# **MultiUUV-Link Prototype Pipeline**
## Communication-Aware Decentralized Multi-UUV Mission Autonomy

**Peacock Dynamics**

**Development Stage:** Jupyter Prototype  
**Foundation:** MicroTrench-2D concepts rebuilt directly within this notebook  
**Primary Objective:** Develop, validate, benchmark, and eventually modularize a decentralized cooperative autonomy architecture for multiple Uncrewed Underwater Vehicles operating under partial perception and limited communication.

---

# Abstract

MultiUUV-Link is an algorithm-first research prototype for communication-aware decentralized multi-UUV mission autonomy.

The project investigates how multiple Uncrewed Underwater Vehicles (UUVs) can cooperatively explore an initially unknown two-dimensional seabed environment while operating with only local perception, persistent private memory, intermittent communication, distributed decision-making, and no continuously available omniscient central controller.

The system is developed using a deliberately lightweight simulation abstraction. Rather than introducing hydrodynamics, six-degree-of-freedom vehicle dynamics, acoustic propagation models, realistic sonar equations, or high-fidelity marine physics during early algorithm development, the environment preserves only the structural elements required to investigate autonomous exploration and distributed coordination.

Each UUV is represented as a point agent moving in a static two-dimensional seabed environment. The vehicle observes only a bounded local Field of View (FOV), maintains its own private history of previously observed regions and discovered resources, and may acquire remote information only through explicit communication with other agents.

The fundamental information constraint is therefore:

$$
K_i(t) \neq K_{\text{world}}
$$

in general, where $K_i(t)$ represents the knowledge possessed by UUV $i$ at time $t$, while $K_{\text{world}}$ represents simulator ground truth.

The simulator itself may maintain complete world information for rendering, evaluation, and debugging, but the autonomy algorithms are prohibited from accessing this information directly.

The project progresses from a rebuilt lightweight MicroTrench-2D-style environment toward multiple autonomous agents, persistent local memory, frontier-based exploration, collision avoidance, dynamic communication graphs, knowledge exchange, distributed task allocation, adaptive replanning, communication degradation, vehicle failure recovery, and a required PyTorch-based learned autonomy component.

The overall development philosophy is:

$$
\boxed{
\text{Build}
\rightarrow
\text{Validate}
\rightarrow
\text{Benchmark}
\rightarrow
\text{Learn}
\rightarrow
\text{Modularize}
}
$$

The final objective is not to claim a physically accurate underwater vehicle simulator.

The objective is to construct a technically transparent experimental platform capable of answering a narrower robotics question:

> How does decentralized autonomous behavior emerge when multiple agents must explore, communicate, allocate work, and recover from failures while possessing incomplete information about the world?

---

# 1. Introduction

Autonomous underwater exploration is fundamentally an information-constrained robotics problem.

A real UUV does not possess a God's-eye representation of the environment.

Instead, knowledge must be accumulated through sensing, navigation, memory, communication, and inference.

Multiple UUVs introduce an additional systems problem.

The mission is no longer simply:

```text
Observe
    ↓
Move
    ↓
Observe again
```

Instead, the system becomes:

```text
Local Observation
       ↓
Private Memory
       ↓
Local Decision
       ↓
Communication Opportunity
       ↓
Knowledge Exchange
       ↓
Distributed Coordination
       ↓
Safety Constraints
       ↓
Motion
       ↓
Environment Changes Relative to Agent
       ↓
Repeat
```

The challenge is therefore not merely moving several markers across an image.

The research problem lies in maintaining a strict distinction between:

```text
what exists
```

and:

```text
what each robot knows exists
```

This distinction defines the architecture of MultiUUV-Link.

---

# 2. Research Question

The central research question is:

> How can multiple UUVs cooperatively explore an initially unknown seabed, discover hidden resources, exchange useful knowledge under limited or intermittent communication, distribute tasks without relying on continuous centralized control, adapt their plans when new information arrives, avoid each other, and continue the mission when communication links or vehicles fail?

Several subordinate questions emerge from this problem.

### Exploration

Can multiple UUVs cover an unknown environment faster than a single UUV?

### Information

Can each vehicle maintain a coherent local world model without accessing simulator ground truth?

### Coordination

Can communication reduce duplicated exploration?

### Allocation

Can newly generated tasks be assigned using only distributed information?

### Communication

How much mission performance is lost when packet loss, latency, limited range, or disconnections are introduced?

### Resilience

Can remaining UUVs recover useful mission performance after another UUV fails?

### Learning

Can a learned decision module outperform an interpretable classical baseline while respecting exactly the same decentralized information constraints?

These questions define the experimental structure of the project.

---

# 3. Development Strategy

MultiUUV-Link follows a Jupyter-first development strategy.

The complete prototype is initially developed inside this notebook.

Stable interfaces and algorithms will be extracted into Python modules only after their behavior has been validated experimentally.

The development sequence is therefore:

$$
\boxed{
\text{Prototype}
\rightarrow
\text{Validate}
\rightarrow
\text{Benchmark}
\rightarrow
\text{Modularize}
}
$$

This avoids prematurely freezing an architecture before the behavior of the system is understood.

The notebook itself acts as:

- executable specification;
- mathematical notebook;
- algorithm-development environment;
- visual debugger;
- experiment workspace;
- architecture prototype;
- record of engineering decisions.

---

# 4. Rebuilding the MicroTrench-2D Foundation

Although MultiUUV-Link conceptually builds upon MicroTrench-2D, the prototype developed here reconstructs the required lightweight environment directly inside the notebook.

This gives complete visibility into the simulator-autonomy boundary before multi-agent behavior is introduced.

The reconstructed foundation will preserve the essential MicroTrench-2D abstractions.

The world consists of:

- a static top-down seabed image;
- hidden resource locations;
- generic resource classes;
- one or more point UUVs;
- discrete planar motion;
- bounded local perception;
- persistent discovery state;
- a global simulator state used only for debugging and evaluation.

The environment intentionally excludes:

- hydrodynamics;
- buoyancy;
- drag;
- currents;
- acceleration dynamics;
- rigid-body vehicle dynamics;
- orientation dynamics;
- depth;
- realistic sonar;
- probabilistic sensing;
- acoustic propagation;
- geological resource modeling;
- extraction mechanics.

This simplification is intentional.

The simulator is designed to preserve the **autonomy problem structure**, not marine physics.

---

# 5. System Boundary

The complete prototype is conceptually divided into two major layers.

```text
┌─────────────────────────────────────────────────────┐
│              SIMULATOR / GROUND TRUTH               │
│                                                     │
│  Seabed Image                                       │
│  Hidden Resources                                   │
│  True UUV Positions                                 │
│  Motion Rules                                       │
│  FOV Geometry                                       │
│  Observation Extraction                             │
│  Discovery Events                                   │
│  Debug Visualization                                │
└───────────────────────┬─────────────────────────────┘
                        │
                        │ LOCAL INFORMATION ONLY
                        ▼
┌─────────────────────────────────────────────────────┐
│                  MULTIUUV-LINK                      │
│                                                     │
│  UUV Agents                                         │
│  Private Memories                                   │
│  Exploration Policies                               │
│  Collision Avoidance                                │
│  Communication Graph                                │
│  Message Exchange                                   │
│  Distributed Tasks                                  │
│  Allocation                                         │
│  Replanning                                         │
│  Failure Recovery                                   │
│  Learned Autonomy                                   │
└─────────────────────────────────────────────────────┘
```

The lower autonomy layer may receive only information that a real agent would legitimately possess within the abstraction.

The upper simulator layer may possess complete ground truth.

No algorithm should silently cross this boundary.

---

# 6. Environment Model

Let the two-dimensional operating domain be:

$$
\Omega
=
\{(x,y)
\mid
0 \le x < W,\;
0 \le y < H
\}
$$

where:

- $W$ is map width;
- $H$ is map height.

The seabed image is represented by:

$$
I:\Omega \rightarrow \mathbb{R}^{3}
$$

for an RGB/BGR image representation.

The image provides spatial texture for perception and visualization.

Pixels initially possess no assumed physical meter-scale interpretation.

---

# 7. UUV State Model

For UUV $i$, the minimum simulator state is:

$$
s_i(t)
=
(x_i(t),y_i(t))
$$

with:

$$
p_i(t)
=
\begin{bmatrix}
x_i(t)\\
y_i(t)
\end{bmatrix}
$$

The initial prototype deliberately excludes:

$$
v_i,\;
a_i,\;
\psi_i,\;
\dot{\psi}_i,\;
z_i,\;
\phi_i,\;
\theta_i
$$

corresponding to velocity, acceleration, heading, angular velocity, depth, roll, and pitch.

The UUV is therefore a kinematic point agent.

This allows the project to isolate mission-autonomy behavior from vehicle-control dynamics.

---

# 8. Action Model

Each UUV initially uses the discrete action space:

$$
a_i(t)
\in
\{0,1,2,3\}
$$

with:

$$
0 = \text{Up}
$$

$$
1 = \text{Right}
$$

$$
2 = \text{Down}
$$

$$
3 = \text{Left}
$$

For a motion step:

$$
\Delta > 0
$$

the nominal displacement vectors are:

$$
u_0 =
\begin{bmatrix}
0\\
-\Delta
\end{bmatrix}
$$

$$
u_1 =
\begin{bmatrix}
\Delta\\
0
\end{bmatrix}
$$

$$
u_2 =
\begin{bmatrix}
0\\
\Delta
\end{bmatrix}
$$

$$
u_3 =
\begin{bmatrix}
-\Delta\\
0
\end{bmatrix}
$$

and therefore:

$$
p_i(t+1)
=
\operatorname{Clamp}
\left(
p_i(t)+u_{a_i(t)},
\Omega
\right)
$$

The clamp operation ensures that the UUV remains inside the image domain.

---

# 9. Local Perception Model

Each vehicle possesses only a bounded local Field of View.

For UUV $i$:

$$
o_i(t)
=
\operatorname{Crop}
\left(
I,p_i(t),FOV_i
\right)
$$

where:

$$
o_i(t)
$$

is the agent's current observation.

For FOV dimensions:

$$
w_F \times h_F
$$

the observation is centered approximately around:

$$
p_i(t)
$$

subject to border handling.

The default prototype will preserve a fixed-size observation using padding near environment boundaries.

The crucial information rule is:

$$
o_i(t)
\neq
I
$$

except in the degenerate case where the FOV covers the complete environment.

---

# 10. Resource Model

Let the environment contain:

$$
N_R
$$

hidden resources.

Resource $j$ is represented as:

$$
R_j
=
(x_j,y_j,c_j,d_j)
$$

where:

- $(x_j,y_j)$ is its true position;
- $c_j$ is its resource class;
- $d_j$ is its discovery state.

Initially:

$$
c_j
\in
\{A,B,C\}
$$

and:

$$
d_j(0)=0
$$

The classes remain generic.

No geological meaning is assigned to classes A, B, or C.

Their purpose is to create heterogeneous task information without making unsupported claims about real mineral deposits.

---

# 11. Resource Discovery

A resource becomes locally discovered when its position falls inside the observing UUV's FOV.

For UUV $i$ and resource $j$:

$$
R_j.position
\in
FOV_i(t)
$$

implies:

$$
d_{ij}^{local}(t):
0
\rightarrow
1
$$

Detection is initially deterministic.

Therefore:

$$
P(\text{detect}\mid R_j\in FOV_i)=1
$$

and:

$$
P(\text{detect}\mid R_j\notin FOV_i)=0
$$

There are initially no:

- false positives;
- false negatives;
- confidence scores;
- sensor noise models;
- range attenuation models.

These are deliberately deferred.

---

# 12. Global Truth Versus Agent Knowledge

This distinction is the most important invariant in the entire project.

The simulator has access to:

$$
K_{\text{world}}
$$

which may include:

- complete seabed image;
- every UUV position;
- every hidden resource;
- every resource class;
- every communication event;
- all task states;
- all failures.

UUV $i$, however, has:

$$
K_i(t)
$$

with:

$$
K_i(t)
\subseteq
K_{\text{world}}
$$

in normal operation.

The UUV may know only:

1. what it observes now;
2. what it observed previously;
3. what another UUV successfully communicated.

Thus:

$$
K_i(t+1)
=
K_i(t)
\cup
O_i(t)
\cup
C_i(t)
$$

where:

- $O_i(t)$ is newly acquired local information;
- $C_i(t)$ is successfully received communication.

Global state must never be used as an invisible shortcut by autonomous decision logic.

---

# 13. Persistent Local Exploration Memory

Every UUV maintains its own exploration memory:

$$
M_i(t)
$$

The memory records which parts of the environment have legitimately been observed by that agent.

Its evolution is:

$$
M_i(t+1)
=
M_i(t)
\cup
FOV_i(t)
$$

Conceptually, cells may initially contain states such as:

```text
UNKNOWN
OBSERVED
RESOURCE_A
RESOURCE_B
RESOURCE_C
```

This creates an important distinction:

$$
M_i(t)
\neq
M_j(t)
$$

for two agents $i\neq j$, unless their observation and communication histories happen to produce identical knowledge.

Private memory is therefore fundamental to decentralized behavior.

---

# 14. Multi-UUV State

The initial swarm size is:

$$
N=4
$$

The collection of agent states is:

$$
S(t)
=
\{
s_1(t),
s_2(t),
\ldots,
s_N(t)
\}
$$

Each agent will independently maintain:

$$
Agent_i
=
\{
id_i,
p_i,
M_i,
D_i,
T_i,
g_i,
H_i
\}
$$

where:

- $id_i$ is the agent identifier;
- $p_i$ is current position;
- $M_i$ is private exploration memory;
- $D_i$ is locally known discovery information;
- $T_i$ represents task state;
- $g_i$ is the current goal;
- $H_i$ is trajectory history.

The agents share an environment but do not automatically share knowledge.

---

# 15. First Multi-Agent Milestone

The first major milestone is intentionally small.

> Four UUVs must coexist inside the same reconstructed MicroTrench environment while receiving only their individual local FOV observations and maintaining independent persistent exploration memories.

At this stage there is:

```text
NO communication
NO auction
NO consensus
NO learned policy
NO failure recovery
```

The milestone exists to validate the information architecture before cooperation is introduced.

Success requires:

- four simultaneous UUV states;
- independent trajectories;
- independent observations;
- independent private memories;
- independent local resource discoveries;
- zero use of global ground truth by agent logic.

---

# 16. Independent Exploration Baseline

Before coordination is introduced, agents must be able to explore independently.

Several classical baselines may be considered:

- random walk;
- deterministic sweeping;
- boustrophedon coverage;
- frontier exploration.

The principal baseline is frontier-based exploration.

A frontier is conceptually the boundary between:

```text
KNOWN
```

and:

```text
UNKNOWN
```

space.

For candidate frontier $g$, UUV $i$ may evaluate:

$$
J_i(g)
=
w_I I_i(g)
-
w_D D_i(g)
-
w_R R_i(g)
$$

where:

- $I_i(g)$ is estimated information gain;
- $D_i(g)$ is travel cost;
- $R_i(g)$ is expected redundancy;
- $w_I,w_D,w_R$ are weighting parameters.

The selected goal is:

$$
g_i^*
=
\arg\max_{g}
J_i(g)
$$

This creates an interpretable classical exploration policy against which later coordination and learning can be compared.

---

# 17. Collision Avoidance

Mission planning and safety are treated as separate logical layers.

For UUVs $i$ and $j$:

$$
d_{ij}(t)
=
\left\|
p_i(t)-p_j(t)
\right\|_2
$$

A safety threshold is defined:

$$
d_{\text{safe}}>0
$$

If:

$$
d_{ij}(t)<d_{\text{safe}}
$$

the safety layer may override the desired motion.

Candidate lightweight strategies include:

- artificial-potential-field repulsion;
- action inhibition;
- priority-based conflict resolution;
- local directional exclusion.

Safety overrides must remain distinct from the exploration objective.

This enables independent measurement of:

```text
mission intent
```

versus:

```text
safety correction
```

---

# 18. Communication Graph

Communication is modeled as a dynamic graph:

$$
G_t=(V,E_t)
$$

where:

$$
V=\{1,2,\ldots,N\}
$$

represents the UUVs.

Initially, communication is range-limited.

An edge exists when:

$$
(i,j)\in E_t
\iff
\left\|
p_i(t)-p_j(t)
\right\|_2
\le
R_c
$$

where:

$$
R_c
$$

is communication range.

Thus the neighbor set of UUV $i$ is:

$$
\mathcal{N}_i(t)
=
\left\{
j:
(i,j)\in E_t
\right\}
$$

The communication topology therefore changes as the swarm moves.

This abstraction models the operational consequence of limited underwater communication without pretending to simulate detailed acoustic propagation.

---

# 19. Knowledge Exchange

When a valid communication link exists, agents may explicitly exchange selected knowledge.

Possible messages include:

- discovered resources;
- explored-region summaries;
- current goals;
- task information;
- bids;
- agent states;
- heartbeat information.

Communication therefore modifies knowledge according to:

$$
K_i(t+1)
=
K_i(t)
\cup
O_i(t)
\cup
C_i(t)
$$

but importantly:

$$
C_i(t)
\neq
K_{\text{world}}
$$

The communication channel may transfer only information actually possessed and transmitted by another agent.

This preserves the decentralized information architecture.

---

# 20. Distributed Task Generation

Mission events may generate tasks.

A generic task is represented by:

$$
T_k
=
(id_k,p_k,type_k,priority_k,status_k)
$$

Possible states include:

```text
UNASSIGNED
BIDDING
ASSIGNED
ACTIVE
COMPLETED
FAILED
```

A discovered resource may therefore become more than a visual marker.

It may create a mission-level object requiring:

- investigation;
- revisit;
- classification;
- monitoring;
- or another abstract operation.

Initial task semantics remain generic.

---

# 21. Distributed Task Allocation

Tasks should not depend on a continuously centralized omniscient controller.

A simple initial auction-style score for UUV $i$ bidding on task $k$ is:

$$
b_{ik}
=
w_pP_k
-
w_dd_{ik}
-
w_qQ_i
-
w_eE_{ik}
$$

where:

- $P_k$ is task priority;
- $d_{ik}$ is distance from UUV $i$ to task $k$;
- $Q_i$ is current workload;
- $E_{ik}$ is estimated execution or travel cost.

A nominal winner is:

$$
i^*
=
\arg\max_i b_{ik}
$$

The challenge is not computing the maximum itself.

The research problem is determining how agents reach consistent task ownership when:

- communication is local;
- information may be stale;
- links may disappear;
- multiple agents may temporarily disagree.

More advanced consensus/auction logic may later be introduced after the simple baseline is validated.

---

# 22. Adaptive Replanning

Autonomous behavior cannot rely on a plan computed once at the beginning of the mission.

Each UUV must respond to events such as:

- new observations;
- new resource discoveries;
- incoming messages;
- newly available tasks;
- task ownership changes;
- collision risks;
- communication loss;
- communication recovery;
- agent failure;
- completed tasks.

The local mission loop is therefore:

```text
Observe
   ↓
Update Private Memory
   ↓
Process Received Messages
   ↓
Update Local Knowledge
   ↓
Update Tasks
   ↓
Select or Replan Goal
   ↓
Apply Safety Layer
   ↓
Execute Motion
   ↓
Transmit Available Information
   ↓
Repeat
```

This loop represents the eventual heart of MultiUUV-Link.

---

# 23. Communication Degradation

Perfect communication provides a necessary baseline but is not the final research condition.

After correct range-limited communication is validated, degradation will be introduced.

## Packet Loss

Let:

$$
P_{\text{loss}}
\in[0,1]
$$

represent packet-loss probability.

Successful delivery probability becomes:

$$
P_{\text{success}}
=
1-P_{\text{loss}}
$$

---

## Latency

For a message transmitted at:

$$
t_{\text{send}}
$$

the arrival time is:

$$
t_{\text{receive}}
=
t_{\text{send}}+\tau
$$

where:

$$
\tau\ge0
$$

is communication delay.

---

## Bandwidth Cost

Communication overhead is measured by:

$$
C_{\text{comm}}
=
\sum_m bytes(m)
$$

This prevents the system from treating unlimited message exchange as free.

---

## Link Failure

A conceptual link-state sequence is:

```text
CONNECTED
    ↓
LINK LOST
    ↓
LOCAL AUTONOMY
    ↓
RECONNECT
    ↓
KNOWLEDGE SYNC
    ↓
CONFLICT RESOLUTION
    ↓
REPLAN
```

Disconnection should therefore degrade cooperation without eliminating local autonomous capability.

---

# 24. Vehicle Failure Recovery

An agent may transition from:

```text
ACTIVE
```

to:

```text
FAILED
```

A heartbeat-based failure condition may use:

$$
t-t_{last}^{i}
>
T_{\text{timeout}}
$$

to infer that UUV $i$ is unavailable.

Tasks owned by a failed UUV may transition toward:

$$
T_k
\rightarrow
UNASSIGNED
$$

allowing surviving agents to redistribute work.

Mission resilience may be summarized using:

$$
R
=
\frac{
Performance_{\text{failure}}
}{
Performance_{\text{nominal}}
}
$$

A resilient system should therefore degrade gracefully rather than catastrophically.

---

# 25. Required PyTorch Learned Component

MultiUUV-Link includes PyTorch as a required project dependency.

Learning is not introduced simply to place machine learning inside the repository.

A learned component must solve a measurable decision problem exposed by the classical autonomy stack.

Potential targets include:

- frontier scoring;
- task-selection scoring;
- communication-value prediction;
- message prioritization;
- mission-state prediction;
- learned replanning;
- learned decentralized coordination.

The exact target is intentionally not predetermined.

The target should be selected only after classical experiments expose a meaningful limitation.

The comparison structure will be:

$$
\boxed{
\text{Classical Baseline}
\rightarrow
\text{Learned Variant}
\rightarrow
\text{Controlled Comparison}
\rightarrow
\text{Ablation}
}
$$

The learned system must obey the same information restrictions as the baseline.

It may not receive simulator ground truth unavailable to the classical controller.

The central learning question is therefore:

> What measurable improvement does learning provide under identical decentralized information constraints?

---

# 26. Coverage Metric

Let:

$$
\Omega_{\text{observed}}(t)
$$

represent the union of legitimately observed map cells.

Coverage is:

$$
Coverage(t)
=
\frac{
|\Omega_{\text{observed}}(t)|
}{
|\Omega|
}
$$

A useful milestone metric is:

$$
T_{90}
=
\min
\left\{
t:
Coverage(t)\ge0.90
\right\}
$$

which measures the time or number of simulation steps required to reach 90% coverage.

---

# 27. Resource Discovery Metric

For:

$$
N_{\text{discovered}}(t)
$$

discovered resources out of:

$$
N_{\text{total}}
$$

total hidden resources:

$$
D(t)
=
\frac{
N_{\text{discovered}}(t)
}{
N_{\text{total}}
}
$$

This measures discovery completeness independently of map coverage.

---

# 28. Exploration Redundancy

Suppose:

$$
A_i
$$

is the area observed by UUV $i$, while:

$$
A_{\cup}
$$

is the union of all observed regions.

A redundancy metric is:

$$
\rho
=
\frac{
\sum_i A_i-A_{\cup}
}{
A_{\cup}
}
$$

Large:

$$
\rho
$$

indicates repeated observation of already explored space.

Communication-aware coordination should ideally reduce redundant exploration while preserving useful overlap where required.

---

# 29. Path Length

For UUV $i$, trajectory length is:

$$
L_i
=
\sum_{t=0}^{T-1}
\left\|
p_i(t+1)-p_i(t)
\right\|_2
$$

Total swarm path length is:

$$
L_{\text{total}}
=
\sum_{i=1}^{N}L_i
$$

This provides a simple proxy for motion effort within the abstract simulator.

It must not be interpreted as a validated real-world energy metric.

---

# 30. Safety Metric

Minimum inter-agent separation is:

$$
d_{\min}
=
\min_{t}
\min_{i\neq j}
\left\|
p_i(t)-p_j(t)
\right\|_2
$$

This gives a direct measure of swarm spatial safety.

Additional safety metrics may include:

- number of threshold violations;
- number of safety overrides;
- near-collision duration;
- conflict-resolution count.

---

# 31. Communication Metrics

Communication performance may be evaluated using:

$$
C_{\text{comm}}
=
\sum_m bytes(m)
$$

together with:

- transmitted packet count;
- received packet count;
- delivery ratio;
- packet loss;
- mean latency;
- communication graph connectivity;
- link uptime;
- reconnection count.

This enables mission performance to be evaluated against communication cost.

---

# 32. Task Allocation Metrics

Task-oriented experiments may measure:

- number of generated tasks;
- assignment latency;
- completion latency;
- assignment conflicts;
- reassignment count;
- duplicate assignment count;
- task completion rate;
- workload distribution.

A simple workload-balance measure may compare individual workloads:

$$
Q_i
$$

against swarm mean workload:

$$
\bar{Q}
=
\frac{1}{N}
\sum_{i=1}^{N}Q_i
$$

The exact balance metric will be selected when the task representation is implemented.

---

# 33. Benchmark Structure

The project is designed around progressive controlled comparisons.

The primary benchmark sequence is:

$$
1\text{ UUV}
$$

versus:

$$
N\text{ independent UUVs}
$$

versus:

$$
N\text{ coordinated UUVs}
$$

versus:

$$
N\text{ coordinated UUVs + limited communication}
$$

versus:

$$
N\text{ coordinated UUVs + communication degradation}
$$

versus:

$$
N\text{ coordinated UUVs + vehicle/link failures}
$$

versus:

$$
N\text{ coordinated UUVs + learned autonomy}
$$

This structure allows every increase in complexity to justify itself quantitatively.

Instead of merely showing that a sophisticated system works, the project asks:

> What did each additional mechanism improve?

---

# 34. Expected Experimental Results

The exact numerical results are not predetermined.

The following are hypotheses to be tested rather than assumed conclusions.

### Hypothesis 1

Multiple independent UUVs should generally reduce exploration time relative to a single UUV, although overlap may create substantial inefficiency.

Expected qualitative relation:

$$
T_{90}^{multi}
<
T_{90}^{single}
$$

under favorable conditions.

---

### Hypothesis 2

Communication-aware coordination should reduce redundant exploration relative to independent multi-UUV behavior.

Expected qualitative relation:

$$
\rho_{\text{coordinated}}
<
\rho_{\text{independent}}
$$

without excessively increasing communication cost.

---

### Hypothesis 3

Collision avoidance should significantly improve minimum separation:

$$
d_{\min}^{safe}
>
d_{\min}^{unsafe}
$$

while introducing some additional path or mission cost.

---

### Hypothesis 4

Communication degradation should reduce swarm coordination quality progressively rather than causing immediate mission collapse.

Performance should ideally vary smoothly with:

$$
P_{\text{loss}},
\quad
\tau,
\quad
R_c
$$

rather than exhibiting brittle all-or-nothing behavior.

---

### Hypothesis 5

When communication becomes unavailable, agents should continue useful local exploration:

$$
Performance_{\text{disconnected}}
>
0
$$

because communication augments local autonomy rather than replacing it.

---

### Hypothesis 6

Following vehicle failure, surviving agents should recover at least part of the lost mission performance through replanning and task redistribution.

Thus:

$$
0<R<1
$$

would represent degraded but useful resilience for many failure scenarios.

---

### Hypothesis 7

The learned autonomy component should be retained only if it produces measurable improvement over its classical baseline.

A useful learned system must satisfy a relation such as:

$$
J_{\text{learned}}
>
J_{\text{classical}}
$$

for a clearly defined evaluation objective $J$, subject to comparable information and resource constraints.

If no meaningful improvement occurs, that result is itself scientifically informative.

---

# 35. Visualization Architecture

The human-facing visualization is allowed to display information unavailable to individual agents.

Possible global overlays include:

- all UUV positions;
- all trajectories;
- individual FOV boundaries;
- communication links;
- current goals;
- task ownership;
- discovered resources;
- failed vehicles;
- communication state;
- coverage;
- mission metrics.

A possible mission HUD is:

```text
Coverage        62.7 %
Discovery       34 / 50
Active UUVs     4 / 4
Comm Packets    184
Packet Loss     10 %
Redundancy      11.8 %
Min Separation  46 px
Tasks Complete  8 / 11
```

This visualization is strictly a debugging and evaluation interface.

It does not define what an agent knows.

Therefore:

$$
\boxed{
\text{Visualization State}
\neq
\text{Agent Knowledge}
}
$$

---

# 36. Planned Prototype Sequence

The notebook is expected to progress through the following implementation stages.

```text
00  Imports and Configuration
        ↓
01  Rebuild Seabed Environment
        ↓
02  Rebuild Point-UUV State
        ↓
03  Rebuild Discrete Motion
        ↓
04  Rebuild Local FOV Extraction
        ↓
05  Rebuild Hidden Resources
        ↓
06  Rebuild Resource Discovery
        ↓
07  Validate MicroTrench-Style Single-UUV Environment
        ↓
08  Create Multiple UUV Agents
        ↓
09  Independent Per-UUV FOVs
        ↓
10  Persistent Private Exploration Memory
        ↓
11  Per-UUV Resource Knowledge
        ↓
12  Independent Exploration Baseline
        ↓
13  Frontier / Coverage Exploration
        ↓
14  Collision Avoidance
        ↓
15  Dynamic Communication Graph
        ↓
16  Explicit Knowledge Exchange
        ↓
17  Distributed Task Generation
        ↓
18  Distributed Task Allocation
        ↓
19  Adaptive Replanning
        ↓
20  Packet Loss / Latency / Link Failure
        ↓
21  Vehicle Failure Injection
        ↓
22  Failure Recovery
        ↓
23  Classical Benchmark Suite
        ↓
24  PyTorch Learned Autonomy Component
        ↓
25  Classical vs Learned Comparison
        ↓
26  Ablation Experiments
        ↓
27  Metrics and Analysis
        ↓
28  Final Visualization / Rollouts
        ↓
29  Architecture Review
        ↓
30  Modularization
```

The exact number of implementation cells may differ because development occurs using validated incremental combos.

---

# 37. Combo-Based Development Protocol

This notebook is developed incrementally using a strict two-cell engineering unit called a **Combo**.

Each Combo contains exactly:

```text
1 Markdown Cell
+
1 Code Cell
```

The Markdown cell explains:

- the system concept;
- relevant mathematics;
- assumptions;
- interfaces;
- information constraints;
- expected behavior;
- expected output.

The Code cell implements only the behavior defined immediately above it.

After execution:

```text
Implement Combo
       ↓
Run Code
       ↓
Inspect Output
       ↓
Evaluate Correctness
       ↓
Discuss / Challenge
       ↓
Approve
       ↓
Proceed
```

A subsequent Combo should not be introduced until the current Combo has been accepted.

This creates an incremental verification process where architectural errors are caught close to their origin.

---

# 38. Technical Honesty

MultiUUV-Link is an algorithm-development environment.

The prototype does not claim to reproduce real underwater vehicle physics.

The following abstractions must be kept explicit.

Pixels do not initially represent calibrated physical distance.

Discrete motion does not represent thruster or rigid-body dynamics.

The rectangular FOV does not represent a validated camera or sonar sensor.

Perfect resource detection does not represent a physical mineral-detection system.

Range-limited graph communication does not reproduce real underwater acoustic propagation.

Communication bandwidth is initially an abstract algorithmic quantity.

Generic resource classes A, B, and C are not geological classifications.

Path length is not automatically equivalent to energy expenditure.

Simulation steps are not automatically equivalent to real-world seconds.

Therefore the system should not be used to claim validated predictions regarding:

- mission duration;
- power consumption;
- real UUV trajectories;
- acoustic communication performance;
- geological resource probability;
- mining feasibility;
- real-world operational safety.

The purpose is narrower:

> Develop and evaluate decentralized exploration, communication, task allocation, replanning, learning, and resilience algorithms under a controlled partial-information abstraction.

---

# 39. Deferred Scope

The following capabilities are deliberately deferred until the lightweight autonomy stack has demonstrated meaningful behavior:

- ROS 2;
- Gazebo;
- realistic UUV geometry;
- rigid-body dynamics;
- 3D motion;
- depth control;
- yaw/pitch/roll dynamics;
- hydrodynamics;
- buoyancy;
- drag;
- current models;
- thruster models;
- sonar equations;
- real acoustic propagation;
- sensor uncertainty;
- localization uncertainty;
- bathymetric reconstruction;
- geological mineral distributions;
- mining and extraction mechanics;
- hardware deployment.

Complexity will be introduced only when a research question requires it.

---

# 40. Expected Modular Architecture

After notebook validation, stable components may be extracted into:

```text
multiuuv_link/
│
├── __init__.py
├── agent.py
├── memory.py
├── planner.py
├── communication.py
├── allocation.py
├── safety.py
├── mission.py
├── learning.py
└── metrics.py
```

Conceptually:

### `agent.py`

Owns individual UUV lifecycle and state.

### `memory.py`

Owns private explored maps, discoveries, timestamps, and knowledge merging.

### `planner.py`

Owns frontier detection, goal generation, scoring, and replanning.

### `communication.py`

Owns graph construction, neighbors, messaging, latency, packet loss, and bandwidth accounting.

### `allocation.py`

Owns task bidding, assignment, conflict resolution, and reassignment.

### `safety.py`

Owns minimum-separation enforcement and motion overrides.

### `mission.py`

Orchestrates simulation execution without becoming a centralized omniscient planner.

### `learning.py`

Owns PyTorch models, features, training utilities, and learned inference.

### `metrics.py`

Owns quantitative experiment evaluation.

The notebook determines these interfaces experimentally before they are frozen into source files.

---

# 41. Long-Term Transition

The intended development path is:

```text
Rebuilt MicroTrench-Style Environment
              ↓
MultiUUV-Link Notebook Prototype
              ↓
Validated Partial-Information Architecture
              ↓
Classical Decentralized Autonomy
              ↓
Communication-Aware Coordination
              ↓
Failure-Resilient Mission Behavior
              ↓
PyTorch Learned Extension
              ↓
Controlled Benchmarks
              ↓
Ablation Studies
              ↓
Modular Python Package
              ↓
ROS 2 Architecture
              ↓
Higher-Fidelity Marine Simulation
              ↓
Physical UUV Research
```

The lightweight notebook therefore serves as the first rung of an incremental-fidelity development ladder.

---

# 42. Project Success Criteria

MultiUUV-Link will be considered successful at the notebook-prototype level if it demonstrates, with reproducible evidence, that:

1. multiple UUVs operate simultaneously;
2. each vehicle receives only local perception;
3. each vehicle maintains private persistent knowledge;
4. no autonomy policy accesses hidden global ground truth;
5. autonomous exploration operates without manual control;
6. collision risks are handled by a dedicated safety layer;
7. communication topology changes with vehicle geometry;
8. knowledge spreads only through explicit messages;
9. coordination reduces measurable inefficiency under suitable conditions;
10. tasks can be generated and distributed without continuous omniscient central control;
11. agents replan after relevant mission events;
12. communication degradation can be simulated and measured;
13. agents retain useful local autonomy during disconnection;
14. vehicle failures trigger meaningful redistribution or replanning;
15. communication and resilience costs are quantitatively measurable;
16. a meaningful PyTorch learned component is implemented;
17. the learned component is compared fairly against a classical baseline;
18. conclusions are supported by metrics rather than visualization alone;
19. the complete prototype remains lightweight and reproducible;
20. validated components can be cleanly modularized afterward.

---

# 43. Expected Final Deliverables

The prototype should eventually generate reproducible artifacts such as:

```text
outputs/
│
├── figures/
│   ├── coverage_curve.png
│   ├── discovery_curve.png
│   ├── redundancy_comparison.png
│   ├── communication_cost.png
│   ├── task_performance.png
│   ├── failure_recovery.png
│   └── learned_vs_classical.png
│
├── videos/
│   └── swarm_rollout.mp4
│
└── logs/
    └── experiment_results.csv
```

The exact artifact set will be determined by the experiments that survive validation.

Generated outputs remain separate from source code.

---

# 44. Peacock Dynamics Work Closure

MultiUUV-Link extends the Peacock Dynamics lightweight-autonomy philosophy into decentralized underwater robotics.

The project begins with an intentionally small abstraction:

```text
Static Seabed
      ↓
Point UUV
      ↓
Local Perception
      ↓
Private Knowledge
```

and progressively introduces:

```text
Multiple Agents
      ↓
Autonomous Exploration
      ↓
Safety
      ↓
Communication
      ↓
Knowledge Exchange
      ↓
Task Allocation
      ↓
Replanning
      ↓
Failure Recovery
      ↓
Learned Autonomy
```

The guiding engineering principle is that complexity must justify its existence through measurable system behavior.

MultiUUV-Link therefore does not begin with the most sophisticated simulator, the largest neural network, or the most realistic underwater model.

It begins with the smallest system capable of exposing the research problem clearly.

The simulator provides truth.

The UUVs receive observations.

Memory produces knowledge.

Communication distributes knowledge.

Planning converts knowledge into intent.

Safety constrains intent.

Allocation distributes work.

Replanning adapts intent.

Learning is introduced only where classical decision-making reveals a measurable limitation.

Evaluation determines whether each additional mechanism earned its place.

The final engineering philosophy can therefore be summarized as:

$$
\boxed{
\text{Local Perception}
\rightarrow
\text{Persistent Knowledge}
\rightarrow
\text{Autonomous Exploration}
\rightarrow
\text{Limited Communication}
\rightarrow
\text{Distributed Cooperation}
\rightarrow
\text{Adaptive Mission Autonomy}
}
$$

---

## Peacock Dynamics

### MultiUUV-Link

**Communication-Aware Decentralized Multi-UUV Mission Autonomy**

*Build the smallest truthful system first. Add complexity only when the experiment earns it.*

---

# Combo 1 — Imports and Global Configuration

## Objective

This first combo establishes the computational environment and the global configuration that will be shared by the rest of the MultiUUV-Link prototype.

No simulator behavior is implemented yet.

This cell only verifies that the notebook has access to the numerical, image-processing, plotting, graph, tabular-data, and PyTorch libraries required by the planned architecture.

The prototype will use:

- **NumPy** for numerical state, coordinates, masks, maps, and array operations;
- **OpenCV** for loading and manipulating the seabed image and later extracting local FOV observations;
- **Matplotlib** for notebook-native visualization and experiment plots;
- **NetworkX** for the later dynamic UUV communication graph;
- **Pandas** for experiment logs and benchmark tables;
- **PyTorch** for the required learned autonomy component.

The development plan deliberately includes PyTorch from the beginning even though the first stages of MultiUUV-Link will use classical autonomy algorithms.

---

## Reproducibility

Stochastic components will eventually include:

- UUV initial positions;
- hidden-resource locations;
- resource classes;
- communication packet loss;
- failure injection;
- potentially learned-model initialization and training.

A common random seed is therefore defined:

$$
s = 42
$$

The exact value is not mathematically important.

What matters is that repeated experiments can reproduce the same pseudorandom sequence when required.

For NumPy:

$$
R_{\mathrm{NumPy}} = f(s)
$$

and for PyTorch:

$$
R_{\mathrm{Torch}} = g(s)
$$

where $s$ is the shared seed.

---

## Initial Prototype Constants

The initial planned swarm size is:

$$
N = 4
$$

The rebuilt MicroTrench-style environment will initially use a square local FOV:

$$
w_F = h_F = 128 \text{ pixels}
$$

and a discrete movement step:

$$
\Delta = 8 \text{ pixels}
$$

The initial hidden-resource count is:

$$
N_R = 20
$$

These values reproduce the lightweight scale of the original MicroTrench-2D concept while providing a clean baseline for later multi-agent experiments.

They are configuration parameters rather than physical quantities.

In particular:

```text
1 pixel ≠ 1 meter
1 simulation step ≠ 1 second
```

unless a physical scaling model is explicitly introduced later.

---

## File Structure Assumption

The notebook is expected to live at:

```text
MultiUUV-Link/
└── notebooks/
    └── MultiUUV_Link_Prototype.ipynb
```

while the simulator seabed image will live at:

```text
MultiUUV-Link/
└── assets/
    └── sample_map.png
```

Therefore the notebook-relative asset path is expected to be:

```text
../assets/sample_map.png
```

The image is **not loaded in this combo**.

That will happen only after the computational environment has been validated.

---

## Expected Output

Running the following code cell should:

1. import all required libraries without errors;
2. initialize deterministic random seeds;
3. define the initial global configuration;
4. report the installed library versions;
5. report whether PyTorch can see a CUDA-capable GPU;
6. print the initial prototype parameters.

A successful output establishes that the notebook environment is ready for the actual simulator reconstruction.

---

In [ ]:
# ============================================================
# Combo 1 — Imports and Global Configuration
# ============================================================

from pathlib import Path
import random

import numpy as np
import cv2
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import torch


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Repository / asset paths
# ------------------------------------------------------------

NOTEBOOK_DIR = Path.cwd()
ASSETS_DIR = NOTEBOOK_DIR.parent / "assets"

SEABED_IMAGE_PATH = ASSETS_DIR / "sample_map.png"


# ------------------------------------------------------------
# Initial prototype configuration
# ------------------------------------------------------------

NUM_UUVS = 4

FOV_WIDTH = 128
FOV_HEIGHT = 128

STEP_SIZE = 8

NUM_RESOURCES = 20

RESOURCE_CLASSES = ("A", "B", "C")


# ------------------------------------------------------------
# Environment report
# ------------------------------------------------------------

print("=" * 60)
print("MultiUUV-Link — Prototype Environment")
print("=" * 60)

print("\nLibrary versions:")
print(f"  NumPy      : {np.__version__}")
print(f"  OpenCV     : {cv2.__version__}")
print(f"  Matplotlib : {plt.matplotlib.__version__}")
print(f"  NetworkX   : {nx.__version__}")
print(f"  Pandas     : {pd.__version__}")
print(f"  PyTorch    : {torch.__version__}")

print("\nCompute:")
print(f"  CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  CUDA device    : {torch.cuda.get_device_name(0)}")

print("\nPrototype configuration:")
print(f"  Random seed       : {SEED}")
print(f"  Number of UUVs    : {NUM_UUVS}")
print(f"  FOV               : {FOV_WIDTH} x {FOV_HEIGHT} px")
print(f"  Motion step       : {STEP_SIZE} px")
print(f"  Hidden resources  : {NUM_RESOURCES}")
print(f"  Resource classes  : {RESOURCE_CLASSES}")

print("\nPaths:")
print(f"  Notebook cwd      : {NOTEBOOK_DIR}")
print(f"  Assets directory  : {ASSETS_DIR}")
print(f"  Seabed image      : {SEABED_IMAGE_PATH}")
print(f"  Image exists      : {SEABED_IMAGE_PATH.exists()}")

print("\nCombo 1 initialization complete.")

# Combo 2 — Load and Validate the Seabed Environment

## Objective

This combo reconstructs the first actual MicroTrench-2D environment component: the static seabed world.

At this stage, the environment contains only the background seabed image.

There is still:

```text
NO UUV
NO motion
NO FOV
NO hidden resources
NO discovery logic
NO communication
```

The purpose is to validate the world representation before any state or autonomy is placed on top of it.

---

## Environment Representation

The seabed is represented as a static image:

$$
I:\Omega \rightarrow \mathbb{R}^{3}
$$

where the spatial domain is:

$$
\Omega
=
\{(x,y)\mid 0\le x<W,\;0\le y<H\}
$$

and:

- $W$ is image width;
- $H$ is image height.

For an 8-bit three-channel image:

$$
I(x,y)
\in
\{0,\ldots,255\}^{3}
$$

The image therefore acts as the spatial substrate on which later UUV motion and sensing will occur.

---

## Coordinate Convention

OpenCV stores an image array using:

```text
image[y, x, channel]
```

not:

```text
image[x, y, channel]
```

Therefore:

$$
\text{array shape}
=
(H,W,C)
$$

with:

$$
C=3
$$

for a standard color image.

Throughout MultiUUV-Link we will use the geometric coordinate convention:

$$
p=
(x,y)
$$

while remembering that NumPy/OpenCV indexing is:

$$
I[y,x]
$$

This distinction is extremely important and will later affect:

- movement;
- FOV extraction;
- resource placement;
- memory masks;
- trajectories;
- communication geometry.

---

## OpenCV Color Convention

OpenCV loads standard color images using BGR ordering:

$$
I_{\mathrm{cv}}
=
(B,G,R)
$$

while Matplotlib expects RGB ordering:

$$
I_{\mathrm{mpl}}
=
(R,G,B)
$$

Therefore the image used for numerical simulation may remain in OpenCV's native BGR representation, while a converted RGB copy is used for notebook visualization.

The transformation is:

$$
I_{\mathrm{RGB}}
=
\operatorname{BGR2RGB}(I_{\mathrm{BGR}})
$$

No autonomous information is created by this transformation.

It changes only channel ordering.

---

## Validation Conditions

The environment image should satisfy:

$$
I \neq \varnothing
$$

and:

$$
H>0,\qquad W>0
$$

with either:

$$
C=3
$$

or a representation that we explicitly convert into three channels.

For the current prototype, the expected final internal representation is:

```text
uint8
H × W × 3
BGR
```

with pixel range:

$$
0\le I(x,y,c)\le255
$$

---

## Information Boundary

At this point the full seabed image is simulator ground truth.

Later, individual UUVs must not directly receive this complete array.

They will instead receive local crops:

$$
o_i(t)
=
\operatorname{Crop}
\left(
I,p_i(t),FOV_i
\right)
$$

Therefore this combo establishes:

```text
GLOBAL WORLD IMAGE
```

but does not yet create:

```text
AGENT OBSERVATION
```

That separation will be enforced when the FOV system is implemented.

---

## Expected Output

The code cell should:

1. load `assets/sample_map.png`;
2. verify that the file is readable;
3. ensure the image has three color channels;
4. report:
   - width;
   - height;
   - number of channels;
   - data type;
   - minimum pixel value;
   - maximum pixel value;
   - approximate memory usage;
5. convert a copy from BGR to RGB for Matplotlib;
6. display the raw seabed image;
7. draw no simulator overlays of any kind.

A successful result establishes:

$$
\boxed{
\text{Valid Static Seabed World}
}
$$

which becomes the spatial foundation for all subsequent simulator components.

---

In [ ]:
# ============================================================
# Combo 2 — Load and Validate the Seabed Environment
# ============================================================

# ------------------------------------------------------------
# Load seabed image
# ------------------------------------------------------------

seabed_bgr = cv2.imread(str(SEABED_IMAGE_PATH), cv2.IMREAD_COLOR)

if seabed_bgr is None:
    raise FileNotFoundError(
        f"OpenCV could not load the seabed image:\n{SEABED_IMAGE_PATH}"
    )


# ------------------------------------------------------------
# Validate representation
# ------------------------------------------------------------

if seabed_bgr.ndim != 3:
    raise ValueError(
        f"Expected a 3D color image array, got shape {seabed_bgr.shape}"
    )

HEIGHT, WIDTH, CHANNELS = seabed_bgr.shape

if CHANNELS != 3:
    raise ValueError(
        f"Expected 3 color channels, got {CHANNELS}"
    )

if HEIGHT <= 0 or WIDTH <= 0:
    raise ValueError(
        f"Invalid image dimensions: {WIDTH} x {HEIGHT}"
    )


# ------------------------------------------------------------
# Pixel statistics
# ------------------------------------------------------------

pixel_min = int(seabed_bgr.min())
pixel_max = int(seabed_bgr.max())

memory_bytes = seabed_bgr.nbytes
memory_mb = memory_bytes / (1024 ** 2)


# ------------------------------------------------------------
# Visualization copy
# ------------------------------------------------------------

seabed_rgb = cv2.cvtColor(
    seabed_bgr,
    cv2.COLOR_BGR2RGB
)


# ------------------------------------------------------------
# Environment report
# ------------------------------------------------------------

print("=" * 60)
print("MultiUUV-Link — Seabed Environment")
print("=" * 60)

print(f"\nImage path       : {SEABED_IMAGE_PATH}")
print(f"Width            : {WIDTH} px")
print(f"Height           : {HEIGHT} px")
print(f"Channels         : {CHANNELS}")
print(f"Array shape      : {seabed_bgr.shape}")
print(f"Data type        : {seabed_bgr.dtype}")
print(f"Pixel range      : [{pixel_min}, {pixel_max}]")
print(f"Memory footprint : {memory_mb:.2f} MiB")

print("\nInternal convention:")
print("  Geometry       : (x, y)")
print("  Array indexing : image[y, x]")
print("  Simulation     : BGR")
print("  Visualization  : RGB")

print("\nStatic seabed environment validated.")


# ------------------------------------------------------------
# Raw world visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))
plt.imshow(seabed_rgb)
plt.title("MultiUUV-Link — Raw Seabed Environment")
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.tight_layout()
plt.show()

# Combo 3 — Point-UUV State Model

## Objective

This combo introduces the first simulated vehicle state into the environment.

The UUV is represented as a **point agent** with only planar image coordinates:

$$
s(t) = (x(t), y(t))
$$

No motion is implemented yet.

This combo only creates, validates, and visualizes one UUV state on top of the static seabed world.

The current system still contains:

```text
NO autonomous policy
NO movement
NO FOV extraction
NO resources
NO memory
NO communication
```

---

## State Definition

The UUV position is:

$$
p(t)
=
\begin{bmatrix}
x(t)\\
y(t)
\end{bmatrix}
$$

subject to the environment bounds:

$$
0 \le x(t) < W
$$

$$
0 \le y(t) < H
$$

where:

$$
W = 1448
$$

and:

$$
H = 1086
$$

for the currently loaded seabed map.

The UUV therefore exists in the same coordinate frame as the image.

---

## Image Coordinate Convention

The geometric state is stored as:

```text
(x, y)
```

while image indexing remains:

```text
image[y, x]
```

The coordinate axes therefore behave as:

```text
+x → right
+y → downward
```

This is an image-space coordinate system rather than the conventional Cartesian plane.

---

## Why a Point Agent?

The current research target is mission autonomy rather than vehicle dynamics.

Therefore the UUV intentionally has no:

- heading;
- velocity;
- acceleration;
- depth;
- roll;
- pitch;
- yaw;
- mass;
- inertia.

The complete state is currently:

$$
s(t) = (x(t), y(t))
$$

rather than something such as:

$$
s(t)
=
(x,y,z,v_x,v_y,v_z,\phi,\theta,\psi,\ldots)
$$

This prevents the simulator from implying physical fidelity that it does not possess.

---

## Initial Position

For this validation combo, the first UUV will be initialized at the geometric center of the map:

$$
x_0
=
\left\lfloor
\frac{W}{2}
\right\rfloor
$$

$$
y_0
=
\left\lfloor
\frac{H}{2}
\right\rfloor
$$

For the current environment this should be approximately:

$$
x_0 = 724
$$

$$
y_0 = 543
$$

The exact values will be computed from the loaded image rather than hard-coded.

---

## State Representation

We will introduce a small data structure representing UUV state.

The minimum fields are:

```text
id
x
y
```

Even though only one UUV exists right now, including an identifier from the start prepares the state representation for the planned multi-agent extension.

For UUV $i$:

$$
s_i(t)
=
(id_i,x_i(t),y_i(t))
$$

At this stage the identifier carries no dynamics.

It only distinguishes agents.

---

## Boundary Invariant

Any valid UUV state must satisfy:

$$
0 \le x_i < W
$$

and:

$$
0 \le y_i < H
$$

The code will explicitly validate this invariant.

Boundary handling during movement will be implemented in a later combo.

---

## Visualization

The human-facing debug view may display the true UUV position.

This is allowed because the plot represents simulator ground truth.

The rendered marker does **not** become part of the seabed image used later for perception.

Therefore:

$$
I_{\text{ground truth}}
\neq
I_{\text{debug render}}
$$

The UUV marker exists only in a copied visualization frame.

This prevents future local observations from accidentally seeing graphical annotations that were added for humans.

---

## Expected Output

The code cell should:

1. define a minimal UUV state structure;
2. initialize `UUV-0` at the center of the map;
3. validate its coordinates against the environment bounds;
4. print its state;
5. create a copy of the RGB seabed image;
6. plot the UUV as a point marker at its true position;
7. leave the original seabed arrays unchanged.

A successful result establishes:

$$
\boxed{
\text{Static World}
+
\text{Valid Point-UUV State}
}
$$

without yet introducing motion or perception.

---

In [ ]:
# ============================================================
# Combo 3 — Point-UUV State Model
# ============================================================

from dataclasses import dataclass


# ------------------------------------------------------------
# UUV state definition
# ------------------------------------------------------------

@dataclass
class UUVState:
    uuv_id: int
    x: int
    y: int


# ------------------------------------------------------------
# Initialize one UUV at the map center
# ------------------------------------------------------------

initial_x = WIDTH // 2
initial_y = HEIGHT // 2

uuv0 = UUVState(
    uuv_id=0,
    x=initial_x,
    y=initial_y,
)


# ------------------------------------------------------------
# Validate state bounds
# ------------------------------------------------------------

if not (0 <= uuv0.x < WIDTH):
    raise ValueError(
        f"UUV x-coordinate out of bounds: {uuv0.x}"
    )

if not (0 <= uuv0.y < HEIGHT):
    raise ValueError(
        f"UUV y-coordinate out of bounds: {uuv0.y}"
    )


# ------------------------------------------------------------
# Verify that simulator ground truth remains untouched
# ------------------------------------------------------------

seabed_rgb_before = seabed_rgb.copy()


# ------------------------------------------------------------
# State report
# ------------------------------------------------------------

print("=" * 60)
print("MultiUUV-Link — Point-UUV State")
print("=" * 60)

print(f"\nUUV ID           : {uuv0.uuv_id}")
print(f"Position         : ({uuv0.x}, {uuv0.y})")
print(f"Map bounds       : x=[0, {WIDTH - 1}], y=[0, {HEIGHT - 1}]")
print(f"State valid      : True")


# ------------------------------------------------------------
# Debug visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))
plt.imshow(seabed_rgb)

plt.scatter(
    uuv0.x,
    uuv0.y,
    s=90,
    marker="o",
)

plt.title("MultiUUV-Link — Point-UUV State")
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Integrity check
# ------------------------------------------------------------

ground_truth_unchanged = np.array_equal(
    seabed_rgb_before,
    seabed_rgb
)

print(f"Ground truth unchanged : {ground_truth_unchanged}")

if not ground_truth_unchanged:
    raise RuntimeError(
        "Debug visualization modified the seabed ground-truth image."
    )

print("\nPoint-UUV state validated.")

# Combo 4 — Discrete Four-Direction Motion

## Objective

This combo gives the point-UUV a deterministic planar motion model.

The UUV will move using four discrete actions:

$$
a(t)\in\{0,1,2,3\}
$$

with:

$$
0=\text{Up}
$$

$$
1=\text{Right}
$$

$$
2=\text{Down}
$$

$$
3=\text{Left}
$$

The motion increment is:

$$
\Delta = 8\text{ pixels}
$$

as defined in the global configuration.

This combo implements only:

```text
state
+
discrete motion
+
boundary clamping
```

There is still:

```text
NO FOV extraction
NO resource generation
NO resource discovery
NO memory
NO planning
NO communication
```

---

## Motion Model

For UUV position:

$$
p(t)=
\begin{bmatrix}
x(t)\\
y(t)
\end{bmatrix}
$$

the four motion vectors are:

$$
u_0=
\begin{bmatrix}
0\\
-\Delta
\end{bmatrix}
$$

$$
u_1=
\begin{bmatrix}
\Delta\\
0
\end{bmatrix}
$$

$$
u_2=
\begin{bmatrix}
0\\
\Delta
\end{bmatrix}
$$

$$
u_3=
\begin{bmatrix}
-\Delta\\
0
\end{bmatrix}
$$

The unconstrained transition is:

$$
\tilde{p}(t+1)
=
p(t)+u_{a(t)}
$$

Because the UUV must remain inside the map, the actual transition is:

$$
p(t+1)
=
\operatorname{Clamp}
\left(
\tilde{p}(t+1),
\Omega
\right)
$$

where:

$$
\Omega=
\{(x,y)\mid 0\le x<W,\;0\le y<H\}
$$

---

## Boundary Clamping

For the x-coordinate:

$$
x_{t+1}
=
\min
\left(
W-1,
\max(0,\tilde{x}_{t+1})
\right)
$$

and for the y-coordinate:

$$
y_{t+1}
=
\min
\left(
H-1,
\max(0,\tilde{y}_{t+1})
\right)
$$

This means the simulator never creates an invalid state.

For example, if the UUV is already at:

$$
x=0
$$

and receives the action:

$$
a=3
$$

corresponding to left motion, then:

$$
x_{t+1}=0
$$

rather than:

$$
x_{t+1}=-8
$$

---

## Functional Separation

The motion function should depend only on:

```text
current UUV state
action
step size
map dimensions
```

It should not depend on:

- visualization;
- resources;
- FOV;
- memory;
- future planner logic.

Conceptually:

$$
f:
(s_t,a_t)
\rightarrow
s_{t+1}
$$

This keeps the motion primitive independent from the autonomy system that will later choose the action.

---

## State Update Policy

The function will return a **new UUV state** rather than mutating the input state in place.

Thus:

$$
s_t
\rightarrow
s_{t+1}
$$

while preserving the original object for inspection.

This is useful during early notebook development because transitions can be validated explicitly.

---

## Validation Sequence

Starting from:

$$
p_0=(724,543)
$$

we will execute:

```text
UP
RIGHT
DOWN
LEFT
```

Using:

$$
\Delta=8
$$

the expected transitions are:

### Up

$$
(724,543)
\rightarrow
(724,535)
$$

### Right

$$
(724,535)
\rightarrow
(732,535)
$$

### Down

$$
(732,535)
\rightarrow
(732,543)
$$

### Left

$$
(732,543)
\rightarrow
(724,543)
$$

Therefore the complete sequence should return the UUV to its initial position:

$$
p_4=p_0
$$

---

## Boundary Tests

The combo will also explicitly test clamping.

At the upper-left corner:

$$
p=(0,0)
$$

an additional `UP` or `LEFT` action must leave the corresponding coordinate unchanged.

At the lower-right corner:

$$
p=(W-1,H-1)
$$

an additional `RIGHT` or `DOWN` action must likewise remain bounded.

These are important invariants before any exploration algorithm is allowed to control the vehicle.

---

## Visualization

The executed path will be rendered on a copy of the seabed image.

The visualization may display:

- initial position;
- intermediate states;
- connected trajectory.

The seabed ground-truth array itself must remain unchanged.

Therefore:

$$
I_{\text{simulation}}
\neq
I_{\text{rendered overlay}}
$$

---

## Expected Output

The code cell should:

1. define the action mapping;
2. define a pure discrete-motion function;
3. execute the sequence `UP → RIGHT → DOWN → LEFT`;
4. verify that the final state equals the initial state;
5. test all four boundary-clamping cases;
6. print each state transition;
7. plot the trajectory;
8. verify that the seabed image remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Valid Point State}
+
\text{Discrete Motion}
+
\text{Boundary Safety}
}
$$

---

In [ ]:
# ============================================================
# Combo 4 — Discrete Four-Direction Motion
# ============================================================

# ------------------------------------------------------------
# Action definitions
# ------------------------------------------------------------

ACTION_UP = 0
ACTION_RIGHT = 1
ACTION_DOWN = 2
ACTION_LEFT = 3

ACTION_NAMES = {
    ACTION_UP: "UP",
    ACTION_RIGHT: "RIGHT",
    ACTION_DOWN: "DOWN",
    ACTION_LEFT: "LEFT",
}

ACTION_DELTAS = {
    ACTION_UP: (0, -STEP_SIZE),
    ACTION_RIGHT: (STEP_SIZE, 0),
    ACTION_DOWN: (0, STEP_SIZE),
    ACTION_LEFT: (-STEP_SIZE, 0),
}


# ------------------------------------------------------------
# Motion primitive
# ------------------------------------------------------------

def move_uuv(state: UUVState, action: int) -> UUVState:
    if action not in ACTION_DELTAS:
        raise ValueError(
            f"Invalid action {action}. "
            f"Valid actions are {tuple(ACTION_DELTAS.keys())}."
        )

    dx, dy = ACTION_DELTAS[action]

    raw_x = state.x + dx
    raw_y = state.y + dy

    new_x = int(np.clip(raw_x, 0, WIDTH - 1))
    new_y = int(np.clip(raw_y, 0, HEIGHT - 1))

    return UUVState(
        uuv_id=state.uuv_id,
        x=new_x,
        y=new_y,
    )


# ------------------------------------------------------------
# Nominal motion sequence
# ------------------------------------------------------------

test_actions = [
    ACTION_UP,
    ACTION_RIGHT,
    ACTION_DOWN,
    ACTION_LEFT,
]

trajectory = [uuv0]
current_state = uuv0

print("=" * 60)
print("MultiUUV-Link — Discrete Motion")
print("=" * 60)

print(f"\nInitial state : ({current_state.x}, {current_state.y})")

for step_index, action in enumerate(test_actions, start=1):
    next_state = move_uuv(current_state, action)

    print(
        f"Step {step_index} | "
        f"{ACTION_NAMES[action]:>5} | "
        f"({current_state.x}, {current_state.y}) "
        f"-> ({next_state.x}, {next_state.y})"
    )

    trajectory.append(next_state)
    current_state = next_state


# ------------------------------------------------------------
# Closed-loop position check
# ------------------------------------------------------------

returned_to_start = (
    current_state.x == uuv0.x
    and current_state.y == uuv0.y
)

print(f"\nReturned to start : {returned_to_start}")

if not returned_to_start:
    raise RuntimeError(
        "UP → RIGHT → DOWN → LEFT did not return to the initial state."
    )


# ------------------------------------------------------------
# Boundary-clamping tests
# ------------------------------------------------------------

top_left = UUVState(
    uuv_id=99,
    x=0,
    y=0,
)

bottom_right = UUVState(
    uuv_id=99,
    x=WIDTH - 1,
    y=HEIGHT - 1,
)

test_top = move_uuv(top_left, ACTION_UP)
test_left = move_uuv(top_left, ACTION_LEFT)

test_right = move_uuv(bottom_right, ACTION_RIGHT)
test_down = move_uuv(bottom_right, ACTION_DOWN)

boundary_checks = {
    "UP at y=0": (
        test_top.x == 0
        and test_top.y == 0
    ),
    "LEFT at x=0": (
        test_left.x == 0
        and test_left.y == 0
    ),
    "RIGHT at x=W-1": (
        test_right.x == WIDTH - 1
        and test_right.y == HEIGHT - 1
    ),
    "DOWN at y=H-1": (
        test_down.x == WIDTH - 1
        and test_down.y == HEIGHT - 1
    ),
}

print("\nBoundary checks:")

for name, passed in boundary_checks.items():
    print(f"  {name:<18}: {passed}")

if not all(boundary_checks.values()):
    raise RuntimeError(
        "At least one boundary-clamping invariant failed."
    )


# ------------------------------------------------------------
# Visualization integrity
# ------------------------------------------------------------

seabed_rgb_before_motion_plot = seabed_rgb.copy()

trajectory_x = [state.x for state in trajectory]
trajectory_y = [state.y for state in trajectory]


# ------------------------------------------------------------
# Trajectory visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))
plt.imshow(seabed_rgb)

plt.plot(
    trajectory_x,
    trajectory_y,
    marker="o",
    linewidth=2,
)

plt.scatter(
    trajectory_x[0],
    trajectory_y[0],
    s=130,
    marker="o",
    label="Start / End",
)

plt.title("MultiUUV-Link — Discrete Motion Validation")
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Ground-truth integrity check
# ------------------------------------------------------------

ground_truth_unchanged = np.array_equal(
    seabed_rgb_before_motion_plot,
    seabed_rgb,
)

print(
    f"\nGround truth unchanged : "
    f"{ground_truth_unchanged}"
)

if not ground_truth_unchanged:
    raise RuntimeError(
        "Trajectory visualization modified the seabed ground truth."
    )

print("\nDiscrete UUV motion validated.")

# Combo 5 — Local FOV Geometry and Observation Extraction

## Objective

This combo introduces the UUV's local Field of View (FOV) and extracts the corresponding local observation from the global seabed image.

This is the first point in the notebook where the simulator explicitly separates:

```text
GLOBAL WORLD
```

from:

```text
AGENT OBSERVATION
```

The UUV still has:

```text
NO resource knowledge
NO memory
NO autonomous planner
NO communication
```

but it no longer needs access to the complete seabed image in order to perceive its surroundings.

---

## Observation Model

For UUV position:

$$
p(t)=
\begin{bmatrix}
x(t)\\
y(t)
\end{bmatrix}
$$

and FOV dimensions:

$$
w_F \times h_F
$$

the local observation is:

$$
o(t)
=
\operatorname{Crop}
\left(
I,p(t),w_F,h_F
\right)
$$

where:

- $I$ is the global seabed image;
- $o(t)$ is the local observation;
- $w_F=128$ pixels;
- $h_F=128$ pixels.

The expected observation shape is therefore:

$$
o(t)
\in
\mathbb{R}^{128\times128\times3}
$$

for the current configuration.

---

## FOV Geometry

For a UUV centered at:

$$
(x_c,y_c)
$$

define half-width and half-height as:

$$
h_x = \left\lfloor \frac{w_F}{2} \right\rfloor
$$

$$
h_y = \left\lfloor \frac{h_F}{2} \right\rfloor
$$

For the current FOV:

$$
h_x=h_y=64
$$

The nominal FOV bounds are:

$$
x_{\min}=x_c-h_x
$$

$$
x_{\max}=x_c-h_x+w_F
$$

$$
y_{\min}=y_c-h_y
$$

$$
y_{\max}=y_c-h_y+h_F
$$

The upper bounds are treated as exclusive array indices.

Therefore the nominal crop is:

```text
image[y_min:y_max, x_min:x_max]
```

---

## Why Padding Is Required

If the UUV is near a map boundary, part of the nominal FOV may fall outside the valid image domain.

For example, near the upper-left corner:

$$
x_c < \frac{w_F}{2}
$$

or:

$$
y_c < \frac{h_F}{2}
$$

would create negative crop indices.

Instead of shrinking the observation, this prototype uses a **fixed-size padded FOV**.

The valid intersection with the map is copied into a blank observation array:

$$
o(t)
\in
\mathbb{R}^{h_F\times w_F\times3}
$$

for every valid UUV position.

Thus:

$$
shape(o(t))=(128,128,3)
$$

regardless of whether the UUV is:

- near the center;
- near an edge;
- or directly in a corner.

---

## Padding Convention

Regions outside the map are initially represented using zero-valued pixels:

$$
I_{\text{pad}}=(0,0,0)
$$

This corresponds to black padding.

This is an abstraction rather than a physical sensor model.

The important property is fixed observation geometry.

---

## Information Architecture

The complete seabed image remains available only to the simulator.

The autonomy layer should eventually receive only:

$$
o_i(t)
$$

rather than:

$$
I
$$

Thus:

$$
o_i(t)
\subset I
$$

for positions where the complete FOV lies inside the environment.

Near a boundary, the observation becomes:

$$
o_i(t)
=
I_{\text{valid}}
\cup
I_{\text{padding}}
$$

The observation function must not add:

- UUV markers;
- trajectory overlays;
- resource markers;
- communication links;
- HUD information.

It extracts only the underlying seabed pixels plus boundary padding.

Therefore:

$$
\boxed{
\text{Raw Observation}
\neq
\text{Debug Visualization}
}
$$

---

## Validation Cases

This combo will test two different positions.

### Case 1 — Center Position

For:

$$
p=(724,543)
$$

the complete $128\times128$ FOV lies inside the environment.

Therefore:

$$
N_{\text{padding}}=0
$$

and the observation should contain only real seabed pixels.

---

### Case 2 — Upper-Left Corner

For:

$$
p=(0,0)
$$

half of the nominal FOV lies outside the map in both x and y.

The observation must still satisfy:

$$
shape(o)=(128,128,3)
$$

but part of it should contain zero padding.

For an even $128\times128$ FOV centered at $(0,0)$, approximately one quarter of the observation contains valid seabed data while the remaining region lies outside the map.

---

## Debug FOV Bounds

The FOV geometry may also be drawn on the human-facing global visualization.

This is allowed because the rectangle is only a debug overlay.

The global image used by the simulator remains unchanged.

---

## Expected Output

The code cell should:

1. define a function that computes FOV bounds;
2. define a fixed-size padded observation extractor;
3. extract the local FOV at the current center-position UUV;
4. verify that its shape is exactly:

$$
(128,128,3)
$$

5. create a test UUV at $(0,0)$;
6. extract its boundary observation;
7. verify that this observation also has shape:

$$
(128,128,3)
$$

8. report the amount of padding required for each case;
9. visualize:
   - the global seabed with the current UUV FOV rectangle;
   - the center FOV observation;
   - the corner FOV observation;
10. verify that the original seabed image remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Global World}
\rightarrow
\text{Local Fixed-Size Observation}
}
$$

without leaking debug annotations into agent perception.

---

In [ ]:
# ============================================================
# Combo 5 — Local FOV Geometry and Observation Extraction
# ============================================================


# ------------------------------------------------------------
# FOV geometry
# ------------------------------------------------------------

def get_fov_bounds(
    state: UUVState,
    fov_width: int = FOV_WIDTH,
    fov_height: int = FOV_HEIGHT,
):
    half_w = fov_width // 2
    half_h = fov_height // 2

    x_min = state.x - half_w
    y_min = state.y - half_h

    x_max = x_min + fov_width
    y_max = y_min + fov_height

    return x_min, y_min, x_max, y_max


# ------------------------------------------------------------
# Fixed-size padded observation extraction
# ------------------------------------------------------------

def get_observation(
    state: UUVState,
    image: np.ndarray,
    fov_width: int = FOV_WIDTH,
    fov_height: int = FOV_HEIGHT,
):
    x_min, y_min, x_max, y_max = get_fov_bounds(
        state,
        fov_width=fov_width,
        fov_height=fov_height,
    )

    # Fixed-size output
    observation = np.zeros(
        (fov_height, fov_width, image.shape[2]),
        dtype=image.dtype,
    )

    # Valid source-image bounds
    src_x0 = max(0, x_min)
    src_y0 = max(0, y_min)

    src_x1 = min(WIDTH, x_max)
    src_y1 = min(HEIGHT, y_max)

    # Matching destination bounds
    dst_x0 = src_x0 - x_min
    dst_y0 = src_y0 - y_min

    dst_x1 = dst_x0 + (src_x1 - src_x0)
    dst_y1 = dst_y0 + (src_y1 - src_y0)

    # Copy only the valid image region
    observation[
        dst_y0:dst_y1,
        dst_x0:dst_x1,
    ] = image[
        src_y0:src_y1,
        src_x0:src_x1,
    ]

    valid_width = max(0, src_x1 - src_x0)
    valid_height = max(0, src_y1 - src_y0)

    valid_pixels = valid_width * valid_height
    total_pixels = fov_width * fov_height
    padded_pixels = total_pixels - valid_pixels

    info = {
        "nominal_bounds": (
            x_min,
            y_min,
            x_max,
            y_max,
        ),
        "source_bounds": (
            src_x0,
            src_y0,
            src_x1,
            src_y1,
        ),
        "valid_pixels": valid_pixels,
        "padded_pixels": padded_pixels,
        "padding_fraction": padded_pixels / total_pixels,
    }

    return observation, info


# ------------------------------------------------------------
# Preserve ground-truth image for integrity checking
# ------------------------------------------------------------

seabed_bgr_before_fov = seabed_bgr.copy()


# ------------------------------------------------------------
# Test 1 — Current center UUV
# ------------------------------------------------------------

center_observation_bgr, center_info = get_observation(
    uuv0,
    seabed_bgr,
)

center_observation_rgb = cv2.cvtColor(
    center_observation_bgr,
    cv2.COLOR_BGR2RGB,
)


# ------------------------------------------------------------
# Test 2 — Upper-left map corner
# ------------------------------------------------------------

corner_uuv = UUVState(
    uuv_id=999,
    x=0,
    y=0,
)

corner_observation_bgr, corner_info = get_observation(
    corner_uuv,
    seabed_bgr,
)

corner_observation_rgb = cv2.cvtColor(
    corner_observation_bgr,
    cv2.COLOR_BGR2RGB,
)


# ------------------------------------------------------------
# Validate output shapes
# ------------------------------------------------------------

expected_shape = (
    FOV_HEIGHT,
    FOV_WIDTH,
    3,
)

if center_observation_bgr.shape != expected_shape:
    raise RuntimeError(
        f"Center observation shape mismatch: "
        f"{center_observation_bgr.shape}"
    )

if corner_observation_bgr.shape != expected_shape:
    raise RuntimeError(
        f"Corner observation shape mismatch: "
        f"{corner_observation_bgr.shape}"
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 60)
print("MultiUUV-Link — Local FOV Observation")
print("=" * 60)

print("\nCenter test:")
print(
    f"  UUV position      : "
    f"({uuv0.x}, {uuv0.y})"
)
print(
    f"  Observation shape : "
    f"{center_observation_bgr.shape}"
)
print(
    f"  Nominal bounds    : "
    f"{center_info['nominal_bounds']}"
)
print(
    f"  Source bounds     : "
    f"{center_info['source_bounds']}"
)
print(
    f"  Valid pixels      : "
    f"{center_info['valid_pixels']}"
)
print(
    f"  Padded pixels     : "
    f"{center_info['padded_pixels']}"
)
print(
    f"  Padding fraction  : "
    f"{center_info['padding_fraction']:.4f}"
)

print("\nCorner test:")
print(
    f"  UUV position      : "
    f"({corner_uuv.x}, {corner_uuv.y})"
)
print(
    f"  Observation shape : "
    f"{corner_observation_bgr.shape}"
)
print(
    f"  Nominal bounds    : "
    f"{corner_info['nominal_bounds']}"
)
print(
    f"  Source bounds     : "
    f"{corner_info['source_bounds']}"
)
print(
    f"  Valid pixels      : "
    f"{corner_info['valid_pixels']}"
)
print(
    f"  Padded pixels     : "
    f"{corner_info['padded_pixels']}"
)
print(
    f"  Padding fraction  : "
    f"{corner_info['padding_fraction']:.4f}"
)


# ------------------------------------------------------------
# Additional correctness checks
# ------------------------------------------------------------

if center_info["padded_pixels"] != 0:
    raise RuntimeError(
        "Center FOV unexpectedly contains padding."
    )

if corner_info["padded_pixels"] <= 0:
    raise RuntimeError(
        "Corner FOV should contain padding, but none was detected."
    )


# ------------------------------------------------------------
# Debug visualization
# ------------------------------------------------------------

x_min, y_min, x_max, y_max = get_fov_bounds(uuv0)

plt.figure(figsize=(12, 7))

plt.imshow(seabed_rgb)

plt.scatter(
    uuv0.x,
    uuv0.y,
    s=90,
    marker="o",
)

fov_rectangle = plt.Rectangle(
    (x_min, y_min),
    FOV_WIDTH,
    FOV_HEIGHT,
    fill=False,
    linewidth=2,
)

plt.gca().add_patch(fov_rectangle)

plt.title("MultiUUV-Link — Global World with UUV FOV")
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 6))

plt.imshow(center_observation_rgb)

plt.title(
    f"Center Local Observation "
    f"({FOV_WIDTH} × {FOV_HEIGHT})"
)

plt.xlabel("local x [pixels]")
plt.ylabel("local y [pixels]")
plt.tight_layout()
plt.show()


plt.figure(figsize=(6, 6))

plt.imshow(corner_observation_rgb)

plt.title(
    f"Corner Local Observation "
    f"({FOV_WIDTH} × {FOV_HEIGHT})"
)

plt.xlabel("local x [pixels]")
plt.ylabel("local y [pixels]")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Ground-truth integrity
# ------------------------------------------------------------

ground_truth_unchanged = np.array_equal(
    seabed_bgr_before_fov,
    seabed_bgr,
)

print(
    f"\nGround truth unchanged : "
    f"{ground_truth_unchanged}"
)

if not ground_truth_unchanged:
    raise RuntimeError(
        "FOV processing modified the global seabed image."
    )

print("\nLocal FOV extraction validated.")

# Combo 6 — Hidden Resource Ground Truth

## Objective

This combo introduces hidden resources into the simulator ground truth.

Resources will exist at fixed positions in the global environment, but they will **not** yet be visible to the UUV and will **not** yet appear inside the raw local observation.

This distinction is essential.

The simulator may know:

$$
R=\{R_1,R_2,\ldots,R_{N_R}\}
$$

while the UUV initially knows:

$$
K_{\text{resource},i}(0)=\varnothing
$$

Therefore:

$$
\boxed{
\text{Resource Exists}
\not\Rightarrow
\text{Resource Is Known}
}
$$

At this stage we implement only:

```text
resource representation
+
random resource placement
+
resource classes
+
ground-truth debug visualization
```

There is still:

```text
NO discovery logic
NO persistent resource memory
NO resource information inside raw FOV observations
NO task generation
NO communication
```

---

## Resource State

Each resource $j$ is represented as:

$$
R_j
=
(id_j,x_j,y_j,c_j)
$$

where:

- $id_j$ is a unique resource identifier;
- $(x_j,y_j)$ is its true global position;
- $c_j$ is its generic resource class.

The class space is:

$$
c_j\in\{A,B,C\}
$$

These classes remain intentionally generic.

They do not represent specific minerals or geological formations.

---

## Number of Resources

The initial resource count is:

$$
N_R=20
$$

as defined in the global notebook configuration.

The complete simulator resource set is therefore:

$$
R=
\{R_1,R_2,\ldots,R_{20}\}
$$

---

## Spatial Sampling Model

Resource positions are sampled inside the valid image domain:

$$
x_j\sim U_{\mathbb{Z}}(0,W-1)
$$

$$
y_j\sim U_{\mathbb{Z}}(0,H-1)
$$

where:

$$
U_{\mathbb{Z}}
$$

denotes a discrete uniform distribution over integer pixel coordinates.

Each valid pixel coordinate is therefore initially treated as equally likely.

This is an algorithmic abstraction.

No geological clustering model is imposed.

---

## Resource-Class Sampling

Each resource class is sampled from:

$$
\{A,B,C\}
$$

with approximately equal probability:

$$
P(c_j=A)
=
P(c_j=B)
=
P(c_j=C)
\approx
\frac{1}{3}
$$

Because only 20 resources are sampled, the exact class counts do not need to be identical.

---

## Reproducibility

The global random seed was already defined as:

$$
s=42
$$

However, this combo will use a dedicated NumPy random generator:

$$
G_R=\operatorname{RNG}(s)
$$

rather than relying only on global NumPy state.

This makes resource generation easier to reproduce independently later.

Conceptually:

$$
R=f(G_R,W,H,N_R)
$$

---

## Duplicate Positions

For this initial prototype, two resources should not occupy the same pixel coordinate.

Therefore, generated coordinates must satisfy:

$$
(x_i,y_i)\neq(x_j,y_j)
$$

for:

$$
i\neq j
$$

This gives:

$$
|\{(x_j,y_j)\}|=N_R
$$

and avoids ambiguous overlapping ground-truth objects.

---

## Information Boundary

The resources belong to the simulator ground truth.

The raw seabed image remains:

$$
I
$$

and the UUV raw observation remains:

$$
o_i(t)
=
\operatorname{Crop}(I,p_i(t),FOV_i)
$$

not:

$$
\operatorname{Crop}(I+R,p_i(t),FOV_i)
$$

In other words, resource markers are **not painted into the image used for perception**.

Their positions exist separately as structured simulator state.

This is one of the most important implementation choices in the prototype.

If resource markers were painted directly into the seabed image, a future autonomous policy could detect resources before the explicit discovery model had been applied.

That would violate the hidden-information architecture.

Therefore:

$$
\boxed{
\text{Raw Seabed}
+
\text{Structured Hidden Resources}
}
$$

are maintained separately.

---

## Debug Visualization

The human-facing global debug view is allowed to display all resource locations.

This does not mean the UUV knows them.

The debug renderer may therefore visualize:

```text
Type A → circle
Type B → triangle
Type C → square
```

The shape convention mirrors the MicroTrench-2D design.

Color is deliberately not used to encode resource class because color may later be useful for distinguishing UUV roles or agent identities.

The visualization is therefore:

$$
V
=
\operatorname{Render}(I,R)
$$

while the agent still receives:

$$
o_i(t)
=
\operatorname{Crop}(I)
$$

Thus:

$$
V\neq I
$$

and:

$$
V\neq o_i
$$

---

## Validation Conditions

The generated resource set must satisfy:

$$
|R|=20
$$

Every resource must satisfy:

$$
0\le x_j<W
$$

$$
0\le y_j<H
$$

and:

$$
c_j\in\{A,B,C\}
$$

All resource coordinates must be unique.

The raw seabed image must remain unchanged.

The previously validated local observation must also remain resource-free because resource markers are stored separately from image pixels.

---

## Expected Output

The code cell should:

1. define a resource data structure;
2. generate exactly 20 resources;
3. assign each resource:
   - unique ID;
   - x-coordinate;
   - y-coordinate;
   - class A/B/C;
4. verify all coordinates are inside the map;
5. verify all resource coordinates are unique;
6. print the full generated resource table;
7. print class counts;
8. visualize the global seabed with all hidden resources for debugging;
9. use:
   - circle for A;
   - triangle for B;
   - square for C;
10. verify that the underlying seabed image remains unchanged;
11. verify that the previously extracted raw center FOV is unchanged.

A successful result establishes:

$$
\boxed{
\text{World Ground Truth}
=
\text{Seabed}
+
\text{Hidden Structured Resources}
}
$$

while:

$$
\boxed{
\text{UUV Knowledge}
=
\text{No Resource Knowledge Yet}
}
$$

---

In [ ]:
# ============================================================
# Combo 6 — Hidden Resource Ground Truth
# ============================================================

from dataclasses import dataclass


# ------------------------------------------------------------
# Resource representation
# ------------------------------------------------------------

@dataclass(frozen=True)
class Resource:
    resource_id: int
    x: int
    y: int
    resource_class: str


# ------------------------------------------------------------
# Scenario configuration
# ------------------------------------------------------------

RESOURCE_COUNT_MIN = 10
RESOURCE_COUNT_MAX = 25

# Set to None for a new random scenario every run.
# Set to an integer (e.g. 42) if you want reproducible scenarios.
RESOURCE_SCENARIO_SEED = None

resource_rng = np.random.default_rng(RESOURCE_SCENARIO_SEED)


# ------------------------------------------------------------
# Resource generation
# ------------------------------------------------------------

def generate_resources(
    width: int,
    height: int,
    min_resources: int,
    max_resources: int,
    classes=RESOURCE_CLASSES,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    if min_resources <= 0:
        raise ValueError("min_resources must be positive.")

    if max_resources < min_resources:
        raise ValueError(
            "max_resources must be greater than or equal to min_resources."
        )

    max_unique_positions = width * height
    if max_resources > max_unique_positions:
        raise ValueError(
            "Requested more possible resources than available unique pixels."
        )

    # Randomize total resource count INCLUSIVELY between min and max
    num_resources = int(rng.integers(min_resources, max_resources + 1))

    resources = []
    used_positions = set()

    while len(resources) < num_resources:
        x = int(rng.integers(0, width))
        y = int(rng.integers(0, height))
        position = (x, y)

        if position in used_positions:
            continue

        resource_class = str(rng.choice(classes))

        resources.append(
            Resource(
                resource_id=len(resources),
                x=x,
                y=y,
                resource_class=resource_class,
            )
        )

        used_positions.add(position)

    return resources


resources = generate_resources(
    width=WIDTH,
    height=HEIGHT,
    min_resources=RESOURCE_COUNT_MIN,
    max_resources=RESOURCE_COUNT_MAX,
    classes=RESOURCE_CLASSES,
    rng=resource_rng,
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

num_resources_generated = len(resources)

if not (RESOURCE_COUNT_MIN <= num_resources_generated <= RESOURCE_COUNT_MAX):
    raise RuntimeError(
        f"Generated resource count {num_resources_generated} is outside "
        f"[{RESOURCE_COUNT_MIN}, {RESOURCE_COUNT_MAX}]."
    )

positions = [(r.x, r.y) for r in resources]

if len(set(positions)) != len(positions):
    raise RuntimeError("Duplicate resource coordinates detected.")

for resource in resources:
    if not (0 <= resource.x < WIDTH):
        raise RuntimeError(
            f"Resource {resource.resource_id} has invalid x={resource.x}"
        )

    if not (0 <= resource.y < HEIGHT):
        raise RuntimeError(
            f"Resource {resource.resource_id} has invalid y={resource.y}"
        )

    if resource.resource_class not in RESOURCE_CLASSES:
        raise RuntimeError(
            f"Resource {resource.resource_id} has invalid class "
            f"{resource.resource_class}"
        )


# ------------------------------------------------------------
# Resource table
# ------------------------------------------------------------

resource_df = pd.DataFrame(
    [
        {
            "resource_id": r.resource_id,
            "x": r.x,
            "y": r.y,
            "class": r.resource_class,
        }
        for r in resources
    ]
)

class_counts = (
    resource_df["class"]
    .value_counts()
    .reindex(RESOURCE_CLASSES, fill_value=0)
)


# ------------------------------------------------------------
# Preserve integrity references
# ------------------------------------------------------------

seabed_bgr_before_resources = seabed_bgr.copy()
center_observation_before_resources = center_observation_bgr.copy()


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 60)
print("MultiUUV-Link — Hidden Resource Ground Truth")
print("=" * 60)

print(f"\nScenario seed       : {RESOURCE_SCENARIO_SEED}")
print(
    f"Resource count mode : random integer in "
    f"[{RESOURCE_COUNT_MIN}, {RESOURCE_COUNT_MAX}]"
)
print(f"Resource count      : {num_resources_generated}")
print(f"Unique positions    : {len(set(positions))}")
print(f"All positions valid : True")
print(f"Valid classes       : {RESOURCE_CLASSES}")

print("\nClass counts:")
for resource_class, count in class_counts.items():
    print(f"  Type {resource_class}: {count}")

print("\nGenerated resources:")
print(resource_df.to_string(index=False))


# ------------------------------------------------------------
# Debug visualization
# ------------------------------------------------------------

marker_by_class = {
    "A": "o",   # Circle
    "B": "^",   # Triangle
    "C": "s",   # Square
}

plt.figure(figsize=(12, 7))
plt.imshow(seabed_rgb)

for resource_class in RESOURCE_CLASSES:
    subset = resource_df[resource_df["class"] == resource_class]

    plt.scatter(
        subset["x"],
        subset["y"],
        marker=marker_by_class[resource_class],
        s=150,
        facecolors="white",
        edgecolors="black",
        linewidths=1.4,
        alpha=0.95,
        label=f"Type {resource_class}",
        zorder=5,
    )

plt.title(
    "MultiUUV-Link — Hidden Resources "
    "(Simulator Ground Truth Only)"
)
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Integrity validation
# ------------------------------------------------------------

seabed_unchanged = np.array_equal(
    seabed_bgr_before_resources,
    seabed_bgr,
)

center_observation_unchanged = np.array_equal(
    center_observation_before_resources,
    center_observation_bgr,
)

print(
    f"\nSeabed ground truth unchanged : "
    f"{seabed_unchanged}"
)

print(
    f"Raw center FOV unchanged      : "
    f"{center_observation_unchanged}"
)

if not seabed_unchanged:
    raise RuntimeError(
        "Resource generation modified the raw seabed image."
    )

if not center_observation_unchanged:
    raise RuntimeError(
        "Hidden resources leaked into the raw UUV observation."
    )

print("\nHidden resource ground truth validated.")

# Combo 7 — Resource Discovery Logic

## Objective

This combo introduces deterministic resource discovery based on the UUV's current Field of View.

The simulator already knows the complete hidden resource set:

$$
R=
\{R_1,R_2,\ldots,R_{N_R}\}
$$

but the UUV does not.

The UUV should acquire resource knowledge only when a resource lies inside its current FOV.

Therefore:

$$
R_j \in FOV_i(t)
\Rightarrow
R_j \text{ becomes locally discovered by UUV } i
$$

while:

$$
R_j \notin FOV_i(t)
\Rightarrow
R_j \text{ remains unknown}
$$

At this stage there is still:

```text
NO persistent exploration memory
NO communication
NO autonomous planner
NO distributed task allocation
```

This combo only implements the first bridge between:

```text
hidden simulator state
```

and:

```text
legitimate agent knowledge
```

---

## Discovery Geometry

For UUV state:

$$
p_i(t)=
(x_i(t),y_i(t))
$$

the FOV bounds are:

$$
x_{\min}\le x < x_{\max}
$$

$$
y_{\min}\le y < y_{\max}
$$

A resource:

$$
R_j=(x_j,y_j,c_j)
$$

is inside the FOV when:

$$
x_{\min}\le x_j < x_{\max}
$$

and:

$$
y_{\min}\le y_j < y_{\max}
$$

The upper bounds are exclusive because the FOV implementation follows NumPy slice semantics.

---

## Deterministic Detection Model

The initial simulator uses perfect detection inside the FOV:

$$
P(\text{detect}\mid R_j\in FOV_i)=1
$$

and:

$$
P(\text{detect}\mid R_j\notin FOV_i)=0
$$

There is currently no:

- detection probability;
- sensor noise;
- false positive;
- false negative;
- confidence score.

The FOV itself is the complete sensing region.

---

## Local Resource Knowledge

The UUV's locally known resource set is:

$$
D_i(t)
$$

Initially:

$$
D_i(0)=\varnothing
$$

After observation:

$$
D_i(t+1)
=
D_i(t)
\cup
\left\{
R_j
\mid
R_j\in FOV_i(t)
\right\}
$$

For this combo we will validate the instantaneous discovery rule first.

Persistent storage across multiple movements will be introduced immediately afterward in a dedicated memory combo.

---

## Important Information Boundary

Discovery must be computed using resource geometry and current FOV geometry.

The raw image observation remains:

$$
o_i(t)=\operatorname{Crop}(I)
$$

The resource marker does not need to physically appear inside the raw pixel observation.

Thus:

$$
\text{Visual Texture}
\neq
\text{Structured Resource State}
$$

The discovery mechanism is currently an abstract perfect detector operating over the same FOV region.

This is consistent with the lightweight simulator philosophy.

---

## Validation Strategy

Because randomly generated resources may or may not happen to lie inside the current center FOV, this combo will test two cases.

### Case 1 — Current UUV Position

For the current UUV:

$$
p=(724,543)
$$

we will compute which of the randomly generated resources happen to lie inside its current FOV.

The number may be:

$$
0,1,2,\ldots
$$

depending on the current random scenario.

A result of zero is valid.

---

### Case 2 — Guaranteed Discovery Test

To verify the logic independently of random chance, we will select one existing hidden resource:

$$
R_k
$$

and construct a temporary test UUV directly at:

$$
p_{\text{test}}=
(x_k,y_k)
$$

Since the resource lies at the center of the test UUV's FOV:

$$
R_k\in FOV_{\text{test}}
$$

must be true.

Therefore the discovery function must return at least that resource.

This gives us a deterministic correctness test even though the environment itself is randomized.

---

## Debug Visualization

The global debug plot may display:

- all hidden resources in faint form;
- the UUV and its FOV;
- currently discovered resources emphasized.

This visualization remains simulator/debug information only.

The UUV's legitimate structured knowledge is represented separately by the discovery result.

---

## Expected Output

The code cell should:

1. define a function that checks whether a resource lies inside a UUV FOV;
2. define a function that returns all resources currently visible to a UUV;
3. evaluate discovery at the current UUV position;
4. print all currently visible resource IDs and classes;
5. select one existing resource for a guaranteed test;
6. place a temporary test UUV at that resource coordinate;
7. verify that the chosen resource is discovered;
8. visualize:
   - current UUV FOV;
   - hidden resources;
   - currently visible/discovered resources;
9. verify that the raw seabed image remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Hidden Resource}
\rightarrow
\text{FOV Intersection}
\rightarrow
\text{Legitimate Local Discovery}
}
$$

---

In [ ]:
# ============================================================
# Combo 7 — Resource Discovery Logic
# ============================================================


# ------------------------------------------------------------
# Resource-in-FOV test
# ------------------------------------------------------------

def resource_in_fov(
    resource: Resource,
    state: UUVState,
    fov_width: int = FOV_WIDTH,
    fov_height: int = FOV_HEIGHT,
) -> bool:
    x_min, y_min, x_max, y_max = get_fov_bounds(
        state,
        fov_width=fov_width,
        fov_height=fov_height,
    )

    return (
        x_min <= resource.x < x_max
        and
        y_min <= resource.y < y_max
    )


# ------------------------------------------------------------
# Discover all resources currently visible to one UUV
# ------------------------------------------------------------

def discover_visible_resources(
    state: UUVState,
    resources,
    fov_width: int = FOV_WIDTH,
    fov_height: int = FOV_HEIGHT,
):
    visible = [
        resource
        for resource in resources
        if resource_in_fov(
            resource,
            state,
            fov_width=fov_width,
            fov_height=fov_height,
        )
    ]

    return visible


# ------------------------------------------------------------
# Preserve seabed integrity
# ------------------------------------------------------------

seabed_bgr_before_discovery = seabed_bgr.copy()


# ------------------------------------------------------------
# Test 1 — Current UUV position
# ------------------------------------------------------------

current_visible_resources = discover_visible_resources(
    uuv0,
    resources,
)

print("=" * 60)
print("MultiUUV-Link — Resource Discovery")
print("=" * 60)

print("\nCurrent UUV test:")
print(f"  UUV position        : ({uuv0.x}, {uuv0.y})")
print(f"  Visible resources   : {len(current_visible_resources)}")

if current_visible_resources:
    for resource in current_visible_resources:
        print(
            f"    ID={resource.resource_id:2d} | "
            f"Type {resource.resource_class} | "
            f"Position=({resource.x}, {resource.y})"
        )
else:
    print("    None")


# ------------------------------------------------------------
# Test 2 — Guaranteed discovery
# ------------------------------------------------------------

guaranteed_resource = resources[0]

guaranteed_test_uuv = UUVState(
    uuv_id=1000,
    x=guaranteed_resource.x,
    y=guaranteed_resource.y,
)

guaranteed_visible_resources = discover_visible_resources(
    guaranteed_test_uuv,
    resources,
)

guaranteed_visible_ids = {
    resource.resource_id
    for resource in guaranteed_visible_resources
}

guaranteed_discovery_passed = (
    guaranteed_resource.resource_id
    in
    guaranteed_visible_ids
)


print("\nGuaranteed discovery test:")
print(
    f"  Target resource ID  : "
    f"{guaranteed_resource.resource_id}"
)
print(
    f"  Target class        : "
    f"{guaranteed_resource.resource_class}"
)
print(
    f"  Target position     : "
    f"({guaranteed_resource.x}, {guaranteed_resource.y})"
)
print(
    f"  Test UUV position   : "
    f"({guaranteed_test_uuv.x}, {guaranteed_test_uuv.y})"
)
print(
    f"  Resources detected  : "
    f"{len(guaranteed_visible_resources)}"
)
print(
    f"  Target discovered   : "
    f"{guaranteed_discovery_passed}"
)

if not guaranteed_discovery_passed:
    raise RuntimeError(
        "Guaranteed resource-discovery test failed."
    )


# ------------------------------------------------------------
# Debug visualization
# ------------------------------------------------------------

x_min, y_min, x_max, y_max = get_fov_bounds(uuv0)

plt.figure(figsize=(12, 7))
plt.imshow(seabed_rgb)

# All resources: simulator ground truth
for resource_class in RESOURCE_CLASSES:
    subset = resource_df[
        resource_df["class"] == resource_class
    ]

    plt.scatter(
        subset["x"],
        subset["y"],
        marker=marker_by_class[resource_class],
        s=100,
        facecolors="none",
        edgecolors="white",
        linewidths=1.2,
        alpha=0.35,
        zorder=3,
    )

# Currently visible resources: emphasize
for resource in current_visible_resources:
    plt.scatter(
        resource.x,
        resource.y,
        marker=marker_by_class[resource.resource_class],
        s=180,
        facecolors="white",
        edgecolors="black",
        linewidths=1.8,
        zorder=6,
    )

# Current UUV
plt.scatter(
    uuv0.x,
    uuv0.y,
    s=110,
    marker="o",
    zorder=7,
    label="UUV-0",
)

# Current FOV rectangle
fov_rectangle = plt.Rectangle(
    (x_min, y_min),
    FOV_WIDTH,
    FOV_HEIGHT,
    fill=False,
    linewidth=2,
    zorder=5,
)

plt.gca().add_patch(fov_rectangle)

plt.title(
    "MultiUUV-Link — Resource Discovery "
    "(Current UUV Observation)"
)
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Seabed integrity
# ------------------------------------------------------------

seabed_unchanged = np.array_equal(
    seabed_bgr_before_discovery,
    seabed_bgr,
)

print(
    f"\nSeabed ground truth unchanged : "
    f"{seabed_unchanged}"
)

if not seabed_unchanged:
    raise RuntimeError(
        "Discovery processing modified the raw seabed image."
    )

print("\nResource discovery logic validated.")

# Combo 8 — Persistent Resource Discovery Memory

## Objective

This combo introduces persistent per-UUV resource memory.

In Combo 7, resource discovery was instantaneous:

$$
R_j \in FOV_i(t)
\Rightarrow
R_j \text{ is visible at time } t
$$

but once the UUV moved away, that instantaneous result alone did not preserve the discovery.

MultiUUV-Link requires each UUV to remember what it has legitimately learned.

Therefore UUV $i$ maintains a private discovered-resource memory:

$$
D_i(t)
$$

with the update rule:

$$
D_i(t+1)
=
D_i(t)
\cup
V_i(t)
$$

where:

- $D_i(t)$ is the set of resources already known by UUV $i$;
- $V_i(t)$ is the set of resources visible in its current FOV.

The union operation makes discovery persistent.

---

## Knowledge State

Initially:

$$
D_i(0)=\varnothing
$$

If resource $R_k$ becomes visible:

$$
R_k\in V_i(t)
$$

then:

$$
R_k\in D_i(t+1)
$$

Once added, it remains known:

$$
R_k\in D_i(t+\tau)
$$

for all:

$$
\tau>0
$$

within the same episode.

This gives the monotonic property:

$$
D_i(t)
\subseteq
D_i(t+1)
$$

Resource knowledge may increase, but it does not spontaneously disappear.

---

## Structured Memory

The memory should store structured information rather than merely resource IDs.

For each known resource, the UUV should retain:

$$
R_j=
(id_j,x_j,y_j,c_j)
$$

Thus the UUV remembers:

- resource ID;
- position;
- class.

A dictionary keyed by `resource_id` provides a convenient prototype representation:

```text
resource_id
    ↓
Resource(...)
```

This automatically prevents the same resource from being duplicated when observed repeatedly.

---

## Private Knowledge Principle

The simulator knows:

$$
R_{\text{world}}
$$

while UUV-0 knows only:

$$
D_0(t)
$$

In general:

$$
D_0(t)
\subseteq
R_{\text{world}}
$$

The UUV memory must never be initialized from the complete resource table.

Resources enter memory only through the already validated discovery function.

Therefore:

$$
\boxed{
\text{Ground Truth}
\rightarrow
\text{FOV Detection}
\rightarrow
\text{Private Memory}
}
$$

and never:

$$
\text{Ground Truth}
\rightarrow
\text{Private Memory}
$$

directly.

---

## Validation Strategy

A random environment makes passive discovery unreliable for testing.

Therefore we will construct a deterministic validation sequence using one existing hidden resource.

Let the selected resource be:

$$
R_k
$$

The test will proceed in three stages.

### Stage 1 — Before Observation

Create an empty UUV memory:

$$
D_0=\varnothing
$$

Therefore:

$$
|D_0|=0
$$

---

### Stage 2 — Observe the Resource

Temporarily place UUV-0 directly at:

$$
p=(x_k,y_k)
$$

The selected resource must lie inside the FOV.

The discovery-memory update should therefore produce:

$$
R_k\in D_0
$$

---

### Stage 3 — Move Away

Move the test UUV far enough away that:

$$
R_k\notin FOV_0
$$

Then the currently visible resource set should no longer contain $R_k$.

However, persistent memory must still satisfy:

$$
R_k\in D_0
$$

Thus:

$$
\boxed{
\text{Visible}=False
\quad\land\quad
\text{Remembered}=True
}
$$

This is the defining validation condition for this combo.

---

## Repeated Observation

If a previously known resource is seen again, memory should not create another copy.

If:

$$
R_k\in D_i(t)
$$

and:

$$
R_k\in V_i(t+1)
$$

then:

$$
|D_i|
$$

should remain unchanged unless another new resource was also discovered.

Using resource IDs as dictionary keys ensures this behavior naturally.

---

## Episode Semantics

Persistent memory applies only within the current mission episode.

A future environment reset should create fresh private memories:

$$
D_i(0)=\varnothing
$$

for every new episode.

Reset behavior will be integrated later when the complete mission/environment lifecycle is assembled.

---

## Expected Output

The code cell should:

1. define a private resource-memory structure;
2. initialize an empty memory for UUV-0;
3. select one existing hidden resource;
4. place a temporary UUV at that resource;
5. update memory from the current FOV;
6. verify that the target enters memory;
7. move the temporary UUV far enough away that the target is no longer visible;
8. verify that:
   - the target is no longer currently visible;
   - the target remains stored in memory;
9. observe the target again and verify that no duplicate memory entry is created;
10. print the full sequence and resulting private knowledge.

A successful result establishes:

$$
\boxed{
\text{Observation}
\rightarrow
\text{Persistent Private Knowledge}
}
$$

---

In [ ]:
# ============================================================
# Combo 8 — Persistent Resource Discovery Memory
# ============================================================


# ------------------------------------------------------------
# Private per-UUV resource memory
# ------------------------------------------------------------

class ResourceMemory:
    def __init__(self, uuv_id: int):
        self.uuv_id = uuv_id
        self._known_resources = {}

    def update(self, visible_resources):
        """
        Add currently visible resources to persistent memory.

        Returns the list of resources that were newly learned
        during this update.
        """
        newly_discovered = []

        for resource in visible_resources:
            if resource.resource_id not in self._known_resources:
                self._known_resources[resource.resource_id] = resource
                newly_discovered.append(resource)

        return newly_discovered

    def knows(self, resource_id: int) -> bool:
        return resource_id in self._known_resources

    def get(self, resource_id: int):
        return self._known_resources.get(resource_id)

    def all_resources(self):
        return list(self._known_resources.values())

    def __len__(self):
        return len(self._known_resources)


# ------------------------------------------------------------
# Initialize empty private memory for UUV-0
# ------------------------------------------------------------

uuv0_resource_memory = ResourceMemory(
    uuv_id=uuv0.uuv_id
)


# ------------------------------------------------------------
# Choose a deterministic target from current world ground truth
# ------------------------------------------------------------

target_resource = resources[0]


# ------------------------------------------------------------
# Stage 1 — Before observation
# ------------------------------------------------------------

initial_memory_size = len(uuv0_resource_memory)

print("=" * 60)
print("MultiUUV-Link — Persistent Resource Memory")
print("=" * 60)

print("\nStage 1 — Before observation")
print(f"  Memory size        : {initial_memory_size}")
print(
    f"  Knows target       : "
    f"{uuv0_resource_memory.knows(target_resource.resource_id)}"
)

if initial_memory_size != 0:
    raise RuntimeError(
        "Resource memory should begin empty."
    )


# ------------------------------------------------------------
# Stage 2 — Place test UUV directly on target resource
# ------------------------------------------------------------

observe_state = UUVState(
    uuv_id=uuv0.uuv_id,
    x=target_resource.x,
    y=target_resource.y,
)

visible_at_target = discover_visible_resources(
    observe_state,
    resources,
)

newly_discovered = uuv0_resource_memory.update(
    visible_at_target
)

target_discovered = uuv0_resource_memory.knows(
    target_resource.resource_id
)

print("\nStage 2 — Observe target resource")
print(
    f"  Target resource    : "
    f"ID={target_resource.resource_id}, "
    f"Type {target_resource.resource_class}, "
    f"Position=({target_resource.x}, {target_resource.y})"
)
print(
    f"  Test UUV position  : "
    f"({observe_state.x}, {observe_state.y})"
)
print(
    f"  Currently visible  : "
    f"{len(visible_at_target)}"
)
print(
    f"  Newly discovered   : "
    f"{len(newly_discovered)}"
)
print(f"  Memory size        : {len(uuv0_resource_memory)}")
print(f"  Knows target       : {target_discovered}")

if not target_discovered:
    raise RuntimeError(
        "Target resource was visible but was not stored in memory."
    )


# ------------------------------------------------------------
# Stage 3 — Move far enough away from target
# ------------------------------------------------------------

# Choose the map corner farthest from the target resource.
candidate_corners = [
    (0, 0),
    (WIDTH - 1, 0),
    (0, HEIGHT - 1),
    (WIDTH - 1, HEIGHT - 1),
]

far_x, far_y = max(
    candidate_corners,
    key=lambda p: (
        (p[0] - target_resource.x) ** 2
        +
        (p[1] - target_resource.y) ** 2
    ),
)

far_state = UUVState(
    uuv_id=uuv0.uuv_id,
    x=far_x,
    y=far_y,
)

visible_far_away = discover_visible_resources(
    far_state,
    resources,
)

visible_far_ids = {
    r.resource_id
    for r in visible_far_away
}

target_still_visible = (
    target_resource.resource_id
    in visible_far_ids
)

target_still_remembered = (
    uuv0_resource_memory.knows(
        target_resource.resource_id
    )
)

print("\nStage 3 — Move away")
print(
    f"  Far UUV position   : "
    f"({far_state.x}, {far_state.y})"
)
print(
    f"  Currently visible  : "
    f"{len(visible_far_away)}"
)
print(
    f"  Target visible     : "
    f"{target_still_visible}"
)
print(
    f"  Target remembered  : "
    f"{target_still_remembered}"
)
print(
    f"  Memory size        : "
    f"{len(uuv0_resource_memory)}"
)

if target_still_visible:
    raise RuntimeError(
        "Far-away validation state still sees the target resource."
    )

if not target_still_remembered:
    raise RuntimeError(
        "Target resource disappeared from persistent memory."
    )


# ------------------------------------------------------------
# Stage 4 — Re-observe target and check duplicate prevention
# ------------------------------------------------------------

memory_size_before_revisit = len(
    uuv0_resource_memory
)

visible_on_revisit = discover_visible_resources(
    observe_state,
    resources,
)

new_on_revisit = uuv0_resource_memory.update(
    visible_on_revisit
)

memory_size_after_revisit = len(
    uuv0_resource_memory
)

print("\nStage 4 — Re-observe target")
print(
    f"  Memory before      : "
    f"{memory_size_before_revisit}"
)
print(
    f"  Newly discovered   : "
    f"{len(new_on_revisit)}"
)
print(
    f"  Memory after       : "
    f"{memory_size_after_revisit}"
)

if memory_size_after_revisit != memory_size_before_revisit:
    raise RuntimeError(
        "Repeated observation created duplicate resource memory."
    )


# ------------------------------------------------------------
# Final private-memory report
# ------------------------------------------------------------

known_resource_df = pd.DataFrame(
    [
        {
            "resource_id": r.resource_id,
            "x": r.x,
            "y": r.y,
            "class": r.resource_class,
        }
        for r in uuv0_resource_memory.all_resources()
    ]
)

print("\nUUV-0 private resource knowledge:")

if known_resource_df.empty:
    print("  None")
else:
    print(
        known_resource_df
        .sort_values("resource_id")
        .to_string(index=False)
    )


# ------------------------------------------------------------
# Final invariants
# ------------------------------------------------------------

print("\nPersistent-memory invariants:")
print(
    f"  Target no longer visible : "
    f"{not target_still_visible}"
)
print(
    f"  Target still remembered  : "
    f"{target_still_remembered}"
)
print(
    f"  No duplicate on revisit  : "
    f"{memory_size_after_revisit == memory_size_before_revisit}"
)

print("\nPersistent resource memory validated.")

# Combo 9 — Persistent Exploration Memory Map

## Objective

This combo introduces a persistent per-UUV exploration-memory map.

The UUV already has:

- a current position;
- a local FOV;
- a resource-discovery mechanism;
- persistent resource memory.

However, it does not yet remember which parts of the seabed it has previously observed.

For autonomous exploration, this is essential.

UUV $i$ therefore maintains a private exploration memory:

$$
M_i(t)
$$

where each map cell records whether that location has ever been inside the UUV's legitimate observation history.

The update rule is:

$$
M_i(t+1)
=
M_i(t)
\cup
FOV_i(t)
$$

This means previously observed regions remain known even after the UUV moves away.

---

## Memory Representation

The global seabed has dimensions:

$$
H\times W
$$

The private exploration memory will therefore use a two-dimensional boolean mask:

$$
M_i
\in
\{0,1\}^{H\times W}
$$

with:

$$
M_i(y,x)=0
$$

meaning:

```text
UNKNOWN
```

and:

$$
M_i(y,x)=1
$$

meaning:

```text
OBSERVED
```

The memory uses the same image-indexing convention as the seabed:

```text
memory[y, x]
```

while UUV geometry remains:

```text
(x, y)
```

---

## Initial Condition

At the beginning of an episode:

$$
M_i(0)=0
$$

for every cell.

Therefore:

$$
\sum_{x,y} M_i(0)=0
$$

and initial coverage is:

$$
Coverage_i(0)=0
$$

---

## FOV-to-Memory Update

Suppose the UUV has nominal FOV bounds:

$$
(x_{\min},y_{\min},x_{\max},y_{\max})
$$

Only the portion that lies inside the valid map should be written into memory.

Define the valid intersection:

$$
x_0=\max(0,x_{\min})
$$

$$
x_1=\min(W,x_{\max})
$$

$$
y_0=\max(0,y_{\min})
$$

$$
y_1=\min(H,y_{\max})
$$

Then the memory update is:

$$
M_i[y_0:y_1,\;x_0:x_1]=1
$$

Padding outside the map is **not** part of the environment and therefore must not count as explored area.

---

## Coverage Metric

For one UUV, private coverage is:

$$
C_i(t)
=
\frac{
\sum M_i(t)
}{
HW
}
$$

where:

$$
\sum M_i(t)
$$

is the number of legitimately observed map cells.

Thus:

$$
0\le C_i(t)\le1
$$

or equivalently:

$$
0\%\le Coverage_i(t)\le100\%
$$

This is a true spatial exploration metric inside the abstract simulator.

---

## Persistent Memory Property

Exploration memory must be monotonic.

If:

$$
M_i(y,x,t)=1
$$

then:

$$
M_i(y,x,t+\tau)=1
$$

for all:

$$
\tau>0
$$

within the same episode.

Therefore:

$$
M_i(t)
\subseteq
M_i(t+1)
$$

and:

$$
Coverage_i(t+1)
\ge
Coverage_i(t)
$$

The UUV may gain observed cells, but it must not forget them.

---

## Validation Strategy

This combo will validate memory using three observation states.

### Stage 1 — Empty Memory

Initialize:

$$
M_0(0)=0
$$

Expected observed cells:

$$
0
$$

---

### Stage 2 — Observe at the Current Center Position

For UUV-0 at:

$$
(724,543)
$$

the full $128\times128$ FOV lies inside the map.

Therefore the expected number of observed cells after one update is:

$$
128\times128
=
16384
$$

Thus:

$$
|M_0|=16384
$$

after the first observation.

---

### Stage 3 — Move Right by One Motion Step

Move the UUV by:

$$
\Delta=8
$$

pixels to the right.

The new FOV overlaps heavily with the old FOV.

The first FOV covers width:

$$
128
$$

and the shifted FOV contributes only an additional vertical strip of width:

$$
8
$$

and height:

$$
128
$$

Therefore the newly explored area should be:

$$
8\times128
=
1024
$$

cells.

Expected total observed cells:

$$
16384+1024
=
17408
$$

This is an excellent validation because it checks that overlapping FOV regions are not double-counted.

---

## Corner Validation

The memory update should also work near map boundaries.

At:

$$
p=(0,0)
$$

the nominal $128\times128$ observation contains only:

$$
64\times64
=
4096
$$

real map cells.

The padded 75% of the observation must **not** be marked as explored because those pixels do not belong to the world.

---

## Private Knowledge Principle

The exploration map belongs to the UUV.

The simulator may know the complete world:

$$
\Omega
$$

but the agent knows only:

$$
M_i(t)
$$

The agent must not initialize its memory from the complete environment.

Instead:

$$
\boxed{
\text{Local FOV History}
\rightarrow
\text{Private Exploration Memory}
}
$$

This map will later support frontier extraction, redundant-exploration analysis, communication-based map sharing, and coverage metrics.

---

## Visualization

The human-facing validation plot will display the private memory mask.

A simple representation is:

```text
black  → UNKNOWN
white  → OBSERVED
```

This is not the seabed image.

It is a visualization of the UUV's knowledge state.

---

## Expected Output

The code cell should:

1. define an exploration-memory structure;
2. initialize a full-map boolean memory mask;
3. confirm initial observed cells are zero;
4. update memory at the center UUV position;
5. verify that exactly 16384 cells become observed;
6. move the UUV one step right;
7. update memory again;
8. verify:
   - newly observed cells = 1024;
   - total observed cells = 17408;
9. test a fresh memory at the upper-left corner;
10. verify that exactly 4096 real map cells are marked observed;
11. compute coverage percentages;
12. visualize the center-plus-right exploration memory.

A successful result establishes:

$$
\boxed{
\text{Local Observation History}
\rightarrow
\text{Persistent Spatial Knowledge}
}
$$

---

In [ ]:
# ============================================================
# Combo 9 — Persistent Exploration Memory Map
# ============================================================


# ------------------------------------------------------------
# Exploration-memory structure
# ------------------------------------------------------------

class ExplorationMemory:
    def __init__(self, uuv_id: int, width: int, height: int):
        self.uuv_id = uuv_id
        self.width = width
        self.height = height

        self.observed = np.zeros(
            (height, width),
            dtype=bool,
        )

    def update_from_fov(
        self,
        state: UUVState,
        fov_width: int = FOV_WIDTH,
        fov_height: int = FOV_HEIGHT,
    ):
        """
        Mark all valid world cells inside the current FOV as observed.

        Returns:
            newly_observed_count
        """

        x_min, y_min, x_max, y_max = get_fov_bounds(
            state,
            fov_width=fov_width,
            fov_height=fov_height,
        )

        x0 = max(0, x_min)
        y0 = max(0, y_min)

        x1 = min(self.width, x_max)
        y1 = min(self.height, y_max)

        before_count = int(self.observed.sum())

        self.observed[
            y0:y1,
            x0:x1,
        ] = True

        after_count = int(self.observed.sum())

        newly_observed = after_count - before_count

        return newly_observed

    def observed_count(self):
        return int(self.observed.sum())

    def coverage(self):
        return self.observed_count() / (
            self.width * self.height
        )


# ------------------------------------------------------------
# Stage 1 — Empty private memory
# ------------------------------------------------------------

uuv0_exploration_memory = ExplorationMemory(
    uuv_id=uuv0.uuv_id,
    width=WIDTH,
    height=HEIGHT,
)

initial_observed = (
    uuv0_exploration_memory.observed_count()
)

print("=" * 60)
print("MultiUUV-Link — Persistent Exploration Memory")
print("=" * 60)

print("\nStage 1 — Empty memory")
print(
    f"  Observed cells      : "
    f"{initial_observed}"
)
print(
    f"  Coverage            : "
    f"{uuv0_exploration_memory.coverage() * 100:.6f}%"
)

if initial_observed != 0:
    raise RuntimeError(
        "Exploration memory should begin completely unknown."
    )


# ------------------------------------------------------------
# Stage 2 — Observe at current center position
# ------------------------------------------------------------

new_center_cells = (
    uuv0_exploration_memory.update_from_fov(
        uuv0
    )
)

observed_after_center = (
    uuv0_exploration_memory.observed_count()
)

expected_center_cells = (
    FOV_WIDTH * FOV_HEIGHT
)

print("\nStage 2 — Observe center FOV")
print(
    f"  UUV position        : "
    f"({uuv0.x}, {uuv0.y})"
)
print(
    f"  Newly observed      : "
    f"{new_center_cells}"
)
print(
    f"  Total observed      : "
    f"{observed_after_center}"
)
print(
    f"  Expected observed   : "
    f"{expected_center_cells}"
)
print(
    f"  Coverage            : "
    f"{uuv0_exploration_memory.coverage() * 100:.6f}%"
)

if new_center_cells != expected_center_cells:
    raise RuntimeError(
        "Center FOV did not mark the expected number of cells."
    )

if observed_after_center != expected_center_cells:
    raise RuntimeError(
        "Unexpected total observed-cell count after center FOV."
    )


# ------------------------------------------------------------
# Stage 3 — Move one step right and observe again
# ------------------------------------------------------------

right_state = move_uuv(
    uuv0,
    ACTION_RIGHT,
)

new_right_cells = (
    uuv0_exploration_memory.update_from_fov(
        right_state
    )
)

observed_after_right = (
    uuv0_exploration_memory.observed_count()
)

expected_new_right_cells = (
    STEP_SIZE * FOV_HEIGHT
)

expected_total_after_right = (
    expected_center_cells
    +
    expected_new_right_cells
)

print("\nStage 3 — Move right and observe again")
print(
    f"  New UUV position    : "
    f"({right_state.x}, {right_state.y})"
)
print(
    f"  Newly observed      : "
    f"{new_right_cells}"
)
print(
    f"  Expected new cells  : "
    f"{expected_new_right_cells}"
)
print(
    f"  Total observed      : "
    f"{observed_after_right}"
)
print(
    f"  Expected total      : "
    f"{expected_total_after_right}"
)
print(
    f"  Coverage            : "
    f"{uuv0_exploration_memory.coverage() * 100:.6f}%"
)

if new_right_cells != expected_new_right_cells:
    raise RuntimeError(
        "Overlapping FOV update produced the wrong number of new cells."
    )

if observed_after_right != expected_total_after_right:
    raise RuntimeError(
        "Unexpected total exploration-memory size after second FOV."
    )


# ------------------------------------------------------------
# Stage 4 — Corner-memory validation
# ------------------------------------------------------------

corner_memory = ExplorationMemory(
    uuv_id=999,
    width=WIDTH,
    height=HEIGHT,
)

corner_state = UUVState(
    uuv_id=999,
    x=0,
    y=0,
)

corner_new_cells = (
    corner_memory.update_from_fov(
        corner_state
    )
)

corner_total = corner_memory.observed_count()

expected_corner_cells = (
    (FOV_WIDTH // 2)
    *
    (FOV_HEIGHT // 2)
)

print("\nStage 4 — Upper-left corner validation")
print(
    f"  UUV position        : "
    f"({corner_state.x}, {corner_state.y})"
)
print(
    f"  Newly observed      : "
    f"{corner_new_cells}"
)
print(
    f"  Expected real cells : "
    f"{expected_corner_cells}"
)
print(
    f"  Total observed      : "
    f"{corner_total}"
)
print(
    f"  Coverage            : "
    f"{corner_memory.coverage() * 100:.6f}%"
)

if corner_new_cells != expected_corner_cells:
    raise RuntimeError(
        "Corner memory incorrectly counted padded FOV pixels."
    )

if corner_total != expected_corner_cells:
    raise RuntimeError(
        "Corner exploration-memory total is incorrect."
    )


# ------------------------------------------------------------
# Monotonic-memory invariant
# ------------------------------------------------------------

monotonic_pass = (
    initial_observed
    <= observed_after_center
    <= observed_after_right
)

print("\nPersistent-memory invariants:")
print(
    f"  Monotonic growth    : "
    f"{monotonic_pass}"
)
print(
    f"  Padding excluded    : "
    f"{corner_total == expected_corner_cells}"
)
print(
    f"  FOV overlap handled : "
    f"{new_right_cells == expected_new_right_cells}"
)

if not monotonic_pass:
    raise RuntimeError(
        "Exploration memory is not monotonic."
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.imshow(
    uuv0_exploration_memory.observed,
    cmap="gray",
    vmin=0,
    vmax=1,
)

plt.scatter(
    uuv0.x,
    uuv0.y,
    s=80,
    marker="o",
    label="Initial UUV position",
)

plt.scatter(
    right_state.x,
    right_state.y,
    s=80,
    marker="x",
    label="After RIGHT action",
)

plt.title(
    "MultiUUV-Link — UUV-0 Private Exploration Memory"
)
plt.xlabel("x [pixels]")
plt.ylabel("y [pixels]")
plt.legend()
plt.tight_layout()
plt.show()


print("\nPersistent exploration memory validated.")

# Combo 10 — Four Independent UUV Agents with Random Safe Spawning

## Objective

This combo extends the validated single-UUV architecture into the first true multi-agent MultiUUV-Link state.

The initial swarm size is:

$$
N=4
$$

Each UUV receives its own independent:

- identity;
- position;
- trajectory;
- resource memory;
- exploration memory.

The UUVs share the same physical environment but do not yet communicate.

Therefore:

$$
M_i(t)\neq M_j(t)
$$

and:

$$
D_i(t)\neq D_j(t)
$$

in general.

At this stage there is still:

```text
NO communication
NO knowledge sharing
NO coordination
NO collision avoidance
NO autonomous exploration
```

This combo establishes only the multi-agent state and information architecture.

---

## Agent State

For UUV $i$:

$$
Agent_i(t)
=
\{
id_i,
p_i(t),
H_i(t),
M_i(t),
D_i(t)
\}
$$

where:

- $id_i$ is the agent identifier;
- $p_i(t)=(x_i,y_i)$ is position;
- $H_i(t)$ is trajectory history;
- $M_i(t)$ is private exploration memory;
- $D_i(t)$ is private resource memory.

The complete swarm state is:

$$
S(t)
=
\{
Agent_0,
Agent_1,
Agent_2,
Agent_3
\}
$$

---

## Random Initial Spawn

Unlike the earlier deterministic validation layout, the final prototype uses randomized initial UUV positions.

For each agent:

$$
p_i(0)\sim U_{\mathbb{Z}}(\Omega)
$$

where:

$$
\Omega=
\{(x,y)\mid 0\le x<W,\;0\le y<H\}
$$

However, completely unconstrained random spawning may place two UUVs almost on top of each other.

Because collision avoidance has not yet been implemented, we impose an initial spawn-separation condition:

$$
\|p_i(0)-p_j(0)\|_2
\ge
d_{\text{spawn,min}}
$$

for all:

$$
i\neq j
$$

The default minimum spawn separation will be:

$$
d_{\text{spawn,min}}=2\max(w_F,h_F)
$$

For the current $128\times128$ FOV:

$$
d_{\text{spawn,min}}=256\text{ pixels}
$$

This does not represent a real UUV safety distance.

It is simply a clean initialization constraint.

---

## Reproducibility

Random spawning should be stochastic by default.

Therefore:

```text
UUV_SPAWN_SEED = None
```

means each execution generates a new spawn layout.

For debugging, an integer seed may be supplied:

```text
UUV_SPAWN_SEED = 42
```

which produces a reproducible layout.

Thus:

$$
\boxed{
\text{Random by Default}
+
\text{Deterministic When Requested}
}
$$

---

## Spawn Rejection Sampling

The spawning algorithm uses rejection sampling.

For a new candidate position:

$$
p_c
$$

the candidate is accepted only if:

$$
\|p_c-p_j\|_2
\ge
d_{\text{spawn,min}}
$$

for every already accepted UUV position $p_j$.

Otherwise the candidate is rejected and another position is sampled.

A maximum-attempt guard prevents an infinite loop if impossible spawn constraints are requested.

---

## Private Exploration Memory

Each UUV receives an independent exploration-memory object:

$$
M_0,M_1,M_2,M_3
$$

Initially:

$$
M_i(0)=0
$$

Each UUV then performs one local observation:

$$
M_i(1)
=
M_i(0)
\cup
FOV_i(0)
$$

Because spawn locations are randomized, the initial observed regions also vary between episodes.

---

## Private Resource Memory

Each UUV independently begins with:

$$
D_i(0)=\varnothing
$$

After its first observation:

$$
D_i(1)
=
D_i(0)
\cup
V_i(0)
$$

where $V_i(0)$ is the set of resources inside that UUV's own FOV.

There is no automatic synchronization.

Therefore:

$$
R_k\in D_i
$$

does not imply:

$$
R_k\in D_j
$$

for:

$$
i\neq j
$$

unless UUV $j$ independently observes the same resource.

---

## Trajectory History

Every UUV begins with its randomized spawn position as the first trajectory point:

$$
H_i(0)=\{p_i(0)\}
$$

Later motion will produce:

$$
H_i(t+1)
=
H_i(t)
\cup
\{p_i(t+1)\}
$$

This history will later support:

- path length;
- visualization;
- redundancy analysis;
- collision analysis.

---

## First Multi-Agent Observation Cycle

After spawning, every UUV executes:

```text
Observe own FOV
      ↓
Update private exploration memory
      ↓
Detect resources in own FOV
      ↓
Update private resource memory
```

There is still no communication between agents.

---

## Information Invariant

The simulator knows:

$$
K_{\text{world}}
$$

while each UUV has its own private knowledge state:

$$
K_0,K_1,K_2,K_3
$$

with:

$$
K_i\subseteq K_{\text{world}}
$$

and generally:

$$
K_i\neq K_j
$$

This is the first true decentralized information architecture in MultiUUV-Link.

---

## Validation Conditions

The generated swarm must satisfy:

$$
N=4
$$

All UUV positions must satisfy:

$$
0\le x_i<W
$$

$$
0\le y_i<H
$$

All IDs must be unique.

Every pair must satisfy:

$$
d_{ij}
=
\|p_i-p_j\|_2
\ge
d_{\text{spawn,min}}
$$

Every UUV must have:

- an independent exploration-memory object;
- an independent resource-memory object;
- an independent trajectory list.

---

## Expected Output

The code cell should:

1. define the higher-level `UUVAgent`;
2. define random safe spawn generation;
3. create exactly four UUV agents;
4. randomize their positions;
5. enforce minimum pairwise separation;
6. create independent exploration memories;
7. create independent resource memories;
8. initialize trajectory histories;
9. perform one private observation cycle per agent;
10. print:
    - UUV positions;
    - pairwise minimum separation;
    - observed-cell counts;
    - coverage;
    - visible resources;
    - known resources;
11. verify that all private state objects are independent;
12. visualize all four randomized UUVs and their FOV rectangles.

A successful result establishes:

$$
\boxed{
\text{One Environment}
+
\text{Four Randomly Spawned Agents}
+
\text{Four Private Knowledge States}
}
$$

---

In [ ]:
# ============================================================
# Combo 10 — Four Independent UUV Agents
# Random Safe Spawning
# ============================================================

from dataclasses import dataclass, field
from itertools import combinations


# ------------------------------------------------------------
# Multi-UUV configuration
# ------------------------------------------------------------

UUV_SPAWN_SEED = None

MIN_SPAWN_SEPARATION = (
    2 * max(FOV_WIDTH, FOV_HEIGHT)
)

MAX_SPAWN_ATTEMPTS = 10_000

spawn_rng = np.random.default_rng(
    UUV_SPAWN_SEED
)


# ------------------------------------------------------------
# Higher-level UUV agent
# ------------------------------------------------------------

@dataclass
class UUVAgent:
    uuv_id: int
    state: UUVState
    exploration_memory: ExplorationMemory
    resource_memory: ResourceMemory
    trajectory: list = field(default_factory=list)

    def observe(self, resources):
        """
        Execute one legitimate private observation cycle.
        """

        newly_observed_cells = (
            self.exploration_memory.update_from_fov(
                self.state
            )
        )

        visible_resources = (
            discover_visible_resources(
                self.state,
                resources,
            )
        )

        newly_discovered_resources = (
            self.resource_memory.update(
                visible_resources
            )
        )

        return (
            newly_observed_cells,
            visible_resources,
            newly_discovered_resources,
        )


# ------------------------------------------------------------
# Distance utility
# ------------------------------------------------------------

def euclidean_distance(p1, p2):
    return float(
        np.hypot(
            p1[0] - p2[0],
            p1[1] - p2[1],
        )
    )


# ------------------------------------------------------------
# Random spawn generation
# ------------------------------------------------------------

def generate_safe_spawn_positions(
    num_uuvs: int,
    width: int,
    height: int,
    min_separation: float,
    rng=None,
    max_attempts: int = MAX_SPAWN_ATTEMPTS,
):
    if rng is None:
        rng = np.random.default_rng()

    positions = []
    attempts = 0

    while len(positions) < num_uuvs:

        if attempts >= max_attempts:
            raise RuntimeError(
                "Could not generate valid UUV spawn positions "
                "within the maximum attempt limit. "
                "Reduce minimum separation or UUV count."
            )

        attempts += 1

        candidate = (
            int(rng.integers(0, width)),
            int(rng.integers(0, height)),
        )

        candidate_valid = all(
            euclidean_distance(
                candidate,
                existing,
            )
            >=
            min_separation

            for existing in positions
        )

        if candidate_valid:
            positions.append(candidate)

    return positions, attempts


# ------------------------------------------------------------
# Generate random safe spawns
# ------------------------------------------------------------

spawn_positions, spawn_attempts = (
    generate_safe_spawn_positions(
        num_uuvs=NUM_UUVS,
        width=WIDTH,
        height=HEIGHT,
        min_separation=MIN_SPAWN_SEPARATION,
        rng=spawn_rng,
    )
)


# ------------------------------------------------------------
# Create independent agents
# ------------------------------------------------------------

agents = []

for uuv_id, (x, y) in enumerate(
    spawn_positions
):

    state = UUVState(
        uuv_id=uuv_id,
        x=x,
        y=y,
    )

    exploration_memory = ExplorationMemory(
        uuv_id=uuv_id,
        width=WIDTH,
        height=HEIGHT,
    )

    resource_memory = ResourceMemory(
        uuv_id=uuv_id,
    )

    agent = UUVAgent(
        uuv_id=uuv_id,
        state=state,
        exploration_memory=exploration_memory,
        resource_memory=resource_memory,
        trajectory=[
            (state.x, state.y)
        ],
    )

    agents.append(agent)


# ------------------------------------------------------------
# Basic structural validation
# ------------------------------------------------------------

if len(agents) != NUM_UUVS:
    raise RuntimeError(
        f"Expected {NUM_UUVS} agents, "
        f"got {len(agents)}."
    )

agent_ids = [
    agent.uuv_id
    for agent in agents
]

if len(set(agent_ids)) != NUM_UUVS:
    raise RuntimeError(
        "Agent IDs are not unique."
    )


# ------------------------------------------------------------
# Spawn bounds validation
# ------------------------------------------------------------

for agent in agents:

    if not (
        0 <= agent.state.x < WIDTH
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id} has invalid x."
        )

    if not (
        0 <= agent.state.y < HEIGHT
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id} has invalid y."
        )


# ------------------------------------------------------------
# Pairwise separation validation
# ------------------------------------------------------------

pairwise_distances = []

for agent_a, agent_b in combinations(
    agents,
    2,
):

    distance = euclidean_distance(
        (
            agent_a.state.x,
            agent_a.state.y,
        ),
        (
            agent_b.state.x,
            agent_b.state.y,
        ),
    )

    pairwise_distances.append(
        {
            "pair":
                f"UUV-{agent_a.uuv_id} / "
                f"UUV-{agent_b.uuv_id}",
            "distance": distance,
        }
    )

minimum_pairwise_distance = min(
    item["distance"]
    for item in pairwise_distances
)

if (
    minimum_pairwise_distance
    <
    MIN_SPAWN_SEPARATION
):
    raise RuntimeError(
        "Minimum spawn-separation invariant failed."
    )


# ------------------------------------------------------------
# Private-memory independence validation
# ------------------------------------------------------------

exploration_object_ids = [
    id(agent.exploration_memory)
    for agent in agents
]

resource_object_ids = [
    id(agent.resource_memory)
    for agent in agents
]

trajectory_object_ids = [
    id(agent.trajectory)
    for agent in agents
]

independent_exploration_memories = (
    len(set(exploration_object_ids))
    ==
    NUM_UUVS
)

independent_resource_memories = (
    len(set(resource_object_ids))
    ==
    NUM_UUVS
)

independent_trajectories = (
    len(set(trajectory_object_ids))
    ==
    NUM_UUVS
)

if not independent_exploration_memories:
    raise RuntimeError(
        "Exploration-memory objects are shared."
    )

if not independent_resource_memories:
    raise RuntimeError(
        "Resource-memory objects are shared."
    )

if not independent_trajectories:
    raise RuntimeError(
        "Trajectory lists are shared."
    )


# ------------------------------------------------------------
# First private observation cycle
# ------------------------------------------------------------

observation_reports = []

for agent in agents:

    (
        newly_observed,
        currently_visible,
        newly_discovered,
    ) = agent.observe(
        resources
    )

    observation_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "x":
                agent.state.x,

            "y":
                agent.state.y,

            "newly_observed":
                newly_observed,

            "total_observed":
                agent.exploration_memory
                .observed_count(),

            "coverage_percent":
                agent.exploration_memory
                .coverage()
                * 100,

            "visible_resources":
                len(currently_visible),

            "new_resources":
                len(newly_discovered),

            "known_resources":
                len(agent.resource_memory),
        }
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 72)
print(
    "MultiUUV-Link — Four Independent "
    "Randomly Spawned UUV Agents"
)
print("=" * 72)

print(
    f"\nSpawn seed             : "
    f"{UUV_SPAWN_SEED}"
)

print(
    f"Minimum separation     : "
    f"{MIN_SPAWN_SEPARATION:.2f} px"
)

print(
    f"Spawn attempts used    : "
    f"{spawn_attempts}"
)

print("\nSpawn positions:")

for agent in agents:
    print(
        f"  UUV-{agent.uuv_id}: "
        f"({agent.state.x}, "
        f"{agent.state.y})"
    )


# ------------------------------------------------------------
# Pairwise distance report
# ------------------------------------------------------------

print("\nPairwise spawn distances:")

for item in pairwise_distances:
    print(
        f"  {item['pair']:<15} : "
        f"{item['distance']:.2f} px"
    )

print(
    f"\nMinimum pairwise distance : "
    f"{minimum_pairwise_distance:.2f} px"
)


# ------------------------------------------------------------
# Private state report
# ------------------------------------------------------------

agent_report_df = pd.DataFrame(
    observation_reports
)

print("\nInitial private-agent states:")

print(
    agent_report_df.to_string(
        index=False,
        formatters={
            "coverage_percent":
                lambda x: f"{x:.6f}"
        },
    )
)


# ------------------------------------------------------------
# Independence report
# ------------------------------------------------------------

print("\nPrivate-state independence:")

print(
    f"  Exploration memories : "
    f"{independent_exploration_memories}"
)

print(
    f"  Resource memories    : "
    f"{independent_resource_memories}"
)

print(
    f"  Trajectory histories : "
    f"{independent_trajectories}"
)


# ------------------------------------------------------------
# Trajectory validation
# ------------------------------------------------------------

print("\nTrajectory initialization:")

for agent in agents:

    print(
        f"  UUV-{agent.uuv_id}: "
        f"{agent.trajectory}"
    )

    if len(agent.trajectory) != 1:
        raise RuntimeError(
            f"UUV-{agent.uuv_id} trajectory "
            "should initially contain one point."
        )


# ------------------------------------------------------------
# Initial observation validation
# ------------------------------------------------------------

for agent in agents:

    if (
        agent.exploration_memory
        .observed_count()
        <= 0
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id} failed to "
            "record its initial local FOV."
        )


# ------------------------------------------------------------
# Debug visualization
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.imshow(
    seabed_rgb
)

for agent in agents:

    x_min, y_min, _, _ = (
        get_fov_bounds(
            agent.state
        )
    )

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=120,
        marker="o",
        label=f"UUV-{agent.uuv_id}",
        zorder=6,
    )

    rectangle = plt.Rectangle(
        (
            x_min,
            y_min,
        ),
        FOV_WIDTH,
        FOV_HEIGHT,
        fill=False,
        linewidth=2,
        zorder=5,
    )

    plt.gca().add_patch(
        rectangle
    )


plt.title(
    "MultiUUV-Link — Random Initial Four-UUV Configuration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nFour independent randomly spawned "
    "UUV agents validated."
)

# Combo 11 — Independent Multi-UUV Evolution

## Objective

This combo advances the four independent UUVs through multiple timesteps while allowing motion large enough to be visually and experimentally meaningful on the current $1448\times1086$ seabed.

The original reconstructed MicroTrench motion primitive used:

$$
\Delta=8\text{ px}
$$

which was useful for validating discrete movement and image-coordinate behavior.

However, relative to the current map dimensions:

$$
W=1448,\qquad H=1086
$$

an $8$ px action represents only:

$$
\frac{8}{1448}\approx0.55\%
$$

of the map width.

For multi-agent exploration experiments this produces trajectories that remain highly localized.

Therefore the dynamic autonomy layer will use a configurable operational movement step:

$$
\Delta_{\text{motion}}=32\text{ px}
$$

This corresponds to:

$$
\frac{32}{128}=0.25
$$

of the local FOV width.

Successive observations therefore still overlap substantially while each action creates visually meaningful displacement.

---

## Separation Between Simulator Primitive and Experimental Configuration

The previously validated motion concept remains:

$$
p_i(t+1)
=
\operatorname{Clamp}
\left(
p_i(t)+u_{a_i(t)},
\Omega
\right)
$$

but the displacement magnitude becomes configurable.

For step magnitude:

$$
\Delta_m
$$

the action vectors are:

$$
u_0=
\begin{bmatrix}
0\\
-\Delta_m
\end{bmatrix}
$$

$$
u_1=
\begin{bmatrix}
\Delta_m\\
0
\end{bmatrix}
$$

$$
u_2=
\begin{bmatrix}
0\\
\Delta_m
\end{bmatrix}
$$

$$
u_3=
\begin{bmatrix}
-\Delta_m\\
0
\end{bmatrix}
$$

For this rollout:

$$
\boxed{\Delta_m=32\text{ px}}
$$

The movement scale remains an algorithmic parameter, not a physical velocity or distance.

---

## Independent Random-Walk Baseline

Each UUV independently samples:

$$
a_i(t)
\sim
U_{\mathbb Z}\{0,1,2,3\}
$$

with:

$$
P(a_i(t)=k)=\frac14
$$

for:

$$
k\in\{0,1,2,3\}
$$

There is still:

```text
NO communication
NO coordination
NO collision avoidance
NO frontier planning
NO task allocation
NO reinforcement learning
```

The random walk exists as the simplest dynamic multi-agent baseline.

---

## Simulation Horizon

The rollout length remains:

$$
T=50
$$

timesteps.

For four UUVs:

$$
N\times T
=
4\times50
=
200
$$

independent motion decisions are generated.

If no boundary clamping occurs, the maximum geometric path length of one UUV is:

$$
L_i=T\Delta_m
$$

and therefore:

$$
L_i=50\times32=1600\text{ px}
$$

This should produce trajectories that are clearly visible at full-map scale.

---

## Exploration Information Gain

After every move, UUV $i$ updates its private exploration memory:

$$
M_i(t+1)
=
M_i(t)
\cup
FOV_i(t+1)
$$

The newly acquired information is:

$$
\Delta A_i(t)
=
|M_i(t+1)|-|M_i(t)|
$$

This quantity is **not an RL reward**.

It is a classical information-gain measurement.

A revisited region may produce:

$$
\Delta A_i(t)=0
$$

while movement into unexplored space gives:

$$
\Delta A_i(t)>0
$$

This will later become useful for evaluating exploration algorithms and scoring frontier candidates.

---

## Private Resource Memory

Every UUV independently performs:

$$
V_i(t)
=
\{R_j\mid R_j\in FOV_i(t)\}
$$

followed by:

$$
D_i(t+1)
=
D_i(t)\cup V_i(t)
$$

No resource discovery is communicated to another UUV.

---

## Important Experimental Reset

Combo 10 already performed an initial observation at every spawn position.

This rollout continues from those states.

Therefore:

- initial FOV knowledge already exists;
- the 50 rollout updates measure additional information acquired afterward;
- total final coverage includes both initial and rollout observations.

---

## Boundary Handling

The larger movement step makes boundary encounters more likely.

Every transition therefore remains clamped:

$$
0\le x_i(t)<W
$$

$$
0\le y_i(t)<H
$$

If an action attempts to leave the environment, actual displacement may be:

$$
0\le d_i(t)\le32
$$

rather than exactly 32 px.

---

## Trajectory Metrics

Each UUV begins Combo 11 with one trajectory point inherited from Combo 10.

After:

$$
T=50
$$

moves, the expected trajectory length is:

$$
|H_i|=51
$$

The geometric path length is:

$$
L_i
=
\sum_{t=0}^{T-1}
\|p_i(t+1)-p_i(t)\|_2
$$

and should usually approach:

$$
1600\text{ px}
$$

unless boundary clamping reduces displacement.

---

## Visualization

The final global visualization will display:

- the seabed;
- each UUV's complete trajectory;
- each initial spawn position;
- each final position.

With:

$$
\Delta_m=32\text{ px}
$$

the paths should now occupy visibly meaningful portions of the environment.

This remains an **independent random-walk baseline**, so compact wandering and inefficient revisits are expected.

That inefficiency is desirable as a baseline because later autonomous exploration should measurably outperform it.

---

## Expected Output

The code cell should:

1. use an operational movement step of 32 px;
2. execute 50 independent random actions per UUV;
3. preserve boundary constraints;
4. update each private trajectory;
5. update each private exploration memory;
6. update each private resource memory;
7. record information gain at every timestep;
8. verify trajectory length equals 51;
9. report:
   - path length;
   - final coverage;
   - discovered resources;
   - newly observed cells;
   - zero-information-gain steps;
10. verify all private memory structures remain independent;
11. display clearly visible four-UUV trajectories.

A successful result establishes:

$$
\boxed{
\text{Independent Multi-UUV Dynamic Baseline}
}
$$

with a movement scale suitable for subsequent autonomy experiments.

---

In [ ]:
# ============================================================
# Combo 11 — Independent Multi-UUV Evolution
# ============================================================


# ------------------------------------------------------------
# Rollout configuration
# ------------------------------------------------------------

ROLLOUT_STEPS = 50

# Larger operational motion step for autonomy experiments.
ROLLOUT_STEP_SIZE = 32

# None = new stochastic rollout every run.
# Integer = reproducible rollout.
MOTION_SEED = None

motion_rng = np.random.default_rng(
    MOTION_SEED
)


# ------------------------------------------------------------
# Parameterized motion primitive
# ------------------------------------------------------------

def move_uuv_scaled(
    state: UUVState,
    action: int,
    step_size: int,
) -> UUVState:

    if step_size <= 0:
        raise ValueError(
            "step_size must be positive."
        )

    action_deltas = {
        ACTION_UP:
            (0, -step_size),

        ACTION_RIGHT:
            (step_size, 0),

        ACTION_DOWN:
            (0, step_size),

        ACTION_LEFT:
            (-step_size, 0),
    }

    if action not in action_deltas:
        raise ValueError(
            f"Invalid action {action}."
        )

    dx, dy = action_deltas[action]

    raw_x = state.x + dx
    raw_y = state.y + dy

    new_x = int(
        np.clip(
            raw_x,
            0,
            WIDTH - 1,
        )
    )

    new_y = int(
        np.clip(
            raw_y,
            0,
            HEIGHT - 1,
        )
    )

    return UUVState(
        uuv_id=state.uuv_id,
        x=new_x,
        y=new_y,
    )


# ------------------------------------------------------------
# Utility — geometric trajectory length
# ------------------------------------------------------------

def trajectory_path_length(
    trajectory
):
    total = 0.0

    for (
        (x0, y0),
        (x1, y1),
    ) in zip(
        trajectory[:-1],
        trajectory[1:],
    ):

        total += euclidean_distance(
            (x0, y0),
            (x1, y1),
        )

    return total


# ------------------------------------------------------------
# Rollout logs
# ------------------------------------------------------------

information_gain_log = {
    agent.uuv_id: []
    for agent in agents
}

action_log = {
    agent.uuv_id: []
    for agent in agents
}

actual_displacement_log = {
    agent.uuv_id: []
    for agent in agents
}


# ------------------------------------------------------------
# Independent multi-UUV rollout
# ------------------------------------------------------------

for timestep in range(
    ROLLOUT_STEPS
):

    for agent in agents:

        # ----------------------------------------------------
        # Independent random action
        # ----------------------------------------------------

        action = int(
            motion_rng.integers(
                0,
                4,
            )
        )

        action_log[
            agent.uuv_id
        ].append(
            action
        )


        # ----------------------------------------------------
        # Save previous position
        # ----------------------------------------------------

        previous_position = (
            agent.state.x,
            agent.state.y,
        )


        # ----------------------------------------------------
        # Scaled discrete motion
        # ----------------------------------------------------

        next_state = move_uuv_scaled(
            state=agent.state,
            action=action,
            step_size=ROLLOUT_STEP_SIZE,
        )

        next_position = (
            next_state.x,
            next_state.y,
        )


        # ----------------------------------------------------
        # Actual displacement
        # ----------------------------------------------------

        actual_displacement = (
            euclidean_distance(
                previous_position,
                next_position,
            )
        )

        actual_displacement_log[
            agent.uuv_id
        ].append(
            actual_displacement
        )


        # ----------------------------------------------------
        # Update agent state
        # ----------------------------------------------------

        agent.state = next_state

        agent.trajectory.append(
            next_position
        )


        # ----------------------------------------------------
        # Private exploration-memory update
        # ----------------------------------------------------

        newly_observed = (
            agent
            .exploration_memory
            .update_from_fov(
                agent.state
            )
        )

        information_gain_log[
            agent.uuv_id
        ].append(
            newly_observed
        )


        # ----------------------------------------------------
        # Private resource discovery
        # ----------------------------------------------------

        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )

        agent.resource_memory.update(
            visible_resources
        )


# ------------------------------------------------------------
# Expected trajectory length
# ------------------------------------------------------------

expected_trajectory_length = (
    ROLLOUT_STEPS + 1
)


# ------------------------------------------------------------
# State / trajectory validation
# ------------------------------------------------------------

for agent in agents:

    if (
        len(agent.trajectory)
        !=
        expected_trajectory_length
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id} trajectory "
            f"contains {len(agent.trajectory)} points; "
            f"expected {expected_trajectory_length}."
        )

    for x, y in agent.trajectory:

        if not (
            0 <= x < WIDTH
            and
            0 <= y < HEIGHT
        ):
            raise RuntimeError(
                f"UUV-{agent.uuv_id} contains "
                "an out-of-bounds trajectory point."
            )


# ------------------------------------------------------------
# Verify private-state independence
# ------------------------------------------------------------

independent_exploration_memories = (
    len(
        {
            id(agent.exploration_memory)
            for agent in agents
        }
    )
    ==
    NUM_UUVS
)

independent_resource_memories = (
    len(
        {
            id(agent.resource_memory)
            for agent in agents
        }
    )
    ==
    NUM_UUVS
)

independent_trajectories = (
    len(
        {
            id(agent.trajectory)
            for agent in agents
        }
    )
    ==
    NUM_UUVS
)

if not independent_exploration_memories:
    raise RuntimeError(
        "Exploration-memory independence lost."
    )

if not independent_resource_memories:
    raise RuntimeError(
        "Resource-memory independence lost."
    )

if not independent_trajectories:
    raise RuntimeError(
        "Trajectory independence lost."
    )


# ------------------------------------------------------------
# Final rollout metrics
# ------------------------------------------------------------

rollout_reports = []

for agent in agents:

    gains = np.asarray(
        information_gain_log[
            agent.uuv_id
        ],
        dtype=int,
    )

    displacements = np.asarray(
        actual_displacement_log[
            agent.uuv_id
        ],
        dtype=float,
    )

    path_length = (
        trajectory_path_length(
            agent.trajectory
        )
    )

    rollout_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "final_x":
                agent.state.x,

            "final_y":
                agent.state.y,

            "trajectory_points":
                len(agent.trajectory),

            "path_length_px":
                path_length,

            "mean_step_px":
                displacements.mean(),

            "observed_cells":
                agent
                .exploration_memory
                .observed_count(),

            "coverage_percent":
                agent
                .exploration_memory
                .coverage()
                * 100,

            "known_resources":
                len(
                    agent.resource_memory
                ),

            "new_cells_rollout":
                int(
                    gains.sum()
                ),

            "mean_info_gain":
                gains.mean(),

            "zero_gain_steps":
                int(
                    (gains == 0)
                    .sum()
                ),
        }
    )


rollout_df = pd.DataFrame(
    rollout_reports
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 78
)

print(
    "MultiUUV-Link — Independent Multi-UUV "
    "Visible-Motion Baseline"
)

print(
    "=" * 78
)

print(
    f"\nRollout steps            : "
    f"{ROLLOUT_STEPS}"
)

print(
    f"Operational step size    : "
    f"{ROLLOUT_STEP_SIZE} px"
)

print(
    f"FOV dimensions           : "
    f"{FOV_WIDTH} × {FOV_HEIGHT} px"
)

print(
    f"Step / FOV ratio         : "
    f"{ROLLOUT_STEP_SIZE / FOV_WIDTH:.3f}"
)

print(
    f"Motion seed              : "
    f"{MOTION_SEED}"
)

print(
    f"Expected trajectory pts  : "
    f"{expected_trajectory_length}"
)


print(
    "\nFinal independent-agent metrics:"
)

print(
    rollout_df.to_string(
        index=False,
        formatters={
            "path_length_px":
                lambda x: f"{x:.2f}",

            "mean_step_px":
                lambda x: f"{x:.2f}",

            "coverage_percent":
                lambda x: f"{x:.6f}",

            "mean_info_gain":
                lambda x: f"{x:.2f}",
        },
    )
)


# ------------------------------------------------------------
# Independence report
# ------------------------------------------------------------

print(
    "\nPrivate-state independence:"
)

print(
    f"  Exploration memories : "
    f"{independent_exploration_memories}"
)

print(
    f"  Resource memories    : "
    f"{independent_resource_memories}"
)

print(
    f"  Trajectories         : "
    f"{independent_trajectories}"
)


# ------------------------------------------------------------
# Information-gain report
# ------------------------------------------------------------

print(
    "\nInformation gain:"
)

for agent in agents:

    gains = np.asarray(
        information_gain_log[
            agent.uuv_id
        ]
    )

    print(
        f"  UUV-{agent.uuv_id}: "
        f"total={int(gains.sum())}, "
        f"mean/step={gains.mean():.2f}, "
        f"zero-gain={int((gains == 0).sum())}/"
        f"{ROLLOUT_STEPS}"
    )


# ------------------------------------------------------------
# Debug trajectory visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)

for agent in agents:

    trajectory_array = np.asarray(
        agent.trajectory
    )

    # Complete path
    plt.plot(
        trajectory_array[:, 0],
        trajectory_array[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
        zorder=5,
    )

    # Starting location
    plt.scatter(
        trajectory_array[0, 0],
        trajectory_array[0, 1],
        marker="o",
        s=90,
        zorder=6,
    )

    # Final location
    plt.scatter(
        trajectory_array[-1, 0],
        trajectory_array[-1, 1],
        marker="x",
        s=130,
        linewidths=2.5,
        zorder=7,
    )


plt.title(
    "MultiUUV-Link — Independent Random-Walk Baseline "
    "(32 px Motion Step)"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nIndependent multi-UUV "
    "visible-motion baseline validated."
)

# Combo 12 — Frontier Extraction from Private Exploration Memory

## Objective

This combo introduces the first building block of autonomous frontier-based exploration.

Until now, the UUVs have moved using random actions.

A frontier-based explorer instead uses its private spatial memory to identify the boundary between:

```text
KNOWN SPACE
```

and:

```text
UNKNOWN SPACE
```

These boundaries indicate locations from which additional unexplored territory may become observable.

The goal of this combo is **not yet to move toward a frontier**.

It only identifies frontier cells correctly from each UUV's private exploration memory.

There is still:

```text
NO frontier scoring
NO goal selection
NO path planner
NO communication
NO collision avoidance
```

---

## Exploration Memory

For UUV $i$, the private exploration memory is:

$$
M_i(y,x)\in\{0,1\}
$$

where:

$$
M_i(y,x)=1
$$

means the cell has been observed, while:

$$
M_i(y,x)=0
$$

means the cell remains unknown.

---

## Frontier Definition

Because the current environment contains no explicit obstacles, a frontier may be defined as an **unknown map cell adjacent to at least one observed cell**.

Let:

$$
\mathcal{N}(x,y)
$$

represent the local neighborhood of cell $(x,y)$.

Then a frontier cell satisfies:

$$
F_i(y,x)=1
$$

if:

$$
M_i(y,x)=0
$$

and:

$$
\exists (x',y')\in\mathcal{N}(x,y)
$$

such that:

$$
M_i(y',x')=1
$$

Thus:

$$
F_i
=
\operatorname{Neighbor}(M_i)
\cap
\neg M_i
$$

---

## Neighborhood Model

The initial implementation uses an 8-connected neighborhood:

```text
↖ ↑ ↗
← ● →
↙ ↓ ↘
```

This corresponds to a:

$$
3\times3
$$

structuring element.

If any observed pixel lies within this neighborhood, an adjacent unknown pixel becomes a frontier candidate.

---

## Morphological Interpretation

Frontier extraction can be implemented efficiently using binary dilation.

Let:

$$
D_i
=
\operatorname{Dilate}(M_i)
$$

Then:

$$
F_i
=
D_i
\land
\neg M_i
$$

The dilation expands the observed region outward by one pixel.

Subtracting the already observed region leaves only the immediate unknown boundary.

---

## Important Information Constraint

Frontiers must be computed entirely from:

$$
M_i
$$

the UUV's private memory.

The algorithm must **not** inspect:

- the complete world exploration state;
- other UUV memories;
- hidden resources;
- simulator-only ground truth.

Therefore:

$$
F_i
=
f(M_i)
$$

rather than:

$$
F_i
=
f(K_{\text{world}})
$$

This maintains decentralized information integrity.

---

## Why Frontier Extraction Matters

Random exploration chooses actions without considering whether they reveal new territory.

A frontier represents a geometrically meaningful exploration opportunity.

The later decision architecture will become:

```text
Private Memory
      ↓
Extract Frontiers
      ↓
Group Frontier Regions
      ↓
Score Candidates
      ↓
Choose Goal
      ↓
Move Toward Goal
      ↓
Observe
      ↓
Update Memory
      ↓
Repeat
```

This combo implements only the first transformation:

$$
M_i
\rightarrow
F_i
$$

---

## Current Validation State

Combo 11 has already produced nontrivial private exploration maps through 50 random-walk steps.

These memories are useful test inputs because they contain:

- explored regions;
- unknown regions;
- irregular boundaries;
- overlap from repeated observations.

We therefore use the existing UUV memories rather than constructing a synthetic map.

---

## Frontier Count

For UUV $i$, define:

$$
N_{F,i}
=
\sum_{x,y}F_i(y,x)
$$

The current partially explored maps should produce:

$$
N_{F,i}>0
$$

for every UUV.

If:

$$
N_{F,i}=0
$$

while coverage is substantially below 100%, frontier extraction has failed.

---

## Frontier Validity Invariants

Every detected frontier cell must satisfy:

$$
M_i(y,x)=0
$$

Therefore:

$$
F_i\cap M_i=\varnothing
$$

No cell may simultaneously be:

```text
OBSERVED
```

and:

```text
FRONTIER
```

Additionally:

$$
F_i\subseteq\Omega
$$

because frontier cells exist only inside the valid map.

---

## Visualization

The private knowledge visualization will use:

```text
black   → unknown
white   → observed
overlay → frontier boundary
```

The frontier should appear as a thin boundary around previously explored regions.

This visualization represents what can be derived from the UUV's own memory.

It does not expose hidden world information.

---

## Expected Output

The code cell should:

1. define a frontier-extraction function;
2. use only private exploration memory;
3. compute one frontier mask per UUV;
4. count frontier cells;
5. verify that:
   - frontier cells are unknown;
   - frontier cells lie inside the map;
   - partially explored agents possess nonzero frontiers;
6. print:
   - observed cells;
   - coverage;
   - frontier-cell count;
7. visualize the frontier map for each UUV.

A successful result establishes:

$$
\boxed{
\text{Private Exploration Memory}
\rightarrow
\text{Frontier Geometry}
}
$$

This becomes the first perception-to-planning interface of the autonomous exploration stack.

---

In [ ]:
# ============================================================
# Combo 12 — Frontier Extraction from Private Memory
# ============================================================


# ------------------------------------------------------------
# Frontier extraction
# ------------------------------------------------------------

def extract_frontier_mask(
    exploration_memory: ExplorationMemory,
):
    """
    Detect unknown cells adjacent to observed cells.

    Frontier definition:
        frontier = dilated(observed) AND NOT observed

    IMPORTANT:
    This function uses only the UUV's private exploration memory.
    """

    observed = (
        exploration_memory.observed
        .astype(np.uint8)
    )

    # 8-connected neighborhood
    kernel = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    dilated = cv2.dilate(
        observed,
        kernel,
        iterations=1,
    )

    frontier = (
        (dilated > 0)
        &
        (~exploration_memory.observed)
    )

    return frontier


# ------------------------------------------------------------
# Resource marker convention
# ------------------------------------------------------------

marker_by_class = {
    "A": "o",   # Circle
    "B": "^",   # Triangle
    "C": "s",   # Square
}


# ------------------------------------------------------------
# Extract private frontier maps
# ------------------------------------------------------------

frontier_masks = {}

frontier_reports = []

for agent in agents:

    memory = agent.exploration_memory

    frontier = extract_frontier_mask(
        memory
    )

    frontier_masks[
        agent.uuv_id
    ] = frontier


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    observed_cells = (
        memory.observed_count()
    )

    frontier_cells = int(
        frontier.sum()
    )

    coverage_percent = (
        memory.coverage()
        * 100
    )

    known_resources = (
        agent.resource_memory.all_resources()
    )

    known_resource_count = len(
        known_resources
    )


    # --------------------------------------------------------
    # Frontier validity
    # --------------------------------------------------------

    overlap = (
        frontier
        &
        memory.observed
    )

    frontier_observed_overlap = int(
        overlap.sum()
    )

    if frontier_observed_overlap != 0:
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "frontier overlaps observed space."
        )

    if (
        coverage_percent < 100.0
        and
        frontier_cells == 0
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "partially explored map has no frontier."
        )


    # --------------------------------------------------------
    # Verify every remembered resource was legitimately known
    # --------------------------------------------------------

    known_resource_ids = {
        resource.resource_id
        for resource in known_resources
    }

    if len(known_resource_ids) != known_resource_count:
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "duplicate resource IDs exist in private memory."
        )


    frontier_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "observed_cells":
                observed_cells,

            "coverage_percent":
                coverage_percent,

            "frontier_cells":
                frontier_cells,

            "known_resources":
                known_resource_count,

            "observed_frontier_overlap":
                frontier_observed_overlap,
        }
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

frontier_df = pd.DataFrame(
    frontier_reports
)

print(
    "=" * 82
)

print(
    "MultiUUV-Link — Private Frontier Extraction "
    "+ Discovered Resource Knowledge"
)

print(
    "=" * 82
)

print(
    "\nFrontier / knowledge statistics:"
)

print(
    frontier_df.to_string(
        index=False,
        formatters={
            "coverage_percent":
                lambda x: f"{x:.6f}",
        },
    )
)


# ------------------------------------------------------------
# Global invariants
# ------------------------------------------------------------

all_nonzero = all(
    report["frontier_cells"] > 0
    for report in frontier_reports
)

all_disjoint = all(
    report["observed_frontier_overlap"] == 0
    for report in frontier_reports
)

print(
    "\nFrontier invariants:"
)

print(
    f"  Nonzero frontiers          : "
    f"{all_nonzero}"
)

print(
    f"  Frontier/observed disjoint : "
    f"{all_disjoint}"
)


if not all_nonzero:
    raise RuntimeError(
        "At least one UUV has no frontier."
    )

if not all_disjoint:
    raise RuntimeError(
        "Frontier and observed regions overlap."
    )


# ------------------------------------------------------------
# Per-agent private knowledge reports
# ------------------------------------------------------------

print(
    "\nPrivate discovered-resource memories:"
)

for agent in agents:

    known_resources = (
        agent.resource_memory.all_resources()
    )

    print(
        f"\nUUV-{agent.uuv_id}: "
        f"{len(known_resources)} known resource(s)"
    )

    if len(known_resources) == 0:
        print("  None")

    else:
        for resource in sorted(
            known_resources,
            key=lambda r: r.resource_id,
        ):
            print(
                f"  ID={resource.resource_id:2d} | "
                f"Type {resource.resource_class} | "
                f"Position=({resource.x}, {resource.y})"
            )


# ------------------------------------------------------------
# Private frontier + knowledge visualizations
# ------------------------------------------------------------

for agent in agents:

    memory = (
        agent.exploration_memory.observed
    )

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    known_resources = (
        agent.resource_memory.all_resources()
    )


    # --------------------------------------------------------
    # Start from seabed as human-facing background
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 7)
    )

    plt.imshow(
        seabed_rgb
    )


    # --------------------------------------------------------
    # UNKNOWN-space darkening
    #
    # This makes explored space visually distinguishable while
    # keeping the actual seabed texture visible.
    # --------------------------------------------------------

    unknown_overlay = np.zeros(
        (HEIGHT, WIDTH, 4),
        dtype=np.float32,
    )

    unknown_overlay[
        ~memory,
        3
    ] = 0.58

    plt.imshow(
        unknown_overlay
    )


    # --------------------------------------------------------
    # Frontier overlay
    # --------------------------------------------------------

    frontier_y, frontier_x = np.where(
        frontier
    )

    plt.scatter(
        frontier_x,
        frontier_y,
        s=3,
        marker=".",
        label="Frontier",
        zorder=5,
    )


    # --------------------------------------------------------
    # Discovered resources
    #
    # IMPORTANT:
    # Only resources present in THIS UUV's private resource
    # memory are drawn.
    # Hidden simulator resources are NOT rendered here.
    # --------------------------------------------------------

    for resource_class in RESOURCE_CLASSES:

        class_resources = [
            resource
            for resource in known_resources
            if resource.resource_class
            ==
            resource_class
        ]

        if not class_resources:
            continue

        resource_x = [
            resource.x
            for resource in class_resources
        ]

        resource_y = [
            resource.y
            for resource in class_resources
        ]

        plt.scatter(
            resource_x,
            resource_y,
            marker=marker_by_class[
                resource_class
            ],
            s=170,
            facecolors="white",
            edgecolors="black",
            linewidths=1.8,
            label=(
                f"Known Type "
                f"{resource_class}"
            ),
            zorder=8,
        )


    # --------------------------------------------------------
    # Current UUV position
    # --------------------------------------------------------

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=130,
        marker="x",
        linewidths=2.5,
        label=f"UUV-{agent.uuv_id}",
        zorder=10,
    )


    # --------------------------------------------------------
    # Plot formatting
    # --------------------------------------------------------

    plt.title(
        f"MultiUUV-Link — UUV-{agent.uuv_id} "
        "Private Exploration + Resource Knowledge"
    )

    plt.xlabel(
        "x [pixels]"
    )

    plt.ylabel(
        "y [pixels]"
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.show()


# ------------------------------------------------------------
# Private-knowledge integrity check
# ------------------------------------------------------------

print(
    "\nPrivate knowledge integrity:"
)

for agent in agents:

    known_ids = {
        resource.resource_id
        for resource
        in agent.resource_memory.all_resources()
    }

    print(
        f"  UUV-{agent.uuv_id}: "
        f"{len(known_ids)} resource(s) rendered "
        "from private memory only"
    )


print(
    "\nPrivate frontier extraction and "
    "resource-knowledge visualization validated."
)

# Combo 13 — Frontier Region Grouping and Candidate Goal Extraction

## Objective

Combo 12 produced a binary frontier mask for each UUV:

$$
F_i(y,x)\in\{0,1\}
$$

where each frontier pixel marks an unknown cell adjacent to observed space.

However, a frontier mask may contain hundreds or thousands of individual frontier pixels.

For example, a UUV may have:

$$
N_{F,i}>1000
$$

frontier cells.

Treating every individual pixel as a planning goal would be inefficient and unstable.

Therefore this combo groups spatially connected frontier pixels into **frontier regions** and converts each sufficiently large region into one representative candidate goal.

The pipeline becomes:

```text
Private Exploration Memory
        ↓
Frontier Mask
        ↓
Connected Frontier Regions
        ↓
Filter Tiny Regions
        ↓
Representative Candidate Point
```

This combo does **not** yet score or select a goal.

There is still:

```text
NO frontier utility function
NO information-gain prediction
NO distance weighting
NO goal selection
NO path planner
NO communication
```

---

## Frontier Connectivity

Let the binary frontier mask for UUV $i$ be:

$$
F_i
$$

Two frontier cells belong to the same region if they are connected through an 8-connected neighborhood.

Thus each connected component forms a frontier region:

$$
\mathcal{F}_{i,k}
$$

where:

$$
k=1,2,\ldots,N_i^{regions}
$$

and:

$$
\mathcal{F}_{i,k}
\subseteq
F_i
$$

---

## Connected Components

The frontier mask can be represented as a graph where:

- every frontier pixel is a vertex;
- edges connect neighboring frontier pixels.

The connected components of this graph are the frontier regions.

Rather than explicitly building this large graph, the prototype uses connected-component labeling directly on the binary image.

For each frontier mask:

$$
F_i
$$

we compute:

$$
L_i(y,x)
$$

where:

$$
L_i(y,x)=k
$$

means pixel $(x,y)$ belongs to frontier component $k$.

---

## Region Size

For frontier component $k$:

$$
A_{i,k}
=
|\mathcal{F}_{i,k}|
$$

where $A_{i,k}$ is the number of frontier pixels in that region.

Very small connected components may arise from:

- tiny irregularities;
- one-pixel gaps;
- narrow artifacts;
- fragmented exploration geometry.

These are poor planning targets.

Therefore we define a minimum frontier-region size:

$$
A_{\min}
$$

and retain only regions satisfying:

$$
A_{i,k}\ge A_{\min}
$$

For the initial prototype:

$$
A_{\min}=10\text{ pixels}
$$

This threshold is algorithmic rather than physical.

---

## Representative Candidate Point

For each valid frontier region, one representative point is required.

The simplest choice is the component centroid:

$$
g_{i,k}
=
\begin{bmatrix}
\bar{x}_{i,k}\\
\bar{y}_{i,k}
\end{bmatrix}
$$

where:

$$
\bar{x}_{i,k}
=
\frac{1}{A_{i,k}}
\sum_{(x,y)\in\mathcal{F}_{i,k}}x
$$

and:

$$
\bar{y}_{i,k}
=
\frac{1}{A_{i,k}}
\sum_{(x,y)\in\mathcal{F}_{i,k}}y
$$

However, the mathematical centroid may fall on a pixel that is not itself a frontier cell.

Therefore the implementation will choose the actual frontier pixel in the component nearest to the centroid.

Thus every candidate satisfies:

$$
g_{i,k}\in\mathcal{F}_{i,k}
$$

This is important because the planner should receive a valid frontier location rather than an arbitrary point inside the component's bounding geometry.

---

## Candidate Representation

Each candidate frontier region will be represented as:

$$
G_{i,k}
=
(
id_k,
x_k,
y_k,
A_k
)
$$

where:

- $id_k$ is the component label;
- $(x_k,y_k)$ is the representative frontier pixel;
- $A_k$ is frontier-region size.

Later we can extend this representation with:

- estimated information gain;
- distance from UUV;
- redundancy;
- communication value;
- learned score.

For now it remains purely geometric.

---

## Information Boundary

Frontier-region extraction must depend only on:

$$
M_i
$$

through:

$$
F_i
$$

Therefore:

$$
G_i
=
f(F_i)
=
f(M_i)
$$

The algorithm must not inspect:

- hidden resources;
- other UUV memories;
- global unexplored truth;
- communication state.

A UUV derives its candidate goals entirely from its own private knowledge.

---

## Candidate Count

If a frontier contains many connected regions:

$$
N_{F,i}
\gg
N_{G,i}
$$

where:

- $N_{F,i}$ = number of frontier pixels;
- $N_{G,i}$ = number of frontier candidates.

The purpose of this combo is therefore a dimensionality reduction:

$$
\text{hundreds of frontier pixels}
\rightarrow
\text{small set of meaningful candidates}
$$

---

## Validation Invariants

Every candidate must satisfy:

$$
F_i(y_k,x_k)=1
$$

Therefore:

$$
g_{i,k}
\in F_i
$$

Each retained component must satisfy:

$$
A_{i,k}\ge A_{\min}
$$

and every candidate must lie inside the map:

$$
0\le x_k<W
$$

$$
0\le y_k<H
$$

Additionally, candidate count should satisfy:

$$
N_{G,i}\le N_{F,i}
$$

for every UUV.

---

## Visualization

Each UUV's private map will display:

```text
darkened region     → unknown
normal seabed       → explored
frontier shell      → frontier pixels
○ / △ / □           → privately discovered resources
×                    → current UUV
numbered marker     → frontier candidate
```

The candidate markers make the future planning problem explicit.

Instead of asking:

> Which of 1200 frontier pixels should I visit?

the UUV will ask:

> Which of these few frontier regions is most useful?

---

## Expected Output

The code cell should:

1. take each UUV's existing frontier mask;
2. perform 8-connected component labeling;
3. compute region sizes;
4. discard components smaller than 10 pixels;
5. compute each retained component centroid;
6. choose the actual frontier pixel nearest that centroid;
7. create a frontier-candidate table;
8. verify every candidate:
   - lies on a frontier;
   - lies inside the map;
   - belongs to a region meeting the size threshold;
9. print:
   - frontier pixels;
   - raw component count;
   - retained component count;
   - candidate coordinates and region sizes;
10. visualize all retained candidate points on each UUV's private frontier map.

A successful result establishes:

$$
\boxed{
\text{Frontier Pixels}
\rightarrow
\text{Frontier Regions}
\rightarrow
\text{Candidate Goals}
}
$$

---

In [ ]:
# ============================================================
# Combo 13 — Frontier Region Grouping and Candidate Extraction
# ============================================================


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MIN_FRONTIER_REGION_SIZE = 10


# ------------------------------------------------------------
# Frontier candidate extraction
# ------------------------------------------------------------

def extract_frontier_candidates(
    frontier_mask: np.ndarray,
    min_region_size: int = MIN_FRONTIER_REGION_SIZE,
):
    """
    Group frontier pixels into 8-connected components and
    return one representative frontier pixel per retained region.
    """

    if frontier_mask.ndim != 2:
        raise ValueError(
            "frontier_mask must be a 2D boolean/binary array."
        )

    frontier_uint8 = (
        frontier_mask.astype(np.uint8)
    )


    # --------------------------------------------------------
    # Connected-component labeling
    # --------------------------------------------------------

    (
        num_labels,
        labels,
        stats,
        centroids,
    ) = cv2.connectedComponentsWithStats(
        frontier_uint8,
        connectivity=8,
    )


    candidates = []


    # Label 0 is background
    for label_id in range(
        1,
        num_labels,
    ):

        region_size = int(
            stats[
                label_id,
                cv2.CC_STAT_AREA
            ]
        )

        if region_size < min_region_size:
            continue


        # ----------------------------------------------------
        # Geometric centroid
        # ----------------------------------------------------

        centroid_x = float(
            centroids[
                label_id,
                0
            ]
        )

        centroid_y = float(
            centroids[
                label_id,
                1
            ]
        )


        # ----------------------------------------------------
        # All actual frontier pixels in this region
        # ----------------------------------------------------

        region_y, region_x = np.where(
            labels == label_id
        )


        # ----------------------------------------------------
        # Select actual frontier pixel nearest centroid
        # ----------------------------------------------------

        squared_distances = (
            (region_x - centroid_x) ** 2
            +
            (region_y - centroid_y) ** 2
        )

        nearest_index = int(
            np.argmin(
                squared_distances
            )
        )

        candidate_x = int(
            region_x[
                nearest_index
            ]
        )

        candidate_y = int(
            region_y[
                nearest_index
            ]
        )


        candidates.append(
            {
                "component_id":
                    int(label_id),

                "x":
                    candidate_x,

                "y":
                    candidate_y,

                "region_size":
                    region_size,

                "centroid_x":
                    centroid_x,

                "centroid_y":
                    centroid_y,
            }
        )


    return (
        candidates,
        num_labels - 1,
        labels,
    )


# ------------------------------------------------------------
# Extract candidates for every UUV
# ------------------------------------------------------------

frontier_candidates = {}

candidate_reports = []

for agent in agents:

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    (
        candidates,
        raw_component_count,
        component_labels,
    ) = extract_frontier_candidates(
        frontier
    )

    frontier_candidates[
        agent.uuv_id
    ] = candidates


    # --------------------------------------------------------
    # Validate candidates
    # --------------------------------------------------------

    for candidate in candidates:

        x = candidate["x"]
        y = candidate["y"]

        if not (
            0 <= x < WIDTH
            and
            0 <= y < HEIGHT
        ):
            raise RuntimeError(
                f"UUV-{agent.uuv_id}: "
                "frontier candidate outside map."
            )

        if not frontier[
            y,
            x
        ]:
            raise RuntimeError(
                f"UUV-{agent.uuv_id}: "
                "candidate is not an actual frontier cell."
            )

        if (
            candidate["region_size"]
            <
            MIN_FRONTIER_REGION_SIZE
        ):
            raise RuntimeError(
                f"UUV-{agent.uuv_id}: "
                "undersized frontier region retained."
            )


    frontier_pixel_count = int(
        frontier.sum()
    )

    retained_count = len(
        candidates
    )

    if retained_count > frontier_pixel_count:
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "candidate count exceeds frontier-pixel count."
        )


    candidate_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "frontier_pixels":
                frontier_pixel_count,

            "raw_components":
                raw_component_count,

            "retained_regions":
                retained_count,
        }
    )


# ------------------------------------------------------------
# Summary report
# ------------------------------------------------------------

candidate_summary_df = pd.DataFrame(
    candidate_reports
)

print(
    "=" * 78
)

print(
    "MultiUUV-Link — Frontier Region Candidate Extraction"
)

print(
    "=" * 78
)

print(
    f"\nMinimum retained region size : "
    f"{MIN_FRONTIER_REGION_SIZE} pixels"
)

print(
    "\nCandidate summary:"
)

print(
    candidate_summary_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Detailed per-UUV candidate report
# ------------------------------------------------------------

for agent in agents:

    candidates = (
        frontier_candidates[
            agent.uuv_id
        ]
    )

    print(
        f"\nUUV-{agent.uuv_id} frontier candidates:"
    )

    if not candidates:

        print(
            "  None"
        )

        continue


    candidate_df = pd.DataFrame(
        candidates
    )

    print(
        candidate_df[
            [
                "component_id",
                "x",
                "y",
                "region_size",
                "centroid_x",
                "centroid_y",
            ]
        ].to_string(
            index=False,
            formatters={
                "centroid_x":
                    lambda x: f"{x:.2f}",

                "centroid_y":
                    lambda x: f"{x:.2f}",
            },
        )
    )


# ------------------------------------------------------------
# Candidate-count invariant
# ------------------------------------------------------------

all_have_candidates = all(
    len(
        frontier_candidates[
            agent.uuv_id
        ]
    )
    > 0
    for agent in agents
)

print(
    "\nCandidate invariants:"
)

print(
    f"  Every UUV has candidate(s) : "
    f"{all_have_candidates}"
)

if not all_have_candidates:
    raise RuntimeError(
        "At least one partially explored UUV "
        "has no retained frontier candidate."
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

for agent in agents:

    memory = (
        agent
        .exploration_memory
        .observed
    )

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    candidates = (
        frontier_candidates[
            agent.uuv_id
        ]
    )

    known_resources = (
        agent
        .resource_memory
        .all_resources()
    )


    # --------------------------------------------------------
    # Base seabed
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 7)
    )

    plt.imshow(
        seabed_rgb
    )


    # --------------------------------------------------------
    # Darken unknown territory
    # --------------------------------------------------------

    unknown_overlay = np.zeros(
        (HEIGHT, WIDTH, 4),
        dtype=np.float32,
    )

    unknown_overlay[
        ~memory,
        3
    ] = 0.58

    plt.imshow(
        unknown_overlay
    )


    # --------------------------------------------------------
    # Frontier pixels
    # --------------------------------------------------------

    frontier_y, frontier_x = np.where(
        frontier
    )

    plt.scatter(
        frontier_x,
        frontier_y,
        s=3,
        marker=".",
        label="Frontier",
        zorder=4,
    )


    # --------------------------------------------------------
    # Privately discovered resources only
    # --------------------------------------------------------

    for resource_class in RESOURCE_CLASSES:

        class_resources = [
            r
            for r in known_resources
            if r.resource_class
            ==
            resource_class
        ]

        if not class_resources:
            continue

        plt.scatter(
            [
                r.x
                for r in class_resources
            ],
            [
                r.y
                for r in class_resources
            ],
            marker=marker_by_class[
                resource_class
            ],
            s=160,
            facecolors="white",
            edgecolors="black",
            linewidths=1.7,
            label=(
                f"Known Type "
                f"{resource_class}"
            ),
            zorder=7,
        )


    # --------------------------------------------------------
    # Frontier candidates
    # --------------------------------------------------------

    for candidate_index, candidate in enumerate(
        candidates
    ):

        x = candidate["x"]
        y = candidate["y"]

        plt.scatter(
            x,
            y,
            s=120,
            marker="D",
            facecolors="none",
            edgecolors="white",
            linewidths=2,
            zorder=8,
        )

        plt.text(
            x + 8,
            y - 8,
            str(
                candidate_index
            ),
            fontsize=9,
            zorder=9,
        )


    # --------------------------------------------------------
    # Current UUV position
    # --------------------------------------------------------

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=140,
        marker="x",
        linewidths=2.5,
        label=f"UUV-{agent.uuv_id}",
        zorder=10,
    )


    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    plt.title(
        f"MultiUUV-Link — UUV-{agent.uuv_id} "
        "Frontier Candidate Regions"
    )

    plt.xlabel(
        "x [pixels]"
    )

    plt.ylabel(
        "y [pixels]"
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.show()


print(
    "\nFrontier-region candidate extraction validated."
)

# Combo 14 — Spatial Sampling of Frontier Goal Candidates

## Objective

Combo 13 correctly grouped frontier pixels into connected frontier regions.

However, the current exploration geometry frequently produces one continuous frontier component surrounding the entire explored region.

Therefore:

$$
N_{\text{components}}=1
$$

does not imply that the UUV has only one meaningful exploration direction.

A single connected frontier may extend around hundreds of pixels and contain possible goals:

- north;
- south;
- east;
- west;
- and intermediate directions.

Representing this entire boundary using one centroid-derived point is therefore too aggressive a reduction for planning.

This combo converts each retained frontier region into **multiple spatially separated candidate goals**.

The planning representation becomes:

```text
Private Exploration Memory
        ↓
Frontier Mask
        ↓
Connected Frontier Region
        ↓
Spatial Sampling
        ↓
Multiple Candidate Goals
```

There is still:

```text
NO candidate scoring
NO selected goal
NO autonomous movement
NO communication
NO collision avoidance
```

---

## Why Multiple Candidates Are Necessary

Suppose one frontier component contains:

$$
A_F=1300
$$

frontier pixels.

A single representative point gives:

$$
N_G=1
$$

even though the frontier may surround a large explored area.

The UUV then cannot distinguish between different exploration directions.

Instead, we seek:

$$
G_i=
\{g_{i,1},g_{i,2},\ldots,g_{i,m}\}
$$

where the candidates are distributed spatially around the frontier.

---

## Minimum Candidate Spacing

Candidate goals should not be nearly identical.

For two candidate points:

$$
g_a=(x_a,y_a)
$$

and:

$$
g_b=(x_b,y_b)
$$

we require approximately:

$$
\|g_a-g_b\|_2
\ge
d_G
$$

where:

$$
d_G
$$

is the minimum candidate spacing.

For the initial prototype:

$$
d_G=128\text{ pixels}
$$

which equals one FOV width.

This provides candidate directions that are meaningfully separated at the current perception scale.

---

## Greedy Spatial Sampling

For each connected frontier component:

1. collect all frontier pixels;
2. choose an initial frontier point;
3. repeatedly choose the frontier pixel farthest from the already selected candidate set;
4. stop when:
   - the desired maximum candidate count is reached; or
   - no remaining point is sufficiently far from the selected candidates.

The distance of frontier point $p$ to candidate set $G$ is:

$$
d(p,G)
=
\min_{g\in G}
\|p-g\|_2
$$

The next candidate is:

$$
p^*
=
\arg\max_{p\in F}
d(p,G)
$$

A candidate is accepted only if:

$$
d(p^*,G)
\ge
d_G
$$

This is a lightweight farthest-point sampling strategy.

---

## Maximum Candidate Count

To keep later planning inexpensive, each UUV will retain at most:

$$
N_{G,\max}=12
$$

frontier candidates.

Thus:

$$
1\le N_{G,i}\le12
$$

for the current partially explored maps.

The exact number depends on frontier geometry.

---

## Candidate Validity

Every candidate must remain an actual frontier cell:

$$
F_i(y_g,x_g)=1
$$

Therefore:

$$
g\in F_i
$$

Every candidate must also satisfy:

$$
0\le x_g<W
$$

$$
0\le y_g<H
$$

No candidate may be created from:

- simulator ground truth;
- hidden resources;
- another UUV's memory.

Candidate generation uses only the private frontier mask.

---

## Expected Geometry

Because the current UUV memories contain relatively compact but irregular explored regions, the sampled frontier candidates should appear distributed around the outside of each explored region.

Conceptually:

```text
       ◇

   ◇       ◇

◇    explored    ◇

   ◇       ◇

       ◇
```

where each diamond is a possible exploration goal.

The exact arrangement depends on the trajectory-generated private memory.

---

## Why This Precedes Scoring

The next planning stage will evaluate candidates using quantities such as:

$$
J_i(g)
=
w_I I_i(g)
-
w_D D_i(g)
-
w_R R_i(g)
$$

Such a score is useful only when:

$$
|G_i|>1
$$

in general.

This combo therefore creates the discrete decision set required by the future planner.

---

## Expected Output

The code cell should:

1. use the existing private frontier masks;
2. perform farthest-point spatial sampling;
3. enforce minimum candidate spacing of 128 pixels;
4. retain no more than 12 candidates per UUV;
5. verify that every selected point is an actual frontier cell;
6. report:
   - frontier pixel count;
   - sampled candidate count;
   - minimum pairwise candidate distance;
7. visualize:
   - private explored region;
   - frontier;
   - privately discovered resources;
   - current UUV position;
   - numbered candidate goals.

A successful result establishes:

$$
\boxed{
\text{Continuous Frontier Geometry}
\rightarrow
\text{Discrete Planning Choice Set}
}
$$

---

In [ ]:
# ============================================================
# Combo 14 — Spatial Sampling of Frontier Goal Candidates
# ============================================================

from itertools import combinations


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

FRONTIER_CANDIDATE_SPACING = 128.0
MAX_FRONTIER_CANDIDATES = 12


# ------------------------------------------------------------
# Spatial frontier sampling
# ------------------------------------------------------------

def sample_frontier_candidates(
    frontier_mask: np.ndarray,
    reference_position,
    min_spacing: float = FRONTIER_CANDIDATE_SPACING,
    max_candidates: int = MAX_FRONTIER_CANDIDATES,
):
    """
    Select multiple well-separated candidate pixels from a
    frontier mask using greedy farthest-point sampling.

    Uses only private frontier geometry.
    """

    frontier_y, frontier_x = np.where(
        frontier_mask
    )

    if len(frontier_x) == 0:
        return []


    frontier_points = np.column_stack(
        (
            frontier_x,
            frontier_y,
        )
    ).astype(float)


    # --------------------------------------------------------
    # First point:
    # choose frontier point farthest from current UUV position
    # --------------------------------------------------------

    ref = np.asarray(
        reference_position,
        dtype=float,
    )

    distances_from_uuv = np.linalg.norm(
        frontier_points - ref,
        axis=1,
    )

    first_index = int(
        np.argmax(
            distances_from_uuv
        )
    )

    selected_indices = [
        first_index
    ]


    # --------------------------------------------------------
    # Greedy farthest-point sampling
    # --------------------------------------------------------

    while (
        len(selected_indices)
        <
        max_candidates
    ):

        selected_points = (
            frontier_points[
                selected_indices
            ]
        )

        # Distance from each frontier pixel to each
        # already-selected candidate
        pairwise_distances = np.linalg.norm(
            frontier_points[:, None, :]
            -
            selected_points[None, :, :],
            axis=2,
        )

        # Distance to nearest existing candidate
        nearest_selected_distance = (
            pairwise_distances.min(
                axis=1
            )
        )

        # Prevent reselection
        nearest_selected_distance[
            selected_indices
        ] = -np.inf

        next_index = int(
            np.argmax(
                nearest_selected_distance
            )
        )

        next_distance = float(
            nearest_selected_distance[
                next_index
            ]
        )


        # Stop when no sufficiently separated point remains
        if next_distance < min_spacing:
            break


        selected_indices.append(
            next_index
        )


    # --------------------------------------------------------
    # Build candidate records
    # --------------------------------------------------------

    candidates = []

    for candidate_id, point_index in enumerate(
        selected_indices
    ):

        x = int(
            frontier_points[
                point_index,
                0
            ]
        )

        y = int(
            frontier_points[
                point_index,
                1
            ]
        )

        candidates.append(
            {
                "candidate_id":
                    candidate_id,

                "x":
                    x,

                "y":
                    y,

                "distance_from_uuv":
                    euclidean_distance(
                        reference_position,
                        (x, y),
                    ),
            }
        )


    return candidates


# ------------------------------------------------------------
# Generate candidates for every UUV
# ------------------------------------------------------------

sampled_frontier_candidates = {}

sampling_reports = []

for agent in agents:

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    current_position = (
        agent.state.x,
        agent.state.y,
    )

    candidates = sample_frontier_candidates(
        frontier_mask=frontier,
        reference_position=current_position,
    )

    sampled_frontier_candidates[
        agent.uuv_id
    ] = candidates


    # --------------------------------------------------------
    # Validate every candidate
    # --------------------------------------------------------

    for candidate in candidates:

        x = candidate["x"]
        y = candidate["y"]

        if not (
            0 <= x < WIDTH
            and
            0 <= y < HEIGHT
        ):
            raise RuntimeError(
                f"UUV-{agent.uuv_id}: "
                "candidate outside map."
            )

        if not frontier[
            y,
            x
        ]:
            raise RuntimeError(
                f"UUV-{agent.uuv_id}: "
                "candidate is not on frontier."
            )


    # --------------------------------------------------------
    # Pairwise spacing report
    # --------------------------------------------------------

    if len(candidates) >= 2:

        candidate_positions = [
            (
                candidate["x"],
                candidate["y"],
            )
            for candidate in candidates
        ]

        candidate_pair_distances = [
            euclidean_distance(
                p1,
                p2,
            )
            for p1, p2 in combinations(
                candidate_positions,
                2,
            )
        ]

        minimum_candidate_distance = min(
            candidate_pair_distances
        )

    else:
        minimum_candidate_distance = np.nan


    sampling_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "frontier_pixels":
                int(
                    frontier.sum()
                ),

            "candidate_count":
                len(
                    candidates
                ),

            "min_candidate_spacing":
                minimum_candidate_distance,
        }
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

sampling_df = pd.DataFrame(
    sampling_reports
)

print(
    "=" * 80
)

print(
    "MultiUUV-Link — Spatial Frontier Candidate Sampling"
)

print(
    "=" * 80
)

print(
    f"\nRequested minimum spacing : "
    f"{FRONTIER_CANDIDATE_SPACING:.1f} px"
)

print(
    f"Maximum candidates / UUV : "
    f"{MAX_FRONTIER_CANDIDATES}"
)

print(
    "\nSampling summary:"
)

print(
    sampling_df.to_string(
        index=False,
        formatters={
            "min_candidate_spacing":
                lambda x: (
                    "N/A"
                    if pd.isna(x)
                    else f"{x:.2f}"
                )
        },
    )
)


# ------------------------------------------------------------
# Detailed candidate tables
# ------------------------------------------------------------

for agent in agents:

    candidates = (
        sampled_frontier_candidates[
            agent.uuv_id
        ]
    )

    print(
        f"\nUUV-{agent.uuv_id} sampled candidates:"
    )

    if not candidates:

        print(
            "  None"
        )

        continue


    candidate_df = pd.DataFrame(
        candidates
    )

    print(
        candidate_df.to_string(
            index=False,
            formatters={
                "distance_from_uuv":
                    lambda x: f"{x:.2f}",
            },
        )
    )


# ------------------------------------------------------------
# Global validation
# ------------------------------------------------------------

all_have_multiple_candidates = all(
    len(
        sampled_frontier_candidates[
            agent.uuv_id
        ]
    )
    >= 2
    for agent in agents
)

print(
    "\nSampling invariants:"
)

print(
    f"  Every UUV has >= 2 candidates : "
    f"{all_have_multiple_candidates}"
)

if not all_have_multiple_candidates:
    raise RuntimeError(
        "At least one UUV does not have enough "
        "frontier candidates for meaningful scoring."
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

for agent in agents:

    memory = (
        agent.exploration_memory.observed
    )

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    candidates = (
        sampled_frontier_candidates[
            agent.uuv_id
        ]
    )

    known_resources = (
        agent.resource_memory.all_resources()
    )


    # --------------------------------------------------------
    # Seabed background
    # --------------------------------------------------------

    plt.figure(
        figsize=(12, 7)
    )

    plt.imshow(
        seabed_rgb
    )


    # --------------------------------------------------------
    # Darken unknown area
    # --------------------------------------------------------

    unknown_overlay = np.zeros(
        (HEIGHT, WIDTH, 4),
        dtype=np.float32,
    )

    unknown_overlay[
        ~memory,
        3
    ] = 0.58

    plt.imshow(
        unknown_overlay
    )


    # --------------------------------------------------------
    # Frontier
    # --------------------------------------------------------

    frontier_y, frontier_x = np.where(
        frontier
    )

    plt.scatter(
        frontier_x,
        frontier_y,
        s=3,
        marker=".",
        label="Frontier",
        zorder=4,
    )


    # --------------------------------------------------------
    # Private discovered resources
    # --------------------------------------------------------

    for resource_class in RESOURCE_CLASSES:

        class_resources = [
            resource
            for resource in known_resources
            if resource.resource_class
            ==
            resource_class
        ]

        if not class_resources:
            continue

        plt.scatter(
            [
                resource.x
                for resource
                in class_resources
            ],
            [
                resource.y
                for resource
                in class_resources
            ],
            marker=marker_by_class[
                resource_class
            ],
            s=160,
            facecolors="white",
            edgecolors="black",
            linewidths=1.7,
            label=(
                f"Known Type "
                f"{resource_class}"
            ),
            zorder=7,
        )


    # --------------------------------------------------------
    # Candidate goals
    # --------------------------------------------------------

    for candidate in candidates:

        x = candidate["x"]
        y = candidate["y"]

        plt.scatter(
            x,
            y,
            s=130,
            marker="D",
            facecolors="none",
            edgecolors="white",
            linewidths=2,
            zorder=8,
        )

        plt.text(
            x + 8,
            y - 8,
            str(
                candidate[
                    "candidate_id"
                ]
            ),
            fontsize=9,
            zorder=9,
        )


    # --------------------------------------------------------
    # Current UUV
    # --------------------------------------------------------

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=150,
        marker="x",
        linewidths=2.5,
        label=f"UUV-{agent.uuv_id}",
        zorder=10,
    )


    plt.title(
        f"MultiUUV-Link — UUV-{agent.uuv_id} "
        "Sampled Frontier Goals"
    )

    plt.xlabel(
        "x [pixels]"
    )

    plt.ylabel(
        "y [pixels]"
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.show()


print(
    "\nSpatial frontier candidate sampling validated."
)

# Combo 15 — Classical Frontier Candidate Scoring and Goal Selection

## Objective

Combo 14 generated a finite planning set for every UUV:

$$
G_i=
\{g_{i,1},g_{i,2},\ldots,g_{i,m}\}
$$

The UUV can now ask:

> Which frontier candidate should I explore next?

This combo implements the first classical frontier utility function.

Each candidate will be evaluated using only the UUV's **private exploration memory** and its own current position.

There is still:

```text
NO movement toward the selected goal
NO path planning
NO communication
NO inter-UUV coordination
NO collision avoidance
NO learned policy
```

This combo performs only:

```text
candidate
    ↓
feature calculation
    ↓
classical score
    ↓
selected goal
```

---

## Candidate Information Gain

For candidate goal:

$$
g=(x_g,y_g)
$$

we estimate the amount of new information that could be obtained if the UUV observed from that location.

Let:

$$
\mathcal{V}(g)
$$

be the valid map cells inside a candidate-centered FOV.

The estimated information gain is:

$$
I_i(g)
=
\left|
\{
p\in\mathcal{V}(g)
\mid
M_i(p)=0
\}
\right|
$$

Thus:

```text
unknown cells inside candidate FOV
        ↓
potential new information
```

A candidate surrounded mostly by unexplored territory receives high information gain.

---

## Valid FOV Area

Near the map boundary, part of the nominal FOV may lie outside the world.

Therefore the valid candidate FOV area is:

$$
A_{\text{valid}}(g)
=
|\mathcal{V}(g)|
$$

The normalized information-gain ratio is:

$$
\hat I_i(g)
=
\frac{I_i(g)}
{A_{\text{valid}}(g)}
$$

with:

$$
0\le\hat I_i(g)\le1
$$

This prevents boundary candidates from being unfairly evaluated using padded pixels.

---

## Redundancy

The already-observed portion of the candidate FOV is:

$$
A_{\text{known}}(g)
=
\left|
\{
p\in\mathcal{V}(g)
\mid
M_i(p)=1
\}
\right|
$$

The redundancy ratio is:

$$
\hat R_i(g)
=
\frac{
A_{\text{known}}(g)
}{
A_{\text{valid}}(g)
}
$$

Therefore:

$$
0\le\hat R_i(g)\le1
$$

A high redundancy ratio means much of the candidate's future observation has already been explored.

For the current binary explored/unknown representation:

$$
\hat I_i(g)+\hat R_i(g)=1
$$

for valid map cells.

They are retained separately because later extensions may introduce richer information states where information gain and redundancy are no longer exact complements.

---

## Travel Cost

For the current UUV position:

$$
p_i=(x_i,y_i)
$$

and frontier goal:

$$
g=(x_g,y_g)
$$

the Euclidean travel cost is:

$$
D_i(g)
=
\|g-p_i\|_2
$$

To combine distance with normalized information quantities, distance is normalized by the map diagonal:

$$
D_{\max}
=
\sqrt{
W^2+H^2
}
$$

and:

$$
\hat D_i(g)
=
\frac{
D_i(g)
}{
D_{\max}
}
$$

Thus:

$$
0\le\hat D_i(g)\le1
$$

---

## Classical Frontier Utility

The candidate utility is:

$$
J_i(g)
=
w_I\hat I_i(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i(g)
$$

The initial weights are:

$$
w_I=1.0
$$

$$
w_D=0.35
$$

$$
w_R=0.25
$$

The priorities are therefore:

1. prefer unexplored information;
2. avoid excessive travel;
3. discourage observations dominated by already-known territory.

These weights are baseline engineering parameters rather than theoretically optimal values.

They may later be tuned experimentally.

---

## Goal Selection

The selected frontier goal is:

$$
g_i^*
=
\arg\max_{g\in G_i}
J_i(g)
$$

Each UUV performs this computation independently.

Therefore:

$$
g_i^*
=
f(
M_i,
p_i,
G_i
)
$$

and not:

$$
f(
K_{\text{world}},
M_j,
R_{\text{hidden}}
)
$$

No other agent's memory or hidden simulator state is used.

---

## Important Information Constraint

Candidate scoring may inspect:

- this UUV's private exploration mask;
- this UUV's current position;
- this UUV's frontier candidates.

It must not inspect:

- hidden resource locations;
- other UUV memories;
- global coverage;
- future communication information.

Thus this remains a genuinely decentralized classical planning decision.

---

## Expected Behavior

A frontier candidate pointing toward largely unknown territory should produce:

$$
\hat I\rightarrow1
$$

and:

$$
\hat R\rightarrow0
$$

which increases utility.

A distant candidate incurs a larger:

$$
\hat D
$$

and therefore a travel penalty.

The winning goal represents a tradeoff:

```text
useful new information
        versus
travel cost
        versus
revisiting known space
```

---

## Validation

For every candidate:

$$
0\le\hat I\le1
$$

$$
0\le\hat R\le1
$$

$$
0\le\hat D\le1
$$

The selected goal must be one of the existing frontier candidates:

$$
g_i^*\in G_i
$$

and must still lie on the frontier:

$$
F_i(g_i^*)=1
$$

Each partially explored UUV should produce exactly one selected goal.

---

## Visualization

For each UUV:

```text
darkened terrain  → unknown
normal terrain    → observed
frontier          → known/unknown boundary
◇                 → candidate goal
★                 → selected best goal
○ / △ / □         → privately discovered resources
×                 → current UUV
```

The selected goal is a planning decision only.

The vehicle does not move toward it in this combo.

---

## Expected Output

The code cell should:

1. compute candidate-centered valid FOV geometry;
2. estimate normalized information gain;
3. compute normalized redundancy;
4. compute normalized travel cost;
5. evaluate:

$$
J_i(g)
=
w_I\hat I_i(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i(g)
$$

6. score all candidates independently for each UUV;
7. select one best frontier goal per UUV;
8. print full scoring tables;
9. verify all feature ranges;
10. verify every selected goal is a valid frontier candidate;
11. visualize candidate goals and the selected goal.

A successful result establishes:

$$
\boxed{
\text{Private Knowledge}
\rightarrow
\text{Candidate Utility}
\rightarrow
\text{Autonomous Goal Selection}
}
$$

---

In [ ]:
# ============================================================
# Combo 15 — Classical Frontier Candidate Scoring
# ============================================================


# ------------------------------------------------------------
# Classical utility weights
# ------------------------------------------------------------

W_INFORMATION = 1.00
W_DISTANCE = 0.35
W_REDUNDANCY = 0.25

MAP_DIAGONAL = float(
    np.hypot(
        WIDTH,
        HEIGHT,
    )
)


# ------------------------------------------------------------
# Candidate FOV statistics
# ------------------------------------------------------------

def candidate_fov_statistics(
    exploration_memory: ExplorationMemory,
    candidate_x: int,
    candidate_y: int,
    fov_width: int = FOV_WIDTH,
    fov_height: int = FOV_HEIGHT,
):
    """
    Estimate information content of an FOV centered on a
    frontier candidate using only private exploration memory.
    """

    temporary_state = UUVState(
        uuv_id=exploration_memory.uuv_id,
        x=int(candidate_x),
        y=int(candidate_y),
    )

    x_min, y_min, x_max, y_max = get_fov_bounds(
        temporary_state,
        fov_width=fov_width,
        fov_height=fov_height,
    )

    # Valid world intersection
    x0 = max(0, x_min)
    y0 = max(0, y_min)

    x1 = min(WIDTH, x_max)
    y1 = min(HEIGHT, y_max)

    memory_patch = exploration_memory.observed[
        y0:y1,
        x0:x1,
    ]

    valid_cells = int(
        memory_patch.size
    )

    if valid_cells <= 0:
        raise RuntimeError(
            "Candidate FOV contains no valid world cells."
        )

    known_cells = int(
        memory_patch.sum()
    )

    unknown_cells = (
        valid_cells - known_cells
    )

    information_ratio = (
        unknown_cells
        /
        valid_cells
    )

    redundancy_ratio = (
        known_cells
        /
        valid_cells
    )

    return {
        "valid_cells":
            valid_cells,

        "unknown_cells":
            unknown_cells,

        "known_cells":
            known_cells,

        "information_ratio":
            float(information_ratio),

        "redundancy_ratio":
            float(redundancy_ratio),
    }


# ------------------------------------------------------------
# Candidate scoring
# ------------------------------------------------------------

def score_frontier_candidate(
    agent: UUVAgent,
    candidate,
):
    x = int(
        candidate["x"]
    )

    y = int(
        candidate["y"]
    )

    fov_stats = candidate_fov_statistics(
        exploration_memory=
            agent.exploration_memory,
        candidate_x=x,
        candidate_y=y,
    )

    distance_px = euclidean_distance(
        (
            agent.state.x,
            agent.state.y,
        ),
        (
            x,
            y,
        ),
    )

    normalized_distance = (
        distance_px
        /
        MAP_DIAGONAL
    )

    information_ratio = (
        fov_stats[
            "information_ratio"
        ]
    )

    redundancy_ratio = (
        fov_stats[
            "redundancy_ratio"
        ]
    )

    utility = (
        W_INFORMATION
        * information_ratio

        -
        W_DISTANCE
        * normalized_distance

        -
        W_REDUNDANCY
        * redundancy_ratio
    )

    return {
        "candidate_id":
            int(
                candidate[
                    "candidate_id"
                ]
            ),

        "x":
            x,

        "y":
            y,

        "distance_px":
            float(
                distance_px
            ),

        "distance_norm":
            float(
                normalized_distance
            ),

        "valid_cells":
            fov_stats[
                "valid_cells"
            ],

        "unknown_cells":
            fov_stats[
                "unknown_cells"
            ],

        "known_cells":
            fov_stats[
                "known_cells"
            ],

        "info_ratio":
            information_ratio,

        "redundancy_ratio":
            redundancy_ratio,

        "utility":
            float(
                utility
            ),
    }


# ------------------------------------------------------------
# Score candidates for every UUV
# ------------------------------------------------------------

frontier_scores = {}

selected_frontier_goals = {}

for agent in agents:

    candidates = (
        sampled_frontier_candidates[
            agent.uuv_id
        ]
    )

    if not candidates:
        raise RuntimeError(
            f"UUV-{agent.uuv_id} has no "
            "frontier candidates to score."
        )

    scored_candidates = [
        score_frontier_candidate(
            agent,
            candidate,
        )
        for candidate in candidates
    ]


    # --------------------------------------------------------
    # Feature-range validation
    # --------------------------------------------------------

    for score in scored_candidates:

        if not (
            0.0
            <=
            score["info_ratio"]
            <=
            1.0
        ):
            raise RuntimeError(
                "Information ratio outside [0,1]."
            )

        if not (
            0.0
            <=
            score["redundancy_ratio"]
            <=
            1.0
        ):
            raise RuntimeError(
                "Redundancy ratio outside [0,1]."
            )

        if not (
            0.0
            <=
            score["distance_norm"]
            <=
            1.0
        ):
            raise RuntimeError(
                "Normalized distance outside [0,1]."
            )

        complement_error = abs(
            (
                score["info_ratio"]
                +
                score["redundancy_ratio"]
            )
            -
            1.0
        )

        if complement_error > 1e-9:
            raise RuntimeError(
                "Information and redundancy ratios "
                "do not sum to one."
            )


    # --------------------------------------------------------
    # Best candidate
    # --------------------------------------------------------

    selected_goal = max(
        scored_candidates,
        key=lambda item:
            item["utility"],
    )


    # --------------------------------------------------------
    # Frontier validity
    # --------------------------------------------------------

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    if not frontier[
        selected_goal["y"],
        selected_goal["x"],
    ]:
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "selected goal is not a frontier cell."
        )


    frontier_scores[
        agent.uuv_id
    ] = scored_candidates

    selected_frontier_goals[
        agent.uuv_id
    ] = selected_goal


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 88
)

print(
    "MultiUUV-Link — Classical Frontier Candidate Scoring"
)

print(
    "=" * 88
)

print(
    "\nUtility weights:"
)

print(
    f"  Information : "
    f"{W_INFORMATION:.2f}"
)

print(
    f"  Distance    : "
    f"{W_DISTANCE:.2f}"
)

print(
    f"  Redundancy  : "
    f"{W_REDUNDANCY:.2f}"
)


# ------------------------------------------------------------
# Detailed per-UUV score tables
# ------------------------------------------------------------

for agent in agents:

    scores = frontier_scores[
        agent.uuv_id
    ]

    selected = selected_frontier_goals[
        agent.uuv_id
    ]

    score_df = pd.DataFrame(
        scores
    )

    score_df[
        "selected"
    ] = (
        score_df[
            "candidate_id"
        ]
        ==
        selected[
            "candidate_id"
        ]
    )


    print(
        f"\nUUV-{agent.uuv_id} candidate scores:"
    )

    print(
        score_df[
            [
                "candidate_id",
                "x",
                "y",
                "distance_px",
                "unknown_cells",
                "info_ratio",
                "redundancy_ratio",
                "utility",
                "selected",
            ]
        ].to_string(
            index=False,
            formatters={
                "distance_px":
                    lambda x: f"{x:.2f}",

                "info_ratio":
                    lambda x: f"{x:.4f}",

                "redundancy_ratio":
                    lambda x: f"{x:.4f}",

                "utility":
                    lambda x: f"{x:.4f}",
            },
        )
    )

    print(
        f"\n  Selected goal: "
        f"candidate "
        f"{selected['candidate_id']} "
        f"at "
        f"({selected['x']}, "
        f"{selected['y']}) "
        f"with J="
        f"{selected['utility']:.4f}"
    )


# ------------------------------------------------------------
# Selection summary
# ------------------------------------------------------------

selection_summary = []

for agent in agents:

    selected = (
        selected_frontier_goals[
            agent.uuv_id
        ]
    )

    selection_summary.append(
        {
            "uuv_id":
                agent.uuv_id,

            "candidate_id":
                selected[
                    "candidate_id"
                ],

            "goal_x":
                selected["x"],

            "goal_y":
                selected["y"],

            "distance_px":
                selected[
                    "distance_px"
                ],

            "info_ratio":
                selected[
                    "info_ratio"
                ],

            "redundancy_ratio":
                selected[
                    "redundancy_ratio"
                ],

            "utility":
                selected[
                    "utility"
                ],
        }
    )


selection_df = pd.DataFrame(
    selection_summary
)

print(
    "\nSelected frontier goals:"
)

print(
    selection_df.to_string(
        index=False,
        formatters={
            "distance_px":
                lambda x: f"{x:.2f}",

            "info_ratio":
                lambda x: f"{x:.4f}",

            "redundancy_ratio":
                lambda x: f"{x:.4f}",

            "utility":
                lambda x: f"{x:.4f}",
        },
    )
)


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

for agent in agents:

    memory = (
        agent
        .exploration_memory
        .observed
    )

    frontier = (
        frontier_masks[
            agent.uuv_id
        ]
    )

    candidates = (
        sampled_frontier_candidates[
            agent.uuv_id
        ]
    )

    selected = (
        selected_frontier_goals[
            agent.uuv_id
        ]
    )

    known_resources = (
        agent
        .resource_memory
        .all_resources()
    )


    plt.figure(
        figsize=(12, 7)
    )

    plt.imshow(
        seabed_rgb
    )


    # --------------------------------------------------------
    # Unknown territory
    # --------------------------------------------------------

    unknown_overlay = np.zeros(
        (HEIGHT, WIDTH, 4),
        dtype=np.float32,
    )

    unknown_overlay[
        ~memory,
        3
    ] = 0.58

    plt.imshow(
        unknown_overlay
    )


    # --------------------------------------------------------
    # Frontier
    # --------------------------------------------------------

    frontier_y, frontier_x = np.where(
        frontier
    )

    plt.scatter(
        frontier_x,
        frontier_y,
        s=3,
        marker=".",
        label="Frontier",
        zorder=4,
    )


    # --------------------------------------------------------
    # Known resources
    # --------------------------------------------------------

    for resource_class in RESOURCE_CLASSES:

        class_resources = [
            resource
            for resource in known_resources
            if resource.resource_class
            ==
            resource_class
        ]

        if not class_resources:
            continue

        plt.scatter(
            [
                r.x
                for r in class_resources
            ],
            [
                r.y
                for r in class_resources
            ],
            marker=marker_by_class[
                resource_class
            ],
            s=160,
            facecolors="white",
            edgecolors="black",
            linewidths=1.7,
            label=(
                f"Known Type "
                f"{resource_class}"
            ),
            zorder=7,
        )


    # --------------------------------------------------------
    # All candidates
    # --------------------------------------------------------

    for candidate in candidates:

        plt.scatter(
            candidate["x"],
            candidate["y"],
            s=110,
            marker="D",
            facecolors="none",
            edgecolors="white",
            linewidths=1.8,
            zorder=8,
        )

        plt.text(
            candidate["x"] + 8,
            candidate["y"] - 8,
            str(
                candidate[
                    "candidate_id"
                ]
            ),
            fontsize=9,
            zorder=9,
        )


    # --------------------------------------------------------
    # Selected goal
    # --------------------------------------------------------

    plt.scatter(
        selected["x"],
        selected["y"],
        s=260,
        marker="*",
        edgecolors="black",
        linewidths=1.5,
        label="Selected Goal",
        zorder=11,
    )


    # --------------------------------------------------------
    # Current UUV
    # --------------------------------------------------------

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=150,
        marker="x",
        linewidths=2.5,
        label=f"UUV-{agent.uuv_id}",
        zorder=12,
    )


    plt.title(
        f"MultiUUV-Link — UUV-{agent.uuv_id} "
        "Classical Frontier Goal Selection"
    )

    plt.xlabel(
        "x [pixels]"
    )

    plt.ylabel(
        "y [pixels]"
    )

    plt.legend(
        loc="best"
    )

    plt.tight_layout()

    plt.show()


print(
    "\nClassical frontier candidate scoring "
    "and goal selection validated."
)

# Combo 16 — Goal-Directed Motion Toward Selected Frontier

## Objective

Combo 15 selected one frontier goal for every UUV:

$$
g_i^*
=
(x_i^*,y_i^*)
$$

The next requirement is to convert this high-level goal into an executable discrete action.

The UUV still uses the four-action motion space:

$$
a_i(t)\in\{0,1,2,3\}
$$

corresponding to:

```text
UP
RIGHT
DOWN
LEFT
```

Therefore this combo implements a simple local goal-directed controller that chooses the discrete action that most reduces the distance to the selected frontier goal.

There is still:

```text
NO multi-UUV coordination
NO collision avoidance
NO communication
NO task allocation
```

Each UUV independently tries to move toward its own selected frontier.

---

## Goal Error

For UUV position:

$$
p_i(t)
=
\begin{bmatrix}
x_i(t)\\
y_i(t)
\end{bmatrix}
$$

and selected goal:

$$
g_i
=
\begin{bmatrix}
x_i^*\\
y_i^*
\end{bmatrix}
$$

define position error:

$$
e_i(t)
=
g_i-p_i(t)
$$

or:

$$
e_i(t)
=
\begin{bmatrix}
x_i^*-x_i(t)\\
y_i^*-y_i(t)
\end{bmatrix}
$$

---

## Discrete Action Selection

Because diagonal motion is not currently available, the controller compares horizontal and vertical goal error.

Let:

$$
e_x=x_i^*-x_i
$$

$$
e_y=y_i^*-y_i
$$

If:

$$
|e_x|\ge|e_y|
$$

the UUV moves horizontally.

If:

$$
e_x>0
$$

choose:

$$
a=\text{RIGHT}
$$

otherwise choose:

$$
a=\text{LEFT}
$$

If:

$$
|e_y|>|e_x|
$$

the UUV moves vertically.

If:

$$
e_y>0
$$

choose:

$$
a=\text{DOWN}
$$

otherwise choose:

$$
a=\text{UP}
$$

This creates a simple greedy Manhattan-style steering rule.

---

## Goal Reached Condition

Because the motion step is discrete:

$$
\Delta_m=32\text{ px}
$$

the UUV may not land exactly on the selected frontier coordinate.

Therefore the goal is considered reached when:

$$
\|p_i-g_i\|_2
\le
r_g
$$

where:

$$
r_g=\Delta_m
$$

for the initial implementation.

Thus:

$$
r_g=32\text{ px}
$$

---

## Why Greedy Motion Is Sufficient Here

The current environment contains no explicit obstacles.

Therefore a full graph-search planner such as A* is not yet necessary.

The world is currently:

```text
continuous free image plane
+
bounded rectangular domain
```

so repeated greedy horizontal/vertical actions are sufficient to reduce distance to a target.

Later, if obstacles or restricted regions are introduced, the planner architecture can be upgraded.

---

## Closed-Loop Exploration

For each UUV, the local loop becomes:

```text
Selected frontier goal
        ↓
Choose greedy action
        ↓
Move one step
        ↓
Observe local FOV
        ↓
Update exploration memory
        ↓
Update resource memory
```

In this combo, the selected frontier goal remains fixed during the short execution segment.

Dynamic replanning will be introduced later.

---

## Validation Horizon

Each UUV will be given at most:

$$
T_g=20
$$

goal-directed steps.

The rollout ends early for that agent if:

$$
\|p_i-g_i\|_2
\le
r_g
$$

---

## Progress Metric

For each UUV, define initial distance:

$$
D_i^{start}
=
\|p_i(0)-g_i\|_2
$$

and final distance:

$$
D_i^{final}
=
\|p_i(T)-g_i\|_2
$$

A successful controller should satisfy:

$$
D_i^{final}
<
D_i^{start}
$$

unless the goal was already within the reach threshold.

---

## Information Gain During Goal Pursuit

Because the frontier goal lies near unknown space, goal-directed motion should often produce new explored area.

We record:

$$
\Delta A_i
=
|M_i(T)|-|M_i(0)|
$$

during goal pursuit.

This allows direct comparison with the previous random-walk baseline.

---

## Resource Discovery

At every goal-directed step, the UUV performs its normal discovery update:

$$
D_i(t+1)
=
D_i(t)
\cup
V_i(t)
$$

Therefore new resources encountered en route are legitimately stored in private memory.

---

## Expected Output

The code cell should:

1. define a greedy discrete action selector;
2. use the selected goal from Combo 15;
3. execute up to 20 goal-directed steps per UUV;
4. stop early if the goal is reached;
5. update:
   - UUV state;
   - trajectory;
   - exploration memory;
   - resource memory;
6. report:
   - start distance;
   - final distance;
   - number of steps;
   - goal reached;
   - new explored cells;
   - newly discovered resources;
7. verify that final distance is smaller than initial distance;
8. visualize:
   - pursuit trajectory;
   - selected frontier goal;
   - final UUV position.

A successful result establishes:

$$
\boxed{
\text{Selected Frontier Goal}
\rightarrow
\text{Goal-Directed Motion}
\rightarrow
\text{New Observation}
}
$$

---

In [ ]:
# ============================================================
# Combo 16 — Goal-Directed Motion Toward Selected Frontier
# ============================================================


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

GOAL_PURSUIT_MAX_STEPS = 20
GOAL_REACHED_RADIUS = ROLLOUT_STEP_SIZE


# ------------------------------------------------------------
# Greedy discrete action selector
# ------------------------------------------------------------

def choose_greedy_action(
    state: UUVState,
    goal_x: int,
    goal_y: int,
):
    """
    Choose the discrete action that reduces the dominant
    component of position error.
    """

    error_x = goal_x - state.x
    error_y = goal_y - state.y


    if abs(error_x) >= abs(error_y):

        if error_x > 0:
            return ACTION_RIGHT

        elif error_x < 0:
            return ACTION_LEFT


    if error_y > 0:
        return ACTION_DOWN

    elif error_y < 0:
        return ACTION_UP


    return None


# ------------------------------------------------------------
# Snapshot starting conditions
# ------------------------------------------------------------

goal_pursuit_reports = []

goal_pursuit_paths = {}

for agent in agents:

    selected_goal = (
        selected_frontier_goals[
            agent.uuv_id
        ]
    )

    goal_position = (
        selected_goal["x"],
        selected_goal["y"],
    )

    start_position = (
        agent.state.x,
        agent.state.y,
    )

    start_distance = euclidean_distance(
        start_position,
        goal_position,
    )

    start_observed_cells = (
        agent
        .exploration_memory
        .observed_count()
    )

    start_known_resources = len(
        agent.resource_memory
    )

    pursuit_path = [
        start_position
    ]


    # --------------------------------------------------------
    # Closed-loop pursuit
    # --------------------------------------------------------

    steps_used = 0
    goal_reached = False

    for step in range(
        GOAL_PURSUIT_MAX_STEPS
    ):

        current_position = (
            agent.state.x,
            agent.state.y,
        )

        current_distance = euclidean_distance(
            current_position,
            goal_position,
        )


        # ----------------------------------------------------
        # Goal condition
        # ----------------------------------------------------

        if current_distance <= GOAL_REACHED_RADIUS:

            goal_reached = True

            break


        # ----------------------------------------------------
        # Select greedy action
        # ----------------------------------------------------

        action = choose_greedy_action(
            agent.state,
            goal_x=goal_position[0],
            goal_y=goal_position[1],
        )

        if action is None:

            goal_reached = True

            break


        # ----------------------------------------------------
        # Move
        # ----------------------------------------------------

        next_state = move_uuv_scaled(
            state=agent.state,
            action=action,
            step_size=ROLLOUT_STEP_SIZE,
        )

        agent.state = next_state

        next_position = (
            next_state.x,
            next_state.y,
        )

        agent.trajectory.append(
            next_position
        )

        pursuit_path.append(
            next_position
        )

        steps_used += 1


        # ----------------------------------------------------
        # Private exploration update
        # ----------------------------------------------------

        agent.exploration_memory.update_from_fov(
            agent.state
        )


        # ----------------------------------------------------
        # Private resource update
        # ----------------------------------------------------

        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )

        agent.resource_memory.update(
            visible_resources
        )


    # --------------------------------------------------------
    # Final metrics
    # --------------------------------------------------------

    final_position = (
        agent.state.x,
        agent.state.y,
    )

    final_distance = euclidean_distance(
        final_position,
        goal_position,
    )

    if final_distance <= GOAL_REACHED_RADIUS:
        goal_reached = True


    final_observed_cells = (
        agent
        .exploration_memory
        .observed_count()
    )

    final_known_resources = len(
        agent.resource_memory
    )

    new_explored_cells = (
        final_observed_cells
        -
        start_observed_cells
    )

    new_resources = (
        final_known_resources
        -
        start_known_resources
    )


    # --------------------------------------------------------
    # Progress validation
    # --------------------------------------------------------

    if (
        start_distance > GOAL_REACHED_RADIUS
        and
        final_distance >= start_distance
    ):
        raise RuntimeError(
            f"UUV-{agent.uuv_id}: "
            "goal pursuit failed to reduce goal distance."
        )


    goal_pursuit_paths[
        agent.uuv_id
    ] = pursuit_path


    goal_pursuit_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "goal_x":
                goal_position[0],

            "goal_y":
                goal_position[1],

            "start_distance":
                start_distance,

            "final_distance":
                final_distance,

            "steps_used":
                steps_used,

            "goal_reached":
                goal_reached,

            "new_explored_cells":
                new_explored_cells,

            "new_resources":
                new_resources,

            "final_x":
                final_position[0],

            "final_y":
                final_position[1],
        }
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

goal_pursuit_df = pd.DataFrame(
    goal_pursuit_reports
)

print(
    "=" * 88
)

print(
    "MultiUUV-Link — Goal-Directed Frontier Pursuit"
)

print(
    "=" * 88
)

print(
    f"\nMaximum pursuit steps : "
    f"{GOAL_PURSUIT_MAX_STEPS}"
)

print(
    f"Goal reached radius   : "
    f"{GOAL_REACHED_RADIUS} px"
)

print(
    f"Motion step           : "
    f"{ROLLOUT_STEP_SIZE} px"
)

print(
    "\nGoal-pursuit results:"
)

print(
    goal_pursuit_df.to_string(
        index=False,
        formatters={
            "start_distance":
                lambda x: f"{x:.2f}",

            "final_distance":
                lambda x: f"{x:.2f}",
        },
    )
)


# ------------------------------------------------------------
# Progress summary
# ------------------------------------------------------------

print(
    "\nProgress checks:"
)

for report in goal_pursuit_reports:

    print(
        f"  UUV-{report['uuv_id']}: "
        f"{report['start_distance']:.2f} "
        f"-> "
        f"{report['final_distance']:.2f} px | "
        f"goal reached={report['goal_reached']}"
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


for agent in agents:

    selected_goal = (
        selected_frontier_goals[
            agent.uuv_id
        ]
    )

    path = np.asarray(
        goal_pursuit_paths[
            agent.uuv_id
        ]
    )


    # --------------------------------------------------------
    # Pursuit path
    # --------------------------------------------------------

    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.5,
        label=f"UUV-{agent.uuv_id}",
        zorder=6,
    )


    # --------------------------------------------------------
    # Start point
    # --------------------------------------------------------

    plt.scatter(
        path[0, 0],
        path[0, 1],
        s=90,
        marker="o",
        zorder=7,
    )


    # --------------------------------------------------------
    # Final point
    # --------------------------------------------------------

    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        s=120,
        marker="x",
        linewidths=2.5,
        zorder=8,
    )


    # --------------------------------------------------------
    # Selected goal
    # --------------------------------------------------------

    plt.scatter(
        selected_goal["x"],
        selected_goal["y"],
        s=220,
        marker="*",
        edgecolors="black",
        linewidths=1.5,
        zorder=9,
    )


plt.title(
    "MultiUUV-Link — Goal-Directed Frontier Pursuit"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nGoal-directed frontier pursuit validated."
)

# Combo 17 — Closed-Loop Autonomous Frontier Exploration

## Objective

The previous combos constructed and validated the complete frontier-planning chain:

```text
Private Memory
      ↓
Frontier Extraction
      ↓
Candidate Sampling
      ↓
Candidate Scoring
      ↓
Goal Selection
      ↓
Goal-Directed Motion
```

However, Combo 16 executed this process only once.

True autonomous exploration requires the UUV to repeatedly:

1. observe;
2. update private memory;
3. recompute its frontier;
4. generate new frontier candidates;
5. score those candidates;
6. select a new goal;
7. move toward that goal;
8. observe newly revealed territory;
9. replan.

This combo closes that loop.

There is still:

```text
NO communication
NO shared map
NO coordination
NO collision avoidance
NO task allocation
NO learning
```

The four UUVs operate as completely independent autonomous explorers.

This provides the principal **classical independent-autonomy baseline** for later swarm-coordination experiments.

---

# Closed-Loop Mission Model

For UUV $i$, planning cycle $k$ consists of:

$$
M_i^k
\rightarrow
F_i^k
\rightarrow
G_i^k
\rightarrow
g_i^{*,k}
\rightarrow
p_i^{k+1}
\rightarrow
M_i^{k+1}
$$

where:

- $M_i^k$ is current private exploration memory;
- $F_i^k$ is the current frontier;
- $G_i^k$ is the sampled frontier candidate set;
- $g_i^{*,k}$ is the selected frontier goal;
- $p_i^{k+1}$ is the state after goal pursuit.

The important distinction is that:

$$
F_i^{k+1}
\neq
F_i^k
$$

in general because new observations modify the private map.

Therefore the planner must regenerate frontiers rather than reusing stale goals.

---

# Autonomous Replanning Cycle

The local autonomous loop is:

```text
Current Private Memory
        ↓
Extract Frontier
        ↓
Sample Candidate Goals
        ↓
Evaluate Candidate Utility
        ↓
Select Best Goal
        ↓
Move Toward Goal
        ↓
Update Local Observation
        ↓
Update Exploration Memory
        ↓
Update Resource Memory
        ↓
REPLAN
```

This is the first persistent sense-plan-act loop in MultiUUV-Link.

---

# Planning Horizon

The initial validation will execute:

$$
K=8
$$

frontier-planning cycles.

Each selected goal may receive at most:

$$
T_g=20
$$

discrete movement actions.

Therefore the theoretical maximum number of actions per UUV is:

$$
K T_g
=
8\times20
=
160
$$

Actual action counts may be smaller because goal pursuit stops when:

$$
\|p_i-g_i\|_2\le r_g
$$

with:

$$
r_g=32\text{ px}
$$

---

# Frontier Recalculation

At every planning cycle:

$$
F_i^k
=
f(M_i^k)
$$

The frontier must therefore be generated from the **latest private memory**.

No frontier information from previous planning cycles is assumed to remain valid.

This prevents stale planning geometry.

---

# Candidate Regeneration

Candidate goals are likewise regenerated:

$$
G_i^k
=
h(F_i^k)
$$

using the spatial frontier sampling developed previously.

Thus candidate locations naturally migrate outward as exploration progresses.

---

# Classical Goal Utility

For each current candidate:

$$
g\in G_i^k
$$

the utility remains:

$$
J_i(g)
=
w_I\hat I_i(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i(g)
$$

with:

$$
w_I=1.0
$$

$$
w_D=0.35
$$

$$
w_R=0.25
$$

The selected goal is:

$$
g_i^{*,k}
=
\arg\max_{g\in G_i^k}J_i(g)
$$

Because private memory evolves, the same physical location may receive a different utility later or disappear from the frontier entirely.

---

# Information Gain

For planning cycle $k$, define:

$$
\Delta A_i^k
=
|M_i^{k+1}|
-
|M_i^k|
$$

This measures the amount of genuinely new world area acquired during that planning cycle.

Cumulative autonomous information gain is:

$$
A_{i,\text{gain}}
=
\sum_{k=1}^{K}
\Delta A_i^k
$$

---

# Resource Discovery

Resource memory continues to update during motion.

At every step:

$$
D_i(t+1)
=
D_i(t)
\cup
V_i(t)
$$

Therefore autonomous exploration may discover additional resources naturally.

No resource coordinates influence frontier scoring.

Resource discovery remains an outcome of exploration rather than privileged planning information.

---

# Termination Conditions

An agent stops autonomous frontier exploration if any of the following occurs:

### Planning budget exhausted

$$
k=K
$$

### No frontier exists

$$
|F_i|=0
$$

which may indicate effectively complete exploration.

### No valid sampled candidate exists

$$
|G_i|=0
$$

Otherwise the agent continues replanning.

---

# Independence Constraint

Each UUV independently computes:

$$
F_i,\quad G_i,\quad J_i,\quad g_i^*
$$

from its own memory.

No UUV knows another UUV's:

- explored map;
- selected goal;
- resource discoveries;
- trajectory.

Therefore duplicated exploration between UUVs is currently possible.

That inefficiency is intentional.

It provides the baseline that later communication-aware coordination must outperform.

---

# Evaluation Metrics

For every UUV, this combo records:

### Initial coverage

$$
C_i^{start}
$$

### Final coverage

$$
C_i^{final}
$$

### Coverage increase

$$
\Delta C_i
=
C_i^{final}
-
C_i^{start}
$$

### Autonomous movement actions

$$
N_{move,i}
$$

### Planning cycles completed

$$
N_{plan,i}
$$

### Resources known before and after

$$
|D_i^{start}|,
\qquad
|D_i^{final}|
$$

### Mean information gain per planning cycle

$$
\bar I_i
=
\frac{1}{N_{plan,i}}
\sum_k\Delta A_i^k
$$

---

# Expected Behavior

Unlike random walk, frontier exploration should generally produce sustained outward growth of private explored regions.

We expect:

$$
C_i^{final}
>
C_i^{start}
$$

for every UUV.

The trajectories should visually move from frontier to frontier instead of wandering repeatedly through the same local patch.

The exact coverage improvement is scenario-dependent and is **not predetermined**.

---

# Expected Output

The code cell should:

1. recompute frontiers every planning cycle;
2. regenerate spatial frontier candidates;
3. score candidates using the classical utility;
4. select one goal independently per UUV;
5. pursue that goal;
6. update exploration memory during motion;
7. update resource memory during motion;
8. replan after reaching each goal;
9. perform up to 8 planning cycles;
10. report:
    - starting coverage;
    - final coverage;
    - coverage gain;
    - planning cycles;
    - motion actions;
    - newly explored cells;
    - newly discovered resources;
11. plot each UUV's autonomous exploration trajectory.

A successful result establishes:

$$
\boxed{
\text{Persistent Autonomous Frontier Exploration}
}
$$

without any inter-agent cooperation.

---

In [ ]:
# ============================================================
# Combo 17 — Closed-Loop Autonomous Frontier Exploration
# ============================================================


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

AUTONOMOUS_PLANNING_CYCLES = 8
MAX_STEPS_PER_GOAL = 20


# ------------------------------------------------------------
# Snapshot starting state
# ------------------------------------------------------------

autonomy_start = {}

for agent in agents:

    autonomy_start[
        agent.uuv_id
    ] = {
        "coverage":
            agent.exploration_memory.coverage(),

        "observed_cells":
            agent.exploration_memory.observed_count(),

        "known_resources":
            len(agent.resource_memory),

        "trajectory_index":
            len(agent.trajectory) - 1,
    }


# ------------------------------------------------------------
# Logs
# ------------------------------------------------------------

autonomy_logs = {
    agent.uuv_id: []
    for agent in agents
}


# ------------------------------------------------------------
# Independent autonomous exploration
# ------------------------------------------------------------

for agent in agents:

    for planning_cycle in range(
        AUTONOMOUS_PLANNING_CYCLES
    ):

        # ----------------------------------------------------
        # 1. Recompute private frontier
        # ----------------------------------------------------

        frontier = extract_frontier_mask(
            agent.exploration_memory
        )

        frontier_pixels = int(
            frontier.sum()
        )

        if frontier_pixels == 0:
            break


        # ----------------------------------------------------
        # 2. Regenerate frontier candidates
        # ----------------------------------------------------

        candidates = sample_frontier_candidates(
            frontier_mask=frontier,
            reference_position=(
                agent.state.x,
                agent.state.y,
            ),
            min_spacing=
                FRONTIER_CANDIDATE_SPACING,
            max_candidates=
                MAX_FRONTIER_CANDIDATES,
        )

        if not candidates:
            break


        # ----------------------------------------------------
        # 3. Score candidates
        # ----------------------------------------------------

        scored_candidates = [
            score_frontier_candidate(
                agent,
                candidate,
            )
            for candidate in candidates
        ]


        # ----------------------------------------------------
        # 4. Select best goal
        # ----------------------------------------------------

        selected_goal = max(
            scored_candidates,
            key=lambda item:
                item["utility"],
        )

        goal_position = (
            selected_goal["x"],
            selected_goal["y"],
        )

        start_position = (
            agent.state.x,
            agent.state.y,
        )

        start_distance = (
            euclidean_distance(
                start_position,
                goal_position,
            )
        )

        observed_before = (
            agent
            .exploration_memory
            .observed_count()
        )

        resources_before = len(
            agent.resource_memory
        )


        # ----------------------------------------------------
        # 5. Pursue selected goal
        # ----------------------------------------------------

        steps_used = 0

        for _ in range(
            MAX_STEPS_PER_GOAL
        ):

            current_position = (
                agent.state.x,
                agent.state.y,
            )

            current_distance = (
                euclidean_distance(
                    current_position,
                    goal_position,
                )
            )

            if (
                current_distance
                <=
                GOAL_REACHED_RADIUS
            ):
                break


            action = choose_greedy_action(
                agent.state,
                goal_x=goal_position[0],
                goal_y=goal_position[1],
            )

            if action is None:
                break


            next_state = move_uuv_scaled(
                state=agent.state,
                action=action,
                step_size=ROLLOUT_STEP_SIZE,
            )


            # ------------------------------------------------
            # Guard against zero-motion boundary lock
            # ------------------------------------------------

            old_position = (
                agent.state.x,
                agent.state.y,
            )

            new_position = (
                next_state.x,
                next_state.y,
            )

            if new_position == old_position:
                break


            # ------------------------------------------------
            # Update state
            # ------------------------------------------------

            agent.state = next_state

            agent.trajectory.append(
                new_position
            )

            steps_used += 1


            # ------------------------------------------------
            # Observation / memory
            # ------------------------------------------------

            agent.exploration_memory.update_from_fov(
                agent.state
            )


            # ------------------------------------------------
            # Resource discovery
            # ------------------------------------------------

            visible_resources = (
                discover_visible_resources(
                    agent.state,
                    resources,
                )
            )

            agent.resource_memory.update(
                visible_resources
            )


        # ----------------------------------------------------
        # 6. Planning-cycle metrics
        # ----------------------------------------------------

        observed_after = (
            agent
            .exploration_memory
            .observed_count()
        )

        resources_after = len(
            agent.resource_memory
        )

        final_distance = (
            euclidean_distance(
                (
                    agent.state.x,
                    agent.state.y,
                ),
                goal_position,
            )
        )

        new_cells = (
            observed_after
            -
            observed_before
        )

        new_resources = (
            resources_after
            -
            resources_before
        )


        autonomy_logs[
            agent.uuv_id
        ].append(
            {
                "planning_cycle":
                    planning_cycle,

                "goal_x":
                    goal_position[0],

                "goal_y":
                    goal_position[1],

                "utility":
                    selected_goal["utility"],

                "start_distance":
                    start_distance,

                "final_distance":
                    final_distance,

                "steps_used":
                    steps_used,

                "new_cells":
                    new_cells,

                "new_resources":
                    new_resources,

                "coverage":
                    agent
                    .exploration_memory
                    .coverage(),
            }
        )


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

autonomy_reports = []

for agent in agents:

    start = autonomy_start[
        agent.uuv_id
    ]

    logs = autonomy_logs[
        agent.uuv_id
    ]

    final_coverage = (
        agent
        .exploration_memory
        .coverage()
    )

    final_observed = (
        agent
        .exploration_memory
        .observed_count()
    )

    final_resources = len(
        agent.resource_memory
    )

    planning_cycles = len(
        logs
    )

    motion_actions = sum(
        log["steps_used"]
        for log in logs
    )

    gained_cells = (
        final_observed
        -
        start["observed_cells"]
    )

    coverage_gain = (
        final_coverage
        -
        start["coverage"]
    )

    resource_gain = (
        final_resources
        -
        start["known_resources"]
    )

    if planning_cycles > 0:

        mean_gain_per_cycle = (
            gained_cells
            /
            planning_cycles
        )

    else:

        mean_gain_per_cycle = 0.0


    autonomy_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "cycles":
                planning_cycles,

            "motion_actions":
                motion_actions,

            "start_coverage_pct":
                start["coverage"]
                * 100,

            "final_coverage_pct":
                final_coverage
                * 100,

            "coverage_gain_pct":
                coverage_gain
                * 100,

            "new_cells":
                gained_cells,

            "mean_cells_cycle":
                mean_gain_per_cycle,

            "resources_before":
                start[
                    "known_resources"
                ],

            "resources_after":
                final_resources,

            "new_resources":
                resource_gain,
        }
    )


autonomy_df = pd.DataFrame(
    autonomy_reports
)


# ------------------------------------------------------------
# Validate coverage progress
# ------------------------------------------------------------

for report in autonomy_reports:

    if (
        report["cycles"] > 0
        and
        report["new_cells"] <= 0
    ):
        raise RuntimeError(
            f"UUV-{report['uuv_id']} "
            "completed planning cycles but "
            "acquired no new territory."
        )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 92
)

print(
    "MultiUUV-Link — Closed-Loop "
    "Independent Frontier Exploration"
)

print(
    "=" * 92
)

print(
    f"\nMaximum planning cycles : "
    f"{AUTONOMOUS_PLANNING_CYCLES}"
)

print(
    f"Maximum steps / goal   : "
    f"{MAX_STEPS_PER_GOAL}"
)

print(
    f"Motion step            : "
    f"{ROLLOUT_STEP_SIZE} px"
)

print(
    "\nAutonomous exploration summary:"
)

print(
    autonomy_df.to_string(
        index=False,
        formatters={
            "start_coverage_pct":
                lambda x: f"{x:.4f}",

            "final_coverage_pct":
                lambda x: f"{x:.4f}",

            "coverage_gain_pct":
                lambda x: f"{x:.4f}",

            "mean_cells_cycle":
                lambda x: f"{x:.2f}",
        },
    )
)


# ------------------------------------------------------------
# Planning-cycle details
# ------------------------------------------------------------

for agent in agents:

    print(
        f"\nUUV-{agent.uuv_id} planning cycles:"
    )

    logs = autonomy_logs[
        agent.uuv_id
    ]

    if not logs:

        print(
            "  No frontier planning cycles."
        )

        continue


    log_df = pd.DataFrame(
        logs
    )

    print(
        log_df.to_string(
            index=False,
            formatters={
                "utility":
                    lambda x: f"{x:.4f}",

                "start_distance":
                    lambda x: f"{x:.2f}",

                "final_distance":
                    lambda x: f"{x:.2f}",

                "coverage":
                    lambda x:
                        f"{100*x:.4f}%",
            },
        )
    )


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_index = (
        autonomy_start[
            agent.uuv_id
        ][
            "trajectory_index"
        ]
    )

    autonomous_path = np.asarray(
        agent.trajectory[
            start_index:
        ]
    )


    # --------------------------------------------------------
    # Autonomous path
    # --------------------------------------------------------

    if len(
        autonomous_path
    ) > 1:

        plt.plot(
            autonomous_path[:, 0],
            autonomous_path[:, 1],
            linewidth=2.5,
            label=f"UUV-{agent.uuv_id}",
            zorder=5,
        )


    # --------------------------------------------------------
    # Start of autonomous phase
    # --------------------------------------------------------

    plt.scatter(
        autonomous_path[0, 0],
        autonomous_path[0, 1],
        s=90,
        marker="o",
        zorder=6,
    )


    # --------------------------------------------------------
    # Final position
    # --------------------------------------------------------

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=130,
        marker="x",
        linewidths=2.5,
        zorder=7,
    )


    # --------------------------------------------------------
    # Planning goals
    # --------------------------------------------------------

    for log in autonomy_logs[
        agent.uuv_id
    ]:

        plt.scatter(
            log["goal_x"],
            log["goal_y"],
            s=90,
            marker="*",
            zorder=6,
        )


plt.title(
    "MultiUUV-Link — Closed-Loop Independent "
    "Frontier Exploration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nClosed-loop independent frontier "
    "exploration validated."
)

# Combo 18 — Inter-UUV Collision-Risk Detection

## Objective

The four UUVs can now autonomously explore using independent frontier-based planning.

However, each agent currently ignores the physical positions of the other UUVs.

Two independently planned trajectories may therefore approach one another arbitrarily closely.

Before modifying motion through collision avoidance, this combo introduces a dedicated **safety-monitoring layer** that measures inter-UUV separation and identifies unsafe proximity events.

This combo performs:

```text
agent positions
      ↓
pairwise distances
      ↓
safety threshold
      ↓
collision-risk state
```

It does **not yet modify any UUV action**.

There is still:

```text
NO avoidance action
NO repulsive controller
NO priority resolution
NO communication
```

The goal is to validate the safety measurement independently before it is allowed to override mission intent.

---

# Pairwise Separation

For UUVs $i$ and $j$ with positions:

$$
p_i(t)=
\begin{bmatrix}
x_i(t)\\
y_i(t)
\end{bmatrix}
$$

and:

$$
p_j(t)=
\begin{bmatrix}
x_j(t)\\
y_j(t)
\end{bmatrix}
$$

the Euclidean separation is:

$$
d_{ij}(t)
=
\|p_i(t)-p_j(t)\|_2
$$

or:

$$
d_{ij}(t)
=
\sqrt{
(x_i-x_j)^2
+
(y_i-y_j)^2
}
$$

For:

$$
N=4
$$

agents, the number of unique UUV pairs is:

$$
\binom{4}{2}=6
$$

Therefore each synchronized safety evaluation contains six pairwise distances.

---

# Safety Threshold

Define a minimum desired separation:

$$
d_{\text{safe}}
$$

The current perception FOV is:

$$
128\times128\text{ px}
$$

and the operational movement step is:

$$
32\text{ px}
$$

For the initial lightweight safety abstraction, define:

$$
d_{\text{safe}}=128\text{ px}
$$

This equals one FOV width.

The value is not intended to represent a validated physical UUV separation distance.

It is an algorithmic safety radius appropriate to the current image-space abstraction.

---

# Safety State

For pair $(i,j)$:

$$
Risk_{ij}(t)
=
\begin{cases}
1, & d_{ij}(t)<d_{\text{safe}}\\
0, & d_{ij}(t)\ge d_{\text{safe}}
\end{cases}
$$

A distance exactly equal to the threshold is considered safe:

$$
d_{ij}=d_{\text{safe}}
\Rightarrow
Risk_{ij}=0
$$

---

# Minimum Swarm Separation

At timestep $t$, define:

$$
d_{\min}(t)
=
\min_{i<j}
d_{ij}(t)
$$

Across an entire rollout:

$$
d_{\min}^{mission}
=
\min_t d_{\min}(t)
$$

This quantity will later become one of the project's principal safety metrics.

---

# Why Detection Is Separated from Avoidance

Mission planning determines:

> Where does this UUV want to go?

Safety determines:

> Is the intended motion acceptable given nearby agents?

These should remain distinct layers.

The future control architecture will therefore be:

```text
Frontier Planner
      ↓
Desired Action
      ↓
Safety Layer
      ↓
Executed Action
```

rather than embedding safety heuristics directly inside frontier utility.

This separation lets us independently evaluate:

- planning performance;
- safety interventions;
- mission cost introduced by avoidance.

---

# Historical Trajectory Analysis

The current notebook already contains trajectory histories from:

- random walk;
- one-shot goal pursuit;
- closed-loop frontier exploration.

Because those trajectories were generated without collision avoidance, they provide useful historical safety data.

This combo will inspect the existing trajectories to determine:

- current pairwise distances;
- historical synchronized minimum separation;
- number of safety-threshold violations;
- which UUV pair came closest.

---

# Synchronization Consideration

Trajectory histories may not have identical lengths because goal-directed planning can require different numbers of movements per UUV.

To compare historical trajectories at shared discrete indices, safety analysis will use the common synchronized prefix:

$$
T_{\text{common}}
=
\min_i |H_i|
$$

For each:

$$
t\in\{0,\ldots,T_{\text{common}}-1\}
$$

the positions:

$$
H_0(t),H_1(t),H_2(t),H_3(t)
$$

are compared.

This is an analysis approximation for the current sequential notebook prototype.

A later mission orchestrator will evolve all UUVs synchronously.

---

# Validation Test

Because naturally generated trajectories may contain no unsafe encounter, this combo also includes a deterministic synthetic safety test.

Two temporary positions will be created with:

$$
d_{ij}<d_{\text{safe}}
$$

The safety detector must classify the pair as unsafe.

A second pair with:

$$
d_{ij}>d_{\text{safe}}
$$

must be classified as safe.

This verifies the logic independently of random mission geometry.

---

# Expected Output

The code cell should:

1. define the safety-distance threshold;
2. compute pairwise UUV distances;
3. identify unsafe pairs;
4. report current pairwise distances;
5. inspect synchronized historical trajectories;
6. compute:
   - historical minimum separation;
   - closest UUV pair;
   - timestep of closest approach;
   - number of threshold violations;
7. validate the detector with synthetic safe and unsafe cases;
8. plot historical minimum separation versus trajectory index;
9. draw the safety threshold on the plot.

A successful result establishes:

$$
\boxed{
\text{Multi-UUV State}
\rightarrow
\text{Quantified Collision Risk}
}
$$

without yet modifying mission behavior.

---

In [ ]:
# ============================================================
# Combo 18 — Inter-UUV Collision-Risk Detection
# ============================================================

from itertools import combinations


# ------------------------------------------------------------
# Safety configuration
# ------------------------------------------------------------

SAFE_SEPARATION = 128.0


# ------------------------------------------------------------
# Pairwise safety evaluation
# ------------------------------------------------------------

def evaluate_pairwise_safety(
    positions,
    safe_separation=SAFE_SEPARATION,
):
    """
    Parameters
    ----------
    positions : dict
        {uuv_id: (x, y)}

    Returns
    -------
    list of dict
        Pairwise distance and safety state.
    """

    results = []

    uuv_ids = sorted(
        positions.keys()
    )

    for uuv_a, uuv_b in combinations(
        uuv_ids,
        2,
    ):

        p_a = positions[uuv_a]
        p_b = positions[uuv_b]

        distance = euclidean_distance(
            p_a,
            p_b,
        )

        unsafe = (
            distance
            <
            safe_separation
        )

        results.append(
            {
                "uuv_a":
                    uuv_a,

                "uuv_b":
                    uuv_b,

                "distance":
                    float(distance),

                "unsafe":
                    bool(unsafe),
            }
        )

    return results


# ------------------------------------------------------------
# Current swarm state
# ------------------------------------------------------------

current_positions = {
    agent.uuv_id: (
        agent.state.x,
        agent.state.y,
    )
    for agent in agents
}

current_safety = (
    evaluate_pairwise_safety(
        current_positions
    )
)


# ------------------------------------------------------------
# Current report
# ------------------------------------------------------------

current_safety_df = pd.DataFrame(
    current_safety
)

print(
    "=" * 82
)

print(
    "MultiUUV-Link — Inter-UUV Collision-Risk Detection"
)

print(
    "=" * 82
)

print(
    f"\nSafety threshold : "
    f"{SAFE_SEPARATION:.2f} px"
)

print(
    "\nCurrent pairwise distances:"
)

print(
    current_safety_df.to_string(
        index=False,
        formatters={
            "distance":
                lambda x: f"{x:.2f}",
        },
    )
)


# ------------------------------------------------------------
# Current minimum separation
# ------------------------------------------------------------

current_closest = min(
    current_safety,
    key=lambda item:
        item["distance"],
)

print(
    f"\nCurrent minimum separation : "
    f"{current_closest['distance']:.2f} px"
)

print(
    f"Current closest pair       : "
    f"UUV-{current_closest['uuv_a']} / "
    f"UUV-{current_closest['uuv_b']}"
)

print(
    f"Currently unsafe           : "
    f"{current_closest['unsafe']}"
)


# ------------------------------------------------------------
# Historical synchronized trajectory analysis
# ------------------------------------------------------------

common_length = min(
    len(agent.trajectory)
    for agent in agents
)

historical_minimums = []

violation_events = []

mission_minimum_distance = np.inf
mission_closest_pair = None
mission_closest_index = None


for trajectory_index in range(
    common_length
):

    positions_at_t = {
        agent.uuv_id:
            agent.trajectory[
                trajectory_index
            ]
        for agent in agents
    }

    pair_results = (
        evaluate_pairwise_safety(
            positions_at_t
        )
    )

    closest_at_t = min(
        pair_results,
        key=lambda item:
            item["distance"],
    )

    historical_minimums.append(
        closest_at_t["distance"]
    )


    if (
        closest_at_t["distance"]
        <
        mission_minimum_distance
    ):

        mission_minimum_distance = (
            closest_at_t["distance"]
        )

        mission_closest_pair = (
            closest_at_t["uuv_a"],
            closest_at_t["uuv_b"],
        )

        mission_closest_index = (
            trajectory_index
        )


    for result in pair_results:

        if result["unsafe"]:

            violation_events.append(
                {
                    "trajectory_index":
                        trajectory_index,

                    "uuv_a":
                        result["uuv_a"],

                    "uuv_b":
                        result["uuv_b"],

                    "distance":
                        result["distance"],
                }
            )


# ------------------------------------------------------------
# Historical report
# ------------------------------------------------------------

print(
    "\nHistorical synchronized analysis:"
)

print(
    f"  Common trajectory length : "
    f"{common_length}"
)

print(
    f"  Mission min separation   : "
    f"{mission_minimum_distance:.2f} px"
)

print(
    f"  Closest pair             : "
    f"UUV-{mission_closest_pair[0]} / "
    f"UUV-{mission_closest_pair[1]}"
)

print(
    f"  Closest trajectory index : "
    f"{mission_closest_index}"
)

print(
    f"  Safety violations        : "
    f"{len(violation_events)}"
)


# ------------------------------------------------------------
# Show first few violations, if any
# ------------------------------------------------------------

if violation_events:

    violation_df = pd.DataFrame(
        violation_events
    )

    print(
        "\nFirst safety violations:"
    )

    print(
        violation_df.head(
            10
        ).to_string(
            index=False,
            formatters={
                "distance":
                    lambda x: f"{x:.2f}",
            },
        )
    )

else:

    print(
        "\nNo historical separation "
        "violations detected."
    )


# ------------------------------------------------------------
# Deterministic detector validation
# ------------------------------------------------------------

synthetic_unsafe_positions = {
    0: (100, 100),
    1: (150, 100),
}

synthetic_safe_positions = {
    0: (100, 100),
    1: (300, 100),
}


synthetic_unsafe_result = (
    evaluate_pairwise_safety(
        synthetic_unsafe_positions
    )[0]
)

synthetic_safe_result = (
    evaluate_pairwise_safety(
        synthetic_safe_positions
    )[0]
)


unsafe_test_passed = (
    synthetic_unsafe_result[
        "unsafe"
    ]
    is True
)

safe_test_passed = (
    synthetic_safe_result[
        "unsafe"
    ]
    is False
)


print(
    "\nSynthetic validation:"
)

print(
    f"  50 px pair classified unsafe : "
    f"{unsafe_test_passed}"
)

print(
    f"  200 px pair classified safe  : "
    f"{safe_test_passed}"
)


if not unsafe_test_passed:
    raise RuntimeError(
        "Unsafe-pair detector test failed."
    )

if not safe_test_passed:
    raise RuntimeError(
        "Safe-pair detector test failed."
    )


# ------------------------------------------------------------
# Distance-history visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    np.arange(
        common_length
    ),
    historical_minimums,
    linewidth=2,
    label="Minimum inter-UUV separation",
)

plt.axhline(
    SAFE_SEPARATION,
    linestyle="--",
    linewidth=2,
    label="Safety threshold",
)

plt.xlabel(
    "Synchronized trajectory index"
)

plt.ylabel(
    "Minimum pairwise distance [px]"
)

plt.title(
    "MultiUUV-Link — Historical Swarm Separation"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nInter-UUV collision-risk "
    "detection validated."
)

# Combo 19 — Predictive Multi-UUV Safety Shield

## Objective

Combo 18 established a collision-risk detector:

$$
d_{ij}<d_{\text{safe}}
\Rightarrow
\text{unsafe}
$$

The detector can identify a problem but cannot yet prevent one.

This combo introduces a dedicated **predictive safety shield** between mission planning and executed motion.

The architecture becomes:

```text
Frontier Planner
      ↓
Desired Actions
      ↓
Predict Next Positions
      ↓
Safety Shield
      ↓
Safe Joint Action Set
      ↓
Executed Actions
```

The safety layer is deliberately separated from the frontier planner.

The planner decides:

> Where does each UUV want to move?

The safety shield decides:

> Can those motions be executed simultaneously without violating minimum separation?

This combo validates the shield itself.

It does **not yet run a complete safety-aware exploration mission**.

---

# Desired Versus Executed Action

For UUV $i$, let:

$$
a_i^{des}(t)
$$

be the action requested by the exploration planner.

The safety shield may produce:

$$
a_i^{exec}(t)
$$

where:

$$
a_i^{exec}(t)
\neq
a_i^{des}(t)
$$

if executing the desired joint action would create an unsafe predicted configuration.

Thus:

$$
\boxed{
\text{Mission Intent}
\neq
\text{Safety-Constrained Execution}
}
$$

when intervention is required.

---

# Predictive Safety

Current separation alone is insufficient.

Two UUVs may currently satisfy:

$$
d_{ij}(t)>d_{\text{safe}}
$$

while their desired actions would produce:

$$
d_{ij}(t+1)<d_{\text{safe}}
$$

Therefore safety is evaluated using predicted next positions.

For joint action vector:

$$
\mathbf{a}
=
(a_0,a_1,\ldots,a_{N-1})
$$

the predicted next state is:

$$
p_i^+
=
f(p_i,a_i)
$$

The joint action is safe only if:

$$
\|p_i^+-p_j^+\|_2
\ge
d_{\text{safe}}
$$

for every pair:

$$
i\neq j
$$

---

# Safety-Only HOLD Action

The normal autonomy action space remains:

$$
\{UP,RIGHT,DOWN,LEFT\}
$$

However, the safety layer is allowed to introduce:

```text
HOLD
```

meaning:

$$
p_i(t+1)=p_i(t)
$$

This is not a new exploration action.

It is a safety override available only when necessary.

Therefore the safety candidate set is:

$$
A_{\text{safety}}
=
\{
UP,RIGHT,DOWN,LEFT,HOLD
\}
$$

---

# Joint Action Search

There are:

$$
5^4=625
$$

possible joint safety actions for four UUVs.

This is tiny computationally.

Therefore rather than using a heuristic pair-by-pair rule, the prototype can evaluate the complete joint action space.

For every joint action:

$$
\mathbf{a}
\in A_{\text{safety}}^4
$$

we:

1. predict all four next positions;
2. reject the action if any pair violates:

$$
d_{ij}<d_{\text{safe}}
$$

3. assign a cost measuring deviation from mission intent.

---

# Intervention Cost

The preferred solution is the one that changes as few planner commands as possible.

Define:

$$
C(\mathbf{a})
=
\sum_i
\mathbf{1}
\left[
a_i\neq a_i^{des}
\right]
$$

where:

$$
\mathbf{1}[\cdot]
$$

is the indicator function.

Thus:

- keeping a desired action costs 0;
- overriding one agent costs 1;
- overriding two agents costs 2.

Among equally low-intervention solutions, we prefer the joint action with the largest predicted minimum separation.

Therefore selection is lexicographic:

1. minimize number of overrides;
2. maximize predicted minimum pairwise distance.

---

# Desired Action Priority

If the desired joint action is already safe:

$$
\min_{i<j}d_{ij}^+
\ge
d_{\text{safe}}
$$

then:

$$
a_i^{exec}
=
a_i^{des}
$$

for every UUV.

The shield should therefore be transparent during normal operation.

Safety intervention occurs only when necessary.

---

# Synthetic Conflict Validation

Natural current positions may not produce a collision-risk event.

Therefore the shield will be tested using a deterministic synthetic configuration.

Two UUVs will be placed on opposite sides of one another and commanded to move toward each other.

The desired action pair should create a predicted separation below:

$$
128\text{ px}
$$

The shield must modify at least one requested action and produce:

$$
d_{ij}^{exec}
\ge128\text{ px}
$$

This proves prevention rather than detection alone.

---

# Current-Swarm Validation

The shield will also evaluate one set of current desired actions from the actual four-UUV state.

Fresh frontier goals will be computed from each UUV's current private memory.

Each planner will request one greedy action toward its selected frontier.

The safety shield will then determine whether those four desired actions may safely execute together.

This current-swarm evaluation is diagnostic only.

Agent state will **not** be mutated in this combo.

---

# Safety Metrics

The shield will report:

### Desired predicted minimum separation

$$
d_{\min}^{des}
$$

### Executed predicted minimum separation

$$
d_{\min}^{exec}
$$

### Intervention count

$$
N_{\text{override}}
=
\sum_i
\mathbf{1}
[
a_i^{exec}\neq a_i^{des}
]
$$

A correct shield must guarantee:

$$
d_{\min}^{exec}
\ge d_{\text{safe}}
$$

whenever a feasible joint action exists.

---

# Expected Output

The code cell should:

1. introduce a safety-only HOLD action;
2. predict next positions for arbitrary joint actions;
3. enumerate all joint action combinations;
4. reject unsafe combinations;
5. select the feasible combination requiring the fewest overrides;
6. break ties using maximum predicted minimum separation;
7. test a deterministic collision scenario;
8. verify that unsafe desired motion becomes safe;
9. compute fresh desired frontier actions for the real four-UUV state;
10. evaluate those actions through the safety shield without mutating the agents;
11. print:
    - desired actions;
    - executed actions;
    - desired minimum separation;
    - executed minimum separation;
    - number of interventions.

A successful result establishes:

$$
\boxed{
\text{Desired Multi-UUV Motion}
\rightarrow
\text{Predictive Safety Shield}
\rightarrow
\text{Safe Joint Motion}
}
$$

---

In [ ]:
# ============================================================
# Combo 19 — Predictive Multi-UUV Safety Shield
# ============================================================

from itertools import product, combinations


# ------------------------------------------------------------
# Safety-only action
# ------------------------------------------------------------

ACTION_HOLD = 4

SAFETY_ACTION_NAMES = {
    ACTION_UP: "UP",
    ACTION_RIGHT: "RIGHT",
    ACTION_DOWN: "DOWN",
    ACTION_LEFT: "LEFT",
    ACTION_HOLD: "HOLD",
}

SAFETY_ACTION_SET = (
    ACTION_UP,
    ACTION_RIGHT,
    ACTION_DOWN,
    ACTION_LEFT,
    ACTION_HOLD,
)


# ------------------------------------------------------------
# Predict one next state
# ------------------------------------------------------------

def predict_safety_state(
    state: UUVState,
    action: int,
    step_size: int = ROLLOUT_STEP_SIZE,
):
    if action == ACTION_HOLD:
        return UUVState(
            uuv_id=state.uuv_id,
            x=state.x,
            y=state.y,
        )

    return move_uuv_scaled(
        state=state,
        action=action,
        step_size=step_size,
    )


# ------------------------------------------------------------
# Predict complete joint state
# ------------------------------------------------------------

def predict_joint_positions(
    states,
    joint_actions,
    step_size=ROLLOUT_STEP_SIZE,
):
    predicted = {}

    for state, action in zip(
        states,
        joint_actions,
    ):
        next_state = predict_safety_state(
            state,
            action,
            step_size=step_size,
        )

        predicted[state.uuv_id] = (
            next_state.x,
            next_state.y,
        )

    return predicted


# ------------------------------------------------------------
# Minimum pairwise separation
# ------------------------------------------------------------

def minimum_pairwise_separation(
    positions,
):
    distances = []

    for uuv_a, uuv_b in combinations(
        sorted(positions.keys()),
        2,
    ):
        distance = euclidean_distance(
            positions[uuv_a],
            positions[uuv_b],
        )

        distances.append(
            (
                distance,
                uuv_a,
                uuv_b,
            )
        )

    if not distances:
        return np.inf, None

    return min(
        distances,
        key=lambda item:
            item[0],
    )


# ------------------------------------------------------------
# Predictive joint safety shield
# ------------------------------------------------------------

def safety_shield(
    states,
    desired_actions,
    safe_separation=SAFE_SEPARATION,
    step_size=ROLLOUT_STEP_SIZE,
):
    """
    Exhaustively search the 5^N safety action combinations.

    Objective:
        1. minimize number of planner overrides
        2. maximize predicted minimum pairwise separation
    """

    if len(states) != len(desired_actions):
        raise ValueError(
            "states and desired_actions must have equal length."
        )

    desired_actions = tuple(
        desired_actions
    )


    # --------------------------------------------------------
    # Evaluate desired motion first
    # --------------------------------------------------------

    desired_positions = predict_joint_positions(
        states,
        desired_actions,
        step_size=step_size,
    )

    (
        desired_min_distance,
        desired_closest_a,
        desired_closest_b,
    ) = minimum_pairwise_separation(
        desired_positions
    )


    # --------------------------------------------------------
    # Exhaustive feasible-action search
    # --------------------------------------------------------

    best_solution = None

    for joint_actions in product(
        SAFETY_ACTION_SET,
        repeat=len(states),
    ):

        predicted_positions = (
            predict_joint_positions(
                states,
                joint_actions,
                step_size=step_size,
            )
        )

        (
            minimum_distance,
            closest_a,
            closest_b,
        ) = minimum_pairwise_separation(
            predicted_positions
        )


        # ----------------------------------------------------
        # Reject unsafe joint actions
        # ----------------------------------------------------

        if (
            minimum_distance
            <
            safe_separation
        ):
            continue


        # ----------------------------------------------------
        # Mission-intent deviation
        # ----------------------------------------------------

        override_count = sum(
            executed != desired
            for executed, desired in zip(
                joint_actions,
                desired_actions,
            )
        )


        candidate_solution = {
            "executed_actions":
                joint_actions,

            "predicted_positions":
                predicted_positions,

            "minimum_distance":
                float(
                    minimum_distance
                ),

            "closest_pair":
                (
                    closest_a,
                    closest_b,
                ),

            "override_count":
                int(
                    override_count
                ),
        }


        # ----------------------------------------------------
        # Lexicographic objective
        # ----------------------------------------------------

        if best_solution is None:
            best_solution = candidate_solution
            continue


        if (
            candidate_solution[
                "override_count"
            ]
            <
            best_solution[
                "override_count"
            ]
        ):
            best_solution = candidate_solution

        elif (
            candidate_solution[
                "override_count"
            ]
            ==
            best_solution[
                "override_count"
            ]
            and
            candidate_solution[
                "minimum_distance"
            ]
            >
            best_solution[
                "minimum_distance"
            ]
        ):
            best_solution = candidate_solution


    if best_solution is None:
        raise RuntimeError(
            "No safe joint action exists for the current state."
        )


    best_solution[
        "desired_actions"
    ] = desired_actions

    best_solution[
        "desired_positions"
    ] = desired_positions

    best_solution[
        "desired_minimum_distance"
    ] = float(
        desired_min_distance
    )

    best_solution[
        "desired_closest_pair"
    ] = (
        desired_closest_a,
        desired_closest_b,
    )

    return best_solution


# ============================================================
# Synthetic conflict validation
# ============================================================

synthetic_states = [
    UUVState(
        uuv_id=0,
        x=400,
        y=500,
    ),
    UUVState(
        uuv_id=1,
        x=560,
        y=500,
    ),
]

# Both move toward each other.
synthetic_desired_actions = (
    ACTION_RIGHT,
    ACTION_LEFT,
)

synthetic_solution = safety_shield(
    states=synthetic_states,
    desired_actions=synthetic_desired_actions,
    safe_separation=SAFE_SEPARATION,
    step_size=ROLLOUT_STEP_SIZE,
)


print(
    "=" * 88
)

print(
    "MultiUUV-Link — Predictive Multi-UUV Safety Shield"
)

print(
    "=" * 88
)

print(
    "\nSynthetic head-on conflict test:"
)

print(
    "  Desired actions       : "
    +
    str(
        [
            SAFETY_ACTION_NAMES[a]
            for a in synthetic_desired_actions
        ]
    )
)

print(
    f"  Desired min distance  : "
    f"{synthetic_solution['desired_minimum_distance']:.2f} px"
)

print(
    "  Executed actions      : "
    +
    str(
        [
            SAFETY_ACTION_NAMES[a]
            for a in synthetic_solution[
                "executed_actions"
            ]
        ]
    )
)

print(
    f"  Executed min distance : "
    f"{synthetic_solution['minimum_distance']:.2f} px"
)

print(
    f"  Overrides             : "
    f"{synthetic_solution['override_count']}"
)


# ------------------------------------------------------------
# Synthetic assertions
# ------------------------------------------------------------

if not (
    synthetic_solution[
        "desired_minimum_distance"
    ]
    <
    SAFE_SEPARATION
):
    raise RuntimeError(
        "Synthetic desired actions were expected to be unsafe."
    )

if not (
    synthetic_solution[
        "minimum_distance"
    ]
    >=
    SAFE_SEPARATION
):
    raise RuntimeError(
        "Safety shield failed synthetic conflict test."
    )

if (
    synthetic_solution[
        "override_count"
    ]
    < 1
):
    raise RuntimeError(
        "Unsafe synthetic motion required no safety override."
    )


# ============================================================
# Fresh planner intent for current swarm
# ============================================================

current_states = [
    agent.state
    for agent in agents
]

current_desired_actions = []

current_goal_records = []


for agent in agents:

    # --------------------------------------------------------
    # Fresh frontier
    # --------------------------------------------------------

    frontier = extract_frontier_mask(
        agent.exploration_memory
    )


    # --------------------------------------------------------
    # Fresh candidate set
    # --------------------------------------------------------

    candidates = sample_frontier_candidates(
        frontier_mask=frontier,
        reference_position=(
            agent.state.x,
            agent.state.y,
        ),
        min_spacing=
            FRONTIER_CANDIDATE_SPACING,
        max_candidates=
            MAX_FRONTIER_CANDIDATES,
    )

    if not candidates:
        desired_action = ACTION_HOLD

        current_desired_actions.append(
            desired_action
        )

        current_goal_records.append(
            {
                "uuv_id":
                    agent.uuv_id,

                "goal_x":
                    None,

                "goal_y":
                    None,

                "desired_action":
                    desired_action,
            }
        )

        continue


    # --------------------------------------------------------
    # Classical candidate scoring
    # --------------------------------------------------------

    scored = [
        score_frontier_candidate(
            agent,
            candidate,
        )
        for candidate in candidates
    ]

    selected = max(
        scored,
        key=lambda item:
            item["utility"],
    )


    # --------------------------------------------------------
    # Desired planner action
    # --------------------------------------------------------

    desired_action = choose_greedy_action(
        agent.state,
        goal_x=selected["x"],
        goal_y=selected["y"],
    )

    if desired_action is None:
        desired_action = ACTION_HOLD


    current_desired_actions.append(
        desired_action
    )

    current_goal_records.append(
        {
            "uuv_id":
                agent.uuv_id,

            "goal_x":
                selected["x"],

            "goal_y":
                selected["y"],

            "desired_action":
                desired_action,
        }
    )


# ------------------------------------------------------------
# Shield current four-UUV intent
# ------------------------------------------------------------

current_solution = safety_shield(
    states=current_states,
    desired_actions=
        current_desired_actions,
    safe_separation=
        SAFE_SEPARATION,
    step_size=
        ROLLOUT_STEP_SIZE,
)


# ------------------------------------------------------------
# Current-swarm report
# ------------------------------------------------------------

print(
    "\nCurrent four-UUV planner intent:"
)

for record, executed_action in zip(
    current_goal_records,
    current_solution[
        "executed_actions"
    ],
):

    goal_text = (
        "None"
        if record["goal_x"] is None
        else
        f"({record['goal_x']}, {record['goal_y']})"
    )

    desired_name = (
        SAFETY_ACTION_NAMES[
            record[
                "desired_action"
            ]
        ]
    )

    executed_name = (
        SAFETY_ACTION_NAMES[
            executed_action
        ]
    )

    print(
        f"  UUV-{record['uuv_id']}: "
        f"goal={goal_text} | "
        f"desired={desired_name:<5} | "
        f"executed={executed_name:<5}"
    )


print(
    f"\nDesired predicted min separation : "
    f"{current_solution['desired_minimum_distance']:.2f} px"
)

print(
    f"Safe predicted min separation    : "
    f"{current_solution['minimum_distance']:.2f} px"
)

print(
    f"Safety overrides                 : "
    f"{current_solution['override_count']}"
)


# ------------------------------------------------------------
# Final invariant
# ------------------------------------------------------------

if (
    current_solution[
        "minimum_distance"
    ]
    <
    SAFE_SEPARATION
):
    raise RuntimeError(
        "Current safety-shield solution is unsafe."
    )


print(
    "\nPredictive multi-UUV safety shield validated."
)

# Combo 20 — Safety-Aware Closed-Loop Autonomous Exploration

## Objective

Combo 17 demonstrated closed-loop autonomous frontier exploration.

Combo 19 demonstrated a predictive multi-UUV safety shield.

This combo integrates the two systems.

Each synchronized swarm timestep will now follow:

```text
Private Memory
      ↓
Frontier Planner
      ↓
Desired Action
      ↓
Predictive Safety Shield
      ↓
Executed Safe Joint Action
      ↓
State Update
      ↓
Observation
      ↓
Memory Update
      ↓
Repeat
```

This produces the first **safety-aware autonomous multi-UUV mission loop**.

There is still:

```text
NO communication
NO shared maps
NO coordinated task allocation
NO consensus
NO learning
```

The agents remain independent explorers, but their physical execution is constrained by a shared safety layer.

---

# Synchronized Mission Execution

Previous notebook stages evolved agents mostly independently.

For collision avoidance to be meaningful, the swarm must now evolve synchronously.

At global timestep $t$:

$$
S(t)=
\{
p_0(t),p_1(t),p_2(t),p_3(t)
\}
$$

Every UUV computes a desired action:

$$
a_i^{des}(t)
$$

from its current private planner state.

The complete desired joint action is:

$$
\mathbf{a}^{des}(t)
=
(
a_0^{des},
a_1^{des},
a_2^{des},
a_3^{des}
)
$$

The safety shield transforms it into:

$$
\mathbf{a}^{exec}(t)
$$

and all agents then execute their safe actions simultaneously.

---

# Local Planning State

Each UUV maintains its own current frontier goal:

$$
g_i(t)
$$

A new goal is generated when:

1. the UUV has no active goal;
2. the previous goal has been reached;
3. the active goal is no longer useful;
4. no previous goal exists.

The planner uses only:

$$
M_i(t)
$$

to produce:

$$
F_i(t)
$$

then candidates:

$$
G_i(t)
$$

and finally:

$$
g_i^*(t)
$$

using the classical frontier utility.

---

# Goal-Reached Condition

The existing condition remains:

$$
\|p_i-g_i\|_2
\le
r_g
$$

with:

$$
r_g=32\text{ px}
$$

When the goal is reached, it is cleared and a fresh goal is selected from updated private memory.

---

# Desired Action

If UUV $i$ has an active frontier goal:

$$
g_i
$$

then its planner requests:

$$
a_i^{des}
=
\pi_{greedy}
(
p_i,
g_i
)
$$

If no frontier candidate exists:

$$
a_i^{des}
=
HOLD
$$

Thus an agent with no remaining exploration target remains stationary.

---

# Predictive Safety Layer

The four desired actions are evaluated jointly.

For predicted next positions:

$$
p_i^+
=
f(
p_i,
a_i
)
$$

the selected executed joint action must satisfy:

$$
\|p_i^+-p_j^+\|_2
\ge
d_{\text{safe}}
$$

for every:

$$
i\neq j
$$

with:

$$
d_{\text{safe}}=128\text{ px}
$$

The shield minimizes intervention count:

$$
N_{\text{override}}
=
\sum_i
\mathbf{1}
[
a_i^{exec}\neq a_i^{des}
]
$$

before maximizing predicted minimum separation.

---

# Safety Override Semantics

If:

$$
a_i^{exec}
\neq
a_i^{des}
$$

the planner's goal is **not immediately discarded**.

The goal represents mission intent.

The safety layer merely modifies this timestep's execution.

At the next synchronized timestep, the planner again attempts to progress toward the same goal unless normal replanning conditions occur.

Thus:

```text
planner goal persists
+
safety may temporarily modify motion
```

This prevents one avoidance action from causing unnecessary goal churn.

---

# Private Observation Update

After executing the safe joint action, every agent independently updates:

$$
M_i(t+1)
=
M_i(t)
\cup
FOV_i(t+1)
$$

and:

$$
D_i(t+1)
=
D_i(t)
\cup
V_i(t+1)
$$

There is still no knowledge transfer between agents.

---

# Mission Horizon

The initial safety-aware validation uses:

$$
T=120
$$

synchronized swarm timesteps.

With four agents this creates up to:

$$
4\times120=480
$$

executed agent actions.

The number of planning decisions may be smaller because goals persist across multiple timesteps.

---

# Safety Metrics

The mission records:

### Desired-action safety violations

Number of timesteps where the planner's raw joint action would have produced:

$$
d_{\min}^{des}<d_{\text{safe}}
$$

### Safety interventions

Total number of individual agent actions overridden:

$$
N_{\text{override,total}}
$$

### Intervention timesteps

Number of global timesteps where:

$$
N_{\text{override}}(t)>0
$$

### Executed minimum separation

$$
d_{\min}^{mission}
=
\min_t
\min_{i<j}
d_{ij}^{exec}(t)
$$

The required safety invariant is:

$$
d_{\min}^{mission}
\ge
d_{\text{safe}}
$$

---

# Exploration Metrics

For every UUV, record:

- starting coverage;
- final coverage;
- newly explored cells;
- starting resource count;
- final resource count;
- movement distance;
- safety overrides.

This allows future comparison between:

```text
Independent Frontier Exploration
```

and:

```text
Safety-Aware Independent Frontier Exploration
```

---

# Information Integrity

The safety layer may inspect current UUV positions because collision avoidance fundamentally requires relative geometry.

However, it must not inspect:

- hidden resource locations;
- another UUV's exploration memory;
- another UUV's resource memory;
- simulator future state.

Therefore safety shares only the minimum physical-state information required for separation enforcement.

Mission knowledge remains private.

---

# Expected Output

The code cell should:

1. maintain one active frontier goal per UUV;
2. replan locally when a goal is absent or reached;
3. compute four desired actions synchronously;
4. apply the predictive safety shield;
5. execute the safe joint action;
6. update every UUV state simultaneously;
7. update private exploration and resource memories;
8. run for 120 synchronized timesteps;
9. record:
   - desired unsafe timesteps;
   - intervention timesteps;
   - total overridden actions;
   - executed mission minimum separation;
   - per-UUV coverage gain;
   - per-UUV resource gain;
10. verify:

$$
d_{\min}^{mission}\ge128\text{ px}
$$

11. visualize the resulting four safety-aware trajectories.

A successful result establishes:

$$
\boxed{
\text{Autonomous Exploration}
+
\text{Predictive Multi-UUV Safety}
}
$$

---

In [ ]:
# ============================================================
# Combo 20 — Safety-Aware Closed-Loop Autonomous Exploration
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SAFE_AUTONOMY_STEPS = 120


# ------------------------------------------------------------
# Fresh mission snapshots
# ------------------------------------------------------------

safe_start = {}

for agent in agents:
    safe_start[agent.uuv_id] = {
        "coverage": agent.exploration_memory.coverage(),
        "observed_cells": agent.exploration_memory.observed_count(),
        "known_resources": len(agent.resource_memory),
        "trajectory_index": len(agent.trajectory) - 1,
    }


# ------------------------------------------------------------
# Per-UUV active goals
# ------------------------------------------------------------

active_goals = {
    agent.uuv_id: None
    for agent in agents
}


# ------------------------------------------------------------
# Logs
# ------------------------------------------------------------

safety_override_counts = {
    agent.uuv_id: 0
    for agent in agents
}

desired_unsafe_timesteps = 0
intervention_timesteps = 0
mission_min_separation = np.inf

executed_min_history = []
desired_min_history = []


# ------------------------------------------------------------
# Helper: generate a fresh private frontier goal
# ------------------------------------------------------------

def generate_private_frontier_goal(agent: UUVAgent):
    frontier = extract_frontier_mask(
        agent.exploration_memory
    )

    if int(frontier.sum()) == 0:
        return None

    candidates = sample_frontier_candidates(
        frontier_mask=frontier,
        reference_position=(
            agent.state.x,
            agent.state.y,
        ),
        min_spacing=FRONTIER_CANDIDATE_SPACING,
        max_candidates=MAX_FRONTIER_CANDIDATES,
    )

    if not candidates:
        return None

    scored = [
        score_frontier_candidate(
            agent,
            candidate,
        )
        for candidate in candidates
    ]

    return max(
        scored,
        key=lambda item: item["utility"],
    )


# ------------------------------------------------------------
# Synchronized mission loop
# ------------------------------------------------------------

for global_step in range(SAFE_AUTONOMY_STEPS):

    # --------------------------------------------------------
    # 1. Refresh goals when required
    # --------------------------------------------------------

    for agent in agents:
        goal = active_goals[agent.uuv_id]

        if goal is not None:
            distance_to_goal = euclidean_distance(
                (
                    agent.state.x,
                    agent.state.y,
                ),
                (
                    goal["x"],
                    goal["y"],
                ),
            )

            if distance_to_goal <= GOAL_REACHED_RADIUS:
                active_goals[agent.uuv_id] = None

        if active_goals[agent.uuv_id] is None:
            active_goals[agent.uuv_id] = (
                generate_private_frontier_goal(agent)
            )


    # --------------------------------------------------------
    # 2. Planner desired actions
    # --------------------------------------------------------

    desired_actions = []

    for agent in agents:
        goal = active_goals[agent.uuv_id]

        if goal is None:
            desired_action = ACTION_HOLD

        else:
            desired_action = choose_greedy_action(
                agent.state,
                goal_x=goal["x"],
                goal_y=goal["y"],
            )

            if desired_action is None:
                desired_action = ACTION_HOLD

        desired_actions.append(
            desired_action
        )


    # --------------------------------------------------------
    # 3. Predictive safety shield
    # --------------------------------------------------------

    current_states = [
        agent.state
        for agent in agents
    ]

    shield_result = safety_shield(
        states=current_states,
        desired_actions=desired_actions,
        safe_separation=SAFE_SEPARATION,
        step_size=ROLLOUT_STEP_SIZE,
    )

    desired_min = (
        shield_result[
            "desired_minimum_distance"
        ]
    )

    executed_min = (
        shield_result[
            "minimum_distance"
        ]
    )

    desired_min_history.append(
        desired_min
    )

    executed_min_history.append(
        executed_min
    )

    if desired_min < SAFE_SEPARATION:
        desired_unsafe_timesteps += 1

    if shield_result["override_count"] > 0:
        intervention_timesteps += 1

    mission_min_separation = min(
        mission_min_separation,
        executed_min,
    )


    # --------------------------------------------------------
    # 4. Record per-UUV overrides
    # --------------------------------------------------------

    executed_actions = (
        shield_result[
            "executed_actions"
        ]
    )

    for agent, desired, executed in zip(
        agents,
        desired_actions,
        executed_actions,
    ):
        if desired != executed:
            safety_override_counts[
                agent.uuv_id
            ] += 1


    # --------------------------------------------------------
    # 5. Compute all next states BEFORE mutating agents
    # --------------------------------------------------------

    next_states = []

    for agent, executed_action in zip(
        agents,
        executed_actions,
    ):
        next_state = predict_safety_state(
            state=agent.state,
            action=executed_action,
            step_size=ROLLOUT_STEP_SIZE,
        )

        next_states.append(
            next_state
        )


    # --------------------------------------------------------
    # 6. Simultaneous state update
    # --------------------------------------------------------

    for agent, next_state in zip(
        agents,
        next_states,
    ):
        agent.state = next_state

        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # --------------------------------------------------------
    # 7. Independent private observations
    # --------------------------------------------------------

    for agent in agents:
        agent.exploration_memory.update_from_fov(
            agent.state
        )

        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )

        agent.resource_memory.update(
            visible_resources
        )


# ------------------------------------------------------------
# Final safety invariant
# ------------------------------------------------------------

if mission_min_separation < SAFE_SEPARATION:
    raise RuntimeError(
        "Safety-aware mission violated minimum separation."
    )


# ------------------------------------------------------------
# Per-agent mission metrics
# ------------------------------------------------------------

safe_reports = []

for agent in agents:
    start = safe_start[
        agent.uuv_id
    ]

    final_coverage = (
        agent.exploration_memory.coverage()
    )

    final_observed = (
        agent.exploration_memory.observed_count()
    )

    final_resources = len(
        agent.resource_memory
    )

    start_index = start[
        "trajectory_index"
    ]

    mission_path = agent.trajectory[
        start_index:
    ]

    mission_path_length = trajectory_path_length(
        mission_path
    )

    safe_reports.append(
        {
            "uuv_id":
                agent.uuv_id,

            "start_coverage_pct":
                start["coverage"] * 100,

            "final_coverage_pct":
                final_coverage * 100,

            "coverage_gain_pct":
                (
                    final_coverage
                    -
                    start["coverage"]
                ) * 100,

            "new_cells":
                (
                    final_observed
                    -
                    start["observed_cells"]
                ),

            "resources_before":
                start["known_resources"],

            "resources_after":
                final_resources,

            "new_resources":
                (
                    final_resources
                    -
                    start["known_resources"]
                ),

            "path_length_px":
                mission_path_length,

            "safety_overrides":
                safety_override_counts[
                    agent.uuv_id
                ],
        }
    )


safe_df = pd.DataFrame(
    safe_reports
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 94)
print(
    "MultiUUV-Link — Safety-Aware Closed-Loop "
    "Independent Frontier Exploration"
)
print("=" * 94)

print(
    f"\nSynchronized mission steps : "
    f"{SAFE_AUTONOMY_STEPS}"
)

print(
    f"Safety threshold           : "
    f"{SAFE_SEPARATION:.2f} px"
)

print(
    f"Desired unsafe timesteps   : "
    f"{desired_unsafe_timesteps}"
)

print(
    f"Intervention timesteps     : "
    f"{intervention_timesteps}"
)

print(
    f"Total safety overrides     : "
    f"{sum(safety_override_counts.values())}"
)

print(
    f"Mission minimum separation : "
    f"{mission_min_separation:.2f} px"
)

print(
    "\nPer-UUV mission metrics:"
)

print(
    safe_df.to_string(
        index=False,
        formatters={
            "start_coverage_pct":
                lambda x: f"{x:.4f}",

            "final_coverage_pct":
                lambda x: f"{x:.4f}",

            "coverage_gain_pct":
                lambda x: f"{x:.4f}",

            "path_length_px":
                lambda x: f"{x:.2f}",
        },
    )
)


# ------------------------------------------------------------
# Safety history visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    desired_min_history,
    linewidth=1.8,
    label="Desired predicted minimum separation",
)

plt.plot(
    executed_min_history,
    linewidth=1.8,
    label="Executed predicted minimum separation",
)

plt.axhline(
    SAFE_SEPARATION,
    linestyle="--",
    linewidth=2,
    label="Safety threshold",
)

plt.xlabel(
    "Synchronized mission timestep"
)

plt.ylabel(
    "Minimum separation [px]"
)

plt.title(
    "MultiUUV-Link — Safety Shield Intervention History"
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Trajectory visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)

for agent in agents:
    start_index = safe_start[
        agent.uuv_id
    ][
        "trajectory_index"
    ]

    path = np.asarray(
        agent.trajectory[
            start_index:
        ]
    )

    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
        zorder=5,
    )

    plt.scatter(
        path[0, 0],
        path[0, 1],
        s=80,
        marker="o",
        zorder=6,
    )

    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        s=120,
        marker="x",
        linewidths=2.5,
        zorder=7,
    )


plt.title(
    "MultiUUV-Link — Safety-Aware Independent "
    "Frontier Exploration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nSafety-aware closed-loop autonomous "
    "exploration validated."
)

# Combo 21 — Dynamic Range-Limited Communication Graph

## Objective

The four UUVs now possess:

- autonomous frontier exploration;
- private exploration memories;
- private resource memories;
- synchronized motion;
- predictive collision avoidance.

However, they remain informationally isolated.

This combo introduces the first communication abstraction in MultiUUV-Link:

$$
G_t=(V,E_t)
$$

where:

- $V$ is the set of UUVs;
- $E_t$ is the set of communication links available at timestep $t$.

A communication link exists only when two UUVs are sufficiently close.

This combo implements only:

```text
UUV positions
      ↓
pairwise distances
      ↓
communication range test
      ↓
dynamic graph
```

There is still:

```text
NO message exchange
NO map sharing
NO resource sharing
NO task allocation
NO packet loss
NO latency
```

The purpose is to establish the communication topology before allowing any information to flow through it.

---

# Communication Graph

For:

$$
N=4
$$

UUVs, define:

$$
V=\{0,1,2,3\}
$$

At timestep $t$, an undirected communication edge exists between UUVs $i$ and $j$ when:

$$
(i,j)\in E_t
\iff
\|p_i(t)-p_j(t)\|_2
\le
R_c
$$

where:

$$
R_c
$$

is the communication range.

---

# Initial Communication Range

For the lightweight prototype we choose:

$$
R_c=400\text{ px}
$$

This value is an algorithmic range in image coordinates.

It is **not** intended to represent a validated underwater acoustic communication distance.

The chosen value is large enough that links may form during the current swarm trajectories, but small enough that the complete swarm should not always form a fully connected graph.

---

# Neighbor Set

For UUV $i$, define its current communication neighbors as:

$$
\mathcal{N}_i(t)
=
\{
j\neq i
\mid
\|p_i-p_j\|_2\le R_c
\}
$$

Thus every agent may have:

$$
0
\le
|\mathcal{N}_i|
\le
N-1
$$

neighbors.

---

# Graph Symmetry

Communication is initially modeled as symmetric.

Therefore:

$$
j\in\mathcal{N}_i
\iff
i\in\mathcal{N}_j
$$

and:

$$
(i,j)\in E_t
\iff
(j,i)\in E_t
$$

Later communication degradation may break reliable information delivery without changing geometric adjacency.

---

# Dynamic Topology

Because UUV positions change:

$$
p_i=p_i(t)
$$

the graph also changes:

$$
G_t\neq G_{t+1}
$$

in general.

Links may therefore:

```text
appear
disappear
reappear
```

as vehicles move.

This dynamic topology is essential because communication availability should emerge from swarm geometry rather than remain permanently enabled.

---

# Graph Metrics

For each communication graph we may compute:

### Number of edges

$$
|E_t|
$$

For four UUVs:

$$
0\le|E_t|\le6
$$

---

### Node degree

For UUV $i$:

$$
deg_i(t)
=
|\mathcal{N}_i(t)|
$$

---

### Connected components

The graph may contain:

$$
C_t
$$

connected components.

If:

$$
C_t=1
$$

the communication graph is fully connected through direct or multi-hop paths.

If:

$$
C_t>1
$$

the swarm is partitioned.

---

### Graph density

For an undirected graph:

$$
\rho_G
=
\frac{2|E|}
{N(N-1)}
$$

with:

$$
0\le\rho_G\le1
$$

---

# Direct Versus Multi-Hop Connectivity

A communication edge represents a **direct link**.

For example:

```text
UUV-0 ↔ UUV-1 ↔ UUV-2
```

means UUV-0 does not necessarily possess a direct link to UUV-2.

However, the graph as a whole contains a multi-hop path.

Actual message forwarding is not implemented in this combo.

The distinction is important:

$$
\text{Direct Neighbor}
\neq
\text{Graph-Reachable Agent}
$$

---

# Historical Communication Analysis

The safety-aware mission from Combo 20 generated synchronized trajectories of equal duration.

Those trajectories provide a clean dataset for reconstructing the communication graph over time.

For every synchronized mission timestep:

$$
t=0,\ldots,T
$$

we can compute:

$$
G_t
$$

and measure:

- number of active edges;
- number of connected components;
- graph density;
- link appearance frequency.

This lets us verify that communication availability changes naturally with motion.

---

# Information Boundary

The communication graph itself may use true current UUV positions because relative geometry is required to determine link availability.

However, establishing an edge does **not** automatically merge any knowledge.

Therefore:

$$
(i,j)\in E_t
$$

does not imply:

$$
M_i=M_j
$$

or:

$$
D_i=D_j
$$

A link means only:

> UUV $i$ and UUV $j$ are currently permitted to exchange messages.

Actual knowledge exchange comes later.

---

# Validation Strategy

The combo will validate three cases.

## Case 1 — Current Swarm

Construct the graph from current UUV positions.

---

## Case 2 — Historical Mission

Reconstruct the graph across the synchronized safety-aware trajectory generated in Combo 20.

We expect graph topology to vary over time.

---

## Case 3 — Synthetic Geometry

Create:

```text
UUV-0 ---- UUV-1          UUV-2
```

such that:

$$
d_{01}<R_c
$$

while:

$$
d_{02}>R_c
$$

and:

$$
d_{12}>R_c
$$

The graph must therefore contain exactly one edge:

$$
E=\{(0,1)\}
$$

---

# Expected Output

The code cell should:

1. define a communication range;
2. construct a NetworkX graph from UUV positions;
3. add edges only when:

$$
d_{ij}\le R_c
$$

4. compute current:
   - neighbors;
   - edge count;
   - connected components;
   - graph density;
5. reconstruct communication topology over the Combo-20 synchronized trajectory;
6. report:
   - minimum edge count;
   - maximum edge count;
   - mean edge count;
   - fraction of timesteps with fully connected graph;
   - fraction of timesteps with no links;
7. validate the synthetic one-edge case;
8. visualize:
   - current UUV positions;
   - current communication links.

A successful result establishes:

$$
\boxed{
\text{Swarm Geometry}
\rightarrow
\text{Dynamic Communication Topology}
}
$$

without yet transferring knowledge.

---

In [ ]:
# ============================================================
# Combo 21 — Dynamic Range-Limited Communication Graph
# ============================================================


# ------------------------------------------------------------
# Communication configuration
# ------------------------------------------------------------

COMMUNICATION_RANGE = 400.0


# ------------------------------------------------------------
# Communication-graph constructor
# ------------------------------------------------------------

def build_communication_graph(
    positions,
    communication_range=COMMUNICATION_RANGE,
):
    """
    Build an undirected range-limited communication graph.

    Parameters
    ----------
    positions : dict
        {uuv_id: (x, y)}

    communication_range : float
        Maximum direct communication distance.

    Returns
    -------
    networkx.Graph
    """

    graph = nx.Graph()

    uuv_ids = sorted(
        positions.keys()
    )

    # Add all UUVs even if isolated
    graph.add_nodes_from(
        uuv_ids
    )


    for uuv_a, uuv_b in combinations(
        uuv_ids,
        2,
    ):

        distance = euclidean_distance(
            positions[uuv_a],
            positions[uuv_b],
        )

        if distance <= communication_range:

            graph.add_edge(
                uuv_a,
                uuv_b,
                distance=float(distance),
            )

    return graph


# ------------------------------------------------------------
# Current swarm graph
# ------------------------------------------------------------

current_positions = {
    agent.uuv_id: (
        agent.state.x,
        agent.state.y,
    )
    for agent in agents
}

current_comm_graph = (
    build_communication_graph(
        current_positions
    )
)


# ------------------------------------------------------------
# Current graph report
# ------------------------------------------------------------

print(
    "=" * 86
)

print(
    "MultiUUV-Link — Dynamic Range-Limited Communication Graph"
)

print(
    "=" * 86
)

print(
    f"\nCommunication range : "
    f"{COMMUNICATION_RANGE:.2f} px"
)

print(
    f"Current edge count  : "
    f"{current_comm_graph.number_of_edges()}"
)

print(
    f"Current components  : "
    f"{nx.number_connected_components(current_comm_graph)}"
)

print(
    f"Current density     : "
    f"{nx.density(current_comm_graph):.4f}"
)


# ------------------------------------------------------------
# Current neighbor sets
# ------------------------------------------------------------

print(
    "\nCurrent neighbor sets:"
)

for uuv_id in sorted(
    current_comm_graph.nodes
):

    neighbors = sorted(
        current_comm_graph.neighbors(
            uuv_id
        )
    )

    print(
        f"  UUV-{uuv_id}: "
        f"{neighbors}"
    )


# ------------------------------------------------------------
# Current edges with distances
# ------------------------------------------------------------

print(
    "\nCurrent communication links:"
)

if current_comm_graph.number_of_edges() == 0:

    print(
        "  None"
    )

else:

    for uuv_a, uuv_b, data in (
        current_comm_graph.edges(
            data=True
        )
    ):

        print(
            f"  UUV-{uuv_a} ↔ UUV-{uuv_b} | "
            f"{data['distance']:.2f} px"
        )


# ============================================================
# Historical synchronized communication reconstruction
# ============================================================

# Combo 20 appended exactly SAFE_AUTONOMY_STEPS new
# synchronized states after each agent's safe_start index.

historical_comm_stats = []

historical_edge_sets = []


for step in range(
    SAFE_AUTONOMY_STEPS + 1
):

    positions_at_t = {}

    for agent in agents:

        start_index = (
            safe_start[
                agent.uuv_id
            ][
                "trajectory_index"
            ]
        )

        trajectory_index = (
            start_index
            +
            step
        )

        positions_at_t[
            agent.uuv_id
        ] = (
            agent.trajectory[
                trajectory_index
            ]
        )


    graph_t = build_communication_graph(
        positions_at_t
    )

    edge_count = (
        graph_t.number_of_edges()
    )

    component_count = (
        nx.number_connected_components(
            graph_t
        )
    )

    density = nx.density(
        graph_t
    )

    fully_connected = (
        component_count == 1
    )

    no_links = (
        edge_count == 0
    )

    canonical_edges = frozenset(
        tuple(sorted(edge))
        for edge in graph_t.edges()
    )

    historical_edge_sets.append(
        canonical_edges
    )


    historical_comm_stats.append(
        {
            "step":
                step,

            "edge_count":
                edge_count,

            "components":
                component_count,

            "density":
                density,

            "fully_connected":
                fully_connected,

            "no_links":
                no_links,
        }
    )


historical_comm_df = pd.DataFrame(
    historical_comm_stats
)


# ------------------------------------------------------------
# Historical topology metrics
# ------------------------------------------------------------

min_edges = int(
    historical_comm_df[
        "edge_count"
    ].min()
)

max_edges = int(
    historical_comm_df[
        "edge_count"
    ].max()
)

mean_edges = float(
    historical_comm_df[
        "edge_count"
    ].mean()
)

fully_connected_fraction = float(
    historical_comm_df[
        "fully_connected"
    ].mean()
)

no_link_fraction = float(
    historical_comm_df[
        "no_links"
    ].mean()
)

unique_topologies = len(
    set(
        historical_edge_sets
    )
)


print(
    "\nHistorical communication topology:"
)

print(
    f"  Timesteps analyzed       : "
    f"{len(historical_comm_df)}"
)

print(
    f"  Minimum active edges     : "
    f"{min_edges}"
)

print(
    f"  Maximum active edges     : "
    f"{max_edges}"
)

print(
    f"  Mean active edges        : "
    f"{mean_edges:.3f}"
)

print(
    f"  Fully connected fraction : "
    f"{fully_connected_fraction:.3f}"
)

print(
    f"  No-link fraction         : "
    f"{no_link_fraction:.3f}"
)

print(
    f"  Unique graph topologies  : "
    f"{unique_topologies}"
)


# ------------------------------------------------------------
# Dynamic-topology check
# ------------------------------------------------------------

topology_changed = (
    unique_topologies > 1
)

print(
    f"  Topology changed         : "
    f"{topology_changed}"
)

if not topology_changed:
    print(
        "\nNOTE: The current trajectory happened to "
        "produce only one topology. This is not a graph "
        "implementation failure, but later scenarios should "
        "exercise topology changes."
    )


# ============================================================
# Synthetic communication validation
# ============================================================

synthetic_positions = {
    0: (100, 100),
    1: (300, 100),
    2: (900, 100),
}

synthetic_graph = build_communication_graph(
    synthetic_positions,
    communication_range=
        COMMUNICATION_RANGE,
)

synthetic_edges = {
    tuple(
        sorted(edge)
    )
    for edge in synthetic_graph.edges()
}

expected_synthetic_edges = {
    (0, 1)
}

synthetic_test_passed = (
    synthetic_edges
    ==
    expected_synthetic_edges
)


print(
    "\nSynthetic communication test:"
)

print(
    f"  Returned edges : "
    f"{sorted(synthetic_edges)}"
)

print(
    f"  Expected edges : "
    f"{sorted(expected_synthetic_edges)}"
)

print(
    f"  Test passed    : "
    f"{synthetic_test_passed}"
)


if not synthetic_test_passed:
    raise RuntimeError(
        "Synthetic communication-graph test failed."
    )


# ------------------------------------------------------------
# Graph symmetry validation
# ------------------------------------------------------------

symmetry_passed = True

for uuv_a, uuv_b in (
    current_comm_graph.edges()
):

    if not (
        current_comm_graph.has_edge(
            uuv_b,
            uuv_a,
        )
    ):
        symmetry_passed = False
        break


print(
    "\nGraph invariants:"
)

print(
    f"  Undirected symmetry : "
    f"{symmetry_passed}"
)

if not symmetry_passed:
    raise RuntimeError(
        "Communication graph is not symmetric."
    )


# ------------------------------------------------------------
# Current graph visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


# ------------------------------------------------------------
# Draw communication links
# ------------------------------------------------------------

for uuv_a, uuv_b in (
    current_comm_graph.edges()
):

    x_a, y_a = (
        current_positions[
            uuv_a
        ]
    )

    x_b, y_b = (
        current_positions[
            uuv_b
        ]
    )

    plt.plot(
        [x_a, x_b],
        [y_a, y_b],
        linewidth=2,
        linestyle="--",
        zorder=5,
    )


# ------------------------------------------------------------
# Draw agents
# ------------------------------------------------------------

for agent in agents:

    plt.scatter(
        agent.state.x,
        agent.state.y,
        s=140,
        marker="o",
        label=f"UUV-{agent.uuv_id}",
        zorder=7,
    )

    plt.text(
        agent.state.x + 10,
        agent.state.y - 10,
        f"{agent.uuv_id}",
        fontsize=10,
        zorder=8,
    )


plt.title(
    "MultiUUV-Link — Current Range-Limited "
    "Communication Graph"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Historical edge-count visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    historical_comm_df[
        "step"
    ],
    historical_comm_df[
        "edge_count"
    ],
    linewidth=2,
)

plt.xlabel(
    "Synchronized mission timestep"
)

plt.ylabel(
    "Active communication edges"
)

plt.title(
    "MultiUUV-Link — Dynamic Communication Connectivity"
)

plt.yticks(
    range(
        0,
        7,
    )
)

plt.tight_layout()

plt.show()


print(
    "\nDynamic range-limited communication "
    "graph validated."
)

# Combo 22 — Explicit Resource-Knowledge Exchange

## Objective

Combo 21 established a dynamic range-limited communication graph:

$$
G_t=(V,E_t)
$$

A communication link now tells us which UUV pairs are currently permitted to exchange information.

However, no knowledge has yet crossed those links.

This combo introduces the first explicit communication payload:

```text
discovered resource knowledge
```

A UUV may transmit only resources that already exist in its own private resource memory.

Therefore communication can propagate legitimate knowledge, but it cannot reveal simulator-hidden resources.

The information path becomes:

```text
World Resource
      ↓
Local FOV Detection
      ↓
Sender Private Memory
      ↓
Communication Link
      ↓
Explicit Message
      ↓
Receiver Private Memory
```

This establishes the key communication invariant:

$$
\boxed{
\text{Remote Knowledge}
=
\text{Previously Legitimate Knowledge from Another Agent}
}
$$

There is still:

```text
NO exploration-map sharing
NO task allocation
NO packet loss
NO latency
NO bandwidth limit
NO multi-hop forwarding
```

This combo implements only direct one-hop resource knowledge exchange.

---

# Sender Knowledge

For UUV $i$, the private resource memory is:

$$
D_i
$$

The sender may transmit only:

$$
R_j\in D_i
$$

The sender may not inspect or transmit:

$$
R_j\in R_{\text{world}}\setminus D_i
$$

Thus:

$$
Message_i
\subseteq D_i
$$

This prevents communication from becoming an accidental ground-truth oracle.

---

# Communication Eligibility

UUV $i$ may send directly to UUV $j$ only if:

$$
(i,j)\in E_t
$$

where $E_t$ is the current communication graph.

For the current swarm state, Combo 21 reported:

```text
UUV-0 ↔ UUV-2
UUV-0 ↔ UUV-3
UUV-2 ↔ UUV-3

UUV-1 isolated
```

Therefore UUV-1 must not receive information during the current one-hop exchange.

---

# Message Representation

A resource-knowledge message contains:

$$
m=
(sender,receiver,payload)
$$

where the payload contains structured resource records:

$$
R_j=
(id_j,x_j,y_j,c_j)
$$

The message does not contain:

- global world state;
- hidden resource records;
- another UUV's map;
- simulator metadata.

---

# Knowledge Merge

Suppose UUV $j$ receives resource set:

$$
P_{i\rightarrow j}
$$

Then:

$$
D_j^+
=
D_j
\cup
P_{i\rightarrow j}
$$

The receiver's memory remains persistent.

Duplicate resource IDs are naturally ignored by the existing dictionary-based memory representation.

---

# Provenance

After communication, it becomes important to distinguish:

```text
personally observed
```

from:

```text
received remotely
```

The current `ResourceMemory` class stores only resource data and does not yet track provenance.

To preserve that distinction without rewriting all previous code, this combo introduces a separate per-agent provenance table.

For each known resource:

$$
P_i(R_j)
\in
\{
LOCAL,
REMOTE
\}
$$

A resource personally observed by the agent is labeled:

```text
LOCAL
```

A resource first learned through communication is labeled:

```text
REMOTE
```

If an already-remotely-known resource is later personally observed, its provenance may later be upgraded to local observation.

For this first communication validation, provenance is used only to distinguish newly received remote knowledge.

---

# Direct One-Hop Exchange

Communication is performed over each current graph edge.

For direct edge:

$$
(i,j)\in E_t
$$

both directions are permitted:

$$
i\rightarrow j
$$

and:

$$
j\rightarrow i
$$

because the current graph is undirected.

However, this combo performs only a **single communication round**.

Therefore information received during this round is not immediately forwarded again during the same round.

This matters.

Suppose:

```text
0 ↔ 2 ↔ 3
```

If UUV-0 knows resource $R_a$, one communication round may send it to UUV-2.

UUV-2 should not then instantly forward newly received $R_a$ to UUV-3 in the same round unless we explicitly model multi-hop propagation.

Thus all outgoing messages are constructed from a snapshot of knowledge taken **before** the communication round begins.

---

# Snapshot Semantics

Let:

$$
D_i^0
$$

be sender knowledge before the round.

Messages are constructed from:

$$
D_i^0
$$

for every UUV.

Only after all messages are generated do receivers merge them.

Therefore:

$$
D_i^0
\rightarrow
Messages
\rightarrow
D_i^1
$$

This creates clean synchronous message semantics.

---

# Communication Gain

For UUV $i$, define:

$$
\Delta D_i^{comm}
=
|D_i^{after}|
-
|D_i^{before}|
$$

This measures how many resource records were learned through communication during the round.

A disconnected UUV should satisfy:

$$
\Delta D_i^{comm}=0
$$

unless it had some direct communication edge.

---

# Global Knowledge Comparison

For validation only, the simulator may compute the union of all private pre-communication knowledge:

$$
D_{\cup}
=
\bigcup_i D_i
$$

This is an evaluation quantity.

It is not exposed to agents.

After direct exchange, connected agents may move closer to this union, but disconnected components need not share identical knowledge.

---

# Synthetic Validation

Natural current memories may occasionally contain no unique knowledge worth exchanging.

Therefore this combo includes a deterministic synthetic test.

Synthetic sender A knows:

$$
R_a
$$

Synthetic receiver B knows nothing.

With a valid communication edge:

$$
A\leftrightarrow B
$$

after one exchange:

$$
R_a\in D_B
$$

The same receiver without a communication edge must remain unaware.

This verifies that knowledge propagation is controlled by graph connectivity.

---

# Expected Output

The code cell should:

1. snapshot every UUV's private resource memory;
2. build direct messages only along current communication edges;
3. construct messages only from sender-known resources;
4. apply all messages after message construction;
5. count:
   - messages sent;
   - resource records transmitted;
   - new remote resources learned per UUV;
6. preserve direct one-hop semantics;
7. verify disconnected UUVs receive no messages;
8. maintain a provenance record for remotely learned resources;
9. run synthetic connected/disconnected validation;
10. print before/after private resource knowledge.

A successful result establishes:

$$
\boxed{
\text{Private Local Knowledge}
+
\text{Valid Communication Edge}
\rightarrow
\text{Legitimate Remote Knowledge}
}
$$

---

In [ ]:
# ============================================================
# Combo 22 — Explicit Resource-Knowledge Exchange
# ============================================================

from copy import deepcopy


# ------------------------------------------------------------
# Initialize provenance tracking
# ------------------------------------------------------------

resource_provenance = {}

for agent in agents:

    provenance = {}

    for resource in (
        agent.resource_memory.all_resources()
    ):
        provenance[
            resource.resource_id
        ] = "LOCAL"

    resource_provenance[
        agent.uuv_id
    ] = provenance


# ------------------------------------------------------------
# Snapshot private resource knowledge
# ------------------------------------------------------------

pre_comm_resource_snapshots = {}

for agent in agents:

    pre_comm_resource_snapshots[
        agent.uuv_id
    ] = {
        resource.resource_id:
            resource
        for resource in (
            agent.resource_memory.all_resources()
        )
    }


# ------------------------------------------------------------
# Before-communication counts
# ------------------------------------------------------------

before_counts = {
    agent.uuv_id:
        len(
            pre_comm_resource_snapshots[
                agent.uuv_id
            ]
        )
    for agent in agents
}


# ------------------------------------------------------------
# Construct one-hop messages FROM SNAPSHOT ONLY
# ------------------------------------------------------------

resource_messages = []

for sender_id, receiver_id in (
    current_comm_graph.edges()
):

    # sender -> receiver
    sender_payload = list(
        pre_comm_resource_snapshots[
            sender_id
        ].values()
    )

    resource_messages.append(
        {
            "sender":
                sender_id,

            "receiver":
                receiver_id,

            "resources":
                sender_payload,
        }
    )


    # receiver -> sender
    reverse_payload = list(
        pre_comm_resource_snapshots[
            receiver_id
        ].values()
    )

    resource_messages.append(
        {
            "sender":
                receiver_id,

            "receiver":
                sender_id,

            "resources":
                reverse_payload,
        }
    )


# ------------------------------------------------------------
# Agent lookup
# ------------------------------------------------------------

agent_by_id = {
    agent.uuv_id:
        agent
    for agent in agents
}


# ------------------------------------------------------------
# Apply messages after construction
# ------------------------------------------------------------

received_message_counts = {
    agent.uuv_id: 0
    for agent in agents
}

transmitted_resource_records = 0

new_remote_resources = {
    agent.uuv_id: []
    for agent in agents
}


for message in resource_messages:

    sender_id = (
        message[
            "sender"
        ]
    )

    receiver_id = (
        message[
            "receiver"
        ]
    )

    payload = (
        message[
            "resources"
        ]
    )

    received_message_counts[
        receiver_id
    ] += 1

    transmitted_resource_records += len(
        payload
    )


    receiver = agent_by_id[
        receiver_id
    ]


    # --------------------------------------------------------
    # Merge resource knowledge
    # --------------------------------------------------------

    newly_learned = (
        receiver
        .resource_memory
        .update(
            payload
        )
    )


    # --------------------------------------------------------
    # Mark new knowledge as REMOTE
    # --------------------------------------------------------

    for resource in newly_learned:

        resource_provenance[
            receiver_id
        ][
            resource.resource_id
        ] = "REMOTE"

        new_remote_resources[
            receiver_id
        ].append(
            {
                "resource_id":
                    resource.resource_id,

                "source_uuv":
                    sender_id,

                "resource":
                    resource,
            }
        )


# ------------------------------------------------------------
# After-communication counts
# ------------------------------------------------------------

after_counts = {
    agent.uuv_id:
        len(
            agent.resource_memory
        )
    for agent in agents
}


# ------------------------------------------------------------
# Communication-gain report
# ------------------------------------------------------------

communication_resource_reports = []

for agent in agents:

    uuv_id = (
        agent.uuv_id
    )

    communication_resource_reports.append(
        {
            "uuv_id":
                uuv_id,

            "neighbors":
                len(
                    list(
                        current_comm_graph
                        .neighbors(
                            uuv_id
                        )
                    )
                ),

            "messages_received":
                received_message_counts[
                    uuv_id
                ],

            "resources_before":
                before_counts[
                    uuv_id
                ],

            "resources_after":
                after_counts[
                    uuv_id
                ],

            "new_remote_resources":
                (
                    after_counts[
                        uuv_id
                    ]
                    -
                    before_counts[
                        uuv_id
                    ]
                ),
        }
    )


communication_resource_df = (
    pd.DataFrame(
        communication_resource_reports
    )
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 90
)

print(
    "MultiUUV-Link — Explicit Resource-Knowledge Exchange"
)

print(
    "=" * 90
)

print(
    f"\nCurrent communication edges : "
    f"{current_comm_graph.number_of_edges()}"
)

print(
    f"Directed messages created   : "
    f"{len(resource_messages)}"
)

print(
    f"Resource records transmitted: "
    f"{transmitted_resource_records}"
)

print(
    "\nPer-UUV communication gain:"
)

print(
    communication_resource_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Detailed newly learned remote resources
# ------------------------------------------------------------

print(
    "\nNew remotely learned resources:"
)

for agent in agents:

    uuv_id = agent.uuv_id

    learned = (
        new_remote_resources[
            uuv_id
        ]
    )

    print(
        f"\nUUV-{uuv_id}:"
    )

    if not learned:

        print(
            "  None"
        )

    else:

        for item in learned:

            resource = (
                item[
                    "resource"
                ]
            )

            print(
                f"  ID={resource.resource_id:2d} | "
                f"Type {resource.resource_class} | "
                f"Position=({resource.x}, {resource.y}) | "
                f"source=UUV-{item['source_uuv']} | "
                f"provenance=REMOTE"
            )


# ------------------------------------------------------------
# Disconnected-UUV invariant
# ------------------------------------------------------------

print(
    "\nDisconnected-agent validation:"
)

for agent in agents:

    degree = (
        current_comm_graph.degree[
            agent.uuv_id
        ]
    )

    if degree == 0:

        gained = (
            after_counts[
                agent.uuv_id
            ]
            -
            before_counts[
                agent.uuv_id
            ]
        )

        print(
            f"  UUV-{agent.uuv_id} isolated | "
            f"knowledge gain={gained}"
        )

        if gained != 0:
            raise RuntimeError(
                f"Disconnected UUV-{agent.uuv_id} "
                "received remote resource knowledge."
            )


# ------------------------------------------------------------
# One-hop snapshot invariant
# ------------------------------------------------------------

# Every transmitted resource must have existed in sender's
# PRE-COMMUNICATION snapshot.

snapshot_integrity = True

for message in resource_messages:

    sender_id = (
        message[
            "sender"
        ]
    )

    sender_snapshot_ids = set(
        pre_comm_resource_snapshots[
            sender_id
        ].keys()
    )

    for resource in (
        message[
            "resources"
        ]
    ):

        if (
            resource.resource_id
            not in
            sender_snapshot_ids
        ):
            snapshot_integrity = False
            break


print(
    "\nMessage integrity:"
)

print(
    f"  Payloads sourced only from "
    f"pre-round sender knowledge : "
    f"{snapshot_integrity}"
)

if not snapshot_integrity:
    raise RuntimeError(
        "Communication violated one-hop snapshot semantics."
    )


# ------------------------------------------------------------
# Private knowledge after exchange
# ------------------------------------------------------------

print(
    "\nPrivate resource memories after communication:"
)

for agent in agents:

    resources_known = sorted(
        agent.resource_memory.all_resources(),
        key=lambda resource:
            resource.resource_id,
    )

    print(
        f"\nUUV-{agent.uuv_id}: "
        f"{len(resources_known)} known"
    )

    if not resources_known:

        print(
            "  None"
        )

        continue

    for resource in resources_known:

        provenance = (
            resource_provenance[
                agent.uuv_id
            ].get(
                resource.resource_id,
                "LOCAL",
            )
        )

        print(
            f"  ID={resource.resource_id:2d} | "
            f"Type {resource.resource_class} | "
            f"Position=({resource.x}, {resource.y}) | "
            f"{provenance}"
        )


# ============================================================
# Synthetic communication validation
# ============================================================

synthetic_resource = Resource(
    resource_id=99999,
    x=500,
    y=500,
    resource_class="A",
)


# ------------------------------------------------------------
# Connected synthetic case
# ------------------------------------------------------------

synthetic_sender_memory = (
    ResourceMemory(
        uuv_id=100
    )
)

synthetic_receiver_memory = (
    ResourceMemory(
        uuv_id=101
    )
)

synthetic_sender_memory.update(
    [
        synthetic_resource
    ]
)

synthetic_connected_graph = nx.Graph()

synthetic_connected_graph.add_nodes_from(
    [
        100,
        101,
    ]
)

synthetic_connected_graph.add_edge(
    100,
    101,
)

if synthetic_connected_graph.has_edge(
    100,
    101,
):

    synthetic_receiver_memory.update(
        synthetic_sender_memory
        .all_resources()
    )


connected_test_passed = (
    synthetic_receiver_memory.knows(
        synthetic_resource.resource_id
    )
)


# ------------------------------------------------------------
# Disconnected synthetic case
# ------------------------------------------------------------

synthetic_receiver_disconnected = (
    ResourceMemory(
        uuv_id=102
    )
)

synthetic_disconnected_graph = nx.Graph()

synthetic_disconnected_graph.add_nodes_from(
    [
        100,
        102,
    ]
)

if synthetic_disconnected_graph.has_edge(
    100,
    102,
):

    synthetic_receiver_disconnected.update(
        synthetic_sender_memory
        .all_resources()
    )


disconnected_test_passed = (
    not
    synthetic_receiver_disconnected.knows(
        synthetic_resource.resource_id
    )
)


print(
    "\nSynthetic resource-communication validation:"
)

print(
    f"  Connected receiver learned resource    : "
    f"{connected_test_passed}"
)

print(
    f"  Disconnected receiver remained unaware : "
    f"{disconnected_test_passed}"
)


if not connected_test_passed:
    raise RuntimeError(
        "Connected synthetic communication test failed."
    )

if not disconnected_test_passed:
    raise RuntimeError(
        "Disconnected synthetic communication test failed."
    )


print(
    "\nExplicit one-hop resource knowledge "
    "exchange validated."
)

# Combo 23 — Exploration-Map Knowledge Exchange

## Objective

Combo 22 demonstrated explicit exchange of discovered-resource knowledge.

The next communication payload is spatial exploration knowledge.

Each UUV currently maintains a private boolean exploration memory:

$$
M_i(y,x)\in\{0,1\}
$$

where:

$$
M_i(y,x)=1
$$

means that UUV $i$ knows the cell has been observed.

Until now:

$$
M_i
$$

contained only cells personally observed by UUV $i$.

This combo allows directly connected neighbors to share their explored-region knowledge.

After communication, UUV $i$ may therefore know a region that it has never personally visited.

The information chain becomes:

```text
Local Observation by UUV-j
        ↓
Private Map M_j
        ↓
Valid Communication Edge
        ↓
Explicit Map Message
        ↓
Merge into UUV-i Knowledge
```

---

# Local Versus Remote Spatial Knowledge

A key distinction must now be preserved.

For UUV $i$, define:

$$
M_i^{local}
$$

as cells personally observed by UUV $i$.

Define:

$$
M_i^{known}
$$

as all cells known by UUV $i$, whether learned locally or through communication.

Therefore:

$$
M_i^{local}
\subseteq
M_i^{known}
$$

Initially, before map communication:

$$
M_i^{known}
=
M_i^{local}
$$

After communication:

$$
M_i^{known}
=
M_i^{local}
\cup
M_j^{known}
$$

for received neighbor knowledge.

The existing `ExplorationMemory.observed` array currently represents personally accumulated map knowledge.

Rather than destroy this semantic distinction, this combo introduces an additional **shared/known exploration map** per agent.

---

# Knowledge Representation

For every UUV:

```text
local exploration memory
+
known exploration memory
```

where:

$$
M_i^{local}
=
\text{agent.exploration_memory.observed}
$$

and:

$$
M_i^{known}
$$

is initialized as a copy of the local map.

The local map remains untouched by communication.

Thus:

$$
\boxed{
\text{Communication cannot fake personal observation}
}
$$

---

# Map Message

A map message contains a boolean knowledge mask:

$$
m_{i\rightarrow j}
=
M_i^{known}
$$

For the initial prototype, the complete boolean mask is transmitted.

This is deliberately simple.

Later communication-cost modeling may replace full-map exchange with:

- sparse cell updates;
- compressed masks;
- changed-region deltas;
- frontier summaries.

For now, correctness is more important than bandwidth realism.

---

# Communication Eligibility

Map exchange occurs only across current direct communication edges:

$$
(i,j)\in E_t
$$

Each graph edge permits:

$$
i\rightarrow j
$$

and:

$$
j\rightarrow i
$$

because the communication graph is currently symmetric.

UUVs without a direct edge receive no map message.

---

# One-Hop Snapshot Semantics

As in Combo 22, map transmission uses synchronous snapshots.

Let:

$$
M_i^0
$$

be UUV $i$'s known map before the communication round.

All outgoing messages are created from:

$$
M_i^0
$$

Only after all messages are created do receivers merge them.

Therefore:

$$
M_i^0
\rightarrow
Messages
\rightarrow
M_i^1
$$

A map received during this round is not immediately forwarded again in the same round.

---

# Merge Operation

For receiver $j$ receiving sender map $M_i$:

$$
M_j^{known,+}
=
M_j^{known}
\lor
M_i^{known}
$$

where:

$$
\lor
$$

is element-wise logical OR.

Thus a cell becomes known if either the receiver or sender already knew it.

Knowledge is monotonic:

$$
M_j^{known}(t)
\subseteq
M_j^{known}(t+1)
$$

---

# Remote Spatial Gain

For UUV $i$, define:

$$
\Delta A_i^{remote}
=
|M_i^{known,after}|
-
|M_i^{known,before}|
$$

This measures how many new map cells were learned through communication.

A disconnected UUV must satisfy:

$$
\Delta A_i^{remote}=0
$$

during that communication round.

---

# Local-Knowledge Integrity

Communication must never modify:

$$
M_i^{local}
$$

Therefore after communication:

$$
M_i^{local,after}
=
M_i^{local,before}
$$

while:

$$
M_i^{known}
$$

may increase.

This is essential because later we may measure separately:

- personally explored area;
- total known area;
- remotely learned area.

---

# Knowledge Coverage

Define local coverage:

$$
C_i^{local}
=
\frac{|M_i^{local}|}{HW}
$$

and total known coverage:

$$
C_i^{known}
=
\frac{|M_i^{known}|}{HW}
$$

After communication:

$$
C_i^{known}
\ge
C_i^{local}
$$

The difference:

$$
C_i^{remote}
=
C_i^{known}
-
C_i^{local}
$$

measures knowledge acquired without personal traversal.

---

# Synthetic Validation

A deterministic synthetic test will create:

- sender map containing one observed block;
- receiver map containing a different observed block.

With a communication edge, the receiver should know the union:

$$
M_R^+
=
M_R
\lor
M_S
$$

Without a communication edge, the receiver must remain unchanged.

This validates map merging independently of current mission geometry.

---

# Expected Output

The code cell should:

1. create one `known_exploration_map` per UUV;
2. initialize it from that UUV's current private exploration memory;
3. snapshot every known map before communication;
4. construct direct one-hop map messages across current communication edges;
5. apply logical-OR map merging;
6. leave local exploration memories unchanged;
7. report:
   - local cells before;
   - known cells before;
   - known cells after;
   - remote cells gained;
   - local coverage;
   - known coverage;
8. verify disconnected UUVs gain zero remote map cells;
9. verify all sender payloads came from pre-round snapshots;
10. run synthetic connected/disconnected tests;
11. visualize one representative UUV:
    - locally observed area;
    - remotely learned area.

A successful result establishes:

$$
\boxed{
\text{Local Spatial Knowledge}
+
\text{Communication}
\rightarrow
\text{Expanded Distributed World Knowledge}
}
$$

---

In [ ]:
# ============================================================
# Combo 23 — Exploration-Map Knowledge Exchange
# ============================================================


# ------------------------------------------------------------
# Initialize total-known exploration maps
# ------------------------------------------------------------

known_exploration_maps = {}

for agent in agents:
    known_exploration_maps[
        agent.uuv_id
    ] = (
        agent.exploration_memory.observed.copy()
    )


# ------------------------------------------------------------
# Preserve local-map integrity snapshots
# ------------------------------------------------------------

local_map_snapshots = {
    agent.uuv_id:
        agent.exploration_memory.observed.copy()
    for agent in agents
}


# ------------------------------------------------------------
# Snapshot known maps BEFORE communication round
# ------------------------------------------------------------

pre_comm_known_maps = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ------------------------------------------------------------
# Before-communication counts
# ------------------------------------------------------------

local_cell_counts = {
    agent.uuv_id:
        int(
            agent
            .exploration_memory
            .observed
            .sum()
        )
    for agent in agents
}

known_before_counts = {
    uuv_id:
        int(
            known_map.sum()
        )
    for uuv_id, known_map
    in pre_comm_known_maps.items()
}


# ------------------------------------------------------------
# Construct one-hop map messages
# ------------------------------------------------------------

map_messages = []

for uuv_a, uuv_b in (
    current_comm_graph.edges()
):

    # UUV-a -> UUV-b
    map_messages.append(
        {
            "sender":
                uuv_a,

            "receiver":
                uuv_b,

            "known_map":
                pre_comm_known_maps[
                    uuv_a
                ].copy(),
        }
    )


    # UUV-b -> UUV-a
    map_messages.append(
        {
            "sender":
                uuv_b,

            "receiver":
                uuv_a,

            "known_map":
                pre_comm_known_maps[
                    uuv_b
                ].copy(),
        }
    )


# ------------------------------------------------------------
# Message counters
# ------------------------------------------------------------

map_messages_received = {
    agent.uuv_id: 0
    for agent in agents
}


# ------------------------------------------------------------
# Apply map messages
# ------------------------------------------------------------

for message in map_messages:

    receiver_id = (
        message[
            "receiver"
        ]
    )

    sender_map = (
        message[
            "known_map"
        ]
    )

    map_messages_received[
        receiver_id
    ] += 1


    # --------------------------------------------------------
    # Knowledge merge
    # --------------------------------------------------------

    known_exploration_maps[
        receiver_id
    ] = np.logical_or(
        known_exploration_maps[
            receiver_id
        ],
        sender_map,
    )


# ------------------------------------------------------------
# After-communication counts
# ------------------------------------------------------------

known_after_counts = {
    uuv_id:
        int(
            known_map.sum()
        )
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ------------------------------------------------------------
# Build report
# ------------------------------------------------------------

map_exchange_reports = []

for agent in agents:

    uuv_id = (
        agent.uuv_id
    )

    local_cells = (
        local_cell_counts[
            uuv_id
        ]
    )

    known_before = (
        known_before_counts[
            uuv_id
        ]
    )

    known_after = (
        known_after_counts[
            uuv_id
        ]
    )

    remote_gain = (
        known_after
        -
        known_before
    )

    total_cells = (
        WIDTH
        *
        HEIGHT
    )


    map_exchange_reports.append(
        {
            "uuv_id":
                uuv_id,

            "neighbors":
                current_comm_graph.degree[
                    uuv_id
                ],

            "messages_received":
                map_messages_received[
                    uuv_id
                ],

            "local_cells":
                local_cells,

            "known_before":
                known_before,

            "known_after":
                known_after,

            "remote_gain":
                remote_gain,

            "local_coverage_pct":
                (
                    local_cells
                    /
                    total_cells
                    *
                    100
                ),

            "known_coverage_pct":
                (
                    known_after
                    /
                    total_cells
                    *
                    100
                ),
        }
    )


map_exchange_df = pd.DataFrame(
    map_exchange_reports
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 92
)

print(
    "MultiUUV-Link — Exploration-Map Knowledge Exchange"
)

print(
    "=" * 92
)

print(
    f"\nCurrent communication edges : "
    f"{current_comm_graph.number_of_edges()}"
)

print(
    f"Directed map messages       : "
    f"{len(map_messages)}"
)

print(
    "\nPer-UUV spatial knowledge gain:"
)

print(
    map_exchange_df.to_string(
        index=False,
        formatters={
            "local_coverage_pct":
                lambda x: f"{x:.4f}",

            "known_coverage_pct":
                lambda x: f"{x:.4f}",
        },
    )
)


# ------------------------------------------------------------
# Local-map integrity validation
# ------------------------------------------------------------

local_integrity = True

for agent in agents:

    if not np.array_equal(
        local_map_snapshots[
            agent.uuv_id
        ],
        agent.exploration_memory.observed,
    ):
        local_integrity = False
        break


print(
    "\nKnowledge integrity:"
)

print(
    f"  Local maps unchanged by communication : "
    f"{local_integrity}"
)

if not local_integrity:
    raise RuntimeError(
        "Communication modified local exploration history."
    )


# ------------------------------------------------------------
# Knowledge supersets local knowledge
# ------------------------------------------------------------

known_superset_passed = True

for agent in agents:

    uuv_id = agent.uuv_id

    local_map = (
        agent
        .exploration_memory
        .observed
    )

    known_map = (
        known_exploration_maps[
            uuv_id
        ]
    )

    if np.any(
        local_map
        &
        (~known_map)
    ):
        known_superset_passed = False
        break


print(
    f"  Known maps contain all local cells     : "
    f"{known_superset_passed}"
)

if not known_superset_passed:
    raise RuntimeError(
        "Known map lost locally observed cells."
    )


# ------------------------------------------------------------
# Disconnected-UUV invariant
# ------------------------------------------------------------

print(
    "\nDisconnected-agent validation:"
)

for agent in agents:

    uuv_id = (
        agent.uuv_id
    )

    degree = (
        current_comm_graph.degree[
            uuv_id
        ]
    )

    if degree == 0:

        gain = (
            known_after_counts[
                uuv_id
            ]
            -
            known_before_counts[
                uuv_id
            ]
        )

        print(
            f"  UUV-{uuv_id} isolated | "
            f"remote map gain={gain}"
        )

        if gain != 0:
            raise RuntimeError(
                f"Disconnected UUV-{uuv_id} "
                "received remote map knowledge."
            )


# ------------------------------------------------------------
# Snapshot message-integrity check
# ------------------------------------------------------------

snapshot_integrity = True

for message in map_messages:

    sender = (
        message[
            "sender"
        ]
    )

    if not np.array_equal(
        message[
            "known_map"
        ],
        pre_comm_known_maps[
            sender
        ],
    ):
        snapshot_integrity = False
        break


print(
    "\nMessage integrity:"
)

print(
    f"  Payloads match pre-round sender maps : "
    f"{snapshot_integrity}"
)

if not snapshot_integrity:
    raise RuntimeError(
        "Map communication violated snapshot semantics."
    )


# ============================================================
# Synthetic map communication tests
# ============================================================

synthetic_sender_map = np.zeros(
    (20, 20),
    dtype=bool,
)

synthetic_receiver_map = np.zeros(
    (20, 20),
    dtype=bool,
)

synthetic_sender_map[
    2:6,
    2:6
] = True

synthetic_receiver_map[
    12:16,
    12:16
] = True


# ------------------------------------------------------------
# Connected merge
# ------------------------------------------------------------

synthetic_connected_result = np.logical_or(
    synthetic_receiver_map,
    synthetic_sender_map,
)

expected_connected_cells = (
    int(
        synthetic_sender_map.sum()
    )
    +
    int(
        synthetic_receiver_map.sum()
    )
)

connected_map_test_passed = (
    int(
        synthetic_connected_result.sum()
    )
    ==
    expected_connected_cells
)


# ------------------------------------------------------------
# Disconnected case
# ------------------------------------------------------------

synthetic_disconnected_result = (
    synthetic_receiver_map.copy()
)

disconnected_map_test_passed = (
    np.array_equal(
        synthetic_disconnected_result,
        synthetic_receiver_map,
    )
)


print(
    "\nSynthetic map-communication validation:"
)

print(
    f"  Connected maps merged correctly    : "
    f"{connected_map_test_passed}"
)

print(
    f"  Disconnected map remained unchanged: "
    f"{disconnected_map_test_passed}"
)


if not connected_map_test_passed:
    raise RuntimeError(
        "Synthetic connected map merge failed."
    )

if not disconnected_map_test_passed:
    raise RuntimeError(
        "Synthetic disconnected map test failed."
    )


# ------------------------------------------------------------
# Representative visualization
# ------------------------------------------------------------

# Pick the UUV with the greatest remote map gain.
representative_uuv = max(
    map_exchange_reports,
    key=lambda item:
        item["remote_gain"],
)[
    "uuv_id"
]


local_map = (
    agent_by_id[
        representative_uuv
    ]
    .exploration_memory
    .observed
)

known_map = (
    known_exploration_maps[
        representative_uuv
    ]
)

remote_only_map = (
    known_map
    &
    (~local_map)
)


# Visualization encoding:
#
# 0.0 = unknown
# 0.5 = remotely learned
# 1.0 = locally observed

knowledge_view = np.zeros(
    (HEIGHT, WIDTH),
    dtype=np.float32,
)

knowledge_view[
    remote_only_map
] = 0.5

knowledge_view[
    local_map
] = 1.0


plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    knowledge_view,
    cmap="gray",
    vmin=0.0,
    vmax=1.0,
)

plt.scatter(
    agent_by_id[
        representative_uuv
    ].state.x,
    agent_by_id[
        representative_uuv
    ].state.y,
    marker="x",
    s=130,
    linewidths=2.5,
    label=f"UUV-{representative_uuv}",
)

plt.title(
    f"MultiUUV-Link — UUV-{representative_uuv} "
    "Local vs Remote Exploration Knowledge"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nVisualization encoding:"
)

print(
    "  black = unknown"
)

print(
    "  gray  = remotely learned explored area"
)

print(
    "  white = personally observed area"
)


print(
    "\nExploration-map knowledge exchange validated."
)

# Combo 24 — Communication-Aware Frontier Planning on Total Known Map

## Objective

Combo 23 introduced two distinct spatial knowledge states for each UUV:

$$
M_i^{local}
$$

representing personally observed cells, and:

$$
M_i^{known}
$$

representing the union of local and remotely communicated exploration knowledge.

Until now, frontier planning has used only:

$$
M_i^{local}
$$

This means a UUV may still target a region that it has not personally visited even if another UUV has already explored that region and communicated the result.

That behavior would create unnecessary duplicate exploration.

This combo modifies frontier planning so that planning operates on:

$$
M_i^{known}
$$

while local exploration history remains unchanged.

The architecture becomes:

```text
Local Observation
      ↓
Local Memory
      ↓
Communication
      ↓
Known Map
      ↓
Communication-Aware Frontier
      ↓
Candidate Goals
      ↓
Goal Selection
```

This is the first point where communication changes autonomous planning behavior.

---

# Local Versus Known Frontier

Previously:

$$
F_i^{local}
=
f(M_i^{local})
$$

Now define:

$$
F_i^{known}
=
f(M_i^{known})
$$

Because:

$$
M_i^{local}
\subseteq
M_i^{known}
$$

the known-map frontier may lie substantially farther from the UUV's personally explored region.

A remotely explored region is treated as already known and therefore should not attract redundant exploration.

---

# Communication-Aware Information Gain

Candidate information gain must also be evaluated against the known map.

For candidate:

$$
g
$$

the communication-aware information gain is:

$$
I_i^{known}(g)
=
\left|
\{
p\in FOV(g)
\mid
M_i^{known}(p)=0
\}
\right|
$$

rather than:

$$
I_i^{local}(g)
$$

This means cells reported by another UUV are no longer incorrectly counted as new information.

---

# Redundancy Reduction

The central hypothesis is:

$$
\rho_{known}
<
\rho_{local}
$$

where:

$$
\rho
$$

represents redundant exploration.

This combo does not yet perform a long coordinated mission, but it creates the planner state required for that later comparison.

---

# Planning Memory Adapter

The existing frontier and scoring functions expect an `ExplorationMemory`-style object.

Rather than rewriting the entire planner, this combo introduces a lightweight planning-memory wrapper containing:

$$
M_i^{known}
$$

This preserves modularity while allowing the classical planner to operate on communicated knowledge.

---

# Communication-Aware Candidate Set

For each UUV:

$$
M_i^{known}
\rightarrow
F_i^{known}
\rightarrow
G_i^{known}
$$

The sampled candidates remain spatially separated using the previously validated frontier-sampling method.

---

# Communication-Aware Goal Utility

The utility remains:

$$
J_i(g)
=
w_I\hat I_i(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i(g)
$$

but:

$$
\hat I_i(g)
$$

and:

$$
\hat R_i(g)
$$

are now computed using:

$$
M_i^{known}
$$

Therefore the planner avoids spending utility on areas that are already known through communication.

---

# Comparison Against Local-Only Planning

For every UUV, this combo computes two planning decisions.

## Local-only planner

$$
g_i^{local}
$$

using:

$$
M_i^{local}
$$

## Communication-aware planner

$$
g_i^{known}
$$

using:

$$
M_i^{known}
$$

The two goals may differ.

For isolated UUV-1:

$$
M_1^{known}
=
M_1^{local}
$$

Therefore its local and communication-aware planning results should generally be identical.

For connected UUVs:

$$
M_i^{known}
\supset M_i^{local}
$$

so the selected goals may differ substantially.

---

# Goal Divergence Metric

Define:

$$
\Delta g_i
=
\|
g_i^{local}
-
g_i^{known}
\|_2
$$

If:

$$
\Delta g_i>0
$$

communication has changed that agent's current planning decision.

For an isolated agent with no remote map gain, we expect:

$$
\Delta g_i=0
$$

subject to deterministic candidate generation.

---

# Known Coverage

The planner also reports:

$$
C_i^{local}
$$

and:

$$
C_i^{known}
$$

with:

$$
C_i^{known}
\ge
C_i^{local}
$$

for every UUV.

---

# Information Integrity

Communication-aware planning may use:

- current UUV position;
- that UUV's total known exploration map;
- current known resource memory if needed later.

It must still not inspect:

- hidden resources;
- simulator-global unexplored state;
- another agent's private map directly.

Remote knowledge may influence planning only after it has been legitimately received and merged into:

$$
M_i^{known}
$$

---

# Expected Output

The code cell should:

1. create a planner-compatible wrapper around each `known_exploration_map`;
2. extract frontiers from local memory and known memory separately;
3. generate spatial candidates from each frontier;
4. score both candidate sets;
5. select:
   - local-only goal;
   - communication-aware goal;
6. report:
   - local coverage;
   - known coverage;
   - local frontier pixels;
   - known frontier pixels;
   - selected local goal;
   - selected known goal;
   - goal divergence;
7. verify:
   - known coverage is never lower than local coverage;
   - isolated agents with identical maps receive identical goals;
8. visualize one connected UUV showing:
   - local frontier;
   - communication-aware frontier;
   - local goal;
   - communication-aware goal.

A successful result establishes:

$$
\boxed{
\text{Communication}
\rightarrow
\text{Different World Knowledge}
\rightarrow
\text{Different Planning Decision}
}
$$

---

In [ ]:
# ============================================================
# Combo 24 — Communication-Aware Frontier Planning
# ============================================================


# ------------------------------------------------------------
# Lightweight planner memory wrapper
# ------------------------------------------------------------

class PlanningMemory:
    def __init__(
        self,
        uuv_id: int,
        observed_mask: np.ndarray,
    ):
        self.uuv_id = uuv_id
        self.observed = observed_mask

        self.height, self.width = (
            observed_mask.shape
        )

    def observed_count(self):
        return int(
            self.observed.sum()
        )

    def coverage(self):
        return (
            self.observed_count()
            /
            (
                self.width
                *
                self.height
            )
        )


# ------------------------------------------------------------
# Communication-aware candidate scoring
# ------------------------------------------------------------

def score_frontier_candidate_with_memory(
    agent: UUVAgent,
    candidate,
    planning_memory,
):
    x = int(
        candidate["x"]
    )

    y = int(
        candidate["y"]
    )

    fov_stats = candidate_fov_statistics(
        exploration_memory=
            planning_memory,
        candidate_x=x,
        candidate_y=y,
    )

    distance_px = euclidean_distance(
        (
            agent.state.x,
            agent.state.y,
        ),
        (
            x,
            y,
        ),
    )

    normalized_distance = (
        distance_px
        /
        MAP_DIAGONAL
    )

    information_ratio = (
        fov_stats[
            "information_ratio"
        ]
    )

    redundancy_ratio = (
        fov_stats[
            "redundancy_ratio"
        ]
    )

    utility = (
        W_INFORMATION
        *
        information_ratio

        -
        W_DISTANCE
        *
        normalized_distance

        -
        W_REDUNDANCY
        *
        redundancy_ratio
    )

    return {
        "candidate_id":
            candidate["candidate_id"],

        "x":
            x,

        "y":
            y,

        "distance_px":
            distance_px,

        "info_ratio":
            information_ratio,

        "redundancy_ratio":
            redundancy_ratio,

        "utility":
            utility,
    }


# ------------------------------------------------------------
# Helper: generate goal from arbitrary knowledge mask
# ------------------------------------------------------------

def generate_goal_from_knowledge(
    agent,
    knowledge_mask,
):
    memory = PlanningMemory(
        uuv_id=agent.uuv_id,
        observed_mask=knowledge_mask,
    )

    frontier = extract_frontier_mask(
        memory
    )

    candidates = sample_frontier_candidates(
        frontier_mask=frontier,
        reference_position=(
            agent.state.x,
            agent.state.y,
        ),
        min_spacing=
            FRONTIER_CANDIDATE_SPACING,
        max_candidates=
            MAX_FRONTIER_CANDIDATES,
    )

    if not candidates:
        return {
            "memory":
                memory,

            "frontier":
                frontier,

            "candidates":
                [],

            "scores":
                [],

            "selected":
                None,
        }


    scores = [
        score_frontier_candidate_with_memory(
            agent,
            candidate,
            memory,
        )
        for candidate in candidates
    ]

    selected = max(
        scores,
        key=lambda item:
            item["utility"],
    )


    return {
        "memory":
            memory,

        "frontier":
            frontier,

        "candidates":
            candidates,

        "scores":
            scores,

        "selected":
            selected,
    }


# ------------------------------------------------------------
# Compute local-only and communication-aware planning
# ------------------------------------------------------------

communication_planning_results = {}

comparison_reports = []


for agent in agents:

    uuv_id = (
        agent.uuv_id
    )


    # --------------------------------------------------------
    # Local-only knowledge
    # --------------------------------------------------------

    local_mask = (
        agent
        .exploration_memory
        .observed
    )


    local_result = generate_goal_from_knowledge(
        agent,
        local_mask,
    )


    # --------------------------------------------------------
    # Total known knowledge
    # --------------------------------------------------------

    known_mask = (
        known_exploration_maps[
            uuv_id
        ]
    )


    known_result = generate_goal_from_knowledge(
        agent,
        known_mask,
    )


    communication_planning_results[
        uuv_id
    ] = {
        "local":
            local_result,

        "known":
            known_result,
    }


    # --------------------------------------------------------
    # Coverage
    # --------------------------------------------------------

    local_coverage = (
        local_result[
            "memory"
        ].coverage()
    )

    known_coverage = (
        known_result[
            "memory"
        ].coverage()
    )


    if known_coverage < local_coverage:
        raise RuntimeError(
            f"UUV-{uuv_id}: "
            "known coverage is lower than local coverage."
        )


    # --------------------------------------------------------
    # Selected goals
    # --------------------------------------------------------

    local_goal = (
        local_result[
            "selected"
        ]
    )

    known_goal = (
        known_result[
            "selected"
        ]
    )


    # --------------------------------------------------------
    # Goal divergence
    # --------------------------------------------------------

    if (
        local_goal is not None
        and
        known_goal is not None
    ):

        goal_divergence = (
            euclidean_distance(
                (
                    local_goal["x"],
                    local_goal["y"],
                ),
                (
                    known_goal["x"],
                    known_goal["y"],
                ),
            )
        )

    elif (
        local_goal is None
        and
        known_goal is None
    ):

        goal_divergence = 0.0

    else:

        goal_divergence = np.nan


    comparison_reports.append(
        {
            "uuv_id":
                uuv_id,

            "neighbors":
                current_comm_graph.degree[
                    uuv_id
                ],

            "local_coverage_pct":
                local_coverage * 100,

            "known_coverage_pct":
                known_coverage * 100,

            "local_frontier_pixels":
                int(
                    local_result[
                        "frontier"
                    ].sum()
                ),

            "known_frontier_pixels":
                int(
                    known_result[
                        "frontier"
                    ].sum()
                ),

            "local_goal_x":
                None
                if local_goal is None
                else local_goal["x"],

            "local_goal_y":
                None
                if local_goal is None
                else local_goal["y"],

            "known_goal_x":
                None
                if known_goal is None
                else known_goal["x"],

            "known_goal_y":
                None
                if known_goal is None
                else known_goal["y"],

            "goal_divergence_px":
                goal_divergence,
        }
    )


comparison_df = pd.DataFrame(
    comparison_reports
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    "=" * 96
)

print(
    "MultiUUV-Link — Communication-Aware "
    "Frontier Planning"
)

print(
    "=" * 96
)

print(
    "\nLocal-only versus known-map planning:"
)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "local_coverage_pct":
                lambda x: f"{x:.4f}",

            "known_coverage_pct":
                lambda x: f"{x:.4f}",

            "goal_divergence_px":
                lambda x: (
                    "N/A"
                    if pd.isna(x)
                    else f"{x:.2f}"
                ),
        },
    )
)


# ------------------------------------------------------------
# Isolated-agent consistency
# ------------------------------------------------------------

print(
    "\nIsolated-agent consistency:"
)

for agent in agents:

    uuv_id = agent.uuv_id

    if (
        current_comm_graph.degree[
            uuv_id
        ]
        !=
        0
    ):
        continue


    local_result = (
        communication_planning_results[
            uuv_id
        ][
            "local"
        ]
    )

    known_result = (
        communication_planning_results[
            uuv_id
        ][
            "known"
        ]
    )


    maps_equal = np.array_equal(
        local_result[
            "memory"
        ].observed,
        known_result[
            "memory"
        ].observed,
    )


    local_goal = (
        local_result[
            "selected"
        ]
    )

    known_goal = (
        known_result[
            "selected"
        ]
    )


    if (
        local_goal is None
        and
        known_goal is None
    ):
        goals_equal = True

    elif (
        local_goal is not None
        and
        known_goal is not None
    ):

        goals_equal = (
            local_goal["x"]
            ==
            known_goal["x"]
            and
            local_goal["y"]
            ==
            known_goal["y"]
        )

    else:
        goals_equal = False


    print(
        f"  UUV-{uuv_id}: "
        f"maps_equal={maps_equal}, "
        f"goals_equal={goals_equal}"
    )


    if not maps_equal:
        raise RuntimeError(
            f"Isolated UUV-{uuv_id} "
            "known map differs from local map."
        )

    if not goals_equal:
        raise RuntimeError(
            f"Isolated UUV-{uuv_id} "
            "planning changed without communication."
        )


# ------------------------------------------------------------
# Count communication-induced goal changes
# ------------------------------------------------------------

changed_goal_count = 0

for report in comparison_reports:

    divergence = (
        report[
            "goal_divergence_px"
        ]
    )

    if (
        not pd.isna(
            divergence
        )
        and
        divergence > 0
    ):
        changed_goal_count += 1


print(
    "\nCommunication planning effect:"
)

print(
    f"  UUVs with changed selected goal : "
    f"{changed_goal_count} / {NUM_UUVS}"
)


# ------------------------------------------------------------
# Choose representative connected UUV
# ------------------------------------------------------------

connected_reports = [
    report
    for report in comparison_reports
    if report["neighbors"] > 0
]

representative = max(
    connected_reports,
    key=lambda item:
        (
            item[
                "known_coverage_pct"
            ]
            -
            item[
                "local_coverage_pct"
            ]
        ),
)

representative_id = (
    representative[
        "uuv_id"
    ]
)

rep_agent = (
    agent_by_id[
        representative_id
    ]
)

rep_results = (
    communication_planning_results[
        representative_id
    ]
)

local_frontier = (
    rep_results[
        "local"
    ][
        "frontier"
    ]
)

known_frontier = (
    rep_results[
        "known"
    ][
        "frontier"
    ]
)

local_goal = (
    rep_results[
        "local"
    ][
        "selected"
    ]
)

known_goal = (
    rep_results[
        "known"
    ][
        "selected"
    ]
)


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


# Local frontier
local_y, local_x = np.where(
    local_frontier
)

plt.scatter(
    local_x,
    local_y,
    s=3,
    marker=".",
    label="Local-memory frontier",
    zorder=4,
)


# Known frontier
known_y, known_x = np.where(
    known_frontier
)

plt.scatter(
    known_x,
    known_y,
    s=4,
    marker=".",
    label="Communication-aware frontier",
    zorder=5,
)


# Local goal
if local_goal is not None:

    plt.scatter(
        local_goal["x"],
        local_goal["y"],
        s=200,
        marker="D",
        facecolors="none",
        linewidths=2,
        label="Local-only goal",
        zorder=8,
    )


# Communication-aware goal
if known_goal is not None:

    plt.scatter(
        known_goal["x"],
        known_goal["y"],
        s=260,
        marker="*",
        edgecolors="black",
        linewidths=1.5,
        label="Known-map goal",
        zorder=9,
    )


# UUV
plt.scatter(
    rep_agent.state.x,
    rep_agent.state.y,
    s=150,
    marker="x",
    linewidths=2.5,
    label=f"UUV-{representative_id}",
    zorder=10,
)


plt.title(
    f"MultiUUV-Link — UUV-{representative_id} "
    "Communication-Aware Planning"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend(
    loc="best"
)

plt.tight_layout()

plt.show()


print(
    "\nCommunication-aware frontier planning validated."
)

# Combo 24.5 — Experimental State Checkpoint and Restore

## Objective

The upcoming experiments compare different cooperative exploration controllers.

Because Jupyter notebook objects are mutable, running one experiment changes:

- UUV positions;
- trajectories;
- local exploration memories;
- known exploration maps;
- resource memories;
- resource provenance.

Therefore sequential experiments would begin from different states and would not constitute a controlled comparison.

This combo creates a reusable checkpoint immediately after Combo 24.

The checkpoint stores the complete experiment-relevant swarm state:

$$
S_{checkpoint}
=
\{
Agents,
M^{known},
P^{resource}
\}
$$

where:

- `Agents` contains UUV state, trajectories, local exploration memories, and resource memories;
- $M^{known}$ contains communication-derived exploration knowledge;
- $P^{resource}$ contains resource provenance.

---

## Controlled Experimental Branching

The desired experimental structure is:

```text
Combo 24
   ↓
Checkpoint
   ├────→ Combo 25 baseline
   │
   └────→ Restore checkpoint
              ↓
           Revised Combo 27
```

Therefore both controllers begin from exactly the same swarm state.

---

## Deep Copy Requirement

A normal Python assignment such as:

```python
backup = agents
```

would not be sufficient because both variables would reference the same mutable objects.

Instead we require:

```python
deepcopy(...)
```

so the checkpoint contains physically independent copies.

---

## Restore Operation

Restoring the checkpoint must recreate:

- all agent objects;
- positions;
- trajectories;
- local exploration memories;
- resource memories;
- known exploration maps;
- resource provenance;
- agent lookup table;
- current communication graph.

After restoration, later experiments may mutate the restored swarm without altering the stored checkpoint.

---

## Validation

Immediately after saving, the combo computes a compact state signature containing:

- UUV positions;
- trajectory lengths;
- local observed-cell counts;
- known-map cell counts;
- known resource counts.

After every restore, the same signature must be recovered exactly.

Thus:

$$
Signature(S_{restored})
=
Signature(S_{checkpoint})
$$

A successful result establishes a reproducible experimental branch point for future controller comparisons.

---

In [ ]:
# ============================================================
# Combo 24.5 — Experimental State Checkpoint & Restore
# ============================================================

from copy import deepcopy


# ------------------------------------------------------------
# State-signature helper
# ------------------------------------------------------------

def get_swarm_state_signature():
    """
    Compact deterministic signature of experiment-relevant
    swarm state.
    """

    signature = []

    for agent in sorted(
        agents,
        key=lambda a: a.uuv_id,
    ):

        uuv_id = agent.uuv_id

        signature.append(
            {
                "uuv_id":
                    uuv_id,

                "position":
                    (
                        agent.state.x,
                        agent.state.y,
                    ),

                "trajectory_length":
                    len(
                        agent.trajectory
                    ),

                "local_cells":
                    agent
                    .exploration_memory
                    .observed_count(),

                "known_cells":
                    int(
                        known_exploration_maps[
                            uuv_id
                        ].sum()
                    ),

                "known_resources":
                    len(
                        agent.resource_memory
                    ),
            }
        )

    return signature


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

CHECKPOINT_24_5 = {
    "agents":
        deepcopy(
            agents
        ),

    "known_exploration_maps":
        deepcopy(
            known_exploration_maps
        ),

    "resource_provenance":
        deepcopy(
            resource_provenance
        ),
}


CHECKPOINT_24_5_SIGNATURE = (
    get_swarm_state_signature()
)


# ------------------------------------------------------------
# Restore function
# ------------------------------------------------------------

def restore_checkpoint_24_5():

    global agents
    global known_exploration_maps
    global resource_provenance
    global agent_by_id
    global current_positions
    global current_comm_graph


    # --------------------------------------------------------
    # Restore independent copies
    # --------------------------------------------------------

    agents = deepcopy(
        CHECKPOINT_24_5[
            "agents"
        ]
    )

    known_exploration_maps = deepcopy(
        CHECKPOINT_24_5[
            "known_exploration_maps"
        ]
    )

    resource_provenance = deepcopy(
        CHECKPOINT_24_5[
            "resource_provenance"
        ]
    )


    # --------------------------------------------------------
    # Rebuild derived lookup structures
    # --------------------------------------------------------

    agent_by_id = {
        agent.uuv_id:
            agent
        for agent in agents
    }


    current_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }


    current_comm_graph = (
        build_communication_graph(
            current_positions,
            communication_range=
                COMMUNICATION_RANGE,
        )
    )


    # --------------------------------------------------------
    # Validate exact restoration
    # --------------------------------------------------------

    restored_signature = (
        get_swarm_state_signature()
    )


    restoration_passed = (
        restored_signature
        ==
        CHECKPOINT_24_5_SIGNATURE
    )


    if not restoration_passed:

        raise RuntimeError(
            "Checkpoint restoration failed: "
            "restored state does not match "
            "Combo 24.5 checkpoint."
        )


    print(
        "=" * 76
    )

    print(
        "MultiUUV-Link — Experimental State Restored"
    )

    print(
        "=" * 76
    )

    print(
        "\nCheckpoint restoration : PASS"
    )

    print(
        "\nRestored swarm state:"
    )


    restored_df = pd.DataFrame(
        restored_signature
    )

    print(
        restored_df.to_string(
            index=False
        )
    )


    return restored_signature


# ------------------------------------------------------------
# Initial checkpoint report
# ------------------------------------------------------------

print(
    "=" * 76
)

print(
    "MultiUUV-Link — Experimental Checkpoint 24.5"
)

print(
    "=" * 76
)

print(
    "\nCheckpoint saved successfully."
)

print(
    "\nCheckpoint state:"
)


checkpoint_df = pd.DataFrame(
    CHECKPOINT_24_5_SIGNATURE
)

print(
    checkpoint_df.to_string(
        index=False
    )
)


print(
    "\nCombo 24.5 experimental checkpoint ready."
)

print(
    "\nTo restore this exact state later, run:"
)

print(
    "    restore_checkpoint_24_5()"
)

# Combo 25 — Dynamic Communication-Aware Cooperative Exploration

## Objective

Previous combos independently validated:

- autonomous frontier exploration;
- predictive multi-UUV safety;
- dynamic range-limited communication graphs;
- resource-knowledge exchange;
- exploration-map exchange;
- communication-aware frontier planning.

This combo integrates those mechanisms into one synchronized mission.

At every timestep, the swarm will:

```text
Current UUV Positions
        ↓
Build Communication Graph
        ↓
Exchange Knowledge Across Current Links
        ↓
Update Each UUV's Known Map
        ↓
Plan Using Known Knowledge
        ↓
Generate Desired Actions
        ↓
Predictive Safety Shield
        ↓
Execute Safe Joint Motion
        ↓
Local Observation
        ↓
Local Resource Discovery
        ↓
Repeat
```

This is the first complete communication-aware decentralized mission loop.

There is still:

```text
NO distributed task auction
NO explicit consensus protocol
NO packet loss
NO latency
NO communication bandwidth limit
NO learned component
```

Those layers will be introduced later.

---

# Two Spatial Memory Layers

Each UUV maintains:

$$
M_i^{local}
$$

and:

$$
M_i^{known}
$$

where:

- $M_i^{local}$ contains personally observed cells;
- $M_i^{known}$ contains both local and communicated exploration knowledge.

The invariant is:

$$
M_i^{local}
\subseteq
M_i^{known}
$$

at all times.

Local sensing updates both:

$$
M_i^{local}(t+1)
=
M_i^{local}(t)
\cup
FOV_i(t+1)
$$

and:

$$
M_i^{known}(t+1)
=
M_i^{known}(t)
\cup
M_i^{local}(t+1)
$$

Communication may expand only:

$$
M_i^{known}
$$

without modifying the historical local-observation map.

---

# Dynamic Communication

At every synchronized mission step, the graph is recomputed:

$$
G_t=(V,E_t)
$$

with:

$$
(i,j)\in E_t
\iff
\|p_i-p_j\|_2
\le R_c
$$

using:

$$
R_c=400\text{ px}
$$

Therefore communication availability naturally changes as the vehicles move.

---

# Resource Knowledge Exchange

Each communication edge permits direct exchange of already-known resource records.

For one synchronous communication round:

$$
D_i^0
\rightarrow
Messages
\rightarrow
D_i^1
$$

All messages are generated from pre-round snapshots.

Information received during a round is therefore not retransmitted until a later mission timestep.

This preserves explicit one-hop temporal propagation.

---

# Exploration-Knowledge Exchange

Similarly:

$$
M_i^{known,0}
\rightarrow
Messages
\rightarrow
M_i^{known,1}
$$

with the merge:

$$
M_j^{known,+}
=
M_j^{known}
\lor
M_i^{known}
$$

Communication therefore gradually propagates explored-region knowledge through the dynamic network.

---

# Communication-Aware Planning

Each agent now computes its frontier using:

$$
M_i^{known}
$$

rather than only:

$$
M_i^{local}
$$

Thus:

$$
F_i(t)
=
f(M_i^{known}(t))
$$

Candidate goals are sampled:

$$
G_i(t)
=
h(F_i(t))
$$

and evaluated with:

$$
J_i(g)
=
w_I\hat I_i^{known}(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i^{known}(g)
$$

The selected goal is:

$$
g_i^*
=
\arg\max_{g\in G_i}J_i(g)
$$

---

# Goal Persistence

A frontier goal persists across timesteps.

A new goal is generated when:

1. no active goal exists;
2. the previous goal is reached;
3. the goal is no longer a frontier under the updated known map.

The third condition is important.

Communication may tell a UUV that another agent has already explored the region toward which it was traveling.

Therefore:

$$
g_i(t)\notin F_i^{known}(t)
$$

causes the stale goal to be discarded and replanned.

This is the first explicit **communication-triggered replanning mechanism**.

---

# Safety Layer

After each UUV chooses a desired action:

$$
a_i^{des}
$$

the predictive joint safety shield determines:

$$
a_i^{exec}
$$

such that:

$$
d_{ij}^{pred}
\ge128\text{ px}
$$

for every UUV pair whenever a feasible joint action exists.

Thus communication improves mission efficiency while the safety layer remains responsible for physical separation.

---

# Mission Horizon

The integrated experiment runs for:

$$
T=180
$$

synchronized timesteps.

This is longer than Combo 20 because communication-aware exploration should have enough time for:

- graph reconnections;
- knowledge propagation;
- replanning;
- divergence of private knowledge states;
- eventual partial convergence.

---

# Communication Metrics

At every timestep record:

### Active graph edges

$$
|E_t|
$$

### Map cells remotely acquired

$$
\Delta A_i^{remote}(t)
$$

### Remote resource discoveries

$$
\Delta D_i^{remote}(t)
$$

### Goal invalidations caused by communication

$$
N_i^{replan,comm}
$$

A communication-triggered replan occurs when an active goal remains physically unreached but stops being a frontier because newly received map knowledge renders it already-known.

---

# Exploration Metrics

For every UUV record:

### Local coverage

$$
C_i^{local}
$$

### Total known coverage

$$
C_i^{known}
$$

### Remote knowledge fraction

$$
C_i^{remote}
=
C_i^{known}
-
C_i^{local}
$$

---

# Team Coverage

Define actual team exploration coverage from the union of local observation maps:

$$
M_{team}
=
\bigvee_i
M_i^{local}
$$

Then:

$$
C_{team}
=
\frac{|M_{team}|}{HW}
$$

This is an evaluation metric only.

No agent receives:

$$
M_{team}
$$

unless the information reaches it through legitimate communication.

---

# Redundant Exploration

The sum of local explored cells is:

$$
A_{\Sigma}
=
\sum_i|M_i^{local}|
$$

while unique team explored area is:

$$
A_{team}
=
\left|
\bigvee_iM_i^{local}
\right|
$$

Define redundant explored-cell count:

$$
A_{redundant}
=
A_{\Sigma}-A_{team}
$$

and redundancy ratio:

$$
\rho
=
\frac{A_{redundant}}{A_{\Sigma}}
$$

This is one of the most important metrics in MultiUUV-Link.

Communication-aware planning should eventually reduce:

$$
\rho
$$

relative to independent exploration.

---

# Information Integrity

The simulator may calculate team-level quantities for evaluation.

Individual agents may use only:

- their current state;
- their local exploration memory;
- their communicated known map;
- their known resources;
- current communication links;
- information explicitly received through those links.

The global team union remains debug/evaluation ground truth.

---

# Expected Output

The code cell should:

1. run 180 synchronized timesteps;
2. rebuild the communication graph every timestep;
3. exchange map knowledge using synchronous one-hop semantics;
4. exchange resource knowledge using synchronous one-hop semantics;
5. update total known maps;
6. invalidate stale goals when communication makes them obsolete;
7. plan using total known knowledge;
8. apply predictive collision safety;
9. execute all UUV actions synchronously;
10. update local observations and resource discoveries;
11. report:
    - final local coverage per UUV;
    - final known coverage per UUV;
    - remote knowledge acquired;
    - resources known;
    - communication-triggered replans;
    - safety overrides;
    - team coverage;
    - redundancy ratio;
    - mean communication edges;
12. visualize:
    - complete cooperative trajectories;
    - communication-edge history;
    - team local-coverage union.

A successful result establishes:

$$
\boxed{
\text{Dynamic Communication-Aware Decentralized Multi-UUV Exploration}
}
$$

---

In [ ]:
# ============================================================
# Combo 25 — Dynamic Communication-Aware Cooperative Exploration
# ============================================================


# ------------------------------------------------------------
# Mission configuration
# ------------------------------------------------------------

COOPERATIVE_MISSION_STEPS = 180


# ------------------------------------------------------------
# Ensure known maps contain current local knowledge
# ------------------------------------------------------------

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ------------------------------------------------------------
# Mission-start snapshots
# ------------------------------------------------------------

coop_start = {}

for agent in agents:

    uuv_id = agent.uuv_id

    coop_start[
        uuv_id
    ] = {
        "local_cells":
            agent.exploration_memory.observed_count(),

        "known_cells":
            int(
                known_exploration_maps[
                    uuv_id
                ].sum()
            ),

        "known_resources":
            len(
                agent.resource_memory
            ),

        "trajectory_index":
            len(
                agent.trajectory
            ) - 1,
    }


# ------------------------------------------------------------
# Active frontier goals
# ------------------------------------------------------------

coop_active_goals = {
    agent.uuv_id: None
    for agent in agents
}


# ------------------------------------------------------------
# Mission logs
# ------------------------------------------------------------

coop_map_remote_gain = {
    agent.uuv_id: 0
    for agent in agents
}

coop_resource_remote_gain = {
    agent.uuv_id: 0
    for agent in agents
}

coop_comm_replans = {
    agent.uuv_id: 0
    for agent in agents
}

coop_safety_overrides = {
    agent.uuv_id: 0
    for agent in agents
}

coop_edge_history = []

coop_team_coverage_history = []

coop_redundancy_history = []

coop_mission_min_separation = np.inf


# ------------------------------------------------------------
# Helper: frontier goal from TOTAL KNOWN map
# ------------------------------------------------------------

def generate_known_map_goal(
    agent,
    known_map,
):

    result = generate_goal_from_knowledge(
        agent,
        known_map,
    )

    return result[
        "selected"
    ]


# ------------------------------------------------------------
# Helper: direct one-hop resource communication round
# ------------------------------------------------------------

def exchange_resource_knowledge_one_hop(
    graph,
):

    # Pre-round snapshots
    snapshots = {
        agent.uuv_id: {
            resource.resource_id:
                resource
            for resource in
            agent.resource_memory.all_resources()
        }
        for agent in agents
    }

    messages = []

    for uuv_a, uuv_b in graph.edges():

        messages.append(
            (
                uuv_a,
                uuv_b,
                list(
                    snapshots[
                        uuv_a
                    ].values()
                ),
            )
        )

        messages.append(
            (
                uuv_b,
                uuv_a,
                list(
                    snapshots[
                        uuv_b
                    ].values()
                ),
            )
        )


    gains = {
        agent.uuv_id: 0
        for agent in agents
    }


    for sender, receiver, payload in messages:

        before = len(
            agent_by_id[
                receiver
            ].resource_memory
        )

        newly_learned = (
            agent_by_id[
                receiver
            ]
            .resource_memory
            .update(
                payload
            )
        )

        for resource in newly_learned:

            if (
                resource.resource_id
                not in
                resource_provenance[
                    receiver
                ]
            ):
                resource_provenance[
                    receiver
                ][
                    resource.resource_id
                ] = "REMOTE"


        after = len(
            agent_by_id[
                receiver
            ].resource_memory
        )

        gains[
            receiver
        ] += (
            after - before
        )


    return gains


# ------------------------------------------------------------
# Helper: direct one-hop map communication round
# ------------------------------------------------------------

def exchange_map_knowledge_one_hop(
    graph,
):

    snapshots = {
        uuv_id:
            known_map.copy()
        for uuv_id, known_map
        in known_exploration_maps.items()
    }

    messages = []

    for uuv_a, uuv_b in graph.edges():

        messages.append(
            (
                uuv_a,
                uuv_b,
                snapshots[
                    uuv_a
                ].copy(),
            )
        )

        messages.append(
            (
                uuv_b,
                uuv_a,
                snapshots[
                    uuv_b
                ].copy(),
            )
        )


    gains = {
        agent.uuv_id: 0
        for agent in agents
    }


    for sender, receiver, payload in messages:

        before = int(
            known_exploration_maps[
                receiver
            ].sum()
        )

        known_exploration_maps[
            receiver
        ] = np.logical_or(
            known_exploration_maps[
                receiver
            ],
            payload,
        )

        after = int(
            known_exploration_maps[
                receiver
            ].sum()
        )

        gains[
            receiver
        ] += (
            after - before
        )


    return gains


# ============================================================
# Cooperative synchronized mission
# ============================================================

for mission_step in range(
    COOPERATIVE_MISSION_STEPS
):

    # --------------------------------------------------------
    # 1. Build current communication graph
    # --------------------------------------------------------

    positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }

    comm_graph = (
        build_communication_graph(
            positions,
            communication_range=
                COMMUNICATION_RANGE,
        )
    )

    coop_edge_history.append(
        comm_graph.number_of_edges()
    )


    # --------------------------------------------------------
    # 2. Save pre-communication known maps
    #    for stale-goal detection
    # --------------------------------------------------------

    known_before_comm = {
        uuv_id:
            known_map.copy()
        for uuv_id, known_map
        in known_exploration_maps.items()
    }


    # --------------------------------------------------------
    # 3. Exchange exploration-map knowledge
    # --------------------------------------------------------

    map_gains = (
        exchange_map_knowledge_one_hop(
            comm_graph
        )
    )

    for uuv_id, gain in map_gains.items():

        coop_map_remote_gain[
            uuv_id
        ] += gain


    # --------------------------------------------------------
    # 4. Exchange resource knowledge
    # --------------------------------------------------------

    resource_gains = (
        exchange_resource_knowledge_one_hop(
            comm_graph
        )
    )

    for uuv_id, gain in resource_gains.items():

        coop_resource_remote_gain[
            uuv_id
        ] += gain


    # --------------------------------------------------------
    # 5. Validate / invalidate active goals
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = (
            agent.uuv_id
        )

        goal = (
            coop_active_goals[
                uuv_id
            ]
        )

        if goal is None:
            continue


        # ----------------------------------------------------
        # Goal reached?
        # ----------------------------------------------------

        distance_to_goal = (
            euclidean_distance(
                (
                    agent.state.x,
                    agent.state.y,
                ),
                (
                    goal["x"],
                    goal["y"],
                ),
            )
        )

        if (
            distance_to_goal
            <=
            GOAL_REACHED_RADIUS
        ):

            coop_active_goals[
                uuv_id
            ] = None

            continue


        # ----------------------------------------------------
        # Is goal still frontier under NEW known map?
        # ----------------------------------------------------

        new_planning_memory = (
            PlanningMemory(
                uuv_id=uuv_id,
                observed_mask=
                    known_exploration_maps[
                        uuv_id
                    ],
            )
        )

        new_frontier = (
            extract_frontier_mask(
                new_planning_memory
            )
        )


        goal_x = int(
            goal["x"]
        )

        goal_y = int(
            goal["y"]
        )


        goal_still_frontier = (
            0 <= goal_x < WIDTH
            and
            0 <= goal_y < HEIGHT
            and
            new_frontier[
                goal_y,
                goal_x
            ]
        )


        if not goal_still_frontier:

            # Determine whether communication actually changed
            # knowledge this timestep.
            communication_changed_map = (
                not np.array_equal(
                    known_before_comm[
                        uuv_id
                    ],
                    known_exploration_maps[
                        uuv_id
                    ],
                )
            )

            if communication_changed_map:

                coop_comm_replans[
                    uuv_id
                ] += 1


            coop_active_goals[
                uuv_id
            ] = None


    # --------------------------------------------------------
    # 6. Generate fresh goals where needed
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = agent.uuv_id

        if (
            coop_active_goals[
                uuv_id
            ]
            is None
        ):

            coop_active_goals[
                uuv_id
            ] = generate_known_map_goal(
                agent,
                known_exploration_maps[
                    uuv_id
                ],
            )


    # --------------------------------------------------------
    # 7. Desired planner actions
    # --------------------------------------------------------

    desired_actions = []

    for agent in agents:

        goal = (
            coop_active_goals[
                agent.uuv_id
            ]
        )

        if goal is None:

            desired_action = (
                ACTION_HOLD
            )

        else:

            desired_action = (
                choose_greedy_action(
                    agent.state,
                    goal_x=
                        goal["x"],
                    goal_y=
                        goal["y"],
                )
            )

            if desired_action is None:
                desired_action = ACTION_HOLD


        desired_actions.append(
            desired_action
        )


    # --------------------------------------------------------
    # 8. Predictive safety shield
    # --------------------------------------------------------

    states = [
        agent.state
        for agent in agents
    ]

    shield_result = safety_shield(
        states=states,
        desired_actions=
            desired_actions,
        safe_separation=
            SAFE_SEPARATION,
        step_size=
            ROLLOUT_STEP_SIZE,
    )

    executed_actions = (
        shield_result[
            "executed_actions"
        ]
    )

    coop_mission_min_separation = min(
        coop_mission_min_separation,
        shield_result[
            "minimum_distance"
        ],
    )


    # --------------------------------------------------------
    # 9. Count interventions
    # --------------------------------------------------------

    for agent, desired, executed in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if desired != executed:

            coop_safety_overrides[
                agent.uuv_id
            ] += 1


    # --------------------------------------------------------
    # 10. Compute all next states
    # --------------------------------------------------------

    next_states = []

    for agent, action in zip(
        agents,
        executed_actions,
    ):

        next_states.append(
            predict_safety_state(
                state=
                    agent.state,
                action=
                    action,
                step_size=
                    ROLLOUT_STEP_SIZE,
            )
        )


    # --------------------------------------------------------
    # 11. Simultaneous state update
    # --------------------------------------------------------

    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = (
            next_state
        )

        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # --------------------------------------------------------
    # 12. Independent LOCAL observations
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = (
            agent.uuv_id
        )


        # Local exploration
        agent.exploration_memory.update_from_fov(
            agent.state
        )


        # Known map must always include local map
        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


        # Local resource discovery
        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )

        newly_local = (
            agent.resource_memory.update(
                visible_resources
            )
        )


        # Upgrade provenance when personally observed
        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # --------------------------------------------------------
    # 13. Team-level EVALUATION metrics only
    # --------------------------------------------------------

    team_union = np.logical_or.reduce(
        [
            agent
            .exploration_memory
            .observed
            for agent in agents
        ]
    )

    team_coverage = (
        team_union.sum()
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )

    sum_local_cells = sum(
        agent
        .exploration_memory
        .observed_count()
        for agent in agents
    )

    team_unique_cells = int(
        team_union.sum()
    )

    redundant_cells = (
        sum_local_cells
        -
        team_unique_cells
    )

    redundancy_ratio = (
        redundant_cells
        /
        sum_local_cells
        if sum_local_cells > 0
        else 0.0
    )


    coop_team_coverage_history.append(
        float(
            team_coverage
        )
    )

    coop_redundancy_history.append(
        float(
            redundancy_ratio
        )
    )


# ============================================================
# Final validation
# ============================================================

if (
    coop_mission_min_separation
    <
    SAFE_SEPARATION
):
    raise RuntimeError(
        "Cooperative mission violated "
        "minimum safety separation."
    )


# ------------------------------------------------------------
# Final reports
# ------------------------------------------------------------

coop_reports = []

for agent in agents:

    uuv_id = agent.uuv_id

    local_cells = (
        agent
        .exploration_memory
        .observed_count()
    )

    known_cells = int(
        known_exploration_maps[
            uuv_id
        ].sum()
    )

    total_cells = (
        WIDTH
        *
        HEIGHT
    )

    coop_reports.append(
        {
            "uuv_id":
                uuv_id,

            "local_coverage_pct":
                local_cells
                /
                total_cells
                *
                100,

            "known_coverage_pct":
                known_cells
                /
                total_cells
                *
                100,

            "remote_cells_gained":
                coop_map_remote_gain[
                    uuv_id
                ],

            "known_resources":
                len(
                    agent.resource_memory
                ),

            "remote_resources_gained":
                coop_resource_remote_gain[
                    uuv_id
                ],

            "comm_replans":
                coop_comm_replans[
                    uuv_id
                ],

            "safety_overrides":
                coop_safety_overrides[
                    uuv_id
                ],
        }
    )


coop_df = pd.DataFrame(
    coop_reports
)


# ------------------------------------------------------------
# Final team metrics
# ------------------------------------------------------------

final_team_union = np.logical_or.reduce(
    [
        agent
        .exploration_memory
        .observed
        for agent in agents
    ]
)

final_team_cells = int(
    final_team_union.sum()
)

final_team_coverage = (
    final_team_cells
    /
    (
        WIDTH
        *
        HEIGHT
    )
)

final_sum_local = sum(
    agent
    .exploration_memory
    .observed_count()
    for agent in agents
)

final_redundant_cells = (
    final_sum_local
    -
    final_team_cells
)

final_redundancy_ratio = (
    final_redundant_cells
    /
    final_sum_local
)


# ------------------------------------------------------------
# Console report
# ------------------------------------------------------------

print(
    "=" * 100
)

print(
    "MultiUUV-Link — Dynamic Communication-Aware "
    "Cooperative Exploration"
)

print(
    "=" * 100
)

print(
    f"\nMission steps              : "
    f"{COOPERATIVE_MISSION_STEPS}"
)

print(
    f"Communication range        : "
    f"{COMMUNICATION_RANGE:.2f} px"
)

print(
    f"Mean active comm edges     : "
    f"{np.mean(coop_edge_history):.3f}"
)

print(
    f"Mission minimum separation : "
    f"{coop_mission_min_separation:.2f} px"
)

print(
    f"Final team coverage        : "
    f"{100 * final_team_coverage:.4f}%"
)

print(
    f"Final redundant cells      : "
    f"{final_redundant_cells}"
)

print(
    f"Final redundancy ratio     : "
    f"{final_redundancy_ratio:.4f}"
)


print(
    "\nPer-UUV cooperative mission state:"
)

print(
    coop_df.to_string(
        index=False,
        formatters={
            "local_coverage_pct":
                lambda x: f"{x:.4f}",

            "known_coverage_pct":
                lambda x: f"{x:.4f}",
        },
    )
)


# ------------------------------------------------------------
# Communication/replanning summary
# ------------------------------------------------------------

print(
    "\nCommunication effects:"
)

print(
    f"  Total remote map cells learned : "
    f"{sum(coop_map_remote_gain.values())}"
)

print(
    f"  Total remote resources learned : "
    f"{sum(coop_resource_remote_gain.values())}"
)

print(
    f"  Communication-triggered replans: "
    f"{sum(coop_comm_replans.values())}"
)

print(
    f"  Total safety overrides         : "
    f"{sum(coop_safety_overrides.values())}"
)


# ------------------------------------------------------------
# Trajectory visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)

for agent in agents:

    start_index = (
        coop_start[
            agent.uuv_id
        ][
            "trajectory_index"
        ]
    )

    path = np.asarray(
        agent.trajectory[
            start_index:
        ]
    )

    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )

    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=80,
    )

    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=120,
        linewidths=2.5,
    )


plt.title(
    "MultiUUV-Link — Dynamic Communication-Aware "
    "Cooperative Exploration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Communication edge history
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    coop_edge_history,
    linewidth=2,
)

plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Active communication edges"
)

plt.title(
    "MultiUUV-Link — Communication Connectivity "
    "During Cooperative Mission"
)

plt.yticks(
    range(
        0,
        7,
    )
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Team coverage + redundancy
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    np.asarray(
        coop_team_coverage_history
    ) * 100,
    label="Team coverage [%]",
    linewidth=2,
)

plt.plot(
    np.asarray(
        coop_redundancy_history
    ) * 100,
    label="Redundancy ratio [%]",
    linewidth=2,
)

plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Percent"
)

plt.title(
    "MultiUUV-Link — Cooperative Coverage "
    "and Redundancy"
)

plt.legend()

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# Final team coverage visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    final_team_union,
    cmap="gray",
    vmin=0,
    vmax=1,
)

plt.title(
    "MultiUUV-Link — Final Unique Team "
    "Exploration Coverage"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.tight_layout()

plt.show()


print(
    "\nDynamic communication-aware cooperative "
    "exploration validated."
)

# Combo 26 — Distributed Frontier Goal Allocation

## Objective

Combo 25 demonstrated successful communication-aware cooperative exploration, reaching:

$$
98.4893\%
$$

unique team coverage.

However, the final redundancy ratio was:

$$
\rho=0.5640
$$

which means a large fraction of personally explored area overlapped with regions explored by other UUVs.

The underlying reason is that communication currently changes each UUV's world knowledge, but every UUV still selects its frontier goal independently.

Thus two communicating UUVs may simultaneously choose nearby goals:

$$
g_i^*
\approx
g_j^*
$$

even though their maps are synchronized.

This combo introduces explicit **distributed frontier task allocation**.

The purpose is:

```text
Shared Knowledge
      ↓
Candidate Frontier Tasks
      ↓
Agent-Task Utilities
      ↓
Distributed Allocation
      ↓
Distinct Assigned Goals
```

No movement occurs in this combo.

The goal is to validate task assignment independently before integrating it into the mission loop.

---

# Communication Components

At the current timestep the communication graph is:

$$
G_t=(V,E_t)
$$

Task allocation is performed independently inside each connected communication component.

For connected component:

$$
\mathcal{C}_k
\subseteq V
$$

the agents may coordinate because information can propagate through the component.

Agents in different disconnected components cannot participate in the same allocation round.

Therefore:

$$
Allocation_k
=
f(
\mathcal{C}_k
)
$$

rather than one centralized global allocation over all UUVs.

This preserves the decentralized communication constraint.

---

# Shared Planning Knowledge

Agents that have exchanged map information maintain:

$$
M_i^{known}
$$

For agents belonging to the same connected communication component, their maps may be similar or identical after sufficient communication.

Each agent still computes candidate utility from its own legitimate known map.

The simulator does not create a global planning map for them.

---

# Frontier Tasks

For each UUV $i$, communication-aware frontier candidates are:

$$
G_i=
\{
g_{i,1},
g_{i,2},
\ldots
\}
$$

To perform allocation across a connected component, candidates from all agents in that component are combined into a task pool:

$$
\mathcal{T}_k
=
\bigcup_{i\in\mathcal{C}_k}
G_i
$$

However, several agents may independently generate nearly identical frontier points.

Therefore the candidate pool must be spatially deduplicated.

---

# Task Deduplication

Two frontier tasks:

$$
g_a
$$

and:

$$
g_b
$$

are treated as the same task when:

$$
\|g_a-g_b\|_2
<
d_{task}
$$

For the initial prototype:

$$
d_{task}=128\text{ px}
$$

equal to one FOV width.

A greedy spatial filtering procedure retains one representative task from each nearby group.

Thus:

$$
|\mathcal{T}_{dedup}|
\le
|\mathcal{T}_{raw}|
$$

---

# Agent-Task Utility

Every agent evaluates every task in its component.

For UUV $i$ and task $g$:

$$
J_{i,g}
=
w_I\hat I_i(g)
-
w_D\hat D_i(g)
-
w_R\hat R_i(g)
$$

using the previously validated communication-aware utility.

This gives a utility matrix:

$$
\mathbf{J}
=
\begin{bmatrix}
J_{1,1} & J_{1,2} & \cdots\\
J_{2,1} & J_{2,2} & \cdots\\
\vdots & \vdots & \ddots
\end{bmatrix}
$$

Rows correspond to UUVs.

Columns correspond to candidate frontier tasks.

---

# Distributed Greedy Auction

The first allocation baseline uses a greedy auction.

Each possible agent-task pair has bid:

$$
b_{i,g}=J_{i,g}
$$

All bids are sorted from highest to lowest.

A bid is accepted only if:

1. the UUV has not yet received a task;
2. the task has not already been assigned.

Thus each UUV receives at most one task:

$$
|\mathcal{A}(i)|\le1
$$

and each task receives at most one UUV:

$$
|\mathcal{A}(g)|\le1
$$

This creates one-to-one frontier allocation inside the connected component.

---

# Why This Reduces Redundancy

Independent planning may produce:

$$
g_i^*\approx g_j^*
$$

for nearby UUVs.

After allocation:

$$
g_i^{alloc}
\neq
g_j^{alloc}
$$

for agents participating in the same communication component.

We additionally measure assigned-goal spacing:

$$
d_{ij}^{goal}
=
\|
g_i^{alloc}
-
g_j^{alloc}
\|_2
$$

A larger value indicates stronger spatial task separation.

---

# Disconnected Agents

A UUV in a singleton communication component:

$$
|\mathcal{C}_k|=1
$$

cannot negotiate with anyone.

It simply selects its highest-utility communication-aware frontier goal:

$$
g_i^{alloc}
=
\arg\max_g J_i(g)
$$

Therefore communication absence degrades gracefully to independent planning.

---

# No Central Ground-Truth Oracle

The allocation procedure may use:

- current communication graph;
- agent positions;
- each participant's legitimate known map;
- candidate goals generated from those maps;
- candidate utilities.

It must not use:

- hidden resources;
- simulator-global unseen map;
- future trajectories.

The algorithm coordinates only legitimately available mission information.

---

# Evaluation Metrics

For each communication component report:

### Number of agents

$$
N_C
$$

### Raw tasks

$$
N_{raw}
$$

### Deduplicated tasks

$$
N_{task}
$$

### Assigned UUVs

$$
N_{assigned}
$$

### Minimum assigned-goal spacing

$$
d_{goal,min}
$$

### Independent-versus-allocated goal change

For UUV $i$:

$$
\Delta g_i^{alloc}
=
\|
g_i^{independent}
-
g_i^{allocated}
\|_2
$$

If:

$$
\Delta g_i^{alloc}>0
$$

then explicit coordination changed that UUV's task.

---

# Expected Output

The code cell should:

1. build the current dynamic communication graph;
2. identify connected components;
3. generate communication-aware frontier candidates for each agent;
4. combine candidates within each connected component;
5. spatially deduplicate the task pool;
6. compute the full agent-task utility matrix;
7. perform greedy one-to-one allocation;
8. give singleton components their normal independent goal;
9. verify:
   - no task is assigned twice;
   - no UUV receives more than one task;
10. compare:
    - independent goal;
    - allocated goal;
11. report goal divergence and assigned-goal spacing;
12. visualize:
    - current UUV positions;
    - available tasks;
    - assignment lines from UUV to task.

A successful result establishes:

$$
\boxed{
\text{Communication-Aware Knowledge}
\rightarrow
\text{Distributed Task Allocation}
\rightarrow
\text{Spatially Separated Mission Goals}
}
$$

---

In [ ]:
# ============================================================
# Combo 26 — Distributed Frontier Goal Allocation
# ============================================================

from itertools import combinations


# ------------------------------------------------------------
# Allocation configuration
# ------------------------------------------------------------

TASK_DEDUPLICATION_DISTANCE = 128.0


# ------------------------------------------------------------
# Current communication graph
# ------------------------------------------------------------

allocation_positions = {
    agent.uuv_id: (
        agent.state.x,
        agent.state.y,
    )
    for agent in agents
}

allocation_graph = build_communication_graph(
    allocation_positions,
    communication_range=COMMUNICATION_RANGE,
)


# ------------------------------------------------------------
# Generate communication-aware candidates for each UUV
# ------------------------------------------------------------

agent_candidate_sets = {}
independent_goals = {}


for agent in agents:

    uuv_id = agent.uuv_id

    known_map = known_exploration_maps[
        uuv_id
    ]

    planning_result = generate_goal_from_knowledge(
        agent,
        known_map,
    )

    candidates = planning_result[
        "candidates"
    ]

    agent_candidate_sets[
        uuv_id
    ] = candidates


    if candidates:

        scores = [
            score_frontier_candidate_with_memory(
                agent,
                candidate,
                planning_result[
                    "memory"
                ],
            )
            for candidate in candidates
        ]

        independent_goals[
            uuv_id
        ] = max(
            scores,
            key=lambda item:
                item["utility"],
        )

    else:

        independent_goals[
            uuv_id
        ] = None


# ------------------------------------------------------------
# Spatial task-pool deduplication
# ------------------------------------------------------------

def deduplicate_frontier_tasks(
    raw_tasks,
    min_distance=
        TASK_DEDUPLICATION_DISTANCE,
):
    """
    Greedy spatial filtering of frontier tasks.
    """

    if not raw_tasks:
        return []

    # Prefer tasks with larger source frontier support if
    # available; otherwise preserve input order.
    retained = []

    for task in raw_tasks:

        candidate_position = (
            task["x"],
            task["y"],
        )

        sufficiently_far = all(
            euclidean_distance(
                candidate_position,
                (
                    existing[
                        "x"
                    ],
                    existing[
                        "y"
                    ],
                ),
            )
            >=
            min_distance

            for existing in retained
        )

        if sufficiently_far:

            new_task = {
                "task_id":
                    len(retained),

                "x":
                    int(
                        task["x"]
                    ),

                "y":
                    int(
                        task["y"]
                    ),

                "source_uuv":
                    int(
                        task[
                            "source_uuv"
                        ]
                    ),
            }

            retained.append(
                new_task
            )

    return retained


# ------------------------------------------------------------
# Connected components
# ------------------------------------------------------------

communication_components = [
    sorted(
        list(component)
    )
    for component in (
        nx.connected_components(
            allocation_graph
        )
    )
]


# ------------------------------------------------------------
# Allocation outputs
# ------------------------------------------------------------

allocated_goals = {}

component_reports = []

allocation_bid_tables = {}


# ============================================================
# Allocate independently inside each communication component
# ============================================================

for component_index, component in enumerate(
    communication_components
):

    # --------------------------------------------------------
    # Raw task pool
    # --------------------------------------------------------

    raw_tasks = []

    for uuv_id in component:

        for candidate in (
            agent_candidate_sets[
                uuv_id
            ]
        ):

            raw_tasks.append(
                {
                    "x":
                        int(
                            candidate[
                                "x"
                            ]
                        ),

                    "y":
                        int(
                            candidate[
                                "y"
                            ]
                        ),

                    "source_uuv":
                        uuv_id,
                }
            )


    # --------------------------------------------------------
    # Deduplicate spatially similar tasks
    # --------------------------------------------------------

    tasks = deduplicate_frontier_tasks(
        raw_tasks,
        min_distance=
            TASK_DEDUPLICATION_DISTANCE,
    )


    # --------------------------------------------------------
    # Singleton component
    # --------------------------------------------------------

    if len(component) == 1:

        uuv_id = component[0]

        independent_goal = (
            independent_goals[
                uuv_id
            ]
        )

        allocated_goals[
            uuv_id
        ] = independent_goal


        component_reports.append(
            {
                "component":
                    component_index,

                "agents":
                    str(component),

                "agent_count":
                    1,

                "raw_tasks":
                    len(
                        raw_tasks
                    ),

                "dedup_tasks":
                    len(
                        tasks
                    ),

                "assigned":
                    int(
                        independent_goal
                        is not None
                    ),
            }
        )

        continue


    # --------------------------------------------------------
    # Build planning-memory wrappers
    # --------------------------------------------------------

    planning_memories = {
        uuv_id:
            PlanningMemory(
                uuv_id=
                    uuv_id,

                observed_mask=
                    known_exploration_maps[
                        uuv_id
                    ],
            )
        for uuv_id in component
    }


    # --------------------------------------------------------
    # Build all agent-task bids
    # --------------------------------------------------------

    bids = []

    for uuv_id in component:

        agent = agent_by_id[
            uuv_id
        ]

        planning_memory = (
            planning_memories[
                uuv_id
            ]
        )


        for task in tasks:

            candidate_for_scoring = {
                "candidate_id":
                    task[
                        "task_id"
                    ],

                "x":
                    task["x"],

                "y":
                    task["y"],
            }


            score = (
                score_frontier_candidate_with_memory(
                    agent,
                    candidate_for_scoring,
                    planning_memory,
                )
            )


            bids.append(
                {
                    "uuv_id":
                        uuv_id,

                    "task_id":
                        task[
                            "task_id"
                        ],

                    "task_x":
                        task[
                            "x"
                        ],

                    "task_y":
                        task[
                            "y"
                        ],

                    "utility":
                        float(
                            score[
                                "utility"
                            ]
                        ),

                    "distance_px":
                        float(
                            score[
                                "distance_px"
                            ]
                        ),

                    "info_ratio":
                        float(
                            score[
                                "info_ratio"
                            ]
                        ),

                    "redundancy_ratio":
                        float(
                            score[
                                "redundancy_ratio"
                            ]
                        ),
                }
            )


    # --------------------------------------------------------
    # Greedy auction
    # --------------------------------------------------------

    bids_sorted = sorted(
        bids,
        key=lambda item:
            item[
                "utility"
            ],
        reverse=True,
    )


    assigned_agents = set()
    assigned_tasks = set()


    for bid in bids_sorted:

        uuv_id = (
            bid[
                "uuv_id"
            ]
        )

        task_id = (
            bid[
                "task_id"
            ]
        )


        if (
            uuv_id
            in assigned_agents
        ):
            continue


        if (
            task_id
            in assigned_tasks
        ):
            continue


        allocated_goals[
            uuv_id
        ] = {
            "candidate_id":
                task_id,

            "x":
                bid[
                    "task_x"
                ],

            "y":
                bid[
                    "task_y"
                ],

            "utility":
                bid[
                    "utility"
                ],

            "distance_px":
                bid[
                    "distance_px"
                ],

            "info_ratio":
                bid[
                    "info_ratio"
                ],

            "redundancy_ratio":
                bid[
                    "redundancy_ratio"
                ],
        }


        assigned_agents.add(
            uuv_id
        )

        assigned_tasks.add(
            task_id
        )


        if (
            len(
                assigned_agents
            )
            ==
            len(
                component
            )
        ):
            break


    # --------------------------------------------------------
    # Any unassigned UUV falls back to independent goal
    # --------------------------------------------------------

    for uuv_id in component:

        if (
            uuv_id
            not in allocated_goals
        ):

            allocated_goals[
                uuv_id
            ] = (
                independent_goals[
                    uuv_id
                ]
            )


    allocation_bid_tables[
        component_index
    ] = pd.DataFrame(
        bids
    )


    component_reports.append(
        {
            "component":
                component_index,

            "agents":
                str(component),

            "agent_count":
                len(
                    component
                ),

            "raw_tasks":
                len(
                    raw_tasks
                ),

            "dedup_tasks":
                len(
                    tasks
                ),

            "assigned":
                len(
                    assigned_agents
                ),
        }
    )


# ------------------------------------------------------------
# Validate one task per communicating UUV
# ------------------------------------------------------------

allocation_positions_by_uuv = {}

for uuv_id, goal in (
    allocated_goals.items()
):

    if goal is None:
        continue

    allocation_positions_by_uuv[
        uuv_id
    ] = (
        int(
            goal[
                "x"
            ]
        ),
        int(
            goal[
                "y"
            ]
        ),
    )


# ------------------------------------------------------------
# Per-component duplicate-task validation
# ------------------------------------------------------------

for component in communication_components:

    goal_positions = []

    for uuv_id in component:

        goal = allocated_goals.get(
            uuv_id
        )

        if goal is None:
            continue

        goal_positions.append(
            (
                goal["x"],
                goal["y"],
            )
        )


    if (
        len(goal_positions)
        !=
        len(
            set(
                goal_positions
            )
        )
    ):
        raise RuntimeError(
            "A communication component assigned "
            "the same task to multiple UUVs."
        )


# ------------------------------------------------------------
# Compare independent and allocated goals
# ------------------------------------------------------------

allocation_comparison = []

for agent in agents:

    uuv_id = (
        agent.uuv_id
    )

    independent = (
        independent_goals[
            uuv_id
        ]
    )

    allocated = (
        allocated_goals.get(
            uuv_id
        )
    )


    if (
        independent is not None
        and
        allocated is not None
    ):

        divergence = (
            euclidean_distance(
                (
                    independent[
                        "x"
                    ],
                    independent[
                        "y"
                    ],
                ),
                (
                    allocated[
                        "x"
                    ],
                    allocated[
                        "y"
                    ],
                ),
            )
        )

    elif (
        independent is None
        and
        allocated is None
    ):

        divergence = 0.0

    else:

        divergence = np.nan


    allocation_comparison.append(
        {
            "uuv_id":
                uuv_id,

            "component":
                next(
                    index
                    for index, component
                    in enumerate(
                        communication_components
                    )
                    if uuv_id in component
                ),

            "independent_x":
                None
                if independent is None
                else independent[
                    "x"
                ],

            "independent_y":
                None
                if independent is None
                else independent[
                    "y"
                ],

            "allocated_x":
                None
                if allocated is None
                else allocated[
                    "x"
                ],

            "allocated_y":
                None
                if allocated is None
                else allocated[
                    "y"
                ],

            "goal_divergence_px":
                divergence,
        }
    )


allocation_comparison_df = (
    pd.DataFrame(
        allocation_comparison
    )
)


# ------------------------------------------------------------
# Assigned-goal pairwise spacing
# ------------------------------------------------------------

assigned_goal_distances = []

for component_index, component in enumerate(
    communication_components
):

    component_goal_records = [
        (
            uuv_id,
            allocation_positions_by_uuv[
                uuv_id
            ],
        )
        for uuv_id in component
        if uuv_id
        in allocation_positions_by_uuv
    ]


    for (
        (uuv_a, goal_a),
        (uuv_b, goal_b),
    ) in combinations(
        component_goal_records,
        2,
    ):

        distance = (
            euclidean_distance(
                goal_a,
                goal_b,
            )
        )

        assigned_goal_distances.append(
            {
                "component":
                    component_index,

                "uuv_a":
                    uuv_a,

                "uuv_b":
                    uuv_b,

                "goal_distance":
                    distance,
            }
        )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

component_df = pd.DataFrame(
    component_reports
)

print(
    "=" * 96
)

print(
    "MultiUUV-Link — Distributed Frontier Goal Allocation"
)

print(
    "=" * 96
)

print(
    f"\nTask deduplication distance : "
    f"{TASK_DEDUPLICATION_DISTANCE:.1f} px"
)

print(
    f"Communication components    : "
    f"{communication_components}"
)

print(
    "\nAllocation components:"
)

print(
    component_df.to_string(
        index=False
    )
)


print(
    "\nIndependent versus allocated goals:"
)

print(
    allocation_comparison_df.to_string(
        index=False,
        formatters={
            "goal_divergence_px":
                lambda x: (
                    "N/A"
                    if pd.isna(x)
                    else f"{x:.2f}"
                )
        },
    )
)


# ------------------------------------------------------------
# Goal-separation report
# ------------------------------------------------------------

if assigned_goal_distances:

    goal_distance_df = (
        pd.DataFrame(
            assigned_goal_distances
        )
    )

    print(
        "\nAllocated goal separations:"
    )

    print(
        goal_distance_df.to_string(
            index=False,
            formatters={
                "goal_distance":
                    lambda x: f"{x:.2f}"
            },
        )
    )

    minimum_allocated_goal_distance = (
        goal_distance_df[
            "goal_distance"
        ].min()
    )

    print(
        f"\nMinimum allocated-goal separation : "
        f"{minimum_allocated_goal_distance:.2f} px"
    )

else:

    minimum_allocated_goal_distance = np.nan

    print(
        "\nNo multi-agent component had enough "
        "assigned goals for spacing analysis."
    )


# ------------------------------------------------------------
# Count allocation-induced changes
# ------------------------------------------------------------

allocation_changed_agents = int(
    (
        allocation_comparison_df[
            "goal_divergence_px"
        ]
        >
        0
    ).sum()
)

print(
    "\nAllocation effect:"
)

print(
    f"  UUVs whose goal changed : "
    f"{allocation_changed_agents} / {NUM_UUVS}"
)


# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


# Draw current communication graph
for uuv_a, uuv_b in (
    allocation_graph.edges()
):

    p_a = (
        allocation_positions[
            uuv_a
        ]
    )

    p_b = (
        allocation_positions[
            uuv_b
        ]
    )

    plt.plot(
        [
            p_a[0],
            p_b[0],
        ],
        [
            p_a[1],
            p_b[1],
        ],
        linestyle="--",
        linewidth=1.2,
        alpha=0.5,
        zorder=3,
    )


# Draw UUVs and assignments
for agent in agents:

    uuv_id = agent.uuv_id

    plt.scatter(
        agent.state.x,
        agent.state.y,
        marker="o",
        s=130,
        label=f"UUV-{uuv_id}",
        zorder=7,
    )


    allocated = (
        allocated_goals.get(
            uuv_id
        )
    )

    if allocated is None:
        continue


    goal_x = (
        allocated[
            "x"
        ]
    )

    goal_y = (
        allocated[
            "y"
        ]
    )


    plt.scatter(
        goal_x,
        goal_y,
        marker="*",
        s=220,
        edgecolors="black",
        linewidths=1.3,
        zorder=8,
    )


    plt.plot(
        [
            agent.state.x,
            goal_x,
        ],
        [
            agent.state.y,
            goal_y,
        ],
        linewidth=1.8,
        zorder=6,
    )


    plt.text(
        goal_x + 8,
        goal_y - 8,
        f"T{uuv_id}",
        fontsize=9,
        zorder=9,
    )


plt.title(
    "MultiUUV-Link — Distributed Frontier "
    "Task Allocation"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nDistributed frontier goal "
    "allocation validated."
)

# Combo 27 — Stable Allocation-Aware Cooperative Exploration

## Objective

Previous work established:

- dynamic communication;
- map and resource knowledge exchange;
- communication-aware frontier planning;
- predictive collision safety;
- distributed frontier allocation.

The first version of Combo 27 exposed three important weaknesses:

1. an unassigned UUV could fall back to an already allocated task;
2. goals were frequently invalidated, producing excessive allocation churn;
3. task allocation encouraged spatially different goal centers but did not explicitly penalize overlapping future FOVs.

This revised combo corrects all three issues.

The integrated architecture becomes:

```text
Local Observation
        ↓
Private Local Memory
        ↓
Dynamic Communication
        ↓
Known Map Fusion
        ↓
Frontier Task Generation
        ↓
Spatial Deduplication
        ↓
Persistent Existing Goals
        +
Overlap-Aware Distributed Auction
        ↓
Desired Actions
        ↓
Predictive Safety Shield
        ↓
Synchronized Motion
        ↓
Repeat
```

---

## 1. No Duplicate Fallback Tasks

Each frontier task may be assigned to at most one UUV.

If a component has fewer valid tasks than active UUVs, an unassigned UUV receives:

```text
HOLD
```

rather than independently selecting a task already assigned to another agent.

Therefore:

$$
g_i^{alloc}=g_j^{alloc}
$$

is prohibited for:

$$
i\neq j
$$

inside the same communication component.

---

## 2. Goal Hysteresis

A goal should not disappear merely because its exact frontier pixel changes after one observation.

Instead, an active goal remains valid while the FOV centered on that goal still contains a meaningful amount of unknown territory.

Define:

$$
I_i(g)
=
\frac{
N_{\text{unknown}}(g)
}{
N_{\text{valid}}(g)
}
$$

A goal remains useful while:

$$
I_i(g)\ge I_{\min}
$$

For the prototype:

$$
I_{\min}=0.15
$$

Thus a goal is discarded only when:

1. the goal is reached; or
2. less than 15% of its candidate-centered FOV remains unknown.

This produces temporal hysteresis and prevents frontier-pixel flicker from causing planning churn.

---

## 3. Controlled Allocation Frequency

Allocation should not run almost every timestep.

The revised system uses:

$$
T_A=12
$$

for periodic allocation and:

$$
T_{cool}=4
$$

as a minimum allocation cooldown.

An allocation round occurs when:

- the periodic interval arrives; or
- topology changes / a goal becomes unavailable,

but emergency reallocation is permitted only when the cooldown has elapsed.

This limits allocation thrashing.

---

## 4. Goal Locking

At an allocation round, an existing goal may remain assigned if:

- it has not been reached;
- its information ratio remains above the usefulness threshold;
- it does not conflict spatially with another locked assignment.

Thus the allocator does not unnecessarily redirect an agent whose existing task remains useful.

---

## 5. Predicted FOV-Overlap Penalty

Goal-center distance alone does not guarantee nonredundant future sensing.

Two different frontier goals may have overlapping future FOVs.

For candidate goal $g$, define its valid FOV region:

$$
\mathcal{V}(g)
$$

For two candidate goals:

$$
g_a,g_b
$$

define FOV Intersection-over-Union:

$$
O(g_a,g_b)
=
\frac{
|\mathcal{V}(g_a)\cap\mathcal{V}(g_b)|
}{
|\mathcal{V}(g_a)\cup\mathcal{V}(g_b)|
}
$$

where:

$$
0\le O\le1
$$

The allocation utility becomes:

$$
J_{i,g}^{alloc}
=
J_{i,g}^{base}
-
\lambda_O
\max_{h\in\mathcal{A}}
O(g,h)
$$

where:

$$
\mathcal{A}
$$

is the set of tasks already assigned during the auction.

For the prototype:

$$
\lambda_O=0.65
$$

This directly discourages assignments whose future sensing footprints overlap.

---

## 6. Minimum Assigned-Goal Separation

In addition to FOV overlap, selected task centers must satisfy:

$$
\|g_a-g_b\|_2
\ge
128\text{ px}
$$

inside the same communication component.

Unlike the previous implementation, this constraint applies to the **final assignments**, including persistent goals.

---

## 7. No-Task Behavior

If no valid unique frontier remains for an agent:

$$
g_i=None
$$

and:

$$
a_i=HOLD
$$

The robot does not invent or steal another agent's assignment.

Later allocation rounds may assign it a new task when topology or knowledge changes.

---

## 8. Communication-Aware Planning

Every frontier and candidate utility is computed from:

$$
M_i^{known}
$$

not merely:

$$
M_i^{local}
$$

Therefore communicated regions are treated as already known.

---

## 9. Safety

All requested actions continue to pass through the predictive safety shield.

The executed swarm must satisfy:

$$
d_{ij}^{exec}\ge128\text{ px}
$$

for every pair whenever a feasible joint action exists.

---

## 10. Mission Metrics

This revised combo distinguishes two redundancy quantities.

### Cumulative redundancy

Using all local observations accumulated throughout the notebook:

$$
\rho_{cum}
=
\frac{
\sum_i|M_i^{local}|
-
|\bigvee_iM_i^{local}|
}{
\sum_i|M_i^{local}|
}
$$

### Mission-incremental redundancy

Let:

$$
\Delta M_i
=
M_i^{local,end}
\setminus
M_i^{local,start}
$$

Then:

$$
\rho_{mission}
=
\frac{
\sum_i|\Delta M_i|
-
|\bigvee_i\Delta M_i|
}{
\sum_i|\Delta M_i|
}
$$

This second quantity is the cleaner measure of overlap introduced specifically during this combo.

---

## Important Experimental Note

Combo 25 and Combo 27 occur sequentially in this development notebook.

Therefore their final metrics are **not yet a controlled algorithm comparison** because they begin from different world-knowledge states.

A later dedicated benchmark must reset both controllers to the same:

- resource realization;
- UUV spawn positions;
- local memories;
- random seeds;

before comparing them quantitatively.

This combo validates the corrected allocation-aware mechanism itself.

---

## Expected Output

The code cell should:

1. run 180 synchronized timesteps;
2. rebuild communication topology dynamically;
3. exchange maps and resource knowledge;
4. preserve valid existing assignments;
5. allocate at a controlled frequency;
6. never assign duplicate final tasks;
7. penalize predicted FOV overlap;
8. HOLD agents when no unique task exists;
9. apply predictive collision safety;
10. report:
    - allocation rounds;
    - goal changes;
    - HOLD actions;
    - minimum final task spacing;
    - mean predicted task-FOV overlap;
    - team coverage gain;
    - mission-incremental redundancy;
    - cumulative redundancy;
    - safety overrides;
11. visualize:
    - allocation-aware trajectories;
    - incremental coverage and redundancy history.

A successful result establishes:

$$
\boxed{
\text{Stable Communication-Aware Distributed Task Allocation}
}
$$

---

In [ ]:
# ============================================================
# Stable Allocation-Aware Cooperative Exploration
# ============================================================

from itertools import combinations


# ------------------------------------------------------------
# Mission configuration
# ------------------------------------------------------------

ALLOC_MISSION_STEPS = 180

ALLOCATION_INTERVAL = 12
ALLOCATION_COOLDOWN = 4

GOAL_MIN_UNKNOWN_RATIO = 0.15

TASK_DEDUPLICATION_DISTANCE = 128.0
MIN_ASSIGNED_GOAL_SEPARATION = 128.0

TASK_OVERLAP_WEIGHT = 0.65


# ------------------------------------------------------------
# Ensure total-known maps contain all current local knowledge
# ------------------------------------------------------------

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

alloc_start = {}

alloc_start_local_maps = {}

for agent in agents:

    uuv_id = agent.uuv_id

    alloc_start_local_maps[
        uuv_id
    ] = (
        agent.exploration_memory
        .observed
        .copy()
    )

    alloc_start[
        uuv_id
    ] = {
        "local_cells":
            agent.exploration_memory
            .observed_count(),

        "known_cells":
            int(
                known_exploration_maps[
                    uuv_id
                ].sum()
            ),

        "known_resources":
            len(
                agent.resource_memory
            ),

        "trajectory_index":
            len(
                agent.trajectory
            ) - 1,
    }


start_team_union = np.logical_or.reduce(
    [
        alloc_start_local_maps[
            agent.uuv_id
        ]
        for agent in agents
    ]
)

start_team_coverage = (
    int(start_team_union.sum())
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Utility functions
# ============================================================

def candidate_valid_fov_rectangle(
    x: int,
    y: int,
):
    """
    Return valid world-space FOV rectangle:

        (x0, y0, x1, y1)

    with x1/y1 exclusive.
    """

    state = UUVState(
        uuv_id=-1,
        x=int(x),
        y=int(y),
    )

    x_min, y_min, x_max, y_max = (
        get_fov_bounds(
            state,
            fov_width=FOV_WIDTH,
            fov_height=FOV_HEIGHT,
        )
    )

    x0 = max(
        0,
        x_min,
    )

    y0 = max(
        0,
        y_min,
    )

    x1 = min(
        WIDTH,
        x_max,
    )

    y1 = min(
        HEIGHT,
        y_max,
    )

    return (
        int(x0),
        int(y0),
        int(x1),
        int(y1),
    )


# ------------------------------------------------------------
# FOV overlap
# ------------------------------------------------------------

def candidate_fov_iou(
    goal_a,
    goal_b,
):
    """
    Intersection-over-Union of two candidate-centered valid FOVs.
    """

    ax0, ay0, ax1, ay1 = (
        candidate_valid_fov_rectangle(
            goal_a["x"],
            goal_a["y"],
        )
    )

    bx0, by0, bx1, by1 = (
        candidate_valid_fov_rectangle(
            goal_b["x"],
            goal_b["y"],
        )
    )


    intersection_width = max(
        0,
        min(
            ax1,
            bx1,
        )
        -
        max(
            ax0,
            bx0,
        ),
    )

    intersection_height = max(
        0,
        min(
            ay1,
            by1,
        )
        -
        max(
            ay0,
            by0,
        ),
    )


    intersection = (
        intersection_width
        *
        intersection_height
    )


    area_a = (
        (ax1 - ax0)
        *
        (ay1 - ay0)
    )

    area_b = (
        (bx1 - bx0)
        *
        (by1 - by0)
    )


    union = (
        area_a
        +
        area_b
        -
        intersection
    )


    if union <= 0:
        return 0.0


    return float(
        intersection
        /
        union
    )


# ------------------------------------------------------------
# Goal information ratio
# ------------------------------------------------------------

def goal_information_ratio(
    agent,
    goal,
):
    """
    Information ratio around an existing goal using the UUV's
    total legitimate known map.
    """

    if goal is None:
        return 0.0


    planning_memory = PlanningMemory(
        uuv_id=
            agent.uuv_id,

        observed_mask=
            known_exploration_maps[
                agent.uuv_id
            ],
    )


    stats = candidate_fov_statistics(
        exploration_memory=
            planning_memory,

        candidate_x=
            int(
                goal["x"]
            ),

        candidate_y=
            int(
                goal["y"]
            ),
    )


    return float(
        stats[
            "information_ratio"
        ]
    )


# ------------------------------------------------------------
# Goal usefulness / hysteresis
# ------------------------------------------------------------

def goal_is_still_useful(
    agent,
    goal,
):
    if goal is None:
        return False


    distance = euclidean_distance(
        (
            agent.state.x,
            agent.state.y,
        ),
        (
            goal["x"],
            goal["y"],
        ),
    )


    # Reached goal
    if (
        distance
        <=
        GOAL_REACHED_RADIUS
    ):
        return False


    info_ratio = (
        goal_information_ratio(
            agent,
            goal,
        )
    )


    return (
        info_ratio
        >=
        GOAL_MIN_UNKNOWN_RATIO
    )


# ------------------------------------------------------------
# Candidate task pool
# ------------------------------------------------------------

def build_component_task_pool(
    component,
):
    raw_tasks = []


    for uuv_id in component:

        agent = agent_by_id[
            uuv_id
        ]


        planning_result = (
            generate_goal_from_knowledge(
                agent,
                known_exploration_maps[
                    uuv_id
                ],
            )
        )


        for candidate in (
            planning_result[
                "candidates"
            ]
        ):

            raw_tasks.append(
                {
                    "x":
                        int(
                            candidate[
                                "x"
                            ]
                        ),

                    "y":
                        int(
                            candidate[
                                "y"
                            ]
                        ),

                    "source_uuv":
                        int(
                            uuv_id
                        ),
                }
            )


    tasks = deduplicate_frontier_tasks(
        raw_tasks,
        min_distance=
            TASK_DEDUPLICATION_DISTANCE,
    )


    return (
        raw_tasks,
        tasks,
    )


# ============================================================
# Corrected component allocator
# ============================================================

def allocate_component_frontiers_stable(
    component,
    previous_goals,
):
    """
    Stable component-level distributed task allocation.

    Corrections:
      - useful old goals are locked;
      - final goals remain separated;
      - no duplicate fallback;
      - overlap penalty included;
      - unassigned UUV receives None/HOLD.
    """

    component = sorted(
        component
    )


    assignments = {
        uuv_id: None
        for uuv_id in component
    }


    # --------------------------------------------------------
    # Build fresh frontier task pool
    # --------------------------------------------------------

    raw_tasks, tasks = (
        build_component_task_pool(
            component
        )
    )


    # --------------------------------------------------------
    # Singleton component
    # --------------------------------------------------------

    if len(component) == 1:

        uuv_id = component[0]

        agent = agent_by_id[
            uuv_id
        ]


        old_goal = previous_goals.get(
            uuv_id
        )


        # Preserve useful old goal
        if goal_is_still_useful(
            agent,
            old_goal,
        ):

            assignments[
                uuv_id
            ] = old_goal

            return (
                assignments,
                {
                    "raw_tasks":
                        len(
                            raw_tasks
                        ),

                    "dedup_tasks":
                        len(
                            tasks
                        ),

                    "locked_goals":
                        1,

                    "auction_assignments":
                        0,

                    "unassigned":
                        0,
                },
            )


        planning_result = (
            generate_goal_from_knowledge(
                agent,
                known_exploration_maps[
                    uuv_id
                ],
            )
        )


        assignments[
            uuv_id
        ] = (
            planning_result[
                "selected"
            ]
        )


        return (
            assignments,
            {
                "raw_tasks":
                    len(
                        raw_tasks
                    ),

                "dedup_tasks":
                    len(
                        tasks
                    ),

                "locked_goals":
                    0,

                "auction_assignments":
                    int(
                        assignments[
                            uuv_id
                        ]
                        is not None
                    ),

                "unassigned":
                    int(
                        assignments[
                            uuv_id
                        ]
                        is None
                    ),
            },
        )


    # ========================================================
    # 1. Lock useful existing goals
    # ========================================================

    locked_goals = []


    # Prefer agents whose existing goal currently has the
    # highest remaining information ratio.
    lock_candidates = []


    for uuv_id in component:

        old_goal = (
            previous_goals.get(
                uuv_id
            )
        )

        agent = agent_by_id[
            uuv_id
        ]


        if not goal_is_still_useful(
            agent,
            old_goal,
        ):
            continue


        info_ratio = (
            goal_information_ratio(
                agent,
                old_goal,
            )
        )


        lock_candidates.append(
            (
                info_ratio,
                uuv_id,
                old_goal,
            )
        )


    lock_candidates.sort(
        key=lambda item:
            item[0],
        reverse=True,
    )


    for (
        _,
        uuv_id,
        old_goal,
    ) in lock_candidates:

        candidate_position = (
            old_goal[
                "x"
            ],
            old_goal[
                "y"
            ],
        )


        spatially_valid = all(
            euclidean_distance(
                candidate_position,
                (
                    locked_goal[
                        "x"
                    ],
                    locked_goal[
                        "y"
                    ],
                ),
            )
            >=
            MIN_ASSIGNED_GOAL_SEPARATION

            for locked_goal
            in locked_goals
        )


        if not spatially_valid:
            continue


        assignments[
            uuv_id
        ] = old_goal

        locked_goals.append(
            old_goal
        )


    # ========================================================
    # 2. Remove tasks conflicting with locked goals
    # ========================================================

    available_tasks = []


    for task in tasks:

        task_position = (
            task[
                "x"
            ],
            task[
                "y"
            ],
        )


        sufficiently_far = all(
            euclidean_distance(
                task_position,
                (
                    goal[
                        "x"
                    ],
                    goal[
                        "y"
                    ],
                ),
            )
            >=
            MIN_ASSIGNED_GOAL_SEPARATION

            for goal in locked_goals
        )


        if sufficiently_far:

            available_tasks.append(
                task
            )


    # ========================================================
    # 3. Greedy overlap-aware auction
    # ========================================================

    unassigned_agents = {
        uuv_id
        for uuv_id in component
        if assignments[
            uuv_id
        ] is None
    }


    assigned_task_ids = set()


    while (
        unassigned_agents
        and
        len(
            assigned_task_ids
        )
        <
        len(
            available_tasks
        )
    ):

        best_bid = None


        current_assigned_goals = [
            goal
            for goal
            in assignments.values()
            if goal is not None
        ]


        for uuv_id in sorted(
            unassigned_agents
        ):

            agent = agent_by_id[
                uuv_id
            ]


            planning_memory = (
                PlanningMemory(
                    uuv_id=
                        uuv_id,

                    observed_mask=
                        known_exploration_maps[
                            uuv_id
                        ],
                )
            )


            for task in available_tasks:

                task_id = (
                    task[
                        "task_id"
                    ]
                )


                if (
                    task_id
                    in assigned_task_ids
                ):
                    continue


                # ------------------------------------------------
                # Hard center-spacing constraint
                # ------------------------------------------------

                task_position = (
                    task[
                        "x"
                    ],
                    task[
                        "y"
                    ],
                )


                if any(
                    euclidean_distance(
                        task_position,
                        (
                            assigned_goal[
                                "x"
                            ],
                            assigned_goal[
                                "y"
                            ],
                        ),
                    )
                    <
                    MIN_ASSIGNED_GOAL_SEPARATION

                    for assigned_goal
                    in current_assigned_goals
                ):
                    continue


                # ------------------------------------------------
                # Base mission utility
                # ------------------------------------------------

                scoring_candidate = {
                    "candidate_id":
                        task_id,

                    "x":
                        task[
                            "x"
                        ],

                    "y":
                        task[
                            "y"
                        ],
                }


                base_score = (
                    score_frontier_candidate_with_memory(
                        agent,
                        scoring_candidate,
                        planning_memory,
                    )
                )


                # ------------------------------------------------
                # Predicted FOV overlap penalty
                # ------------------------------------------------

                if current_assigned_goals:

                    max_overlap = max(
                        candidate_fov_iou(
                            task,
                            assigned_goal,
                        )
                        for assigned_goal
                        in current_assigned_goals
                    )

                else:

                    max_overlap = 0.0


                adjusted_utility = (
                    base_score[
                        "utility"
                    ]
                    -
                    TASK_OVERLAP_WEIGHT
                    *
                    max_overlap
                )


                bid = {
                    "uuv_id":
                        uuv_id,

                    "task_id":
                        task_id,

                    "x":
                        int(
                            task[
                                "x"
                            ]
                        ),

                    "y":
                        int(
                            task[
                                "y"
                            ]
                        ),

                    "base_utility":
                        float(
                            base_score[
                                "utility"
                            ]
                        ),

                    "overlap":
                        float(
                            max_overlap
                        ),

                    "utility":
                        float(
                            adjusted_utility
                        ),

                    "distance_px":
                        float(
                            base_score[
                                "distance_px"
                            ]
                        ),

                    "info_ratio":
                        float(
                            base_score[
                                "info_ratio"
                            ]
                        ),

                    "redundancy_ratio":
                        float(
                            base_score[
                                "redundancy_ratio"
                            ]
                        ),
                }


                if (
                    best_bid is None
                    or
                    bid[
                        "utility"
                    ]
                    >
                    best_bid[
                        "utility"
                    ]
                ):

                    best_bid = bid


        # No valid unique task remains.
        if best_bid is None:
            break


        uuv_id = (
            best_bid[
                "uuv_id"
            ]
        )


        assignments[
            uuv_id
        ] = {
            "candidate_id":
                best_bid[
                    "task_id"
                ],

            "x":
                best_bid[
                    "x"
                ],

            "y":
                best_bid[
                    "y"
                ],

            "base_utility":
                best_bid[
                    "base_utility"
                ],

            "overlap":
                best_bid[
                    "overlap"
                ],

            "utility":
                best_bid[
                    "utility"
                ],

            "distance_px":
                best_bid[
                    "distance_px"
                ],

            "info_ratio":
                best_bid[
                    "info_ratio"
                ],

            "redundancy_ratio":
                best_bid[
                    "redundancy_ratio"
                ],
        }


        assigned_task_ids.add(
            best_bid[
                "task_id"
            ]
        )


        unassigned_agents.remove(
            uuv_id
        )


    # --------------------------------------------------------
    # IMPORTANT:
    # No independent fallback.
    #
    # Any agent still unassigned remains None and will HOLD.
    # --------------------------------------------------------


    stats = {
        "raw_tasks":
            len(
                raw_tasks
            ),

        "dedup_tasks":
            len(
                tasks
            ),

        "locked_goals":
            len(
                locked_goals
            ),

        "auction_assignments":
            sum(
                goal is not None
                for goal in assignments.values()
            )
            -
            len(
                locked_goals
            ),

        "unassigned":
            sum(
                goal is None
                for goal in assignments.values()
            ),
    }


    return (
        assignments,
        stats,
    )


# ============================================================
# Complete allocation round
# ============================================================

def perform_stable_allocation_round(
    graph,
    previous_goals,
):
    assignments = {}

    round_stats = []

    spacing_samples = []

    overlap_samples = []


    components = [
        sorted(
            component
        )
        for component in (
            nx.connected_components(
                graph
            )
        )
    ]


    for component_index, component in enumerate(
        components
    ):

        (
            component_assignments,
            component_stats,
        ) = allocate_component_frontiers_stable(
            component,
            previous_goals,
        )


        assignments.update(
            component_assignments
        )


        component_stats[
            "component"
        ] = component_index

        component_stats[
            "agents"
        ] = str(
            component
        )

        round_stats.append(
            component_stats
        )


        # ----------------------------------------------------
        # Final goal spacing and overlap validation
        # ----------------------------------------------------

        assigned_records = [
            (
                uuv_id,
                component_assignments[
                    uuv_id
                ],
            )
            for uuv_id in component
            if component_assignments[
                uuv_id
            ] is not None
        ]


        for (
            (uuv_a, goal_a),
            (uuv_b, goal_b),
        ) in combinations(
            assigned_records,
            2,
        ):

            distance = (
                euclidean_distance(
                    (
                        goal_a[
                            "x"
                        ],
                        goal_a[
                            "y"
                        ],
                    ),
                    (
                        goal_b[
                            "x"
                        ],
                        goal_b[
                            "y"
                        ],
                    ),
                )
            )


            if (
                distance
                <
                MIN_ASSIGNED_GOAL_SEPARATION
            ):

                raise RuntimeError(
                    "Final component assignments violate "
                    "minimum goal separation."
                )


            spacing_samples.append(
                distance
            )


            overlap_samples.append(
                candidate_fov_iou(
                    goal_a,
                    goal_b,
                )
            )


    # --------------------------------------------------------
    # Goal-change count
    # --------------------------------------------------------

    changed_count = 0


    for agent in agents:

        uuv_id = agent.uuv_id

        previous = (
            previous_goals.get(
                uuv_id
            )
        )

        current = (
            assignments.get(
                uuv_id
            )
        )


        if (
            previous is None
            and
            current is None
        ):
            continue


        if (
            previous is None
            or
            current is None
        ):

            changed_count += 1
            continue


        if (
            previous[
                "x"
            ]
            !=
            current[
                "x"
            ]
            or
            previous[
                "y"
            ]
            !=
            current[
                "y"
            ]
        ):

            changed_count += 1


    return (
        assignments,
        changed_count,
        spacing_samples,
        overlap_samples,
        round_stats,
    )


# ============================================================
# Mission state
# ============================================================

mission_allocated_goals = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step = (
    -ALLOCATION_COOLDOWN
)

previous_graph_edges = None


# ------------------------------------------------------------
# Logs
# ------------------------------------------------------------

allocation_round_count = 0
allocation_goal_changes = 0

allocation_goal_spacing_samples = []
allocation_overlap_samples = []

allocation_hold_counts = {
    agent.uuv_id: 0
    for agent in agents
}

allocation_safety_overrides = {
    agent.uuv_id: 0
    for agent in agents
}

allocation_remote_map_gain = {
    agent.uuv_id: 0
    for agent in agents
}

allocation_remote_resource_gain = {
    agent.uuv_id: 0
    for agent in agents
}


alloc_edge_history = []

alloc_team_coverage_history = []
alloc_incremental_redundancy_history = []

alloc_mission_min_separation = np.inf

allocation_round_logs = []


# ============================================================
# Synchronized mission
# ============================================================

for mission_step in range(
    ALLOC_MISSION_STEPS
):

    # --------------------------------------------------------
    # 1. Current communication graph
    # --------------------------------------------------------

    positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }


    graph = build_communication_graph(
        positions,
        communication_range=
            COMMUNICATION_RANGE,
    )


    alloc_edge_history.append(
        graph.number_of_edges()
    )


    canonical_edges = frozenset(
        tuple(
            sorted(
                edge
            )
        )
        for edge
        in graph.edges()
    )


    topology_changed = (
        previous_graph_edges
        is None
        or
        canonical_edges
        !=
        previous_graph_edges
    )


    previous_graph_edges = (
        canonical_edges
    )


    # --------------------------------------------------------
    # 2. One-hop communication
    # --------------------------------------------------------

    map_gains = (
        exchange_map_knowledge_one_hop(
            graph
        )
    )


    resource_gains = (
        exchange_resource_knowledge_one_hop(
            graph
        )
    )


    for agent in agents:

        uuv_id = agent.uuv_id


        allocation_remote_map_gain[
            uuv_id
        ] += (
            map_gains[
                uuv_id
            ]
        )


        allocation_remote_resource_gain[
            uuv_id
        ] += (
            resource_gains[
                uuv_id
            ]
        )


        # Ensure local knowledge can never be absent
        # from the total known map.
        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # --------------------------------------------------------
    # 3. Goal hysteresis check
    # --------------------------------------------------------

    goal_became_unavailable = False


    for agent in agents:

        uuv_id = agent.uuv_id

        goal = (
            mission_allocated_goals[
                uuv_id
            ]
        )


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            mission_allocated_goals[
                uuv_id
            ] = None

            goal_became_unavailable = True


    # --------------------------------------------------------
    # 4. Controlled allocation trigger
    # --------------------------------------------------------

    periodic_trigger = (
        mission_step
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    missing_goal_exists = any(
        mission_allocated_goals[
            agent.uuv_id
        ]
        is None
        for agent in agents
    )


    urgent_trigger = (
        topology_changed
        or
        goal_became_unavailable
        or
        missing_goal_exists
    )


    cooldown_satisfied = (
        mission_step
        -
        last_allocation_step
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate = (
        periodic_trigger
        or
        (
            urgent_trigger
            and
            cooldown_satisfied
        )
    )


    if should_allocate:

        (
            new_assignments,
            changed_count,
            spacing_samples,
            overlap_samples,
            round_stats,
        ) = perform_stable_allocation_round(
            graph,
            mission_allocated_goals,
        )


        mission_allocated_goals = (
            new_assignments
        )


        allocation_round_count += 1

        allocation_goal_changes += (
            changed_count
        )


        allocation_goal_spacing_samples.extend(
            spacing_samples
        )


        allocation_overlap_samples.extend(
            overlap_samples
        )


        last_allocation_step = (
            mission_step
        )


        allocation_round_logs.append(
            {
                "mission_step":
                    mission_step,

                "goal_changes":
                    changed_count,

                "mean_spacing":
                    (
                        float(
                            np.mean(
                                spacing_samples
                            )
                        )
                        if spacing_samples
                        else np.nan
                    ),

                "min_spacing":
                    (
                        float(
                            np.min(
                                spacing_samples
                            )
                        )
                        if spacing_samples
                        else np.nan
                    ),

                "mean_overlap":
                    (
                        float(
                            np.mean(
                                overlap_samples
                            )
                        )
                        if overlap_samples
                        else 0.0
                    ),

                "unassigned":
                    sum(
                        goal is None
                        for goal
                        in mission_allocated_goals.values()
                    ),
            }
        )


    # --------------------------------------------------------
    # 5. Desired actions
    # --------------------------------------------------------

    desired_actions = []


    for agent in agents:

        uuv_id = agent.uuv_id

        goal = (
            mission_allocated_goals[
                uuv_id
            ]
        )


        if goal is None:

            desired_action = (
                ACTION_HOLD
            )

            allocation_hold_counts[
                uuv_id
            ] += 1


        else:

            desired_action = (
                choose_greedy_action(
                    agent.state,
                    goal_x=
                        goal[
                            "x"
                        ],
                    goal_y=
                        goal[
                            "y"
                        ],
                )
            )


            if desired_action is None:

                desired_action = (
                    ACTION_HOLD
                )

                allocation_hold_counts[
                    uuv_id
                ] += 1


        desired_actions.append(
            desired_action
        )


    # --------------------------------------------------------
    # 6. Predictive safety shield
    # --------------------------------------------------------

    shield_result = safety_shield(
        states=[
            agent.state
            for agent in agents
        ],
        desired_actions=
            desired_actions,
        safe_separation=
            SAFE_SEPARATION,
        step_size=
            ROLLOUT_STEP_SIZE,
    )


    executed_actions = (
        shield_result[
            "executed_actions"
        ]
    )


    alloc_mission_min_separation = min(
        alloc_mission_min_separation,
        shield_result[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if desired != executed:

            allocation_safety_overrides[
                agent.uuv_id
            ] += 1


    # --------------------------------------------------------
    # 7. Predict next states
    # --------------------------------------------------------

    next_states = [
        predict_safety_state(
            state=
                agent.state,

            action=
                action,

            step_size=
                ROLLOUT_STEP_SIZE,
        )

        for agent, action in zip(
            agents,
            executed_actions,
        )
    ]


    # --------------------------------------------------------
    # 8. Simultaneous state update
    # --------------------------------------------------------

    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = next_state


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # --------------------------------------------------------
    # 9. Independent local observations
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = agent.uuv_id


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        # Personally observed knowledge takes LOCAL provenance.
        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 10. Evaluation-only team metrics
    # ========================================================

    current_local_maps = {
        agent.uuv_id:
            agent
            .exploration_memory
            .observed
        for agent in agents
    }


    team_union = (
        np.logical_or.reduce(
            list(
                current_local_maps.values()
            )
        )
    )


    team_coverage = (
        int(
            team_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    alloc_team_coverage_history.append(
        float(
            team_coverage
        )
    )


    # --------------------------------------------------------
    # Mission-INCREMENTAL exploration masks
    # --------------------------------------------------------

    mission_new_masks = {
        agent.uuv_id:
            (
                current_local_maps[
                    agent.uuv_id
                ]
                &
                (
                    ~alloc_start_local_maps[
                        agent.uuv_id
                    ]
                )
            )
        for agent in agents
    }


    sum_new_local_cells = sum(
        int(
            mask.sum()
        )
        for mask
        in mission_new_masks.values()
    )


    mission_new_union = (
        np.logical_or.reduce(
            list(
                mission_new_masks.values()
            )
        )
    )


    unique_new_team_cells = int(
        mission_new_union.sum()
    )


    incremental_redundant_cells = (
        sum_new_local_cells
        -
        unique_new_team_cells
    )


    incremental_redundancy_ratio = (
        incremental_redundant_cells
        /
        sum_new_local_cells

        if sum_new_local_cells > 0

        else 0.0
    )


    alloc_incremental_redundancy_history.append(
        float(
            incremental_redundancy_ratio
        )
    )


# ============================================================
# Final invariants
# ============================================================

if (
    alloc_mission_min_separation
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Allocation-aware mission violated "
        "minimum safety separation."
    )


# ------------------------------------------------------------
# Final team coverage
# ------------------------------------------------------------

final_local_maps = {
    agent.uuv_id:
        agent.exploration_memory.observed
    for agent in agents
}


final_team_union = (
    np.logical_or.reduce(
        list(
            final_local_maps.values()
        )
    )
)


final_team_cells = int(
    final_team_union.sum()
)


final_team_coverage = (
    final_team_cells
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


team_coverage_gain = (
    final_team_coverage
    -
    start_team_coverage
)


# ============================================================
# Final cumulative redundancy
# ============================================================

final_sum_local_cells = sum(
    int(
        local_map.sum()
    )
    for local_map
    in final_local_maps.values()
)


final_cumulative_redundant_cells = (
    final_sum_local_cells
    -
    final_team_cells
)


final_cumulative_redundancy_ratio = (
    final_cumulative_redundant_cells
    /
    final_sum_local_cells

    if final_sum_local_cells > 0

    else 0.0
)


# ============================================================
# Final mission-incremental redundancy
# ============================================================

final_new_masks = {
    uuv_id:
        (
            final_local_maps[
                uuv_id
            ]
            &
            (
                ~alloc_start_local_maps[
                    uuv_id
                ]
            )
        )
    for uuv_id in final_local_maps
}


final_sum_new_local = sum(
    int(
        mask.sum()
    )
    for mask
    in final_new_masks.values()
)


final_new_union = (
    np.logical_or.reduce(
        list(
            final_new_masks.values()
        )
    )
)


final_unique_new_team = int(
    final_new_union.sum()
)


final_incremental_redundant_cells = (
    final_sum_new_local
    -
    final_unique_new_team
)


final_incremental_redundancy_ratio = (
    final_incremental_redundant_cells
    /
    final_sum_new_local

    if final_sum_new_local > 0

    else 0.0
)


# ============================================================
# Allocation statistics
# ============================================================

if allocation_goal_spacing_samples:

    mean_goal_spacing = float(
        np.mean(
            allocation_goal_spacing_samples
        )
    )

    minimum_goal_spacing = float(
        np.min(
            allocation_goal_spacing_samples
        )
    )

else:

    mean_goal_spacing = np.nan
    minimum_goal_spacing = np.nan


if allocation_overlap_samples:

    mean_task_overlap = float(
        np.mean(
            allocation_overlap_samples
        )
    )

    maximum_task_overlap = float(
        np.max(
            allocation_overlap_samples
        )
    )

else:

    mean_task_overlap = 0.0
    maximum_task_overlap = 0.0


# ============================================================
# Per-UUV final metrics
# ============================================================

final_reports = []


for agent in agents:

    uuv_id = agent.uuv_id

    local_cells = (
        agent.exploration_memory
        .observed_count()
    )


    known_cells = int(
        known_exploration_maps[
            uuv_id
        ].sum()
    )


    new_local_cells = int(
        final_new_masks[
            uuv_id
        ].sum()
    )


    total_cells = (
        WIDTH
        *
        HEIGHT
    )


    final_reports.append(
        {
            "uuv_id":
                uuv_id,

            "local_coverage_pct":
                (
                    local_cells
                    /
                    total_cells
                    *
                    100
                ),

            "known_coverage_pct":
                (
                    known_cells
                    /
                    total_cells
                    *
                    100
                ),

            "new_local_cells":
                new_local_cells,

            "remote_map_gain":
                allocation_remote_map_gain[
                    uuv_id
                ],

            "known_resources":
                len(
                    agent.resource_memory
                ),

            "remote_resource_gain":
                allocation_remote_resource_gain[
                    uuv_id
                ],

            "hold_steps":
                allocation_hold_counts[
                    uuv_id
                ],

            "safety_overrides":
                allocation_safety_overrides[
                    uuv_id
                ],
        }
    )


final_df = pd.DataFrame(
    final_reports
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 106
)

print(
    "MultiUUV-Link — Revised Stable Allocation-Aware "
    "Cooperative Exploration"
)

print(
    "=" * 106
)


print(
    f"\nMission steps                     : "
    f"{ALLOC_MISSION_STEPS}"
)

print(
    f"Periodic allocation interval      : "
    f"{ALLOCATION_INTERVAL}"
)

print(
    f"Minimum allocation cooldown       : "
    f"{ALLOCATION_COOLDOWN}"
)

print(
    f"Goal usefulness threshold         : "
    f"{GOAL_MIN_UNKNOWN_RATIO:.2f}"
)

print(
    f"Task-overlap penalty weight       : "
    f"{TASK_OVERLAP_WEIGHT:.2f}"
)


print(
    f"\nAllocation rounds                 : "
    f"{allocation_round_count}"
)

print(
    f"Allocation-induced goal changes   : "
    f"{allocation_goal_changes}"
)

print(
    f"Mean assigned-goal spacing        : "
    f"{mean_goal_spacing:.2f} px"
)

print(
    f"Minimum assigned-goal spacing     : "
    f"{minimum_goal_spacing:.2f} px"
)

print(
    f"Mean assigned-task FOV overlap    : "
    f"{mean_task_overlap:.4f}"
)

print(
    f"Maximum assigned-task FOV overlap : "
    f"{maximum_task_overlap:.4f}"
)


print(
    f"\nMean active communication edges   : "
    f"{np.mean(alloc_edge_history):.3f}"
)

print(
    f"Mission minimum separation        : "
    f"{alloc_mission_min_separation:.2f} px"
)


print(
    f"\nStart team coverage               : "
    f"{100 * start_team_coverage:.4f}%"
)

print(
    f"Final team coverage               : "
    f"{100 * final_team_coverage:.4f}%"
)

print(
    f"Team coverage gain                : "
    f"{100 * team_coverage_gain:.4f} percentage points"
)


print(
    f"\nMission new local cells (sum)     : "
    f"{final_sum_new_local}"
)

print(
    f"Mission unique new team cells     : "
    f"{final_unique_new_team}"
)

print(
    f"Mission redundant new cells       : "
    f"{final_incremental_redundant_cells}"
)

print(
    f"Mission incremental redundancy    : "
    f"{final_incremental_redundancy_ratio:.4f}"
)


print(
    f"\nCumulative redundant cells        : "
    f"{final_cumulative_redundant_cells}"
)

print(
    f"Cumulative redundancy ratio       : "
    f"{final_cumulative_redundancy_ratio:.4f}"
)


print(
    f"\nTotal HOLD planner steps          : "
    f"{sum(allocation_hold_counts.values())}"
)

print(
    f"Total safety overrides            : "
    f"{sum(allocation_safety_overrides.values())}"
)


print(
    "\nPer-UUV final state:"
)

print(
    final_df.to_string(
        index=False,
        formatters={
            "local_coverage_pct":
                lambda x: f"{x:.4f}",

            "known_coverage_pct":
                lambda x: f"{x:.4f}",
        },
    )
)


print(
    "\nImportant interpretation:"
)

print(
    "  Mission incremental redundancy measures overlap "
    "created specifically during this combo."
)

print(
    "  Cumulative redundancy includes exploration from "
    "all earlier notebook phases."
)

print(
    "  Combo 25 and Combo 27 are not yet a controlled "
    "A/B comparison because they begin from different states."
)


# ============================================================
# Allocation-round diagnostics
# ============================================================

allocation_round_df = pd.DataFrame(
    allocation_round_logs
)


if not allocation_round_df.empty:

    print(
        "\nAllocation-round diagnostics:"
    )

    print(
        allocation_round_df.to_string(
            index=False,
            formatters={
                "mean_spacing":
                    lambda x: (
                        "N/A"
                        if pd.isna(x)
                        else f"{x:.2f}"
                    ),

                "min_spacing":
                    lambda x: (
                        "N/A"
                        if pd.isna(x)
                        else f"{x:.2f}"
                    ),

                "mean_overlap":
                    lambda x: f"{x:.4f}",
            },
        )
    )


# ============================================================
# Visualization 1 — Mission trajectories
# ============================================================

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_index = (
        alloc_start[
            agent.uuv_id
        ][
            "trajectory_index"
        ]
    )


    path = np.asarray(
        agent.trajectory[
            start_index:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=80,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=120,
        linewidths=2.5,
    )


plt.title(
    "MultiUUV-Link — Revised Stable "
    "Allocation-Aware Exploration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Visualization 2 — Coverage and mission redundancy
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    np.asarray(
        alloc_team_coverage_history
    )
    *
    100,
    linewidth=2,
    label="Team coverage [%]",
)


plt.plot(
    np.asarray(
        alloc_incremental_redundancy_history
    )
    *
    100,
    linewidth=2,
    label="Mission incremental redundancy [%]",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Percent"
)

plt.title(
    "MultiUUV-Link — Revised Allocation-Aware "
    "Coverage and Incremental Redundancy"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Final validation
# ============================================================

if (
    not pd.isna(
        minimum_goal_spacing
    )
    and
    minimum_goal_spacing
    <
    MIN_ASSIGNED_GOAL_SEPARATION
):

    raise RuntimeError(
        "Final allocation history contains "
        "insufficiently separated goals."
    )


if (
    alloc_mission_min_separation
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Safety separation invariant failed."
    )


print(
    "\nRevised stable allocation-aware "
    "cooperative exploration validated."
)

# Combo 28 — Stochastic Packet-Loss Communication

## Objective

All communication implemented so far has been perfect whenever a geometric link exists.

For UUVs $i$ and $j$:

$$
(i,j)\in E_t
$$

has so far implied that a transmitted message is always successfully received.

Real limited communication should instead distinguish between:

```text
LINK AVAILABLE
```

and:

```text
MESSAGE SUCCESSFULLY DELIVERED
```

This combo introduces the first communication-degradation mechanism:

$$
\boxed{\text{Stochastic Packet Loss}}
$$

The geometric communication graph remains unchanged.

Packet loss affects only message delivery.

This combo validates the degraded communication channel independently before integrating it into a complete exploration mission.

There is still:

```text
NO latency
NO bandwidth limit
NO persistent hard link failure
NO vehicle failure
NO learned communication policy
```

---

# Geometric Link Availability

The existing communication graph remains:

$$
G_t=(V,E_t)
$$

where:

$$
(i,j)\in E_t
\iff
\|p_i-p_j\|_2\le R_c
$$

with:

$$
R_c=400\text{ px}
$$

This determines whether transmission is physically permitted.

However, an available edge does not guarantee successful delivery.

---

# Packet Delivery Model

For every directed message:

$$
m_{i\rightarrow j}
$$

define a Bernoulli delivery variable:

$$
Z_{ij}
\sim
Bernoulli(1-P_{loss})
$$

where:

$$
P_{loss}
$$

is the probability of message loss.

Thus:

$$
P(Z_{ij}=1)
=
1-P_{loss}
$$

and:

$$
P(Z_{ij}=0)
=
P_{loss}
$$

A message is delivered only when:

$$
Z_{ij}=1
$$

---

# Initial Packet-Loss Probability

For the first degraded-channel validation:

$$
P_{loss}=0.25
$$

Therefore the expected delivery probability is:

$$
P_{delivery}=0.75
$$

This value is algorithmic rather than a claim about a specific underwater modem.

It simply creates a meaningful imperfect-communication regime.

---

# Directed Delivery

Although the geometric graph is undirected, packet delivery events are evaluated independently in each direction.

Therefore it is possible that:

$$
i\rightarrow j
$$

succeeds while:

$$
j\rightarrow i
$$

fails during the same communication round.

This creates realistic asymmetric information states even when geometric connectivity is symmetric.

---

# Message Semantics

The same one-hop synchronous communication rule remains.

At round start:

$$
K_i^0
$$

is snapshotted.

Messages are generated from:

$$
K_i^0
$$

Then packet-loss sampling determines which messages survive.

Finally successful payloads are merged.

Therefore:

$$
Snapshot
\rightarrow
Generate
\rightarrow
Drop/Deliver
\rightarrow
Merge
$$

No message received during the current round is retransmitted during that same round.

---

# Resource Messages

For resource communication:

$$
m_{i\rightarrow j}^{resource}
$$

contains only resource records already known by sender $i$.

A dropped resource message contributes no receiver knowledge.

---

# Map Messages

For map communication:

$$
m_{i\rightarrow j}^{map}
$$

contains sender $i$'s known exploration mask at round start.

A dropped map message leaves receiver knowledge unchanged for that message.

---

# Communication Statistics

For a communication round define:

### Attempted packets

$$
N_{tx}
$$

### Successfully delivered packets

$$
N_{rx}
$$

### Dropped packets

$$
N_{drop}
=
N_{tx}-N_{rx}
$$

### Delivery ratio

$$
\eta
=
\frac{N_{rx}}{N_{tx}}
$$

when:

$$
N_{tx}>0
$$

The observed delivery ratio will vary stochastically around:

$$
1-P_{loss}
$$

---

# Reproducibility

Communication degradation should normally be stochastic.

Therefore:

```text
PACKET_LOSS_SEED = None
```

produces a different delivery realization each run.

For debugging:

```text
PACKET_LOSS_SEED = 42
```

provides deterministic message-loss behavior.

---

# Important Separation of Concerns

Packet loss must not alter the communication graph.

A link can exist:

$$
(i,j)\in E_t
$$

while a particular message still fails.

Therefore:

$$
\boxed{
\text{Connectivity}
\neq
\text{Successful Information Transfer}
}
$$

This distinction becomes essential for later:

- latency;
- link-loss detection;
- heartbeat timeout;
- reconnection;
- knowledge synchronization.

---

# Synthetic Statistical Validation

A single communication round contains too few packets to verify a stochastic probability reliably.

Therefore this combo also performs a synthetic Monte Carlo channel test with:

$$
N=10{,}000
$$

independent transmissions.

For:

$$
P_{loss}=0.25
$$

we expect approximately:

$$
\eta\approx0.75
$$

The measured value should lie reasonably close to the theoretical probability.

A tolerance such as:

$$
|\eta-0.75|<0.03
$$

is sufficient for this validation.

---

# Current-Swarm Validation

The current communication graph is used to perform one degraded message-exchange round on **temporary copies** of the current knowledge state.

This is important.

Combo 28 should validate the channel without mutating the real mission state required for later controlled experiments.

Thus:

$$
K_i^{test}
=
copy(K_i)
$$

and degraded communication operates only on:

$$
K_i^{test}
$$

The live agents and their memories remain untouched.

---

# Expected Output

The code cell should:

1. define:

$$
P_{loss}=0.25
$$

2. implement independent directed packet delivery;
3. retain the existing range-limited communication graph;
4. perform degraded resource-message exchange on temporary memories;
5. perform degraded map-message exchange on temporary known maps;
6. count:
   - attempted messages;
   - delivered messages;
   - dropped messages;
   - delivery ratio;
7. verify live agent memories remain unchanged;
8. perform a 10,000-packet Monte Carlo validation;
9. verify empirical delivery rate approximately matches:

$$
1-P_{loss}
$$

10. print per-direction delivery results for the current communication graph.

A successful result establishes:

$$
\boxed{
\text{Available Link}
+
\text{Stochastic Packet Delivery}
}
$$

---

In [ ]:
# ============================================================
# Combo 28 — Stochastic Packet-Loss Communication
# ============================================================

from copy import deepcopy


# ------------------------------------------------------------
# Packet-loss configuration
# ------------------------------------------------------------

PACKET_LOSS_PROBABILITY = 0.25

# None = stochastic every run
# integer = deterministic debugging
PACKET_LOSS_SEED = None

packet_rng = np.random.default_rng(
    PACKET_LOSS_SEED
)


# ------------------------------------------------------------
# Basic packet-delivery primitive
# ------------------------------------------------------------

def packet_delivered(
    rng,
    loss_probability=
        PACKET_LOSS_PROBABILITY,
):
    """
    Return True if one directed packet survives the channel.
    """

    if not (
        0.0
        <=
        loss_probability
        <=
        1.0
    ):
        raise ValueError(
            "loss_probability must lie inside [0, 1]."
        )


    return bool(
        rng.random()
        >=
        loss_probability
    )


# ------------------------------------------------------------
# Preserve LIVE state for non-mutation validation
# ------------------------------------------------------------

live_resource_signatures_before = {
    agent.uuv_id:
        sorted(
            resource.resource_id
            for resource
            in agent.resource_memory.all_resources()
        )
    for agent in agents
}


live_known_map_counts_before = {
    uuv_id:
        int(
            known_map.sum()
        )
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Temporary degraded-communication state
# ============================================================

temporary_resource_memories = {
    agent.uuv_id:
        deepcopy(
            agent.resource_memory
        )
    for agent in agents
}


temporary_known_maps = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ------------------------------------------------------------
# Current geometric communication graph
# ------------------------------------------------------------

packet_test_positions = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )
    for agent in agents
}


packet_test_graph = (
    build_communication_graph(
        packet_test_positions,
        communication_range=
            COMMUNICATION_RANGE,
    )
)


# ============================================================
# Resource communication with packet loss
# ============================================================

# Pre-round snapshots
resource_snapshots = {
    uuv_id: {
        resource.resource_id:
            resource
        for resource
        in memory.all_resources()
    }
    for uuv_id, memory
    in temporary_resource_memories.items()
}


resource_delivery_records = []


for uuv_a, uuv_b in (
    packet_test_graph.edges()
):

    for sender, receiver in (
        (uuv_a, uuv_b),
        (uuv_b, uuv_a),
    ):

        delivered = packet_delivered(
            packet_rng
        )


        payload = list(
            resource_snapshots[
                sender
            ].values()
        )


        before_count = len(
            temporary_resource_memories[
                receiver
            ]
        )


        if delivered:

            temporary_resource_memories[
                receiver
            ].update(
                payload
            )


        after_count = len(
            temporary_resource_memories[
                receiver
            ]
        )


        resource_delivery_records.append(
            {
                "sender":
                    sender,

                "receiver":
                    receiver,

                "payload_records":
                    len(
                        payload
                    ),

                "delivered":
                    delivered,

                "new_resources":
                    (
                        after_count
                        -
                        before_count
                    ),
            }
        )


# ============================================================
# Map communication with packet loss
# ============================================================

# Snapshot before map round
map_snapshots = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in temporary_known_maps.items()
}


map_delivery_records = []


for uuv_a, uuv_b in (
    packet_test_graph.edges()
):

    for sender, receiver in (
        (uuv_a, uuv_b),
        (uuv_b, uuv_a),
    ):

        delivered = packet_delivered(
            packet_rng
        )


        before_count = int(
            temporary_known_maps[
                receiver
            ].sum()
        )


        if delivered:

            temporary_known_maps[
                receiver
            ] = np.logical_or(
                temporary_known_maps[
                    receiver
                ],
                map_snapshots[
                    sender
                ],
            )


        after_count = int(
            temporary_known_maps[
                receiver
            ].sum()
        )


        map_delivery_records.append(
            {
                "sender":
                    sender,

                "receiver":
                    receiver,

                "delivered":
                    delivered,

                "new_map_cells":
                    (
                        after_count
                        -
                        before_count
                    ),
            }
        )


# ============================================================
# Combined packet statistics
# ============================================================

all_delivery_records = (
    [
        {
            "type":
                "resource",

            "sender":
                record[
                    "sender"
                ],

            "receiver":
                record[
                    "receiver"
                ],

            "delivered":
                record[
                    "delivered"
                ],
        }
        for record in resource_delivery_records
    ]
    +
    [
        {
            "type":
                "map",

            "sender":
                record[
                    "sender"
                ],

            "receiver":
                record[
                    "receiver"
                ],

            "delivered":
                record[
                    "delivered"
                ],
        }
        for record in map_delivery_records
    ]
)


attempted_packets = len(
    all_delivery_records
)


delivered_packets = sum(
    record[
        "delivered"
    ]
    for record
    in all_delivery_records
)


dropped_packets = (
    attempted_packets
    -
    delivered_packets
)


observed_delivery_ratio = (
    delivered_packets
    /
    attempted_packets

    if attempted_packets > 0

    else np.nan
)


# ============================================================
# Verify LIVE state was not mutated
# ============================================================

live_resource_signatures_after = {
    agent.uuv_id:
        sorted(
            resource.resource_id
            for resource
            in agent.resource_memory.all_resources()
        )
    for agent in agents
}


live_known_map_counts_after = {
    uuv_id:
        int(
            known_map.sum()
        )
    for uuv_id, known_map
    in known_exploration_maps.items()
}


resource_state_unchanged = (
    live_resource_signatures_before
    ==
    live_resource_signatures_after
)


known_maps_unchanged = (
    live_known_map_counts_before
    ==
    live_known_map_counts_after
)


if not resource_state_unchanged:

    raise RuntimeError(
        "Packet-loss validation mutated "
        "live resource memories."
    )


if not known_maps_unchanged:

    raise RuntimeError(
        "Packet-loss validation mutated "
        "live known exploration maps."
    )


# ============================================================
# Monte Carlo channel validation
# ============================================================

MONTE_CARLO_PACKETS = 10_000

monte_carlo_rng = np.random.default_rng(
    123456
)


monte_carlo_deliveries = np.fromiter(
    (
        packet_delivered(
            monte_carlo_rng,
            loss_probability=
                PACKET_LOSS_PROBABILITY,
        )
        for _ in range(
            MONTE_CARLO_PACKETS
        )
    ),
    dtype=bool,
    count=
        MONTE_CARLO_PACKETS,
)


monte_carlo_delivery_ratio = float(
    monte_carlo_deliveries.mean()
)


expected_delivery_ratio = (
    1.0
    -
    PACKET_LOSS_PROBABILITY
)


delivery_error = abs(
    monte_carlo_delivery_ratio
    -
    expected_delivery_ratio
)


MONTE_CARLO_TOLERANCE = 0.03


monte_carlo_passed = (
    delivery_error
    <
    MONTE_CARLO_TOLERANCE
)


if not monte_carlo_passed:

    raise RuntimeError(
        "Monte Carlo packet-loss validation "
        "fell outside tolerance."
    )


# ============================================================
# Console report
# ============================================================

print(
    "=" * 92
)

print(
    "MultiUUV-Link — Stochastic Packet-Loss Communication"
)

print(
    "=" * 92
)


print(
    f"\nCommunication range         : "
    f"{COMMUNICATION_RANGE:.2f} px"
)

print(
    f"Packet-loss probability     : "
    f"{PACKET_LOSS_PROBABILITY:.2f}"
)

print(
    f"Expected delivery ratio     : "
    f"{expected_delivery_ratio:.2f}"
)

print(
    f"Packet-loss seed            : "
    f"{PACKET_LOSS_SEED}"
)

print(
    f"Current geometric edges     : "
    f"{packet_test_graph.number_of_edges()}"
)


# ------------------------------------------------------------
# Resource packet table
# ------------------------------------------------------------

print(
    "\nResource-message delivery:"
)


if resource_delivery_records:

    resource_delivery_df = pd.DataFrame(
        resource_delivery_records
    )

    print(
        resource_delivery_df.to_string(
            index=False
        )
    )

else:

    print(
        "  No current communication links."
    )


# ------------------------------------------------------------
# Map packet table
# ------------------------------------------------------------

print(
    "\nMap-message delivery:"
)


if map_delivery_records:

    map_delivery_df = pd.DataFrame(
        map_delivery_records
    )

    print(
        map_delivery_df.to_string(
            index=False
        )
    )

else:

    print(
        "  No current communication links."
    )


# ------------------------------------------------------------
# Round statistics
# ------------------------------------------------------------

print(
    "\nCurrent degraded communication round:"
)

print(
    f"  Attempted packets : "
    f"{attempted_packets}"
)

print(
    f"  Delivered packets : "
    f"{delivered_packets}"
)

print(
    f"  Dropped packets   : "
    f"{dropped_packets}"
)

print(
    f"  Delivery ratio    : "
    f"{observed_delivery_ratio:.4f}"
    if attempted_packets > 0
    else
    "  Delivery ratio    : N/A"
)


# ------------------------------------------------------------
# Monte Carlo statistics
# ------------------------------------------------------------

print(
    "\nMonte Carlo channel validation:"
)

print(
    f"  Trials                  : "
    f"{MONTE_CARLO_PACKETS}"
)

print(
    f"  Expected delivery ratio : "
    f"{expected_delivery_ratio:.4f}"
)

print(
    f"  Observed delivery ratio : "
    f"{monte_carlo_delivery_ratio:.4f}"
)

print(
    f"  Absolute error          : "
    f"{delivery_error:.4f}"
)

print(
    f"  Tolerance               : "
    f"{MONTE_CARLO_TOLERANCE:.4f}"
)

print(
    f"  Statistical test passed : "
    f"{monte_carlo_passed}"
)


# ------------------------------------------------------------
# Integrity report
# ------------------------------------------------------------

print(
    "\nState-integrity checks:"
)

print(
    f"  Live resource memories unchanged : "
    f"{resource_state_unchanged}"
)

print(
    f"  Live known maps unchanged        : "
    f"{known_maps_unchanged}"
)


print(
    "\nStochastic packet-loss "
    "communication validated."
)

# Combo 29 — Delayed Communication and Stale Messages

## Objective

Combo 28 introduced stochastic packet loss while preserving instantaneous delivery for surviving packets.

This combo introduces communication latency.

A transmitted message may now:

1. be dropped;
2. survive;
3. arrive after a nonzero delay.

Therefore communication becomes:

```text
Generate Message
      ↓
Packet-Loss Test
      ↓
If Delivered:
Assign Arrival Time
      ↓
Place in Message Queue
      ↓
Time Advances
      ↓
Deliver When Due
```

This combo validates delayed-message mechanics independently.

It does not yet integrate latency into the full cooperative mission.

---

# Message Generation Time

For message:

$$
m
$$

generated at discrete communication timestep:

$$
t_s
$$

define:

$$
t_{send}=t_s
$$

---

# Communication Delay

Each surviving packet receives an integer delay:

$$
\tau
$$

measured in simulation timesteps.

The initial latency model is:

$$
\tau\sim U_{\mathbb Z}
\{1,2,3,4,5\}
$$

Therefore:

$$
1\le\tau\le5
$$

and every successfully transmitted packet is delayed by at least one timestep.

The arrival timestep is:

$$
t_{arrival}
=
t_{send}
+
\tau
$$

---

# Why Delay Is Discrete

The current autonomy simulator evolves in discrete timesteps.

Therefore communication latency is represented using the same temporal resolution.

This is an algorithmic abstraction rather than a physical acoustic propagation model.

Later parameter calibration may map one simulation step to physical time.

---

# Message Queue

Pending communication is represented using a queue:

$$
Q(t)
$$

containing messages whose:

$$
t_{arrival}>t
$$

At each simulation timestep:

1. inspect all queued messages;
2. deliver every message satisfying:

$$
t_{arrival}\le t
$$

3. retain later messages in the queue.

Thus:

$$
Q(t+1)
=
Q(t)
-
Messages_{delivered}(t)
+
Messages_{new}(t)
$$

---

# Message Staleness

A delayed message contains a snapshot of sender knowledge from:

$$
t_{send}
$$

not sender knowledge at:

$$
t_{arrival}
$$

Therefore its age on delivery is:

$$
A_m
=
t_{arrival}
-
t_{send}
=
\tau
$$

The payload is intentionally not refreshed while waiting in the queue.

This creates realistic stale information.

---

# Monotonic Knowledge Merge

The current map and resource knowledge representations are monotonic.

For maps:

$$
M_i^{known,+}
=
M_i^{known}
\lor
M_{message}
$$

For resources:

$$
D_i^+
=
D_i
\cup
D_{message}
$$

Therefore an old message cannot erase newer information.

This is an important property:

$$
\boxed{
\text{Stale Knowledge May Be Incomplete,
But It Cannot Regress Knowledge}
}
$$

with the present boolean/set-based representations.

Later richer state estimates may require timestamps or conflict-resolution logic.

---

# Packet Loss Plus Latency

The complete communication-channel model now becomes:

$$
m
\xrightarrow{Bernoulli}
\begin{cases}
DROP\\
QUEUE(\tau)
\end{cases}
$$

where:

$$
P(DROP)=P_{loss}
$$

and for surviving messages:

$$
\tau
\sim
U_{\mathbb Z}
\{1,\ldots,5\}
$$

---

# Independent Directional Delay

Just as packet loss is directional, latency is sampled independently for each directed message.

Thus:

$$
\tau_{ij}
\neq
\tau_{ji}
$$

in general.

---

# Controlled Validation Scenario

The validation uses temporary communication state.

At timestep:

$$
t=0
$$

several synthetic messages are generated.

The test advances:

$$
t=0,1,\ldots,6
$$

and records:

- queue length;
- delivered packets;
- message age;
- receiver knowledge gain.

The live swarm state remains unchanged.

---

# Latency Distribution Validation

A Monte Carlo validation samples:

$$
N=10{,}000
$$

latencies.

For:

$$
\tau\sim U_{\mathbb Z}\{1,2,3,4,5\}
$$

the theoretical mean is:

$$
E[\tau]
=
\frac{1+5}{2}
=
3
$$

The observed mean should therefore satisfy approximately:

$$
\bar\tau\approx3
$$

A tolerance of:

$$
|\bar\tau-3|<0.1
$$

is sufficient.

Additionally:

$$
\min(\tau)=1
$$

and:

$$
\max(\tau)=5
$$

should appear in a sufficiently large sample.

---

# Queue Invariants

Every queued message must satisfy:

$$
t_{arrival}>t_{current}
$$

Every delivered message must satisfy:

$$
t_{arrival}\le t_{current}
$$

A message must be delivered exactly once.

Thus:

$$
N_{queued}
+
N_{delivered}
+
N_{dropped}
=
N_{generated}
$$

for the completed test horizon.

---

# State Integrity

As in Combo 28, all delayed communication tests operate on temporary copies.

The live:

- UUV states;
- local exploration memories;
- known exploration maps;
- resource memories;

must remain unchanged.

---

# Expected Output

The code cell should:

1. define latency range:

$$
1\le\tau\le5
$$

2. implement delayed-message queue records;
3. combine packet loss with latency;
4. generate temporary map/resource messages;
5. advance simulation time;
6. deliver messages only when due;
7. report:
   - generated packets;
   - dropped packets;
   - queued packets over time;
   - delivered packets;
   - individual message ages;
8. validate no message is delivered early or twice;
9. run a 10,000-sample latency Monte Carlo test;
10. verify:

$$
\bar\tau\approx3
$$

11. verify live mission state remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Packet Loss}
+
\text{Delayed Delivery}
+
\text{Stale Knowledge}
}
$$

---

In [ ]:
# ============================================================
# Combo 29 — Delayed Communication and Stale Messages
# ============================================================

from copy import deepcopy


# ------------------------------------------------------------
# Latency configuration
# ------------------------------------------------------------

MIN_COMM_DELAY = 1
MAX_COMM_DELAY = 5

LATENCY_TEST_SEED = None

latency_rng = np.random.default_rng(
    LATENCY_TEST_SEED
)


# ------------------------------------------------------------
# Sample integer communication delay
# ------------------------------------------------------------

def sample_message_delay(
    rng,
    min_delay=MIN_COMM_DELAY,
    max_delay=MAX_COMM_DELAY,
):
    if min_delay < 0:
        raise ValueError(
            "min_delay must be non-negative."
        )

    if max_delay < min_delay:
        raise ValueError(
            "max_delay must be >= min_delay."
        )

    return int(
        rng.integers(
            min_delay,
            max_delay + 1,
        )
    )


# ------------------------------------------------------------
# Message constructor
# ------------------------------------------------------------

def create_delayed_message(
    sender,
    receiver,
    message_type,
    payload,
    send_time,
    rng,
    loss_probability=
        PACKET_LOSS_PROBABILITY,
):
    """
    Create one directed packet.

    Returns:
        None      -> packet dropped
        dict      -> packet survives and enters queue
    """

    delivered = packet_delivered(
        rng,
        loss_probability=
            loss_probability,
    )

    if not delivered:
        return None


    delay = sample_message_delay(
        rng
    )


    return {
        "message_id":
            None,

        "sender":
            sender,

        "receiver":
            receiver,

        "message_type":
            message_type,

        "payload":
            deepcopy(
                payload
            ),

        "send_time":
            int(
                send_time
            ),

        "delay":
            int(
                delay
            ),

        "arrival_time":
            int(
                send_time
                +
                delay
            ),
    }


# ============================================================
# Preserve LIVE state
# ============================================================

live_signature_before_29 = (
    get_swarm_state_signature()
)


live_known_maps_before_29 = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Temporary knowledge copies
# ============================================================

temp_resource_memories_29 = {
    agent.uuv_id:
        deepcopy(
            agent.resource_memory
        )
    for agent in agents
}


temp_known_maps_29 = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ------------------------------------------------------------
# Communication graph for validation
# ------------------------------------------------------------

latency_test_positions = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )
    for agent in agents
}


latency_test_graph = (
    build_communication_graph(
        latency_test_positions,
        communication_range=
            COMMUNICATION_RANGE,
    )
)


# ============================================================
# Snapshot payloads at t = 0
# ============================================================

resource_snapshot_29 = {
    uuv_id:
        list(
            memory.all_resources()
        )
    for uuv_id, memory
    in temp_resource_memories_29.items()
}


map_snapshot_29 = {
    uuv_id:
        known_map.copy()
    for uuv_id, known_map
    in temp_known_maps_29.items()
}


# ============================================================
# Generate delayed messages
# ============================================================

generated_packet_count = 0
dropped_packet_count = 0

message_queue = []

next_message_id = 0


for uuv_a, uuv_b in (
    latency_test_graph.edges()
):

    for sender, receiver in (
        (uuv_a, uuv_b),
        (uuv_b, uuv_a),
    ):

        # ----------------------------------------------------
        # Resource message
        # ----------------------------------------------------

        generated_packet_count += 1


        resource_message = (
            create_delayed_message(
                sender=
                    sender,

                receiver=
                    receiver,

                message_type=
                    "resource",

                payload=
                    resource_snapshot_29[
                        sender
                    ],

                send_time=
                    0,

                rng=
                    latency_rng,
            )
        )


        if resource_message is None:

            dropped_packet_count += 1

        else:

            resource_message[
                "message_id"
            ] = next_message_id

            next_message_id += 1

            message_queue.append(
                resource_message
            )


        # ----------------------------------------------------
        # Map message
        # ----------------------------------------------------

        generated_packet_count += 1


        map_message = (
            create_delayed_message(
                sender=
                    sender,

                receiver=
                    receiver,

                message_type=
                    "map",

                payload=
                    map_snapshot_29[
                        sender
                    ],

                send_time=
                    0,

                rng=
                    latency_rng,
            )
        )


        if map_message is None:

            dropped_packet_count += 1

        else:

            map_message[
                "message_id"
            ] = next_message_id

            next_message_id += 1

            message_queue.append(
                map_message
            )


# ============================================================
# Simulate queue progression
# ============================================================

delivered_messages = []

queue_history = []

delivered_message_ids = set()


TEST_FINAL_TIME = (
    MAX_COMM_DELAY
    +
    1
)


for current_time in range(
    TEST_FINAL_TIME + 1
):

    # --------------------------------------------------------
    # Queue invariant before delivery
    # --------------------------------------------------------

    for message in message_queue:

        if (
            message[
                "arrival_time"
            ]
            <=
            current_time - 1
        ):

            raise RuntimeError(
                "Overdue message remained in queue."
            )


    due_messages = [
        message
        for message in message_queue
        if (
            message[
                "arrival_time"
            ]
            <=
            current_time
        )
    ]


    future_messages = [
        message
        for message in message_queue
        if (
            message[
                "arrival_time"
            ]
            >
            current_time
        )
    ]


    # --------------------------------------------------------
    # Deliver all due messages
    # --------------------------------------------------------

    for message in due_messages:

        message_id = (
            message[
                "message_id"
            ]
        )


        if (
            message_id
            in
            delivered_message_ids
        ):

            raise RuntimeError(
                "Message delivered more than once."
            )


        if (
            current_time
            <
            message[
                "arrival_time"
            ]
        ):

            raise RuntimeError(
                "Message delivered before arrival time."
            )


        receiver = (
            message[
                "receiver"
            ]
        )


        before_resource_count = len(
            temp_resource_memories_29[
                receiver
            ]
        )


        before_map_count = int(
            temp_known_maps_29[
                receiver
            ].sum()
        )


        if (
            message[
                "message_type"
            ]
            ==
            "resource"
        ):

            temp_resource_memories_29[
                receiver
            ].update(
                message[
                    "payload"
                ]
            )


        elif (
            message[
                "message_type"
            ]
            ==
            "map"
        ):

            temp_known_maps_29[
                receiver
            ] = np.logical_or(
                temp_known_maps_29[
                    receiver
                ],
                message[
                    "payload"
                ],
            )


        else:

            raise RuntimeError(
                "Unknown message type."
            )


        after_resource_count = len(
            temp_resource_memories_29[
                receiver
            ]
        )


        after_map_count = int(
            temp_known_maps_29[
                receiver
            ].sum()
        )


        delivered_record = {
            "message_id":
                message_id,

            "type":
                message[
                    "message_type"
                ],

            "sender":
                message[
                    "sender"
                ],

            "receiver":
                receiver,

            "send_time":
                message[
                    "send_time"
                ],

            "arrival_time":
                message[
                    "arrival_time"
                ],

            "delivery_time":
                current_time,

            "age":
                (
                    current_time
                    -
                    message[
                        "send_time"
                    ]
                ),

            "new_resources":
                (
                    after_resource_count
                    -
                    before_resource_count
                ),

            "new_map_cells":
                (
                    after_map_count
                    -
                    before_map_count
                ),
        }


        delivered_messages.append(
            delivered_record
        )


        delivered_message_ids.add(
            message_id
        )


    message_queue = (
        future_messages
    )


    queue_history.append(
        {
            "time":
                current_time,

            "delivered_this_step":
                len(
                    due_messages
                ),

            "queue_remaining":
                len(
                    message_queue
                ),

            "cumulative_delivered":
                len(
                    delivered_messages
                ),
        }
    )


# ============================================================
# Queue completion validation
# ============================================================

if len(message_queue) != 0:

    raise RuntimeError(
        "Delayed-message queue did not empty."
    )


surviving_packet_count = (
    generated_packet_count
    -
    dropped_packet_count
)


if (
    len(
        delivered_messages
    )
    !=
    surviving_packet_count
):

    raise RuntimeError(
        "Delivered message count does not match "
        "surviving packet count."
    )


if (
    generated_packet_count
    !=
    (
        dropped_packet_count
        +
        len(
            delivered_messages
        )
    )
):

    raise RuntimeError(
        "Packet accounting invariant failed."
    )


# ============================================================
# Monte Carlo latency validation
# ============================================================

LATENCY_MONTE_CARLO_SAMPLES = 10_000

latency_mc_rng = np.random.default_rng(
    654321
)


latency_samples = np.fromiter(
    (
        sample_message_delay(
            latency_mc_rng
        )
        for _ in range(
            LATENCY_MONTE_CARLO_SAMPLES
        )
    ),
    dtype=int,
    count=
        LATENCY_MONTE_CARLO_SAMPLES,
)


observed_latency_mean = float(
    latency_samples.mean()
)


theoretical_latency_mean = (
    (
        MIN_COMM_DELAY
        +
        MAX_COMM_DELAY
    )
    /
    2.0
)


latency_mean_error = abs(
    observed_latency_mean
    -
    theoretical_latency_mean
)


LATENCY_MEAN_TOLERANCE = 0.10


latency_test_passed = (
    latency_mean_error
    <
    LATENCY_MEAN_TOLERANCE
    and
    int(
        latency_samples.min()
    )
    ==
    MIN_COMM_DELAY
    and
    int(
        latency_samples.max()
    )
    ==
    MAX_COMM_DELAY
)


if not latency_test_passed:

    raise RuntimeError(
        "Latency Monte Carlo validation failed."
    )


# ============================================================
# Verify live state unchanged
# ============================================================

live_signature_after_29 = (
    get_swarm_state_signature()
)


live_signature_unchanged = (
    live_signature_before_29
    ==
    live_signature_after_29
)


live_known_maps_unchanged = all(
    np.array_equal(
        live_known_maps_before_29[
            uuv_id
        ],
        known_exploration_maps[
            uuv_id
        ],
    )
    for uuv_id
    in known_exploration_maps
)


if not live_signature_unchanged:

    raise RuntimeError(
        "Latency test mutated live swarm state."
    )


if not live_known_maps_unchanged:

    raise RuntimeError(
        "Latency test mutated live known maps."
    )


# ============================================================
# Reports
# ============================================================

print(
    "=" * 96
)

print(
    "MultiUUV-Link — Delayed Communication "
    "and Stale Messages"
)

print(
    "=" * 96
)


print(
    f"\nPacket-loss probability    : "
    f"{PACKET_LOSS_PROBABILITY:.2f}"
)

print(
    f"Delay range                : "
    f"[{MIN_COMM_DELAY}, "
    f"{MAX_COMM_DELAY}] timesteps"
)

print(
    f"Latency test seed          : "
    f"{LATENCY_TEST_SEED}"
)

print(
    f"Current communication edges: "
    f"{latency_test_graph.number_of_edges()}"
)


print(
    "\nPacket accounting:"
)

print(
    f"  Generated packets : "
    f"{generated_packet_count}"
)

print(
    f"  Dropped packets   : "
    f"{dropped_packet_count}"
)

print(
    f"  Surviving packets : "
    f"{surviving_packet_count}"
)

print(
    f"  Delivered packets : "
    f"{len(delivered_messages)}"
)


# ------------------------------------------------------------
# Queue history
# ------------------------------------------------------------

queue_history_df = pd.DataFrame(
    queue_history
)


print(
    "\nQueue progression:"
)

print(
    queue_history_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Delivery details
# ------------------------------------------------------------

if delivered_messages:

    delivered_df = pd.DataFrame(
        delivered_messages
    )


    print(
        "\nDelivered message details:"
    )


    print(
        delivered_df.to_string(
            index=False
        )
    )


    print(
        f"\nObserved delivered-message "
        f"mean age : "
        f"{delivered_df['age'].mean():.3f} "
        f"timesteps"
    )

else:

    print(
        "\nNo packets survived this "
        "particular degraded round."
    )


# ------------------------------------------------------------
# Latency Monte Carlo report
# ------------------------------------------------------------

print(
    "\nMonte Carlo latency validation:"
)

print(
    f"  Samples                  : "
    f"{LATENCY_MONTE_CARLO_SAMPLES}"
)

print(
    f"  Theoretical mean latency : "
    f"{theoretical_latency_mean:.4f}"
)

print(
    f"  Observed mean latency    : "
    f"{observed_latency_mean:.4f}"
)

print(
    f"  Absolute mean error      : "
    f"{latency_mean_error:.4f}"
)

print(
    f"  Minimum sampled delay    : "
    f"{int(latency_samples.min())}"
)

print(
    f"  Maximum sampled delay    : "
    f"{int(latency_samples.max())}"
)

print(
    f"  Statistical test passed  : "
    f"{latency_test_passed}"
)


# ------------------------------------------------------------
# Integrity
# ------------------------------------------------------------

print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm signature unchanged : "
    f"{live_signature_unchanged}"
)

print(
    f"  Live known maps unchanged      : "
    f"{live_known_maps_unchanged}"
)


print(
    "\nDelayed communication and "
    "stale-message handling validated."
)

# Combo 30 — Persistent Communication Blackout and Link-Failure Model

## Objective

Combos 28 and 29 introduced:

- stochastic packet loss;
- delayed message delivery;
- stale communication payloads.

However, both mechanisms assume the underlying geometric communication link remains operational.

This combo introduces a more severe communication fault:

$$
\boxed{\text{Persistent Link Blackout}}
$$

A UUV pair may remain geometrically within communication range while its logical communication link is temporarily unavailable.

Therefore the system must now distinguish three states:

```text
GEOMETRICALLY OUT OF RANGE
GEOMETRICALLY IN RANGE + LINK HEALTHY
GEOMETRICALLY IN RANGE + LINK FAILED
```

This creates the first explicit **link-failure abstraction** required by the MultiUUV-Link development plan.

This combo validates blackout generation and link suppression independently.

It does not yet integrate link recovery into the complete cooperative mission.

---

# Geometric Communication Graph

The original range-limited communication graph remains:

$$
G_t^{geo}
=
(V,E_t^{geo})
$$

where:

$$
(i,j)\in E_t^{geo}
\iff
\|p_i-p_j\|_2\le R_c
$$

This graph represents physical geometric eligibility.

---

# Operational Communication Graph

A second graph is introduced:

$$
G_t^{op}
=
(V,E_t^{op})
$$

An operational edge requires both:

1. geometric range eligibility;
2. healthy link state.

Therefore:

$$
(i,j)\in E_t^{op}
$$

if and only if:

$$
(i,j)\in E_t^{geo}
$$

and:

$$
H_{ij}(t)=1
$$

where:

$$
H_{ij}(t)
$$

is the binary link-health state.

---

# Link Health

For each possible UUV pair:

$$
(i,j)
$$

define:

$$
H_{ij}(t)
\in
\{0,1\}
$$

where:

$$
H_{ij}=1
$$

means healthy and:

$$
H_{ij}=0
$$

means blackout.

Thus:

$$
E_t^{op}
=
\{
(i,j)\in E_t^{geo}
\mid
H_{ij}(t)=1
\}
$$

---

# Blackout Event

A blackout event is represented by:

$$
B=
(i,j,t_s,T_B)
$$

where:

- $(i,j)$ is the affected link;
- $t_s$ is blackout start timestep;
- $T_B$ is blackout duration.

The link remains failed for:

$$
t_s
\le
t
<
t_s+T_B
$$

and automatically recovers afterward.

---

# Initial Blackout Duration

For the prototype, blackout duration is sampled as:

$$
T_B
\sim
U_{\mathbb Z}
\{10,\ldots,30\}
$$

timesteps.

This represents a sustained communication interruption rather than isolated packet loss.

---

# Blackout Trigger Probability

At each timestep, each healthy geometric link may independently begin a blackout with probability:

$$
P_B=0.02
$$

Thus:

$$
P(
\text{blackout begins}
)
=
0.02
$$

per eligible healthy link per timestep.

Once a link is already blacked out, it cannot trigger another blackout until the current one ends.

---

# Bidirectional Failure

Unlike packet loss, which is directional, the first blackout model is symmetric.

If link:

$$
(i,j)
$$

fails, then both:

$$
i\rightarrow j
$$

and:

$$
j\rightarrow i
$$

are unavailable.

Therefore:

$$
H_{ij}
=
H_{ji}
$$

This represents failure of the shared communication relationship rather than failure of one packet direction.

---

# Interaction with Packet Loss and Latency

The complete future communication stack becomes:

```text
Geometric Range Test
        ↓
Link Health Test
        ↓
If Operational:
    Generate Packet
        ↓
    Packet Loss
        ↓
    If Survives:
        Latency Queue
```

Therefore:

$$
\boxed{
\text{Range}
\rightarrow
\text{Link Health}
\rightarrow
\text{Packet Loss}
\rightarrow
\text{Latency}
}
$$

Blackout operates before packet-level effects.

If the link is failed:

$$
H_{ij}=0
$$

then no packet is attempted at all.

---

# Blackout Manager

The simulator maintains a table:

$$
L_{ij}
$$

containing the remaining blackout duration for every UUV pair.

If:

$$
L_{ij}=0
$$

the link is healthy.

If:

$$
L_{ij}>0
$$

the link is failed.

At each timestep:

1. decrement active blackout timers;
2. identify geometric links;
3. sample new blackout events only for healthy geometric links;
4. remove failed links from the operational graph.

---

# Recovery

Recovery is automatic when:

$$
L_{ij}\rightarrow0
$$

Therefore a link can transition:

```text
HEALTHY
   ↓
BLACKOUT
   ↓
RECOVERED
```

No knowledge is automatically synchronized on recovery in this combo.

Later mission logic will decide what information should be exchanged after reconnection.

---

# Blackout Metrics

Over a test horizon define:

### Blackout starts

$$
N_B
$$

### Total failed-link timesteps

$$
T_{failed}
=
\sum_t
\sum_{i<j}
\mathbf{1}
[
H_{ij}(t)=0
]
$$

### Operational edge count

$$
|E_t^{op}|
$$

### Suppressed geometric links

$$
N_{suppressed}(t)
=
|E_t^{geo}|
-
|E_t^{op}|
$$

### Recovery count

Number of blackout events whose timers return to zero.

---

# Deterministic Synthetic Validation

Stochastic blackout generation alone is insufficient for validation.

A deterministic test will manually fail one geometric link for:

$$
T_B=5
$$

timesteps.

The operational graph must exclude that edge for exactly five timesteps and restore it afterward.

Thus:

$$
H_{ij}(t)=0
$$

for exactly:

$$
5
$$

steps.

---

# Stochastic Validation

A longer temporary simulation will then run for:

$$
T=500
$$

timesteps using fixed synthetic geometry where all four UUVs remain within communication range.

Therefore:

$$
|E^{geo}|=6
$$

at all times.

The operational edge count should occasionally fall below six due to blackouts.

This verifies that link failure emerges independently of geometry.

---

# Important State Boundary

Combo 30 operates entirely on temporary blackout state and temporary communication graphs.

It must not modify:

- live UUV positions;
- trajectories;
- local memories;
- known maps;
- resource memories.

---

# Expected Output

The code cell should:

1. define blackout probability:

$$
P_B=0.02
$$

2. define blackout duration:

$$
10\le T_B\le30
$$

3. create a symmetric per-link blackout manager;
4. construct:
   - geometric graph;
   - operational graph;
5. suppress failed links even when vehicles remain in range;
6. automatically recover links when timers expire;
7. run deterministic 5-step blackout validation;
8. run 500-step stochastic fully-connected validation;
9. report:
   - blackout events;
   - recoveries;
   - failed-link timesteps;
   - minimum operational edge count;
   - mean operational edge count;
10. verify live swarm state remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Geometric Connectivity}
\neq
\text{Operational Connectivity}
}
$$

and introduces explicit persistent link failure into MultiUUV-Link.

---

In [ ]:
# ============================================================
# Combo 30 — Persistent Communication Blackout / Link Failure
# ============================================================

from copy import deepcopy
from itertools import combinations


# ------------------------------------------------------------
# Blackout configuration
# ------------------------------------------------------------

BLACKOUT_START_PROBABILITY = 0.02

BLACKOUT_MIN_DURATION = 10
BLACKOUT_MAX_DURATION = 30

BLACKOUT_TEST_SEED = None

blackout_rng = np.random.default_rng(
    BLACKOUT_TEST_SEED
)


# ============================================================
# Link canonicalization
# ============================================================

def canonical_link(
    uuv_a,
    uuv_b,
):
    """
    Represent an undirected link deterministically.
    """

    return tuple(
        sorted(
            (
                int(uuv_a),
                int(uuv_b),
            )
        )
    )


# ============================================================
# Blackout manager
# ============================================================

class LinkBlackoutManager:

    def __init__(
        self,
        uuv_ids,
        start_probability=
            BLACKOUT_START_PROBABILITY,
        min_duration=
            BLACKOUT_MIN_DURATION,
        max_duration=
            BLACKOUT_MAX_DURATION,
        rng=None,
    ):

        self.uuv_ids = sorted(
            list(
                uuv_ids
            )
        )

        self.start_probability = float(
            start_probability
        )

        self.min_duration = int(
            min_duration
        )

        self.max_duration = int(
            max_duration
        )

        self.rng = (
            np.random.default_rng()
            if rng is None
            else rng
        )


        if not (
            0.0
            <=
            self.start_probability
            <=
            1.0
        ):

            raise ValueError(
                "Blackout probability must lie in [0, 1]."
            )


        if (
            self.min_duration
            <=
            0
        ):

            raise ValueError(
                "Blackout duration must be positive."
            )


        if (
            self.max_duration
            <
            self.min_duration
        ):

            raise ValueError(
                "max_duration must be >= min_duration."
            )


        # ----------------------------------------------------
        # Remaining blackout duration for every pair
        # ----------------------------------------------------

        self.remaining = {
            canonical_link(
                uuv_a,
                uuv_b,
            ):
                0

            for uuv_a, uuv_b
            in combinations(
                self.uuv_ids,
                2,
            )
        }


        self.total_blackout_starts = 0

        self.total_recoveries = 0


    # --------------------------------------------------------
    # Link-health query
    # --------------------------------------------------------

    def is_operational(
        self,
        uuv_a,
        uuv_b,
    ):

        link = canonical_link(
            uuv_a,
            uuv_b,
        )

        return (
            self.remaining[
                link
            ]
            ==
            0
        )


    # --------------------------------------------------------
    # Remaining-duration query
    # --------------------------------------------------------

    def remaining_duration(
        self,
        uuv_a,
        uuv_b,
    ):

        return self.remaining[
            canonical_link(
                uuv_a,
                uuv_b,
            )
        ]


    # --------------------------------------------------------
    # Manual deterministic blackout
    # --------------------------------------------------------

    def force_blackout(
        self,
        uuv_a,
        uuv_b,
        duration,
    ):

        if duration <= 0:

            raise ValueError(
                "Forced blackout duration must be positive."
            )


        link = canonical_link(
            uuv_a,
            uuv_b,
        )


        if (
            self.remaining[
                link
            ]
            ==
            0
        ):

            self.total_blackout_starts += 1


        self.remaining[
            link
        ] = int(
            duration
        )


    # --------------------------------------------------------
    # Advance one timestep
    # --------------------------------------------------------

    def step(
        self,
        geometric_graph,
        allow_new_blackouts=True,
    ):
        """
        One blackout-state transition.

        Semantics:
          1. active blackout links remain unavailable for the
             CURRENT timestep;
          2. their timers are decremented after health is known;
          3. newly triggered blackouts become unavailable
             immediately for the current timestep.
        """

        # ----------------------------------------------------
        # Snapshot active failures before decrement
        # ----------------------------------------------------

        active_before = {
            link
            for link, remaining
            in self.remaining.items()
            if remaining > 0
        }


        # ----------------------------------------------------
        # Sample new failures on healthy geometric links
        # ----------------------------------------------------

        newly_started = []


        if allow_new_blackouts:

            for uuv_a, uuv_b in (
                geometric_graph.edges()
            ):

                link = canonical_link(
                    uuv_a,
                    uuv_b,
                )


                if (
                    self.remaining[
                        link
                    ]
                    >
                    0
                ):

                    continue


                if (
                    self.rng.random()
                    <
                    self.start_probability
                ):

                    duration = int(
                        self.rng.integers(
                            self.min_duration,
                            self.max_duration
                            +
                            1,
                        )
                    )


                    self.remaining[
                        link
                    ] = duration


                    self.total_blackout_starts += 1


                    newly_started.append(
                        (
                            link,
                            duration,
                        )
                    )


        # ----------------------------------------------------
        # Construct operational graph BEFORE decrement
        # ----------------------------------------------------

        operational_graph = nx.Graph()

        operational_graph.add_nodes_from(
            geometric_graph.nodes()
        )


        suppressed_links = []


        for uuv_a, uuv_b, data in (
            geometric_graph.edges(
                data=True
            )
        ):

            link = canonical_link(
                uuv_a,
                uuv_b,
            )


            if (
                self.remaining[
                    link
                ]
                >
                0
            ):

                suppressed_links.append(
                    link
                )

            else:

                operational_graph.add_edge(
                    uuv_a,
                    uuv_b,
                    **deepcopy(
                        data
                    ),
                )


        # ----------------------------------------------------
        # Failed-link timestep contribution
        # ----------------------------------------------------

        failed_link_count = sum(
            1
            for remaining
            in self.remaining.values()
            if remaining > 0
        )


        # ----------------------------------------------------
        # Decrement blackout timers
        # ----------------------------------------------------

        recovered_links = []


        for link in list(
            self.remaining.keys()
        ):

            if (
                self.remaining[
                    link
                ]
                <=
                0
            ):

                continue


            self.remaining[
                link
            ] -= 1


            if (
                self.remaining[
                    link
                ]
                ==
                0
            ):

                recovered_links.append(
                    link
                )

                self.total_recoveries += 1


        return {
            "operational_graph":
                operational_graph,

            "suppressed_links":
                suppressed_links,

            "new_blackouts":
                newly_started,

            "recoveries":
                recovered_links,

            "failed_link_count":
                failed_link_count,

            "active_before":
                active_before,
        }


# ============================================================
# Preserve live state
# ============================================================

live_signature_before_30 = (
    get_swarm_state_signature()
)


live_known_maps_before_30 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Deterministic blackout validation
# ============================================================

deterministic_positions = {
    0: (100, 100),
    1: (200, 100),
    2: (300, 100),
}


deterministic_geo_graph = (
    build_communication_graph(
        deterministic_positions,
        communication_range=
            400.0,
    )
)


deterministic_manager = (
    LinkBlackoutManager(
        uuv_ids=[
            0,
            1,
            2,
        ],
        start_probability=
            0.0,
        min_duration=
            1,
        max_duration=
            1,
        rng=np.random.default_rng(
            123
        ),
    )
)


# ------------------------------------------------------------
# Force UUV-0 ↔ UUV-1 blackout for exactly 5 steps
# ------------------------------------------------------------

DETERMINISTIC_BLACKOUT_DURATION = 5


deterministic_manager.force_blackout(
    0,
    1,
    DETERMINISTIC_BLACKOUT_DURATION,
)


deterministic_history = []


for t in range(
    DETERMINISTIC_BLACKOUT_DURATION
    +
    2
):

    result = deterministic_manager.step(
        deterministic_geo_graph,
        allow_new_blackouts=False,
    )


    link_operational = (
        result[
            "operational_graph"
        ].has_edge(
            0,
            1,
        )
    )


    deterministic_history.append(
        {
            "time":
                t,

            "link_0_1_operational":
                link_operational,

            "remaining_after_step":
                deterministic_manager
                .remaining_duration(
                    0,
                    1,
                ),

            "recoveries_this_step":
                len(
                    result[
                        "recoveries"
                    ]
                ),
        }
    )


deterministic_df = pd.DataFrame(
    deterministic_history
)


# ------------------------------------------------------------
# Validate exact blackout length
# ------------------------------------------------------------

failed_steps = int(
    (
        ~deterministic_df[
            "link_0_1_operational"
        ]
    ).sum()
)


deterministic_duration_passed = (
    failed_steps
    ==
    DETERMINISTIC_BLACKOUT_DURATION
)


if not deterministic_duration_passed:

    raise RuntimeError(
        "Forced blackout did not last "
        "the requested number of timesteps."
    )


# ------------------------------------------------------------
# Verify recovery
# ------------------------------------------------------------

deterministic_recovery_passed = bool(
    deterministic_df.iloc[
        DETERMINISTIC_BLACKOUT_DURATION
    ][
        "link_0_1_operational"
    ]
)


if not deterministic_recovery_passed:

    raise RuntimeError(
        "Forced blackout link did not recover."
    )


# ============================================================
# Stochastic blackout validation
# ============================================================

# Fixed fully-connected geometry:
#
# every pair remains geometrically eligible,
# therefore topology changes can only come from blackouts.

stochastic_positions = {
    0: (400, 400),
    1: (500, 400),
    2: (400, 500),
    3: (500, 500),
}


stochastic_geo_graph = (
    build_communication_graph(
        stochastic_positions,
        communication_range=
            COMMUNICATION_RANGE,
    )
)


if (
    stochastic_geo_graph
    .number_of_edges()
    !=
    6
):

    raise RuntimeError(
        "Synthetic stochastic geometry "
        "must form complete graph K4."
    )


BLACKOUT_STOCHASTIC_STEPS = 500


stochastic_manager = (
    LinkBlackoutManager(
        uuv_ids=[
            0,
            1,
            2,
            3,
        ],
        start_probability=
            BLACKOUT_START_PROBABILITY,
        min_duration=
            BLACKOUT_MIN_DURATION,
        max_duration=
            BLACKOUT_MAX_DURATION,
        rng=np.random.default_rng(
            777777
        ),
    )
)


blackout_history = []

failed_link_timesteps = 0

stochastic_blackout_events = []


for t in range(
    BLACKOUT_STOCHASTIC_STEPS
):

    result = stochastic_manager.step(
        stochastic_geo_graph,
        allow_new_blackouts=True,
    )


    operational_edges = (
        result[
            "operational_graph"
        ].number_of_edges()
    )


    suppressed_count = len(
        result[
            "suppressed_links"
        ]
    )


    failed_link_timesteps += (
        suppressed_count
    )


    for link, duration in (
        result[
            "new_blackouts"
        ]
    ):

        stochastic_blackout_events.append(
            {
                "start_time":
                    t,

                "uuv_a":
                    link[0],

                "uuv_b":
                    link[1],

                "duration":
                    duration,
            }
        )


    blackout_history.append(
        {
            "time":
                t,

            "geometric_edges":
                6,

            "operational_edges":
                operational_edges,

            "suppressed_edges":
                suppressed_count,

            "new_blackouts":
                len(
                    result[
                        "new_blackouts"
                    ]
                ),

            "recoveries":
                len(
                    result[
                        "recoveries"
                    ]
                ),
        }
    )


blackout_history_df = pd.DataFrame(
    blackout_history
)


# ============================================================
# Stochastic metrics
# ============================================================

stochastic_blackout_count = (
    stochastic_manager
    .total_blackout_starts
)


stochastic_recovery_count = (
    stochastic_manager
    .total_recoveries
)


minimum_operational_edges = int(
    blackout_history_df[
        "operational_edges"
    ].min()
)


maximum_operational_edges = int(
    blackout_history_df[
        "operational_edges"
    ].max()
)


mean_operational_edges = float(
    blackout_history_df[
        "operational_edges"
    ].mean()
)


fraction_full_connectivity = float(
    (
        blackout_history_df[
            "operational_edges"
        ]
        ==
        6
    ).mean()
)


fraction_any_blackout = float(
    (
        blackout_history_df[
            "suppressed_edges"
        ]
        >
        0
    ).mean()
)


stochastic_blackout_observed = (
    stochastic_blackout_count
    >
    0
)


operational_graph_changed = (
    blackout_history_df[
        "operational_edges"
    ].nunique()
    >
    1
)


if not stochastic_blackout_observed:

    raise RuntimeError(
        "No stochastic blackout occurred "
        "during validation horizon."
    )


if not operational_graph_changed:

    raise RuntimeError(
        "Operational graph never changed "
        "under stochastic blackout."
    )


# ============================================================
# Live state integrity
# ============================================================

live_signature_after_30 = (
    get_swarm_state_signature()
)


live_swarm_unchanged = (
    live_signature_before_30
    ==
    live_signature_after_30
)


live_known_maps_unchanged_30 = all(
    np.array_equal(
        live_known_maps_before_30[
            uuv_id
        ],
        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged:

    raise RuntimeError(
        "Blackout test mutated live swarm state."
    )


if not live_known_maps_unchanged_30:

    raise RuntimeError(
        "Blackout test mutated live known maps."
    )


# ============================================================
# Report
# ============================================================

print(
    "=" * 98
)

print(
    "MultiUUV-Link — Persistent Communication "
    "Blackout / Link Failure"
)

print(
    "=" * 98
)


print(
    f"\nBlackout start probability : "
    f"{BLACKOUT_START_PROBABILITY:.3f}"
)

print(
    f"Blackout duration range    : "
    f"[{BLACKOUT_MIN_DURATION}, "
    f"{BLACKOUT_MAX_DURATION}] steps"
)


# ------------------------------------------------------------
# Deterministic report
# ------------------------------------------------------------

print(
    "\nDeterministic 5-step blackout validation:"
)


print(
    deterministic_df.to_string(
        index=False
    )
)


print(
    f"\n  Failed timesteps       : "
    f"{failed_steps}"
)

print(
    f"  Requested duration     : "
    f"{DETERMINISTIC_BLACKOUT_DURATION}"
)

print(
    f"  Exact-duration test    : "
    f"{deterministic_duration_passed}"
)

print(
    f"  Recovery test          : "
    f"{deterministic_recovery_passed}"
)


# ------------------------------------------------------------
# Stochastic report
# ------------------------------------------------------------

print(
    "\nStochastic fully-connected blackout validation:"
)


print(
    f"  Timesteps simulated          : "
    f"{BLACKOUT_STOCHASTIC_STEPS}"
)

print(
    f"  Geometric edges              : "
    f"{stochastic_geo_graph.number_of_edges()}"
)

print(
    f"  Blackout events              : "
    f"{stochastic_blackout_count}"
)

print(
    f"  Recoveries                   : "
    f"{stochastic_recovery_count}"
)

print(
    f"  Failed-link timesteps        : "
    f"{failed_link_timesteps}"
)

print(
    f"  Minimum operational edges    : "
    f"{minimum_operational_edges}"
)

print(
    f"  Maximum operational edges    : "
    f"{maximum_operational_edges}"
)

print(
    f"  Mean operational edges       : "
    f"{mean_operational_edges:.3f}"
)

print(
    f"  Fully operational fraction   : "
    f"{fraction_full_connectivity:.3f}"
)

print(
    f"  Any-blackout fraction        : "
    f"{fraction_any_blackout:.3f}"
)

print(
    f"  Operational topology changed : "
    f"{operational_graph_changed}"
)


# ------------------------------------------------------------
# First blackout events
# ------------------------------------------------------------

if stochastic_blackout_events:

    blackout_events_df = pd.DataFrame(
        stochastic_blackout_events
    )


    print(
        "\nFirst stochastic blackout events:"
    )


    print(
        blackout_events_df.head(
            12
        ).to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# Integrity
# ------------------------------------------------------------

print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm unchanged      : "
    f"{live_swarm_unchanged}"
)

print(
    f"  Live known maps unchanged : "
    f"{live_known_maps_unchanged_30}"
)


print(
    "\nPersistent communication blackout "
    "and automatic link recovery validated."
)

# Combo 31 — Unified Degraded Communication Channel

## Objective

The previous communication-degradation combos independently introduced:

### Combo 28

$$
\text{Packet Loss}
$$

### Combo 29

$$
\text{Latency}
$$

### Combo 30

$$
\text{Persistent Link Blackout}
$$

These mechanisms have so far been validated separately.

This combo combines them into one unified communication channel.

The complete communication path becomes:

```text
UUV Positions
      ↓
Geometric Range Test
      ↓
Operational Link / Blackout Test
      ↓
Packet Transmission Attempt
      ↓
Packet-Loss Test
      ↓
If Surviving:
Sample Communication Delay
      ↓
Delayed Message Queue
      ↓
Deliver When Due
      ↓
Merge Knowledge
```

The communication system must therefore distinguish:

```text
OUT OF RANGE
IN RANGE BUT BLACKED OUT
LINK OPERATIONAL BUT PACKET DROPPED
PACKET SURVIVED AND IS QUEUED
PACKET DELIVERED
```

This is the first complete degraded communication-channel abstraction in MultiUUV-Link.

This combo validates the channel on temporary state only.

The live autonomy mission is not yet modified.

---

# Communication-Layer Ordering

For a potential directed message:

$$
m_{i\rightarrow j}
$$

communication proceeds in the following order.

## Stage 1 — Geometric Eligibility

A direct geometric link exists if:

$$
d_{ij}
\le
R_c
$$

where:

$$
R_c=400\text{ px}
$$

If:

$$
d_{ij}>R_c
$$

no packet is generated.

---

## Stage 2 — Link Health

If the pair is geometrically connected, communication is allowed only if:

$$
H_{ij}(t)=1
$$

If:

$$
H_{ij}(t)=0
$$

the link is currently blacked out.

Therefore no packet transmission is attempted.

This distinction is important:

$$
\boxed{
\text{Blackout}
\neq
\text{Packet Loss}
}
$$

A blackout prevents packet creation.

Packet loss affects an attempted packet.

---

## Stage 3 — Packet Loss

For an operational link:

$$
Z_{ij}
\sim
Bernoulli
(
1-P_{loss}
)
$$

with:

$$
P_{loss}=0.25
$$

If:

$$
Z_{ij}=0
$$

the packet is dropped.

---

## Stage 4 — Latency

If the packet survives, sample:

$$
\tau
\sim
U_{\mathbb Z}
\{1,2,3,4,5\}
$$

and assign:

$$
t_{arrival}
=
t_{send}
+
\tau
$$

The payload is inserted into the pending-message queue.

---

# Message Types

Two message types are supported:

```text
MAP
RESOURCE
```

For every operational undirected communication edge:

$$
(i,j)
$$

the system may attempt:

$$
i\rightarrow j
$$

and:

$$
j\rightarrow i
$$

for both payload types.

Thus a fully operational four-UUV complete graph could create at most:

$$
6
\times
2
\times
2
=
24
$$

directed packets per communication round.

---

# Snapshot Semantics

Every outgoing payload is created from sender knowledge at:

$$
t_{send}
$$

For exploration maps:

$$
Payload^{map}_{i}(t)
=
M_i^{known}(t)
$$

For resource knowledge:

$$
Payload^{resource}_{i}(t)
=
D_i(t)
$$

The queued payload is a deep snapshot.

It does not update while waiting.

Thus a message delivered later may be stale.

---

# Delayed Delivery After Link Loss

Once a packet has successfully entered the communication queue, later loss of the geometric or operational link does **not** cancel that packet.

The communication event has already occurred.

Therefore:

$$
m\in Q
$$

continues toward its scheduled arrival even if:

$$
(i,j)\notin E_{t'}^{op}
$$

at some later:

$$
t'>t_{send}
$$

This separates transmission-time availability from propagation delay.

---

# Knowledge Merge

Delivered map messages use:

$$
M_j^{known,+}
=
M_j^{known}
\lor
M_{message}
$$

Delivered resource messages use:

$$
D_j^+
=
D_j
\cup
D_{message}
$$

Because both representations are monotonic, stale messages cannot erase newer knowledge.

---

# Channel Accounting

For every mission timestep we distinguish:

### Geometric directed opportunities

Potential communication directions created by current range geometry.

### Blackout-suppressed packets

Packets that would have been attempted but were suppressed because:

$$
H_{ij}=0
$$

### Attempted packets

Packets created after passing geometry and link-health tests.

### Dropped packets

Attempted packets removed by packet loss.

### Queued packets

Packets surviving packet loss and waiting for arrival.

### Delivered packets

Packets reaching their arrival timestep.

---

# Conservation Relationship

For packets actually attempted:

$$
N_{attempted}
=
N_{dropped}
+
N_{survived}
$$

and eventually:

$$
N_{survived}
=
N_{delivered}
+
N_{pending}
$$

Therefore at any finite test time:

$$
N_{attempted}
=
N_{dropped}
+
N_{delivered}
+
N_{pending}
$$

Blackout-suppressed packets are **not** counted as attempted packets.

---

# Test Scenario

The unified channel is tested over:

$$
T=120
$$

communication timesteps.

The UUV geometry is held fixed in a fully connected synthetic configuration:

$$
|E^{geo}|=6
$$

This ensures changes in communication performance arise only from:

- link blackout;
- packet loss;
- latency.

The live mission state remains untouched.

---

# Temporary Knowledge Evolution

To make delayed messages meaningful, temporary sender knowledge evolves during the test.

At selected timesteps, synthetic map cells and synthetic resource records are inserted into temporary sender memories.

These changes mimic new exploration/discovery without modifying live agents.

Communication then determines when and whether other UUVs learn those updates.

This allows measurement of:

$$
\text{Knowledge Propagation Delay}
$$

rather than merely queue mechanics.

---

# Knowledge-Propagation Metrics

For injected synthetic knowledge item:

$$
K^*
$$

record:

### Injection time

$$
t_{inject}
$$

### First remote acquisition time

$$
t_{remote}
$$

### Propagation delay

$$
T_{prop}
=
t_{remote}
-
t_{inject}
$$

Some knowledge may not reach every UUV within the finite test horizon due to blackout and packet loss.

That is acceptable.

---

# Expected Output

The code cell should:

1. create a unified degraded-channel manager;
2. apply communication layers in the order:

$$
\text{Range}
\rightarrow
\text{Blackout}
\rightarrow
\text{Packet Loss}
\rightarrow
\text{Latency}
$$

3. maintain a delayed-message queue;
4. support map and resource packets;
5. run 120 temporary communication timesteps;
6. inject temporary new knowledge during the run;
7. report:
   - geometric opportunities;
   - blackout-suppressed packets;
   - attempted packets;
   - dropped packets;
   - delivered packets;
   - pending packets;
   - blackout starts;
   - blackout recoveries;
   - mean packet age;
8. verify packet accounting;
9. report propagation of injected knowledge;
10. verify live swarm state remains unchanged.

A successful result establishes:

$$
\boxed{
\text{Unified Degraded Communication Channel}
}
$$

ready for integration into autonomous swarm behavior.

---

In [ ]:
# ============================================================
# Combo 31 — Unified Degraded Communication Channel
# ============================================================

from copy import deepcopy


# ------------------------------------------------------------
# Unified-channel test configuration
# ------------------------------------------------------------

UNIFIED_CHANNEL_STEPS = 120

UNIFIED_CHANNEL_SEED = 310031

channel_rng = np.random.default_rng(
    UNIFIED_CHANNEL_SEED
)


# ============================================================
# Unified degraded communication manager
# ============================================================

class DegradedCommunicationChannel:

    def __init__(
        self,
        uuv_ids,
        communication_range=
            COMMUNICATION_RANGE,
        packet_loss_probability=
            PACKET_LOSS_PROBABILITY,
        min_delay=
            MIN_COMM_DELAY,
        max_delay=
            MAX_COMM_DELAY,
        blackout_start_probability=
            BLACKOUT_START_PROBABILITY,
        blackout_min_duration=
            BLACKOUT_MIN_DURATION,
        blackout_max_duration=
            BLACKOUT_MAX_DURATION,
        rng=None,
    ):

        self.uuv_ids = sorted(
            list(
                uuv_ids
            )
        )

        self.communication_range = float(
            communication_range
        )

        self.packet_loss_probability = float(
            packet_loss_probability
        )

        self.min_delay = int(
            min_delay
        )

        self.max_delay = int(
            max_delay
        )

        self.rng = (
            np.random.default_rng()
            if rng is None
            else rng
        )


        # ----------------------------------------------------
        # Blackout manager uses the SAME RNG stream
        # ----------------------------------------------------

        self.blackout_manager = (
            LinkBlackoutManager(
                uuv_ids=
                    self.uuv_ids,

                start_probability=
                    blackout_start_probability,

                min_duration=
                    blackout_min_duration,

                max_duration=
                    blackout_max_duration,

                rng=
                    self.rng,
            )
        )


        # ----------------------------------------------------
        # Delayed packet queue
        # ----------------------------------------------------

        self.queue = []

        self.next_message_id = 0


        # ----------------------------------------------------
        # Channel statistics
        # ----------------------------------------------------

        self.geometric_packet_opportunities = 0

        self.blackout_suppressed_packets = 0

        self.attempted_packets = 0

        self.dropped_packets = 0

        self.queued_packets = 0

        self.delivered_packets = 0


        self.delivery_records = []


    # ========================================================
    # Build one payload snapshot
    # ========================================================

    @staticmethod
    def snapshot_payload(
        message_type,
        sender_id,
        known_maps,
        resource_memories,
    ):

        if message_type == "map":

            return (
                known_maps[
                    sender_id
                ].copy()
            )


        if message_type == "resource":

            return deepcopy(
                resource_memories[
                    sender_id
                ].all_resources()
            )


        raise ValueError(
            "Unknown message type."
        )


    # ========================================================
    # Queue surviving packet
    # ========================================================

    def queue_packet(
        self,
        sender,
        receiver,
        message_type,
        payload,
        current_time,
    ):

        self.attempted_packets += 1


        # ----------------------------------------------------
        # Packet loss
        # ----------------------------------------------------

        survives = packet_delivered(
            self.rng,
            loss_probability=
                self.packet_loss_probability,
        )


        if not survives:

            self.dropped_packets += 1

            return None


        # ----------------------------------------------------
        # Latency
        # ----------------------------------------------------

        delay = sample_message_delay(
            self.rng,
            min_delay=
                self.min_delay,
            max_delay=
                self.max_delay,
        )


        message = {
            "message_id":
                self.next_message_id,

            "sender":
                int(
                    sender
                ),

            "receiver":
                int(
                    receiver
                ),

            "message_type":
                message_type,

            "payload":
                deepcopy(
                    payload
                ),

            "send_time":
                int(
                    current_time
                ),

            "delay":
                int(
                    delay
                ),

            "arrival_time":
                int(
                    current_time
                    +
                    delay
                ),
        }


        self.next_message_id += 1


        self.queue.append(
            message
        )


        self.queued_packets += 1


        return message


    # ========================================================
    # Generate packets from current operational graph
    # ========================================================

    def transmit_round(
        self,
        current_time,
        positions,
        known_maps,
        resource_memories,
    ):

        # ----------------------------------------------------
        # Geometric graph
        # ----------------------------------------------------

        geometric_graph = (
            build_communication_graph(
                positions,
                communication_range=
                    self.communication_range,
            )
        )


        # ----------------------------------------------------
        # Apply persistent blackout state
        # ----------------------------------------------------

        blackout_result = (
            self.blackout_manager.step(
                geometric_graph,
                allow_new_blackouts=True,
            )
        )


        operational_graph = (
            blackout_result[
                "operational_graph"
            ]
        )


        # ----------------------------------------------------
        # Every geometric edge has:
        #
        # 2 directions × 2 message types = 4 opportunities
        # ----------------------------------------------------

        geometric_opportunities = (
            geometric_graph.number_of_edges()
            *
            4
        )


        operational_opportunities = (
            operational_graph.number_of_edges()
            *
            4
        )


        suppressed = (
            geometric_opportunities
            -
            operational_opportunities
        )


        self.geometric_packet_opportunities += (
            geometric_opportunities
        )


        self.blackout_suppressed_packets += (
            suppressed
        )


        # ----------------------------------------------------
        # Packet generation over OPERATIONAL edges only
        # ----------------------------------------------------

        for uuv_a, uuv_b in (
            operational_graph.edges()
        ):

            for sender, receiver in (
                (uuv_a, uuv_b),
                (uuv_b, uuv_a),
            ):

                for message_type in (
                    "map",
                    "resource",
                ):

                    payload = (
                        self.snapshot_payload(
                            message_type=
                                message_type,

                            sender_id=
                                sender,

                            known_maps=
                                known_maps,

                            resource_memories=
                                resource_memories,
                        )
                    )


                    self.queue_packet(
                        sender=
                            sender,

                        receiver=
                            receiver,

                        message_type=
                            message_type,

                        payload=
                            payload,

                        current_time=
                            current_time,
                    )


        return {
            "geometric_graph":
                geometric_graph,

            "operational_graph":
                operational_graph,

            "new_blackouts":
                blackout_result[
                    "new_blackouts"
                ],

            "recoveries":
                blackout_result[
                    "recoveries"
                ],

            "suppressed_packets":
                suppressed,
        }


    # ========================================================
    # Deliver due queued messages
    # ========================================================

    def deliver_due(
        self,
        current_time,
        known_maps,
        resource_memories,
    ):

        due = [
            message
            for message in self.queue
            if (
                message[
                    "arrival_time"
                ]
                <=
                current_time
            )
        ]


        future = [
            message
            for message in self.queue
            if (
                message[
                    "arrival_time"
                ]
                >
                current_time
            )
        ]


        delivered_now = []


        for message in due:

            receiver = (
                message[
                    "receiver"
                ]
            )


            before_map_cells = int(
                known_maps[
                    receiver
                ].sum()
            )


            before_resource_count = len(
                resource_memories[
                    receiver
                ]
            )


            # ------------------------------------------------
            # Knowledge merge
            # ------------------------------------------------

            if (
                message[
                    "message_type"
                ]
                ==
                "map"
            ):

                known_maps[
                    receiver
                ] = np.logical_or(
                    known_maps[
                        receiver
                    ],
                    message[
                        "payload"
                    ],
                )


            elif (
                message[
                    "message_type"
                ]
                ==
                "resource"
            ):

                resource_memories[
                    receiver
                ].update(
                    message[
                        "payload"
                    ]
                )


            else:

                raise RuntimeError(
                    "Unknown queued message type."
                )


            after_map_cells = int(
                known_maps[
                    receiver
                ].sum()
            )


            after_resource_count = len(
                resource_memories[
                    receiver
                ]
            )


            record = {
                "message_id":
                    message[
                        "message_id"
                    ],

                "message_type":
                    message[
                        "message_type"
                    ],

                "sender":
                    message[
                        "sender"
                    ],

                "receiver":
                    receiver,

                "send_time":
                    message[
                        "send_time"
                    ],

                "arrival_time":
                    message[
                        "arrival_time"
                    ],

                "delivery_time":
                    int(
                        current_time
                    ),

                "age":
                    int(
                        current_time
                        -
                        message[
                            "send_time"
                        ]
                    ),

                "new_map_cells":
                    (
                        after_map_cells
                        -
                        before_map_cells
                    ),

                "new_resources":
                    (
                        after_resource_count
                        -
                        before_resource_count
                    ),
            }


            delivered_now.append(
                record
            )


            self.delivery_records.append(
                record
            )


            self.delivered_packets += 1


        self.queue = future


        return delivered_now


# ============================================================
# Preserve LIVE mission state
# ============================================================

live_signature_before_31 = (
    get_swarm_state_signature()
)


live_known_maps_before_31 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Fully-connected fixed temporary geometry
# ============================================================

temp_positions_31 = {
    0: (500, 500),
    1: (620, 500),
    2: (500, 620),
    3: (620, 620),
}


temp_geo_graph_31 = (
    build_communication_graph(
        temp_positions_31,
        communication_range=
            COMMUNICATION_RANGE,
    )
)


if (
    temp_geo_graph_31
    .number_of_edges()
    !=
    6
):

    raise RuntimeError(
        "Combo 31 test geometry must be K4."
    )


# ============================================================
# Temporary knowledge state
# ============================================================

temp_known_maps_31 = {
    uuv_id:
        np.zeros(
            (
                HEIGHT,
                WIDTH,
            ),
            dtype=bool,
        )

    for uuv_id in [
        0,
        1,
        2,
        3,
    ]
}


temp_resource_memories_31 = {
    uuv_id:
        ResourceMemory(
            uuv_id=
                uuv_id
        )

    for uuv_id in [
        0,
        1,
        2,
        3,
    ]
}


# ============================================================
# Synthetic knowledge injections
# ============================================================

# Map block injected into UUV-0 at t=10
INJECT_MAP_TIME = 10

INJECT_MAP_REGION = (
    slice(
        100,
        140,
    ),
    slice(
        100,
        140,
    ),
)


# Resource injected into UUV-2 at t=35
INJECT_RESOURCE_TIME = 35

INJECTED_RESOURCE = Resource(
    resource_id=
        88888,

    x=
        700,

    y=
        700,

    resource_class=
        "B",
)


map_injection_remote_first_seen = {}

resource_injection_remote_first_seen = {}


# ============================================================
# Create unified channel
# ============================================================

channel_31 = (
    DegradedCommunicationChannel(
        uuv_ids=[
            0,
            1,
            2,
            3,
        ],

        communication_range=
            COMMUNICATION_RANGE,

        packet_loss_probability=
            PACKET_LOSS_PROBABILITY,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,

        blackout_start_probability=
            BLACKOUT_START_PROBABILITY,

        blackout_min_duration=
            BLACKOUT_MIN_DURATION,

        blackout_max_duration=
            BLACKOUT_MAX_DURATION,

        rng=
            channel_rng,
    )
)


# ------------------------------------------------------------
# Mission-channel logs
# ------------------------------------------------------------

channel_step_logs = []

blackout_event_count = 0

recovery_event_count = 0


# ============================================================
# Run temporary communication simulation
# ============================================================

for t in range(
    UNIFIED_CHANNEL_STEPS
):

    # --------------------------------------------------------
    # Inject synthetic map information
    # --------------------------------------------------------

    if (
        t
        ==
        INJECT_MAP_TIME
    ):

        temp_known_maps_31[
            0
        ][
            INJECT_MAP_REGION
        ] = True


    # --------------------------------------------------------
    # Inject synthetic resource knowledge
    # --------------------------------------------------------

    if (
        t
        ==
        INJECT_RESOURCE_TIME
    ):

        temp_resource_memories_31[
            2
        ].update(
            [
                INJECTED_RESOURCE
            ]
        )


    # --------------------------------------------------------
    # Deliver packets that have ARRIVED by current time
    # --------------------------------------------------------

    delivered_now = (
        channel_31.deliver_due(
            current_time=
                t,

            known_maps=
                temp_known_maps_31,

            resource_memories=
                temp_resource_memories_31,
        )
    )


    # --------------------------------------------------------
    # Record first remote acquisition of injected MAP
    # --------------------------------------------------------

    for receiver in [
        1,
        2,
        3,
    ]:

        if (
            receiver
            not in
            map_injection_remote_first_seen
        ):

            injected_block_known = bool(
                temp_known_maps_31[
                    receiver
                ][
                    INJECT_MAP_REGION
                ].all()
            )


            if injected_block_known:

                map_injection_remote_first_seen[
                    receiver
                ] = t


    # --------------------------------------------------------
    # Record first remote acquisition of injected RESOURCE
    # --------------------------------------------------------

    for receiver in [
        0,
        1,
        3,
    ]:

        if (
            receiver
            not in
            resource_injection_remote_first_seen
        ):

            if (
                temp_resource_memories_31[
                    receiver
                ].knows(
                    INJECTED_RESOURCE
                    .resource_id
                )
            ):

                resource_injection_remote_first_seen[
                    receiver
                ] = t


    # --------------------------------------------------------
    # Generate this timestep's transmissions
    # --------------------------------------------------------

    transmission_result = (
        channel_31.transmit_round(
            current_time=
                t,

            positions=
                temp_positions_31,

            known_maps=
                temp_known_maps_31,

            resource_memories=
                temp_resource_memories_31,
        )
    )


    blackout_event_count += len(
        transmission_result[
            "new_blackouts"
        ]
    )


    recovery_event_count += len(
        transmission_result[
            "recoveries"
        ]
    )


    channel_step_logs.append(
        {
            "time":
                t,

            "geometric_edges":
                transmission_result[
                    "geometric_graph"
                ].number_of_edges(),

            "operational_edges":
                transmission_result[
                    "operational_graph"
                ].number_of_edges(),

            "blackout_suppressed_packets":
                transmission_result[
                    "suppressed_packets"
                ],

            "delivered_this_step":
                len(
                    delivered_now
                ),

            "queue_size":
                len(
                    channel_31.queue
                ),
        }
    )


# ============================================================
# Flush surviving delayed packets after final transmission
# ============================================================

flush_time = (
    UNIFIED_CHANNEL_STEPS
)


while (
    len(
        channel_31.queue
    )
    >
    0
):

    channel_31.deliver_due(
        current_time=
            flush_time,

        known_maps=
            temp_known_maps_31,

        resource_memories=
            temp_resource_memories_31,
    )


    flush_time += 1


    if (
        flush_time
        >
        UNIFIED_CHANNEL_STEPS
        +
        MAX_COMM_DELAY
        +
        5
    ):

        raise RuntimeError(
            "Unified communication queue "
            "failed to drain."
        )


# ============================================================
# Packet-accounting validation
# ============================================================

pending_packets = len(
    channel_31.queue
)


packet_accounting_passed = (
    channel_31.attempted_packets
    ==
    (
        channel_31.dropped_packets
        +
        channel_31.delivered_packets
        +
        pending_packets
    )
)


if not packet_accounting_passed:

    raise RuntimeError(
        "Unified communication packet "
        "accounting failed."
    )


# ============================================================
# Knowledge propagation delays
# ============================================================

map_propagation_records = []


for receiver in [
    1,
    2,
    3,
]:

    first_seen = (
        map_injection_remote_first_seen
        .get(
            receiver
        )
    )


    propagation_delay = (
        None
        if first_seen is None
        else (
            first_seen
            -
            INJECT_MAP_TIME
        )
    )


    map_propagation_records.append(
        {
            "receiver":
                receiver,

            "injection_time":
                INJECT_MAP_TIME,

            "first_remote_time":
                first_seen,

            "propagation_delay":
                propagation_delay,
        }
    )


resource_propagation_records = []


for receiver in [
    0,
    1,
    3,
]:

    first_seen = (
        resource_injection_remote_first_seen
        .get(
            receiver
        )
    )


    propagation_delay = (
        None
        if first_seen is None
        else (
            first_seen
            -
            INJECT_RESOURCE_TIME
        )
    )


    resource_propagation_records.append(
        {
            "receiver":
                receiver,

            "injection_time":
                INJECT_RESOURCE_TIME,

            "first_remote_time":
                first_seen,

            "propagation_delay":
                propagation_delay,
        }
    )


# ============================================================
# Delivery statistics
# ============================================================

delivery_df_31 = pd.DataFrame(
    channel_31.delivery_records
)


if not delivery_df_31.empty:

    mean_delivered_packet_age = float(
        delivery_df_31[
            "age"
        ].mean()
    )

else:

    mean_delivered_packet_age = np.nan


channel_steps_df_31 = pd.DataFrame(
    channel_step_logs
)


mean_operational_edges_31 = float(
    channel_steps_df_31[
        "operational_edges"
    ].mean()
)


minimum_operational_edges_31 = int(
    channel_steps_df_31[
        "operational_edges"
    ].min()
)


# ============================================================
# Live-state integrity
# ============================================================

live_signature_after_31 = (
    get_swarm_state_signature()
)


live_swarm_unchanged_31 = (
    live_signature_before_31
    ==
    live_signature_after_31
)


live_known_maps_unchanged_31 = all(
    np.array_equal(
        live_known_maps_before_31[
            uuv_id
        ],
        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged_31:

    raise RuntimeError(
        "Unified channel test mutated live swarm state."
    )


if not live_known_maps_unchanged_31:

    raise RuntimeError(
        "Unified channel test mutated live known maps."
    )


# ============================================================
# Report
# ============================================================

print(
    "=" * 104
)

print(
    "MultiUUV-Link — Unified Degraded "
    "Communication Channel"
)

print(
    "=" * 104
)


print(
    f"\nTest timesteps                    : "
    f"{UNIFIED_CHANNEL_STEPS}"
)

print(
    f"Geometric communication edges    : "
    f"{temp_geo_graph_31.number_of_edges()}"
)

print(
    f"Packet-loss probability           : "
    f"{PACKET_LOSS_PROBABILITY:.2f}"
)

print(
    f"Latency range                     : "
    f"[{MIN_COMM_DELAY}, "
    f"{MAX_COMM_DELAY}] steps"
)

print(
    f"Blackout start probability        : "
    f"{BLACKOUT_START_PROBABILITY:.3f}"
)


print(
    "\nUnified channel accounting:"
)

print(
    f"  Geometric packet opportunities : "
    f"{channel_31.geometric_packet_opportunities}"
)

print(
    f"  Blackout-suppressed packets    : "
    f"{channel_31.blackout_suppressed_packets}"
)

print(
    f"  Attempted packets              : "
    f"{channel_31.attempted_packets}"
)

print(
    f"  Dropped packets                : "
    f"{channel_31.dropped_packets}"
)

print(
    f"  Delivered packets              : "
    f"{channel_31.delivered_packets}"
)

print(
    f"  Pending packets                : "
    f"{pending_packets}"
)

print(
    f"  Packet accounting passed       : "
    f"{packet_accounting_passed}"
)


print(
    "\nLink degradation:"
)

print(
    f"  Blackout starts                : "
    f"{blackout_event_count}"
)

print(
    f"  Recoveries observed            : "
    f"{recovery_event_count}"
)

print(
    f"  Mean operational edges         : "
    f"{mean_operational_edges_31:.3f}"
)

print(
    f"  Minimum operational edges      : "
    f"{minimum_operational_edges_31}"
)


print(
    "\nDelivered-message latency:"
)

print(
    f"  Mean delivered packet age      : "
    f"{mean_delivered_packet_age:.3f} steps"
)


# ------------------------------------------------------------
# Propagation reports
# ------------------------------------------------------------

map_propagation_df_31 = pd.DataFrame(
    map_propagation_records
)


resource_propagation_df_31 = pd.DataFrame(
    resource_propagation_records
)


print(
    "\nInjected map-knowledge propagation:"
)

print(
    map_propagation_df_31.to_string(
        index=False
    )
)


print(
    "\nInjected resource-knowledge propagation:"
)

print(
    resource_propagation_df_31.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# State integrity
# ------------------------------------------------------------

print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm unchanged      : "
    f"{live_swarm_unchanged_31}"
)

print(
    f"  Live known maps unchanged : "
    f"{live_known_maps_unchanged_31}"
)


print(
    "\nUnified degraded communication "
    "channel validated."
)

# Combo 32 — Cooperative Exploration Under Degraded Communication

## Objective

Combo 31 validated a unified degraded communication channel containing:

$$
\boxed{
\text{Persistent Blackouts}
+
\text{Packet Loss}
+
\text{Latency}
}
$$

This combo integrates that channel into the live decentralized cooperative exploration architecture.

The complete mission loop becomes:

```text
Local Observation
        ↓
Local Memory
        ↓
Current UUV Geometry
        ↓
Geometric Communication Graph
        ↓
Persistent Link Blackouts
        ↓
Packet Loss
        ↓
Latency Queue
        ↓
Delayed Knowledge Delivery
        ↓
Known Map / Resource Update
        ↓
Communication-Aware Planning
        ↓
Distributed Frontier Allocation
        ↓
Desired Motion
        ↓
Predictive Safety Shield
        ↓
Synchronized Motion
        ↓
Repeat
```

This is the first complete mission in which the UUVs must operate while communication is:

- intermittent;
- lossy;
- delayed;
- occasionally unavailable for sustained periods.

---

# Experimental Starting State

This degraded mission must begin from the reusable experimental checkpoint created after Combo 24.

Therefore before running this experiment:

```python
restore_checkpoint_24_5()
```

must be called.

This prevents the experiment from inheriting state from previous missions.

The mission therefore begins from the same nominal swarm state used for controlled controller evaluations.

---

# Communication State

Each UUV maintains:

$$
M_i^{local}
$$

for personally observed cells and:

$$
M_i^{known}
$$

for all legitimately known spatial information.

Communication can update only:

$$
M_i^{known}
$$

while local sensing updates both:

$$
M_i^{local}
$$

and:

$$
M_i^{known}
$$

through:

$$
M_i^{known}
\leftarrow
M_i^{known}
\lor
M_i^{local}
$$

Resource knowledge follows the same local-versus-remote distinction.

---

# Degraded Communication Pipeline

At timestep $t$:

## 1. Geometric graph

$$
G_t^{geo}
$$

is constructed from current UUV positions.

## 2. Persistent blackout state

Geometric links may be removed to produce:

$$
G_t^{op}
$$

## 3. Packet generation

Only operational links generate messages.

## 4. Packet loss

Each directed packet survives with probability:

$$
1-P_{loss}=0.75
$$

## 5. Latency

Surviving packets receive:

$$
\tau\in\{1,2,3,4,5\}
$$

and enter the delayed queue.

## 6. Delayed delivery

Packets are merged only when:

$$
t\ge t_{arrival}
$$

Thus the planner may operate using incomplete or stale knowledge.

---

# Planner Behavior Under Communication Loss

The planner must never stop simply because communication disappears.

Each UUV always retains its own:

$$
M_i^{local}
$$

and whatever:

$$
M_i^{known}
$$

it accumulated previously.

Therefore when isolated:

$$
G_t^{op}
$$

may contain no communication edges, but autonomous exploration continues.

This is a core resilience property:

$$
\boxed{
\text{Communication Loss}
\not\Rightarrow
\text{Mission Halt}
}
$$

---

# Distributed Allocation Under Partial Connectivity

Task allocation is performed inside each **operational communication component**.

For connected component:

$$
\mathcal{C}_k^{op}(t)
$$

agents may coordinate using whatever knowledge they currently possess.

If an agent is isolated:

$$
|\mathcal{C}|=1
$$

it falls back to independent communication-aware planning using its own known map.

Thus:

```text
Connected
→ coordinated allocation

Disconnected
→ independent autonomous exploration
```

---

# Delayed-Knowledge Consequence

Because map messages can arrive several timesteps late, two UUVs may temporarily believe the same region remains unexplored.

Therefore degraded communication is expected to increase redundant exploration relative to ideal communication.

This is not considered a software failure.

It is a measurable consequence of communication limitations.

---

# Reconnection

When a failed link recovers, new messages may again be transmitted.

Previously queued packets are unaffected by current connectivity.

Knowledge therefore gradually resynchronizes after reconnection.

No instantaneous global synchronization occurs.

---

# Allocation Stability

The stable allocation logic from revised Combo 27 remains active:

- useful goals persist;
- final assigned goals respect minimum separation;
- FOV overlap is penalized;
- agents without unique tasks HOLD;
- allocation cooldown limits thrashing.

However, degraded communication may split one large allocation component into multiple smaller components.

---

# Safety Independence

Physical safety does not depend on successful communication messages.

The predictive safety shield still uses current UUV positions and jointly evaluates desired motion.

Therefore:

$$
d_{ij}^{exec}
\ge
128\text{ px}
$$

must remain satisfied even when communication is degraded.

This deliberately separates:

$$
\text{Mission Communication}
$$

from:

$$
\text{Immediate Collision Safety}
$$

in the current simulator architecture.

---

# Mission Horizon

Run:

$$
T=180
$$

synchronized timesteps.

This matches the previous cooperative mission horizons.

---

# Communication Metrics

Record:

### Geometric communication opportunities

$$
N_{geo}
$$

### Blackout-suppressed packets

$$
N_{blackout}
$$

### Attempted packets

$$
N_{attempt}
$$

### Packet drops

$$
N_{drop}
$$

### Delivered packets

$$
N_{deliver}
$$

### Mean delivered-message age

$$
\bar\tau
$$

### Blackout events

$$
N_B
$$

### Mean geometric edges

$$
\bar{|E^{geo}|}
$$

### Mean operational edges

$$
\bar{|E^{op}|}
$$

---

# Mission Metrics

Record:

### Team coverage

$$
C_{team}
$$

### Mission incremental redundancy

$$
\rho_{mission}
$$

### Per-UUV local coverage

$$
C_i^{local}
$$

### Per-UUV known coverage

$$
C_i^{known}
$$

### Resources known

$$
|D_i|
$$

### Allocation rounds

$$
N_A
$$

### HOLD actions

$$
N_{hold}
$$

### Safety overrides

$$
N_{safe}
$$

---

# Communication Resilience Metric

Define the fraction of mission timesteps with at least one operational communication link:

$$
R_{comm}
=
\frac{
|\{t:|E_t^{op}|>0\}|
}{
T
}
$$

Also define complete operational isolation fraction:

$$
R_{iso}
=
\frac{
|\{t:|E_t^{op}|=0\}|
}{
T
}
$$

The mission should continue even during isolated intervals.

---

# Validation Conditions

The following are required:

### Safety

$$
d_{\min}^{mission}\ge128
$$

### Packet accounting

After queue flush:

$$
N_{attempt}
=
N_{drop}
+
N_{deliver}
$$

### Knowledge monotonicity

For every UUV:

$$
|M_i^{known}(t+1)|
\ge
|M_i^{known}(t)|
$$

### Continued autonomy

The mission must produce movement and exploration even when communication becomes temporarily unavailable.

---

# Important Experimental Interpretation

This combo validates autonomous operation under degraded communication.

It does not yet prove robustness superiority over another controller.

A later benchmark will compare identical checkpoint starts under:

- perfect communication;
- packet loss only;
- packet loss + latency;
- packet loss + latency + blackout.

That experiment will quantify communication sensitivity.

---

# Expected Output

The code cell should:

1. restore the Combo 24.5 experimental checkpoint;
2. create the unified degraded communication channel;
3. run 180 synchronized mission steps;
4. deliver queued packets when due;
5. create new communication packets through:
   - geometric range;
   - blackout state;
   - packet loss;
   - latency;
6. use received knowledge for planning;
7. perform stable distributed task allocation on operational components;
8. continue autonomous exploration during communication outages;
9. apply the predictive safety shield;
10. report:
    - communication degradation metrics;
    - team coverage;
    - incremental redundancy;
    - local/known coverage;
    - resources;
    - allocation rounds;
    - HOLD steps;
    - safety overrides;
11. visualize:
    - degraded-communication trajectories;
    - geometric versus operational communication edge history;
    - team coverage and incremental redundancy.

A successful result establishes:

$$
\boxed{
\text{Communication-Aware Autonomous Exploration}
\text{ Under Loss, Delay, and Persistent Link Failure}
}
$$

---

In [ ]:
# ============================================================
# Combo 32 — Cooperative Mission Under Degraded Communication
# ============================================================

# IMPORTANT:
# Controlled experiment starts from Combo 24.5 state.

restore_checkpoint_24_5()


# ------------------------------------------------------------
# Mission configuration
# ------------------------------------------------------------

DEGRADED_MISSION_STEPS = 180

DEGRADED_CHANNEL_SEED = 320032

degraded_rng = np.random.default_rng(
    DEGRADED_CHANNEL_SEED
)


# ------------------------------------------------------------
# Ensure known maps contain local knowledge
# ------------------------------------------------------------

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent
        .exploration_memory
        .observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

degraded_start = {}

degraded_start_local_maps = {}


for agent in agents:

    uuv_id = agent.uuv_id

    degraded_start_local_maps[
        uuv_id
    ] = (
        agent.exploration_memory
        .observed
        .copy()
    )

    degraded_start[
        uuv_id
    ] = {
        "local_cells":
            agent
            .exploration_memory
            .observed_count(),

        "known_cells":
            int(
                known_exploration_maps[
                    uuv_id
                ].sum()
            ),

        "resources":
            len(
                agent.resource_memory
            ),

        "trajectory_index":
            len(
                agent.trajectory
            ) - 1,
    }


start_team_union_32 = (
    np.logical_or.reduce(
        list(
            degraded_start_local_maps
            .values()
        )
    )
)


start_team_coverage_32 = (
    int(
        start_team_union_32.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Unified degraded channel
# ============================================================

channel_32 = DegradedCommunicationChannel(
    uuv_ids=[
        agent.uuv_id
        for agent in agents
    ],

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        degraded_rng,
)


# ============================================================
# Stable allocation state
# ============================================================

degraded_allocated_goals = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step_32 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_32 = None


allocation_rounds_32 = 0

allocation_goal_changes_32 = 0

hold_counts_32 = {
    agent.uuv_id: 0
    for agent in agents
}

safety_overrides_32 = {
    agent.uuv_id: 0
    for agent in agents
}


# ============================================================
# Histories
# ============================================================

geometric_edge_history_32 = []

operational_edge_history_32 = []

coverage_history_32 = []

incremental_redundancy_history_32 = []

known_cell_history_32 = {
    agent.uuv_id: []
    for agent in agents
}


mission_min_separation_32 = np.inf

blackout_starts_32 = 0
blackout_recoveries_32 = 0


# ============================================================
# Mission loop
# ============================================================

for mission_step in range(
    DEGRADED_MISSION_STEPS
):

    # --------------------------------------------------------
    # 1. Deliver all messages due now
    # --------------------------------------------------------

    channel_32.deliver_due(
        current_time=
            mission_step,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory
            for agent in agents
        },
    )


    # --------------------------------------------------------
    # 2. Ensure local knowledge remains included
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = agent.uuv_id

        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # --------------------------------------------------------
    # 3. Current positions
    # --------------------------------------------------------

    positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }


    # --------------------------------------------------------
    # 4. Generate degraded communication round
    # --------------------------------------------------------

    transmission_result = (
        channel_32.transmit_round(
            current_time=
                mission_step,

            positions=
                positions,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory
                for agent in agents
            },
        )
    )


    geometric_graph = (
        transmission_result[
            "geometric_graph"
        ]
    )


    operational_graph = (
        transmission_result[
            "operational_graph"
        ]
    )


    geometric_edge_history_32.append(
        geometric_graph
        .number_of_edges()
    )


    operational_edge_history_32.append(
        operational_graph
        .number_of_edges()
    )


    blackout_starts_32 += len(
        transmission_result[
            "new_blackouts"
        ]
    )


    blackout_recoveries_32 += len(
        transmission_result[
            "recoveries"
        ]
    )


    # ========================================================
    # 5. Goal usefulness / hysteresis
    # ========================================================

    goal_became_unavailable = False


    for agent in agents:

        uuv_id = agent.uuv_id

        goal = (
            degraded_allocated_goals[
                uuv_id
            ]
        )


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            degraded_allocated_goals[
                uuv_id
            ] = None

            goal_became_unavailable = True


    # ========================================================
    # 6. Operational topology change
    # ========================================================

    canonical_operational_edges = (
        frozenset(
            tuple(
                sorted(
                    edge
                )
            )
            for edge
            in operational_graph.edges()
        )
    )


    topology_changed = (
        previous_operational_edges_32
        is None
        or
        canonical_operational_edges
        !=
        previous_operational_edges_32
    )


    previous_operational_edges_32 = (
        canonical_operational_edges
    )


    # ========================================================
    # 7. Allocation trigger
    # ========================================================

    periodic_trigger = (
        mission_step
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    missing_goal_exists = any(
        degraded_allocated_goals[
            agent.uuv_id
        ]
        is None
        for agent in agents
    )


    urgent_trigger = (
        topology_changed
        or
        goal_became_unavailable
        or
        missing_goal_exists
    )


    cooldown_satisfied = (
        mission_step
        -
        last_allocation_step_32
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate = (
        periodic_trigger
        or
        (
            urgent_trigger
            and
            cooldown_satisfied
        )
    )


    if should_allocate:

        (
            new_assignments,
            changed_count,
            _,
            _,
            _,
        ) = perform_stable_allocation_round(
            operational_graph,
            degraded_allocated_goals,
        )


        degraded_allocated_goals = (
            new_assignments
        )


        allocation_rounds_32 += 1


        allocation_goal_changes_32 += (
            changed_count
        )


        last_allocation_step_32 = (
            mission_step
        )


    # ========================================================
    # 8. Desired mission actions
    # ========================================================

    desired_actions = []


    for agent in agents:

        uuv_id = agent.uuv_id

        goal = (
            degraded_allocated_goals[
                uuv_id
            ]
        )


        if goal is None:

            desired_action = (
                ACTION_HOLD
            )

            hold_counts_32[
                uuv_id
            ] += 1


        else:

            desired_action = (
                choose_greedy_action(
                    agent.state,
                    goal_x=
                        goal[
                            "x"
                        ],
                    goal_y=
                        goal[
                            "y"
                        ],
                )
            )


            if desired_action is None:

                desired_action = (
                    ACTION_HOLD
                )

                hold_counts_32[
                    uuv_id
                ] += 1


        desired_actions.append(
            desired_action
        )


    # ========================================================
    # 9. Safety shield
    # ========================================================

    shield_result = safety_shield(
        states=[
            agent.state
            for agent in agents
        ],

        desired_actions=
            desired_actions,

        safe_separation=
            SAFE_SEPARATION,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    executed_actions = (
        shield_result[
            "executed_actions"
        ]
    )


    mission_min_separation_32 = min(
        mission_min_separation_32,
        shield_result[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if desired != executed:

            safety_overrides_32[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 10. Predict and execute simultaneous motion
    # ========================================================

    next_states = [
        predict_safety_state(
            state=
                agent.state,

            action=
                action,

            step_size=
                ROLLOUT_STEP_SIZE,
        )

        for agent, action
        in zip(
            agents,
            executed_actions,
        )
    ]


    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = next_state


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # ========================================================
    # 11. Local observations
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in (
            visible_resources
        ):

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 12. Evaluation metrics
    # ========================================================

    local_maps = {
        agent.uuv_id:
            agent
            .exploration_memory
            .observed

        for agent in agents
    }


    team_union = (
        np.logical_or.reduce(
            list(
                local_maps.values()
            )
        )
    )


    team_coverage = (
        int(
            team_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    coverage_history_32.append(
        float(
            team_coverage
        )
    )


    # --------------------------------------------------------
    # Mission incremental redundancy
    # --------------------------------------------------------

    mission_new_masks = {
        uuv_id:
            (
                local_maps[
                    uuv_id
                ]
                &
                (
                    ~degraded_start_local_maps[
                        uuv_id
                    ]
                )
            )

        for uuv_id
        in local_maps
    }


    sum_new_local = sum(
        int(
            mask.sum()
        )
        for mask
        in mission_new_masks.values()
    )


    unique_new_union = (
        np.logical_or.reduce(
            list(
                mission_new_masks.values()
            )
        )
    )


    unique_new_cells = int(
        unique_new_union.sum()
    )


    redundant_new_cells = (
        sum_new_local
        -
        unique_new_cells
    )


    redundancy_ratio = (
        redundant_new_cells
        /
        sum_new_local

        if sum_new_local > 0

        else 0.0
    )


    incremental_redundancy_history_32.append(
        float(
            redundancy_ratio
        )
    )


    # --------------------------------------------------------
    # Knowledge monotonicity history
    # --------------------------------------------------------

    for agent in agents:

        known_cell_history_32[
            agent.uuv_id
        ].append(
            int(
                known_exploration_maps[
                    agent.uuv_id
                ].sum()
            )
        )


# ============================================================
# Flush delayed packets after mission
# ============================================================

flush_time = (
    DEGRADED_MISSION_STEPS
)


while (
    len(
        channel_32.queue
    )
    >
    0
):

    channel_32.deliver_due(
        current_time=
            flush_time,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory
            for agent in agents
        },
    )


    flush_time += 1


    if (
        flush_time
        >
        DEGRADED_MISSION_STEPS
        +
        MAX_COMM_DELAY
        +
        10
    ):

        raise RuntimeError(
            "Combo 32 delayed-message queue "
            "failed to drain."
        )


# ============================================================
# Validation
# ============================================================

if (
    mission_min_separation_32
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Degraded communication mission "
        "violated safety separation."
    )


# ------------------------------------------------------------
# Packet accounting
# ------------------------------------------------------------

packet_accounting_32 = (
    channel_32.attempted_packets
    ==
    (
        channel_32.dropped_packets
        +
        channel_32.delivered_packets
    )
)


if not packet_accounting_32:

    raise RuntimeError(
        "Combo 32 packet accounting failed."
    )


# ------------------------------------------------------------
# Knowledge monotonicity
# ------------------------------------------------------------

knowledge_monotonic_32 = True


for uuv_id, history in (
    known_cell_history_32.items()
):

    if any(
        later < earlier

        for earlier, later
        in zip(
            history[:-1],
            history[1:],
        )
    ):

        knowledge_monotonic_32 = False

        break


if not knowledge_monotonic_32:

    raise RuntimeError(
        "Known exploration memory regressed."
    )


# ============================================================
# Final mission metrics
# ============================================================

final_local_maps_32 = {
    agent.uuv_id:
        agent.exploration_memory.observed

    for agent in agents
}


final_team_union_32 = (
    np.logical_or.reduce(
        list(
            final_local_maps_32
            .values()
        )
    )
)


final_team_coverage_32 = (
    int(
        final_team_union_32.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


team_coverage_gain_32 = (
    final_team_coverage_32
    -
    start_team_coverage_32
)


# ------------------------------------------------------------
# Final incremental redundancy
# ------------------------------------------------------------

final_new_masks_32 = {
    uuv_id:
        (
            final_local_maps_32[
                uuv_id
            ]
            &
            (
                ~degraded_start_local_maps[
                    uuv_id
                ]
            )
        )

    for uuv_id
    in final_local_maps_32
}


final_sum_new_local_32 = sum(
    int(
        mask.sum()
    )
    for mask
    in final_new_masks_32.values()
)


final_unique_new_union_32 = (
    np.logical_or.reduce(
        list(
            final_new_masks_32.values()
        )
    )
)


final_unique_new_cells_32 = int(
    final_unique_new_union_32.sum()
)


final_redundant_new_cells_32 = (
    final_sum_new_local_32
    -
    final_unique_new_cells_32
)


final_incremental_redundancy_32 = (
    final_redundant_new_cells_32
    /
    final_sum_new_local_32

    if final_sum_new_local_32 > 0

    else 0.0
)


# ============================================================
# Communication statistics
# ============================================================

geometric_edges_mean_32 = float(
    np.mean(
        geometric_edge_history_32
    )
)


operational_edges_mean_32 = float(
    np.mean(
        operational_edge_history_32
    )
)


operational_link_fraction_32 = float(
    np.mean(
        np.asarray(
            operational_edge_history_32
        )
        >
        0
    )
)


complete_isolation_fraction_32 = float(
    np.mean(
        np.asarray(
            operational_edge_history_32
        )
        ==
        0
    )
)


if (
    channel_32.delivery_records
):

    mean_packet_age_32 = float(
        np.mean(
            [
                record[
                    "age"
                ]
                for record
                in channel_32
                .delivery_records
            ]
        )
    )

else:

    mean_packet_age_32 = np.nan


# ============================================================
# Per-UUV report
# ============================================================

degraded_reports_32 = []


for agent in agents:

    uuv_id = agent.uuv_id

    total_cells = (
        WIDTH
        *
        HEIGHT
    )


    local_cells = (
        agent
        .exploration_memory
        .observed_count()
    )


    known_cells = int(
        known_exploration_maps[
            uuv_id
        ].sum()
    )


    new_local_cells = int(
        final_new_masks_32[
            uuv_id
        ].sum()
    )


    degraded_reports_32.append(
        {
            "uuv_id":
                uuv_id,

            "local_coverage_pct":
                (
                    local_cells
                    /
                    total_cells
                    *
                    100
                ),

            "known_coverage_pct":
                (
                    known_cells
                    /
                    total_cells
                    *
                    100
                ),

            "new_local_cells":
                new_local_cells,

            "known_resources":
                len(
                    agent
                    .resource_memory
                ),

            "hold_steps":
                hold_counts_32[
                    uuv_id
                ],

            "safety_overrides":
                safety_overrides_32[
                    uuv_id
                ],
        }
    )


degraded_df_32 = pd.DataFrame(
    degraded_reports_32
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 108
)

print(
    "MultiUUV-Link — Cooperative Exploration "
    "Under Degraded Communication"
)

print(
    "=" * 108
)


print(
    f"\nMission steps                      : "
    f"{DEGRADED_MISSION_STEPS}"
)


print(
    "\nCommunication degradation:"
)

print(
    f"  Mean geometric edges            : "
    f"{geometric_edges_mean_32:.3f}"
)

print(
    f"  Mean operational edges          : "
    f"{operational_edges_mean_32:.3f}"
)

print(
    f"  Blackout starts                 : "
    f"{blackout_starts_32}"
)

print(
    f"  Recoveries observed             : "
    f"{blackout_recoveries_32}"
)

print(
    f"  Geometric packet opportunities  : "
    f"{channel_32.geometric_packet_opportunities}"
)

print(
    f"  Blackout-suppressed packets     : "
    f"{channel_32.blackout_suppressed_packets}"
)

print(
    f"  Attempted packets               : "
    f"{channel_32.attempted_packets}"
)

print(
    f"  Dropped packets                 : "
    f"{channel_32.dropped_packets}"
)

print(
    f"  Delivered packets               : "
    f"{channel_32.delivered_packets}"
)

print(
    f"  Mean delivered packet age       : "
    f"{mean_packet_age_32:.3f} steps"
)

print(
    f"  Operational-link timestep frac. : "
    f"{operational_link_fraction_32:.4f}"
)

print(
    f"  Complete isolation fraction     : "
    f"{complete_isolation_fraction_32:.4f}"
)

print(
    f"  Packet accounting passed        : "
    f"{packet_accounting_32}"
)


print(
    "\nMission performance:"
)

print(
    f"  Start team coverage             : "
    f"{100 * start_team_coverage_32:.4f}%"
)

print(
    f"  Final team coverage             : "
    f"{100 * final_team_coverage_32:.4f}%"
)

print(
    f"  Team coverage gain              : "
    f"{100 * team_coverage_gain_32:.4f} percentage points"
)

print(
    f"  Mission new local cells (sum)   : "
    f"{final_sum_new_local_32}"
)

print(
    f"  Mission unique new team cells   : "
    f"{final_unique_new_cells_32}"
)

print(
    f"  Mission redundant new cells     : "
    f"{final_redundant_new_cells_32}"
)

print(
    f"  Mission incremental redundancy  : "
    f"{final_incremental_redundancy_32:.4f}"
)


print(
    "\nPlanning / safety:"
)

print(
    f"  Allocation rounds               : "
    f"{allocation_rounds_32}"
)

print(
    f"  Allocation goal changes         : "
    f"{allocation_goal_changes_32}"
)

print(
    f"  Total HOLD steps                : "
    f"{sum(hold_counts_32.values())}"
)

print(
    f"  Total safety overrides          : "
    f"{sum(safety_overrides_32.values())}"
)

print(
    f"  Mission minimum separation      : "
    f"{mission_min_separation_32:.2f} px"
)

print(
    f"  Known-memory monotonicity       : "
    f"{knowledge_monotonic_32}"
)


print(
    "\nPer-UUV final state:"
)

print(
    degraded_df_32.to_string(
        index=False,
        formatters={
            "local_coverage_pct":
                lambda x: f"{x:.4f}",

            "known_coverage_pct":
                lambda x: f"{x:.4f}",
        },
    )
)


# ============================================================
# Visualization 1 — Trajectories
# ============================================================

plt.figure(
    figsize=(12, 7)
)

plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_index = (
        degraded_start[
            agent.uuv_id
        ][
            "trajectory_index"
        ]
    )


    path = np.asarray(
        agent.trajectory[
            start_index:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=80,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=120,
        linewidths=2.5,
    )


plt.title(
    "MultiUUV-Link — Exploration Under "
    "Degraded Communication"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Visualization 2 — Geometric vs operational communication
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    geometric_edge_history_32,
    linewidth=2,
    label="Geometric edges",
)


plt.plot(
    operational_edge_history_32,
    linewidth=2,
    label="Operational edges",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Communication edges"
)

plt.title(
    "MultiUUV-Link — Geometric vs Operational "
    "Connectivity"
)

plt.yticks(
    range(
        0,
        7,
    )
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Visualization 3 — Coverage and redundancy
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    np.asarray(
        coverage_history_32
    )
    *
    100,
    linewidth=2,
    label="Team coverage [%]",
)


plt.plot(
    np.asarray(
        incremental_redundancy_history_32
    )
    *
    100,
    linewidth=2,
    label="Mission incremental redundancy [%]",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Percent"
)

plt.title(
    "MultiUUV-Link — Degraded-Communication "
    "Coverage and Redundancy"
)

plt.legend()

plt.tight_layout()

plt.show()


print(
    "\nCooperative exploration under degraded "
    "communication validated."
)

# Combo 33 — UUV Failure Injection and Orphaned Task Detection

## Objective

Previous development established autonomous cooperative exploration under:

- limited perception;
- dynamic communication;
- packet loss;
- latency;
- persistent communication blackout.

This combo introduces the first permanent **vehicle failure**.

A failed UUV:

- becomes permanently immobile;
- stops sensing;
- stops transmitting;
- stops receiving;
- preserves all knowledge collected before failure;
- remains a physical obstacle;
- may leave an unfinished assigned frontier task.

That unfinished task becomes an:

$$
\boxed{\text{Orphaned Task}}
$$

Actual redistribution of orphaned work will be implemented in the next combo.

---

# Failure State

For UUV $i$:

$$
F_i(t)\in\{0,1\}
$$

where:

$$
F_i=0
$$

means healthy and:

$$
F_i=1
$$

means permanently failed.

For the validation experiment:

$$
i_f=1
$$

and:

$$
t_f=60
$$

Therefore:

$$
F_1(t)=
\begin{cases}
0,&t<60\\
1,&t\ge60
\end{cases}
$$

---

# Motion Failure

After failure:

$$
a_f(t)=HOLD
$$

and:

$$
p_f(t)=p_f(t_f)
$$

for all later timesteps.

The safety system must treat the failed UUV as an immobile physical obstacle.

Therefore healthy robots may change their actions, but the failed UUV must never be moved by the safety layer.

---

# Failure-Aware Safety

The ordinary predictive safety shield assumes all UUVs remain controllable.

That assumption is invalid after vehicle failure.

Therefore this combo introduces a failure-aware safety search.

For failed UUV $f$:

$$
\mathcal A_f=\{HOLD\}
$$

while each healthy UUV retains:

$$
\mathcal A_i=
\{
UP,RIGHT,DOWN,LEFT,HOLD
\}
$$

The safety search selects the feasible joint action minimizing healthy-agent action overrides while satisfying:

$$
d_{ij}\ge d_{safe}
$$

with:

$$
d_{safe}=128\text{ px}
$$

Thus the dead vehicle remains frozen even during collision avoidance.

---

# Sensing Failure

After failure:

$$
M_f^{local}(t+1)
=
M_f^{local}(t)
$$

The failed vehicle collects no new FOV observations and discovers no new resources.

Therefore:

$$
\Delta |M_f^{local}|=0
$$

after failure.

---

# Communication Failure

A failed UUV no longer participates in communication.

It is excluded from:

$$
G_t^{geo}
$$

and therefore from:

$$
G_t^{op}
$$

after failure.

Furthermore, packets already in transit whose receiver is the failed UUV are discarded when they arrive.

Therefore failure means:

```text
NO TRANSMIT
NO RECEIVE
```

rather than merely removing future transmissions.

---

# Knowledge Persistence

Failure does not erase previously acquired information.

At failure time the UUV retains:

$$
M_f^{local}(t_f)
$$

$$
M_f^{known}(t_f)
$$

and:

$$
D_f(t_f)
$$

but these quantities cannot grow afterward.

---

# Orphaned Task

If the failed UUV owns active goal:

$$
g_f
$$

at the failure instant, record:

$$
g_f^{orphan}
$$

containing:

- failed UUV ID;
- failure timestep;
- goal coordinates;
- remaining information ratio.

The failed UUV then loses ownership:

$$
g_f\leftarrow None
$$

The orphan record is preserved for later reassignment.

---

# Mission Continuation

All healthy UUVs continue normal autonomy:

```text
Local Sensing
    ↓
Degraded Communication
    ↓
Known Memory
    ↓
Frontier Planning
    ↓
Distributed Allocation
    ↓
Failure-Aware Safety
    ↓
Motion
```

Therefore:

$$
\boxed{
\text{Single-UUV Failure}
\not\Rightarrow
\text{Mission Failure}
}
$$

---

# Validation

The experiment runs:

$$
T=180
$$

timesteps with UUV-1 failing at:

$$
t=60.
$$

The following invariants must hold.

### Failed UUV displacement

$$
\Delta p_f=0
$$

### Failed local-map growth

$$
\Delta |M_f^{local}|=0
$$

### Failed known-map growth

$$
\Delta |M_f^{known}|=0
$$

### Failed resource-memory growth

$$
\Delta |D_f|=0
$$

### Failed communication degree

$$
degree_f=0
$$

### Healthy mission continuation

At least one healthy UUV must continue moving after failure.

### Safety

$$
d_{\min}^{mission}\ge128\text{ px}
$$

---

# Expected Output

The code should report:

- failed UUV;
- failure timestep;
- failure position;
- orphaned task;
- discarded packets addressed to failed UUV;
- failed-UUV displacement;
- local-map growth;
- known-map growth;
- resource-memory growth;
- final communication degree;
- healthy post-failure motion;
- team coverage;
- safety overrides;
- mission minimum separation.

It should also plot all trajectories and mark the failure position.

A successful result establishes:

$$
\boxed{
\text{Permanent UUV Failure}
+
\text{Orphaned Task Detection}
}
$$

---

In [ ]:
# ============================================================
# Combo 33 — Corrected UUV Failure Injection
#              + Orphaned Task Detection
# ============================================================

from itertools import product


# ============================================================
# Controlled experiment reset
# ============================================================

restore_checkpoint_24_5()


# ------------------------------------------------------------
# Mission configuration
# ------------------------------------------------------------

FAILURE_MISSION_STEPS = 180

FAILED_UUV_ID = 1

FAILURE_TIME = 60

FAILURE_CHANNEL_SEED = 330033


failure_rng = np.random.default_rng(
    FAILURE_CHANNEL_SEED
)


# ============================================================
# Failure state
# ============================================================

failed_flags_33 = {
    agent.uuv_id: False
    for agent in agents
}


failure_position_33 = None

failure_local_cells_33 = None

failure_known_cells_33 = None

failure_resource_count_33 = None

failure_trajectory_index_33 = None

orphaned_task_33 = None


discarded_packets_to_failed_33 = 0


# ============================================================
# Ensure known maps contain local knowledge
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent
        .exploration_memory
        .observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

failure_start_local_maps_33 = {
    agent.uuv_id:
        agent
        .exploration_memory
        .observed
        .copy()

    for agent in agents
}


failure_start_traj_idx_33 = {
    agent.uuv_id:
        len(
            agent.trajectory
        ) - 1

    for agent in agents
}


failure_start_union_33 = (
    np.logical_or.reduce(
        list(
            failure_start_local_maps_33.values()
        )
    )
)


failure_start_coverage_33 = (
    int(
        failure_start_union_33.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Communication channel
# ============================================================

channel_33 = DegradedCommunicationChannel(
    uuv_ids=[
        agent.uuv_id
        for agent in agents
    ],

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        failure_rng,
)


# ============================================================
# Failure-aware delayed-message delivery
# ============================================================

def deliver_due_failure_aware_33(
    channel,
    current_time,
    known_maps,
    resource_memories,
    failed_flags,
):
    """
    Deliver due packets only to healthy receivers.

    Packets already in flight toward a UUV that has since
    failed are discarded rather than merged.
    """

    due_messages = [
        message
        for message
        in channel.queue
        if (
            message[
                "arrival_time"
            ]
            <=
            current_time
        )
    ]


    future_messages = [
        message
        for message
        in channel.queue
        if (
            message[
                "arrival_time"
            ]
            >
            current_time
        )
    ]


    delivered_now = []

    discarded_now = 0


    for message in due_messages:

        receiver = (
            message[
                "receiver"
            ]
        )


        # ----------------------------------------------------
        # Receiver failed before packet arrival.
        # ----------------------------------------------------

        if failed_flags.get(
            receiver,
            False,
        ):

            discarded_now += 1

            continue


        before_map_cells = int(
            known_maps[
                receiver
            ].sum()
        )


        before_resource_count = len(
            resource_memories[
                receiver
            ]
        )


        # ----------------------------------------------------
        # Merge payload
        # ----------------------------------------------------

        if (
            message[
                "message_type"
            ]
            ==
            "map"
        ):

            known_maps[
                receiver
            ] = np.logical_or(
                known_maps[
                    receiver
                ],
                message[
                    "payload"
                ],
            )


        elif (
            message[
                "message_type"
            ]
            ==
            "resource"
        ):

            resource_memories[
                receiver
            ].update(
                message[
                    "payload"
                ]
            )


        else:

            raise RuntimeError(
                "Unknown message type in Combo 33."
            )


        after_map_cells = int(
            known_maps[
                receiver
            ].sum()
        )


        after_resource_count = len(
            resource_memories[
                receiver
            ]
        )


        record = {
            "message_id":
                message[
                    "message_id"
                ],

            "message_type":
                message[
                    "message_type"
                ],

            "sender":
                message[
                    "sender"
                ],

            "receiver":
                receiver,

            "send_time":
                message[
                    "send_time"
                ],

            "arrival_time":
                message[
                    "arrival_time"
                ],

            "delivery_time":
                current_time,

            "age":
                (
                    current_time
                    -
                    message[
                        "send_time"
                    ]
                ),

            "new_map_cells":
                (
                    after_map_cells
                    -
                    before_map_cells
                ),

            "new_resources":
                (
                    after_resource_count
                    -
                    before_resource_count
                ),
        }


        channel.delivery_records.append(
            record
        )


        channel.delivered_packets += 1


        delivered_now.append(
            record
        )


    channel.queue = (
        future_messages
    )


    return (
        delivered_now,
        discarded_now,
    )


# ============================================================
# Failure-aware safety helpers
# ============================================================

SAFETY_ACTIONS_33 = [
    ACTION_UP,
    ACTION_RIGHT,
    ACTION_DOWN,
    ACTION_LEFT,
    ACTION_HOLD,
]


def minimum_state_separation_33(
    states,
):
    """
    Minimum Euclidean separation between UUVState objects.
    """

    if len(states) < 2:
        return np.inf


    minimum_distance = np.inf


    for i in range(
        len(states)
    ):

        for j in range(
            i + 1,
            len(states)
        ):

            distance = euclidean_distance(
                (
                    states[i].x,
                    states[i].y,
                ),
                (
                    states[j].x,
                    states[j].y,
                ),
            )


            minimum_distance = min(
                minimum_distance,
                distance,
            )


    return float(
        minimum_distance
    )


def predict_joint_states_33(
    states,
    actions,
    step_size=
        ROLLOUT_STEP_SIZE,
):
    """
    Predict joint next states without relying on the existing
    predict_joint_positions() calling convention.
    """

    return [
        predict_safety_state(
            state=
                state,

            action=
                action,

            step_size=
                step_size,
        )

        for state, action
        in zip(
            states,
            actions,
        )
    ]


def failure_aware_safety_shield_33(
    states,
    desired_actions,
    failed_mask,
    safe_separation=
        SAFE_SEPARATION,
    step_size=
        ROLLOUT_STEP_SIZE,
):
    """
    Exhaustive joint-action safety search.

    Failed agents have exactly one possible action:
        HOLD

    Healthy agents may use all five motion actions.

    Objective:
        1. minimize overrides of healthy desired actions
        2. maximize predicted minimum pairwise separation
    """

    action_options = []


    for index, failed in enumerate(
        failed_mask
    ):

        if failed:

            action_options.append(
                [
                    ACTION_HOLD
                ]
            )

        else:

            action_options.append(
                SAFETY_ACTIONS_33
            )


    best_feasible = None


    for candidate_actions in product(
        *action_options
    ):

        predicted_states = (
            predict_joint_states_33(
                states=
                    states,

                actions=
                    candidate_actions,

                step_size=
                    step_size,
            )
        )


        minimum_distance = (
            minimum_state_separation_33(
                predicted_states
            )
        )


        if (
            minimum_distance
            <
            safe_separation
        ):

            continue


        healthy_override_count = sum(
            1
            for index, (
                desired,
                candidate,
            )
            in enumerate(
                zip(
                    desired_actions,
                    candidate_actions,
                )
            )
            if (
                not failed_mask[
                    index
                ]
                and
                desired
                !=
                candidate
            )
        )


        candidate_record = {
            "executed_actions":
                list(
                    candidate_actions
                ),

            "minimum_distance":
                float(
                    minimum_distance
                ),

            "override_count":
                int(
                    healthy_override_count
                ),
        }


        if best_feasible is None:

            best_feasible = (
                candidate_record
            )

            continue


        # ----------------------------------------------------
        # Lexicographic objective:
        #
        # 1. fewer healthy action overrides
        # 2. greater safety margin
        # ----------------------------------------------------

        if (
            candidate_record[
                "override_count"
            ]
            <
            best_feasible[
                "override_count"
            ]
        ):

            best_feasible = (
                candidate_record
            )


        elif (
            candidate_record[
                "override_count"
            ]
            ==
            best_feasible[
                "override_count"
            ]
            and
            candidate_record[
                "minimum_distance"
            ]
            >
            best_feasible[
                "minimum_distance"
            ]
        ):

            best_feasible = (
                candidate_record
            )


    # --------------------------------------------------------
    # Emergency fallback
    # --------------------------------------------------------

    if best_feasible is None:

        fallback_actions = [
            ACTION_HOLD
            for _ in states
        ]


        fallback_states = (
            predict_joint_states_33(
                states=
                    states,

                actions=
                    fallback_actions,

                step_size=
                    step_size,
            )
        )


        return {
            "executed_actions":
                fallback_actions,

            "minimum_distance":
                minimum_state_separation_33(
                    fallback_states
                ),

            "override_count":
                sum(
                    1
                    for index, desired
                    in enumerate(
                        desired_actions
                    )
                    if (
                        not failed_mask[
                            index
                        ]
                        and
                        desired
                        !=
                        ACTION_HOLD
                    )
                ),

            "fallback":
                True,
        }


    best_feasible[
        "fallback"
    ] = False


    return best_feasible


# ============================================================
# Allocation state
# ============================================================

goals_33 = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step_33 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_33 = None


# ============================================================
# Mission logs
# ============================================================

operational_edge_history_33 = []

coverage_history_33 = []

min_separation_33 = np.inf


safety_overrides_33 = {
    agent.uuv_id: 0
    for agent in agents
}


hold_counts_33 = {
    agent.uuv_id: 0
    for agent in agents
}


allocation_rounds_33 = 0


post_failure_healthy_motion_steps_33 = 0


# ============================================================
# Mission loop
# ============================================================

for mission_step in range(
    FAILURE_MISSION_STEPS
):

    # ========================================================
    # 1. Inject permanent failure
    # ========================================================

    if (
        mission_step
        ==
        FAILURE_TIME
    ):

        failed_flags_33[
            FAILED_UUV_ID
        ] = True


        failed_agent = agent_by_id[
            FAILED_UUV_ID
        ]


        failure_position_33 = (
            failed_agent.state.x,
            failed_agent.state.y,
        )


        failure_local_cells_33 = (
            failed_agent
            .exploration_memory
            .observed_count()
        )


        failure_known_cells_33 = int(
            known_exploration_maps[
                FAILED_UUV_ID
            ].sum()
        )


        failure_resource_count_33 = len(
            failed_agent
            .resource_memory
        )


        failure_trajectory_index_33 = (
            len(
                failed_agent.trajectory
            )
            -
            1
        )


        # ----------------------------------------------------
        # Capture current task as orphan
        # ----------------------------------------------------

        active_goal = goals_33[
            FAILED_UUV_ID
        ]


        if active_goal is not None:

            orphaned_task_33 = {
                "failed_uuv":
                    FAILED_UUV_ID,

                "failure_time":
                    FAILURE_TIME,

                "x":
                    int(
                        active_goal[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        active_goal[
                            "y"
                        ]
                    ),

                "information_ratio":
                    float(
                        goal_information_ratio(
                            failed_agent,
                            active_goal,
                        )
                    ),
            }


        goals_33[
            FAILED_UUV_ID
        ] = None


    # ========================================================
    # 2. Deliver due communication
    # ========================================================

    (
        _,
        discarded_now,
    ) = deliver_due_failure_aware_33(
        channel=
            channel_33,

        current_time=
            mission_step,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },

        failed_flags=
            failed_flags_33,
    )


    discarded_packets_to_failed_33 += (
        discarded_now
    )


    # ========================================================
    # 3. Preserve local knowledge inside known memory
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        # Even a failed UUV retains its old local memory.
        # Since it no longer senses, this OR cannot add cells.
        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # ========================================================
    # 4. Communication only among healthy UUVs
    # ========================================================

    healthy_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents

        if not failed_flags_33[
            agent.uuv_id
        ]
    }


    if len(
        healthy_positions
    ) >= 2:

        transmission_result = (
            channel_33.transmit_round(
                current_time=
                    mission_step,

                positions=
                    healthy_positions,

                known_maps=
                    known_exploration_maps,

                resource_memories={
                    agent.uuv_id:
                        agent.resource_memory

                    for agent in agents
                },
            )
        )


        operational_graph = (
            transmission_result[
                "operational_graph"
            ]
        )


    else:

        operational_graph = nx.Graph()

        operational_graph.add_nodes_from(
            healthy_positions.keys()
        )


    operational_edge_history_33.append(
        operational_graph
        .number_of_edges()
    )


    # ========================================================
    # 5. Goal usefulness
    # ========================================================

    goal_became_unavailable = False


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_33[
            uuv_id
        ]:

            continue


        goal = goals_33[
            uuv_id
        ]


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_33[
                uuv_id
            ] = None

            goal_became_unavailable = True


    # ========================================================
    # 6. Operational topology change
    # ========================================================

    canonical_edges = frozenset(
        tuple(
            sorted(
                edge
            )
        )

        for edge
        in operational_graph.edges()
    )


    topology_changed = (
        previous_operational_edges_33
        is None
        or
        canonical_edges
        !=
        previous_operational_edges_33
    )


    previous_operational_edges_33 = (
        canonical_edges
    )


    # ========================================================
    # 7. Healthy-agent allocation
    # ========================================================

    healthy_ids = [
        agent.uuv_id

        for agent in agents

        if not failed_flags_33[
            agent.uuv_id
        ]
    ]


    operational_graph.add_nodes_from(
        healthy_ids
    )


    missing_goal_exists = any(
        goals_33[
            uuv_id
        ]
        is None

        for uuv_id
        in healthy_ids
    )


    periodic_trigger = (
        mission_step
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger = (
        topology_changed
        or
        goal_became_unavailable
        or
        missing_goal_exists
    )


    cooldown_satisfied = (
        mission_step
        -
        last_allocation_step_33
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate = (
        periodic_trigger
        or
        (
            urgent_trigger
            and
            cooldown_satisfied
        )
    )


    if (
        should_allocate
        and
        len(
            healthy_ids
        )
        >
        0
    ):

        (
            new_assignments,
            _,
            _,
            _,
            _,
        ) = perform_stable_allocation_round(
            operational_graph,
            goals_33,
        )


        for uuv_id in healthy_ids:

            goals_33[
                uuv_id
            ] = (
                new_assignments.get(
                    uuv_id
                )
            )


        # Permanently enforce no task ownership by dead UUV.
        goals_33[
            FAILED_UUV_ID
        ] = None


        allocation_rounds_33 += 1


        last_allocation_step_33 = (
            mission_step
        )


    # ========================================================
    # 8. Desired actions
    # ========================================================

    desired_actions = []


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_33[
            uuv_id
        ]:

            desired_action = (
                ACTION_HOLD
            )


        else:

            goal = goals_33[
                uuv_id
            ]


            if goal is None:

                desired_action = (
                    ACTION_HOLD
                )


                hold_counts_33[
                    uuv_id
                ] += 1


            else:

                desired_action = (
                    choose_greedy_action(
                        agent.state,

                        goal_x=
                            goal[
                                "x"
                            ],

                        goal_y=
                            goal[
                                "y"
                            ],
                    )
                )


                if desired_action is None:

                    desired_action = (
                        ACTION_HOLD
                    )


                    hold_counts_33[
                        uuv_id
                    ] += 1


        desired_actions.append(
            desired_action
        )


    # ========================================================
    # 9. Failure-aware predictive safety
    # ========================================================

    states_now = [
        agent.state
        for agent in agents
    ]


    failed_mask_now = [
        failed_flags_33[
            agent.uuv_id
        ]
        for agent in agents
    ]


    shield_result_33 = (
        failure_aware_safety_shield_33(
            states=
                states_now,

            desired_actions=
                desired_actions,

            failed_mask=
                failed_mask_now,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )
    )


    executed_actions = (
        shield_result_33[
            "executed_actions"
        ]
    )


    # --------------------------------------------------------
    # Hard invariant:
    # failed UUV must ALWAYS execute HOLD.
    # --------------------------------------------------------

    for index, agent in enumerate(
        agents
    ):

        if failed_flags_33[
            agent.uuv_id
        ]:

            if (
                executed_actions[
                    index
                ]
                !=
                ACTION_HOLD
            ):

                raise RuntimeError(
                    "Failure-aware safety shield "
                    "attempted to move failed UUV."
                )


    min_separation_33 = min(
        min_separation_33,
        shield_result_33[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if (
            not failed_flags_33[
                agent.uuv_id
            ]
            and
            desired != executed
        ):

            safety_overrides_33[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 10. Execute synchronized motion
    # ========================================================

    previous_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    next_states = (
        predict_joint_states_33(
            states=
                states_now,

            actions=
                executed_actions,

            step_size=
                ROLLOUT_STEP_SIZE,
        )
    )


    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = (
            next_state
        )


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # ========================================================
    # 11. Verify healthy mission motion after failure
    # ========================================================

    if (
        mission_step
        >=
        FAILURE_TIME
    ):

        any_healthy_moved = any(
            (
                agent.state.x,
                agent.state.y,
            )
            !=
            previous_positions[
                agent.uuv_id
            ]

            for agent in agents

            if not failed_flags_33[
                agent.uuv_id
            ]
        )


        if any_healthy_moved:

            post_failure_healthy_motion_steps_33 += 1


    # ========================================================
    # 12. Healthy local sensing only
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_33[
            uuv_id
        ]:

            continue


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 13. Team coverage
    # ========================================================

    team_union = (
        np.logical_or.reduce(
            [
                agent
                .exploration_memory
                .observed

                for agent in agents
            ]
        )
    )


    coverage_history_33.append(
        float(
            team_union.sum()
            /
            (
                WIDTH
                *
                HEIGHT
            )
        )
    )


# ============================================================
# Flush remaining queued messages
# ============================================================

flush_time = (
    FAILURE_MISSION_STEPS
)


while len(
    channel_33.queue
) > 0:

    (
        _,
        discarded_now,
    ) = deliver_due_failure_aware_33(
        channel=
            channel_33,

        current_time=
            flush_time,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },

        failed_flags=
            failed_flags_33,
    )


    discarded_packets_to_failed_33 += (
        discarded_now
    )


    flush_time += 1


    if (
        flush_time
        >
        FAILURE_MISSION_STEPS
        +
        MAX_COMM_DELAY
        +
        10
    ):

        raise RuntimeError(
            "Combo 33 queue failed to drain."
        )


# ============================================================
# Final failure invariants
# ============================================================

failed_agent = agent_by_id[
    FAILED_UUV_ID
]


final_failure_position_33 = (
    failed_agent.state.x,
    failed_agent.state.y,
)


failed_post_failure_displacement_33 = (
    euclidean_distance(
        failure_position_33,
        final_failure_position_33,
    )
)


failed_local_growth_33 = (
    failed_agent
    .exploration_memory
    .observed_count()
    -
    failure_local_cells_33
)


failed_known_growth_33 = (
    int(
        known_exploration_maps[
            FAILED_UUV_ID
        ].sum()
    )
    -
    failure_known_cells_33
)


failed_resource_growth_33 = (
    len(
        failed_agent.resource_memory
    )
    -
    failure_resource_count_33
)


# ============================================================
# Final failed-UUV communication degree
# ============================================================

healthy_positions_final_33 = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )

    for agent in agents

    if not failed_flags_33[
        agent.uuv_id
    ]
}


final_comm_graph_33 = nx.Graph()


final_comm_graph_33.add_nodes_from(
    [
        agent.uuv_id
        for agent in agents
    ]
)


if len(
    healthy_positions_final_33
) >= 2:

    healthy_final_graph_33 = (
        build_communication_graph(
            healthy_positions_final_33,
            communication_range=
                COMMUNICATION_RANGE,
        )
    )


    final_comm_graph_33.add_edges_from(
        healthy_final_graph_33.edges()
    )


failed_final_degree_33 = (
    final_comm_graph_33.degree[
        FAILED_UUV_ID
    ]
)


# ============================================================
# Final coverage
# ============================================================

final_team_union_33 = (
    np.logical_or.reduce(
        [
            agent
            .exploration_memory
            .observed

            for agent in agents
        ]
    )
)


final_team_coverage_33 = (
    int(
        final_team_union_33.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Assertions
# ============================================================

if (
    failed_post_failure_displacement_33
    !=
    0.0
):

    raise RuntimeError(
        "Failed UUV moved after failure."
    )


if (
    failed_local_growth_33
    !=
    0
):

    raise RuntimeError(
        "Failed UUV continued local sensing."
    )


if (
    failed_known_growth_33
    !=
    0
):

    raise RuntimeError(
        "Failed UUV received communication after failure."
    )


if (
    failed_resource_growth_33
    !=
    0
):

    raise RuntimeError(
        "Failed UUV resource memory changed after failure."
    )


if (
    failed_final_degree_33
    !=
    0
):

    raise RuntimeError(
        "Failed UUV remained communication-active."
    )


if (
    post_failure_healthy_motion_steps_33
    <=
    0
):

    raise RuntimeError(
        "Healthy UUVs stopped after failure."
    )


if (
    min_separation_33
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Failure mission violated safety separation."
    )


# ============================================================
# Console report
# ============================================================

print(
    "=" * 108
)

print(
    "MultiUUV-Link — Permanent UUV Failure "
    "and Orphaned Task Detection"
)

print(
    "=" * 108
)


print(
    f"\nMission steps                       : "
    f"{FAILURE_MISSION_STEPS}"
)

print(
    f"Failed UUV                          : "
    f"{FAILED_UUV_ID}"
)

print(
    f"Failure timestep                    : "
    f"{FAILURE_TIME}"
)

print(
    f"Failure position                    : "
    f"{failure_position_33}"
)


print(
    "\nFailure invariants:"
)

print(
    f"  Post-failure displacement        : "
    f"{failed_post_failure_displacement_33:.2f} px"
)

print(
    f"  Post-failure local-map growth    : "
    f"{failed_local_growth_33}"
)

print(
    f"  Post-failure known-map growth    : "
    f"{failed_known_growth_33}"
)

print(
    f"  Post-failure resource growth     : "
    f"{failed_resource_growth_33}"
)

print(
    f"  Final communication degree       : "
    f"{failed_final_degree_33}"
)

print(
    f"  Packets discarded after failure  : "
    f"{discarded_packets_to_failed_33}"
)


print(
    "\nOrphaned task:"
)


if orphaned_task_33 is None:

    print(
        "  None existed at the failure instant."
    )


else:

    print(
        f"  Failed UUV                       : "
        f"{orphaned_task_33['failed_uuv']}"
    )

    print(
        f"  Failure time                     : "
        f"{orphaned_task_33['failure_time']}"
    )

    print(
        f"  Goal                             : "
        f"({orphaned_task_33['x']}, "
        f"{orphaned_task_33['y']})"
    )

    print(
        f"  Remaining information ratio      : "
        f"{orphaned_task_33['information_ratio']:.4f}"
    )


print(
    "\nMission continuation:"
)

print(
    f"  Start team coverage              : "
    f"{100 * failure_start_coverage_33:.4f}%"
)

print(
    f"  Final team coverage              : "
    f"{100 * final_team_coverage_33:.4f}%"
)

print(
    f"  Healthy post-failure move steps  : "
    f"{post_failure_healthy_motion_steps_33}"
)

print(
    f"  Allocation rounds                : "
    f"{allocation_rounds_33}"
)

print(
    f"  Total healthy HOLD steps         : "
    f"{sum(hold_counts_33.values())}"
)

print(
    f"  Total healthy safety overrides   : "
    f"{sum(safety_overrides_33.values())}"
)

print(
    f"  Mission minimum separation       : "
    f"{min_separation_33:.2f} px"
)


print(
    "\nPermanent UUV failure and "
    "orphaned-task detection validated."
)


# ============================================================
# Visualization — trajectories + failure point
# ============================================================

plt.figure(
    figsize=(12, 7)
)


plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_idx = (
        failure_start_traj_idx_33[
            agent.uuv_id
        ]
    )


    path = np.asarray(
        agent.trajectory[
            start_idx:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=70,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=110,
        linewidths=2.5,
    )


plt.scatter(
    failure_position_33[0],
    failure_position_33[1],
    marker="X",
    s=240,
    label=f"UUV-{FAILED_UUV_ID} FAILURE",
)


plt.title(
    "MultiUUV-Link — Mission with Permanent UUV Failure"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 34 — Fault-Aware Orphaned Task Reassignment

## Objective

Combo 33 established permanent UUV failure and orphaned-task detection.

However, in that particular stochastic mission UUV-1 happened to have no active goal at the exact failure instant.

This combo therefore performs a deterministic validation of the actual recovery mechanism:

$$
\boxed{
\text{Failed UUV}
\rightarrow
\text{Orphaned Goal}
\rightarrow
\text{Healthy Successor}
\rightarrow
\text{Task Completion}
}
$$

The experiment again uses:

$$
i_f=1
$$

with permanent failure at:

$$
t_f=60
$$

but now an orphaned frontier task is guaranteed to exist.

---

# 1. Guaranteed Orphan Creation

At the failure instant, if UUV-1 already owns an active valid goal, that goal becomes the orphan.

Otherwise a valid frontier goal is generated from UUV-1's legitimate knowledge immediately before failure.

Thus:

$$
g_f^{orphan}\neq None
$$

is guaranteed for the validation experiment.

This does not fabricate an arbitrary coordinate.

The fallback task must still originate from the failed UUV's frontier-planning mechanism.

---

# 2. Failure Semantics

After failure, UUV-1:

- remains physically stationary;
- performs no sensing;
- sends no communication;
- receives no communication;
- owns no active mission task.

Therefore:

$$
a_f=HOLD
$$

and:

$$
g_f=None
$$

for all:

$$
t\ge t_f.
$$

Its former task survives independently as:

$$
g^{orphan}.
$$

---

# 3. Fault-Aware Task Reassignment

Once permanent failure is confirmed, all healthy UUVs are evaluated as possible successors.

For healthy UUV $i$, define:

$$
J_i^{reassign}
=
w_I I_i(g^{orphan})
-
w_D
\frac{d_i(g^{orphan})}{d_{map}}
$$

where:

- $I_i$ is the remaining unknown-information ratio around the orphan goal according to UUV $i$'s known map;
- $d_i$ is Euclidean distance to the orphan goal;
- $d_{map}$ is the map diagonal.

Use:

$$
w_I=0.65
$$

and:

$$
w_D=0.35.
$$

The successor is:

$$
i^*
=
\arg\max_i
J_i^{reassign}.
$$

---

# 4. Immediate Confirmed-Failure Abstraction

This combo assumes that the simulator's permanent-failure event is immediately confirmed by the fault-management layer.

Therefore the orphan record is available to the surviving swarm at:

$$
t=t_f.
$$

This combo validates **task recovery after confirmed failure**.

Heartbeat-based distributed failure detection can be added separately later if required.

---

# 5. Orphan Priority

Once assigned, the inherited task receives temporary priority.

The successor keeps:

$$
g_{i^*}=g^{orphan}
$$

until the task is completed.

Ordinary frontier replanning is not allowed to overwrite that assignment.

Other healthy UUVs continue normal distributed frontier allocation.

Any ordinary task lying within:

$$
128\text{ px}
$$

of the active orphan goal is suppressed to avoid two UUVs being directed toward essentially the same region.

---

# 6. Completion Criterion

The inherited task is considered completed once the successor enters:

$$
d(i^*,g^{orphan})
\le
R_{goal}
$$

where the existing mission value is:

$$
R_{goal}=32\text{ px}.
$$

Define recovery delay:

$$
T_{recover}
=
t_{complete}
-
t_f.
$$

This measures how long the surviving swarm requires to physically recover the failed vehicle's responsibility.

---

# 7. Post-Completion Behavior

After completing the orphaned goal:

$$
g^{orphan}
\rightarrow COMPLETE
$$

and the successor returns to the ordinary distributed frontier allocator.

Thus:

```text
NORMAL EXPLORATION
       ↓
UUV FAILURE
       ↓
ORPHAN TASK
       ↓
SUCCESSOR SELECTION
       ↓
PRIORITY PURSUIT
       ↓
TASK COMPLETED
       ↓
NORMAL EXPLORATION
```

---

# 8. Communication Degradation Remains Active

The mission continues to use the complete degraded channel:

$$
\text{Range}
\rightarrow
\text{Blackout}
\rightarrow
\text{Packet Loss}
\rightarrow
\text{Latency}.
$$

Therefore healthy UUVs still operate under intermittent communication while recovering the failed task.

---

# 9. Failure-Aware Safety

The failed UUV remains a stationary physical obstacle.

The failure-aware safety shield from Combo 33 remains active:

$$
\mathcal A_f=\{HOLD\}.
$$

Healthy agents may change actions to satisfy:

$$
d_{ij}\ge128\text{ px}.
$$

---

# 10. Required Validation

The experiment must verify:

### Orphan exists

$$
g^{orphan}\neq None
$$

### Healthy successor exists

$$
i^*\neq i_f
$$

### Successor reaches orphan goal

$$
d(i^*,g^{orphan})
\le32\text{ px}
$$

### Failed UUV stays frozen

$$
\Delta p_f=0
$$

### Failed UUV remains communication-dead

$$
degree_f=0
$$

### Healthy swarm continues mission

Team coverage must continue increasing after failure.

### Safety

$$
d_{\min}^{mission}\ge128\text{ px}.
$$

---

# Expected Output

The code cell should report:

- failed UUV;
- failure time and position;
- orphan source:
  - existing assignment, or
  - generated frontier task;
- orphan coordinates;
- successor-candidate utilities;
- selected successor;
- initial successor-to-task distance;
- task-completion timestep;
- recovery delay;
- final orphan-task distance;
- failed-UUV displacement;
- team coverage;
- safety overrides;
- mission minimum separation.

The trajectory plot should mark:

- the failure position;
- the orphaned goal;
- the selected successor.

A successful result establishes:

$$
\boxed{
\text{Permanent Vehicle Failure}
+
\text{Fault-Aware Task Reallocation}
+
\text{Mission Recovery}
}
$$

---

In [ ]:
# ============================================================
# Combo 34 — Fault-Aware Orphaned Task Reassignment
# ============================================================

restore_checkpoint_24_5()


# ============================================================
# Mission configuration
# ============================================================

REASSIGN_MISSION_STEPS = 180

FAILED_UUV_ID_34 = 1

FAILURE_TIME_34 = 60

REASSIGN_CHANNEL_SEED = 340034


REASSIGN_INFO_WEIGHT = 0.65
REASSIGN_DISTANCE_WEIGHT = 0.35


reassign_rng_34 = np.random.default_rng(
    REASSIGN_CHANNEL_SEED
)


# ============================================================
# Failure state
# ============================================================

failed_flags_34 = {
    agent.uuv_id: False
    for agent in agents
}


failure_position_34 = None
failure_local_cells_34 = None
failure_known_cells_34 = None
failure_resource_count_34 = None


# ============================================================
# Orphan-task state
# ============================================================

orphan_task_34 = None

orphan_source_34 = None

orphan_active_34 = False

orphan_completed_34 = False


orphan_successor_34 = None

orphan_assignment_time_34 = None

orphan_completion_time_34 = None

orphan_initial_distance_34 = None

orphan_final_distance_34 = None


successor_candidate_records_34 = []


# ============================================================
# Communication diagnostics
# ============================================================

discarded_packets_to_failed_34 = 0


# ============================================================
# Known-map initialization
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent
        .exploration_memory
        .observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

reassign_start_local_maps_34 = {
    agent.uuv_id:
        agent
        .exploration_memory
        .observed
        .copy()

    for agent in agents
}


reassign_start_traj_idx_34 = {
    agent.uuv_id:
        len(
            agent.trajectory
        ) - 1

    for agent in agents
}


start_team_union_34 = (
    np.logical_or.reduce(
        list(
            reassign_start_local_maps_34.values()
        )
    )
)


start_team_coverage_34 = (
    int(
        start_team_union_34.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Communication channel
# ============================================================

channel_34 = DegradedCommunicationChannel(
    uuv_ids=[
        agent.uuv_id
        for agent in agents
    ],

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        reassign_rng_34,
)


# ============================================================
# Utility: guarantee a legitimate orphan goal
# ============================================================

def get_or_create_orphan_goal_34(
    failed_agent,
    existing_goal,
):
    """
    Prefer the failed UUV's actual active assignment.

    If none exists, generate a legitimate frontier goal from
    its known exploration memory.

    If its known-map planner unexpectedly has no candidate,
    retry using strictly local memory.
    """

    # --------------------------------------------------------
    # Existing mission assignment
    # --------------------------------------------------------

    if (
        existing_goal is not None
        and
        goal_is_still_useful(
            failed_agent,
            existing_goal,
        )
    ):

        return (
            deepcopy(
                existing_goal
            ),
            "existing_assignment",
        )


    # --------------------------------------------------------
    # Frontier from total legitimate known map
    # --------------------------------------------------------

    known_result = (
        generate_goal_from_knowledge(
            failed_agent,
            known_exploration_maps[
                failed_agent.uuv_id
            ],
        )
    )


    known_goal = (
        known_result[
            "selected"
        ]
    )


    if known_goal is not None:

        return (
            deepcopy(
                known_goal
            ),
            "generated_from_known_frontier",
        )


    # --------------------------------------------------------
    # Last legitimate fallback:
    # frontier from personally observed memory
    # --------------------------------------------------------

    local_result = (
        generate_goal_from_knowledge(
            failed_agent,
            failed_agent
            .exploration_memory
            .observed,
        )
    )


    local_goal = (
        local_result[
            "selected"
        ]
    )


    if local_goal is not None:

        return (
            deepcopy(
                local_goal
            ),
            "generated_from_local_frontier",
        )


    raise RuntimeError(
        "Combo 34 could not create a legitimate "
        "frontier orphan task."
    )


# ============================================================
# Utility: score potential successor
# ============================================================

def score_orphan_successor_34(
    agent,
    orphan_goal,
):
    """
    Fault-reassignment utility:

        J = w_I * information_ratio
            - w_D * normalized_distance
    """

    planning_memory = (
        PlanningMemory(
            uuv_id=
                agent.uuv_id,

            observed_mask=
                known_exploration_maps[
                    agent.uuv_id
                ],
        )
    )


    stats = candidate_fov_statistics(
        exploration_memory=
            planning_memory,

        candidate_x=
            int(
                orphan_goal[
                    "x"
                ]
            ),

        candidate_y=
            int(
                orphan_goal[
                    "y"
                ]
            ),
    )


    distance_px = euclidean_distance(
        (
            agent.state.x,
            agent.state.y,
        ),
        (
            orphan_goal[
                "x"
            ],
            orphan_goal[
                "y"
            ],
        ),
    )


    distance_norm = (
        distance_px
        /
        MAP_DIAGONAL
    )


    utility = (
        REASSIGN_INFO_WEIGHT
        *
        float(
            stats[
                "information_ratio"
            ]
        )
        -
        REASSIGN_DISTANCE_WEIGHT
        *
        distance_norm
    )


    return {
        "uuv_id":
            agent.uuv_id,

        "information_ratio":
            float(
                stats[
                    "information_ratio"
                ]
            ),

        "distance_px":
            float(
                distance_px
            ),

        "distance_norm":
            float(
                distance_norm
            ),

        "utility":
            float(
                utility
            ),
    }


# ============================================================
# Utility: choose orphan-task successor
# ============================================================

def choose_orphan_successor_34(
    orphan_goal,
    failed_flags,
):
    candidates = []


    for agent in agents:

        if failed_flags[
            agent.uuv_id
        ]:

            continue


        record = (
            score_orphan_successor_34(
                agent,
                orphan_goal,
            )
        )


        candidates.append(
            record
        )


    if not candidates:

        raise RuntimeError(
            "No healthy successor exists."
        )


    candidates.sort(
        key=lambda record:
            (
                record[
                    "utility"
                ],
                -
                record[
                    "distance_px"
                ],
            ),
        reverse=True,
    )


    return (
        candidates[0][
            "uuv_id"
        ],
        candidates,
    )


# ============================================================
# Allocation state
# ============================================================

goals_34 = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step_34 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_34 = None


# ============================================================
# Logs
# ============================================================

allocation_rounds_34 = 0

hold_counts_34 = {
    agent.uuv_id: 0
    for agent in agents
}


safety_overrides_34 = {
    agent.uuv_id: 0
    for agent in agents
}


coverage_history_34 = []

operational_edge_history_34 = []


mission_min_separation_34 = np.inf

healthy_motion_steps_after_failure_34 = 0


# ============================================================
# Mission loop
# ============================================================

for mission_step in range(
    REASSIGN_MISSION_STEPS
):

    # ========================================================
    # 1. Inject permanent vehicle failure
    # ========================================================

    if (
        mission_step
        ==
        FAILURE_TIME_34
    ):

        failed_agent = agent_by_id[
            FAILED_UUV_ID_34
        ]


        # ----------------------------------------------------
        # FIRST capture / construct its task.
        # ----------------------------------------------------

        (
            orphan_task_34,
            orphan_source_34,
        ) = get_or_create_orphan_goal_34(
            failed_agent=
                failed_agent,

            existing_goal=
                goals_34[
                    FAILED_UUV_ID_34
                ],
        )


        # ----------------------------------------------------
        # THEN fail the UUV.
        # ----------------------------------------------------

        failed_flags_34[
            FAILED_UUV_ID_34
        ] = True


        failure_position_34 = (
            failed_agent.state.x,
            failed_agent.state.y,
        )


        failure_local_cells_34 = (
            failed_agent
            .exploration_memory
            .observed_count()
        )


        failure_known_cells_34 = int(
            known_exploration_maps[
                FAILED_UUV_ID_34
            ].sum()
        )


        failure_resource_count_34 = len(
            failed_agent.resource_memory
        )


        # Failed UUV loses task ownership.
        goals_34[
            FAILED_UUV_ID_34
        ] = None


        # ----------------------------------------------------
        # Select healthy successor.
        # ----------------------------------------------------

        (
            orphan_successor_34,
            successor_candidate_records_34,
        ) = choose_orphan_successor_34(
            orphan_goal=
                orphan_task_34,

            failed_flags=
                failed_flags_34,
        )


        successor_agent = agent_by_id[
            orphan_successor_34
        ]


        orphan_initial_distance_34 = (
            euclidean_distance(
                (
                    successor_agent.state.x,
                    successor_agent.state.y,
                ),
                (
                    orphan_task_34[
                        "x"
                    ],
                    orphan_task_34[
                        "y"
                    ],
                ),
            )
        )


        orphan_assignment_time_34 = (
            mission_step
        )


        orphan_active_34 = True


        # ----------------------------------------------------
        # Priority inheritance
        # ----------------------------------------------------

        goals_34[
            orphan_successor_34
        ] = deepcopy(
            orphan_task_34
        )


    # ========================================================
    # 2. Deliver due communication
    # ========================================================

    (
        _,
        discarded_now,
    ) = deliver_due_failure_aware_33(
        channel=
            channel_34,

        current_time=
            mission_step,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },

        failed_flags=
            failed_flags_34,
    )


    discarded_packets_to_failed_34 += (
        discarded_now
    )


    # ========================================================
    # 3. Local memory always contributes to known memory
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # ========================================================
    # 4. Communication among healthy UUVs only
    # ========================================================

    healthy_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents

        if not failed_flags_34[
            agent.uuv_id
        ]
    }


    if len(
        healthy_positions
    ) >= 2:

        transmission_result = (
            channel_34.transmit_round(
                current_time=
                    mission_step,

                positions=
                    healthy_positions,

                known_maps=
                    known_exploration_maps,

                resource_memories={
                    agent.uuv_id:
                        agent.resource_memory

                    for agent in agents
                },
            )
        )


        operational_graph = (
            transmission_result[
                "operational_graph"
            ]
        )


    else:

        operational_graph = nx.Graph()

        operational_graph.add_nodes_from(
            healthy_positions.keys()
        )


    operational_edge_history_34.append(
        operational_graph
        .number_of_edges()
    )


    # ========================================================
    # 5. Goal usefulness
    # ========================================================

    goal_became_unavailable = False


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_34[
            uuv_id
        ]:

            continue


        # ----------------------------------------------------
        # Priority orphan assignment survives normal
        # goal-hysteresis invalidation.
        # ----------------------------------------------------

        if (
            orphan_active_34
            and
            uuv_id
            ==
            orphan_successor_34
        ):

            continue


        goal = goals_34[
            uuv_id
        ]


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_34[
                uuv_id
            ] = None

            goal_became_unavailable = True


    # ========================================================
    # 6. Operational topology change
    # ========================================================

    canonical_edges_34 = frozenset(
        tuple(
            sorted(
                edge
            )
        )

        for edge
        in operational_graph.edges()
    )


    topology_changed_34 = (
        previous_operational_edges_34
        is None
        or
        canonical_edges_34
        !=
        previous_operational_edges_34
    )


    previous_operational_edges_34 = (
        canonical_edges_34
    )


    # ========================================================
    # 7. Ordinary allocation
    # ========================================================

    healthy_ids_34 = [
        agent.uuv_id

        for agent in agents

        if not failed_flags_34[
            agent.uuv_id
        ]
    ]


    operational_graph.add_nodes_from(
        healthy_ids_34
    )


    missing_goal_exists_34 = any(
        goals_34[
            uuv_id
        ]
        is None

        for uuv_id
        in healthy_ids_34
    )


    periodic_trigger_34 = (
        mission_step
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger_34 = (
        topology_changed_34
        or
        goal_became_unavailable
        or
        missing_goal_exists_34
    )


    cooldown_satisfied_34 = (
        mission_step
        -
        last_allocation_step_34
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate_34 = (
        periodic_trigger_34
        or
        (
            urgent_trigger_34
            and
            cooldown_satisfied_34
        )
    )


    if (
        should_allocate_34
        and
        len(
            healthy_ids_34
        )
        >
        0
    ):

        (
            new_assignments_34,
            _,
            _,
            _,
            _,
        ) = perform_stable_allocation_round(
            operational_graph,
            goals_34,
        )


        for uuv_id in healthy_ids_34:

            goals_34[
                uuv_id
            ] = (
                new_assignments_34.get(
                    uuv_id
                )
            )


        # ----------------------------------------------------
        # Failed agent can never receive a task.
        # ----------------------------------------------------

        goals_34[
            FAILED_UUV_ID_34
        ] = None


        # ----------------------------------------------------
        # Active inherited orphan task has priority.
        # ----------------------------------------------------

        if orphan_active_34:

            goals_34[
                orphan_successor_34
            ] = deepcopy(
                orphan_task_34
            )


            # ------------------------------------------------
            # Suppress competing ordinary goals near orphan.
            # ------------------------------------------------

            orphan_xy = (
                orphan_task_34[
                    "x"
                ],
                orphan_task_34[
                    "y"
                ],
            )


            for uuv_id in healthy_ids_34:

                if (
                    uuv_id
                    ==
                    orphan_successor_34
                ):

                    continue


                other_goal = goals_34[
                    uuv_id
                ]


                if other_goal is None:
                    continue


                separation = (
                    euclidean_distance(
                        orphan_xy,
                        (
                            other_goal[
                                "x"
                            ],
                            other_goal[
                                "y"
                            ],
                        ),
                    )
                )


                if (
                    separation
                    <
                    MIN_ASSIGNED_GOAL_SEPARATION
                ):

                    goals_34[
                        uuv_id
                    ] = None


        allocation_rounds_34 += 1


        last_allocation_step_34 = (
            mission_step
        )


    # ========================================================
    # 8. Desired actions
    # ========================================================

    desired_actions_34 = []


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_34[
            uuv_id
        ]:

            desired_action = (
                ACTION_HOLD
            )


        else:

            goal = goals_34[
                uuv_id
            ]


            if goal is None:

                desired_action = (
                    ACTION_HOLD
                )


                hold_counts_34[
                    uuv_id
                ] += 1


            else:

                desired_action = (
                    choose_greedy_action(
                        agent.state,

                        goal_x=
                            goal[
                                "x"
                            ],

                        goal_y=
                            goal[
                                "y"
                            ],
                    )
                )


                if desired_action is None:

                    desired_action = (
                        ACTION_HOLD
                    )


                    hold_counts_34[
                        uuv_id
                    ] += 1


        desired_actions_34.append(
            desired_action
        )


    # ========================================================
    # 9. Failure-aware safety
    # ========================================================

    states_now_34 = [
        agent.state
        for agent in agents
    ]


    failed_mask_34 = [
        failed_flags_34[
            agent.uuv_id
        ]

        for agent in agents
    ]


    shield_result_34 = (
        failure_aware_safety_shield_33(
            states=
                states_now_34,

            desired_actions=
                desired_actions_34,

            failed_mask=
                failed_mask_34,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )
    )


    executed_actions_34 = (
        shield_result_34[
            "executed_actions"
        ]
    )


    mission_min_separation_34 = min(
        mission_min_separation_34,
        shield_result_34[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions_34,
        executed_actions_34,
    ):

        if (
            not failed_flags_34[
                agent.uuv_id
            ]
            and
            desired
            !=
            executed
        ):

            safety_overrides_34[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 10. Execute synchronized motion
    # ========================================================

    previous_positions_34 = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    next_states_34 = (
        predict_joint_states_33(
            states=
                states_now_34,

            actions=
                executed_actions_34,

            step_size=
                ROLLOUT_STEP_SIZE,
        )
    )


    for agent, next_state in zip(
        agents,
        next_states_34,
    ):

        agent.state = (
            next_state
        )


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # ========================================================
    # 11. Healthy post-failure motion
    # ========================================================

    if (
        mission_step
        >=
        FAILURE_TIME_34
    ):

        any_healthy_moved_34 = any(
            (
                agent.state.x,
                agent.state.y,
            )
            !=
            previous_positions_34[
                agent.uuv_id
            ]

            for agent in agents

            if not failed_flags_34[
                agent.uuv_id
            ]
        )


        if any_healthy_moved_34:

            healthy_motion_steps_after_failure_34 += 1


    # ========================================================
    # 12. Healthy sensing
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_34[
            uuv_id
        ]:

            continue


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 13. Check orphan-task completion
    # ========================================================

    if orphan_active_34:

        successor_agent = agent_by_id[
            orphan_successor_34
        ]


        distance_to_orphan = (
            euclidean_distance(
                (
                    successor_agent.state.x,
                    successor_agent.state.y,
                ),
                (
                    orphan_task_34[
                        "x"
                    ],
                    orphan_task_34[
                        "y"
                    ],
                ),
            )
        )


        if (
            distance_to_orphan
            <=
            GOAL_REACHED_RADIUS
        ):

            orphan_completed_34 = True

            orphan_active_34 = False

            orphan_completion_time_34 = (
                mission_step
            )


            orphan_final_distance_34 = (
                distance_to_orphan
            )


            # Successor returns to ordinary mission planning.
            goals_34[
                orphan_successor_34
            ] = None


    # ========================================================
    # 14. Coverage history
    # ========================================================

    team_union_34 = (
        np.logical_or.reduce(
            [
                agent
                .exploration_memory
                .observed

                for agent in agents
            ]
        )
    )


    coverage_history_34.append(
        float(
            team_union_34.sum()
            /
            (
                WIDTH
                *
                HEIGHT
            )
        )
    )


# ============================================================
# Flush delayed communication
# ============================================================

flush_time_34 = (
    REASSIGN_MISSION_STEPS
)


while len(
    channel_34.queue
) > 0:

    (
        _,
        discarded_now,
    ) = deliver_due_failure_aware_33(
        channel=
            channel_34,

        current_time=
            flush_time_34,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },

        failed_flags=
            failed_flags_34,
    )


    discarded_packets_to_failed_34 += (
        discarded_now
    )


    flush_time_34 += 1


    if (
        flush_time_34
        >
        REASSIGN_MISSION_STEPS
        +
        MAX_COMM_DELAY
        +
        10
    ):

        raise RuntimeError(
            "Combo 34 queue failed to drain."
        )


# ============================================================
# Final metrics
# ============================================================

failed_agent_34 = agent_by_id[
    FAILED_UUV_ID_34
]


failed_final_position_34 = (
    failed_agent_34.state.x,
    failed_agent_34.state.y,
)


failed_displacement_34 = (
    euclidean_distance(
        failure_position_34,
        failed_final_position_34,
    )
)


failed_local_growth_34 = (
    failed_agent_34
    .exploration_memory
    .observed_count()
    -
    failure_local_cells_34
)


failed_known_growth_34 = (
    int(
        known_exploration_maps[
            FAILED_UUV_ID_34
        ].sum()
    )
    -
    failure_known_cells_34
)


failed_resource_growth_34 = (
    len(
        failed_agent_34
        .resource_memory
    )
    -
    failure_resource_count_34
)


# ------------------------------------------------------------
# Final failed communication degree
# ------------------------------------------------------------

healthy_positions_final_34 = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )

    for agent in agents

    if not failed_flags_34[
        agent.uuv_id
    ]
}


final_comm_graph_34 = nx.Graph()


final_comm_graph_34.add_nodes_from(
    [
        agent.uuv_id
        for agent in agents
    ]
)


if len(
    healthy_positions_final_34
) >= 2:

    healthy_graph_final_34 = (
        build_communication_graph(
            healthy_positions_final_34,

            communication_range=
                COMMUNICATION_RANGE,
        )
    )


    final_comm_graph_34.add_edges_from(
        healthy_graph_final_34.edges()
    )


failed_final_degree_34 = (
    final_comm_graph_34.degree[
        FAILED_UUV_ID_34
    ]
)


# ------------------------------------------------------------
# Final team coverage
# ------------------------------------------------------------

final_union_34 = (
    np.logical_or.reduce(
        [
            agent
            .exploration_memory
            .observed

            for agent in agents
        ]
    )
)


final_team_coverage_34 = (
    int(
        final_union_34.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ------------------------------------------------------------
# Final successor distance if task was not completed
# ------------------------------------------------------------

if (
    orphan_successor_34
    is not None
    and
    orphan_final_distance_34
    is None
):

    successor_agent = agent_by_id[
        orphan_successor_34
    ]


    orphan_final_distance_34 = (
        euclidean_distance(
            (
                successor_agent.state.x,
                successor_agent.state.y,
            ),
            (
                orphan_task_34[
                    "x"
                ],
                orphan_task_34[
                    "y"
                ],
            ),
        )
    )


# ------------------------------------------------------------
# Recovery delay
# ------------------------------------------------------------

recovery_delay_34 = (
    None

    if orphan_completion_time_34
    is None

    else (
        orphan_completion_time_34
        -
        FAILURE_TIME_34
    )
)


# ============================================================
# Assertions
# ============================================================

if orphan_task_34 is None:

    raise RuntimeError(
        "Combo 34 failed to create an orphan task."
    )


if orphan_successor_34 is None:

    raise RuntimeError(
        "Combo 34 failed to select a successor."
    )


if (
    orphan_successor_34
    ==
    FAILED_UUV_ID_34
):

    raise RuntimeError(
        "Failed UUV was selected as its own successor."
    )


if not orphan_completed_34:

    raise RuntimeError(
        "Reassigned orphan task was not completed "
        "within the mission horizon."
    )


if (
    failed_displacement_34
    !=
    0.0
):

    raise RuntimeError(
        "Failed UUV moved after failure."
    )


if (
    failed_local_growth_34
    !=
    0
):

    raise RuntimeError(
        "Failed UUV continued sensing."
    )


if (
    failed_known_growth_34
    !=
    0
):

    raise RuntimeError(
        "Failed UUV continued receiving map knowledge."
    )


if (
    failed_resource_growth_34
    !=
    0
):

    raise RuntimeError(
        "Failed UUV resource knowledge changed."
    )


if (
    failed_final_degree_34
    !=
    0
):

    raise RuntimeError(
        "Failed UUV remained communication-active."
    )


if (
    healthy_motion_steps_after_failure_34
    <=
    0
):

    raise RuntimeError(
        "Healthy swarm did not continue after failure."
    )


if (
    mission_min_separation_34
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Combo 34 safety invariant failed."
    )


# ============================================================
# Successor-candidate table
# ============================================================

successor_candidates_df_34 = pd.DataFrame(
    successor_candidate_records_34
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 110
)

print(
    "MultiUUV-Link — Fault-Aware Orphaned "
    "Task Reassignment"
)

print(
    "=" * 110
)


print(
    f"\nMission steps                       : "
    f"{REASSIGN_MISSION_STEPS}"
)

print(
    f"Failed UUV                          : "
    f"{FAILED_UUV_ID_34}"
)

print(
    f"Failure timestep                    : "
    f"{FAILURE_TIME_34}"
)

print(
    f"Failure position                    : "
    f"{failure_position_34}"
)


print(
    "\nOrphaned task:"
)

print(
    f"  Source                            : "
    f"{orphan_source_34}"
)

print(
    f"  Goal                              : "
    f"({orphan_task_34['x']}, "
    f"{orphan_task_34['y']})"
)


print(
    "\nSuccessor candidates:"
)

print(
    successor_candidates_df_34.to_string(
        index=False,
        formatters={
            "information_ratio":
                lambda x: f"{x:.4f}",

            "distance_px":
                lambda x: f"{x:.2f}",

            "distance_norm":
                lambda x: f"{x:.4f}",

            "utility":
                lambda x: f"{x:.4f}",
        },
    )
)


print(
    "\nRecovery assignment:"
)

print(
    f"  Selected successor                : "
    f"UUV-{orphan_successor_34}"
)

print(
    f"  Initial successor distance        : "
    f"{orphan_initial_distance_34:.2f} px"
)

print(
    f"  Assignment timestep               : "
    f"{orphan_assignment_time_34}"
)

print(
    f"  Completion timestep               : "
    f"{orphan_completion_time_34}"
)

print(
    f"  Recovery delay                    : "
    f"{recovery_delay_34} steps"
)

print(
    f"  Final task distance               : "
    f"{orphan_final_distance_34:.2f} px"
)

print(
    f"  Task completed                    : "
    f"{orphan_completed_34}"
)


print(
    "\nFailure invariants:"
)

print(
    f"  Failed-UUV displacement           : "
    f"{failed_displacement_34:.2f} px"
)

print(
    f"  Failed local-map growth           : "
    f"{failed_local_growth_34}"
)

print(
    f"  Failed known-map growth           : "
    f"{failed_known_growth_34}"
)

print(
    f"  Failed resource growth            : "
    f"{failed_resource_growth_34}"
)

print(
    f"  Failed final communication degree : "
    f"{failed_final_degree_34}"
)

print(
    f"  Discarded packets to failed UUV   : "
    f"{discarded_packets_to_failed_34}"
)


print(
    "\nMission continuation:"
)

print(
    f"  Start team coverage               : "
    f"{100 * start_team_coverage_34:.4f}%"
)

print(
    f"  Final team coverage               : "
    f"{100 * final_team_coverage_34:.4f}%"
)

print(
    f"  Coverage gain                     : "
    f"{100 * (final_team_coverage_34 - start_team_coverage_34):.4f} pp"
)

print(
    f"  Healthy post-failure moving steps : "
    f"{healthy_motion_steps_after_failure_34}"
)

print(
    f"  Allocation rounds                 : "
    f"{allocation_rounds_34}"
)

print(
    f"  Healthy HOLD steps                : "
    f"{sum(hold_counts_34.values())}"
)

print(
    f"  Healthy safety overrides          : "
    f"{sum(safety_overrides_34.values())}"
)

print(
    f"  Mission minimum separation        : "
    f"{mission_min_separation_34:.2f} px"
)


print(
    "\nFault-aware orphaned-task "
    "reassignment and completion validated."
)


# ============================================================
# Visualization
# ============================================================

plt.figure(
    figsize=(12, 7)
)


plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_idx = (
        reassign_start_traj_idx_34[
            agent.uuv_id
        ]
    )


    path = np.asarray(
        agent.trajectory[
            start_idx:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=70,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=110,
        linewidths=2.5,
    )


# Failure marker
plt.scatter(
    failure_position_34[0],
    failure_position_34[1],
    marker="X",
    s=240,
    label=f"UUV-{FAILED_UUV_ID_34} FAILURE",
)


# Orphaned task marker
plt.scatter(
    orphan_task_34[
        "x"
    ],
    orphan_task_34[
        "y"
    ],
    marker="*",
    s=300,
    label="Orphaned task",
)


plt.title(
    "MultiUUV-Link — Fault-Aware Task Reassignment"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 35 — Distributed Heartbeat Failure Detection

## Objective

Combo 34 assumed that a permanent UUV failure was immediately known by the fault-management layer.

That was sufficient to validate:

$$
\text{Failure}
\rightarrow
\text{Orphan}
\rightarrow
\text{Reassignment}
$$

but a decentralized swarm cannot normally assume instantaneous failure knowledge.

This combo introduces distributed **heartbeat-based peer failure detection**.

Each healthy UUV periodically broadcasts a small heartbeat message.

Every UUV independently maintains:

$$
L_{ij}(t)
$$

the last timestep at which UUV $i$ received a heartbeat from UUV $j$.

If no heartbeat has been received for sufficiently long:

$$
t-L_{ij}(t)>T_{timeout}
$$

then UUV $i$ declares peer $j$ failed.

No global controller performs this decision.

---

# Heartbeat Message

A heartbeat contains:

```text
sender_id
send_time
```

and no map or resource payload.

For healthy UUV $j$:

$$
h_j(t)
=
(j,t)
$$

is generated periodically.

For this prototype:

$$
T_{heartbeat}=1
$$

simulation timestep.

Thus every healthy UUV attempts to broadcast one heartbeat every timestep to every currently operational neighbor.

---

# Communication Degradation

Heartbeat traffic experiences the same communication conditions as mission messages:

$$
\text{Range}
\rightarrow
\text{Blackout}
\rightarrow
\text{Packet Loss}
\rightarrow
\text{Latency}
$$

Therefore a missed heartbeat does **not** immediately imply vehicle failure.

Possible explanations include:

- packet loss;
- communication blackout;
- delayed delivery;
- temporary disconnection.

This is why a timeout is necessary.

---

# Peer Liveness State

Every UUV $i$ maintains an independent table:

$$
L_i
=
\{
j:t_{last}
\}
$$

For peer $j$:

$$
Age_{ij}(t)
=
t-L_{ij}(t)
$$

The peer is considered alive while:

$$
Age_{ij}(t)
\le
T_{timeout}.
$$

It is declared failed when:

$$
Age_{ij}(t)
>
T_{timeout}.
$$

---

# Timeout Selection

Communication blackouts currently last at most:

$$
30
$$

timesteps.

Maximum packet latency is:

$$
5
$$

timesteps.

We therefore choose:

$$
T_{timeout}=40
$$

timesteps.

This provides margin above:

$$
30+5=35.
$$

The additional margin reduces false positives caused by an unlucky packet-loss sequence immediately surrounding a blackout.

This is an engineering prototype parameter, not an optimized physical-modem timeout.

---

# Permanent Failure Scenario

Use four UUVs in fixed fully connected geometry.

UUV-1 permanently fails at:

$$
t_f=50.
$$

Before failure it transmits heartbeat packets normally.

After failure:

$$
h_1(t)=\varnothing
\qquad
\forall t\ge50.
$$

The remaining UUVs continue transmitting.

---

# Local Detection

Healthy UUV $i$ detects UUV-1 independently when:

$$
t-L_{i1}(t)>40.
$$

Because the last successfully received heartbeat may have been generated before failure and delivered later, detection times need not be identical.

Thus:

$$
t_{detect}^{(0)}
\neq
t_{detect}^{(2)}
\neq
t_{detect}^{(3)}
$$

is allowed.

This is an important decentralized property.

---

# No False Positive Requirement

For every healthy pair:

$$
i,j\neq1
$$

the test requires that UUV $i$ never declares healthy UUV $j$ failed.

Therefore:

$$
N_{false}=0.
$$

A fixed random seed is used for this validation so the condition is reproducible.

---

# Detection Delay

For detector UUV $i$:

$$
T_{detect,i}
=
t_{detect,i}
-
t_f.
$$

Because timeout-based detection requires evidence accumulated over time:

$$
T_{detect,i}>0.
$$

We expect detection approximately around the timeout scale rather than immediately at failure.

---

# Confirmation State

Once a peer is declared failed:

$$
D_{ij}=1
$$

the declaration is persistent for this permanent-failure prototype.

Later work could support:

- heartbeat recovery;
- temporary fault suspicion;
- confidence scores;
- consensus-based failure confirmation.

For now:

$$
\boxed{
\text{Detected Permanent Failure}
\rightarrow
\text{Persistent Failed State}
}
$$

---

# Relationship to Combo 34

The autonomy pipeline can now evolve from:

```text
Vehicle Failure
    ↓
Instant Failure Knowledge
    ↓
Task Reassignment
```

to:

```text
Vehicle Failure
    ↓
Heartbeat Silence
    ↓
Timeout
    ↓
Local Failure Detection
    ↓
Orphaned Task
    ↓
Task Reassignment
```

This removes the unrealistic instantaneous-failure assumption.

---

# Experimental Isolation

Combo 35 validates only the failure detector.

It uses temporary heartbeat state and fixed synthetic UUV geometry.

The live exploration mission state must remain unchanged.

---

# Required Validation

The test must verify:

### Failed UUV stops transmitting

$$
N_{heartbeat,1}^{post}=0
$$

### All healthy UUVs detect UUV-1

$$
D_{01}=D_{21}=D_{31}=1
$$

### Detection occurs after physical failure

$$
t_{detect}>t_f
$$

### No healthy peer is falsely declared failed

$$
N_{false}=0
$$

### Communication remains degraded

Heartbeat packets must still experience:

- blackouts;
- packet drops;
- latency.

### Live swarm state remains unchanged

The test must not mutate the exploration experiment.

---

# Expected Output

The code cell should report:

- physical failure timestep;
- heartbeat timeout;
- packet attempts;
- packet drops;
- blackout-suppressed heartbeat opportunities;
- delivered heartbeats;
- last heartbeat time for failed UUV at every healthy detector;
- detection timestep for each healthy UUV;
- detection delay;
- false-positive count;
- live-state integrity.

A successful result establishes:

$$
\boxed{
\text{Distributed Failure Detection from Communication Silence}
}
$$

---

In [ ]:
# ============================================================
# Combo 35 — Distributed Heartbeat Failure Detection
# ============================================================

from copy import deepcopy


# ============================================================
# Configuration
# ============================================================

HEARTBEAT_TEST_STEPS = 130

HEARTBEAT_FAILURE_UUV = 1
HEARTBEAT_FAILURE_TIME = 50

HEARTBEAT_TIMEOUT = 40

HEARTBEAT_TEST_SEED = 350035


heartbeat_rng_35 = np.random.default_rng(
    HEARTBEAT_TEST_SEED
)


UUV_IDS_35 = [
    0,
    1,
    2,
    3,
]


# ============================================================
# Preserve live state
# ============================================================

live_signature_before_35 = (
    get_swarm_state_signature()
)


live_known_maps_before_35 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Fixed fully-connected test geometry
# ============================================================

heartbeat_positions_35 = {
    0: (500, 500),
    1: (620, 500),
    2: (500, 620),
    3: (620, 620),
}


heartbeat_geo_graph_35 = (
    build_communication_graph(
        heartbeat_positions_35,

        communication_range=
            COMMUNICATION_RANGE,
    )
)


if (
    heartbeat_geo_graph_35
    .number_of_edges()
    !=
    6
):

    raise RuntimeError(
        "Combo 35 heartbeat geometry must be fully connected K4."
    )


# ============================================================
# Dedicated blackout manager
# ============================================================

heartbeat_blackout_manager_35 = (
    LinkBlackoutManager(
        uuv_ids=
            UUV_IDS_35,

        start_probability=
            BLACKOUT_START_PROBABILITY,

        min_duration=
            BLACKOUT_MIN_DURATION,

        max_duration=
            BLACKOUT_MAX_DURATION,

        rng=
            heartbeat_rng_35,
    )
)


# ============================================================
# Distributed last-seen tables
# ============================================================

# At t=0 each UUV begins knowing all peers are initially alive.
#
# L_i[j] = most recent DELIVERED heartbeat send-time from j.

last_seen_35 = {
    receiver: {
        sender: 0

        for sender in UUV_IDS_35

        if sender != receiver
    }

    for receiver in UUV_IDS_35
}


# ------------------------------------------------------------
# Persistent local failed-peer declarations
# ------------------------------------------------------------

declared_failed_35 = {
    detector: set()

    for detector in UUV_IDS_35
}


# ------------------------------------------------------------
# Detection timestamps
# ------------------------------------------------------------

detection_time_35 = {
    detector: {}

    for detector in UUV_IDS_35
}


# ============================================================
# Delayed heartbeat queue
# ============================================================

heartbeat_queue_35 = []

next_heartbeat_id_35 = 0


# ============================================================
# Communication counters
# ============================================================

heartbeat_geometric_opportunities_35 = 0

heartbeat_blackout_suppressed_35 = 0

heartbeat_attempted_35 = 0

heartbeat_dropped_35 = 0

heartbeat_queued_35 = 0

heartbeat_delivered_35 = 0


blackout_starts_35 = 0
blackout_recoveries_35 = 0


post_failure_transmissions_from_failed_35 = 0


# ============================================================
# Logs
# ============================================================

heartbeat_step_logs_35 = []

heartbeat_delivery_records_35 = []


# ============================================================
# Helper — queue one heartbeat
# ============================================================

def queue_heartbeat_35(
    sender,
    receiver,
    current_time,
):
    global next_heartbeat_id_35

    global heartbeat_attempted_35
    global heartbeat_dropped_35
    global heartbeat_queued_35


    heartbeat_attempted_35 += 1


    # --------------------------------------------------------
    # Packet-loss test
    # --------------------------------------------------------

    survives = packet_delivered(
        heartbeat_rng_35,

        loss_probability=
            PACKET_LOSS_PROBABILITY,
    )


    if not survives:

        heartbeat_dropped_35 += 1

        return


    # --------------------------------------------------------
    # Communication latency
    # --------------------------------------------------------

    delay = sample_message_delay(
        heartbeat_rng_35,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,
    )


    heartbeat_queue_35.append(
        {
            "message_id":
                next_heartbeat_id_35,

            "sender":
                sender,

            "receiver":
                receiver,

            "send_time":
                current_time,

            "arrival_time":
                (
                    current_time
                    +
                    delay
                ),
        }
    )


    next_heartbeat_id_35 += 1

    heartbeat_queued_35 += 1


# ============================================================
# Helper — deliver due heartbeats
# ============================================================

def deliver_heartbeats_35(
    current_time,
):

    global heartbeat_queue_35
    global heartbeat_delivered_35


    due = [
        message

        for message
        in heartbeat_queue_35

        if (
            message[
                "arrival_time"
            ]
            <=
            current_time
        )
    ]


    future = [
        message

        for message
        in heartbeat_queue_35

        if (
            message[
                "arrival_time"
            ]
            >
            current_time
        )
    ]


    for message in due:

        sender = (
            message[
                "sender"
            ]
        )

        receiver = (
            message[
                "receiver"
            ]
        )


        # ----------------------------------------------------
        # Failed UUV no longer receives heartbeats either.
        # ----------------------------------------------------

        if (
            receiver
            ==
            HEARTBEAT_FAILURE_UUV
            and
            current_time
            >=
            HEARTBEAT_FAILURE_TIME
        ):

            continue


        # ----------------------------------------------------
        # Record sender's transmitted timestamp.
        #
        # We use send_time rather than arrival time so the
        # detector knows how old the heartbeat information is.
        # ----------------------------------------------------

        last_seen_35[
            receiver
        ][
            sender
        ] = max(
            last_seen_35[
                receiver
            ][
                sender
            ],

            message[
                "send_time"
            ],
        )


        heartbeat_delivered_35 += 1


        heartbeat_delivery_records_35.append(
            {
                "sender":
                    sender,

                "receiver":
                    receiver,

                "send_time":
                    message[
                        "send_time"
                    ],

                "arrival_time":
                    message[
                        "arrival_time"
                    ],

                "delivery_time":
                    current_time,

                "age":
                    (
                        current_time
                        -
                        message[
                            "send_time"
                        ]
                    ),
            }
        )


    heartbeat_queue_35 = (
        future
    )


# ============================================================
# Main heartbeat simulation
# ============================================================

for t in range(
    HEARTBEAT_TEST_STEPS
):

    # ========================================================
    # 1. Deliver delayed heartbeats
    # ========================================================

    deliver_heartbeats_35(
        current_time=
            t
    )


    # ========================================================
    # 2. Determine physically alive UUVs
    # ========================================================

    alive_ids = [
        uuv_id

        for uuv_id
        in UUV_IDS_35

        if not (
            uuv_id
            ==
            HEARTBEAT_FAILURE_UUV
            and
            t
            >=
            HEARTBEAT_FAILURE_TIME
        )
    ]


    alive_positions = {
        uuv_id:
            heartbeat_positions_35[
                uuv_id
            ]

        for uuv_id
        in alive_ids
    }


    # ========================================================
    # 3. Build current geometric graph
    # ========================================================

    if len(
        alive_positions
    ) >= 2:

        geometric_graph = (
            build_communication_graph(
                alive_positions,

                communication_range=
                    COMMUNICATION_RANGE,
            )
        )

    else:

        geometric_graph = nx.Graph()

        geometric_graph.add_nodes_from(
            alive_ids
        )


    # ========================================================
    # 4. Apply persistent blackout layer
    # ========================================================

    blackout_result = (
        heartbeat_blackout_manager_35.step(
            geometric_graph,

            allow_new_blackouts=True,
        )
    )


    operational_graph = (
        blackout_result[
            "operational_graph"
        ]
    )


    blackout_starts_35 += len(
        blackout_result[
            "new_blackouts"
        ]
    )


    blackout_recoveries_35 += len(
        blackout_result[
            "recoveries"
        ]
    )


    # ========================================================
    # 5. Heartbeat opportunities
    #
    # One heartbeat per DIRECTION per geometric edge.
    # ========================================================

    geometric_directed_opportunities = (
        geometric_graph
        .number_of_edges()
        *
        2
    )


    operational_directed_opportunities = (
        operational_graph
        .number_of_edges()
        *
        2
    )


    suppressed = (
        geometric_directed_opportunities
        -
        operational_directed_opportunities
    )


    heartbeat_geometric_opportunities_35 += (
        geometric_directed_opportunities
    )


    heartbeat_blackout_suppressed_35 += (
        suppressed
    )


    # ========================================================
    # 6. Queue heartbeat packets
    # ========================================================

    for uuv_a, uuv_b in (
        operational_graph.edges()
    ):

        for sender, receiver in (
            (uuv_a, uuv_b),
            (uuv_b, uuv_a),
        ):

            # ------------------------------------------------
            # Sanity check:
            # dead UUV must never transmit.
            # ------------------------------------------------

            if (
                sender
                ==
                HEARTBEAT_FAILURE_UUV
                and
                t
                >=
                HEARTBEAT_FAILURE_TIME
            ):

                post_failure_transmissions_from_failed_35 += 1

                continue


            queue_heartbeat_35(
                sender=
                    sender,

                receiver=
                    receiver,

                current_time=
                    t,
            )


    # ========================================================
    # 7. Local timeout evaluation
    # ========================================================

    for detector in alive_ids:

        for peer in UUV_IDS_35:

            if (
                peer
                ==
                detector
            ):

                continue


            if (
                peer
                in
                declared_failed_35[
                    detector
                ]
            ):

                continue


            last_seen = (
                last_seen_35[
                    detector
                ][
                    peer
                ]
            )


            heartbeat_age = (
                t
                -
                last_seen
            )


            if (
                heartbeat_age
                >
                HEARTBEAT_TIMEOUT
            ):

                declared_failed_35[
                    detector
                ].add(
                    peer
                )


                detection_time_35[
                    detector
                ][
                    peer
                ] = t


    # ========================================================
    # 8. Step logging
    # ========================================================

    heartbeat_step_logs_35.append(
        {
            "time":
                t,

            "alive_uuvs":
                len(
                    alive_ids
                ),

            "geometric_edges":
                geometric_graph
                .number_of_edges(),

            "operational_edges":
                operational_graph
                .number_of_edges(),

            "queue_size":
                len(
                    heartbeat_queue_35
                ),

            "declared_failures_total":
                sum(
                    len(
                        failed_set
                    )

                    for failed_set
                    in declared_failed_35.values()
                ),
        }
    )


# ============================================================
# Flush remaining heartbeat queue
# ============================================================

flush_time_35 = (
    HEARTBEAT_TEST_STEPS
)


while (
    len(
        heartbeat_queue_35
    )
    >
    0
):

    deliver_heartbeats_35(
        current_time=
            flush_time_35
    )


    flush_time_35 += 1


    if (
        flush_time_35
        >
        HEARTBEAT_TEST_STEPS
        +
        MAX_COMM_DELAY
        +
        10
    ):

        raise RuntimeError(
            "Combo 35 heartbeat queue failed to drain."
        )


# ============================================================
# Detection records for failed UUV
# ============================================================

healthy_detectors_35 = [
    uuv_id

    for uuv_id
    in UUV_IDS_35

    if (
        uuv_id
        !=
        HEARTBEAT_FAILURE_UUV
    )
]


failed_detection_records_35 = []


for detector in healthy_detectors_35:

    detection_time = (
        detection_time_35[
            detector
        ].get(
            HEARTBEAT_FAILURE_UUV
        )
    )


    last_heartbeat = (
        last_seen_35[
            detector
        ][
            HEARTBEAT_FAILURE_UUV
        ]
    )


    detection_delay = (
        None

        if detection_time is None

        else (
            detection_time
            -
            HEARTBEAT_FAILURE_TIME
        )
    )


    failed_detection_records_35.append(
        {
            "detector":
                detector,

            "failed_peer":
                HEARTBEAT_FAILURE_UUV,

            "last_seen_send_time":
                last_heartbeat,

            "detection_time":
                detection_time,

            "detection_delay":
                detection_delay,
        }
    )


failed_detection_df_35 = pd.DataFrame(
    failed_detection_records_35
)


# ============================================================
# False-positive analysis
# ============================================================

false_positive_records_35 = []


for detector in healthy_detectors_35:

    for declared_peer in (
        declared_failed_35[
            detector
        ]
    ):

        if (
            declared_peer
            !=
            HEARTBEAT_FAILURE_UUV
        ):

            false_positive_records_35.append(
                {
                    "detector":
                        detector,

                    "healthy_peer":
                        declared_peer,

                    "detection_time":
                        detection_time_35[
                            detector
                        ][
                            declared_peer
                        ],
                }
            )


false_positive_count_35 = len(
    false_positive_records_35
)


# ============================================================
# Validation
# ============================================================

all_healthy_detected_failure_35 = all(
    (
        HEARTBEAT_FAILURE_UUV
        in
        declared_failed_35[
            detector
        ]
    )

    for detector
    in healthy_detectors_35
)


all_detections_after_failure_35 = all(
    (
        detection_time_35[
            detector
        ][
            HEARTBEAT_FAILURE_UUV
        ]
        >
        HEARTBEAT_FAILURE_TIME
    )

    for detector
    in healthy_detectors_35

    if (
        HEARTBEAT_FAILURE_UUV
        in
        detection_time_35[
            detector
        ]
    )
)


if not all_healthy_detected_failure_35:

    raise RuntimeError(
        "Not all healthy UUVs detected the failed peer."
    )


if not all_detections_after_failure_35:

    raise RuntimeError(
        "Failure was detected before the physical failure event."
    )


if (
    false_positive_count_35
    !=
    0
):

    raise RuntimeError(
        "Heartbeat detector generated false positives."
    )


if (
    post_failure_transmissions_from_failed_35
    !=
    0
):

    raise RuntimeError(
        "Failed UUV transmitted heartbeat after failure."
    )


# ============================================================
# Communication statistics
# ============================================================

if heartbeat_delivery_records_35:

    heartbeat_delivery_df_35 = pd.DataFrame(
        heartbeat_delivery_records_35
    )


    mean_heartbeat_age_35 = float(
        heartbeat_delivery_df_35[
            "age"
        ].mean()
    )

else:

    mean_heartbeat_age_35 = np.nan


# ============================================================
# Live-state integrity
# ============================================================

live_signature_after_35 = (
    get_swarm_state_signature()
)


live_swarm_unchanged_35 = (
    live_signature_before_35
    ==
    live_signature_after_35
)


live_known_maps_unchanged_35 = all(
    np.array_equal(
        live_known_maps_before_35[
            uuv_id
        ],
        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged_35:

    raise RuntimeError(
        "Heartbeat test mutated live swarm state."
    )


if not live_known_maps_unchanged_35:

    raise RuntimeError(
        "Heartbeat test mutated live known maps."
    )


# ============================================================
# Console report
# ============================================================

print(
    "=" * 108
)

print(
    "MultiUUV-Link — Distributed Heartbeat "
    "Failure Detection"
)

print(
    "=" * 108
)


print(
    f"\nTest timesteps                     : "
    f"{HEARTBEAT_TEST_STEPS}"
)

print(
    f"Physical failed UUV                : "
    f"{HEARTBEAT_FAILURE_UUV}"
)

print(
    f"Physical failure timestep          : "
    f"{HEARTBEAT_FAILURE_TIME}"
)

print(
    f"Heartbeat timeout                  : "
    f"{HEARTBEAT_TIMEOUT} steps"
)


print(
    "\nDegraded heartbeat communication:"
)

print(
    f"  Geometric heartbeat opportunities : "
    f"{heartbeat_geometric_opportunities_35}"
)

print(
    f"  Blackout-suppressed heartbeats    : "
    f"{heartbeat_blackout_suppressed_35}"
)

print(
    f"  Attempted heartbeat packets       : "
    f"{heartbeat_attempted_35}"
)

print(
    f"  Dropped heartbeat packets         : "
    f"{heartbeat_dropped_35}"
)

print(
    f"  Delivered heartbeat packets       : "
    f"{heartbeat_delivered_35}"
)

print(
    f"  Mean delivered heartbeat age      : "
    f"{mean_heartbeat_age_35:.3f} steps"
)

print(
    f"  Blackout starts                   : "
    f"{blackout_starts_35}"
)

print(
    f"  Blackout recoveries               : "
    f"{blackout_recoveries_35}"
)


print(
    "\nFailure detection:"
)

print(
    failed_detection_df_35.to_string(
        index=False
    )
)


print(
    f"\nAll healthy UUVs detected failure : "
    f"{all_healthy_detected_failure_35}"
)

print(
    f"All detections occurred post-fail  : "
    f"{all_detections_after_failure_35}"
)

print(
    f"False-positive declarations        : "
    f"{false_positive_count_35}"
)

print(
    f"Post-failure failed-UUV heartbeats : "
    f"{post_failure_transmissions_from_failed_35}"
)


print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm unchanged      : "
    f"{live_swarm_unchanged_35}"
)

print(
    f"  Live known maps unchanged : "
    f"{live_known_maps_unchanged_35}"
)


print(
    "\nDistributed heartbeat-based "
    "failure detection validated."
)

# Combo 36 — Safety-Feasible Heartbeat-Triggered Fault Recovery

## Objective

The previous version introduced contact-aware heartbeat failure detection correctly, but exposed another systems-level issue.

A failed UUV remains a stationary physical obstacle.

Therefore an orphaned task located too close to the failed UUV may be impossible for a healthy successor to complete while satisfying:

$$
d_{safe}=128\text{ px}
$$

and simultaneously reaching:

$$
d_{goal}\le32\text{ px}.
$$

This revised combo explicitly enforces **recovery-task feasibility**.

---

# 1. Physical Failure

UUV-1 permanently fails at:

$$
t_f=50.
$$

After failure:

- motion stops;
- sensing stops;
- communication stops;
- the vehicle remains physically present.

Therefore:

$$
p_f(t)=p_f(t_f)
$$

for:

$$
t\ge t_f.
$$

---

# 2. Contact-Aware Heartbeat Detection

Failure evidence accumulates only while detector $i$ is geometrically within communication range of peer $j$.

Define:

$$
C_{ij}(t)
=
\mathbf{1}
[
\|p_i-p_j\|\le R_c
].
$$

If:

$$
C_{ij}=0
$$

no suspicion is accumulated.

If geometric contact exists but no fresh heartbeat is received:

$$
S_{ij}\leftarrow S_{ij}+1.
$$

A fresh heartbeat resets:

$$
S_{ij}\leftarrow0.
$$

Peer failure is locally declared when:

$$
S_{ij}\ge40.
$$

---

# 3. Majority Confirmation

There are three healthy UUVs after UUV-1 fails.

Recovery begins after at least two independent declarations:

$$
N_{confirm}=2.
$$

Thus:

$$
t_c>t_f.
$$

---

# 4. Recovery-Task Safety Feasibility

Suppose orphan goal $g$ is at distance:

$$
d_f(g)
=
\|g-p_f\|
$$

from the stationary failed UUV.

The successor must ultimately reach within:

$$
R_{goal}=32
$$

of $g$ while staying at least:

$$
d_{safe}=128
$$

from the failed vehicle.

A conservative feasibility condition is therefore imposed:

$$
d_f(g)
\ge
d_{safe}+R_{goal}.
$$

For this prototype:

$$
d_f(g)\ge160\text{ px}.
$$

This intentionally leaves geometric margin.

---

# 5. Orphan Selection

At physical failure time, save:

- failed-UUV position;
- frozen known map;
- active assignment.

After majority confirmation:

1. use the physical-failure assignment if it remains safety-feasible;
2. otherwise generate frontier candidates from the frozen failure-time map;
3. reject any candidate inside the failed-UUV exclusion radius;
4. choose the highest-utility remaining legitimate frontier.

Therefore:

$$
g^{orphan}
$$

is both information-driven and physically recoverable.

---

# 6. Recovery Priority

Once successor $i^*$ is selected:

$$
g_{i^*}=g^{orphan}
$$

and ordinary allocation cannot overwrite it.

The task remains active until:

$$
d(i^*,g^{orphan})\le32\text{ px}.
$$

---

# 7. Extended Mission Horizon

Heartbeat confirmation may require significant time because:

- agents move;
- geometric contact is intermittent;
- links blackout;
- packets drop;
- messages are delayed.

Therefore the validation horizon is increased to:

$$
T=320
$$

timesteps.

This does not alter controller behavior. It merely ensures the experiment has enough time to observe both delayed detection and physical recovery.

---

# 8. Recovery Timing

Measure:

$$
T_{detect}=t_c-t_f
$$

$$
T_{execute}=t_{complete}-t_c
$$

$$
T_{total}=t_{complete}-t_f.
$$

---

# 9. Required Validation

The experiment must satisfy:

$$
t_c>t_f
$$

$$
N_{false}=0
$$

$$
d_f(g^{orphan})\ge160
$$

$$
i^*\neq i_f
$$

$$
d(i^*,g^{orphan})\le32
$$

$$
\Delta p_f=0
$$

and:

$$
d_{\min}^{mission}\ge128.
$$

A successful run establishes:

$$
\boxed{
\text{Distributed Failure Detection}
+
\text{Safety-Feasible Task Recovery}
}
$$

---

In [ ]:
# ============================================================
# Combo 36 — Safety-Feasible Heartbeat Fault Recovery
# ============================================================

restore_checkpoint_24_5()


# ============================================================
# Configuration
# ============================================================

RECOVERY_MISSION_STEPS_36 = 320

FAILED_UUV_ID_36 = 1
PHYSICAL_FAILURE_TIME_36 = 50

HEARTBEAT_SUSPICION_THRESHOLD_36 = 40
MAJORITY_CONFIRMATIONS_36 = 2

RECOVERY_CHANNEL_SEED_36 = 360036

RECOVERY_INFO_WEIGHT_36 = 0.65
RECOVERY_DISTANCE_WEIGHT_36 = 0.35

ORPHAN_FAILED_UUV_CLEARANCE_36 = (
    SAFE_SEPARATION
    +
    GOAL_REACHED_RADIUS
)


rng_36 = np.random.default_rng(
    RECOVERY_CHANNEL_SEED_36
)


uuv_ids_36 = [
    agent.uuv_id
    for agent in agents
]


# ============================================================
# Failure state
# ============================================================

failed_flags_36 = {
    uuv_id: False
    for uuv_id in uuv_ids_36
}


physical_failure_position_36 = None

physical_failure_local_cells_36 = None
physical_failure_known_cells_36 = None
physical_failure_resource_count_36 = None

physical_failure_known_snapshot_36 = None
physical_failure_goal_snapshot_36 = None


# ============================================================
# Detection state
# ============================================================

suspicion_count_36 = {
    detector: {
        peer: 0
        for peer in uuv_ids_36
        if peer != detector
    }
    for detector in uuv_ids_36
}


declared_failed_36 = {
    detector: set()
    for detector in uuv_ids_36
}


detection_time_36 = {
    detector: {}
    for detector in uuv_ids_36
}


fault_confirmed_36 = False
majority_confirmation_time_36 = None


# ============================================================
# Heartbeat communication
# ============================================================

heartbeat_queue_36 = []

next_heartbeat_id_36 = 0

heartbeat_attempted_36 = 0
heartbeat_dropped_36 = 0
heartbeat_delivered_36 = 0
heartbeat_blackout_suppressed_36 = 0


fresh_heartbeat_pairs_36 = set()


# ============================================================
# Recovery task state
# ============================================================

orphan_task_36 = None
orphan_source_36 = None

orphan_successor_36 = None

orphan_active_36 = False
orphan_completed_36 = False

orphan_assignment_time_36 = None
orphan_completion_time_36 = None

orphan_initial_distance_36 = None
orphan_final_distance_36 = None

orphan_failed_uuv_distance_36 = None


successor_candidate_records_36 = []


# ============================================================
# Initialize known maps
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ============================================================
# Starting snapshots
# ============================================================

start_local_maps_36 = {
    agent.uuv_id:
        agent.exploration_memory.observed.copy()
    for agent in agents
}


start_traj_idx_36 = {
    agent.uuv_id:
        len(agent.trajectory) - 1
    for agent in agents
}


start_union_36 = np.logical_or.reduce(
    list(
        start_local_maps_36.values()
    )
)


start_team_coverage_36 = (
    int(start_union_36.sum())
    /
    (WIDTH * HEIGHT)
)


# ============================================================
# Mission communication channel
# ============================================================

channel_36 = DegradedCommunicationChannel(
    uuv_ids=
        uuv_ids_36,

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        rng_36,
)


# ============================================================
# Dedicated heartbeat blackout manager
# ============================================================

heartbeat_blackout_36 = LinkBlackoutManager(
    uuv_ids=
        uuv_ids_36,

    start_probability=
        BLACKOUT_START_PROBABILITY,

    min_duration=
        BLACKOUT_MIN_DURATION,

    max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        rng_36,
)


# ============================================================
# Allocation state
# ============================================================

goals_36 = {
    uuv_id: None
    for uuv_id in uuv_ids_36
}


last_allocation_step_36 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_36 = None

allocation_rounds_36 = 0


# ============================================================
# Logs
# ============================================================

coverage_history_36 = []

mission_min_separation_36 = np.inf


hold_counts_36 = {
    uuv_id: 0
    for uuv_id in uuv_ids_36
}


safety_overrides_36 = {
    uuv_id: 0
    for uuv_id in uuv_ids_36
}


healthy_motion_steps_after_failure_36 = 0


# ============================================================
# Heartbeat helper
# ============================================================

def queue_heartbeat_36(
    sender,
    receiver,
    current_time,
):

    global next_heartbeat_id_36
    global heartbeat_attempted_36
    global heartbeat_dropped_36


    heartbeat_attempted_36 += 1


    survives = packet_delivered(
        rng_36,
        loss_probability=
            PACKET_LOSS_PROBABILITY,
    )


    if not survives:

        heartbeat_dropped_36 += 1

        return


    delay = sample_message_delay(
        rng_36,
        min_delay=
            MIN_COMM_DELAY,
        max_delay=
            MAX_COMM_DELAY,
    )


    heartbeat_queue_36.append(
        {
            "message_id":
                next_heartbeat_id_36,

            "sender":
                sender,

            "receiver":
                receiver,

            "send_time":
                current_time,

            "arrival_time":
                current_time
                +
                delay,
        }
    )


    next_heartbeat_id_36 += 1


# ============================================================
# Heartbeat delivery
# ============================================================

def deliver_heartbeats_36(
    current_time,
):

    global heartbeat_queue_36
    global heartbeat_delivered_36
    global fresh_heartbeat_pairs_36


    fresh_heartbeat_pairs_36 = set()


    due = [
        message
        for message in heartbeat_queue_36
        if (
            message[
                "arrival_time"
            ]
            <=
            current_time
        )
    ]


    future = [
        message
        for message in heartbeat_queue_36
        if (
            message[
                "arrival_time"
            ]
            >
            current_time
        )
    ]


    for message in due:

        sender = message[
            "sender"
        ]

        receiver = message[
            "receiver"
        ]


        if failed_flags_36[
            receiver
        ]:

            continue


        fresh_heartbeat_pairs_36.add(
            (
                receiver,
                sender,
            )
        )


        heartbeat_delivered_36 += 1


    heartbeat_queue_36 = future


# ============================================================
# Safety-feasibility helper
# ============================================================

def orphan_goal_safe_from_failed_uuv_36(
    goal,
):

    if goal is None:
        return False


    return (
        euclidean_distance(
            (
                goal[
                    "x"
                ],
                goal[
                    "y"
                ],
            ),
            physical_failure_position_36,
        )
        >=
        ORPHAN_FAILED_UUV_CLEARANCE_36
    )


# ============================================================
# Create safety-feasible orphan task
# ============================================================

def create_safe_orphan_task_36(
    failed_agent,
):

    # --------------------------------------------------------
    # Prefer real active task if physically recoverable.
    # --------------------------------------------------------

    if (
        physical_failure_goal_snapshot_36
        is not None
        and
        orphan_goal_safe_from_failed_uuv_36(
            physical_failure_goal_snapshot_36
        )
    ):

        return (
            deepcopy(
                physical_failure_goal_snapshot_36
            ),
            "active_goal_at_physical_failure",
        )


    # --------------------------------------------------------
    # Frozen failure-time planning memory
    # --------------------------------------------------------

    frozen_memory = PlanningMemory(
        uuv_id=
            failed_agent.uuv_id,

        observed_mask=
            physical_failure_known_snapshot_36,
    )


    frontier_mask = extract_frontier_mask(
        frozen_memory
    )


    candidates = sample_frontier_candidates(
        frontier_mask,

        reference_position=
            physical_failure_position_36,

        min_spacing=
            TASK_DEDUPLICATION_DISTANCE,

        max_candidates=
            24,
    )


    # --------------------------------------------------------
    # Reject candidates too close to dead vehicle.
    # --------------------------------------------------------

    feasible_candidates = [
        candidate
        for candidate in candidates

        if (
            euclidean_distance(
                (
                    candidate[
                        "x"
                    ],
                    candidate[
                        "y"
                    ],
                ),
                physical_failure_position_36,
            )
            >=
            ORPHAN_FAILED_UUV_CLEARANCE_36
        )
    ]


    if not feasible_candidates:

        raise RuntimeError(
            "No safety-feasible orphan frontier "
            "exists in failure snapshot."
        )


    scored = [
        score_frontier_candidate_with_memory(
            failed_agent,
            candidate,
            frozen_memory,
        )
        for candidate
        in feasible_candidates
    ]


    selected = max(
        scored,
        key=lambda record:
            record[
                "utility"
            ],
    )


    return (
        deepcopy(selected),
        "generated_safe_frontier_from_failure_snapshot",
    )


# ============================================================
# Successor scoring
# ============================================================

def score_successor_36(
    agent,
    orphan_goal,
):

    planning_memory = PlanningMemory(
        uuv_id=
            agent.uuv_id,

        observed_mask=
            known_exploration_maps[
                agent.uuv_id
            ],
    )


    stats = candidate_fov_statistics(
        exploration_memory=
            planning_memory,

        candidate_x=
            orphan_goal[
                "x"
            ],

        candidate_y=
            orphan_goal[
                "y"
            ],
    )


    distance_px = euclidean_distance(
        (
            agent.state.x,
            agent.state.y,
        ),
        (
            orphan_goal[
                "x"
            ],
            orphan_goal[
                "y"
            ],
        ),
    )


    distance_norm = (
        distance_px
        /
        MAP_DIAGONAL
    )


    utility = (
        RECOVERY_INFO_WEIGHT_36
        *
        stats[
            "information_ratio"
        ]
        -
        RECOVERY_DISTANCE_WEIGHT_36
        *
        distance_norm
    )


    return {
        "uuv_id":
            agent.uuv_id,

        "information_ratio":
            float(
                stats[
                    "information_ratio"
                ]
            ),

        "distance_px":
            float(
                distance_px
            ),

        "utility":
            float(
                utility
            ),
    }


# ============================================================
# Mission loop
# ============================================================

for t in range(
    RECOVERY_MISSION_STEPS_36
):

    # ========================================================
    # 1. Physical failure
    # ========================================================

    if (
        t
        ==
        PHYSICAL_FAILURE_TIME_36
    ):

        failed_agent = agent_by_id[
            FAILED_UUV_ID_36
        ]


        physical_failure_position_36 = (
            failed_agent.state.x,
            failed_agent.state.y,
        )


        physical_failure_local_cells_36 = (
            failed_agent
            .exploration_memory
            .observed_count()
        )


        physical_failure_known_cells_36 = int(
            known_exploration_maps[
                FAILED_UUV_ID_36
            ].sum()
        )


        physical_failure_resource_count_36 = len(
            failed_agent.resource_memory
        )


        physical_failure_known_snapshot_36 = (
            known_exploration_maps[
                FAILED_UUV_ID_36
            ].copy()
        )


        physical_failure_goal_snapshot_36 = deepcopy(
            goals_36[
                FAILED_UUV_ID_36
            ]
        )


        failed_flags_36[
            FAILED_UUV_ID_36
        ] = True


        goals_36[
            FAILED_UUV_ID_36
        ] = None


    # ========================================================
    # 2. Deliver mission messages
    # ========================================================

    deliver_due_failure_aware_33(
        channel=
            channel_36,

        current_time=
            t,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory
            for agent in agents
        },

        failed_flags=
            failed_flags_36,
    )


    # ========================================================
    # 3. Deliver heartbeat packets
    # ========================================================

    deliver_heartbeats_36(
        current_time=
            t
    )


    # ========================================================
    # 4. Local -> known
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


    # ========================================================
    # 5. Physical geometry
    # ========================================================

    physical_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }


    heartbeat_geo_graph = (
        build_communication_graph(
            physical_positions,

            communication_range=
                COMMUNICATION_RANGE,
        )
    )


    # ========================================================
    # 6. Heartbeat blackout state
    # ========================================================

    heartbeat_blackout_result = (
        heartbeat_blackout_36.step(
            heartbeat_geo_graph,

            allow_new_blackouts=True,
        )
    )


    heartbeat_operational_graph = (
        heartbeat_blackout_result[
            "operational_graph"
        ]
    )


    heartbeat_blackout_suppressed_36 += (
        (
            heartbeat_geo_graph
            .number_of_edges()
            -
            heartbeat_operational_graph
            .number_of_edges()
        )
        *
        2
    )


    # ========================================================
    # 7. Healthy heartbeat transmissions
    # ========================================================

    for uuv_a, uuv_b in (
        heartbeat_operational_graph.edges()
    ):

        for sender, receiver in (
            (uuv_a, uuv_b),
            (uuv_b, uuv_a),
        ):

            if (
                failed_flags_36[
                    sender
                ]
                or
                failed_flags_36[
                    receiver
                ]
            ):

                continue


            queue_heartbeat_36(
                sender=
                    sender,

                receiver=
                    receiver,

                current_time=
                    t,
            )


    # ========================================================
    # 8. Contact-aware suspicion
    # ========================================================

    healthy_ids = [
        uuv_id
        for uuv_id in uuv_ids_36
        if not failed_flags_36[
            uuv_id
        ]
    ]


    for detector in healthy_ids:

        for peer in uuv_ids_36:

            if peer == detector:
                continue


            if (
                peer
                in
                declared_failed_36[
                    detector
                ]
            ):

                continue


            geometric_contact = (
                heartbeat_geo_graph.has_edge(
                    detector,
                    peer,
                )
            )


            if not geometric_contact:
                continue


            if (
                (
                    detector,
                    peer,
                )
                in
                fresh_heartbeat_pairs_36
            ):

                suspicion_count_36[
                    detector
                ][
                    peer
                ] = 0


            else:

                suspicion_count_36[
                    detector
                ][
                    peer
                ] += 1


            if (
                suspicion_count_36[
                    detector
                ][
                    peer
                ]
                >=
                HEARTBEAT_SUSPICION_THRESHOLD_36
            ):

                declared_failed_36[
                    detector
                ].add(
                    peer
                )


                detection_time_36[
                    detector
                ][
                    peer
                ] = t


    # ========================================================
    # 9. Majority confirmation + recovery assignment
    # ========================================================

    if (
        not fault_confirmed_36
        and
        t
        >
        PHYSICAL_FAILURE_TIME_36
    ):

        detectors = [
            detector
            for detector in healthy_ids
            if (
                FAILED_UUV_ID_36
                in
                declared_failed_36[
                    detector
                ]
            )
        ]


        if (
            len(detectors)
            >=
            MAJORITY_CONFIRMATIONS_36
        ):

            fault_confirmed_36 = True

            majority_confirmation_time_36 = t


            failed_agent = agent_by_id[
                FAILED_UUV_ID_36
            ]


            (
                orphan_task_36,
                orphan_source_36,
            ) = create_safe_orphan_task_36(
                failed_agent
            )


            orphan_failed_uuv_distance_36 = (
                euclidean_distance(
                    (
                        orphan_task_36[
                            "x"
                        ],
                        orphan_task_36[
                            "y"
                        ],
                    ),
                    physical_failure_position_36,
                )
            )


            successor_candidate_records_36 = [
                score_successor_36(
                    agent,
                    orphan_task_36,
                )
                for agent in agents
                if not failed_flags_36[
                    agent.uuv_id
                ]
            ]


            successor_candidate_records_36.sort(
                key=lambda record:
                    (
                        record[
                            "utility"
                        ],
                        -
                        record[
                            "distance_px"
                        ],
                    ),
                reverse=True,
            )


            orphan_successor_36 = (
                successor_candidate_records_36[
                    0
                ][
                    "uuv_id"
                ]
            )


            successor_agent = agent_by_id[
                orphan_successor_36
            ]


            orphan_initial_distance_36 = (
                euclidean_distance(
                    (
                        successor_agent.state.x,
                        successor_agent.state.y,
                    ),
                    (
                        orphan_task_36[
                            "x"
                        ],
                        orphan_task_36[
                            "y"
                        ],
                    ),
                )
            )


            orphan_assignment_time_36 = t
            orphan_active_36 = True


            goals_36[
                orphan_successor_36
            ] = deepcopy(
                orphan_task_36
            )


    # ========================================================
    # 10. Mission communication among healthy UUVs
    # ========================================================

    healthy_positions = {
        uuv_id:
            physical_positions[
                uuv_id
            ]
        for uuv_id
        in healthy_ids
    }


    if len(
        healthy_positions
    ) >= 2:

        transmission_result = (
            channel_36.transmit_round(
                current_time=
                    t,

                positions=
                    healthy_positions,

                known_maps=
                    known_exploration_maps,

                resource_memories={
                    agent.uuv_id:
                        agent.resource_memory
                    for agent in agents
                },
            )
        )


        operational_graph = (
            transmission_result[
                "operational_graph"
            ]
        )


    else:

        operational_graph = nx.Graph()

        operational_graph.add_nodes_from(
            healthy_ids
        )


    # ========================================================
    # 11. Ordinary goal validity
    # ========================================================

    goal_became_unavailable = False


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_36[
            uuv_id
        ]:

            continue


        if (
            orphan_active_36
            and
            uuv_id
            ==
            orphan_successor_36
        ):

            continue


        goal = goals_36[
            uuv_id
        ]


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_36[
                uuv_id
            ] = None

            goal_became_unavailable = True


    # ========================================================
    # 12. Stable allocation
    # ========================================================

    operational_graph.add_nodes_from(
        healthy_ids
    )


    canonical_edges = frozenset(
        tuple(
            sorted(edge)
        )
        for edge
        in operational_graph.edges()
    )


    topology_changed = (
        previous_operational_edges_36
        is None
        or
        canonical_edges
        !=
        previous_operational_edges_36
    )


    previous_operational_edges_36 = (
        canonical_edges
    )


    missing_goal = any(
        goals_36[
            uuv_id
        ]
        is None
        for uuv_id
        in healthy_ids
    )


    periodic_trigger = (
        t
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger = (
        topology_changed
        or
        goal_became_unavailable
        or
        missing_goal
    )


    cooldown_satisfied = (
        t
        -
        last_allocation_step_36
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate = (
        periodic_trigger
        or
        (
            urgent_trigger
            and
            cooldown_satisfied
        )
    )


    if (
        should_allocate
        and
        healthy_ids
    ):

        (
            assignments,
            _,
            _,
            _,
            _,
        ) = perform_stable_allocation_round(
            operational_graph,
            goals_36,
        )


        for uuv_id in healthy_ids:

            goals_36[
                uuv_id
            ] = assignments.get(
                uuv_id
            )


        goals_36[
            FAILED_UUV_ID_36
        ] = None


        # ----------------------------------------------------
        # Preserve recovery priority.
        # ----------------------------------------------------

        if orphan_active_36:

            goals_36[
                orphan_successor_36
            ] = deepcopy(
                orphan_task_36
            )


            for uuv_id in healthy_ids:

                if (
                    uuv_id
                    ==
                    orphan_successor_36
                ):
                    continue


                competing = goals_36[
                    uuv_id
                ]


                if competing is None:
                    continue


                if (
                    euclidean_distance(
                        (
                            competing[
                                "x"
                            ],
                            competing[
                                "y"
                            ],
                        ),
                        (
                            orphan_task_36[
                                "x"
                            ],
                            orphan_task_36[
                                "y"
                            ],
                        ),
                    )
                    <
                    MIN_ASSIGNED_GOAL_SEPARATION
                ):

                    goals_36[
                        uuv_id
                    ] = None


        allocation_rounds_36 += 1

        last_allocation_step_36 = t


    # ========================================================
    # 13. Desired actions
    # ========================================================

    desired_actions = []


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_36[
            uuv_id
        ]:

            desired_action = ACTION_HOLD


        else:

            goal = goals_36[
                uuv_id
            ]


            if goal is None:

                desired_action = ACTION_HOLD

                hold_counts_36[
                    uuv_id
                ] += 1


            else:

                desired_action = (
                    choose_greedy_action(
                        agent.state,

                        goal_x=
                            goal[
                                "x"
                            ],

                        goal_y=
                            goal[
                                "y"
                            ],
                    )
                )


                if desired_action is None:

                    desired_action = ACTION_HOLD

                    hold_counts_36[
                        uuv_id
                    ] += 1


        desired_actions.append(
            desired_action
        )


    # ========================================================
    # 14. Failure-aware safety
    # ========================================================

    states_now = [
        agent.state
        for agent in agents
    ]


    failed_mask = [
        failed_flags_36[
            agent.uuv_id
        ]
        for agent in agents
    ]


    shield = failure_aware_safety_shield_33(
        states=
            states_now,

        desired_actions=
            desired_actions,

        failed_mask=
            failed_mask,

        safe_separation=
            SAFE_SEPARATION,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    executed_actions = shield[
        "executed_actions"
    ]


    mission_min_separation_36 = min(
        mission_min_separation_36,
        shield[
            "minimum_distance"
        ],
    )


    for agent, desired, executed in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if (
            not failed_flags_36[
                agent.uuv_id
            ]
            and
            desired != executed
        ):

            safety_overrides_36[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 15. Execute motion
    # ========================================================

    previous_positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )
        for agent in agents
    }


    next_states = predict_joint_states_33(
        states=
            states_now,

        actions=
            executed_actions,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = next_state


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    if (
        t
        >=
        PHYSICAL_FAILURE_TIME_36
    ):

        if any(
            (
                agent.state.x,
                agent.state.y,
            )
            !=
            previous_positions[
                agent.uuv_id
            ]
            for agent in agents
            if not failed_flags_36[
                agent.uuv_id
            ]
        ):

            healthy_motion_steps_after_failure_36 += 1


    # ========================================================
    # 16. Healthy sensing
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_36[
            uuv_id
        ]:
            continue


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 17. Recovery-task completion
    # ========================================================

    if orphan_active_36:

        successor_agent = agent_by_id[
            orphan_successor_36
        ]


        distance_to_orphan = (
            euclidean_distance(
                (
                    successor_agent.state.x,
                    successor_agent.state.y,
                ),
                (
                    orphan_task_36[
                        "x"
                    ],
                    orphan_task_36[
                        "y"
                    ],
                ),
            )
        )


        if (
            distance_to_orphan
            <=
            GOAL_REACHED_RADIUS
        ):

            orphan_completed_36 = True

            orphan_active_36 = False

            orphan_completion_time_36 = t

            orphan_final_distance_36 = (
                distance_to_orphan
            )


            goals_36[
                orphan_successor_36
            ] = None


    # ========================================================
    # 18. Coverage
    # ========================================================

    current_union = np.logical_or.reduce(
        [
            agent.exploration_memory.observed
            for agent in agents
        ]
    )


    coverage_history_36.append(
        float(
            current_union.sum()
            /
            (
                WIDTH
                *
                HEIGHT
            )
        )
    )


# ============================================================
# Final metrics
# ============================================================

failed_agent = agent_by_id[
    FAILED_UUV_ID_36
]


failed_displacement_36 = (
    euclidean_distance(
        physical_failure_position_36,
        (
            failed_agent.state.x,
            failed_agent.state.y,
        ),
    )
)


final_union_36 = np.logical_or.reduce(
    [
        agent.exploration_memory.observed
        for agent in agents
    ]
)


final_team_coverage_36 = (
    int(
        final_union_36.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Detection table
# ============================================================

detection_records_36 = []


for detector in uuv_ids_36:

    if detector == FAILED_UUV_ID_36:
        continue


    detected_time = (
        detection_time_36[
            detector
        ].get(
            FAILED_UUV_ID_36
        )
    )


    detection_records_36.append(
        {
            "detector":
                detector,

            "detection_time":
                detected_time,

            "delay_from_failure":
                (
                    None
                    if detected_time is None
                    else
                    detected_time
                    -
                    PHYSICAL_FAILURE_TIME_36
                ),

            "suspicion_count":
                suspicion_count_36[
                    detector
                ][
                    FAILED_UUV_ID_36
                ],
        }
    )


detection_df_36 = pd.DataFrame(
    detection_records_36
)


# ============================================================
# False positives
# ============================================================

false_positive_count_36 = 0


for detector in uuv_ids_36:

    if detector == FAILED_UUV_ID_36:
        continue


    for peer in declared_failed_36[
        detector
    ]:

        if peer != FAILED_UUV_ID_36:

            false_positive_count_36 += 1


# ============================================================
# Recovery timing
# ============================================================

if fault_confirmed_36:

    detection_delay_36 = (
        majority_confirmation_time_36
        -
        PHYSICAL_FAILURE_TIME_36
    )

else:

    detection_delay_36 = None


if orphan_completed_36:

    execution_delay_36 = (
        orphan_completion_time_36
        -
        majority_confirmation_time_36
    )


    total_recovery_delay_36 = (
        orphan_completion_time_36
        -
        PHYSICAL_FAILURE_TIME_36
    )

else:

    execution_delay_36 = None
    total_recovery_delay_36 = None


# ============================================================
# Assertions
# ============================================================

if not fault_confirmed_36:

    raise RuntimeError(
        "Majority failure confirmation never occurred."
    )


if (
    majority_confirmation_time_36
    <=
    PHYSICAL_FAILURE_TIME_36
):

    raise RuntimeError(
        "Failure confirmation was not delayed."
    )


if false_positive_count_36 != 0:

    raise RuntimeError(
        "False-positive failure declaration occurred."
    )


if orphan_task_36 is None:

    raise RuntimeError(
        "No orphan task was created."
    )


if (
    orphan_failed_uuv_distance_36
    <
    ORPHAN_FAILED_UUV_CLEARANCE_36
):

    raise RuntimeError(
        "Orphan task violates failed-UUV "
        "safety-clearance requirement."
    )


if orphan_successor_36 is None:

    raise RuntimeError(
        "No healthy successor was selected."
    )


if not orphan_completed_36:

    raise RuntimeError(
        "Safety-feasible orphan task was still "
        "not completed within mission horizon."
    )


if failed_displacement_36 != 0.0:

    raise RuntimeError(
        "Failed UUV moved after failure."
    )


if (
    mission_min_separation_36
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Combo 36 safety invariant failed."
    )


# ============================================================
# Candidate report
# ============================================================

successor_df_36 = pd.DataFrame(
    successor_candidate_records_36
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 112
)

print(
    "MultiUUV-Link — Safety-Feasible "
    "Heartbeat-Triggered Fault Recovery"
)

print(
    "=" * 112
)


print(
    f"\nPhysical failed UUV               : "
    f"{FAILED_UUV_ID_36}"
)

print(
    f"Physical failure timestep         : "
    f"{PHYSICAL_FAILURE_TIME_36}"
)

print(
    f"Contact suspicion threshold       : "
    f"{HEARTBEAT_SUSPICION_THRESHOLD_36}"
)

print(
    f"Required majority                 : "
    f"{MAJORITY_CONFIRMATIONS_36}"
)


print(
    "\nDistributed failure detections:"
)

print(
    detection_df_36.to_string(
        index=False
    )
)


print(
    f"\nMajority confirmation timestep    : "
    f"{majority_confirmation_time_36}"
)

print(
    f"Detection delay                   : "
    f"{detection_delay_36} steps"
)

print(
    f"False positives                   : "
    f"{false_positive_count_36}"
)


print(
    "\nRecovery task:"
)

print(
    f"  Source                          : "
    f"{orphan_source_36}"
)

print(
    f"  Goal                            : "
    f"({orphan_task_36['x']}, "
    f"{orphan_task_36['y']})"
)

print(
    f"  Distance from failed UUV        : "
    f"{orphan_failed_uuv_distance_36:.2f} px"
)

print(
    f"  Required failed-UUV clearance   : "
    f"{ORPHAN_FAILED_UUV_CLEARANCE_36:.2f} px"
)


print(
    "\nSuccessor candidates:"
)

print(
    successor_df_36.to_string(
        index=False,
        formatters={
            "information_ratio":
                lambda x: f"{x:.4f}",

            "distance_px":
                lambda x: f"{x:.2f}",

            "utility":
                lambda x: f"{x:.4f}",
        },
    )
)


print(
    "\nRecovery performance:"
)

print(
    f"  Selected successor              : "
    f"UUV-{orphan_successor_36}"
)

print(
    f"  Initial successor distance      : "
    f"{orphan_initial_distance_36:.2f} px"
)

print(
    f"  Completion timestep             : "
    f"{orphan_completion_time_36}"
)

print(
    f"  Detection delay                 : "
    f"{detection_delay_36} steps"
)

print(
    f"  Execution delay                 : "
    f"{execution_delay_36} steps"
)

print(
    f"  Total recovery delay            : "
    f"{total_recovery_delay_36} steps"
)

print(
    f"  Final orphan distance           : "
    f"{orphan_final_distance_36:.2f} px"
)

print(
    f"  Task completed                  : "
    f"{orphan_completed_36}"
)


print(
    "\nMission continuation:"
)

print(
    f"  Start team coverage             : "
    f"{100 * start_team_coverage_36:.4f}%"
)

print(
    f"  Final team coverage             : "
    f"{100 * final_team_coverage_36:.4f}%"
)

print(
    f"  Coverage gain                   : "
    f"{100 * (final_team_coverage_36 - start_team_coverage_36):.4f} pp"
)

print(
    f"  Healthy post-failure move steps : "
    f"{healthy_motion_steps_after_failure_36}"
)

print(
    f"  Allocation rounds               : "
    f"{allocation_rounds_36}"
)

print(
    f"  Healthy HOLD steps              : "
    f"{sum(hold_counts_36.values())}"
)

print(
    f"  Safety overrides                : "
    f"{sum(safety_overrides_36.values())}"
)

print(
    f"  Failed-UUV displacement         : "
    f"{failed_displacement_36:.2f} px"
)

print(
    f"  Mission minimum separation      : "
    f"{mission_min_separation_36:.2f} px"
)


print(
    "\nDistributed failure detection and "
    "safety-feasible autonomous recovery validated."
)


# ============================================================
# Visualization
# ============================================================

plt.figure(
    figsize=(12, 7)
)


plt.imshow(
    seabed_rgb
)


for agent in agents:

    start_idx = (
        start_traj_idx_36[
            agent.uuv_id
        ]
    )


    path = np.asarray(
        agent.trajectory[
            start_idx:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=70,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=110,
        linewidths=2.5,
    )


plt.scatter(
    physical_failure_position_36[0],
    physical_failure_position_36[1],
    marker="X",
    s=240,
    label=f"UUV-{FAILED_UUV_ID_36} FAILURE",
)


plt.scatter(
    orphan_task_36[
        "x"
    ],
    orphan_task_36[
        "y"
    ],
    marker="*",
    s=300,
    label="Safety-feasible orphan task",
)


plt.title(
    "MultiUUV-Link — Distributed Fault Detection and Recovery"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 37 — Learned Frontier-Value Dataset Generation

## Objective

The classical autonomy stack is now sufficiently mature to introduce the required PyTorch-based learned component.

Rather than immediately replacing a working planner with an unvalidated neural network, the learned-development path begins with a supervised frontier-value model.

The first learned module will estimate:

$$
\hat J_\theta(i,g)
$$

for UUV $i$ and frontier candidate $g$.

The model will later be used as a learned frontier-ranking component.

This combo only constructs and validates the training dataset.

Training begins in the next combo.

---

# Classical Frontier Utility

The current classical planner scores candidate goal $g$ using:

$$
J_i(g)
=
w_I I_i(g)
-
w_D D_i(g)
-
w_R R_i(g)
$$

where:

- $I_i(g)$ is predicted information gain;
- $D_i(g)$ is normalized travel distance;
- $R_i(g)$ is predicted observation redundancy.

The existing classical planner therefore provides a useful initial supervision signal.

---

# Why Begin with Classical Supervision?

The goal is not to claim that the neural model already improves on the classical controller.

Instead, this establishes:

$$
\boxed{
\text{PyTorch Learning Infrastructure}
}
$$

inside the autonomy pipeline.

The development sequence is:

```text
Classical Frontier Planner
        ↓
Generate Training Samples
        ↓
Learn Frontier Value Function
        ↓
Validate Learned Ranking
        ↓
Integrate into Mission
        ↓
Compare Classical vs Learned
```

Later experiments can replace or augment the imitation target with actual mission-outcome data.

---

# Feature Vector

For each UUV-candidate pair define:

$$
x_{i,g}
=
[
I,
D,
R,
C_i,
K_i,
d_i^{comm}
]
$$

where:

### Information ratio

$$
I
=
\frac{N_{unknown}}{N_{valid}}
$$

### Normalized distance

$$
D
=
\frac{d_i(g)}{d_{map}}
$$

### Redundancy ratio

$$
R
=
\frac{N_{known}}{N_{valid}}
$$

### Local coverage

$$
C_i
=
\frac{|M_i^{local}|}{WH}
$$

### Known coverage

$$
K_i
=
\frac{|M_i^{known}|}{WH}
$$

### Normalized communication degree

For four UUVs:

$$
d_i^{comm}
=
\frac{degree_i}{3}.
$$

Thus:

$$
x\in\mathbb R^6.
$$

---

# Learning Target

The initial target is the existing classical utility:

$$
y_{i,g}
=
J_i^{classical}(g).
$$

The network will therefore initially learn to approximate the classical frontier ranking.

This provides a controlled baseline before introducing outcome-based learning.

---

# Dataset Diversity

A useful dataset should not come from only one final swarm state.

Therefore this combo generates samples from multiple synthetic partial-knowledge states derived from the current checkpoint maps.

For each original UUV memory, stochastic masks retain different fractions of its known cells.

This creates knowledge states spanning approximately:

$$
20\%-100\%
$$

of available known information.

The original local/known maps themselves remain unchanged.

---

# Synthetic Partial Knowledge

For mask realization $k$:

$$
M_{i,k}^{synthetic}
=
M_i^{known}
\land
B_k
$$

where:

$$
B_k\sim Bernoulli(p_k).
$$

Retention probabilities are sampled over:

$$
p_k\in[0.2,1.0].
$$

This produces many frontier configurations without rerunning hundreds of full missions.

---

# Candidate Generation

For every synthetic memory:

1. extract the frontier mask;
2. sample spatially separated frontier candidates;
3. score every candidate using the existing classical scoring function;
4. record feature-target pairs.

Each sample becomes:

$$
(x_{i,g},y_{i,g}).
$$

---

# Dataset Requirements

The generated dataset must:

- contain finite numeric features;
- contain finite targets;
- contain samples from all four UUVs;
- contain a broad range of information ratios;
- contain both positive and negative utilities;
- contain enough samples for train/validation/test splitting.

Target minimum:

$$
N\ge1000.
$$

---

# Reproducibility

Use:

```text
LEARN_DATASET_SEED = 370037
```

so later training and comparisons can reproduce exactly the same dataset.

---

# No Live-State Mutation

This combo operates only on copied/synthetic exploration masks.

It must not modify:

- UUV positions;
- trajectories;
- local memories;
- known maps;
- resources;
- communication state.

---

# Output

The code should report:

- number of samples;
- samples per UUV;
- feature statistics;
- target statistics;
- positive/negative target counts;
- NaN/infinite checks;
- live-state integrity.

It should also create:

```text
frontier_dataset_37
X_frontier_37
y_frontier_37
```

for the next combo.

A successful result establishes:

$$
\boxed{
\text{Training Data Pipeline for Learned Frontier Valuation}
}
$$

---

In [ ]:
# ============================================================
# Combo 37 — Learned Frontier-Value Dataset Generation
# ============================================================

import torch


# ============================================================
# Configuration
# ============================================================

LEARN_DATASET_SEED = 370037

SYNTHETIC_STATES_PER_UUV = 80

MAX_DATASET_CANDIDATES = 20

MIN_SYNTHETIC_RETENTION = 0.20

MAX_SYNTHETIC_RETENTION = 1.00


dataset_rng_37 = np.random.default_rng(
    LEARN_DATASET_SEED
)


# ============================================================
# Preserve live experiment state
# ============================================================

live_signature_before_37 = (
    get_swarm_state_signature()
)


live_known_maps_before_37 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Helper: synthetic planning memory
# ============================================================

def build_synthetic_known_map_37(
    source_mask,
    retention_probability,
    rng,
):
    """
    Randomly retain known cells from an existing legitimate
    knowledge mask.

    Unknown cells remain unknown.
    """

    keep_mask = (
        rng.random(
            source_mask.shape
        )
        <
        retention_probability
    )


    synthetic_mask = (
        source_mask
        &
        keep_mask
    )


    return synthetic_mask


# ============================================================
# Helper: feature extraction
# ============================================================

def build_frontier_features_37(
    agent,
    candidate,
    planning_memory,
    communication_degree,
):
    """
    Six-dimensional learned-frontier feature vector.
    """

    scored = (
        score_frontier_candidate_with_memory(
            agent,
            candidate,
            planning_memory,
        )
    )


    local_coverage = (
        agent
        .exploration_memory
        .coverage()
    )


    known_coverage = (
        planning_memory
        .coverage()
    )


    normalized_comm_degree = (
        communication_degree
        /
        max(
            1,
            len(
                agents
            ) - 1,
        )
    )


    features = {
        "information_ratio":
            float(
                scored[
                    "info_ratio"
                ]
            ),

        "distance_norm":
            float(
                scored[
                    "distance_px"
                ]
                /
                MAP_DIAGONAL
            ),

        "redundancy_ratio":
            float(
                scored[
                    "redundancy_ratio"
                ]
            ),

        "local_coverage":
            float(
                local_coverage
            ),

        "known_coverage":
            float(
                known_coverage
            ),

        "comm_degree_norm":
            float(
                normalized_comm_degree
            ),
    }


    target = float(
        scored[
            "utility"
        ]
    )


    return (
        features,
        target,
        scored,
    )


# ============================================================
# Reference communication degree
# ============================================================

current_positions_37 = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )

    for agent in agents
}


reference_graph_37 = (
    build_communication_graph(
        current_positions_37,

        communication_range=
            COMMUNICATION_RANGE,
    )
)


reference_degree_37 = {
    agent.uuv_id:
        int(
            reference_graph_37.degree[
                agent.uuv_id
            ]
        )

    for agent in agents
}


# ============================================================
# Generate dataset
# ============================================================

dataset_records_37 = []


for agent in agents:

    uuv_id = agent.uuv_id


    source_known_map = (
        known_exploration_maps[
            uuv_id
        ]
        .copy()
    )


    for synthetic_index in range(
        SYNTHETIC_STATES_PER_UUV
    ):

        retention_probability = float(
            dataset_rng_37.uniform(
                MIN_SYNTHETIC_RETENTION,
                MAX_SYNTHETIC_RETENTION,
            )
        )


        synthetic_mask = (
            build_synthetic_known_map_37(
                source_mask=
                    source_known_map,

                retention_probability=
                    retention_probability,

                rng=
                    dataset_rng_37,
            )
        )


        planning_memory = (
            PlanningMemory(
                uuv_id=
                    uuv_id,

                observed_mask=
                    synthetic_mask,
            )
        )


        # ----------------------------------------------------
        # Frontier extraction
        # ----------------------------------------------------

        frontier_mask = (
            extract_frontier_mask(
                planning_memory
            )
        )


        # ----------------------------------------------------
        # Spatially separated candidates
        # ----------------------------------------------------

        candidates = (
            sample_frontier_candidates(
                frontier_mask,

                reference_position=(
                    agent.state.x,
                    agent.state.y,
                ),

                min_spacing=
                    96,

                max_candidates=
                    MAX_DATASET_CANDIDATES,
            )
        )


        for candidate_index, candidate in enumerate(
            candidates
        ):

            (
                features,
                target,
                scored,
            ) = build_frontier_features_37(
                agent=
                    agent,

                candidate=
                    candidate,

                planning_memory=
                    planning_memory,

                communication_degree=
                    reference_degree_37[
                        uuv_id
                    ],
            )


            dataset_records_37.append(
                {
                    "uuv_id":
                        uuv_id,

                    "synthetic_state":
                        synthetic_index,

                    "retention_probability":
                        retention_probability,

                    "candidate_index":
                        candidate_index,

                    "candidate_x":
                        int(
                            candidate[
                                "x"
                            ]
                        ),

                    "candidate_y":
                        int(
                            candidate[
                                "y"
                            ]
                        ),

                    **features,

                    "classical_utility":
                        target,
                }
            )


# ============================================================
# DataFrame
# ============================================================

frontier_dataset_37 = pd.DataFrame(
    dataset_records_37
)


if frontier_dataset_37.empty:

    raise RuntimeError(
        "Combo 37 generated an empty dataset."
    )


# ============================================================
# Tensor matrices
# ============================================================

FEATURE_COLUMNS_37 = [
    "information_ratio",
    "distance_norm",
    "redundancy_ratio",
    "local_coverage",
    "known_coverage",
    "comm_degree_norm",
]


TARGET_COLUMN_37 = (
    "classical_utility"
)


X_frontier_np_37 = (
    frontier_dataset_37[
        FEATURE_COLUMNS_37
    ]
    .to_numpy(
        dtype=np.float32
    )
)


y_frontier_np_37 = (
    frontier_dataset_37[
        TARGET_COLUMN_37
    ]
    .to_numpy(
        dtype=np.float32
    )
    .reshape(
        -1,
        1,
    )
)


X_frontier_37 = torch.tensor(
    X_frontier_np_37,
    dtype=torch.float32,
)


y_frontier_37 = torch.tensor(
    y_frontier_np_37,
    dtype=torch.float32,
)


# ============================================================
# Dataset validation
# ============================================================

sample_count_37 = len(
    frontier_dataset_37
)


finite_features_37 = bool(
    np.isfinite(
        X_frontier_np_37
    ).all()
)


finite_targets_37 = bool(
    np.isfinite(
        y_frontier_np_37
    ).all()
)


all_uuvs_present_37 = (
    set(
        frontier_dataset_37[
            "uuv_id"
        ].unique()
    )
    ==
    set(
        agent.uuv_id
        for agent in agents
    )
)


positive_targets_37 = int(
    (
        frontier_dataset_37[
            TARGET_COLUMN_37
        ]
        >
        0
    ).sum()
)


negative_targets_37 = int(
    (
        frontier_dataset_37[
            TARGET_COLUMN_37
        ]
        <
        0
    ).sum()
)


zero_targets_37 = int(
    (
        frontier_dataset_37[
            TARGET_COLUMN_37
        ]
        ==
        0
    ).sum()
)


if sample_count_37 < 1000:

    raise RuntimeError(
        f"Dataset too small: {sample_count_37} samples."
    )


if not finite_features_37:

    raise RuntimeError(
        "Non-finite frontier features detected."
    )


if not finite_targets_37:

    raise RuntimeError(
        "Non-finite training targets detected."
    )


if not all_uuvs_present_37:

    raise RuntimeError(
        "Dataset does not contain all UUV IDs."
    )


# ============================================================
# Live-state integrity
# ============================================================

live_signature_after_37 = (
    get_swarm_state_signature()
)


live_swarm_unchanged_37 = (
    live_signature_before_37
    ==
    live_signature_after_37
)


live_known_maps_unchanged_37 = all(
    np.array_equal(
        live_known_maps_before_37[
            uuv_id
        ],

        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged_37:

    raise RuntimeError(
        "Dataset generation mutated live swarm state."
    )


if not live_known_maps_unchanged_37:

    raise RuntimeError(
        "Dataset generation mutated live known maps."
    )


# ============================================================
# Statistics
# ============================================================

samples_per_uuv_37 = (
    frontier_dataset_37
    .groupby(
        "uuv_id"
    )
    .size()
    .rename(
        "samples"
    )
)


feature_statistics_37 = (
    frontier_dataset_37[
        FEATURE_COLUMNS_37
    ]
    .describe()
    .T
)


target_statistics_37 = (
    frontier_dataset_37[
        TARGET_COLUMN_37
    ]
    .describe()
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 108
)

print(
    "MultiUUV-Link — Learned Frontier-Value "
    "Dataset Generation"
)

print(
    "=" * 108
)


print(
    f"\nPyTorch version               : "
    f"{torch.__version__}"
)

print(
    f"Dataset seed                  : "
    f"{LEARN_DATASET_SEED}"
)

print(
    f"Synthetic states per UUV      : "
    f"{SYNTHETIC_STATES_PER_UUV}"
)

print(
    f"Total frontier samples        : "
    f"{sample_count_37}"
)

print(
    f"Feature dimension             : "
    f"{X_frontier_37.shape[1]}"
)


print(
    "\nSamples per UUV:"
)

print(
    samples_per_uuv_37.to_string()
)


print(
    "\nFeature statistics:"
)

print(
    feature_statistics_37[
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    ].to_string(
        float_format=
            lambda x: f"{x:.4f}"
    )
)


print(
    "\nTarget statistics:"
)

print(
    target_statistics_37.to_string(
        float_format=
            lambda x: f"{x:.4f}"
    )
)


print(
    "\nTarget sign distribution:"
)

print(
    f"  Positive utilities : "
    f"{positive_targets_37}"
)

print(
    f"  Negative utilities : "
    f"{negative_targets_37}"
)

print(
    f"  Zero utilities     : "
    f"{zero_targets_37}"
)


print(
    "\nTensor shapes:"
)

print(
    f"  X_frontier_37 : "
    f"{tuple(X_frontier_37.shape)}"
)

print(
    f"  y_frontier_37 : "
    f"{tuple(y_frontier_37.shape)}"
)


print(
    "\nValidation:"
)

print(
    f"  Finite features          : "
    f"{finite_features_37}"
)

print(
    f"  Finite targets           : "
    f"{finite_targets_37}"
)

print(
    f"  All UUVs represented     : "
    f"{all_uuvs_present_37}"
)

print(
    f"  Live swarm unchanged     : "
    f"{live_swarm_unchanged_37}"
)

print(
    f"  Live known maps unchanged: "
    f"{live_known_maps_unchanged_37}"
)


print(
    "\nLearned frontier-value dataset "
    "generated and validated."
)

# Combo 38 — PyTorch Learned Frontier-Value Model

## Objective

Combo 37 produced:

$$
N=6400
$$

frontier-candidate samples with:

$$
x\in\mathbb{R}^6
$$

and classical frontier utility:

$$
y=J^{classical}(g).
$$

This combo trains the first required PyTorch learned component:

$$
\boxed{
\hat J_\theta(x)
}
$$

The model learns to estimate frontier utility from the six-dimensional feature vector.

---

# Model

Use a small multilayer perceptron:

$$
6
\rightarrow
32
\rightarrow
32
\rightarrow
1
$$

with ReLU nonlinearities:

$$
h_1=ReLU(W_1x+b_1)
$$

$$
h_2=ReLU(W_2h_1+b_2)
$$

$$
\hat J=W_3h_2+b_3.
$$

This is intentionally lightweight.

The point is not network size.

The point is introducing a trainable PyTorch decision component into the autonomy stack.

---

# Data Splitting

Candidates generated from the same synthetic knowledge state are correlated.

Therefore we must **not randomly split individual rows**.

Instead, the dataset is grouped by:

```text
(UUV ID, synthetic state)
```

All candidates belonging to one synthetic state remain entirely inside one split.

Use approximately:

$$
70\%\;train
$$

$$
15\%\;validation
$$

$$
15\%\;test.
$$

This prevents candidate-level leakage between training and evaluation.

---

# Feature Standardization

Feature normalization parameters are computed using **training data only**:

$$
\mu_{train}
$$

and:

$$
\sigma_{train}.
$$

Then:

$$
x'
=
\frac{x-\mu_{train}}
{\sigma_{train}+\epsilon}.
$$

Validation and test sets reuse the same training statistics.

---

# Training Objective

Use mean squared error:

$$
\mathcal L
=
\frac{1}{N}
\sum_n
(
\hat J_n-J_n
)^2.
$$

Optimizer:

```text
Adam
```

with:

$$
\eta=10^{-3}.
$$

---

# Early Model Validation

Because this model ultimately ranks frontier candidates, regression error alone is not enough.

Evaluate:

### Mean Squared Error

$$
MSE
$$

### Mean Absolute Error

$$
MAE
$$

### Coefficient of Determination

$$
R^2
$$

### Pearson Correlation

Measures linear agreement between predicted and true utilities.

### Spearman Correlation

Measures ranking agreement.

### Top-1 Frontier Agreement

For each held-out synthetic state:

1. find the candidate with maximum classical utility;
2. find the candidate with maximum predicted utility;
3. check whether they are the same candidate.

Define:

$$
A_{top1}
=
\frac{
N_{\text{matching best candidates}}
}{
N_{\text{held-out states}}
}.
$$

This is particularly important because the network will later replace or augment candidate ranking.

---

# Training Progress

Training uses:

$$
150
$$

epochs.

A live progress bar should display:

```text
epoch
training loss
validation loss
```

so long-running training is never hidden behind a Jupyter `[*]`.

---

# Device

Use CUDA when available:

$$
device=
\begin{cases}
CUDA,&\text{if available}\\
CPU,&\text{otherwise}
\end{cases}
$$

The current system should normally use the RTX 4060.

---

# Outputs

This combo should create persistent notebook objects:

```text
frontier_model_38
feature_mean_38
feature_std_38
FEATURE_COLUMNS_37
```

These will be reused when the learned model is integrated into frontier planning.

---

# Expected Result

Because the target is generated from a relatively simple classical utility function, the neural model should learn it accurately.

We expect:

- low validation/test error;
- strong Pearson correlation;
- strong Spearman ranking correlation;
- meaningful top-1 frontier agreement.

A successful result establishes:

$$
\boxed{
\text{First Trained PyTorch Autonomy Component}
}
$$

---

In [ ]:
# ============================================================
# Combo 38 — Train PyTorch Frontier-Value Network
# ============================================================

import random
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# Reproducibility
# ============================================================

TRAIN_SEED_38 = 380038

random.seed(
    TRAIN_SEED_38
)

np.random.seed(
    TRAIN_SEED_38
)

torch.manual_seed(
    TRAIN_SEED_38
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        TRAIN_SEED_38
    )


# ============================================================
# Device
# ============================================================

device_38 = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


# ============================================================
# Group-aware split
#
# One group =
#     (uuv_id, synthetic_state)
# ============================================================

group_keys_38 = (
    frontier_dataset_37[
        [
            "uuv_id",
            "synthetic_state",
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


group_indices_38 = np.arange(
    len(
        group_keys_38
    )
)


split_rng_38 = np.random.default_rng(
    TRAIN_SEED_38
)


split_rng_38.shuffle(
    group_indices_38
)


num_groups_38 = len(
    group_indices_38
)


num_train_groups_38 = int(
    round(
        0.70
        *
        num_groups_38
    )
)


num_val_groups_38 = int(
    round(
        0.15
        *
        num_groups_38
    )
)


train_group_indices_38 = (
    group_indices_38[
        :
        num_train_groups_38
    ]
)


val_group_indices_38 = (
    group_indices_38[
        num_train_groups_38
        :
        num_train_groups_38
        +
        num_val_groups_38
    ]
)


test_group_indices_38 = (
    group_indices_38[
        num_train_groups_38
        +
        num_val_groups_38
        :
    ]
)


def group_tuple_set_38(
    indices,
):

    return set(
        tuple(
            row
        )
        for row
        in group_keys_38.iloc[
            indices
        ][
            [
                "uuv_id",
                "synthetic_state",
            ]
        ].to_numpy()
    )


train_groups_38 = (
    group_tuple_set_38(
        train_group_indices_38
    )
)


val_groups_38 = (
    group_tuple_set_38(
        val_group_indices_38
    )
)


test_groups_38 = (
    group_tuple_set_38(
        test_group_indices_38
    )
)


# ============================================================
# Row masks
# ============================================================

dataset_group_pairs_38 = list(
    zip(
        frontier_dataset_37[
            "uuv_id"
        ].astype(
            int
        ),

        frontier_dataset_37[
            "synthetic_state"
        ].astype(
            int
        ),
    )
)


train_mask_38 = np.array(
    [
        pair in train_groups_38
        for pair
        in dataset_group_pairs_38
    ],
    dtype=bool,
)


val_mask_38 = np.array(
    [
        pair in val_groups_38
        for pair
        in dataset_group_pairs_38
    ],
    dtype=bool,
)


test_mask_38 = np.array(
    [
        pair in test_groups_38
        for pair
        in dataset_group_pairs_38
    ],
    dtype=bool,
)


# ============================================================
# Split arrays
# ============================================================

X_train_np_38 = (
    X_frontier_np_37[
        train_mask_38
    ]
)


y_train_np_38 = (
    y_frontier_np_37[
        train_mask_38
    ]
)


X_val_np_38 = (
    X_frontier_np_37[
        val_mask_38
    ]
)


y_val_np_38 = (
    y_frontier_np_37[
        val_mask_38
    ]
)


X_test_np_38 = (
    X_frontier_np_37[
        test_mask_38
    ]
)


y_test_np_38 = (
    y_frontier_np_37[
        test_mask_38
    ]
)


# ============================================================
# Feature normalization from TRAIN ONLY
# ============================================================

feature_mean_np_38 = (
    X_train_np_38.mean(
        axis=0,
        keepdims=True,
    )
)


feature_std_np_38 = (
    X_train_np_38.std(
        axis=0,
        keepdims=True,
    )
)


feature_std_np_38 = np.where(
    feature_std_np_38
    <
    1e-8,

    1.0,

    feature_std_np_38,
)


feature_mean_38 = torch.tensor(
    feature_mean_np_38,
    dtype=torch.float32,
)


feature_std_38 = torch.tensor(
    feature_std_np_38,
    dtype=torch.float32,
)


def normalize_features_38(
    X_np,
):

    return (
        (
            X_np
            -
            feature_mean_np_38
        )
        /
        feature_std_np_38
    ).astype(
        np.float32
    )


X_train_norm_38 = (
    normalize_features_38(
        X_train_np_38
    )
)


X_val_norm_38 = (
    normalize_features_38(
        X_val_np_38
    )
)


X_test_norm_38 = (
    normalize_features_38(
        X_test_np_38
    )
)


# ============================================================
# Torch tensors
# ============================================================

X_train_38 = torch.tensor(
    X_train_norm_38,
    dtype=torch.float32,
)


y_train_38 = torch.tensor(
    y_train_np_38,
    dtype=torch.float32,
)


X_val_38 = torch.tensor(
    X_val_norm_38,
    dtype=torch.float32,
)


y_val_38 = torch.tensor(
    y_val_np_38,
    dtype=torch.float32,
)


X_test_38 = torch.tensor(
    X_test_norm_38,
    dtype=torch.float32,
)


y_test_38 = torch.tensor(
    y_test_np_38,
    dtype=torch.float32,
)


# ============================================================
# Data loader
# ============================================================

BATCH_SIZE_38 = 256


train_loader_38 = DataLoader(
    TensorDataset(
        X_train_38,
        y_train_38,
    ),

    batch_size=
        BATCH_SIZE_38,

    shuffle=
        True,

    generator=
        torch.Generator()
        .manual_seed(
            TRAIN_SEED_38
        ),
)


# ============================================================
# Model
# ============================================================

class FrontierValueNetwork38(
    nn.Module
):

    def __init__(
        self,
        input_dim=6,
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                32,
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                32,
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                1,
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.network(
            x
        )


frontier_model_38 = (
    FrontierValueNetwork38(
        input_dim=
            len(
                FEATURE_COLUMNS_37
            )
    )
    .to(
        device_38
    )
)


# ============================================================
# Optimization
# ============================================================

criterion_38 = nn.MSELoss()


optimizer_38 = torch.optim.Adam(
    frontier_model_38.parameters(),

    lr=1e-3,
)


NUM_EPOCHS_38 = 150


train_loss_history_38 = []
val_loss_history_38 = []


# ============================================================
# Progress bar
# ============================================================

try:

    from tqdm.auto import tqdm

    epoch_iterator_38 = tqdm(
        range(
            1,
            NUM_EPOCHS_38 + 1
        ),

        desc=
            "Training frontier model",
    )

    use_tqdm_38 = True


except Exception:

    epoch_iterator_38 = range(
        1,
        NUM_EPOCHS_38 + 1
    )

    use_tqdm_38 = False


# ============================================================
# Training
# ============================================================

X_val_device_38 = (
    X_val_38.to(
        device_38
    )
)


y_val_device_38 = (
    y_val_38.to(
        device_38
    )
)


for epoch in epoch_iterator_38:

    frontier_model_38.train()


    epoch_loss_sum = 0.0

    epoch_samples = 0


    for (
        batch_X,
        batch_y,
    ) in train_loader_38:

        batch_X = batch_X.to(
            device_38
        )

        batch_y = batch_y.to(
            device_38
        )


        optimizer_38.zero_grad()


        predictions = (
            frontier_model_38(
                batch_X
            )
        )


        loss = criterion_38(
            predictions,
            batch_y,
        )


        loss.backward()

        optimizer_38.step()


        batch_count = (
            batch_X.shape[
                0
            ]
        )


        epoch_loss_sum += (
            loss.item()
            *
            batch_count
        )


        epoch_samples += (
            batch_count
        )


    train_loss = (
        epoch_loss_sum
        /
        epoch_samples
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    frontier_model_38.eval()


    with torch.no_grad():

        val_predictions = (
            frontier_model_38(
                X_val_device_38
            )
        )


        val_loss = criterion_38(
            val_predictions,
            y_val_device_38,
        ).item()


    train_loss_history_38.append(
        train_loss
    )


    val_loss_history_38.append(
        val_loss
    )


    # --------------------------------------------------------
    # Live progress display
    # --------------------------------------------------------

    if use_tqdm_38:

        epoch_iterator_38.set_postfix(
            train=f"{train_loss:.6f}",
            val=f"{val_loss:.6f}",
        )


    elif (
        epoch == 1
        or
        epoch % 10 == 0
        or
        epoch == NUM_EPOCHS_38
    ):

        progress_fraction = (
            epoch
            /
            NUM_EPOCHS_38
        )


        bar_width = 30


        filled = int(
            progress_fraction
            *
            bar_width
        )


        bar = (
            "█" * filled
            +
            "░"
            *
            (
                bar_width
                -
                filled
            )
        )


        print(
            f"[{bar}] "
            f"{epoch:3d}/{NUM_EPOCHS_38} "
            f"| train={train_loss:.6f} "
            f"| val={val_loss:.6f}"
        )


# ============================================================
# Predictions
# ============================================================

frontier_model_38.eval()


with torch.no_grad():

    train_pred_np_38 = (
        frontier_model_38(
            X_train_38.to(
                device_38
            )
        )
        .cpu()
        .numpy()
        .reshape(
            -1
        )
    )


    val_pred_np_38 = (
        frontier_model_38(
            X_val_38.to(
                device_38
            )
        )
        .cpu()
        .numpy()
        .reshape(
            -1
        )
    )


    test_pred_np_38 = (
        frontier_model_38(
            X_test_38.to(
                device_38
            )
        )
        .cpu()
        .numpy()
        .reshape(
            -1
        )
    )


y_train_flat_38 = (
    y_train_np_38.reshape(
        -1
    )
)


y_val_flat_38 = (
    y_val_np_38.reshape(
        -1
    )
)


y_test_flat_38 = (
    y_test_np_38.reshape(
        -1
    )
)


# ============================================================
# Metrics helpers
# ============================================================

def regression_metrics_38(
    y_true,
    y_pred,
):

    mse = float(
        np.mean(
            (
                y_true
                -
                y_pred
            )
            **
            2
        )
    )


    mae = float(
        np.mean(
            np.abs(
                y_true
                -
                y_pred
            )
        )
    )


    ss_res = float(
        np.sum(
            (
                y_true
                -
                y_pred
            )
            **
            2
        )
    )


    ss_tot = float(
        np.sum(
            (
                y_true
                -
                y_true.mean()
            )
            **
            2
        )
    )


    r2 = (
        1.0
        -
        ss_res
        /
        ss_tot
    )


    pearson = float(
        np.corrcoef(
            y_true,
            y_pred,
        )[
            0,
            1
        ]
    )


    true_ranks = pd.Series(
        y_true
    ).rank().to_numpy()


    pred_ranks = pd.Series(
        y_pred
    ).rank().to_numpy()


    spearman = float(
        np.corrcoef(
            true_ranks,
            pred_ranks,
        )[
            0,
            1
        ]
    )


    return {
        "MSE":
            mse,

        "MAE":
            mae,

        "R2":
            r2,

        "Pearson":
            pearson,

        "Spearman":
            spearman,
    }


train_metrics_38 = regression_metrics_38(
    y_train_flat_38,
    train_pred_np_38,
)


val_metrics_38 = regression_metrics_38(
    y_val_flat_38,
    val_pred_np_38,
)


test_metrics_38 = regression_metrics_38(
    y_test_flat_38,
    test_pred_np_38,
)


# ============================================================
# Top-1 frontier ranking agreement
# ============================================================

test_dataset_38 = (
    frontier_dataset_37.loc[
        test_mask_38
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


test_dataset_38[
    "predicted_utility"
] = test_pred_np_38


top1_total_groups_38 = 0

top1_matches_38 = 0


for (
    uuv_id,
    synthetic_state,
), group in (
    test_dataset_38.groupby(
        [
            "uuv_id",
            "synthetic_state",
        ]
    )
):

    if len(
        group
    ) == 0:

        continue


    classical_best_index = (
        group[
            "classical_utility"
        ].idxmax()
    )


    learned_best_index = (
        group[
            "predicted_utility"
        ].idxmax()
    )


    top1_total_groups_38 += 1


    if (
        classical_best_index
        ==
        learned_best_index
    ):

        top1_matches_38 += 1


top1_accuracy_38 = (
    top1_matches_38
    /
    top1_total_groups_38
)


# ============================================================
# Model parameter count
# ============================================================

model_parameter_count_38 = sum(
    parameter.numel()

    for parameter
    in frontier_model_38.parameters()
)


# ============================================================
# Report
# ============================================================

metrics_df_38 = pd.DataFrame(
    [
        {
            "split":
                "train",
            **train_metrics_38,
        },

        {
            "split":
                "validation",
            **val_metrics_38,
        },

        {
            "split":
                "test",
            **test_metrics_38,
        },
    ]
)


print(
    "=" * 108
)

print(
    "MultiUUV-Link — PyTorch Frontier-Value "
    "Network Training"
)

print(
    "=" * 108
)


print(
    f"\nDevice                    : "
    f"{device_38}"
)

if torch.cuda.is_available():

    print(
        f"GPU                       : "
        f"{torch.cuda.get_device_name(0)}"
    )


print(
    f"Training seed             : "
    f"{TRAIN_SEED_38}"
)

print(
    f"Model parameters          : "
    f"{model_parameter_count_38}"
)


print(
    "\nGroup-aware split:"
)

print(
    f"  Total groups            : "
    f"{num_groups_38}"
)

print(
    f"  Training groups         : "
    f"{len(train_groups_38)}"
)

print(
    f"  Validation groups       : "
    f"{len(val_groups_38)}"
)

print(
    f"  Test groups             : "
    f"{len(test_groups_38)}"
)


print(
    "\nSamples:"
)

print(
    f"  Training                : "
    f"{len(X_train_38)}"
)

print(
    f"  Validation              : "
    f"{len(X_val_38)}"
)

print(
    f"  Test                    : "
    f"{len(X_test_38)}"
)


print(
    "\nRegression metrics:"
)

print(
    metrics_df_38.to_string(
        index=False,
        formatters={
            "MSE":
                lambda x: f"{x:.6f}",

            "MAE":
                lambda x: f"{x:.6f}",

            "R2":
                lambda x: f"{x:.5f}",

            "Pearson":
                lambda x: f"{x:.5f}",

            "Spearman":
                lambda x: f"{x:.5f}",
        },
    )
)


print(
    "\nFrontier-ranking metric:"
)

print(
    f"  Held-out synthetic states : "
    f"{top1_total_groups_38}"
)

print(
    f"  Exact top-1 matches        : "
    f"{top1_matches_38}"
)

print(
    f"  Top-1 agreement            : "
    f"{100 * top1_accuracy_38:.2f}%"
)


print(
    "\nFinal losses:"
)

print(
    f"  Training loss            : "
    f"{train_loss_history_38[-1]:.8f}"
)

print(
    f"  Validation loss          : "
    f"{val_loss_history_38[-1]:.8f}"
)


# ============================================================
# Plot 1 — Training progress
# ============================================================

plt.figure(
    figsize=(10, 5)
)


plt.plot(
    range(
        1,
        NUM_EPOCHS_38 + 1
    ),

    train_loss_history_38,

    label="Training loss",
)


plt.plot(
    range(
        1,
        NUM_EPOCHS_38 + 1
    ),

    val_loss_history_38,

    label="Validation loss",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "MSE loss"
)

plt.title(
    "MultiUUV-Link — Frontier-Value Network Training"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2 — Test prediction quality
# ============================================================

plt.figure(
    figsize=(6, 6)
)


plt.scatter(
    y_test_flat_38,
    test_pred_np_38,
    alpha=0.5,
)


plot_min_38 = min(
    float(
        y_test_flat_38.min()
    ),
    float(
        test_pred_np_38.min()
    ),
)


plot_max_38 = max(
    float(
        y_test_flat_38.max()
    ),
    float(
        test_pred_np_38.max()
    ),
)


plt.plot(
    [
        plot_min_38,
        plot_max_38,
    ],

    [
        plot_min_38,
        plot_max_38,
    ],

    linestyle="--",
)


plt.xlabel(
    "Classical utility"
)

plt.ylabel(
    "Predicted utility"
)

plt.title(
    "Held-Out Frontier Utility Prediction"
)

plt.tight_layout()

plt.show()


print(
    "\nFirst PyTorch frontier-value model "
    "trained and evaluated."
)

# Combo 39 — Integrating the Learned Frontier-Value Model

## Objective

Combo 38 trained the first PyTorch autonomy component:

$$
\hat J_\theta(x)
$$

which estimates the classical frontier utility from:

$$
x=
[
I,
D,
R,
C_i,
K_i,
d_i^{comm}
].
$$

This combo integrates the learned model into frontier-candidate ranking.

The goal is not yet to claim that learning outperforms the classical planner.

The purpose is to prove:

$$
\boxed{
\text{A Trained PyTorch Model Can Directly Participate in Autonomy Decisions}
}
$$

---

# Learned Candidate Utility

For candidate frontier goal $g$:

$$
x_{i,g}
=
[
I,
D,
R,
C_i,
K_i,
d_i^{comm}
]
$$

The normalized feature vector is:

$$
x'
=
\frac{
x-\mu_{train}
}{
\sigma_{train}
}.
$$

The learned utility is then:

$$
J_{i,g}^{learned}
=
\hat J_\theta(x').
$$

---

# Classical vs Learned Ranking

For every current candidate, compute:

### Classical utility

$$
J^{classical}_{i,g}
$$

### Learned utility

$$
J^{learned}_{i,g}
$$

Then independently select:

$$
g_i^{classical}
=
\arg\max_g
J^{classical}_{i,g}
$$

and:

$$
g_i^{learned}
=
\arg\max_g
J^{learned}_{i,g}.
$$

---

# Decision Agreement

For each UUV define:

$$
A_i
=
\mathbf{1}
[
g_i^{classical}
=
g_i^{learned}
].
$$

The overall decision agreement is:

$$
A
=
\frac{
1
}{
N
}
\sum_i A_i.
$$

---

# Utility Correlation

Across all current frontier candidates, compare:

$$
J^{classical}
$$

and:

$$
J^{learned}.
$$

Measure:

- Pearson correlation;
- Spearman correlation;
- mean absolute utility error.

---

# Learned Planner Function

This combo introduces:

```text
score_frontier_candidate_learned(...)
```

and:

```text
generate_goal_from_knowledge_learned(...)
```

These functions mirror the existing classical planner but use the trained PyTorch network for ranking.

The original classical planner remains untouched.

---

# Communication-Degree Feature

The learned model was trained using normalized communication degree:

$$
d_i^{comm}
=
\frac{
degree_i
}{
N-1
}.
$$

Therefore the current geometric communication graph is used to compute this feature.

---

# Important Scientific Interpretation

The network was trained to imitate the existing classical frontier score.

Therefore strong agreement is expected.

This combo validates **learned decision integration**, not learned superiority.

A later benchmark will compare:

```text
Classical Planner
vs
Learned Planner
```

under identical checkpoint starts.

---

# No Mission Mutation

This combo performs only one-step planning analysis.

It must not move any UUV or alter:

- trajectories;
- local memories;
- known maps;
- resources.

---

# Expected Output

The code should report:

- current candidate count per UUV;
- classical selected goal;
- learned selected goal;
- goal-distance difference;
- per-UUV agreement;
- aggregate agreement;
- utility MAE;
- Pearson correlation;
- Spearman correlation;
- live-state integrity.

A successful result establishes:

$$
\boxed{
\text{PyTorch Frontier Ranking Integrated Into Planner}
}
$$

---

In [ ]:
# ============================================================
# Combo 39 — Learned Frontier Ranking Integration
# ============================================================

frontier_model_38.eval()


# ============================================================
# Preserve live state
# ============================================================

live_signature_before_39 = (
    get_swarm_state_signature()
)


live_known_maps_before_39 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Current communication graph
# ============================================================

current_positions_39 = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )

    for agent in agents
}


current_graph_39 = (
    build_communication_graph(
        current_positions_39,

        communication_range=
            COMMUNICATION_RANGE,
    )
)


current_degree_39 = {
    agent.uuv_id:
        int(
            current_graph_39.degree[
                agent.uuv_id
            ]
        )

    for agent in agents
}


# ============================================================
# Learned scoring function
# ============================================================

def score_frontier_candidate_learned_39(
    agent,
    candidate,
    planning_memory,
    communication_degree,
):
    """
    Compute classical diagnostic features, then estimate
    frontier value using the trained PyTorch model.
    """

    classical_score = (
        score_frontier_candidate_with_memory(
            agent,
            candidate,
            planning_memory,
        )
    )


    local_coverage = (
        agent
        .exploration_memory
        .coverage()
    )


    known_coverage = (
        planning_memory
        .coverage()
    )


    comm_degree_norm = (
        communication_degree
        /
        max(
            1,
            len(agents) - 1,
        )
    )


    feature_vector = np.array(
        [
            classical_score[
                "info_ratio"
            ],

            classical_score[
                "distance_px"
            ]
            /
            MAP_DIAGONAL,

            classical_score[
                "redundancy_ratio"
            ],

            local_coverage,

            known_coverage,

            comm_degree_norm,
        ],

        dtype=np.float32,
    )


    # --------------------------------------------------------
    # Normalize using TRAINING statistics only
    # --------------------------------------------------------

    feature_vector_norm = (
        (
            feature_vector
            -
            feature_mean_np_38.reshape(-1)
        )
        /
        feature_std_np_38.reshape(-1)
    )


    feature_tensor = torch.tensor(
        feature_vector_norm,
        dtype=torch.float32,
        device=device_38,
    ).unsqueeze(0)


    with torch.no_grad():

        predicted_utility = float(
            frontier_model_38(
                feature_tensor
            )
            .cpu()
            .item()
        )


    return {
        "candidate_id":
            candidate.get(
                "candidate_id",
                None,
            ),

        "x":
            int(
                candidate[
                    "x"
                ]
            ),

        "y":
            int(
                candidate[
                    "y"
                ]
            ),

        "learned_utility":
            predicted_utility,

        "classical_utility":
            float(
                classical_score[
                    "utility"
                ]
            ),

        "info_ratio":
            float(
                classical_score[
                    "info_ratio"
                ]
            ),

        "distance_px":
            float(
                classical_score[
                    "distance_px"
                ]
            ),

        "redundancy_ratio":
            float(
                classical_score[
                    "redundancy_ratio"
                ]
            ),

        "feature_vector":
            feature_vector,
    }


# ============================================================
# Learned goal generator
# ============================================================

def generate_goal_from_knowledge_learned_39(
    agent,
    knowledge_mask,
    communication_degree,
):
    """
    Frontier extraction + spatial candidate sampling +
    learned PyTorch candidate ranking.
    """

    planning_memory = PlanningMemory(
        uuv_id=
            agent.uuv_id,

        observed_mask=
            knowledge_mask,
    )


    frontier_mask = extract_frontier_mask(
        planning_memory
    )


    candidates = sample_frontier_candidates(
        frontier_mask,

        reference_position=(
            agent.state.x,
            agent.state.y,
        ),

        min_spacing=
            TASK_DEDUPLICATION_DISTANCE,

        max_candidates=
            12,
    )


    if not candidates:

        return {
            "selected":
                None,

            "candidates":
                [],

            "frontier_mask":
                frontier_mask,
        }


    learned_scores = [
        score_frontier_candidate_learned_39(
            agent=
                agent,

            candidate=
                candidate,

            planning_memory=
                planning_memory,

            communication_degree=
                communication_degree,
        )

        for candidate in candidates
    ]


    selected = max(
        learned_scores,

        key=lambda record:
            record[
                "learned_utility"
            ],
    )


    return {
        "selected":
            selected,

        "candidates":
            learned_scores,

        "frontier_mask":
            frontier_mask,
    }


# ============================================================
# Compare current classical and learned decisions
# ============================================================

comparison_records_39 = []

candidate_utility_records_39 = []


for agent in agents:

    uuv_id = agent.uuv_id


    planning_memory = PlanningMemory(
        uuv_id=
            uuv_id,

        observed_mask=
            known_exploration_maps[
                uuv_id
            ],
    )


    frontier_mask = extract_frontier_mask(
        planning_memory
    )


    candidates = sample_frontier_candidates(
        frontier_mask,

        reference_position=(
            agent.state.x,
            agent.state.y,
        ),

        min_spacing=
            TASK_DEDUPLICATION_DISTANCE,

        max_candidates=
            12,
    )


    if not candidates:

        comparison_records_39.append(
            {
                "uuv_id":
                    uuv_id,

                "candidate_count":
                    0,

                "classical_goal":
                    None,

                "learned_goal":
                    None,

                "goal_difference_px":
                    0.0,

                "agreement":
                    True,
            }
        )

        continue


    classical_scores = [
        score_frontier_candidate_with_memory(
            agent,
            candidate,
            planning_memory,
        )

        for candidate in candidates
    ]


    learned_scores = [
        score_frontier_candidate_learned_39(
            agent=
                agent,

            candidate=
                candidate,

            planning_memory=
                planning_memory,

            communication_degree=
                current_degree_39[
                    uuv_id
                ],
        )

        for candidate in candidates
    ]


    classical_selected = max(
        classical_scores,

        key=lambda record:
            record[
                "utility"
            ],
    )


    learned_selected = max(
        learned_scores,

        key=lambda record:
            record[
                "learned_utility"
            ],
    )


    classical_goal = (
        int(
            classical_selected[
                "x"
            ]
        ),
        int(
            classical_selected[
                "y"
            ]
        ),
    )


    learned_goal = (
        int(
            learned_selected[
                "x"
            ]
        ),
        int(
            learned_selected[
                "y"
            ]
        ),
    )


    agreement = (
        classical_goal
        ==
        learned_goal
    )


    goal_difference = (
        euclidean_distance(
            classical_goal,
            learned_goal,
        )
    )


    comparison_records_39.append(
        {
            "uuv_id":
                uuv_id,

            "candidate_count":
                len(
                    candidates
                ),

            "classical_goal":
                classical_goal,

            "learned_goal":
                learned_goal,

            "goal_difference_px":
                float(
                    goal_difference
                ),

            "agreement":
                bool(
                    agreement
                ),
        }
    )


    for classical, learned in zip(
        classical_scores,
        learned_scores,
    ):

        candidate_utility_records_39.append(
            {
                "uuv_id":
                    uuv_id,

                "x":
                    int(
                        learned[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        learned[
                            "y"
                        ]
                    ),

                "classical_utility":
                    float(
                        classical[
                            "utility"
                        ]
                    ),

                "learned_utility":
                    float(
                        learned[
                            "learned_utility"
                        ]
                    ),
            }
        )


# ============================================================
# DataFrames
# ============================================================

comparison_df_39 = pd.DataFrame(
    comparison_records_39
)


utility_df_39 = pd.DataFrame(
    candidate_utility_records_39
)


# ============================================================
# Aggregate decision agreement
# ============================================================

decision_agreement_39 = float(
    comparison_df_39[
        "agreement"
    ].mean()
)


# ============================================================
# Utility agreement metrics
# ============================================================

if not utility_df_39.empty:

    classical_values_39 = (
        utility_df_39[
            "classical_utility"
        ]
        .to_numpy()
    )


    learned_values_39 = (
        utility_df_39[
            "learned_utility"
        ]
        .to_numpy()
    )


    utility_mae_39 = float(
        np.mean(
            np.abs(
                classical_values_39
                -
                learned_values_39
            )
        )
    )


    utility_pearson_39 = float(
        np.corrcoef(
            classical_values_39,
            learned_values_39,
        )[
            0,
            1
        ]
    )


    classical_ranks_39 = (
        pd.Series(
            classical_values_39
        )
        .rank()
        .to_numpy()
    )


    learned_ranks_39 = (
        pd.Series(
            learned_values_39
        )
        .rank()
        .to_numpy()
    )


    utility_spearman_39 = float(
        np.corrcoef(
            classical_ranks_39,
            learned_ranks_39,
        )[
            0,
            1
        ]
    )


else:

    utility_mae_39 = np.nan
    utility_pearson_39 = np.nan
    utility_spearman_39 = np.nan


# ============================================================
# Live-state integrity
# ============================================================

live_signature_after_39 = (
    get_swarm_state_signature()
)


live_swarm_unchanged_39 = (
    live_signature_before_39
    ==
    live_signature_after_39
)


live_known_maps_unchanged_39 = all(
    np.array_equal(
        live_known_maps_before_39[
            uuv_id
        ],

        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged_39:

    raise RuntimeError(
        "Learned planning test mutated live swarm state."
    )


if not live_known_maps_unchanged_39:

    raise RuntimeError(
        "Learned planning test mutated live known maps."
    )


# ============================================================
# Console report
# ============================================================

print(
    "=" * 112
)

print(
    "MultiUUV-Link — Learned Frontier Ranking Integration"
)

print(
    "=" * 112
)


print(
    f"\nModel device                : "
    f"{device_38}"
)

print(
    f"Current communication edges : "
    f"{current_graph_39.number_of_edges()}"
)


print(
    "\nPer-UUV planning comparison:"
)

print(
    comparison_df_39.to_string(
        index=False,
        formatters={
            "goal_difference_px":
                lambda x: f"{x:.2f}",
        },
    )
)


print(
    "\nAggregate decision metrics:"
)

print(
    f"  Exact goal agreement      : "
    f"{100 * decision_agreement_39:.2f}%"
)

print(
    f"  Candidate utility MAE     : "
    f"{utility_mae_39:.6f}"
)

print(
    f"  Utility Pearson           : "
    f"{utility_pearson_39:.6f}"
)

print(
    f"  Utility Spearman          : "
    f"{utility_spearman_39:.6f}"
)


print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm unchanged      : "
    f"{live_swarm_unchanged_39}"
)

print(
    f"  Live known maps unchanged : "
    f"{live_known_maps_unchanged_39}"
)


print(
    "\nLearned PyTorch frontier ranking "
    "integrated into planning."
)

# Combo 40 — Full Mission Using the Learned Frontier Ranker

## Objective

Combo 39 proved that the trained PyTorch model can rank real frontier candidates correctly.

This combo places that learned ranker inside a complete cooperative exploration mission.

The mission still uses:

- local FOV sensing;
- persistent local and communicated memory;
- degraded communication;
- distributed task allocation;
- packet loss;
- latency;
- persistent blackouts;
- predictive collision safety.

The principal change is:

$$
J^{classical}(g)
\rightarrow
\hat J_\theta(g)
$$

during frontier-task ranking.

---

# Learned Allocation

For every connected operational communication component:

1. frontier candidates are generated from legitimate UUV knowledge;
2. candidates are spatially deduplicated;
3. every healthy UUV evaluates candidate utility using the PyTorch model;
4. the best available UUV-task pair is assigned;
5. assigned goals must remain spatially separated.

Thus the learned network directly influences mission-level motion.

---

# Controlled Starting State

The mission begins from:

```python
restore_checkpoint_24_5()
```

Therefore the learned mission starts from the same state used by the previous baseline experiments.

---

# Communication Conditions

Use the same degraded communication configuration as Combo 32:

$$
P_{loss}=0.25
$$

$$
\tau\in\{1,\ldots,5\}
$$

with persistent link blackouts.

For reproducibility, use the same communication seed:

```text
320032
```

This helps make later classical-vs-learned comparison more controlled.

---

# Mission Horizon

Run:

$$
T=180
$$

synchronized timesteps.

---

# Learned Task Utility

For candidate $g$ and UUV $i$:

$$
x_{i,g}
=
[
I,D,R,C_i,K_i,d_i^{comm}
]
$$

and:

$$
J^{learned}_{i,g}
=
\hat J_\theta(x_{i,g}).
$$

The network remains in:

```text
eval()
```

mode and performs inference using:

```text
torch.no_grad()
```

---

# Task Uniqueness

A candidate may be assigned to at most one UUV.

Additionally, simultaneous assigned goals must satisfy approximately:

$$
d(g_i,g_j)\ge128\text{ px}.
$$

If no unique useful task exists, that UUV executes:

$$
HOLD.
$$

---

# Safety

The existing predictive safety shield remains unchanged.

Therefore:

$$
d_{ij}\ge128\text{ px}
$$

must hold throughout the mission.

---

# Metrics

Record:

- start and final team coverage;
- team coverage gain;
- mission new local cells;
- unique new team cells;
- redundant new cells;
- mission incremental redundancy;
- allocation rounds;
- learned goal changes;
- HOLD steps;
- safety overrides;
- minimum separation;
- communication statistics.

---

# Scientific Interpretation

Because the network learned the classical utility function, similar mission behavior is expected.

This combo establishes:

$$
\boxed{
\text{Learned Model}
\rightarrow
\text{Planning Decision}
\rightarrow
\text{Physical Mission Motion}
}
$$

It does not yet establish superiority.

The next controlled experiment will compare classical and learned missions under matched conditions.

---

In [ ]:
# ============================================================
# Combo 40 — Full Learned-Frontier Cooperative Mission
# ============================================================

restore_checkpoint_24_5()

frontier_model_38.eval()


# ============================================================
# Configuration
# ============================================================

LEARNED_MISSION_STEPS_40 = 180

LEARNED_CHANNEL_SEED_40 = 320032

learned_rng_40 = np.random.default_rng(
    LEARNED_CHANNEL_SEED_40
)


# ============================================================
# Ensure local knowledge is included in known maps
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

start_local_maps_40 = {
    agent.uuv_id:
        agent.exploration_memory.observed.copy()

    for agent in agents
}


start_traj_idx_40 = {
    agent.uuv_id:
        len(
            agent.trajectory
        ) - 1

    for agent in agents
}


start_team_union_40 = (
    np.logical_or.reduce(
        list(
            start_local_maps_40.values()
        )
    )
)


start_team_coverage_40 = (
    int(
        start_team_union_40.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Communication channel
# ============================================================

channel_40 = DegradedCommunicationChannel(
    uuv_ids=[
        agent.uuv_id
        for agent in agents
    ],

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        learned_rng_40,
)


# ============================================================
# Learned task-generation helper
# ============================================================

def learned_candidate_tasks_40(
    agent,
    communication_degree,
):
    """
    Generate frontier candidates from legitimate known memory
    and rank them using the trained PyTorch model.
    """

    uuv_id = agent.uuv_id


    planning_memory = PlanningMemory(
        uuv_id=
            uuv_id,

        observed_mask=
            known_exploration_maps[
                uuv_id
            ],
    )


    frontier_mask = (
        extract_frontier_mask(
            planning_memory
        )
    )


    candidates = (
        sample_frontier_candidates(
            frontier_mask,

            reference_position=(
                agent.state.x,
                agent.state.y,
            ),

            min_spacing=
                TASK_DEDUPLICATION_DISTANCE,

            max_candidates=
                12,
        )
    )


    records = []


    for fallback_candidate_id, candidate in enumerate(
        candidates
    ):

        # ----------------------------------------------------
        # Defensive interface normalization
        #
        # Existing classical scoring function requires:
        # candidate["candidate_id"]
        # ----------------------------------------------------

        normalized_candidate = {
            "candidate_id":
                candidate.get(
                    "candidate_id",
                    fallback_candidate_id,
                ),

            "x":
                int(
                    candidate[
                        "x"
                    ]
                ),

            "y":
                int(
                    candidate[
                        "y"
                    ]
                ),
        }


        record = (
            score_frontier_candidate_learned_39(
                agent=
                    agent,

                candidate=
                    normalized_candidate,

                planning_memory=
                    planning_memory,

                communication_degree=
                    communication_degree,
            )
        )


        records.append(
            record
        )


    return records


# ============================================================
# Learned distributed allocator
# ============================================================

def perform_learned_allocation_round_40(
    operational_graph,
    previous_goals,
):
    """
    Greedy one-to-one distributed assignment using learned
    frontier utility.

    Important correction:
    every shared task preserves a valid candidate_id.
    """

    assignments = {
        agent.uuv_id: None
        for agent in agents
    }


    assignment_records = []


    # --------------------------------------------------------
    # Work on graph copy
    # --------------------------------------------------------

    graph = operational_graph.copy()


    graph.add_nodes_from(
        [
            agent.uuv_id
            for agent in agents
        ]
    )


    components = list(
        nx.connected_components(
            graph
        )
    )


    # ========================================================
    # Allocate independently inside each comm component
    # ========================================================

    for component_index, component in enumerate(
        components
    ):

        component = sorted(
            component
        )


        # ----------------------------------------------------
        # Shared spatially deduplicated task pool
        # ----------------------------------------------------

        task_pool = []


        for uuv_id in component:

            agent = agent_by_id[
                uuv_id
            ]


            communication_degree = (
                graph.degree[
                    uuv_id
                ]
            )


            candidate_records = (
                learned_candidate_tasks_40(
                    agent=
                        agent,

                    communication_degree=
                        communication_degree,
                )
            )


            for candidate in candidate_records:

                candidate_xy = (
                    candidate[
                        "x"
                    ],
                    candidate[
                        "y"
                    ],
                )


                duplicate = any(
                    euclidean_distance(
                        candidate_xy,

                        (
                            task[
                                "x"
                            ],
                            task[
                                "y"
                            ],
                        ),
                    )
                    <
                    TASK_DEDUPLICATION_DISTANCE

                    for task in task_pool
                )


                if duplicate:
                    continue


                # ------------------------------------------------
                # Create a NEW canonical task ID.
                #
                # This ID belongs to the shared task pool and is
                # preserved during every later scoring call.
                # ------------------------------------------------

                shared_task_id = (
                    component_index
                    *
                    1000
                    +
                    len(
                        task_pool
                    )
                )


                task_pool.append(
                    {
                        "candidate_id":
                            shared_task_id,

                        "x":
                            int(
                                candidate[
                                    "x"
                                ]
                            ),

                        "y":
                            int(
                                candidate[
                                    "y"
                                ]
                            ),
                    }
                )


        # ----------------------------------------------------
        # Nothing available in this component
        # ----------------------------------------------------

        if not task_pool:
            continue


        # ----------------------------------------------------
        # Score every UUV-task pair using learned model
        # ----------------------------------------------------

        component_pairs = []


        for uuv_id in component:

            agent = agent_by_id[
                uuv_id
            ]


            planning_memory = PlanningMemory(
                uuv_id=
                    uuv_id,

                observed_mask=
                    known_exploration_maps[
                        uuv_id
                    ],
            )


            communication_degree = (
                graph.degree[
                    uuv_id
                ]
            )


            for task_index, task in enumerate(
                task_pool
            ):

                # --------------------------------------------
                # task now contains:
                # candidate_id + x + y
                # --------------------------------------------

                score = (
                    score_frontier_candidate_learned_39(
                        agent=
                            agent,

                        candidate=
                            task,

                        planning_memory=
                            planning_memory,

                        communication_degree=
                            communication_degree,
                    )
                )


                component_pairs.append(
                    {
                        "uuv_id":
                            uuv_id,

                        "task_index":
                            task_index,

                        "candidate_id":
                            task[
                                "candidate_id"
                            ],

                        "x":
                            task[
                                "x"
                            ],

                        "y":
                            task[
                                "y"
                            ],

                        "utility":
                            float(
                                score[
                                    "learned_utility"
                                ]
                            ),
                    }
                )


        # ----------------------------------------------------
        # Highest learned utility first
        # ----------------------------------------------------

        component_pairs.sort(
            key=lambda record:
                record[
                    "utility"
                ],

            reverse=True,
        )


        assigned_uuvs = set()

        assigned_tasks = set()

        assigned_positions = []


        # ----------------------------------------------------
        # Greedy one-to-one assignment
        # ----------------------------------------------------

        for record in component_pairs:

            uuv_id = record[
                "uuv_id"
            ]


            task_index = record[
                "task_index"
            ]


            if uuv_id in assigned_uuvs:
                continue


            if task_index in assigned_tasks:
                continue


            candidate_position = (
                record[
                    "x"
                ],
                record[
                    "y"
                ],
            )


            spatial_conflict = any(
                euclidean_distance(
                    candidate_position,
                    assigned_position,
                )
                <
                MIN_ASSIGNED_GOAL_SEPARATION

                for assigned_position
                in assigned_positions
            )


            if spatial_conflict:
                continue


            assignments[
                uuv_id
            ] = {
                "candidate_id":
                    int(
                        record[
                            "candidate_id"
                        ]
                    ),

                "x":
                    int(
                        record[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        record[
                            "y"
                        ]
                    ),

                "learned_utility":
                    float(
                        record[
                            "utility"
                        ]
                    ),
            }


            assigned_uuvs.add(
                uuv_id
            )


            assigned_tasks.add(
                task_index
            )


            assigned_positions.append(
                candidate_position
            )


            assignment_records.append(
                deepcopy(
                    record
                )
            )


    # ========================================================
    # Goal-change count
    # ========================================================

    goal_changes = 0


    for uuv_id in assignments:

        old_goal = previous_goals.get(
            uuv_id
        )


        new_goal = assignments[
            uuv_id
        ]


        old_xy = (
            None

            if old_goal is None

            else (
                old_goal[
                    "x"
                ],
                old_goal[
                    "y"
                ],
            )
        )


        new_xy = (
            None

            if new_goal is None

            else (
                new_goal[
                    "x"
                ],
                new_goal[
                    "y"
                ],
            )
        )


        if old_xy != new_xy:

            goal_changes += 1


    return (
        assignments,
        goal_changes,
        assignment_records,
    )


# ============================================================
# Mission state
# ============================================================

goals_40 = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step_40 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_40 = None


allocation_rounds_40 = 0

learned_goal_changes_40 = 0


hold_counts_40 = {
    agent.uuv_id: 0
    for agent in agents
}


safety_overrides_40 = {
    agent.uuv_id: 0
    for agent in agents
}


mission_min_separation_40 = np.inf


# ============================================================
# Histories
# ============================================================

coverage_history_40 = []

redundancy_history_40 = []

geometric_edges_history_40 = []

operational_edges_history_40 = []


# ============================================================
# Mission loop
# ============================================================

for t in range(
    LEARNED_MISSION_STEPS_40
):

    # ========================================================
    # 1. Deliver delayed communication
    # ========================================================

    channel_40.deliver_due(
        current_time=
            t,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },
    )


    # ========================================================
    # 2. Preserve local knowledge in known map
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


    # ========================================================
    # 3. Degraded communication round
    # ========================================================

    positions = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    transmission_result = (
        channel_40.transmit_round(
            current_time=
                t,

            positions=
                positions,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )
    )


    geometric_graph = (
        transmission_result[
            "geometric_graph"
        ]
    )


    operational_graph = (
        transmission_result[
            "operational_graph"
        ]
    )


    geometric_edges_history_40.append(
        geometric_graph.number_of_edges()
    )


    operational_edges_history_40.append(
        operational_graph.number_of_edges()
    )


    # ========================================================
    # 4. Existing-goal validity
    # ========================================================

    goal_invalidated = False


    for agent in agents:

        uuv_id = agent.uuv_id


        goal = goals_40[
            uuv_id
        ]


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_40[
                uuv_id
            ] = None


            goal_invalidated = True


    # ========================================================
    # 5. Operational topology change
    # ========================================================

    canonical_edges = frozenset(
        tuple(
            sorted(
                edge
            )
        )

        for edge in operational_graph.edges()
    )


    topology_changed = (
        previous_operational_edges_40
        is None

        or
        canonical_edges
        !=
        previous_operational_edges_40
    )


    previous_operational_edges_40 = (
        canonical_edges
    )


    # ========================================================
    # 6. Allocation trigger
    # ========================================================

    missing_goal = any(
        goals_40[
            agent.uuv_id
        ]
        is None

        for agent in agents
    )


    periodic_trigger = (
        t
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger = (
        topology_changed
        or
        goal_invalidated
        or
        missing_goal
    )


    cooldown_satisfied = (
        t
        -
        last_allocation_step_40
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate = (
        periodic_trigger
        or
        (
            urgent_trigger
            and
            cooldown_satisfied
        )
    )


    if should_allocate:

        (
            new_goals,
            changes,
            _,
        ) = perform_learned_allocation_round_40(
            operational_graph,
            goals_40,
        )


        goals_40 = (
            new_goals
        )


        learned_goal_changes_40 += (
            changes
        )


        allocation_rounds_40 += 1


        last_allocation_step_40 = t


    # ========================================================
    # 7. Desired actions
    # ========================================================

    desired_actions = []


    for agent in agents:

        uuv_id = agent.uuv_id


        goal = goals_40[
            uuv_id
        ]


        if goal is None:

            action = ACTION_HOLD


            hold_counts_40[
                uuv_id
            ] += 1


        else:

            action = choose_greedy_action(
                agent.state,

                goal_x=
                    goal[
                        "x"
                    ],

                goal_y=
                    goal[
                        "y"
                    ],
            )


            if action is None:

                action = ACTION_HOLD


                hold_counts_40[
                    uuv_id
                ] += 1


        desired_actions.append(
            action
        )


    # ========================================================
    # 8. Predictive safety
    # ========================================================

    shield = safety_shield(
        states=[
            agent.state
            for agent in agents
        ],

        desired_actions=
            desired_actions,

        safe_separation=
            SAFE_SEPARATION,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    executed_actions = (
        shield[
            "executed_actions"
        ]
    )


    mission_min_separation_40 = min(
        mission_min_separation_40,

        shield[
            "minimum_distance"
        ],
    )


    for agent, desired, executed in zip(
        agents,
        desired_actions,
        executed_actions,
    ):

        if desired != executed:

            safety_overrides_40[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 9. Synchronized motion
    # ========================================================

    next_states = [
        predict_safety_state(
            state=
                agent.state,

            action=
                action,

            step_size=
                ROLLOUT_STEP_SIZE,
        )

        for agent, action
        in zip(
            agents,
            executed_actions,
        )
    ]


    for agent, next_state in zip(
        agents,
        next_states,
    ):

        agent.state = (
            next_state
        )


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # ========================================================
    # 10. Local sensing
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 11. Team coverage
    # ========================================================

    local_maps = {
        agent.uuv_id:
            agent.exploration_memory.observed

        for agent in agents
    }


    team_union = np.logical_or.reduce(
        list(
            local_maps.values()
        )
    )


    team_coverage = (
        int(
            team_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    coverage_history_40.append(
        float(
            team_coverage
        )
    )


    # ========================================================
    # 12. Mission incremental redundancy
    # ========================================================

    mission_new_masks = {
        uuv_id:
            (
                local_maps[
                    uuv_id
                ]
                &
                (
                    ~start_local_maps_40[
                        uuv_id
                    ]
                )
            )

        for uuv_id in local_maps
    }


    sum_new = sum(
        int(
            mask.sum()
        )

        for mask in mission_new_masks.values()
    )


    unique_new = int(
        np.logical_or.reduce(
            list(
                mission_new_masks.values()
            )
        ).sum()
    )


    redundant_new = (
        sum_new
        -
        unique_new
    )


    redundancy_ratio = (
        redundant_new
        /
        sum_new

        if sum_new > 0

        else 0.0
    )


    redundancy_history_40.append(
        float(
            redundancy_ratio
        )
    )


# ============================================================
# Flush communication queue
# ============================================================

flush_time_40 = (
    LEARNED_MISSION_STEPS_40
)


while len(
    channel_40.queue
) > 0:

    channel_40.deliver_due(
        current_time=
            flush_time_40,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },
    )


    flush_time_40 += 1


    if (
        flush_time_40
        >
        LEARNED_MISSION_STEPS_40
        +
        MAX_COMM_DELAY
        +
        10
    ):

        raise RuntimeError(
            "Combo 40 queue failed to drain."
        )


# ============================================================
# Final mission metrics
# ============================================================

final_local_maps_40 = {
    agent.uuv_id:
        agent.exploration_memory.observed

    for agent in agents
}


final_team_union_40 = np.logical_or.reduce(
    list(
        final_local_maps_40.values()
    )
)


final_team_coverage_40 = (
    int(
        final_team_union_40.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


final_new_masks_40 = {
    uuv_id:
        (
            final_local_maps_40[
                uuv_id
            ]
            &
            (
                ~start_local_maps_40[
                    uuv_id
                ]
            )
        )

    for uuv_id
    in final_local_maps_40
}


final_sum_new_40 = sum(
    int(
        mask.sum()
    )

    for mask in final_new_masks_40.values()
)


final_unique_new_40 = int(
    np.logical_or.reduce(
        list(
            final_new_masks_40.values()
        )
    ).sum()
)


final_redundant_new_40 = (
    final_sum_new_40
    -
    final_unique_new_40
)


final_redundancy_40 = (
    final_redundant_new_40
    /
    final_sum_new_40

    if final_sum_new_40 > 0

    else 0.0
)


# ============================================================
# Packet accounting
# ============================================================

packet_accounting_40 = (
    channel_40.attempted_packets
    ==
    (
        channel_40.dropped_packets
        +
        channel_40.delivered_packets
    )
)


if not packet_accounting_40:

    raise RuntimeError(
        "Combo 40 packet accounting failed."
    )


# ============================================================
# Safety validation
# ============================================================

if (
    mission_min_separation_40
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Combo 40 safety invariant failed."
    )


# ============================================================
# Report
# ============================================================

print(
    "=" * 110
)

print(
    "MultiUUV-Link — Full Learned-Frontier "
    "Cooperative Mission"
)

print(
    "=" * 110
)


print(
    f"\nMission steps                    : "
    f"{LEARNED_MISSION_STEPS_40}"
)

print(
    f"Learned model device             : "
    f"{device_38}"
)


print(
    "\nMission performance:"
)

print(
    f"  Start team coverage           : "
    f"{100 * start_team_coverage_40:.4f}%"
)

print(
    f"  Final team coverage           : "
    f"{100 * final_team_coverage_40:.4f}%"
)

print(
    f"  Coverage gain                 : "
    f"{100 * (final_team_coverage_40 - start_team_coverage_40):.4f} pp"
)

print(
    f"  Mission new local cells       : "
    f"{final_sum_new_40}"
)

print(
    f"  Unique new team cells         : "
    f"{final_unique_new_40}"
)

print(
    f"  Redundant new cells           : "
    f"{final_redundant_new_40}"
)

print(
    f"  Incremental redundancy        : "
    f"{final_redundancy_40:.4f}"
)


print(
    "\nLearned planning:"
)

print(
    f"  Allocation rounds             : "
    f"{allocation_rounds_40}"
)

print(
    f"  Learned goal changes          : "
    f"{learned_goal_changes_40}"
)

print(
    f"  Total HOLD steps              : "
    f"{sum(hold_counts_40.values())}"
)


print(
    "\nCommunication:"
)

print(
    f"  Mean geometric edges          : "
    f"{np.mean(geometric_edges_history_40):.3f}"
)

print(
    f"  Mean operational edges        : "
    f"{np.mean(operational_edges_history_40):.3f}"
)

print(
    f"  Attempted packets             : "
    f"{channel_40.attempted_packets}"
)

print(
    f"  Dropped packets               : "
    f"{channel_40.dropped_packets}"
)

print(
    f"  Delivered packets             : "
    f"{channel_40.delivered_packets}"
)

print(
    f"  Packet accounting passed      : "
    f"{packet_accounting_40}"
)


print(
    "\nSafety:"
)

print(
    f"  Safety overrides              : "
    f"{sum(safety_overrides_40.values())}"
)

print(
    f"  Mission minimum separation    : "
    f"{mission_min_separation_40:.2f} px"
)


print(
    "\nFull cooperative mission executed "
    "using learned PyTorch frontier ranking."
)


# ============================================================
# Plot 1 — Learned mission trajectories
# ============================================================

plt.figure(
    figsize=(12, 7)
)


plt.imshow(
    seabed_rgb
)


for agent in agents:

    path = np.asarray(
        agent.trajectory[
            start_traj_idx_40[
                agent.uuv_id
            ]:
        ]
    )


    plt.plot(
        path[:, 0],
        path[:, 1],
        linewidth=2.2,
        label=f"UUV-{agent.uuv_id}",
    )


    plt.scatter(
        path[0, 0],
        path[0, 1],
        marker="o",
        s=70,
    )


    plt.scatter(
        path[-1, 0],
        path[-1, 1],
        marker="x",
        s=110,
    )


plt.title(
    "MultiUUV-Link — Learned Frontier Cooperative Exploration"
)

plt.xlabel(
    "x [pixels]"
)

plt.ylabel(
    "y [pixels]"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2 — Coverage and redundancy
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    np.asarray(
        coverage_history_40
    )
    *
    100,

    label="Team coverage [%]",
)


plt.plot(
    np.asarray(
        redundancy_history_40
    )
    *
    100,

    label="Mission redundancy [%]",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Percent"
)

plt.title(
    "MultiUUV-Link — Learned Mission Performance"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 41 — Controlled Classical vs Learned Frontier Benchmark

## Objective

The learned controller has now successfully executed a complete mission.

However, a successful learned mission does not by itself tell us whether the learned frontier ranker changes mission performance relative to the classical ranker.

This combo performs the first controlled A/B comparison:

$$
\boxed{
\text{Classical Frontier Ranking}
\quad vs \quad
\text{Learned Frontier Ranking}
}
$$

Both experiments begin from the exact same Combo 24.5 checkpoint.

---

# Experimental Controls

Both arms use:

$$
T=180
$$

mission timesteps.

Both use:

```text
same checkpoint
same map
same resources
same initial UUV positions
same communication parameters
same packet-loss probability
same latency range
same blackout model
same communication random seed
same safety layer
same motion model
```

The intended planner difference is:

### Classical

$$
J(g)
=
J^{classical}(g)
$$

### Learned

$$
J(g)
=
\hat J_\theta(g)
$$

---

# Important Stochastic Qualification

The same random seed is used for both communication channels.

However, once planners choose different trajectories, their communication graphs may differ.

That can change how many random channel samples are consumed.

Therefore this is a **matched-seed controlled benchmark**, not a mathematically identical packet-by-packet communication realization.

A later higher-rigor experiment could pre-sample an exogenous communication-noise schedule.

For the current prototype, matched seed + identical starting state is sufficient.

---

# Primary Metrics

Compare:

### Final coverage

$$
C_f
$$

### Coverage gain

$$
\Delta C
$$

### Mission incremental redundancy

$$
\rho
$$

### HOLD actions

$$
N_{hold}
$$

### Safety overrides

$$
N_{safe}
$$

### Minimum pairwise separation

$$
d_{\min}
$$

### Allocation rounds

$$
N_A
$$

### Goal changes

$$
N_G
$$

---

# Learned-vs-Classical Differences

For any metric $m$ define:

$$
\Delta m
=
m_{learned}
-
m_{classical}.
$$

For coverage:

$$
\Delta C_f>0
$$

favors learned ranking.

For redundancy:

$$
\Delta\rho<0
$$

favors learned ranking.

No single metric is sufficient to claim superiority.

---

# Expected Interpretation

Because the PyTorch model was trained to imitate the classical utility, we expect broadly similar behavior.

Differences can still emerge because small ranking perturbations cause different trajectories, communication topology, subsequent observations, and future frontier sets.

This is a closed-loop system:

$$
\text{Tiny Ranking Difference}
\rightarrow
\text{Different Motion}
\rightarrow
\text{Different Observation}
\rightarrow
\text{Different Knowledge}
\rightarrow
\text{Different Future Decisions}.
$$

---

# Benchmark Source

Combo 32 already produced the classical degraded-communication mission.

Combo 40 produced the learned degraded-communication mission.

Both used:

```text
restore_checkpoint_24_5()
```

and:

```text
communication seed = 320032
```

with:

$$
T=180.
$$

Therefore this combo can construct the controlled comparison directly from those already validated mission outputs without rerunning expensive experiments.

---

# Validation

The benchmark requires:

- identical starting team coverage;
- identical mission horizon;
- identical channel seed;
- both missions passing safety;
- both missions passing packet accounting.

A successful result establishes:

$$
\boxed{
\text{Controlled Classical-vs-Learned Mission Evaluation}
}
$$

---

In [ ]:
# ============================================================
# Combo 41 — Controlled Classical vs Learned A/B Benchmark
# FULLY CORRECTED FOR NUMPY 2.x
# ============================================================


# ============================================================
# Validate prerequisite experiment compatibility
# ============================================================

same_start_coverage_41 = bool(
    np.isclose(
        start_team_coverage_32,
        start_team_coverage_40,
        atol=1e-12,
    )
)


same_horizon_41 = (
    DEGRADED_MISSION_STEPS
    ==
    LEARNED_MISSION_STEPS_40
)


same_channel_seed_41 = (
    DEGRADED_CHANNEL_SEED
    ==
    LEARNED_CHANNEL_SEED_40
)


classical_safety_pass_41 = (
    mission_min_separation_32
    >=
    SAFE_SEPARATION
)


learned_safety_pass_41 = (
    mission_min_separation_40
    >=
    SAFE_SEPARATION
)


classical_packet_pass_41 = bool(
    packet_accounting_32
)


learned_packet_pass_41 = bool(
    packet_accounting_40
)


if not same_start_coverage_41:

    raise RuntimeError(
        "Classical and learned missions did not "
        "begin from identical team coverage."
    )


if not same_horizon_41:

    raise RuntimeError(
        "Classical and learned mission horizons differ."
    )


if not same_channel_seed_41:

    raise RuntimeError(
        "Classical and learned communication seeds differ."
    )


if not (
    classical_safety_pass_41
    and
    learned_safety_pass_41
):

    raise RuntimeError(
        "One benchmark arm failed safety validation."
    )


if not (
    classical_packet_pass_41
    and
    learned_packet_pass_41
):

    raise RuntimeError(
        "One benchmark arm failed packet accounting."
    )


# ============================================================
# Classical benchmark record
# ============================================================

classical_record_41 = {
    "controller":
        "Classical",

    "start_coverage_pct":
        100.0
        *
        start_team_coverage_32,

    "final_coverage_pct":
        100.0
        *
        final_team_coverage_32,

    "coverage_gain_pp":
        100.0
        *
        (
            final_team_coverage_32
            -
            start_team_coverage_32
        ),

    "new_local_cells":
        final_sum_new_local_32,

    "unique_new_cells":
        final_unique_new_cells_32,

    "redundant_new_cells":
        final_redundant_new_cells_32,

    "incremental_redundancy":
        final_incremental_redundancy_32,

    "allocation_rounds":
        allocation_rounds_32,

    "goal_changes":
        allocation_goal_changes_32,

    "hold_steps":
        sum(
            hold_counts_32.values()
        ),

    "safety_overrides":
        sum(
            safety_overrides_32.values()
        ),

    "min_separation_px":
        mission_min_separation_32,

    "mean_geometric_edges":
        geometric_edges_mean_32,

    "mean_operational_edges":
        operational_edges_mean_32,

    "attempted_packets":
        channel_32.attempted_packets,

    "dropped_packets":
        channel_32.dropped_packets,

    "delivered_packets":
        channel_32.delivered_packets,
}


# ============================================================
# Learned benchmark record
# ============================================================

learned_record_41 = {
    "controller":
        "Learned",

    "start_coverage_pct":
        100.0
        *
        start_team_coverage_40,

    "final_coverage_pct":
        100.0
        *
        final_team_coverage_40,

    "coverage_gain_pp":
        100.0
        *
        (
            final_team_coverage_40
            -
            start_team_coverage_40
        ),

    "new_local_cells":
        final_sum_new_40,

    "unique_new_cells":
        final_unique_new_40,

    "redundant_new_cells":
        final_redundant_new_40,

    "incremental_redundancy":
        final_redundancy_40,

    "allocation_rounds":
        allocation_rounds_40,

    "goal_changes":
        learned_goal_changes_40,

    "hold_steps":
        sum(
            hold_counts_40.values()
        ),

    "safety_overrides":
        sum(
            safety_overrides_40.values()
        ),

    "min_separation_px":
        mission_min_separation_40,

    "mean_geometric_edges":
        float(
            np.mean(
                geometric_edges_history_40
            )
        ),

    "mean_operational_edges":
        float(
            np.mean(
                operational_edges_history_40
            )
        ),

    "attempted_packets":
        channel_40.attempted_packets,

    "dropped_packets":
        channel_40.dropped_packets,

    "delivered_packets":
        channel_40.delivered_packets,
}


benchmark_df_41 = pd.DataFrame(
    [
        classical_record_41,
        learned_record_41,
    ]
)


# ============================================================
# Learned minus classical differences
# ============================================================

difference_record_41 = {
    "final_coverage_pp":
        (
            learned_record_41[
                "final_coverage_pct"
            ]
            -
            classical_record_41[
                "final_coverage_pct"
            ]
        ),

    "coverage_gain_pp":
        (
            learned_record_41[
                "coverage_gain_pp"
            ]
            -
            classical_record_41[
                "coverage_gain_pp"
            ]
        ),

    "incremental_redundancy":
        (
            learned_record_41[
                "incremental_redundancy"
            ]
            -
            classical_record_41[
                "incremental_redundancy"
            ]
        ),

    "unique_new_cells":
        (
            learned_record_41[
                "unique_new_cells"
            ]
            -
            classical_record_41[
                "unique_new_cells"
            ]
        ),

    "hold_steps":
        (
            learned_record_41[
                "hold_steps"
            ]
            -
            classical_record_41[
                "hold_steps"
            ]
        ),

    "safety_overrides":
        (
            learned_record_41[
                "safety_overrides"
            ]
            -
            classical_record_41[
                "safety_overrides"
            ]
        ),

    "goal_changes":
        (
            learned_record_41[
                "goal_changes"
            ]
            -
            classical_record_41[
                "goal_changes"
            ]
        ),

    "min_separation_px":
        (
            learned_record_41[
                "min_separation_px"
            ]
            -
            classical_record_41[
                "min_separation_px"
            ]
        ),
}


difference_df_41 = pd.DataFrame(
    [
        difference_record_41
    ]
)


# ============================================================
# Coverage histories
# ============================================================

classical_coverage_curve_41 = (
    np.asarray(
        coverage_history_32,
        dtype=float,
    )
    *
    100.0
)


learned_coverage_curve_41 = (
    np.asarray(
        coverage_history_40,
        dtype=float,
    )
    *
    100.0
)


classical_redundancy_curve_41 = (
    np.asarray(
        incremental_redundancy_history_32,
        dtype=float,
    )
    *
    100.0
)


learned_redundancy_curve_41 = (
    np.asarray(
        redundancy_history_40,
        dtype=float,
    )
    *
    100.0
)


# ============================================================
# History-length validation
# ============================================================

if (
    len(
        classical_coverage_curve_41
    )
    !=
    len(
        learned_coverage_curve_41
    )
):

    raise RuntimeError(
        "Classical and learned coverage histories "
        "have different lengths."
    )


if (
    len(
        classical_redundancy_curve_41
    )
    !=
    len(
        learned_redundancy_curve_41
    )
):

    raise RuntimeError(
        "Classical and learned redundancy histories "
        "have different lengths."
    )


# ============================================================
# Area under coverage curve
#
# NumPy 2.x uses np.trapezoid()
# ============================================================

classical_coverage_auc_41 = float(
    np.trapezoid(
        classical_coverage_curve_41
    )
)


learned_coverage_auc_41 = float(
    np.trapezoid(
        learned_coverage_curve_41
    )
)


coverage_auc_difference_41 = (
    learned_coverage_auc_41
    -
    classical_coverage_auc_41
)


# ============================================================
# Time-to-coverage helper
# ============================================================

def time_to_coverage_41(
    coverage_curve,
    threshold,
):

    matches = np.where(
        coverage_curve
        >=
        threshold
    )[0]


    if len(
        matches
    ) == 0:

        return None


    return int(
        matches[
            0
        ]
    )


# ============================================================
# Coverage milestone comparison
# ============================================================

coverage_threshold_records_41 = []


for threshold in [
    80.0,
    90.0,
    95.0,
    98.0,
]:

    classical_time = (
        time_to_coverage_41(
            classical_coverage_curve_41,
            threshold,
        )
    )


    learned_time = (
        time_to_coverage_41(
            learned_coverage_curve_41,
            threshold,
        )
    )


    coverage_threshold_records_41.append(
        {
            "coverage_threshold_pct":
                threshold,

            "classical_step":
                classical_time,

            "learned_step":
                learned_time,

            "learned_minus_classical":
                (
                    None

                    if (
                        classical_time is None
                        or
                        learned_time is None
                    )

                    else (
                        learned_time
                        -
                        classical_time
                    )
                ),
        }
    )


coverage_threshold_df_41 = pd.DataFrame(
    coverage_threshold_records_41
)


# ============================================================
# Compact interpretation flags
# ============================================================

learned_final_coverage_better_41 = (
    difference_record_41[
        "final_coverage_pp"
    ]
    >
    0
)


learned_redundancy_better_41 = (
    difference_record_41[
        "incremental_redundancy"
    ]
    <
    0
)


learned_auc_better_41 = (
    coverage_auc_difference_41
    >
    0
)


# ============================================================
# Additional benchmark summary
# ============================================================

classical_final_coverage_41 = (
    classical_record_41[
        "final_coverage_pct"
    ]
)


learned_final_coverage_41 = (
    learned_record_41[
        "final_coverage_pct"
    ]
)


classical_redundancy_41 = (
    classical_record_41[
        "incremental_redundancy"
    ]
)


learned_redundancy_41 = (
    learned_record_41[
        "incremental_redundancy"
    ]
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 112
)

print(
    "MultiUUV-Link — Controlled Classical vs "
    "Learned Frontier Benchmark"
)

print(
    "=" * 112
)


print(
    "\nExperimental controls:"
)

print(
    f"  Same start coverage      : "
    f"{same_start_coverage_41}"
)

print(
    f"  Same mission horizon     : "
    f"{same_horizon_41}"
)

print(
    f"  Same communication seed  : "
    f"{same_channel_seed_41}"
)

print(
    f"  Classical safety pass    : "
    f"{classical_safety_pass_41}"
)

print(
    f"  Learned safety pass      : "
    f"{learned_safety_pass_41}"
)

print(
    f"  Classical packet pass    : "
    f"{classical_packet_pass_41}"
)

print(
    f"  Learned packet pass      : "
    f"{learned_packet_pass_41}"
)


print(
    "\nController comparison:"
)

print(
    benchmark_df_41[
        [
            "controller",
            "start_coverage_pct",
            "final_coverage_pct",
            "coverage_gain_pp",
            "incremental_redundancy",
            "unique_new_cells",
            "allocation_rounds",
            "goal_changes",
            "hold_steps",
            "safety_overrides",
            "min_separation_px",
        ]
    ].to_string(
        index=False,

        formatters={
            "start_coverage_pct":
                lambda x: f"{x:.4f}",

            "final_coverage_pct":
                lambda x: f"{x:.4f}",

            "coverage_gain_pp":
                lambda x: f"{x:.4f}",

            "incremental_redundancy":
                lambda x: f"{x:.4f}",

            "min_separation_px":
                lambda x: f"{x:.2f}",
        },
    )
)


print(
    "\nLearned minus classical:"
)

print(
    difference_df_41.to_string(
        index=False,

        formatters={
            "final_coverage_pp":
                lambda x: f"{x:+.4f}",

            "coverage_gain_pp":
                lambda x: f"{x:+.4f}",

            "incremental_redundancy":
                lambda x: f"{x:+.4f}",

            "min_separation_px":
                lambda x: f"{x:+.2f}",
        },
    )
)


print(
    "\nCoverage timing:"
)

print(
    coverage_threshold_df_41.to_string(
        index=False
    )
)


print(
    "\nCoverage-curve area:"
)

print(
    f"  Classical coverage AUC   : "
    f"{classical_coverage_auc_41:.2f}"
)

print(
    f"  Learned coverage AUC     : "
    f"{learned_coverage_auc_41:.2f}"
)

print(
    f"  Learned - classical      : "
    f"{coverage_auc_difference_41:+.2f}"
)


print(
    "\nDirectional indicators:"
)

print(
    f"  Learned higher final coverage : "
    f"{learned_final_coverage_better_41}"
)

print(
    f"  Learned lower redundancy      : "
    f"{learned_redundancy_better_41}"
)

print(
    f"  Learned higher coverage AUC   : "
    f"{learned_auc_better_41}"
)


print(
    "\nScientific interpretation:"
)

print(
    "  This is a matched-checkpoint, matched-seed "
    "closed-loop comparison."
)

print(
    "  Because trajectories can diverge, communication "
    "noise realizations are not guaranteed packet-identical."
)


print(
    "\nControlled classical-vs-learned "
    "benchmark completed."
)


# ============================================================
# Plot 1 — Coverage comparison
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    classical_coverage_curve_41,
    linewidth=2,
    label="Classical frontier ranker",
)


plt.plot(
    learned_coverage_curve_41,
    linewidth=2,
    label="Learned frontier ranker",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Team coverage [%]"
)

plt.title(
    "MultiUUV-Link — Classical vs Learned Coverage"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2 — Incremental redundancy comparison
# ============================================================

plt.figure(
    figsize=(12, 5)
)


plt.plot(
    classical_redundancy_curve_41,
    linewidth=2,
    label="Classical frontier ranker",
)


plt.plot(
    learned_redundancy_curve_41,
    linewidth=2,
    label="Learned frontier ranker",
)


plt.xlabel(
    "Mission timestep"
)

plt.ylabel(
    "Mission incremental redundancy [%]"
)

plt.title(
    "MultiUUV-Link — Classical vs Learned Redundancy"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 42 — Multi-Seed Classical vs Learned Robustness Benchmark

## Objective

Combo 41 compared the classical and learned frontier rankers under one matched communication seed.

The result showed:

- nearly identical final coverage;
- lower redundancy for the learned planner;
- slower coverage progression for the learned planner;
- more learned goal changes and safety interventions.

However, one stochastic realization is insufficient to determine whether those differences are robust.

This combo therefore runs a paired multi-seed benchmark:

$$
\boxed{
\text{Classical}
\quad vs \quad
\text{Learned}
}
$$

across several degraded-communication realizations.

---

# Paired Experimental Design

For each seed:

$$
s_k
$$

run two missions:

```text
Classical planner + seed s_k
Learned planner   + seed s_k
```

Both begin from:

```python
restore_checkpoint_24_5()
```

and use identical:

- map;
- resources;
- starting states;
- mission horizon;
- communication parameters;
- safety parameters;
- seed value.

The only intended controller difference is frontier ranking.

---

# Seeds

Use five communication seeds:

$$
S=
\{
420001,
420002,
420003,
420004,
420005
\}.
$$

Therefore:

$$
N_{runs}=10.
$$

---

# Mission Horizon

Each mission runs:

$$
T=180
$$

timesteps.

---

# Metrics

For every run record:

### Final coverage

$$
C_f
$$

### Coverage gain

$$
\Delta C
$$

### Coverage AUC

$$
A_C
=
\int C(t)\,dt
$$

### Incremental redundancy

$$
\rho
$$

### HOLD steps

$$
N_{hold}
$$

### Safety overrides

$$
N_{safe}
$$

### Goal changes

$$
N_G
$$

### Minimum separation

$$
d_{\min}
$$

---

# Paired Differences

For every seed compute:

$$
\Delta m_k
=
m_k^{learned}
-
m_k^{classical}.
$$

Important signs:

### Coverage

$$
\Delta C_f>0
$$

favors learned.

### Coverage AUC

$$
\Delta A_C>0
$$

favors learned.

### Redundancy

$$
\Delta\rho<0
$$

favors learned.

### Safety overrides

Lower is operationally preferable, although both controllers must still satisfy the hard separation constraint.

---

# Summary Statistics

For each metric compute:

$$
\bar x
$$

and:

$$
\sigma_x.
$$

For paired differences also report:

- mean difference;
- median difference;
- number of seeds where learned is better.

---

# Scientific Interpretation

This remains a matched-seed stochastic benchmark.

Once controller trajectories diverge, communication topology may also diverge, so the two arms are not guaranteed to consume identical random packet events.

Therefore the experiment evaluates:

$$
\boxed{
\text{Closed-Loop Robustness Across Matched Random Seeds}
}
$$

rather than packet-identical counterfactual communication.

---

# Progress Visualization

Because this combo runs ten complete missions, a progress bar is required.

It should display:

```text
seed
controller
completed runs / total runs
```

so execution progress remains visible.

---

# Final State

After all experiments finish:

```python
restore_checkpoint_24_5()
```

must be called once more.

Thus the benchmark should leave the notebook in the reusable checkpoint state rather than in the state of the final simulation.

---

# Expected Result

This combo should answer:

> Is the lower learned redundancy seen in Combo 41 repeatable across multiple stochastic communication realizations?

and:

> Is the classical planner's faster coverage progression repeatable?

A successful result establishes:

$$
\boxed{
\text{Multi-Seed Robustness Evaluation}
}
$$

---

In [ ]:
# ============================================================
# Combo 42 — Multi-Seed Classical vs Learned Benchmark
# ============================================================

import time


# ============================================================
# Configuration
# ============================================================

BENCHMARK_SEEDS_42 = [
    420001,
    420002,
    420003,
    420004,
    420005,
]

BENCHMARK_STEPS_42 = 180


# ============================================================
# Classical mission runner
# ============================================================

def run_classical_benchmark_mission_42(
    channel_seed,
):
    """
    Restore the common checkpoint and execute one complete
    classical degraded-communication mission.
    """

    restore_checkpoint_24_5()


    rng = np.random.default_rng(
        channel_seed
    )


    # --------------------------------------------------------
    # Known-map consistency
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = agent.uuv_id

        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


    # --------------------------------------------------------
    # Start snapshots
    # --------------------------------------------------------

    start_maps = {
        agent.uuv_id:
            agent.exploration_memory
            .observed.copy()

        for agent in agents
    }


    start_union = np.logical_or.reduce(
        list(
            start_maps.values()
        )
    )


    start_coverage = (
        int(
            start_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    # --------------------------------------------------------
    # Communication channel
    # --------------------------------------------------------

    channel = DegradedCommunicationChannel(
        uuv_ids=[
            agent.uuv_id
            for agent in agents
        ],

        communication_range=
            COMMUNICATION_RANGE,

        packet_loss_probability=
            PACKET_LOSS_PROBABILITY,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,

        blackout_start_probability=
            BLACKOUT_START_PROBABILITY,

        blackout_min_duration=
            BLACKOUT_MIN_DURATION,

        blackout_max_duration=
            BLACKOUT_MAX_DURATION,

        rng=
            rng,
    )


    # --------------------------------------------------------
    # Planner state
    # --------------------------------------------------------

    goals = {
        agent.uuv_id: None
        for agent in agents
    }


    last_allocation_step = (
        -ALLOCATION_COOLDOWN
    )


    previous_edges = None


    allocation_rounds = 0
    goal_changes = 0


    hold_counts = {
        agent.uuv_id: 0
        for agent in agents
    }


    safety_overrides = {
        agent.uuv_id: 0
        for agent in agents
    }


    minimum_separation = np.inf


    coverage_curve = []


    # ========================================================
    # Mission
    # ========================================================

    for t in range(
        BENCHMARK_STEPS_42
    ):

        # ----------------------------------------------------
        # Deliver delayed knowledge
        # ----------------------------------------------------

        channel.deliver_due(
            current_time=
                t,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )


        # ----------------------------------------------------
        # Local knowledge is always known locally
        # ----------------------------------------------------

        for agent in agents:

            uuv_id = agent.uuv_id

            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


        # ----------------------------------------------------
        # Communication
        # ----------------------------------------------------

        positions = {
            agent.uuv_id:
                (
                    agent.state.x,
                    agent.state.y,
                )

            for agent in agents
        }


        transmission = channel.transmit_round(
            current_time=
                t,

            positions=
                positions,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )


        operational_graph = transmission[
            "operational_graph"
        ]


        # ----------------------------------------------------
        # Goal validity
        # ----------------------------------------------------

        goal_invalidated = False


        for agent in agents:

            uuv_id = agent.uuv_id

            goal = goals[
                uuv_id
            ]


            if goal is None:
                continue


            if not goal_is_still_useful(
                agent,
                goal,
            ):

                goals[
                    uuv_id
                ] = None

                goal_invalidated = True


        # ----------------------------------------------------
        # Topology
        # ----------------------------------------------------

        canonical_edges = frozenset(
            tuple(
                sorted(
                    edge
                )
            )

            for edge
            in operational_graph.edges()
        )


        topology_changed = (
            previous_edges is None
            or
            canonical_edges
            !=
            previous_edges
        )


        previous_edges = (
            canonical_edges
        )


        # ----------------------------------------------------
        # Allocation trigger
        # ----------------------------------------------------

        missing_goal = any(
            goals[
                agent.uuv_id
            ]
            is None

            for agent in agents
        )


        periodic_trigger = (
            t
            %
            ALLOCATION_INTERVAL
            ==
            0
        )


        urgent_trigger = (
            topology_changed
            or
            goal_invalidated
            or
            missing_goal
        )


        cooldown_satisfied = (
            t
            -
            last_allocation_step
            >=
            ALLOCATION_COOLDOWN
        )


        should_allocate = (
            periodic_trigger
            or
            (
                urgent_trigger
                and
                cooldown_satisfied
            )
        )


        if should_allocate:

            (
                new_goals,
                changes,
                _,
                _,
                _,
            ) = perform_stable_allocation_round(
                operational_graph,
                goals,
            )


            goals = new_goals

            goal_changes += (
                changes
            )

            allocation_rounds += 1

            last_allocation_step = t


        # ----------------------------------------------------
        # Desired actions
        # ----------------------------------------------------

        desired_actions = []


        for agent in agents:

            uuv_id = agent.uuv_id

            goal = goals[
                uuv_id
            ]


            if goal is None:

                action = ACTION_HOLD

                hold_counts[
                    uuv_id
                ] += 1


            else:

                action = choose_greedy_action(
                    agent.state,

                    goal_x=
                        goal[
                            "x"
                        ],

                    goal_y=
                        goal[
                            "y"
                        ],
                )


                if action is None:

                    action = ACTION_HOLD

                    hold_counts[
                        uuv_id
                    ] += 1


            desired_actions.append(
                action
            )


        # ----------------------------------------------------
        # Safety
        # ----------------------------------------------------

        shield = safety_shield(
            states=[
                agent.state
                for agent in agents
            ],

            desired_actions=
                desired_actions,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )


        executed_actions = shield[
            "executed_actions"
        ]


        minimum_separation = min(
            minimum_separation,
            shield[
                "minimum_distance"
            ],
        )


        for (
            agent,
            desired,
            executed,
        ) in zip(
            agents,
            desired_actions,
            executed_actions,
        ):

            if desired != executed:

                safety_overrides[
                    agent.uuv_id
                ] += 1


        # ----------------------------------------------------
        # Motion
        # ----------------------------------------------------

        next_states = [
            predict_safety_state(
                state=
                    agent.state,

                action=
                    action,

                step_size=
                    ROLLOUT_STEP_SIZE,
            )

            for agent, action
            in zip(
                agents,
                executed_actions,
            )
        ]


        for agent, state in zip(
            agents,
            next_states,
        ):

            agent.state = state

            agent.trajectory.append(
                (
                    state.x,
                    state.y,
                )
            )


        # ----------------------------------------------------
        # Sensing
        # ----------------------------------------------------

        for agent in agents:

            uuv_id = agent.uuv_id


            agent.exploration_memory.update_from_fov(
                agent.state
            )


            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


            visible_resources = (
                discover_visible_resources(
                    agent.state,
                    resources,
                )
            )


            agent.resource_memory.update(
                visible_resources
            )


        # ----------------------------------------------------
        # Coverage
        # ----------------------------------------------------

        union = np.logical_or.reduce(
            [
                agent.exploration_memory.observed
                for agent in agents
            ]
        )


        coverage_curve.append(
            float(
                union.sum()
                /
                (
                    WIDTH
                    *
                    HEIGHT
                )
            )
        )


    # ========================================================
    # Final metrics
    # ========================================================

    final_maps = {
        agent.uuv_id:
            agent.exploration_memory.observed

        for agent in agents
    }


    final_union = np.logical_or.reduce(
        list(
            final_maps.values()
        )
    )


    final_coverage = (
        int(
            final_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    new_masks = {
        uuv_id:
            (
                final_maps[
                    uuv_id
                ]
                &
                (
                    ~start_maps[
                        uuv_id
                    ]
                )
            )

        for uuv_id
        in final_maps
    }


    sum_new = sum(
        int(
            mask.sum()
        )

        for mask
        in new_masks.values()
    )


    unique_new = int(
        np.logical_or.reduce(
            list(
                new_masks.values()
            )
        ).sum()
    )


    redundant_new = (
        sum_new
        -
        unique_new
    )


    redundancy = (
        redundant_new
        /
        sum_new

        if sum_new > 0

        else 0.0
    )


    coverage_auc = float(
        np.trapezoid(
            np.asarray(
                coverage_curve
            )
            *
            100.0
        )
    )


    if (
        minimum_separation
        <
        SAFE_SEPARATION
    ):

        raise RuntimeError(
            "Classical benchmark run violated safety."
        )


    return {
        "controller":
            "Classical",

        "seed":
            channel_seed,

        "start_coverage_pct":
            start_coverage
            *
            100.0,

        "final_coverage_pct":
            final_coverage
            *
            100.0,

        "coverage_gain_pp":
            (
                final_coverage
                -
                start_coverage
            )
            *
            100.0,

        "coverage_auc":
            coverage_auc,

        "incremental_redundancy":
            redundancy,

        "unique_new_cells":
            unique_new,

        "hold_steps":
            sum(
                hold_counts.values()
            ),

        "safety_overrides":
            sum(
                safety_overrides.values()
            ),

        "goal_changes":
            goal_changes,

        "allocation_rounds":
            allocation_rounds,

        "min_separation_px":
            minimum_separation,
    }


# ============================================================
# Learned mission runner
# ============================================================

def run_learned_benchmark_mission_42(
    channel_seed,
):
    """
    Same experiment as classical runner, but task allocation
    uses the trained PyTorch learned frontier ranker.
    """

    restore_checkpoint_24_5()

    frontier_model_38.eval()


    rng = np.random.default_rng(
        channel_seed
    )


    for agent in agents:

        uuv_id = agent.uuv_id

        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


    start_maps = {
        agent.uuv_id:
            agent.exploration_memory.observed.copy()

        for agent in agents
    }


    start_union = np.logical_or.reduce(
        list(
            start_maps.values()
        )
    )


    start_coverage = (
        int(
            start_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    channel = DegradedCommunicationChannel(
        uuv_ids=[
            agent.uuv_id
            for agent in agents
        ],

        communication_range=
            COMMUNICATION_RANGE,

        packet_loss_probability=
            PACKET_LOSS_PROBABILITY,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,

        blackout_start_probability=
            BLACKOUT_START_PROBABILITY,

        blackout_min_duration=
            BLACKOUT_MIN_DURATION,

        blackout_max_duration=
            BLACKOUT_MAX_DURATION,

        rng=
            rng,
    )


    goals = {
        agent.uuv_id: None
        for agent in agents
    }


    last_allocation_step = (
        -ALLOCATION_COOLDOWN
    )


    previous_edges = None

    allocation_rounds = 0
    goal_changes = 0


    hold_counts = {
        agent.uuv_id: 0
        for agent in agents
    }


    safety_overrides = {
        agent.uuv_id: 0
        for agent in agents
    }


    minimum_separation = np.inf

    coverage_curve = []


    # ========================================================
    # Mission
    # ========================================================

    for t in range(
        BENCHMARK_STEPS_42
    ):

        channel.deliver_due(
            current_time=
                t,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )


        for agent in agents:

            uuv_id = agent.uuv_id

            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


        positions = {
            agent.uuv_id:
                (
                    agent.state.x,
                    agent.state.y,
                )

            for agent in agents
        }


        transmission = channel.transmit_round(
            current_time=
                t,

            positions=
                positions,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )


        operational_graph = transmission[
            "operational_graph"
        ]


        # ----------------------------------------------------
        # Goal validity
        # ----------------------------------------------------

        goal_invalidated = False


        for agent in agents:

            uuv_id = agent.uuv_id

            goal = goals[
                uuv_id
            ]


            if goal is None:
                continue


            if not goal_is_still_useful(
                agent,
                goal,
            ):

                goals[
                    uuv_id
                ] = None

                goal_invalidated = True


        # ----------------------------------------------------
        # Topology
        # ----------------------------------------------------

        canonical_edges = frozenset(
            tuple(
                sorted(
                    edge
                )
            )

            for edge
            in operational_graph.edges()
        )


        topology_changed = (
            previous_edges is None
            or
            canonical_edges
            !=
            previous_edges
        )


        previous_edges = canonical_edges


        # ----------------------------------------------------
        # Allocation
        # ----------------------------------------------------

        missing_goal = any(
            goals[
                agent.uuv_id
            ]
            is None

            for agent in agents
        )


        periodic_trigger = (
            t
            %
            ALLOCATION_INTERVAL
            ==
            0
        )


        urgent_trigger = (
            topology_changed
            or
            goal_invalidated
            or
            missing_goal
        )


        cooldown_satisfied = (
            t
            -
            last_allocation_step
            >=
            ALLOCATION_COOLDOWN
        )


        should_allocate = (
            periodic_trigger
            or
            (
                urgent_trigger
                and
                cooldown_satisfied
            )
        )


        if should_allocate:

            (
                new_goals,
                changes,
                _,
            ) = perform_learned_allocation_round_40(
                operational_graph,
                goals,
            )


            goals = new_goals

            goal_changes += (
                changes
            )

            allocation_rounds += 1

            last_allocation_step = t


        # ----------------------------------------------------
        # Desired actions
        # ----------------------------------------------------

        desired_actions = []


        for agent in agents:

            uuv_id = agent.uuv_id

            goal = goals[
                uuv_id
            ]


            if goal is None:

                action = ACTION_HOLD

                hold_counts[
                    uuv_id
                ] += 1


            else:

                action = choose_greedy_action(
                    agent.state,

                    goal_x=
                        goal[
                            "x"
                        ],

                    goal_y=
                        goal[
                            "y"
                        ],
                )


                if action is None:

                    action = ACTION_HOLD

                    hold_counts[
                        uuv_id
                    ] += 1


            desired_actions.append(
                action
            )


        # ----------------------------------------------------
        # Safety
        # ----------------------------------------------------

        shield = safety_shield(
            states=[
                agent.state
                for agent in agents
            ],

            desired_actions=
                desired_actions,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )


        executed_actions = shield[
            "executed_actions"
        ]


        minimum_separation = min(
            minimum_separation,
            shield[
                "minimum_distance"
            ],
        )


        for (
            agent,
            desired,
            executed,
        ) in zip(
            agents,
            desired_actions,
            executed_actions,
        ):

            if desired != executed:

                safety_overrides[
                    agent.uuv_id
                ] += 1


        # ----------------------------------------------------
        # Motion
        # ----------------------------------------------------

        next_states = [
            predict_safety_state(
                state=
                    agent.state,

                action=
                    action,

                step_size=
                    ROLLOUT_STEP_SIZE,
            )

            for agent, action
            in zip(
                agents,
                executed_actions,
            )
        ]


        for agent, state in zip(
            agents,
            next_states,
        ):

            agent.state = state

            agent.trajectory.append(
                (
                    state.x,
                    state.y,
                )
            )


        # ----------------------------------------------------
        # Sensing
        # ----------------------------------------------------

        for agent in agents:

            uuv_id = agent.uuv_id


            agent.exploration_memory.update_from_fov(
                agent.state
            )


            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


            visible_resources = (
                discover_visible_resources(
                    agent.state,
                    resources,
                )
            )


            agent.resource_memory.update(
                visible_resources
            )


        union = np.logical_or.reduce(
            [
                agent.exploration_memory.observed
                for agent in agents
            ]
        )


        coverage_curve.append(
            float(
                union.sum()
                /
                (
                    WIDTH
                    *
                    HEIGHT
                )
            )
        )


    # ========================================================
    # Final metrics
    # ========================================================

    final_maps = {
        agent.uuv_id:
            agent.exploration_memory.observed

        for agent in agents
    }


    final_union = np.logical_or.reduce(
        list(
            final_maps.values()
        )
    )


    final_coverage = (
        int(
            final_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    new_masks = {
        uuv_id:
            (
                final_maps[
                    uuv_id
                ]
                &
                (
                    ~start_maps[
                        uuv_id
                    ]
                )
            )

        for uuv_id
        in final_maps
    }


    sum_new = sum(
        int(
            mask.sum()
        )

        for mask
        in new_masks.values()
    )


    unique_new = int(
        np.logical_or.reduce(
            list(
                new_masks.values()
            )
        ).sum()
    )


    redundant_new = (
        sum_new
        -
        unique_new
    )


    redundancy = (
        redundant_new
        /
        sum_new

        if sum_new > 0

        else 0.0
    )


    coverage_auc = float(
        np.trapezoid(
            np.asarray(
                coverage_curve
            )
            *
            100.0
        )
    )


    if (
        minimum_separation
        <
        SAFE_SEPARATION
    ):

        raise RuntimeError(
            "Learned benchmark run violated safety."
        )


    return {
        "controller":
            "Learned",

        "seed":
            channel_seed,

        "start_coverage_pct":
            start_coverage
            *
            100.0,

        "final_coverage_pct":
            final_coverage
            *
            100.0,

        "coverage_gain_pp":
            (
                final_coverage
                -
                start_coverage
            )
            *
            100.0,

        "coverage_auc":
            coverage_auc,

        "incremental_redundancy":
            redundancy,

        "unique_new_cells":
            unique_new,

        "hold_steps":
            sum(
                hold_counts.values()
            ),

        "safety_overrides":
            sum(
                safety_overrides.values()
            ),

        "goal_changes":
            goal_changes,

        "allocation_rounds":
            allocation_rounds,

        "min_separation_px":
            minimum_separation,
    }


# ============================================================
# Execute paired benchmark
# ============================================================

benchmark_records_42 = []


try:

    from tqdm.auto import tqdm

    progress_42 = tqdm(
        total=
            len(
                BENCHMARK_SEEDS_42
            )
            *
            2,

        desc=
            "Multi-seed A/B benchmark",
    )

    use_progress_42 = True


except Exception:

    progress_42 = None

    use_progress_42 = False


benchmark_start_time_42 = time.time()


for seed in BENCHMARK_SEEDS_42:

    # --------------------------------------------------------
    # Classical arm
    # --------------------------------------------------------

    classical_result = (
        run_classical_benchmark_mission_42(
            seed
        )
    )


    benchmark_records_42.append(
        classical_result
    )


    if use_progress_42:

        progress_42.set_postfix(
            seed=seed,
            controller="Classical",
        )

        progress_42.update(
            1
        )


    # --------------------------------------------------------
    # Learned arm
    # --------------------------------------------------------

    learned_result = (
        run_learned_benchmark_mission_42(
            seed
        )
    )


    benchmark_records_42.append(
        learned_result
    )


    if use_progress_42:

        progress_42.set_postfix(
            seed=seed,
            controller="Learned",
        )

        progress_42.update(
            1
        )


if use_progress_42:

    progress_42.close()


benchmark_elapsed_42 = (
    time.time()
    -
    benchmark_start_time_42
)


# ============================================================
# Restore checkpoint after benchmark
# ============================================================

restore_checkpoint_24_5()


# ============================================================
# Results table
# ============================================================

benchmark_runs_df_42 = pd.DataFrame(
    benchmark_records_42
)


# ============================================================
# Controller-level statistics
# ============================================================

metric_columns_42 = [
    "final_coverage_pct",
    "coverage_gain_pp",
    "coverage_auc",
    "incremental_redundancy",
    "unique_new_cells",
    "hold_steps",
    "safety_overrides",
    "goal_changes",
    "min_separation_px",
]


controller_summary_42 = (
    benchmark_runs_df_42
    .groupby(
        "controller"
    )[
        metric_columns_42
    ]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)


# ============================================================
# Paired learned-minus-classical table
# ============================================================

classical_by_seed_42 = (
    benchmark_runs_df_42[
        benchmark_runs_df_42[
            "controller"
        ]
        ==
        "Classical"
    ]
    .set_index(
        "seed"
    )
)


learned_by_seed_42 = (
    benchmark_runs_df_42[
        benchmark_runs_df_42[
            "controller"
        ]
        ==
        "Learned"
    ]
    .set_index(
        "seed"
    )
)


paired_records_42 = []


for seed in BENCHMARK_SEEDS_42:

    classical_row = (
        classical_by_seed_42.loc[
            seed
        ]
    )


    learned_row = (
        learned_by_seed_42.loc[
            seed
        ]
    )


    paired_records_42.append(
        {
            "seed":
                seed,

            "final_coverage_diff_pp":
                (
                    learned_row[
                        "final_coverage_pct"
                    ]
                    -
                    classical_row[
                        "final_coverage_pct"
                    ]
                ),

            "coverage_auc_diff":
                (
                    learned_row[
                        "coverage_auc"
                    ]
                    -
                    classical_row[
                        "coverage_auc"
                    ]
                ),

            "redundancy_diff":
                (
                    learned_row[
                        "incremental_redundancy"
                    ]
                    -
                    classical_row[
                        "incremental_redundancy"
                    ]
                ),

            "hold_diff":
                (
                    learned_row[
                        "hold_steps"
                    ]
                    -
                    classical_row[
                        "hold_steps"
                    ]
                ),

            "safety_override_diff":
                (
                    learned_row[
                        "safety_overrides"
                    ]
                    -
                    classical_row[
                        "safety_overrides"
                    ]
                ),

            "goal_change_diff":
                (
                    learned_row[
                        "goal_changes"
                    ]
                    -
                    classical_row[
                        "goal_changes"
                    ]
                ),
        }
    )


paired_df_42 = pd.DataFrame(
    paired_records_42
)


# ============================================================
# Paired summary
# ============================================================

paired_summary_42 = {
    "mean_final_coverage_diff_pp":
        float(
            paired_df_42[
                "final_coverage_diff_pp"
            ].mean()
        ),

    "mean_coverage_auc_diff":
        float(
            paired_df_42[
                "coverage_auc_diff"
            ].mean()
        ),

    "mean_redundancy_diff":
        float(
            paired_df_42[
                "redundancy_diff"
            ].mean()
        ),

    "median_redundancy_diff":
        float(
            paired_df_42[
                "redundancy_diff"
            ].median()
        ),

    "learned_lower_redundancy_seeds":
        int(
            (
                paired_df_42[
                    "redundancy_diff"
                ]
                <
                0
            ).sum()
        ),

    "learned_higher_coverage_auc_seeds":
        int(
            (
                paired_df_42[
                    "coverage_auc_diff"
                ]
                >
                0
            ).sum()
        ),

    "learned_higher_final_coverage_seeds":
        int(
            (
                paired_df_42[
                    "final_coverage_diff_pp"
                ]
                >
                0
            ).sum()
        ),
}


# ============================================================
# Console report
# ============================================================

print(
    "=" * 114
)

print(
    "MultiUUV-Link — Multi-Seed Classical vs "
    "Learned Robustness Benchmark"
)

print(
    "=" * 114
)


print(
    f"\nSeeds tested              : "
    f"{len(BENCHMARK_SEEDS_42)}"
)

print(
    f"Total missions executed   : "
    f"{len(benchmark_records_42)}"
)

print(
    f"Mission steps per run     : "
    f"{BENCHMARK_STEPS_42}"
)

print(
    f"Benchmark runtime         : "
    f"{benchmark_elapsed_42:.2f} s"
)


print(
    "\nPer-run results:"
)

print(
    benchmark_runs_df_42[
        [
            "seed",
            "controller",
            "final_coverage_pct",
            "coverage_auc",
            "incremental_redundancy",
            "hold_steps",
            "safety_overrides",
            "goal_changes",
            "min_separation_px",
        ]
    ].to_string(
        index=False,

        formatters={
            "final_coverage_pct":
                lambda x: f"{x:.4f}",

            "coverage_auc":
                lambda x: f"{x:.2f}",

            "incremental_redundancy":
                lambda x: f"{x:.4f}",

            "min_separation_px":
                lambda x: f"{x:.2f}",
        },
    )
)


print(
    "\nLearned minus classical by seed:"
)

print(
    paired_df_42.to_string(
        index=False,

        formatters={
            "final_coverage_diff_pp":
                lambda x: f"{x:+.4f}",

            "coverage_auc_diff":
                lambda x: f"{x:+.2f}",

            "redundancy_diff":
                lambda x: f"{x:+.4f}",
        },
    )
)


print(
    "\nPaired robustness summary:"
)

print(
    f"  Mean final coverage difference : "
    f"{paired_summary_42['mean_final_coverage_diff_pp']:+.4f} pp"
)

print(
    f"  Mean coverage-AUC difference   : "
    f"{paired_summary_42['mean_coverage_auc_diff']:+.2f}"
)

print(
    f"  Mean redundancy difference     : "
    f"{paired_summary_42['mean_redundancy_diff']:+.4f}"
)

print(
    f"  Median redundancy difference   : "
    f"{paired_summary_42['median_redundancy_diff']:+.4f}"
)

print(
    f"  Learned lower redundancy       : "
    f"{paired_summary_42['learned_lower_redundancy_seeds']}"
    f"/{len(BENCHMARK_SEEDS_42)} seeds"
)

print(
    f"  Learned higher coverage AUC    : "
    f"{paired_summary_42['learned_higher_coverage_auc_seeds']}"
    f"/{len(BENCHMARK_SEEDS_42)} seeds"
)

print(
    f"  Learned higher final coverage  : "
    f"{paired_summary_42['learned_higher_final_coverage_seeds']}"
    f"/{len(BENCHMARK_SEEDS_42)} seeds"
)


print(
    "\nMulti-seed robustness benchmark completed."
)


# ============================================================
# Plot 1 — Final coverage by seed
# ============================================================

plt.figure(
    figsize=(11, 5)
)


for controller in [
    "Classical",
    "Learned",
]:

    subset = benchmark_runs_df_42[
        benchmark_runs_df_42[
            "controller"
        ]
        ==
        controller
    ]


    plt.plot(
        subset[
            "seed"
        ].astype(
            str
        ),

        subset[
            "final_coverage_pct"
        ],

        marker="o",

        label=
            controller,
    )


plt.xlabel(
    "Communication seed"
)

plt.ylabel(
    "Final team coverage [%]"
)

plt.title(
    "MultiUUV-Link — Multi-Seed Final Coverage"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2 — Redundancy by seed
# ============================================================

plt.figure(
    figsize=(11, 5)
)


for controller in [
    "Classical",
    "Learned",
]:

    subset = benchmark_runs_df_42[
        benchmark_runs_df_42[
            "controller"
        ]
        ==
        controller
    ]


    plt.plot(
        subset[
            "seed"
        ].astype(
            str
        ),

        subset[
            "incremental_redundancy"
        ]
        *
        100.0,

        marker="o",

        label=
            controller,
    )


plt.xlabel(
    "Communication seed"
)

plt.ylabel(
    "Mission incremental redundancy [%]"
)

plt.title(
    "MultiUUV-Link — Multi-Seed Exploration Redundancy"
)

plt.legend()

plt.tight_layout()

plt.show()

# Combo 43 — Confidence-Gated Classical + Learned Frontier Ranking

## Objective

The multi-seed benchmark showed that the learned frontier-value network is accurate as an approximation model but does not consistently improve closed-loop mission performance.

Therefore the learned model should not blindly replace the classical planner.

Instead, introduce a hybrid decision architecture:

$$
\boxed{
\text{Classical Planner}
+
\text{Learned Advisor}
+
\text{Confidence Gate}
}
$$

The learned model is used when its ranking agrees strongly with the classical planner.

When learned confidence is weak or disagreement is substantial, the system falls back to the classical decision.

---

# Why Hybridize?

Combo 38 demonstrated excellent supervised accuracy:

$$
R^2\approx0.9999
$$

and high ranking agreement.

But Combo 42 demonstrated:

$$
\text{small ranking perturbation}
\rightarrow
\text{different trajectories}
\rightarrow
\text{different communication}
\rightarrow
\text{different future map state}.
$$

Therefore prediction accuracy alone is not enough.

The hybrid architecture explicitly limits closed-loop divergence.

---

# Candidate Utilities

For every candidate $g$ compute both:

$$
J_C(g)
$$

and:

$$
J_L(g).
$$

where:

- $J_C$ = classical frontier utility;
- $J_L$ = learned PyTorch utility.

---

# Learned Ranking Margin

Let:

$$
J_L^{(1)}
$$

be the highest learned utility and:

$$
J_L^{(2)}
$$

the second-highest.

Define learned ranking margin:

$$
M_L
=
J_L^{(1)}
-
J_L^{(2)}.
$$

A small margin means the neural model itself sees several candidates as nearly equivalent.

---

# Classical-Learned Agreement

Let:

$$
g_C
=
\arg\max_g J_C(g)
$$

and:

$$
g_L
=
\arg\max_g J_L(g).
$$

If:

$$
g_C=g_L
$$

the learned recommendation is accepted.

If the goals differ, learned control is allowed only if:

$$
M_L\ge M_{min}
$$

and the classical penalty for choosing the learned candidate is small:

$$
J_C(g_C)-J_C(g_L)\le\Delta J_{max}.
$$

Use:

$$
M_{min}=0.03
$$

and:

$$
\Delta J_{max}=0.02.
$$

---

# Decision Rule

The hybrid planner uses:

$$
g_H=
\begin{cases}
g_L,
&
g_L=g_C
\\
g_L,
&
M_L\ge M_{min}
\land
[J_C(g_C)-J_C(g_L)]\le\Delta J_{max}
\\
g_C,
&
\text{otherwise}
\end{cases}
$$

This means the neural model is allowed to influence decisions but cannot pull the planner far away from a strong classical optimum without sufficient confidence.

---

# Why Not Average the Scores?

A simple score blend:

$$
\alpha J_C+(1-\alpha)J_L
$$

would hide disagreement rather than diagnose it.

The confidence gate provides explicit interpretable behavior:

```text
AGREE
→ learned accepted

DISAGREE but learned confident + classical cost small
→ learned accepted

DISAGREE substantially
→ classical fallback
```

---

# Validation Dataset

Use all current frontier candidates from all UUVs.

Also evaluate the complete Combo 37 dataset to estimate how frequently the gate chooses:

- learned agreement;
- learned override;
- classical fallback.

---

# Required Metrics

Report:

### Exact classical-learned agreement

$$
A_{CL}
$$

### Hybrid equals classical

$$
A_{HC}
$$

### Hybrid equals learned

$$
A_{HL}
$$

### Learned accepted through agreement

$$
N_{agree}
$$

### Learned accepted through confident override

$$
N_{override}
$$

### Classical fallback

$$
N_{fallback}
$$

### Mean classical regret of hybrid decisions

For each state:

$$
R_H
=
J_C(g_C)-J_C(g_H).
$$

Ideal:

$$
R_H\approx0.
$$

---

# Interpretation

The aim is not to prove that learning wins.

The aim is to build a learned autonomy component that is:

- useful;
- interpretable;
- bounded by classical knowledge;
- robust to learned ranking errors.

A successful result establishes:

$$
\boxed{
\text{Confidence-Gated Hybrid Autonomy}
}
$$

The next step will benchmark this hybrid policy against both classical and fully learned planners.

---

In [ ]:
# ============================================================
# Combo 43 — Confidence-Gated Hybrid Frontier Ranker
# ============================================================

frontier_model_38.eval()


# ============================================================
# Configuration
# ============================================================

HYBRID_LEARNED_MARGIN_MIN_43 = 0.03

HYBRID_CLASSICAL_REGRET_MAX_43 = 0.02


# ============================================================
# Preserve live state
# ============================================================

live_signature_before_43 = (
    get_swarm_state_signature()
)


live_known_maps_before_43 = {
    uuv_id:
        known_map.copy()

    for uuv_id, known_map
    in known_exploration_maps.items()
}


# ============================================================
# Hybrid selection helper
# ============================================================

def choose_hybrid_frontier_43(
    classical_scores,
    learned_scores,
):
    """
    Confidence-gated frontier selection.

    Both input lists must refer to the SAME candidate ordering.
    """

    if (
        len(classical_scores)
        !=
        len(learned_scores)
    ):

        raise RuntimeError(
            "Classical and learned candidate lists "
            "have different lengths."
        )


    if not classical_scores:

        return {
            "selected":
                None,

            "decision_source":
                "NO_TASK",

            "learned_margin":
                np.nan,

            "classical_regret":
                0.0,

            "classical_goal":
                None,

            "learned_goal":
                None,
        }


    # --------------------------------------------------------
    # Classical best
    # --------------------------------------------------------

    classical_best_index = int(
        np.argmax(
            [
                record[
                    "utility"
                ]

                for record
                in classical_scores
            ]
        )
    )


    classical_best = (
        classical_scores[
            classical_best_index
        ]
    )


    # --------------------------------------------------------
    # Learned ordering
    # --------------------------------------------------------

    learned_utilities = np.asarray(
        [
            record[
                "learned_utility"
            ]

            for record
            in learned_scores
        ],

        dtype=float,
    )


    learned_order = np.argsort(
        learned_utilities
    )[::-1]


    learned_best_index = int(
        learned_order[
            0
        ]
    )


    learned_best = (
        learned_scores[
            learned_best_index
        ]
    )


    # --------------------------------------------------------
    # Learned margin
    # --------------------------------------------------------

    if len(
        learned_order
    ) >= 2:

        learned_margin = float(
            learned_utilities[
                learned_order[
                    0
                ]
            ]
            -
            learned_utilities[
                learned_order[
                    1
                ]
            ]
        )

    else:

        learned_margin = np.inf


    classical_goal = (
        int(
            classical_best[
                "x"
            ]
        ),
        int(
            classical_best[
                "y"
            ]
        ),
    )


    learned_goal = (
        int(
            learned_best[
                "x"
            ]
        ),
        int(
            learned_best[
                "y"
            ]
        ),
    )


    # --------------------------------------------------------
    # Case 1:
    # Both planners already agree.
    # --------------------------------------------------------

    if (
        classical_goal
        ==
        learned_goal
    ):

        selected_index = (
            learned_best_index
        )


        decision_source = (
            "LEARNED_AGREEMENT"
        )


    else:

        # ----------------------------------------------------
        # Classical utility of learned-selected candidate
        # ----------------------------------------------------

        learned_candidate_classical_utility = float(
            classical_scores[
                learned_best_index
            ][
                "utility"
            ]
        )


        best_classical_utility = float(
            classical_best[
                "utility"
            ]
        )


        candidate_regret = (
            best_classical_utility
            -
            learned_candidate_classical_utility
        )


        # ----------------------------------------------------
        # Case 2:
        # learned model is confident AND the classical
        # sacrifice is very small.
        # ----------------------------------------------------

        if (
            learned_margin
            >=
            HYBRID_LEARNED_MARGIN_MIN_43
            and
            candidate_regret
            <=
            HYBRID_CLASSICAL_REGRET_MAX_43
        ):

            selected_index = (
                learned_best_index
            )


            decision_source = (
                "LEARNED_OVERRIDE"
            )


        # ----------------------------------------------------
        # Case 3:
        # classical fallback
        # ----------------------------------------------------

        else:

            selected_index = (
                classical_best_index
            )


            decision_source = (
                "CLASSICAL_FALLBACK"
            )


    # --------------------------------------------------------
    # Final hybrid record
    # --------------------------------------------------------

    selected_classical = (
        classical_scores[
            selected_index
        ]
    )


    selected_learned = (
        learned_scores[
            selected_index
        ]
    )


    classical_regret = float(
        classical_best[
            "utility"
        ]
        -
        selected_classical[
            "utility"
        ]
    )


    return {
        "selected":
            {
                "candidate_id":
                    selected_classical[
                        "candidate_id"
                    ],

                "x":
                    int(
                        selected_classical[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        selected_classical[
                            "y"
                        ]
                    ),

                "classical_utility":
                    float(
                        selected_classical[
                            "utility"
                        ]
                    ),

                "learned_utility":
                    float(
                        selected_learned[
                            "learned_utility"
                        ]
                    ),
            },

        "decision_source":
            decision_source,

        "learned_margin":
            float(
                learned_margin
            ),

        "classical_regret":
            classical_regret,

        "classical_goal":
            classical_goal,

        "learned_goal":
            learned_goal,
    }


# ============================================================
# Hybrid goal generator
# ============================================================

def generate_goal_from_knowledge_hybrid_43(
    agent,
    knowledge_mask,
    communication_degree,
):
    """
    Generate candidate frontiers once and compare classical
    and learned rankings on the exact same candidate set.
    """

    planning_memory = PlanningMemory(
        uuv_id=
            agent.uuv_id,

        observed_mask=
            knowledge_mask,
    )


    frontier_mask = (
        extract_frontier_mask(
            planning_memory
        )
    )


    candidates = (
        sample_frontier_candidates(
            frontier_mask,

            reference_position=(
                agent.state.x,
                agent.state.y,
            ),

            min_spacing=
                TASK_DEDUPLICATION_DISTANCE,

            max_candidates=
                12,
        )
    )


    if not candidates:

        return {
            "selected":
                None,

            "decision_source":
                "NO_TASK",

            "candidates":
                [],
        }


    normalized_candidates = []


    for candidate_index, candidate in enumerate(
        candidates
    ):

        normalized_candidates.append(
            {
                "candidate_id":
                    candidate.get(
                        "candidate_id",
                        candidate_index,
                    ),

                "x":
                    int(
                        candidate[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        candidate[
                            "y"
                        ]
                    ),
            }
        )


    classical_scores = [
        score_frontier_candidate_with_memory(
            agent,
            candidate,
            planning_memory,
        )

        for candidate
        in normalized_candidates
    ]


    learned_scores = [
        score_frontier_candidate_learned_39(
            agent=
                agent,

            candidate=
                candidate,

            planning_memory=
                planning_memory,

            communication_degree=
                communication_degree,
        )

        for candidate
        in normalized_candidates
    ]


    hybrid_result = (
        choose_hybrid_frontier_43(
            classical_scores=
                classical_scores,

            learned_scores=
                learned_scores,
        )
    )


    hybrid_result[
        "candidates"
    ] = normalized_candidates


    return hybrid_result


# ============================================================
# Current live-state validation
# ============================================================

current_positions_43 = {
    agent.uuv_id:
        (
            agent.state.x,
            agent.state.y,
        )

    for agent in agents
}


current_graph_43 = (
    build_communication_graph(
        current_positions_43,

        communication_range=
            COMMUNICATION_RANGE,
    )
)


current_records_43 = []


for agent in agents:

    uuv_id = agent.uuv_id


    result = (
        generate_goal_from_knowledge_hybrid_43(
            agent=
                agent,

            knowledge_mask=
                known_exploration_maps[
                    uuv_id
                ],

            communication_degree=
                current_graph_43.degree[
                    uuv_id
                ],
        )
    )


    current_records_43.append(
        {
            "uuv_id":
                uuv_id,

            "decision_source":
                result[
                    "decision_source"
                ],

            "hybrid_goal":
                (
                    None
                    if result[
                        "selected"
                    ]
                    is None
                    else (
                        result[
                            "selected"
                        ][
                            "x"
                        ],
                        result[
                            "selected"
                        ][
                            "y"
                        ],
                    )
                ),

            "classical_goal":
                result.get(
                    "classical_goal"
                ),

            "learned_goal":
                result.get(
                    "learned_goal"
                ),

            "learned_margin":
                result.get(
                    "learned_margin",
                    np.nan,
                ),

            "classical_regret":
                result.get(
                    "classical_regret",
                    0.0,
                ),
        }
    )


current_hybrid_df_43 = pd.DataFrame(
    current_records_43
)


# ============================================================
# Evaluate gate on Combo 37 synthetic states
# ============================================================

gate_records_43 = []


grouped_dataset_43 = (
    frontier_dataset_37.groupby(
        [
            "uuv_id",
            "synthetic_state",
        ]
    )
)


for (
    uuv_id,
    synthetic_state,
), group in grouped_dataset_43:

    # --------------------------------------------------------
    # Classical utilities directly available from dataset
    # --------------------------------------------------------

    classical_utilities = (
        group[
            "classical_utility"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Learned inference on stored features
    # --------------------------------------------------------

    feature_matrix = (
        group[
            FEATURE_COLUMNS_37
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    normalized_matrix = (
        (
            feature_matrix
            -
            feature_mean_np_38
        )
        /
        feature_std_np_38
    )


    feature_tensor = torch.tensor(
        normalized_matrix,
        dtype=torch.float32,
        device=device_38,
    )


    with torch.no_grad():

        learned_utilities = (
            frontier_model_38(
                feature_tensor
            )
            .cpu()
            .numpy()
            .reshape(
                -1
            )
        )


    # --------------------------------------------------------
    # Classical best
    # --------------------------------------------------------

    classical_best_index = int(
        np.argmax(
            classical_utilities
        )
    )


    # --------------------------------------------------------
    # Learned ranking
    # --------------------------------------------------------

    learned_order = np.argsort(
        learned_utilities
    )[::-1]


    learned_best_index = int(
        learned_order[
            0
        ]
    )


    if len(
        learned_order
    ) >= 2:

        learned_margin = float(
            learned_utilities[
                learned_order[
                    0
                ]
            ]
            -
            learned_utilities[
                learned_order[
                    1
                ]
            ]
        )

    else:

        learned_margin = np.inf


    # --------------------------------------------------------
    # Gate
    # --------------------------------------------------------

    if (
        learned_best_index
        ==
        classical_best_index
    ):

        hybrid_index = (
            learned_best_index
        )

        source = (
            "LEARNED_AGREEMENT"
        )


    else:

        classical_regret_if_learned = float(
            classical_utilities[
                classical_best_index
            ]
            -
            classical_utilities[
                learned_best_index
            ]
        )


        if (
            learned_margin
            >=
            HYBRID_LEARNED_MARGIN_MIN_43
            and
            classical_regret_if_learned
            <=
            HYBRID_CLASSICAL_REGRET_MAX_43
        ):

            hybrid_index = (
                learned_best_index
            )

            source = (
                "LEARNED_OVERRIDE"
            )


        else:

            hybrid_index = (
                classical_best_index
            )

            source = (
                "CLASSICAL_FALLBACK"
            )


    hybrid_classical_regret = float(
        classical_utilities[
            classical_best_index
        ]
        -
        classical_utilities[
            hybrid_index
        ]
    )


    gate_records_43.append(
        {
            "uuv_id":
                int(
                    uuv_id
                ),

            "synthetic_state":
                int(
                    synthetic_state
                ),

            "source":
                source,

            "classical_learned_agree":
                bool(
                    classical_best_index
                    ==
                    learned_best_index
                ),

            "hybrid_equals_classical":
                bool(
                    hybrid_index
                    ==
                    classical_best_index
                ),

            "hybrid_equals_learned":
                bool(
                    hybrid_index
                    ==
                    learned_best_index
                ),

            "learned_margin":
                learned_margin,

            "classical_regret":
                hybrid_classical_regret,
        }
    )


gate_df_43 = pd.DataFrame(
    gate_records_43
)


# ============================================================
# Aggregate gate statistics
# ============================================================

decision_counts_43 = (
    gate_df_43[
        "source"
    ]
    .value_counts()
)


total_states_43 = len(
    gate_df_43
)


classical_learned_agreement_43 = float(
    gate_df_43[
        "classical_learned_agree"
    ].mean()
)


hybrid_classical_agreement_43 = float(
    gate_df_43[
        "hybrid_equals_classical"
    ].mean()
)


hybrid_learned_agreement_43 = float(
    gate_df_43[
        "hybrid_equals_learned"
    ].mean()
)


mean_hybrid_regret_43 = float(
    gate_df_43[
        "classical_regret"
    ].mean()
)


max_hybrid_regret_43 = float(
    gate_df_43[
        "classical_regret"
    ].max()
)


learned_agreement_count_43 = int(
    decision_counts_43.get(
        "LEARNED_AGREEMENT",
        0,
    )
)


learned_override_count_43 = int(
    decision_counts_43.get(
        "LEARNED_OVERRIDE",
        0,
    )
)


classical_fallback_count_43 = int(
    decision_counts_43.get(
        "CLASSICAL_FALLBACK",
        0,
    )
)


# ============================================================
# Validation
# ============================================================

if (
    max_hybrid_regret_43
    >
    HYBRID_CLASSICAL_REGRET_MAX_43
    +
    1e-9
):

    raise RuntimeError(
        "Hybrid gate allowed excessive classical regret."
    )


# ============================================================
# Live-state integrity
# ============================================================

live_signature_after_43 = (
    get_swarm_state_signature()
)


live_swarm_unchanged_43 = (
    live_signature_before_43
    ==
    live_signature_after_43
)


live_known_maps_unchanged_43 = all(
    np.array_equal(
        live_known_maps_before_43[
            uuv_id
        ],

        known_exploration_maps[
            uuv_id
        ],
    )

    for uuv_id
    in known_exploration_maps
)


if not live_swarm_unchanged_43:

    raise RuntimeError(
        "Hybrid planner analysis mutated live swarm state."
    )


if not live_known_maps_unchanged_43:

    raise RuntimeError(
        "Hybrid planner analysis mutated known maps."
    )


# ============================================================
# Console report
# ============================================================

print(
    "=" * 112
)

print(
    "MultiUUV-Link — Confidence-Gated "
    "Hybrid Frontier Ranking"
)

print(
    "=" * 112
)


print(
    f"\nLearned margin threshold       : "
    f"{HYBRID_LEARNED_MARGIN_MIN_43:.4f}"
)

print(
    f"Maximum classical regret       : "
    f"{HYBRID_CLASSICAL_REGRET_MAX_43:.4f}"
)


print(
    "\nCurrent live-state decisions:"
)

print(
    current_hybrid_df_43.to_string(
        index=False,
        formatters={
            "learned_margin":
                lambda x: f"{x:.4f}",

            "classical_regret":
                lambda x: f"{x:.6f}",
        },
    )
)


print(
    "\nSynthetic-state gate evaluation:"
)

print(
    f"  States evaluated             : "
    f"{total_states_43}"
)

print(
    f"  Classical-learned agreement  : "
    f"{100 * classical_learned_agreement_43:.2f}%"
)

print(
    f"  Hybrid equals classical      : "
    f"{100 * hybrid_classical_agreement_43:.2f}%"
)

print(
    f"  Hybrid equals learned        : "
    f"{100 * hybrid_learned_agreement_43:.2f}%"
)


print(
    "\nGate decisions:"
)

print(
    f"  Learned through agreement    : "
    f"{learned_agreement_count_43}"
)

print(
    f"  Learned confident overrides  : "
    f"{learned_override_count_43}"
)

print(
    f"  Classical fallbacks          : "
    f"{classical_fallback_count_43}"
)


print(
    "\nHybrid classical regret:"
)

print(
    f"  Mean regret                  : "
    f"{mean_hybrid_regret_43:.8f}"
)

print(
    f"  Maximum regret               : "
    f"{max_hybrid_regret_43:.8f}"
)


print(
    "\nState-integrity checks:"
)

print(
    f"  Live swarm unchanged         : "
    f"{live_swarm_unchanged_43}"
)

print(
    f"  Live known maps unchanged    : "
    f"{live_known_maps_unchanged_43}"
)


print(
    "\nConfidence-gated hybrid frontier "
    "ranking validated."
)

# Combo 44 — Three-Way Classical vs Learned vs Hybrid Benchmark

## Objective

We now have three frontier-ranking strategies:

### Classical

$$
J_C(g)
$$

### Learned

$$
J_L(g)
=
\hat J_\theta(g)
$$

### Hybrid

$$
g_H
=
\text{confidence-gated}(J_C,J_L)
$$

This combo compares all three under matched multi-seed degraded-communication missions.

---

# Experimental Design

Use the same five communication seeds from Combo 42:

$$
\{
420001,
420002,
420003,
420004,
420005
\}.
$$

For each seed run:

```text
Classical
Learned
Hybrid
```

Therefore:

$$
N_{runs}=15.
$$

Each mission begins from:

```python
restore_checkpoint_24_5()
```

with identical:

- map;
- resources;
- initial UUV states;
- communication parameters;
- mission horizon;
- safety logic.

---

# Mission Horizon

Use:

$$
T=180.
$$

---

# Hybrid Allocation

The hybrid allocator uses the same shared frontier-task pool as the learned allocator.

For every UUV-task pair:

1. compute classical utility;
2. compute learned utility;
3. apply the confidence gate;
4. produce one effective hybrid utility.

For exact classical-learned agreement:

$$
J_H=J_C.
$$

For allowed learned overrides:

$$
J_H=J_L.
$$

For fallback:

$$
J_H=J_C.
$$

Because Combo 43 produced no learned overrides on the synthetic validation set, the current expectation is that hybrid behavior will remain close to classical behavior.

---

# Metrics

For every run measure:

- final coverage;
- coverage gain;
- coverage AUC;
- incremental redundancy;
- HOLD steps;
- safety overrides;
- goal changes;
- allocation rounds;
- minimum separation.

---

# Comparison Questions

The benchmark should answer:

### 1. Does hybrid preserve classical robustness?

$$
m_H\approx m_C
$$

### 2. Does hybrid avoid the variability seen in fully learned control?

### 3. Does the hybrid architecture introduce any measurable penalty?

### 4. Does the hybrid ever select a learned override during full missions?

---

# Hybrid Decision Statistics

Across all hybrid runs record:

$$
N_{agree}
$$

$$
N_{override}
$$

$$
N_{fallback}.
$$

This shows how much authority the neural model actually receives in closed-loop operation.

---

# Interpretation

If hybrid results closely match classical results while retaining a trained neural decision layer, then the project obtains a useful architecture:

$$
\boxed{
\text{Learning With Classical Guardrails}
}
$$

rather than forcing learning to replace a controller that already works well.

---

# Progress

This combo runs 15 full missions.

Use a progress bar showing:

```text
seed
controller
completed runs / total runs
```

---

# Final State

After benchmarking:

```python
restore_checkpoint_24_5()
```

must be called again.

---

# Next Phase

After this benchmark, the main autonomy architecture is considered complete.

The following combos move into:

- mission visualization recorder;
- GIF/MP4 generation;
- final summary metrics;
- end-to-end demonstration.

---

In [ ]:
# ============================================================
# Combo 44 — Three-Way Classical vs Learned vs Hybrid Benchmark
# ============================================================

import time


# ============================================================
# Configuration
# ============================================================

BENCHMARK_SEEDS_44 = [
    420001,
    420002,
    420003,
    420004,
    420005,
]

BENCHMARK_STEPS_44 = 180


# ============================================================
# Hybrid pair scoring helper
# ============================================================

def hybrid_effective_score_44(
    agent,
    task,
    planning_memory,
    communication_degree,
):
    """
    Evaluate one UUV-task pair using both classical and learned
    utilities.

    The returned 'effective_utility' follows the confidence-gated
    policy from Combo 43.
    """

    classical_score = (
        score_frontier_candidate_with_memory(
            agent,
            task,
            planning_memory,
        )
    )


    learned_score = (
        score_frontier_candidate_learned_39(
            agent=
                agent,

            candidate=
                task,

            planning_memory=
                planning_memory,

            communication_degree=
                communication_degree,
        )
    )


    return {
        "candidate_id":
            task[
                "candidate_id"
            ],

        "x":
            task[
                "x"
            ],

        "y":
            task[
                "y"
            ],

        "classical_utility":
            float(
                classical_score[
                    "utility"
                ]
            ),

        "learned_utility":
            float(
                learned_score[
                    "learned_utility"
                ]
            ),
    }


# ============================================================
# Hybrid distributed allocator
# ============================================================

def perform_hybrid_allocation_round_44(
    operational_graph,
    previous_goals,
):
    """
    Distributed task allocation with confidence-gated
    classical + learned ranking.
    """

    assignments = {
        agent.uuv_id: None
        for agent in agents
    }


    decision_counts = {
        "LEARNED_AGREEMENT":
            0,

        "LEARNED_OVERRIDE":
            0,

        "CLASSICAL_FALLBACK":
            0,
    }


    graph = operational_graph.copy()


    graph.add_nodes_from(
        [
            agent.uuv_id
            for agent in agents
        ]
    )


    components = list(
        nx.connected_components(
            graph
        )
    )


    for component_index, component in enumerate(
        components
    ):

        component = sorted(
            component
        )


        # ----------------------------------------------------
        # Build shared spatially deduplicated task pool
        # ----------------------------------------------------

        task_pool = []


        for uuv_id in component:

            agent = agent_by_id[
                uuv_id
            ]


            planning_memory = PlanningMemory(
                uuv_id=
                    uuv_id,

                observed_mask=
                    known_exploration_maps[
                        uuv_id
                    ],
            )


            frontier_mask = extract_frontier_mask(
                planning_memory
            )


            candidates = sample_frontier_candidates(
                frontier_mask,

                reference_position=(
                    agent.state.x,
                    agent.state.y,
                ),

                min_spacing=
                    TASK_DEDUPLICATION_DISTANCE,

                max_candidates=
                    12,
            )


            for candidate in candidates:

                candidate_xy = (
                    candidate[
                        "x"
                    ],
                    candidate[
                        "y"
                    ],
                )


                duplicate = any(
                    euclidean_distance(
                        candidate_xy,
                        (
                            task[
                                "x"
                            ],
                            task[
                                "y"
                            ],
                        ),
                    )
                    <
                    TASK_DEDUPLICATION_DISTANCE

                    for task in task_pool
                )


                if duplicate:
                    continue


                task_pool.append(
                    {
                        "candidate_id":
                            component_index
                            *
                            1000
                            +
                            len(
                                task_pool
                            ),

                        "x":
                            int(
                                candidate[
                                    "x"
                                ]
                            ),

                        "y":
                            int(
                                candidate[
                                    "y"
                                ]
                            ),
                    }
                )


        if not task_pool:
            continue


        # ----------------------------------------------------
        # Score all UUV-task pairs
        # ----------------------------------------------------

        pair_records = []


        for uuv_id in component:

            agent = agent_by_id[
                uuv_id
            ]


            planning_memory = PlanningMemory(
                uuv_id=
                    uuv_id,

                observed_mask=
                    known_exploration_maps[
                        uuv_id
                    ],
            )


            comm_degree = (
                graph.degree[
                    uuv_id
                ]
            )


            # ------------------------------------------------
            # Score every task
            # ------------------------------------------------

            task_scores = [
                hybrid_effective_score_44(
                    agent=
                        agent,

                    task=
                        task,

                    planning_memory=
                        planning_memory,

                    communication_degree=
                        comm_degree,
                )

                for task in task_pool
            ]


            if not task_scores:
                continue


            # ------------------------------------------------
            # Build classical / learned ranking lists
            # ------------------------------------------------

            classical_scores = [
                record[
                    "classical_utility"
                ]

                for record
                in task_scores
            ]


            learned_scores = [
                record[
                    "learned_utility"
                ]

                for record
                in task_scores
            ]


            classical_best_index = int(
                np.argmax(
                    classical_scores
                )
            )


            learned_order = np.argsort(
                learned_scores
            )[::-1]


            learned_best_index = int(
                learned_order[
                    0
                ]
            )


            # ------------------------------------------------
            # Learned confidence margin
            # ------------------------------------------------

            if len(
                learned_order
            ) >= 2:

                learned_margin = float(
                    learned_scores[
                        learned_order[
                            0
                        ]
                    ]
                    -
                    learned_scores[
                        learned_order[
                            1
                        ]
                    ]
                )

            else:

                learned_margin = np.inf


            # ------------------------------------------------
            # Gate decision
            # ------------------------------------------------

            if (
                learned_best_index
                ==
                classical_best_index
            ):

                preferred_index = (
                    classical_best_index
                )

                source = (
                    "LEARNED_AGREEMENT"
                )


            else:

                classical_regret = float(
                    classical_scores[
                        classical_best_index
                    ]
                    -
                    classical_scores[
                        learned_best_index
                    ]
                )


                if (
                    learned_margin
                    >=
                    HYBRID_LEARNED_MARGIN_MIN_43
                    and
                    classical_regret
                    <=
                    HYBRID_CLASSICAL_REGRET_MAX_43
                ):

                    preferred_index = (
                        learned_best_index
                    )

                    source = (
                        "LEARNED_OVERRIDE"
                    )


                else:

                    preferred_index = (
                        classical_best_index
                    )

                    source = (
                        "CLASSICAL_FALLBACK"
                    )


            decision_counts[
                source
            ] += 1


            # ------------------------------------------------
            # Effective utility:
            #
            # Give preferred task a tiny deterministic bonus
            # so greedy assignment respects the hybrid choice.
            # ------------------------------------------------

            for task_index, record in enumerate(
                task_scores
            ):

                effective_utility = (
                    record[
                        "classical_utility"
                    ]
                )


                if (
                    task_index
                    ==
                    preferred_index
                ):

                    effective_utility += (
                        1e-6
                    )


                pair_records.append(
                    {
                        "uuv_id":
                            uuv_id,

                        "task_index":
                            task_index,

                        "candidate_id":
                            record[
                                "candidate_id"
                            ],

                        "x":
                            record[
                                "x"
                            ],

                        "y":
                            record[
                                "y"
                            ],

                        "effective_utility":
                            float(
                                effective_utility
                            ),

                        "source":
                            source,
                    }
                )


        # ----------------------------------------------------
        # Global greedy assignment within component
        # ----------------------------------------------------

        pair_records.sort(
            key=lambda record:
                record[
                    "effective_utility"
                ],

            reverse=True,
        )


        assigned_uuvs = set()
        assigned_tasks = set()
        assigned_positions = []


        for record in pair_records:

            uuv_id = record[
                "uuv_id"
            ]


            task_index = record[
                "task_index"
            ]


            if uuv_id in assigned_uuvs:
                continue


            if task_index in assigned_tasks:
                continue


            candidate_position = (
                record[
                    "x"
                ],
                record[
                    "y"
                ],
            )


            conflict = any(
                euclidean_distance(
                    candidate_position,
                    assigned_position,
                )
                <
                MIN_ASSIGNED_GOAL_SEPARATION

                for assigned_position
                in assigned_positions
            )


            if conflict:
                continue


            assignments[
                uuv_id
            ] = {
                "candidate_id":
                    int(
                        record[
                            "candidate_id"
                        ]
                    ),

                "x":
                    int(
                        record[
                            "x"
                        ]
                    ),

                "y":
                    int(
                        record[
                            "y"
                        ]
                    ),
            }


            assigned_uuvs.add(
                uuv_id
            )


            assigned_tasks.add(
                task_index
            )


            assigned_positions.append(
                candidate_position
            )


    # ========================================================
    # Goal-change count
    # ========================================================

    goal_changes = 0


    for uuv_id in assignments:

        old_goal = previous_goals.get(
            uuv_id
        )


        new_goal = assignments[
            uuv_id
        ]


        old_xy = (
            None
            if old_goal is None
            else (
                old_goal[
                    "x"
                ],
                old_goal[
                    "y"
                ],
            )
        )


        new_xy = (
            None
            if new_goal is None
            else (
                new_goal[
                    "x"
                ],
                new_goal[
                    "y"
                ],
            )
        )


        if old_xy != new_xy:

            goal_changes += 1


    return (
        assignments,
        goal_changes,
        decision_counts,
    )


# ============================================================
# Generic benchmark mission runner
# ============================================================

def run_controller_mission_44(
    controller,
    channel_seed,
):
    """
    controller:
        "Classical"
        "Learned"
        "Hybrid"
    """

    restore_checkpoint_24_5()


    frontier_model_38.eval()


    rng = np.random.default_rng(
        channel_seed
    )


    # --------------------------------------------------------
    # Local -> known consistency
    # --------------------------------------------------------

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent.exploration_memory.observed,
        )


    # --------------------------------------------------------
    # Start state
    # --------------------------------------------------------

    start_maps = {
        agent.uuv_id:
            agent.exploration_memory
            .observed.copy()

        for agent in agents
    }


    start_union = np.logical_or.reduce(
        list(
            start_maps.values()
        )
    )


    start_coverage = (
        int(
            start_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    # --------------------------------------------------------
    # Channel
    # --------------------------------------------------------

    channel = DegradedCommunicationChannel(
        uuv_ids=[
            agent.uuv_id
            for agent in agents
        ],

        communication_range=
            COMMUNICATION_RANGE,

        packet_loss_probability=
            PACKET_LOSS_PROBABILITY,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,

        blackout_start_probability=
            BLACKOUT_START_PROBABILITY,

        blackout_min_duration=
            BLACKOUT_MIN_DURATION,

        blackout_max_duration=
            BLACKOUT_MAX_DURATION,

        rng=
            rng,
    )


    # --------------------------------------------------------
    # Mission state
    # --------------------------------------------------------

    goals = {
        agent.uuv_id: None
        for agent in agents
    }


    previous_edges = None

    last_allocation_step = (
        -ALLOCATION_COOLDOWN
    )


    allocation_rounds = 0
    goal_changes = 0


    hold_counts = {
        agent.uuv_id: 0
        for agent in agents
    }


    safety_overrides = {
        agent.uuv_id: 0
        for agent in agents
    }


    min_separation = np.inf


    coverage_curve = []


    hybrid_decision_totals = {
        "LEARNED_AGREEMENT":
            0,

        "LEARNED_OVERRIDE":
            0,

        "CLASSICAL_FALLBACK":
            0,
    }


    # ========================================================
    # Mission loop
    # ========================================================

    for t in range(
        BENCHMARK_STEPS_44
    ):

        # ----------------------------------------------------
        # Deliver delayed messages
        # ----------------------------------------------------

        channel.deliver_due(
            current_time=
                t,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )


        # ----------------------------------------------------
        # Preserve local knowledge
        # ----------------------------------------------------

        for agent in agents:

            uuv_id = agent.uuv_id


            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


        # ----------------------------------------------------
        # Communication
        # ----------------------------------------------------

        positions = {
            agent.uuv_id:
                (
                    agent.state.x,
                    agent.state.y,
                )

            for agent in agents
        }


        transmission = (
            channel.transmit_round(
                current_time=
                    t,

                positions=
                    positions,

                known_maps=
                    known_exploration_maps,

                resource_memories={
                    agent.uuv_id:
                        agent.resource_memory

                    for agent in agents
                },
            )
        )


        operational_graph = (
            transmission[
                "operational_graph"
            ]
        )


        # ----------------------------------------------------
        # Goal validity
        # ----------------------------------------------------

        goal_invalidated = False


        for agent in agents:

            uuv_id = agent.uuv_id


            goal = goals[
                uuv_id
            ]


            if goal is None:
                continue


            if not goal_is_still_useful(
                agent,
                goal,
            ):

                goals[
                    uuv_id
                ] = None

                goal_invalidated = True


        # ----------------------------------------------------
        # Topology
        # ----------------------------------------------------

        canonical_edges = frozenset(
            tuple(
                sorted(
                    edge
                )
            )

            for edge
            in operational_graph.edges()
        )


        topology_changed = (
            previous_edges is None
            or
            canonical_edges
            !=
            previous_edges
        )


        previous_edges = (
            canonical_edges
        )


        # ----------------------------------------------------
        # Allocation trigger
        # ----------------------------------------------------

        missing_goal = any(
            goals[
                agent.uuv_id
            ]
            is None

            for agent in agents
        )


        periodic_trigger = (
            t
            %
            ALLOCATION_INTERVAL
            ==
            0
        )


        urgent_trigger = (
            topology_changed
            or
            goal_invalidated
            or
            missing_goal
        )


        cooldown_satisfied = (
            t
            -
            last_allocation_step
            >=
            ALLOCATION_COOLDOWN
        )


        should_allocate = (
            periodic_trigger
            or
            (
                urgent_trigger
                and
                cooldown_satisfied
            )
        )


        # ----------------------------------------------------
        # Controller-specific allocation
        # ----------------------------------------------------

        if should_allocate:

            if (
                controller
                ==
                "Classical"
            ):

                (
                    new_goals,
                    changes,
                    _,
                    _,
                    _,
                ) = perform_stable_allocation_round(
                    operational_graph,
                    goals,
                )


            elif (
                controller
                ==
                "Learned"
            ):

                (
                    new_goals,
                    changes,
                    _,
                ) = perform_learned_allocation_round_40(
                    operational_graph,
                    goals,
                )


            elif (
                controller
                ==
                "Hybrid"
            ):

                (
                    new_goals,
                    changes,
                    decision_counts,
                ) = perform_hybrid_allocation_round_44(
                    operational_graph,
                    goals,
                )


                for key in hybrid_decision_totals:

                    hybrid_decision_totals[
                        key
                    ] += (
                        decision_counts[
                            key
                        ]
                    )


            else:

                raise ValueError(
                    f"Unknown controller: {controller}"
                )


            goals = new_goals

            goal_changes += changes

            allocation_rounds += 1

            last_allocation_step = t


        # ----------------------------------------------------
        # Desired actions
        # ----------------------------------------------------

        desired_actions = []


        for agent in agents:

            uuv_id = agent.uuv_id


            goal = goals[
                uuv_id
            ]


            if goal is None:

                action = ACTION_HOLD

                hold_counts[
                    uuv_id
                ] += 1


            else:

                action = choose_greedy_action(
                    agent.state,

                    goal_x=
                        goal[
                            "x"
                        ],

                    goal_y=
                        goal[
                            "y"
                        ],
                )


                if action is None:

                    action = ACTION_HOLD

                    hold_counts[
                        uuv_id
                    ] += 1


            desired_actions.append(
                action
            )


        # ----------------------------------------------------
        # Safety shield
        # ----------------------------------------------------

        shield = safety_shield(
            states=[
                agent.state
                for agent in agents
            ],

            desired_actions=
                desired_actions,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )


        executed_actions = (
            shield[
                "executed_actions"
            ]
        )


        min_separation = min(
            min_separation,

            shield[
                "minimum_distance"
            ],
        )


        for (
            agent,
            desired,
            executed,
        ) in zip(
            agents,
            desired_actions,
            executed_actions,
        ):

            if desired != executed:

                safety_overrides[
                    agent.uuv_id
                ] += 1


        # ----------------------------------------------------
        # Motion
        # ----------------------------------------------------

        next_states = [
            predict_safety_state(
                state=
                    agent.state,

                action=
                    action,

                step_size=
                    ROLLOUT_STEP_SIZE,
            )

            for agent, action
            in zip(
                agents,
                executed_actions,
            )
        ]


        for agent, state in zip(
            agents,
            next_states,
        ):

            agent.state = state


            agent.trajectory.append(
                (
                    state.x,
                    state.y,
                )
            )


        # ----------------------------------------------------
        # Local sensing
        # ----------------------------------------------------

        for agent in agents:

            uuv_id = agent.uuv_id


            agent.exploration_memory.update_from_fov(
                agent.state
            )


            known_exploration_maps[
                uuv_id
            ] = np.logical_or(
                known_exploration_maps[
                    uuv_id
                ],
                agent.exploration_memory.observed,
            )


            visible_resources = (
                discover_visible_resources(
                    agent.state,
                    resources,
                )
            )


            agent.resource_memory.update(
                visible_resources
            )


        # ----------------------------------------------------
        # Coverage history
        # ----------------------------------------------------

        union = np.logical_or.reduce(
            [
                agent.exploration_memory.observed
                for agent in agents
            ]
        )


        coverage_curve.append(
            float(
                union.sum()
                /
                (
                    WIDTH
                    *
                    HEIGHT
                )
            )
        )


    # ========================================================
    # Final metrics
    # ========================================================

    final_maps = {
        agent.uuv_id:
            agent.exploration_memory.observed

        for agent in agents
    }


    final_union = np.logical_or.reduce(
        list(
            final_maps.values()
        )
    )


    final_coverage = (
        int(
            final_union.sum()
        )
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    new_masks = {
        uuv_id:
            (
                final_maps[
                    uuv_id
                ]
                &
                (
                    ~start_maps[
                        uuv_id
                    ]
                )
            )

        for uuv_id
        in final_maps
    }


    sum_new = sum(
        int(
            mask.sum()
        )

        for mask
        in new_masks.values()
    )


    unique_new = int(
        np.logical_or.reduce(
            list(
                new_masks.values()
            )
        ).sum()
    )


    redundant_new = (
        sum_new
        -
        unique_new
    )


    redundancy = (
        redundant_new
        /
        sum_new

        if sum_new > 0

        else 0.0
    )


    coverage_auc = float(
        np.trapezoid(
            np.asarray(
                coverage_curve
            )
            *
            100.0
        )
    )


    # --------------------------------------------------------
    # Safety invariant
    # --------------------------------------------------------

    if (
        min_separation
        <
        SAFE_SEPARATION
    ):

        raise RuntimeError(
            f"{controller} run violated safety."
        )


    return {
        "seed":
            channel_seed,

        "controller":
            controller,

        "final_coverage_pct":
            final_coverage
            *
            100.0,

        "coverage_gain_pp":
            (
                final_coverage
                -
                start_coverage
            )
            *
            100.0,

        "coverage_auc":
            coverage_auc,

        "incremental_redundancy":
            redundancy,

        "unique_new_cells":
            unique_new,

        "hold_steps":
            sum(
                hold_counts.values()
            ),

        "safety_overrides":
            sum(
                safety_overrides.values()
            ),

        "goal_changes":
            goal_changes,

        "allocation_rounds":
            allocation_rounds,

        "min_separation_px":
            min_separation,

        "hybrid_agreements":
            hybrid_decision_totals[
                "LEARNED_AGREEMENT"
            ],

        "hybrid_overrides":
            hybrid_decision_totals[
                "LEARNED_OVERRIDE"
            ],

        "hybrid_fallbacks":
            hybrid_decision_totals[
                "CLASSICAL_FALLBACK"
            ],
    }


# ============================================================
# Execute 15 missions
# ============================================================

benchmark_records_44 = []


try:

    from tqdm.auto import tqdm


    progress_44 = tqdm(
        total=
            len(
                BENCHMARK_SEEDS_44
            )
            *
            3,

        desc=
            "Three-way benchmark",
    )


    use_progress_44 = True


except Exception:

    progress_44 = None

    use_progress_44 = False


start_time_44 = time.time()


for seed in BENCHMARK_SEEDS_44:

    for controller in [
        "Classical",
        "Learned",
        "Hybrid",
    ]:

        result = (
            run_controller_mission_44(
                controller=
                    controller,

                channel_seed=
                    seed,
            )
        )


        benchmark_records_44.append(
            result
        )


        if use_progress_44:

            progress_44.set_postfix(
                seed=
                    seed,

                controller=
                    controller,
            )


            progress_44.update(
                1
            )


if use_progress_44:

    progress_44.close()


runtime_44 = (
    time.time()
    -
    start_time_44
)


# ============================================================
# Restore common checkpoint
# ============================================================

restore_checkpoint_24_5()


# ============================================================
# Results
# ============================================================

benchmark_df_44 = pd.DataFrame(
    benchmark_records_44
)


# ============================================================
# Controller summary
# ============================================================

summary_metrics_44 = [
    "final_coverage_pct",
    "coverage_auc",
    "incremental_redundancy",
    "hold_steps",
    "safety_overrides",
    "goal_changes",
    "min_separation_px",
]


summary_rows_44 = []


for controller in [
    "Classical",
    "Learned",
    "Hybrid",
]:

    subset = benchmark_df_44[
        benchmark_df_44[
            "controller"
        ]
        ==
        controller
    ]


    summary_rows_44.append(
        {
            "controller":
                controller,

            "final_coverage_mean":
                subset[
                    "final_coverage_pct"
                ].mean(),

            "final_coverage_std":
                subset[
                    "final_coverage_pct"
                ].std(),

            "coverage_auc_mean":
                subset[
                    "coverage_auc"
                ].mean(),

            "redundancy_mean":
                subset[
                    "incremental_redundancy"
                ].mean(),

            "redundancy_std":
                subset[
                    "incremental_redundancy"
                ].std(),

            "hold_mean":
                subset[
                    "hold_steps"
                ].mean(),

            "safety_override_mean":
                subset[
                    "safety_overrides"
                ].mean(),

            "goal_changes_mean":
                subset[
                    "goal_changes"
                ].mean(),

            "min_separation_mean":
                subset[
                    "min_separation_px"
                ].mean(),
        }
    )


summary_df_44 = pd.DataFrame(
    summary_rows_44
)


# ============================================================
# Hybrid authority statistics
# ============================================================

hybrid_runs_44 = benchmark_df_44[
    benchmark_df_44[
        "controller"
    ]
    ==
    "Hybrid"
]


hybrid_agreement_total_44 = int(
    hybrid_runs_44[
        "hybrid_agreements"
    ].sum()
)


hybrid_override_total_44 = int(
    hybrid_runs_44[
        "hybrid_overrides"
    ].sum()
)


hybrid_fallback_total_44 = int(
    hybrid_runs_44[
        "hybrid_fallbacks"
    ].sum()
)


hybrid_decision_total_44 = (
    hybrid_agreement_total_44
    +
    hybrid_override_total_44
    +
    hybrid_fallback_total_44
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 114
)

print(
    "MultiUUV-Link — Three-Way Classical vs "
    "Learned vs Hybrid Benchmark"
)

print(
    "=" * 114
)


print(
    f"\nSeeds tested              : "
    f"{len(BENCHMARK_SEEDS_44)}"
)

print(
    f"Total missions executed   : "
    f"{len(benchmark_records_44)}"
)

print(
    f"Mission steps per run     : "
    f"{BENCHMARK_STEPS_44}"
)

print(
    f"Benchmark runtime         : "
    f"{runtime_44:.2f} s"
)


print(
    "\nController summary:"
)

print(
    summary_df_44.to_string(
        index=False,

        formatters={
            "final_coverage_mean":
                lambda x: f"{x:.4f}",

            "final_coverage_std":
                lambda x: f"{x:.4f}",

            "coverage_auc_mean":
                lambda x: f"{x:.2f}",

            "redundancy_mean":
                lambda x: f"{x:.4f}",

            "redundancy_std":
                lambda x: f"{x:.4f}",

            "hold_mean":
                lambda x: f"{x:.2f}",

            "safety_override_mean":
                lambda x: f"{x:.2f}",

            "goal_changes_mean":
                lambda x: f"{x:.2f}",

            "min_separation_mean":
                lambda x: f"{x:.2f}",
        },
    )
)


print(
    "\nHybrid authority:"
)

print(
    f"  Total hybrid decisions        : "
    f"{hybrid_decision_total_44}"
)

print(
    f"  Learned agreements            : "
    f"{hybrid_agreement_total_44}"
)

print(
    f"  Learned overrides             : "
    f"{hybrid_override_total_44}"
)

print(
    f"  Classical fallbacks           : "
    f"{hybrid_fallback_total_44}"
)


if hybrid_decision_total_44 > 0:

    print(
        f"  Learned-authorized fraction   : "
        f"{100 * (hybrid_agreement_total_44 + hybrid_override_total_44) / hybrid_decision_total_44:.2f}%"
    )


print(
    "\nThree-way controller benchmark completed."
)


# ============================================================
# Plot 1 — Mean final coverage
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.bar(
    summary_df_44[
        "controller"
    ],

    summary_df_44[
        "final_coverage_mean"
    ],
)


plt.ylabel(
    "Mean final coverage [%]"
)

plt.title(
    "MultiUUV-Link — Mean Final Coverage"
)

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2 — Mean redundancy
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.bar(
    summary_df_44[
        "controller"
    ],

    summary_df_44[
        "redundancy_mean"
    ]
    *
    100.0,
)


plt.ylabel(
    "Mean incremental redundancy [%]"
)

plt.title(
    "MultiUUV-Link — Mean Exploration Redundancy"
)

plt.tight_layout()

plt.show()

# Combo 45 — Hybrid Mission Visualization Recorder

## Objective

The autonomy architecture is now stable enough to record an end-to-end mission.

This combo builds a reusable visualization recorder around the validated hybrid controller.

The animation should show:

- all four UUVs moving;
- each UUV's trajectory;
- each UUV's local FOV;
- active hybrid frontier goals;
- operational communication links;
- persistent resource discoveries;
- team coverage;
- mission timestep.

The output will be saved as:

```text
artifacts/multiuuv_hybrid_mission.gif
```

---

# Visualization Layers

For each frame:

```text
Seabed map
    ↓
Operational communication links
    ↓
Previously discovered resources
    ↓
UUV trajectories
    ↓
Current UUV positions
    ↓
FOV rectangles
    ↓
Current assigned goals
    ↓
Status text
```

---

# Resource Discovery

Resources remain hidden until discovered by at least one UUV.

Define team-known resources:

$$
D_{team}(t)
=
\bigcup_i D_i(t)
$$

Once a resource enters this set, its marker remains visible in all future frames.

Thus the animation visually demonstrates persistent discovery memory.

Resource classes retain the existing symbols:

- A: circle;
- B: triangle;
- C: square.

---

# Communication Links

Only **operational** links are shown.

Therefore:

$$
(i,j)\in E^{op}_t
$$

is displayed as a line between UUVs.

A link may disappear because of:

- range;
- persistent blackout.

Packet-level losses are not individually animated because they occur at message level rather than topology level.

---

# FOV

For each UUV, draw its current:

$$
128\times128
$$

pixel FOV footprint.

This visually emphasizes that each UUV perceives only a local part of the global seabed.

The global seabed image remains simulator/debug ground truth.

---

# Hybrid Goals

Current frontier assignments are marked with:

```text
*
```

and connected conceptually to the UUV executing them.

Goal markers disappear when the goal is completed or invalidated.

---

# Frame Sampling

A 180-step mission does not need 180 rendered frames.

Record every:

$$
2
$$

timesteps.

Therefore approximately:

$$
90
$$

mission frames are produced.

This keeps GIF size and rendering time manageable.

---

# Mission Controller

Use the validated confidence-gated hybrid allocator from Combo 44.

The mission uses:

- degraded communication;
- packet loss;
- latency;
- blackouts;
- hybrid frontier allocation;
- safety shield;
- persistent exploration memory;
- persistent resource memory.

---

# Reproducibility

Use:

```text
SHOWCASE_SEED = 450045
```

The mission starts from:

```python
restore_checkpoint_24_5()
```

---

# Expected Output

The combo should produce:

```text
../artifacts/multiuuv_hybrid_mission.gif
```

and report:

- frame count;
- mission steps;
- final coverage;
- resources discovered;
- safety overrides;
- minimum separation;
- GIF output path.

A successful result establishes the first reusable mission visualization artifact:

$$
\boxed{
\text{MultiUUV-Link Animated Mission Recorder}
}
$$

---

In [ ]:
# ============================================================
# Combo 45 — Hybrid Mission GIF Recorder
# ============================================================

from pathlib import Path
import imageio.v2 as imageio
from matplotlib.patches import Rectangle


# ============================================================
# Configuration
# ============================================================

SHOWCASE_STEPS_45 = 180

SHOWCASE_SEED_45 = 450045

FRAME_INTERVAL_45 = 2

GIF_FPS_45 = 8


showcase_rng_45 = np.random.default_rng(
    SHOWCASE_SEED_45
)


# ============================================================
# Output directory
# ============================================================

ARTIFACT_DIR_45 = (
    Path.cwd().parent
    /
    "artifacts"
)


ARTIFACT_DIR_45.mkdir(
    parents=True,
    exist_ok=True,
)


GIF_PATH_45 = (
    ARTIFACT_DIR_45
    /
    "multiuuv_hybrid_mission_prototype.gif"
)


# ============================================================
# Reset controlled experiment
# ============================================================

restore_checkpoint_24_5()

frontier_model_38.eval()


# ============================================================
# Ensure local knowledge is included in known maps
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id

    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ============================================================
# Mission-start snapshots
# ============================================================

showcase_start_maps_45 = {
    agent.uuv_id:
        agent.exploration_memory
        .observed.copy()

    for agent in agents
}


showcase_start_traj_idx_45 = {
    agent.uuv_id:
        len(
            agent.trajectory
        ) - 1

    for agent in agents
}


showcase_start_union_45 = (
    np.logical_or.reduce(
        list(
            showcase_start_maps_45.values()
        )
    )
)


showcase_start_coverage_45 = (
    int(
        showcase_start_union_45.sum()
    )
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Degraded communication channel
# ============================================================

channel_45 = DegradedCommunicationChannel(
    uuv_ids=[
        agent.uuv_id
        for agent in agents
    ],

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        showcase_rng_45,
)


# ============================================================
# Planner state
# ============================================================

goals_45 = {
    agent.uuv_id: None
    for agent in agents
}


last_allocation_step_45 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_45 = None


allocation_rounds_45 = 0

goal_changes_45 = 0


hold_counts_45 = {
    agent.uuv_id: 0
    for agent in agents
}


safety_overrides_45 = {
    agent.uuv_id: 0
    for agent in agents
}


mission_min_separation_45 = np.inf


# ============================================================
# Persistent team-discovered resources
# ============================================================

team_discovered_resources_45 = {}


# ============================================================
# Visualization frame storage
# ============================================================

frames_45 = []


# ============================================================
# Hybrid authority counters
# ============================================================

hybrid_decision_totals_45 = {
    "LEARNED_AGREEMENT":
        0,

    "LEARNED_OVERRIDE":
        0,

    "CLASSICAL_FALLBACK":
        0,
}


# ============================================================
# Visualization helper
# ============================================================

def render_showcase_frame_45(
    timestep,
    operational_graph,
):
    """
    Render one mission frame and return RGB uint8 image.
    """

    fig, ax = plt.subplots(
        figsize=(12, 7)
    )


    ax.imshow(
        seabed_rgb
    )


    # ========================================================
    # 1. Operational communication links
    # ========================================================

    for uuv_a, uuv_b in (
        operational_graph.edges()
    ):

        agent_a = agent_by_id[
            uuv_a
        ]

        agent_b = agent_by_id[
            uuv_b
        ]


        ax.plot(
            [
                agent_a.state.x,
                agent_b.state.x,
            ],
            [
                agent_a.state.y,
                agent_b.state.y,
            ],
            linestyle="--",
            linewidth=1.2,
            alpha=0.65,
        )


    # ========================================================
    # 2. Persistent discovered resources
    # ========================================================

    for resource in (
        team_discovered_resources_45.values()
    ):

        marker = {
            "A": "o",
            "B": "^",
            "C": "s",
        }[
            resource.resource_class
        ]


        ax.scatter(
            resource.x,
            resource.y,
            marker=marker,
            s=95,
            edgecolors="white",
            linewidths=1.2,
            zorder=6,
        )


    # ========================================================
    # 3. Trajectories + FOV + positions + goals
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        # ----------------------------------------------------
        # Mission trajectory only
        # ----------------------------------------------------

        trajectory = np.asarray(
            agent.trajectory[
                showcase_start_traj_idx_45[
                    uuv_id
                ]:
            ]
        )


        if len(
            trajectory
        ) > 1:

            ax.plot(
                trajectory[:, 0],
                trajectory[:, 1],
                linewidth=1.7,
                alpha=0.8,
            )


        # ----------------------------------------------------
        # FOV rectangle
        # ----------------------------------------------------

        x0, y0, x1, y1 = (
            get_fov_bounds(
                agent.state
            )
        )


        fov_rectangle = Rectangle(
            (
                x0,
                y0,
            ),

            x1 - x0,
            y1 - y0,

            fill=False,
            linewidth=1.3,
            alpha=0.75,
        )


        ax.add_patch(
            fov_rectangle
        )


        # ----------------------------------------------------
        # Current UUV
        # ----------------------------------------------------

        ax.scatter(
            agent.state.x,
            agent.state.y,
            s=115,
            marker="o",
            zorder=8,
            label=f"UUV-{uuv_id}",
        )


        # ----------------------------------------------------
        # UUV ID label
        # ----------------------------------------------------

        ax.text(
            agent.state.x + 12,
            agent.state.y - 12,
            f"{uuv_id}",
            fontsize=9,
            weight="bold",
        )


        # ----------------------------------------------------
        # Current frontier goal
        # ----------------------------------------------------

        goal = goals_45[
            uuv_id
        ]


        if goal is not None:

            ax.scatter(
                goal[
                    "x"
                ],
                goal[
                    "y"
                ],
                marker="*",
                s=145,
                zorder=7,
            )


    # ========================================================
    # 4. Coverage status
    # ========================================================

    team_union = np.logical_or.reduce(
        [
            agent
            .exploration_memory
            .observed

            for agent in agents
        ]
    )


    coverage_pct = (
        100.0
        *
        team_union.sum()
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    # ========================================================
    # 5. Frame status box
    # ========================================================

    status_text = (
        f"t = {timestep:03d} / "
        f"{SHOWCASE_STEPS_45}\n"
        f"Coverage = {coverage_pct:.2f}%\n"
        f"Resources found = "
        f"{len(team_discovered_resources_45)}"
        f"/{len(resources)}\n"
        f"Operational links = "
        f"{operational_graph.number_of_edges()}"
    )


    ax.text(
        0.015,
        0.985,
        status_text,

        transform=
            ax.transAxes,

        verticalalignment=
            "top",

        bbox={
            "boxstyle":
                "round",

            "facecolor":
                "white",

            "alpha":
                0.78,
        },
    )


    ax.set_title(
        "MultiUUV-Link — "
        "Hybrid Cooperative Underwater Exploration"
    )


    ax.set_xlabel(
        "x [pixels]"
    )


    ax.set_ylabel(
        "y [pixels]"
    )


    ax.set_xlim(
        0,
        WIDTH
    )


    ax.set_ylim(
        HEIGHT,
        0
    )


    ax.legend(
        loc="lower left",
        ncol=2,
    )


    fig.tight_layout()


    # ========================================================
    # Convert canvas to RGB array
    # ========================================================

    fig.canvas.draw()


    rgba = np.asarray(
        fig.canvas.buffer_rgba()
    )


    rgb = (
        rgba[
            ...,
            :3
        ]
        .copy()
    )


    plt.close(
        fig
    )


    return rgb


# ============================================================
# Mission loop
# ============================================================

try:

    from tqdm.auto import tqdm


    mission_iterator_45 = tqdm(
        range(
            SHOWCASE_STEPS_45
        ),

        desc=
            "Recording hybrid mission",
    )


except Exception:

    mission_iterator_45 = range(
        SHOWCASE_STEPS_45
    )


for t in mission_iterator_45:

    # ========================================================
    # 1. Deliver delayed messages
    # ========================================================

    channel_45.deliver_due(
        current_time=
            t,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },
    )


    # ========================================================
    # 2. Preserve local knowledge
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # ========================================================
    # 3. Communication round
    # ========================================================

    positions_45 = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    transmission_45 = (
        channel_45.transmit_round(
            current_time=
                t,

            positions=
                positions_45,

            known_maps=
                known_exploration_maps,

            resource_memories={
                agent.uuv_id:
                    agent.resource_memory

                for agent in agents
            },
        )
    )


    operational_graph_45 = (
        transmission_45[
            "operational_graph"
        ]
    )


    # ========================================================
    # 4. Goal validity
    # ========================================================

    goal_invalidated_45 = False


    for agent in agents:

        uuv_id = agent.uuv_id


        goal = goals_45[
            uuv_id
        ]


        if goal is None:
            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_45[
                uuv_id
            ] = None


            goal_invalidated_45 = True


    # ========================================================
    # 5. Topology change
    # ========================================================

    canonical_edges_45 = frozenset(
        tuple(
            sorted(
                edge
            )
        )

        for edge
        in operational_graph_45.edges()
    )


    topology_changed_45 = (
        previous_operational_edges_45
        is None

        or
        canonical_edges_45
        !=
        previous_operational_edges_45
    )


    previous_operational_edges_45 = (
        canonical_edges_45
    )


    # ========================================================
    # 6. Allocation trigger
    # ========================================================

    missing_goal_45 = any(
        goals_45[
            agent.uuv_id
        ]
        is None

        for agent in agents
    )


    periodic_trigger_45 = (
        t
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger_45 = (
        topology_changed_45
        or
        goal_invalidated_45
        or
        missing_goal_45
    )


    cooldown_satisfied_45 = (
        t
        -
        last_allocation_step_45
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate_45 = (
        periodic_trigger_45

        or
        (
            urgent_trigger_45
            and
            cooldown_satisfied_45
        )
    )


    if should_allocate_45:

        (
            new_goals_45,
            changes_45,
            decision_counts_45,
        ) = perform_hybrid_allocation_round_44(
            operational_graph_45,
            goals_45,
        )


        goals_45 = (
            new_goals_45
        )


        goal_changes_45 += (
            changes_45
        )


        allocation_rounds_45 += 1


        last_allocation_step_45 = t


        for key in (
            hybrid_decision_totals_45
        ):

            hybrid_decision_totals_45[
                key
            ] += (
                decision_counts_45[
                    key
                ]
            )


    # ========================================================
    # 7. Desired actions
    # ========================================================

    desired_actions_45 = []


    for agent in agents:

        uuv_id = agent.uuv_id


        goal = goals_45[
            uuv_id
        ]


        if goal is None:

            action = (
                ACTION_HOLD
            )


            hold_counts_45[
                uuv_id
            ] += 1


        else:

            action = (
                choose_greedy_action(
                    agent.state,

                    goal_x=
                        goal[
                            "x"
                        ],

                    goal_y=
                        goal[
                            "y"
                        ],
                )
            )


            if action is None:

                action = (
                    ACTION_HOLD
                )


                hold_counts_45[
                    uuv_id
                ] += 1


        desired_actions_45.append(
            action
        )


    # ========================================================
    # 8. Safety
    # ========================================================

    shield_45 = safety_shield(
        states=[
            agent.state
            for agent in agents
        ],

        desired_actions=
            desired_actions_45,

        safe_separation=
            SAFE_SEPARATION,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    executed_actions_45 = (
        shield_45[
            "executed_actions"
        ]
    )


    mission_min_separation_45 = min(
        mission_min_separation_45,

        shield_45[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions_45,
        executed_actions_45,
    ):

        if desired != executed:

            safety_overrides_45[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 9. Motion
    # ========================================================

    next_states_45 = [
        predict_safety_state(
            state=
                agent.state,

            action=
                action,

            step_size=
                ROLLOUT_STEP_SIZE,
        )

        for agent, action
        in zip(
            agents,
            executed_actions_45,
        )
    ]


    for agent, next_state in zip(
        agents,
        next_states_45,
    ):

        agent.state = (
            next_state
        )


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    # ========================================================
    # 10. Local sensing + discovery
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources
        )


        for resource in visible_resources:

            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


            team_discovered_resources_45[
                resource.resource_id
            ] = resource


    # ========================================================
    # 11. Record frame
    # ========================================================

    if (
        t
        %
        FRAME_INTERVAL_45
        ==
        0
    ):

        frame_45 = (
            render_showcase_frame_45(
                timestep=
                    t,

                operational_graph=
                    operational_graph_45,
            )
        )


        frames_45.append(
            frame_45
        )


# ============================================================
# Final frame
# ============================================================

frames_45.append(
    render_showcase_frame_45(
        timestep=
            SHOWCASE_STEPS_45,

        operational_graph=
            operational_graph_45,
    )
)


# ============================================================
# Validation
# ============================================================

if (
    mission_min_separation_45
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Combo 45 showcase mission violated safety."
    )


# ============================================================
# Save GIF
# ============================================================

imageio.mimsave(
    GIF_PATH_45,

    frames_45,

    fps=
        GIF_FPS_45,

    loop=
        0,
)


# ============================================================
# Final coverage
# ============================================================

final_union_45 = np.logical_or.reduce(
    [
        agent
        .exploration_memory
        .observed

        for agent in agents
    ]
)


final_coverage_45 = (
    final_union_45.sum()
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 108
)

print(
    "MultiUUV-Link — Hybrid Mission GIF Recorder"
)

print(
    "=" * 108
)


print(
    f"\nMission steps                : "
    f"{SHOWCASE_STEPS_45}"
)

print(
    f"Recorded frames              : "
    f"{len(frames_45)}"
)

print(
    f"GIF frame rate               : "
    f"{GIF_FPS_45} FPS"
)

print(
    f"Start team coverage          : "
    f"{100 * showcase_start_coverage_45:.4f}%"
)

print(
    f"Final team coverage          : "
    f"{100 * final_coverage_45:.4f}%"
)

print(
    f"Resources discovered         : "
    f"{len(team_discovered_resources_45)}"
    f"/{len(resources)}"
)

print(
    f"Allocation rounds            : "
    f"{allocation_rounds_45}"
)

print(
    f"Goal changes                 : "
    f"{goal_changes_45}"
)

print(
    f"Safety overrides             : "
    f"{sum(safety_overrides_45.values())}"
)

print(
    f"Mission minimum separation   : "
    f"{mission_min_separation_45:.2f} px"
)


print(
    "\nHybrid authority:"
)

print(
    f"  Learned agreements         : "
    f"{hybrid_decision_totals_45['LEARNED_AGREEMENT']}"
)

print(
    f"  Learned overrides          : "
    f"{hybrid_decision_totals_45['LEARNED_OVERRIDE']}"
)

print(
    f"  Classical fallbacks        : "
    f"{hybrid_decision_totals_45['CLASSICAL_FALLBACK']}"
)


print(
    f"\nGIF saved to:\n"
    f"{GIF_PATH_45.resolve()}"
)


print(
    "\nAnimated hybrid swarm mission generated."
)

# Combo 46 — End-to-End Fault-Recovery Showcase Animation

## Objective

Combo 45 produced a clean animation of normal hybrid cooperative exploration.

The final showcase should demonstrate the more distinctive MultiUUV-Link behavior:

$$
\boxed{
\text{Explore}
+
\text{Communicate}
+
\text{Discover}
+
\text{Lose a Vehicle}
+
\text{Detect Failure}
+
\text{Recover Task}
+
\text{Continue Mission}
}
$$

This combo generates the primary project showcase animation.

---

# Scenario

Run a hybrid mission for:

$$
T=260
$$

timesteps.

UUV-1 physically fails at:

$$
t_f=70.
$$

After failure, UUV-1:

- stops moving;
- stops sensing;
- stops sending;
- stops receiving;
- remains a stationary physical obstacle.

---

# Failure Detection

Use the validated contact-aware heartbeat mechanism.

For detector UUV $i$ and failed peer $f$:

$$
S_{if}(t)
$$

increments only while geometric communication contact exists without a fresh heartbeat.

The local failure threshold is:

$$
S_{if}\ge40.
$$

Swarm-level failure confirmation requires:

$$
N_{confirm}=2
$$

independent detections.

---

# Failure Visualization

The animation should distinguish three phases.

### Before physical failure

UUV-1 appears normally.

### After physical failure but before confirmation

UUV-1 remains frozen and is marked:

```text
FAILED?
```

The rest of the swarm has not yet completed distributed confirmation.

### After majority confirmation

UUV-1 becomes visibly marked:

```text
FAILED
```

and an orphaned frontier task is created.

---

# Orphan Recovery

At physical failure, freeze:

- UUV-1's position;
- known map;
- current assignment.

After majority confirmation:

1. recover the active task if safe;
2. otherwise generate a safety-feasible frontier task from the frozen knowledge snapshot;
3. select a healthy successor;
4. give that task temporary priority;
5. mark completion when:

$$
d\le32\text{ px}.
$$

---

# Visual Events

The animation should visibly mark:

### Physical failure

```text
X
```

at the failed UUV.

### Orphan task

```text
*
```

at the inherited frontier.

### Recovery successor

The selected successor should have a visible annotation.

### Completion

Once the orphan task is reached, status should display:

```text
RECOVERY COMPLETE
```

---

# Resource Discovery

Resources remain invisible until discovered.

Once discovered:

$$
r\in D_{team}
$$

the marker remains permanently visible.

The animation therefore shows autonomous resource discovery unfolding over time.

---

# Communication

Display only currently operational communication links.

This naturally visualizes:

- range loss;
- blackout;
- reconnection.

---

# FOV

Every healthy UUV keeps its local:

$$
128\times128
$$

FOV rectangle.

The failed UUV's FOV disappears after physical failure because it no longer senses.

---

# Hybrid Planner

The mission uses the confidence-gated hybrid frontier ranker.

Therefore the showcase represents the final autonomy architecture rather than a simplified visualization-only controller.

---

# Output

Save:

```text
../artifacts/multiuuv_fault_recovery_showcase.gif
```

Record approximately every:

$$
2
$$

simulation timesteps.

At:

$$
8\text{ FPS}
$$

the resulting GIF should remain compact enough for portfolio/demo use.

---

# Required Validation

The showcase must satisfy:

$$
\Delta p_f=0
$$

after failure.

It must obtain:

$$
t_{confirm}>t_f.
$$

It must create:

$$
g^{orphan}\neq None.
$$

It must select:

$$
i^*\neq f.
$$

The successor must complete the task:

$$
d(i^*,g^{orphan})\le32.
$$

The swarm must remain safe:

$$
d_{\min}\ge128.
$$

The other healthy UUVs must continue moving after failure.

---

# Expected Output

Report:

- physical failure timestep;
- majority confirmation timestep;
- detection delay;
- selected recovery UUV;
- orphan goal;
- task completion timestep;
- total recovery delay;
- resources discovered;
- final team coverage;
- safety overrides;
- minimum separation;
- frame count;
- GIF path.

A successful result establishes the final visual proof of:

$$
\boxed{
\text{Communication-Aware Decentralized Multi-UUV Mission Autonomy}
}
$$

---

In [ ]:
# ============================================================
# Combo 46 — End-to-End Fault-Recovery Showcase GIF
# ============================================================

from pathlib import Path
import imageio.v2 as imageio
from matplotlib.patches import Rectangle


# ============================================================
# Configuration
# ============================================================

SHOWCASE_STEPS_46 = 260

SHOWCASE_SEED_46 = 460046

FAILED_UUV_ID_46 = 1

PHYSICAL_FAILURE_TIME_46 = 70

HEARTBEAT_THRESHOLD_46 = 40

MAJORITY_CONFIRMATIONS_46 = 2

FRAME_INTERVAL_46 = 2

GIF_FPS_46 = 8


rng_46 = np.random.default_rng(
    SHOWCASE_SEED_46
)


uuv_ids_46 = [
    agent.uuv_id
    for agent in agents
]


# ============================================================
# Output
# ============================================================

ARTIFACT_DIR_46 = (
    Path.cwd().parent
    /
    "artifacts"
)


ARTIFACT_DIR_46.mkdir(
    parents=True,
    exist_ok=True,
)


GIF_PATH_46 = (
    ARTIFACT_DIR_46
    /
    "multiuuv_fault_recovery_showcase_prototype.gif"
)


# ============================================================
# Restore experiment
# ============================================================

restore_checkpoint_24_5()

frontier_model_38.eval()


# ============================================================
# Known-map consistency
# ============================================================

for agent in agents:

    uuv_id = agent.uuv_id


    known_exploration_maps[
        uuv_id
    ] = np.logical_or(
        known_exploration_maps[
            uuv_id
        ],
        agent.exploration_memory.observed,
    )


# ============================================================
# Mission start
# ============================================================

start_maps_46 = {
    agent.uuv_id:
        agent.exploration_memory
        .observed.copy()

    for agent in agents
}


start_traj_idx_46 = {
    agent.uuv_id:
        len(
            agent.trajectory
        ) - 1

    for agent in agents
}


start_union_46 = np.logical_or.reduce(
    list(
        start_maps_46.values()
    )
)


start_coverage_46 = (
    start_union_46.sum()
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Physical failure state
# ============================================================

failed_flags_46 = {
    uuv_id: False
    for uuv_id in uuv_ids_46
}


physical_failure_position_46 = None

failure_known_snapshot_46 = None

failure_goal_snapshot_46 = None


# ============================================================
# Failure detection
# ============================================================

suspicion_count_46 = {
    detector: {
        peer: 0
        for peer in uuv_ids_46
        if peer != detector
    }

    for detector in uuv_ids_46
}


declared_failed_46 = {
    detector: set()
    for detector in uuv_ids_46
}


detection_time_46 = {
    detector: {}
    for detector in uuv_ids_46
}


fault_confirmed_46 = False

majority_confirmation_time_46 = None


# ============================================================
# Heartbeat channel
# ============================================================

heartbeat_queue_46 = []

next_heartbeat_id_46 = 0

fresh_heartbeat_pairs_46 = set()


heartbeat_blackout_46 = LinkBlackoutManager(
    uuv_ids=
        uuv_ids_46,

    start_probability=
        BLACKOUT_START_PROBABILITY,

    min_duration=
        BLACKOUT_MIN_DURATION,

    max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        rng_46,
)


# ============================================================
# Mission communication channel
# ============================================================

channel_46 = DegradedCommunicationChannel(
    uuv_ids=
        uuv_ids_46,

    communication_range=
        COMMUNICATION_RANGE,

    packet_loss_probability=
        PACKET_LOSS_PROBABILITY,

    min_delay=
        MIN_COMM_DELAY,

    max_delay=
        MAX_COMM_DELAY,

    blackout_start_probability=
        BLACKOUT_START_PROBABILITY,

    blackout_min_duration=
        BLACKOUT_MIN_DURATION,

    blackout_max_duration=
        BLACKOUT_MAX_DURATION,

    rng=
        rng_46,
)


# ============================================================
# Planner state
# ============================================================

goals_46 = {
    uuv_id: None
    for uuv_id in uuv_ids_46
}


last_allocation_step_46 = (
    -ALLOCATION_COOLDOWN
)


previous_operational_edges_46 = None


# ============================================================
# Recovery state
# ============================================================

orphan_task_46 = None

orphan_source_46 = None

orphan_successor_46 = None

orphan_active_46 = False

orphan_completed_46 = False

orphan_assignment_time_46 = None

orphan_completion_time_46 = None


# ============================================================
# Mission statistics
# ============================================================

allocation_rounds_46 = 0

goal_changes_46 = 0


hold_counts_46 = {
    uuv_id: 0
    for uuv_id in uuv_ids_46
}


safety_overrides_46 = {
    uuv_id: 0
    for uuv_id in uuv_ids_46
}


mission_min_separation_46 = np.inf

healthy_motion_steps_after_failure_46 = 0


# ============================================================
# Persistent resources
# ============================================================

team_discovered_resources_46 = {}


# ============================================================
# Frames
# ============================================================

frames_46 = []


# ============================================================
# Heartbeat helpers
# ============================================================

def queue_heartbeat_46(
    sender,
    receiver,
    current_time,
):
    global next_heartbeat_id_46


    survives = packet_delivered(
        rng_46,

        loss_probability=
            PACKET_LOSS_PROBABILITY,
    )


    if not survives:

        return


    delay = sample_message_delay(
        rng_46,

        min_delay=
            MIN_COMM_DELAY,

        max_delay=
            MAX_COMM_DELAY,
    )


    heartbeat_queue_46.append(
        {
            "message_id":
                next_heartbeat_id_46,

            "sender":
                sender,

            "receiver":
                receiver,

            "send_time":
                current_time,

            "arrival_time":
                current_time
                +
                delay,
        }
    )


    next_heartbeat_id_46 += 1


def deliver_heartbeats_46(
    current_time,
):
    global heartbeat_queue_46
    global fresh_heartbeat_pairs_46


    fresh_heartbeat_pairs_46 = set()


    due = [
        message

        for message
        in heartbeat_queue_46

        if (
            message[
                "arrival_time"
            ]
            <=
            current_time
        )
    ]


    future = [
        message

        for message
        in heartbeat_queue_46

        if (
            message[
                "arrival_time"
            ]
            >
            current_time
        )
    ]


    for message in due:

        receiver = message[
            "receiver"
        ]


        sender = message[
            "sender"
        ]


        if failed_flags_46[
            receiver
        ]:

            continue


        fresh_heartbeat_pairs_46.add(
            (
                receiver,
                sender,
            )
        )


    heartbeat_queue_46 = (
        future
    )


# ============================================================
# Recovery-task helper
# ============================================================

def create_orphan_task_46(
    failed_agent,
):
    """
    Prefer failed UUV's actual active assignment if safe.
    Otherwise generate a safety-feasible frontier from the
    frozen failure-time known map.
    """

    minimum_clearance = (
        SAFE_SEPARATION
        +
        GOAL_REACHED_RADIUS
    )


    # --------------------------------------------------------
    # Existing assignment
    # --------------------------------------------------------

    if (
        failure_goal_snapshot_46
        is not None
    ):

        candidate_distance = (
            euclidean_distance(
                (
                    failure_goal_snapshot_46[
                        "x"
                    ],
                    failure_goal_snapshot_46[
                        "y"
                    ],
                ),

                physical_failure_position_46,
            )
        )


        if (
            candidate_distance
            >=
            minimum_clearance
        ):

            return (
                deepcopy(
                    failure_goal_snapshot_46
                ),

                "active_goal_at_failure",
            )


    # --------------------------------------------------------
    # Frozen-memory frontier
    # --------------------------------------------------------

    frozen_memory = PlanningMemory(
        uuv_id=
            failed_agent.uuv_id,

        observed_mask=
            failure_known_snapshot_46,
    )


    frontier_mask = extract_frontier_mask(
        frozen_memory
    )


    candidates = sample_frontier_candidates(
        frontier_mask,

        reference_position=
            physical_failure_position_46,

        min_spacing=
            TASK_DEDUPLICATION_DISTANCE,

        max_candidates=
            24,
    )


    feasible_candidates = []


    for candidate_index, candidate in enumerate(
        candidates
    ):

        normalized = {
            "candidate_id":
                candidate.get(
                    "candidate_id",
                    candidate_index,
                ),

            "x":
                int(
                    candidate[
                        "x"
                    ]
                ),

            "y":
                int(
                    candidate[
                        "y"
                    ]
                ),
        }


        distance_from_failure = (
            euclidean_distance(
                (
                    normalized[
                        "x"
                    ],
                    normalized[
                        "y"
                    ],
                ),
                physical_failure_position_46,
            )
        )


        if (
            distance_from_failure
            >=
            minimum_clearance
        ):

            feasible_candidates.append(
                normalized
            )


    if not feasible_candidates:

        raise RuntimeError(
            "Combo 46 could not generate a "
            "safety-feasible orphan task."
        )


    scored = [
        score_frontier_candidate_with_memory(
            failed_agent,
            candidate,
            frozen_memory,
        )

        for candidate
        in feasible_candidates
    ]


    selected = max(
        scored,

        key=lambda record:
            record[
                "utility"
            ],
    )


    return (
        deepcopy(
            selected
        ),

        "generated_safe_frontier",
    )


# ============================================================
# Recovery successor
# ============================================================

def select_successor_46():
    records = []


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_46[
            uuv_id
        ]:

            continue


        distance = euclidean_distance(
            (
                agent.state.x,
                agent.state.y,
            ),

            (
                orphan_task_46[
                    "x"
                ],
                orphan_task_46[
                    "y"
                ],
            ),
        )


        planning_memory = PlanningMemory(
            uuv_id=
                uuv_id,

            observed_mask=
                known_exploration_maps[
                    uuv_id
                ],
        )


        stats = candidate_fov_statistics(
            exploration_memory=
                planning_memory,

            candidate_x=
                orphan_task_46[
                    "x"
                ],

            candidate_y=
                orphan_task_46[
                    "y"
                ],
        )


        utility = (
            0.65
            *
            stats[
                "information_ratio"
            ]

            -
            0.35
            *
            distance
            /
            MAP_DIAGONAL
        )


        records.append(
            {
                "uuv_id":
                    uuv_id,

                "distance":
                    distance,

                "utility":
                    utility,
            }
        )


    records.sort(
        key=lambda record:
            (
                record[
                    "utility"
                ],

                -
                record[
                    "distance"
                ],
            ),

        reverse=True,
    )


    return records[
        0
    ][
        "uuv_id"
    ]


# ============================================================
# Frame renderer
# ============================================================

def render_frame_46(
    timestep,
    operational_graph,
):
    fig, ax = plt.subplots(
        figsize=(12, 7)
    )


    ax.imshow(
        seabed_rgb
    )


    # ========================================================
    # Communication links
    # ========================================================

    for uuv_a, uuv_b in (
        operational_graph.edges()
    ):

        a = agent_by_id[
            uuv_a
        ]

        b = agent_by_id[
            uuv_b
        ]


        ax.plot(
            [
                a.state.x,
                b.state.x,
            ],
            [
                a.state.y,
                b.state.y,
            ],
            linestyle="--",
            linewidth=1.2,
            alpha=0.65,
        )


    # ========================================================
    # Persistent resources
    # ========================================================

    for resource in (
        team_discovered_resources_46.values()
    ):

        marker = {
            "A":
                "o",

            "B":
                "^",

            "C":
                "s",
        }[
            resource.resource_class
        ]


        ax.scatter(
            resource.x,
            resource.y,

            marker=
                marker,

            s=
                95,

            edgecolors=
                "white",

            linewidths=
                1.2,

            zorder=
                6,
        )


    # ========================================================
    # UUVs
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        trajectory = np.asarray(
            agent.trajectory[
                start_traj_idx_46[
                    uuv_id
                ]:
            ]
        )


        if len(
            trajectory
        ) > 1:

            ax.plot(
                trajectory[:, 0],
                trajectory[:, 1],

                linewidth=
                    1.7,

                alpha=
                    0.8,
            )


        # ----------------------------------------------------
        # Healthy FOV only
        # ----------------------------------------------------

        if not failed_flags_46[
            uuv_id
        ]:

            x0, y0, x1, y1 = (
                get_fov_bounds(
                    agent.state
                )
            )


            ax.add_patch(
                Rectangle(
                    (
                        x0,
                        y0,
                    ),

                    x1 - x0,
                    y1 - y0,

                    fill=False,

                    linewidth=
                        1.2,

                    alpha=
                        0.75,
                )
            )


        # ----------------------------------------------------
        # Vehicle marker
        # ----------------------------------------------------

        if failed_flags_46[
            uuv_id
        ]:

            marker = "X"

            size = 175


        else:

            marker = "o"

            size = 110


        ax.scatter(
            agent.state.x,
            agent.state.y,

            marker=
                marker,

            s=
                size,

            zorder=
                8,

            label=
                f"UUV-{uuv_id}",
        )


        # ----------------------------------------------------
        # Vehicle text
        # ----------------------------------------------------

        label_text = (
            f"{uuv_id}"
        )


        if (
            uuv_id
            ==
            FAILED_UUV_ID_46
            and
            failed_flags_46[
                uuv_id
            ]
        ):

            if fault_confirmed_46:

                label_text += (
                    " FAILED"
                )

            else:

                label_text += (
                    " FAILED?"
                )


        if (
            orphan_successor_46
            ==
            uuv_id
            and
            orphan_active_46
        ):

            label_text += (
                " RECOVERY"
            )


        ax.text(
            agent.state.x + 12,
            agent.state.y - 12,

            label_text,

            fontsize=
                9,

            weight=
                "bold",
        )


        # ----------------------------------------------------
        # Normal mission goals
        # ----------------------------------------------------

        goal = goals_46[
            uuv_id
        ]


        if (
            goal is not None
            and
            not failed_flags_46[
                uuv_id
            ]
        ):

            ax.scatter(
                goal[
                    "x"
                ],
                goal[
                    "y"
                ],

                marker=
                    "*",

                s=
                    130,

                zorder=
                    7,
            )


    # ========================================================
    # Orphan marker
    # ========================================================

    if (
        orphan_task_46
        is not None
    ):

        ax.scatter(
            orphan_task_46[
                "x"
            ],
            orphan_task_46[
                "y"
            ],

            marker=
                "*",

            s=
                260,

            edgecolors=
                "black",

            linewidths=
                1.0,

            zorder=
                9,
        )


    # ========================================================
    # Coverage
    # ========================================================

    team_union = np.logical_or.reduce(
        [
            agent
            .exploration_memory
            .observed

            for agent in agents
        ]
    )


    coverage_pct = (
        100.0
        *
        team_union.sum()
        /
        (
            WIDTH
            *
            HEIGHT
        )
    )


    # ========================================================
    # Mission status
    # ========================================================

    if (
        timestep
        <
        PHYSICAL_FAILURE_TIME_46
    ):

        mission_phase = (
            "NORMAL EXPLORATION"
        )


    elif not fault_confirmed_46:

        mission_phase = (
            "PHYSICAL FAILURE / DETECTING"
        )


    elif (
        orphan_active_46
    ):

        mission_phase = (
            "FAULT CONFIRMED / TASK RECOVERY"
        )


    elif orphan_completed_46:

        mission_phase = (
            "RECOVERY COMPLETE"
        )


    else:

        mission_phase = (
            "FAULT CONFIRMED"
        )


    status_text = (
        f"t = {timestep:03d} / "
        f"{SHOWCASE_STEPS_46}\n"

        f"{mission_phase}\n"

        f"Coverage = "
        f"{coverage_pct:.2f}%\n"

        f"Resources = "
        f"{len(team_discovered_resources_46)}"
        f"/{len(resources)}\n"

        f"Operational links = "
        f"{operational_graph.number_of_edges()}"
    )


    if fault_confirmed_46:

        status_text += (
            f"\nFailure confirmed @ "
            f"t={majority_confirmation_time_46}"
        )


    if (
        orphan_successor_46
        is not None
    ):

        status_text += (
            f"\nRecovery UUV = "
            f"{orphan_successor_46}"
        )


    ax.text(
        0.015,
        0.985,

        status_text,

        transform=
            ax.transAxes,

        verticalalignment=
            "top",

        bbox={
            "boxstyle":
                "round",

            "facecolor":
                "white",

            "alpha":
                0.82,
        },
    )


    ax.set_title(
        "MultiUUV-Link — "
        "Decentralized Fault-Recovery Mission"
    )


    ax.set_xlabel(
        "x [pixels]"
    )


    ax.set_ylabel(
        "y [pixels]"
    )


    ax.set_xlim(
        0,
        WIDTH
    )


    ax.set_ylim(
        HEIGHT,
        0
    )


    ax.legend(
        loc=
            "lower left",

        ncol=
            2,
    )


    fig.tight_layout()

    fig.canvas.draw()


    rgba = np.asarray(
        fig.canvas.buffer_rgba()
    )


    rgb = (
        rgba[
            ...,
            :3
        ]
        .copy()
    )


    plt.close(
        fig
    )


    return rgb


# ============================================================
# Mission loop
# ============================================================

try:

    from tqdm.auto import tqdm


    mission_iterator_46 = tqdm(
        range(
            SHOWCASE_STEPS_46
        ),

        desc=
            "Recording fault-recovery showcase",
    )


except Exception:

    mission_iterator_46 = range(
        SHOWCASE_STEPS_46
    )


for t in mission_iterator_46:

    # ========================================================
    # 1. Inject physical failure
    # ========================================================

    if (
        t
        ==
        PHYSICAL_FAILURE_TIME_46
    ):

        failed_agent = agent_by_id[
            FAILED_UUV_ID_46
        ]


        physical_failure_position_46 = (
            failed_agent.state.x,
            failed_agent.state.y,
        )


        failure_known_snapshot_46 = (
            known_exploration_maps[
                FAILED_UUV_ID_46
            ].copy()
        )


        failure_goal_snapshot_46 = deepcopy(
            goals_46[
                FAILED_UUV_ID_46
            ]
        )


        failed_flags_46[
            FAILED_UUV_ID_46
        ] = True


        goals_46[
            FAILED_UUV_ID_46
        ] = None


    # ========================================================
    # 2. Deliver mission communication
    # ========================================================

    deliver_due_failure_aware_33(
        channel=
            channel_46,

        current_time=
            t,

        known_maps=
            known_exploration_maps,

        resource_memories={
            agent.uuv_id:
                agent.resource_memory

            for agent in agents
        },

        failed_flags=
            failed_flags_46,
    )


    # ========================================================
    # 3. Deliver heartbeats
    # ========================================================

    deliver_heartbeats_46(
        t
    )


    # ========================================================
    # 4. Preserve local knowledge
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


    # ========================================================
    # 5. Physical geometry including failed UUV
    # ========================================================

    physical_positions_46 = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    heartbeat_geo_graph_46 = (
        build_communication_graph(
            physical_positions_46,

            communication_range=
                COMMUNICATION_RANGE,
        )
    )


    heartbeat_blackout_result_46 = (
        heartbeat_blackout_46.step(
            heartbeat_geo_graph_46,

            allow_new_blackouts=
                True,
        )
    )


    heartbeat_operational_graph_46 = (
        heartbeat_blackout_result_46[
            "operational_graph"
        ]
    )


    # ========================================================
    # 6. Heartbeat transmissions among healthy agents
    # ========================================================

    for uuv_a, uuv_b in (
        heartbeat_operational_graph_46.edges()
    ):

        for sender, receiver in (
            (
                uuv_a,
                uuv_b,
            ),

            (
                uuv_b,
                uuv_a,
            ),
        ):

            if (
                failed_flags_46[
                    sender
                ]
                or
                failed_flags_46[
                    receiver
                ]
            ):

                continue


            queue_heartbeat_46(
                sender=
                    sender,

                receiver=
                    receiver,

                current_time=
                    t,
            )


    # ========================================================
    # 7. Contact-aware failure suspicion
    # ========================================================

    healthy_ids_46 = [
        uuv_id

        for uuv_id
        in uuv_ids_46

        if not failed_flags_46[
            uuv_id
        ]
    ]


    for detector in healthy_ids_46:

        for peer in uuv_ids_46:

            if peer == detector:
                continue


            if (
                peer
                in
                declared_failed_46[
                    detector
                ]
            ):

                continue


            if not (
                heartbeat_geo_graph_46
                .has_edge(
                    detector,
                    peer,
                )
            ):

                continue


            if (
                (
                    detector,
                    peer,
                )
                in
                fresh_heartbeat_pairs_46
            ):

                suspicion_count_46[
                    detector
                ][
                    peer
                ] = 0


            else:

                suspicion_count_46[
                    detector
                ][
                    peer
                ] += 1


            if (
                suspicion_count_46[
                    detector
                ][
                    peer
                ]
                >=
                HEARTBEAT_THRESHOLD_46
            ):

                declared_failed_46[
                    detector
                ].add(
                    peer
                )


                detection_time_46[
                    detector
                ][
                    peer
                ] = t


    # ========================================================
    # 8. Majority confirmation
    # ========================================================

    if (
        not fault_confirmed_46
        and
        t
        >
        PHYSICAL_FAILURE_TIME_46
    ):

        detectors_46 = [
            detector

            for detector
            in healthy_ids_46

            if (
                FAILED_UUV_ID_46
                in
                declared_failed_46[
                    detector
                ]
            )
        ]


        if (
            len(
                detectors_46
            )
            >=
            MAJORITY_CONFIRMATIONS_46
        ):

            fault_confirmed_46 = True


            majority_confirmation_time_46 = (
                t
            )


            failed_agent = agent_by_id[
                FAILED_UUV_ID_46
            ]


            (
                orphan_task_46,
                orphan_source_46,
            ) = create_orphan_task_46(
                failed_agent
            )


            orphan_successor_46 = (
                select_successor_46()
            )


            orphan_assignment_time_46 = (
                t
            )


            orphan_active_46 = True


            goals_46[
                orphan_successor_46
            ] = deepcopy(
                orphan_task_46
            )


    # ========================================================
    # 9. Healthy mission communication
    # ========================================================

    healthy_positions_46 = {
        uuv_id:
            physical_positions_46[
                uuv_id
            ]

        for uuv_id
        in healthy_ids_46
    }


    if len(
        healthy_positions_46
    ) >= 2:

        transmission_46 = (
            channel_46.transmit_round(
                current_time=
                    t,

                positions=
                    healthy_positions_46,

                known_maps=
                    known_exploration_maps,

                resource_memories={
                    agent.uuv_id:
                        agent.resource_memory

                    for agent in agents
                },
            )
        )


        operational_graph_46 = (
            transmission_46[
                "operational_graph"
            ]
        )


    else:

        operational_graph_46 = nx.Graph()


        operational_graph_46.add_nodes_from(
            healthy_ids_46
        )


    # ========================================================
    # 10. Goal validity
    # ========================================================

    goal_invalidated_46 = False


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_46[
            uuv_id
        ]:

            continue


        if (
            orphan_active_46
            and
            uuv_id
            ==
            orphan_successor_46
        ):

            continue


        goal = goals_46[
            uuv_id
        ]


        if goal is None:

            continue


        if not goal_is_still_useful(
            agent,
            goal,
        ):

            goals_46[
                uuv_id
            ] = None


            goal_invalidated_46 = True


    # ========================================================
    # 11. Allocation trigger
    # ========================================================

    operational_graph_46.add_nodes_from(
        healthy_ids_46
    )


    canonical_edges_46 = frozenset(
        tuple(
            sorted(
                edge
            )
        )

        for edge
        in operational_graph_46.edges()
    )


    topology_changed_46 = (
        previous_operational_edges_46
        is None

        or
        canonical_edges_46
        !=
        previous_operational_edges_46
    )


    previous_operational_edges_46 = (
        canonical_edges_46
    )


    missing_goal_46 = any(
        goals_46[
            uuv_id
        ]
        is None

        for uuv_id
        in healthy_ids_46
    )


    periodic_trigger_46 = (
        t
        %
        ALLOCATION_INTERVAL
        ==
        0
    )


    urgent_trigger_46 = (
        topology_changed_46
        or
        goal_invalidated_46
        or
        missing_goal_46
    )


    cooldown_satisfied_46 = (
        t
        -
        last_allocation_step_46
        >=
        ALLOCATION_COOLDOWN
    )


    should_allocate_46 = (
        periodic_trigger_46

        or
        (
            urgent_trigger_46
            and
            cooldown_satisfied_46
        )
    )


    if (
        should_allocate_46
        and
        healthy_ids_46
    ):

        (
            new_goals_46,
            changes_46,
            _,
        ) = perform_hybrid_allocation_round_44(
            operational_graph_46,
            goals_46,
        )


        for uuv_id in healthy_ids_46:

            goals_46[
                uuv_id
            ] = new_goals_46.get(
                uuv_id
            )


        goals_46[
            FAILED_UUV_ID_46
        ] = None


        # ----------------------------------------------------
        # Preserve recovery priority
        # ----------------------------------------------------

        if orphan_active_46:

            goals_46[
                orphan_successor_46
            ] = deepcopy(
                orphan_task_46
            )


        goal_changes_46 += (
            changes_46
        )


        allocation_rounds_46 += 1


        last_allocation_step_46 = (
            t
        )


    # ========================================================
    # 12. Desired actions
    # ========================================================

    desired_actions_46 = []


    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_46[
            uuv_id
        ]:

            action = (
                ACTION_HOLD
            )


        else:

            goal = goals_46[
                uuv_id
            ]


            if goal is None:

                action = (
                    ACTION_HOLD
                )


                hold_counts_46[
                    uuv_id
                ] += 1


            else:

                action = (
                    choose_greedy_action(
                        agent.state,

                        goal_x=
                            goal[
                                "x"
                            ],

                        goal_y=
                            goal[
                                "y"
                            ],
                    )
                )


                if action is None:

                    action = (
                        ACTION_HOLD
                    )


                    hold_counts_46[
                        uuv_id
                    ] += 1


        desired_actions_46.append(
            action
        )


    # ========================================================
    # 13. Failure-aware safety
    # ========================================================

    states_now_46 = [
        agent.state
        for agent in agents
    ]


    failed_mask_46 = [
        failed_flags_46[
            agent.uuv_id
        ]

        for agent in agents
    ]


    shield_46 = (
        failure_aware_safety_shield_33(
            states=
                states_now_46,

            desired_actions=
                desired_actions_46,

            failed_mask=
                failed_mask_46,

            safe_separation=
                SAFE_SEPARATION,

            step_size=
                ROLLOUT_STEP_SIZE,
        )
    )


    executed_actions_46 = (
        shield_46[
            "executed_actions"
        ]
    )


    mission_min_separation_46 = min(
        mission_min_separation_46,

        shield_46[
            "minimum_distance"
        ],
    )


    for (
        agent,
        desired,
        executed,
    ) in zip(
        agents,
        desired_actions_46,
        executed_actions_46,
    ):

        if (
            not failed_flags_46[
                agent.uuv_id
            ]
            and
            desired
            !=
            executed
        ):

            safety_overrides_46[
                agent.uuv_id
            ] += 1


    # ========================================================
    # 14. Motion
    # ========================================================

    previous_positions_46 = {
        agent.uuv_id:
            (
                agent.state.x,
                agent.state.y,
            )

        for agent in agents
    }


    next_states_46 = predict_joint_states_33(
        states=
            states_now_46,

        actions=
            executed_actions_46,

        step_size=
            ROLLOUT_STEP_SIZE,
    )


    for agent, next_state in zip(
        agents,
        next_states_46,
    ):

        agent.state = (
            next_state
        )


        agent.trajectory.append(
            (
                next_state.x,
                next_state.y,
            )
        )


    if (
        t
        >=
        PHYSICAL_FAILURE_TIME_46
    ):

        if any(
            (
                agent.state.x,
                agent.state.y,
            )
            !=
            previous_positions_46[
                agent.uuv_id
            ]

            for agent in agents

            if not failed_flags_46[
                agent.uuv_id
            ]
        ):

            healthy_motion_steps_after_failure_46 += 1


    # ========================================================
    # 15. Healthy sensing
    # ========================================================

    for agent in agents:

        uuv_id = agent.uuv_id


        if failed_flags_46[
            uuv_id
        ]:

            continue


        agent.exploration_memory.update_from_fov(
            agent.state
        )


        known_exploration_maps[
            uuv_id
        ] = np.logical_or(
            known_exploration_maps[
                uuv_id
            ],
            agent
            .exploration_memory
            .observed,
        )


        visible_resources_46 = (
            discover_visible_resources(
                agent.state,
                resources,
            )
        )


        agent.resource_memory.update(
            visible_resources_46
        )


        for resource in visible_resources_46:

            team_discovered_resources_46[
                resource.resource_id
            ] = resource


            resource_provenance[
                uuv_id
            ][
                resource.resource_id
            ] = "LOCAL"


    # ========================================================
    # 16. Recovery completion
    # ========================================================

    if orphan_active_46:

        successor = agent_by_id[
            orphan_successor_46
        ]


        orphan_distance_46 = (
            euclidean_distance(
                (
                    successor.state.x,
                    successor.state.y,
                ),

                (
                    orphan_task_46[
                        "x"
                    ],
                    orphan_task_46[
                        "y"
                    ],
                ),
            )
        )


        if (
            orphan_distance_46
            <=
            GOAL_REACHED_RADIUS
        ):

            orphan_completed_46 = True


            orphan_active_46 = False


            orphan_completion_time_46 = (
                t
            )


            goals_46[
                orphan_successor_46
            ] = None


    # ========================================================
    # 17. Record frame
    # ========================================================

    if (
        t
        %
        FRAME_INTERVAL_46
        ==
        0
    ):

        frames_46.append(
            render_frame_46(
                timestep=
                    t,

                operational_graph=
                    operational_graph_46,
            )
        )


# ============================================================
# Final frame
# ============================================================

frames_46.append(
    render_frame_46(
        timestep=
            SHOWCASE_STEPS_46,

        operational_graph=
            operational_graph_46,
    )
)


# ============================================================
# Final metrics
# ============================================================

failed_agent_46 = agent_by_id[
    FAILED_UUV_ID_46
]


failed_displacement_46 = (
    euclidean_distance(
        physical_failure_position_46,

        (
            failed_agent_46.state.x,
            failed_agent_46.state.y,
        ),
    )
)


final_union_46 = np.logical_or.reduce(
    [
        agent
        .exploration_memory
        .observed

        for agent in agents
    ]
)


final_coverage_46 = (
    final_union_46.sum()
    /
    (
        WIDTH
        *
        HEIGHT
    )
)


# ============================================================
# Recovery timing
# ============================================================

if fault_confirmed_46:

    detection_delay_46 = (
        majority_confirmation_time_46
        -
        PHYSICAL_FAILURE_TIME_46
    )

else:

    detection_delay_46 = None


if orphan_completed_46:

    total_recovery_delay_46 = (
        orphan_completion_time_46
        -
        PHYSICAL_FAILURE_TIME_46
    )

else:

    total_recovery_delay_46 = None


# ============================================================
# Validation
# ============================================================

if not fault_confirmed_46:

    raise RuntimeError(
        "Combo 46 failure was never confirmed."
    )


if (
    majority_confirmation_time_46
    <=
    PHYSICAL_FAILURE_TIME_46
):

    raise RuntimeError(
        "Combo 46 failure confirmation was not delayed."
    )


if orphan_task_46 is None:

    raise RuntimeError(
        "Combo 46 failed to create orphan task."
    )


if orphan_successor_46 is None:

    raise RuntimeError(
        "Combo 46 failed to select recovery UUV."
    )


if not orphan_completed_46:

    raise RuntimeError(
        "Combo 46 orphan task was not completed."
    )


if failed_displacement_46 != 0.0:

    raise RuntimeError(
        "Combo 46 failed UUV moved after failure."
    )


if (
    healthy_motion_steps_after_failure_46
    <=
    0
):

    raise RuntimeError(
        "Healthy swarm failed to continue mission."
    )


if (
    mission_min_separation_46
    <
    SAFE_SEPARATION
):

    raise RuntimeError(
        "Combo 46 safety invariant failed."
    )


# ============================================================
# Save GIF
# ============================================================

imageio.mimsave(
    GIF_PATH_46,

    frames_46,

    fps=
        GIF_FPS_46,

    loop=
        0,
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 112
)

print(
    "MultiUUV-Link — End-to-End Fault-Recovery Showcase"
)

print(
    "=" * 112
)


print(
    f"\nMission steps                  : "
    f"{SHOWCASE_STEPS_46}"
)

print(
    f"Physical failed UUV            : "
    f"{FAILED_UUV_ID_46}"
)

print(
    f"Physical failure timestep      : "
    f"{PHYSICAL_FAILURE_TIME_46}"
)

print(
    f"Majority confirmation timestep : "
    f"{majority_confirmation_time_46}"
)

print(
    f"Detection delay                : "
    f"{detection_delay_46} steps"
)


print(
    "\nRecovery:"
)

print(
    f"  Orphan source               : "
    f"{orphan_source_46}"
)

print(
    f"  Orphan goal                 : "
    f"({orphan_task_46['x']}, "
    f"{orphan_task_46['y']})"
)

print(
    f"  Recovery UUV                : "
    f"{orphan_successor_46}"
)

print(
    f"  Completion timestep         : "
    f"{orphan_completion_time_46}"
)

print(
    f"  Total recovery delay        : "
    f"{total_recovery_delay_46} steps"
)

print(
    f"  Recovery completed          : "
    f"{orphan_completed_46}"
)


print(
    "\nMission performance:"
)

print(
    f"  Start coverage              : "
    f"{100 * start_coverage_46:.4f}%"
)

print(
    f"  Final coverage              : "
    f"{100 * final_coverage_46:.4f}%"
)

print(
    f"  Resources discovered        : "
    f"{len(team_discovered_resources_46)}"
    f"/{len(resources)}"
)

print(
    f"  Healthy post-failure motion : "
    f"{healthy_motion_steps_after_failure_46} steps"
)

print(
    f"  Failed-UUV displacement     : "
    f"{failed_displacement_46:.2f} px"
)

print(
    f"  Safety overrides            : "
    f"{sum(safety_overrides_46.values())}"
)

print(
    f"  Minimum separation          : "
    f"{mission_min_separation_46:.2f} px"
)


print(
    "\nAnimation:"
)

print(
    f"  Frames                      : "
    f"{len(frames_46)}"
)

print(
    f"  GIF FPS                     : "
    f"{GIF_FPS_46}"
)

print(
    f"  Output                      : "
    f"{GIF_PATH_46.resolve()}"
)


print(
    "\nEnd-to-end decentralized fault-recovery "
    "showcase generated."
)

# Combo 47 — Final Prototype Metrics and Artifact Export

## Objective

The autonomy architecture and showcase demonstrations are now complete.

This combo consolidates the validated experimental results into reusable artifacts.

The goal is to produce:

```text
artifacts/
├── multiuuv_final_metrics_prototype.csv
├── multiuuv_final_summary_prototype.json
└── frontier_value_model_prototype.pt
```

These files provide:

- quantitative project results;
- machine-readable benchmark summaries;
- trained PyTorch model weights.

---

# Included Results

The final metrics table should summarize:

### Learned model

- test MSE;
- test MAE;
- test $R^2$;
- Pearson correlation;
- Spearman correlation;
- top-1 ranking agreement.

### Multi-seed controllers

For:

```text
Classical
Learned
Hybrid
```

include:

- mean final coverage;
- final coverage standard deviation;
- mean coverage AUC;
- mean incremental redundancy;
- redundancy standard deviation;
- mean HOLD steps;
- mean safety overrides;
- mean goal changes;
- mean minimum separation.

### Hybrid authority

Include:

- learned agreements;
- learned overrides;
- classical fallbacks;
- learned-authorized fraction.

### Fault-recovery showcase

Include:

- physical failure time;
- confirmation time;
- detection delay;
- orphan-task completion time;
- total recovery delay;
- failed-UUV displacement;
- final coverage;
- resources discovered;
- safety overrides;
- minimum separation.

---

# Scientific Summary

The final prototype supports:

$$
\boxed{
\text{Local Perception}
+
\text{Persistent Private Memory}
+
\text{Frontier Exploration}
+
\text{Distributed Task Allocation}
}
$$

plus:

$$
\boxed{
\text{Limited Communication}
+
\text{Packet Loss}
+
\text{Latency}
+
\text{Persistent Link Blackout}
}
$$

plus:

$$
\boxed{
\text{Collision Avoidance}
+
\text{Failure Detection}
+
\text{Task Recovery}
}
$$

and:

$$
\boxed{
\text{PyTorch Learned Frontier Valuation}
+
\text{Confidence-Gated Hybrid Planning}
}
$$

---

# Important Learned-Controller Result

The learned model accurately approximated the classical scoring function, but fully learned closed-loop control did not consistently outperform the classical planner across stochastic seeds.

The hybrid architecture therefore retains the learned component while using classical guardrails.

This should be stated explicitly in exported results.

---

# Model Export

Store:

```python
frontier_model_38.state_dict()
```

along with:

- feature mean;
- feature standard deviation;
- feature column names;
- model architecture information.

This allows later modularization without retraining the notebook model.

---

# Expected Output

Report:

- CSV path;
- JSON path;
- PyTorch model path;
- file existence;
- file sizes.

A successful result establishes:

$$
\boxed{
\text{Reproducible Final Prototype Artifacts}
}
$$

---

In [ ]:
# ============================================================
# Combo 47 — Final Metrics + Artifact Export
# ============================================================

from pathlib import Path
import json
import torch


# ============================================================
# Artifact directory
# ============================================================

ARTIFACT_DIR_47 = (
    Path.cwd().parent
    /
    "artifacts"
)

ARTIFACT_DIR_47.mkdir(
    parents=True,
    exist_ok=True,
)


CSV_PATH_47 = (
    ARTIFACT_DIR_47
    /
    "multiuuv_final_metrics_prototype.csv"
)


JSON_PATH_47 = (
    ARTIFACT_DIR_47
    /
    "multiuuv_final_summary_prototype.json"
)


MODEL_PATH_47 = (
    ARTIFACT_DIR_47
    /
    "frontier_value_model_prototype.pt"
)


# ============================================================
# Learned model metrics
# ============================================================

learned_model_metrics_47 = {
    "test_mse":
        float(
            test_metrics_38[
                "MSE"
            ]
        ),

    "test_mae":
        float(
            test_metrics_38[
                "MAE"
            ]
        ),

    "test_r2":
        float(
            test_metrics_38[
                "R2"
            ]
        ),

    "test_pearson":
        float(
            test_metrics_38[
                "Pearson"
            ]
        ),

    "test_spearman":
        float(
            test_metrics_38[
                "Spearman"
            ]
        ),

    "top1_agreement":
        float(
            top1_accuracy_38
        ),

    "model_parameters":
        int(
            model_parameter_count_38
        ),
}


# ============================================================
# Controller summary from Combo 44
# ============================================================

controller_records_47 = []


for _, row in summary_df_44.iterrows():

    controller_records_47.append(
        {
            "controller":
                str(
                    row[
                        "controller"
                    ]
                ),

            "final_coverage_mean_pct":
                float(
                    row[
                        "final_coverage_mean"
                    ]
                ),

            "final_coverage_std_pct":
                float(
                    row[
                        "final_coverage_std"
                    ]
                ),

            "coverage_auc_mean":
                float(
                    row[
                        "coverage_auc_mean"
                    ]
                ),

            "redundancy_mean":
                float(
                    row[
                        "redundancy_mean"
                    ]
                ),

            "redundancy_std":
                float(
                    row[
                        "redundancy_std"
                    ]
                ),

            "hold_mean":
                float(
                    row[
                        "hold_mean"
                    ]
                ),

            "safety_override_mean":
                float(
                    row[
                        "safety_override_mean"
                    ]
                ),

            "goal_changes_mean":
                float(
                    row[
                        "goal_changes_mean"
                    ]
                ),

            "min_separation_mean_px":
                float(
                    row[
                        "min_separation_mean"
                    ]
                ),
        }
    )


controller_metrics_df_47 = pd.DataFrame(
    controller_records_47
)


# ============================================================
# Hybrid authority
# ============================================================

hybrid_authority_47 = {
    "learned_agreements":
        int(
            hybrid_agreement_total_44
        ),

    "learned_overrides":
        int(
            hybrid_override_total_44
        ),

    "classical_fallbacks":
        int(
            hybrid_fallback_total_44
        ),

    "total_decisions":
        int(
            hybrid_decision_total_44
        ),

    "learned_authorized_fraction":
        (
            float(
                (
                    hybrid_agreement_total_44
                    +
                    hybrid_override_total_44
                )
                /
                hybrid_decision_total_44
            )
            if hybrid_decision_total_44 > 0
            else 0.0
        ),
}


# ============================================================
# Fault-recovery showcase metrics
# ============================================================

fault_showcase_metrics_47 = {
    "mission_steps":
        int(
            SHOWCASE_STEPS_46
        ),

    "failed_uuv":
        int(
            FAILED_UUV_ID_46
        ),

    "physical_failure_time":
        int(
            PHYSICAL_FAILURE_TIME_46
        ),

    "majority_confirmation_time":
        int(
            majority_confirmation_time_46
        ),

    "detection_delay":
        int(
            detection_delay_46
        ),

    "orphan_source":
        str(
            orphan_source_46
        ),

    "orphan_goal_x":
        int(
            orphan_task_46[
                "x"
            ]
        ),

    "orphan_goal_y":
        int(
            orphan_task_46[
                "y"
            ]
        ),

    "recovery_uuv":
        int(
            orphan_successor_46
        ),

    "completion_time":
        int(
            orphan_completion_time_46
        ),

    "total_recovery_delay":
        int(
            total_recovery_delay_46
        ),

    "failed_displacement_px":
        float(
            failed_displacement_46
        ),

    "final_coverage_pct":
        float(
            final_coverage_46
            *
            100.0
        ),

    "resources_discovered":
        int(
            len(
                team_discovered_resources_46
            )
        ),

    "resources_total":
        int(
            len(
                resources
            )
        ),

    "safety_overrides":
        int(
            sum(
                safety_overrides_46.values()
            )
        ),

    "minimum_separation_px":
        float(
            mission_min_separation_46
        ),
}


# ============================================================
# CSV export
#
# One compact controller-comparison table.
# ============================================================

controller_metrics_df_47.to_csv(
    CSV_PATH_47,
    index=False,
)


# ============================================================
# JSON summary
# ============================================================

final_summary_47 = {
    "project":
        "MultiUUV-Link",

    "prototype_status":
        "validated_research_prototype",

    "architecture":
        {
            "agents":
                int(
                    len(
                        agents
                    )
                ),

            "local_fov_pixels":
                [
                    int(
                        FOV_WIDTH
                    ),
                    int(
                        FOV_HEIGHT
                    ),
                ],

            "motion_step_pixels":
                int(
                    ROLLOUT_STEP_SIZE
                ),

            "safe_separation_pixels":
                float(
                    SAFE_SEPARATION
                ),

            "communication_range_pixels":
                float(
                    COMMUNICATION_RANGE
                ),

            "packet_loss_probability":
                float(
                    PACKET_LOSS_PROBABILITY
                ),

            "communication_delay_range":
                [
                    int(
                        MIN_COMM_DELAY
                    ),
                    int(
                        MAX_COMM_DELAY
                    ),
                ],

            "blackout_start_probability":
                float(
                    BLACKOUT_START_PROBABILITY
                ),
        },

    "learned_model":
        learned_model_metrics_47,

    "controllers":
        controller_records_47,

    "hybrid_authority":
        hybrid_authority_47,

    "fault_recovery_showcase":
        fault_showcase_metrics_47,

    "scientific_interpretation":
        {
            "learned_model_role":
                (
                    "The PyTorch model learns frontier-value "
                    "ranking from classical supervision."
                ),

            "closed_loop_result":
                (
                    "Fully learned ranking did not consistently "
                    "outperform the classical planner across "
                    "matched stochastic communication seeds."
                ),

            "final_architecture_choice":
                (
                    "Confidence-gated hybrid planning retains "
                    "the learned model while using classical "
                    "fallbacks to bound ranking error."
                ),
        },

    "artifacts":
        {
            "normal_mission_gif":
                str(
                    GIF_PATH_45.resolve()
                ),

            "fault_recovery_gif":
                str(
                    GIF_PATH_46.resolve()
                ),

            "metrics_csv":
                str(
                    CSV_PATH_47.resolve()
                ),

            "summary_json":
                str(
                    JSON_PATH_47.resolve()
                ),

            "model":
                str(
                    MODEL_PATH_47.resolve()
                ),
        },
}


with open(
    JSON_PATH_47,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_summary_47,
        file,
        indent=2,
    )


# ============================================================
# PyTorch model export
# ============================================================

model_export_47 = {
    "model_state_dict":
        frontier_model_38
        .state_dict(),

    "feature_columns":
        list(
            FEATURE_COLUMNS_37
        ),

    "feature_mean":
        feature_mean_np_38
        .astype(
            np.float32
        ),

    "feature_std":
        feature_std_np_38
        .astype(
            np.float32
        ),

    "architecture":
        {
            "input_dim":
                int(
                    len(
                        FEATURE_COLUMNS_37
                    )
                ),

            "hidden_layers":
                [
                    32,
                    32,
                ],

            "output_dim":
                1,

            "activation":
                "ReLU",
        },

    "training_seed":
        int(
            TRAIN_SEED_38
        ),
}


torch.save(
    model_export_47,
    MODEL_PATH_47,
)


# ============================================================
# Validation
# ============================================================

artifact_paths_47 = [
    CSV_PATH_47,
    JSON_PATH_47,
    MODEL_PATH_47,
]


for artifact_path in artifact_paths_47:

    if not artifact_path.exists():

        raise RuntimeError(
            f"Artifact was not created: "
            f"{artifact_path}"
        )


    if artifact_path.stat().st_size <= 0:

        raise RuntimeError(
            f"Artifact is empty: "
            f"{artifact_path}"
        )


# ============================================================
# Artifact report
# ============================================================

artifact_records_47 = []


for artifact_path in artifact_paths_47:

    artifact_records_47.append(
        {
            "artifact":
                artifact_path.name,

            "size_bytes":
                int(
                    artifact_path
                    .stat()
                    .st_size
                ),

            "path":
                str(
                    artifact_path.resolve()
                ),
        }
    )


artifact_df_47 = pd.DataFrame(
    artifact_records_47
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 110
)

print(
    "MultiUUV-Link — Final Prototype "
    "Metrics and Artifact Export"
)

print(
    "=" * 110
)


print(
    "\nLearned frontier model:"
)

print(
    f"  Test R²                    : "
    f"{learned_model_metrics_47['test_r2']:.6f}"
)

print(
    f"  Test Spearman              : "
    f"{learned_model_metrics_47['test_spearman']:.6f}"
)

print(
    f"  Top-1 agreement            : "
    f"{100 * learned_model_metrics_47['top1_agreement']:.2f}%"
)


print(
    "\nFinal controller benchmark:"
)

print(
    controller_metrics_df_47.to_string(
        index=False,
        formatters={
            "final_coverage_mean_pct":
                lambda x: f"{x:.4f}",

            "final_coverage_std_pct":
                lambda x: f"{x:.4f}",

            "coverage_auc_mean":
                lambda x: f"{x:.2f}",

            "redundancy_mean":
                lambda x: f"{x:.4f}",

            "redundancy_std":
                lambda x: f"{x:.4f}",

            "hold_mean":
                lambda x: f"{x:.2f}",

            "safety_override_mean":
                lambda x: f"{x:.2f}",

            "goal_changes_mean":
                lambda x: f"{x:.2f}",

            "min_separation_mean_px":
                lambda x: f"{x:.2f}",
        },
    )
)


print(
    "\nFault-recovery showcase:"
)

print(
    f"  Detection delay            : "
    f"{detection_delay_46} steps"
)

print(
    f"  Total recovery delay       : "
    f"{total_recovery_delay_46} steps"
)

print(
    f"  Final coverage             : "
    f"{100 * final_coverage_46:.4f}%"
)

print(
    f"  Resources discovered       : "
    f"{len(team_discovered_resources_46)}"
    f"/{len(resources)}"
)

print(
    f"  Minimum separation         : "
    f"{mission_min_separation_46:.2f} px"
)


print(
    "\nExported artifacts:"
)

print(
    artifact_df_47.to_string(
        index=False
    )
)


print(
    "\nFinal prototype metrics, model, "
    "and machine-readable summary exported."
)

# Combo 48 — Final Prototype Validation and Completion Report

## Objective

The MultiUUV-Link research prototype has now completed:

- decentralized exploration;
- persistent local memory;
- communication-aware knowledge sharing;
- distributed frontier allocation;
- collision avoidance;
- packet loss;
- latency;
- persistent communication blackout;
- permanent UUV failure;
- distributed heartbeat-based failure detection;
- orphaned-task reassignment;
- PyTorch frontier-value learning;
- confidence-gated hybrid planning;
- multi-seed controller benchmarking;
- animated mission visualization;
- machine-readable artifact export.

This final combo performs a compact validation sweep over the finished notebook state and produces the final prototype completion report.

---

# Final Architecture

The validated autonomy stack is:

```text
Local FOV Perception
        ↓
Persistent Private Exploration Memory
        ↓
Frontier Extraction
        ↓
Communication-Aware Knowledge Fusion
        ↓
Distributed Task Allocation
        ↓
Confidence-Gated Classical + Learned Ranking
        ↓
Predictive Safety Shield
        ↓
Discrete UUV Motion
```

Communication is degraded through:

```text
Geometric Range
        ↓
Persistent Link Blackout
        ↓
Packet Loss
        ↓
Latency
```

Fault recovery adds:

```text
Physical UUV Failure
        ↓
Heartbeat Silence
        ↓
Contact-Aware Suspicion
        ↓
Majority Confirmation
        ↓
Orphaned Task Creation
        ↓
Healthy Successor Reassignment
        ↓
Mission Continuation
```

---

# Final Controller Choice

The final controller is the confidence-gated hybrid architecture.

Across the multi-seed benchmark:

- classical control remained strong;
- fully learned control showed higher closed-loop variability;
- hybrid control achieved the best mean final coverage;
- hybrid control achieved the lowest mean redundancy;
- hybrid control produced the smallest coverage variance.

Therefore the final prototype architecture is:

$$
\boxed{
\text{Classical Autonomy}
+
\text{PyTorch Learned Advisor}
+
\text{Confidence Gate}
}
$$

---

# Final Validation Checklist

The notebook should verify that the following objects exist:

```text
resources
agents
known_exploration_maps
frontier_model_38
summary_df_44
GIF_PATH_45
GIF_PATH_46
CSV_PATH_47
JSON_PATH_47
MODEL_PATH_47
```

The final exported files must exist and have non-zero size.

---

# Core Numerical Invariants

Validate:

### Agent count

$$
N=4
$$

### Local FOV

$$
128\times128
$$

### Safety radius

$$
d_{safe}=128
$$

### Communication range

$$
R_c=400
$$

### Packet loss

$$
P_{loss}=0.25
$$

### Communication delay

$$
\tau\in[1,5]
$$

### Fault showcase safety

$$
d_{min}>128
$$

### Failed-UUV immobility

$$
\Delta p_f=0
$$

### Fault recovery

$$
g^{orphan}\rightarrow\text{completed}
$$

### Resource discovery

$$
10/10
$$

---

# Learned Model Validation

Verify:

$$
R^2>0.99
$$

$$
\rho_{Spearman}>0.99
$$

and:

$$
A_{top1}>0.90.
$$

This confirms that the PyTorch model genuinely learned the intended frontier-value function.

---

# Final Status

If all checks pass, declare:

```text
PROTOTYPE STATUS: COMPLETE
```

This status refers to the current research prototype.

Future work such as:

- ROS 2 integration;
- Gazebo;
- real UUV hardware;
- hydrodynamics;
- acoustic modem modeling;
- richer learned policies;
- larger swarms;

belongs to later development phases rather than this notebook prototype.

---

# Expected Output

The code cell should print:

- architecture checklist;
- exported artifact checklist;
- learned-model checks;
- controller benchmark winner summary;
- fault-recovery checks;
- final prototype status.

A successful run establishes:

$$
\boxed{
\text{MultiUUV-Link Prototype Complete}
}
$$

---

In [ ]:
# ============================================================
# Combo 48 — Final Prototype Validation and Completion Report
# ============================================================

from pathlib import Path


# ============================================================
# Object existence checks
# ============================================================

required_object_names_48 = [
    "resources",
    "agents",
    "known_exploration_maps",
    "frontier_model_38",
    "summary_df_44",
    "GIF_PATH_45",
    "GIF_PATH_46",
    "CSV_PATH_47",
    "JSON_PATH_47",
    "MODEL_PATH_47",
]


object_checks_48 = {
    name:
        (
            name
            in
            globals()
        )

    for name
    in required_object_names_48
}


all_objects_present_48 = all(
    object_checks_48.values()
)


# ============================================================
# Artifact existence checks
# ============================================================

artifact_paths_48 = {
    "normal_mission_gif":
        Path(
            GIF_PATH_45
        ),

    "fault_recovery_gif":
        Path(
            GIF_PATH_46
        ),

    "metrics_csv":
        Path(
            CSV_PATH_47
        ),

    "summary_json":
        Path(
            JSON_PATH_47
        ),

    "frontier_model":
        Path(
            MODEL_PATH_47
        ),
}


artifact_checks_48 = {}


for name, path in (
    artifact_paths_48.items()
):

    artifact_checks_48[
        name
    ] = {
        "exists":
            path.exists(),

        "nonempty":
            (
                path.exists()
                and
                path.stat().st_size
                >
                0
            ),

        "size_bytes":
            (
                int(
                    path.stat().st_size
                )
                if path.exists()
                else 0
            ),

        "path":
            str(
                path.resolve()
            ),
    }


all_artifacts_valid_48 = all(
    record[
        "exists"
    ]
    and
    record[
        "nonempty"
    ]

    for record
    in artifact_checks_48.values()
)


# ============================================================
# Architecture invariant checks
# ============================================================

architecture_checks_48 = {
    "four_agents":
        (
            len(
                agents
            )
            ==
            4
        ),

    "fov_width_128":
        (
            FOV_WIDTH
            ==
            128
        ),

    "fov_height_128":
        (
            FOV_HEIGHT
            ==
            128
        ),

    "rollout_step_32":
        (
            ROLLOUT_STEP_SIZE
            ==
            32
        ),

    "safe_separation_128":
        bool(
            np.isclose(
                SAFE_SEPARATION,
                128.0,
            )
        ),

    "communication_range_400":
        bool(
            np.isclose(
                COMMUNICATION_RANGE,
                400.0,
            )
        ),

    "packet_loss_025":
        bool(
            np.isclose(
                PACKET_LOSS_PROBABILITY,
                0.25,
            )
        ),

    "minimum_delay_1":
        (
            MIN_COMM_DELAY
            ==
            1
        ),

    "maximum_delay_5":
        (
            MAX_COMM_DELAY
            ==
            5
        ),
}


all_architecture_checks_48 = all(
    architecture_checks_48.values()
)


# ============================================================
# Learned-model checks
# ============================================================

learned_checks_48 = {
    "test_r2_gt_099":
        (
            test_metrics_38[
                "R2"
            ]
            >
            0.99
        ),

    "test_spearman_gt_099":
        (
            test_metrics_38[
                "Spearman"
            ]
            >
            0.99
        ),

    "top1_gt_090":
        (
            top1_accuracy_38
            >
            0.90
        ),

    "model_parameters_positive":
        (
            model_parameter_count_38
            >
            0
        ),
}


all_learned_checks_48 = all(
    learned_checks_48.values()
)


# ============================================================
# Controller benchmark winner summary
# ============================================================

controller_summary_48 = (
    summary_df_44.copy()
)


coverage_winner_48 = (
    controller_summary_48.loc[
        controller_summary_48[
            "final_coverage_mean"
        ].idxmax(),
        "controller",
    ]
)


coverage_auc_winner_48 = (
    controller_summary_48.loc[
        controller_summary_48[
            "coverage_auc_mean"
        ].idxmax(),
        "controller",
    ]
)


redundancy_winner_48 = (
    controller_summary_48.loc[
        controller_summary_48[
            "redundancy_mean"
        ].idxmin(),
        "controller",
    ]
)


coverage_variance_winner_48 = (
    controller_summary_48.loc[
        controller_summary_48[
            "final_coverage_std"
        ].idxmin(),
        "controller",
    ]
)


hybrid_benchmark_checks_48 = {
    "hybrid_best_mean_coverage":
        (
            coverage_winner_48
            ==
            "Hybrid"
        ),

    "hybrid_best_coverage_auc":
        (
            coverage_auc_winner_48
            ==
            "Hybrid"
        ),

    "hybrid_lowest_redundancy":
        (
            redundancy_winner_48
            ==
            "Hybrid"
        ),

    "hybrid_lowest_coverage_variance":
        (
            coverage_variance_winner_48
            ==
            "Hybrid"
        ),
}


all_hybrid_benchmark_checks_48 = all(
    hybrid_benchmark_checks_48.values()
)


# ============================================================
# Fault-recovery checks
# ============================================================

fault_checks_48 = {
    "failure_confirmed_after_failure":
        (
            majority_confirmation_time_46
            >
            PHYSICAL_FAILURE_TIME_46
        ),

    "recovery_task_created":
        (
            orphan_task_46
            is not None
        ),

    "healthy_successor_selected":
        (
            orphan_successor_46
            is not None
            and
            orphan_successor_46
            !=
            FAILED_UUV_ID_46
        ),

    "recovery_completed":
        bool(
            orphan_completed_46
        ),

    "failed_uuv_immobile":
        bool(
            np.isclose(
                failed_displacement_46,
                0.0,
            )
        ),

    "safety_preserved":
        (
            mission_min_separation_46
            >=
            SAFE_SEPARATION
        ),

    "all_resources_discovered":
        (
            len(
                team_discovered_resources_46
            )
            ==
            len(
                resources
            )
        ),

    "healthy_swarm_continued":
        (
            healthy_motion_steps_after_failure_46
            >
            0
        ),
}


all_fault_checks_48 = all(
    fault_checks_48.values()
)


# ============================================================
# Final prototype status
# ============================================================

prototype_complete_48 = all(
    [
        all_objects_present_48,
        all_artifacts_valid_48,
        all_architecture_checks_48,
        all_learned_checks_48,
        all_hybrid_benchmark_checks_48,
        all_fault_checks_48,
    ]
)


prototype_status_48 = (
    "COMPLETE"
    if prototype_complete_48
    else
    "VALIDATION FAILED"
)


# ============================================================
# Tables
# ============================================================

object_df_48 = pd.DataFrame(
    [
        {
            "object":
                name,

            "present":
                status,
        }

        for name, status
        in object_checks_48.items()
    ]
)


artifact_df_48 = pd.DataFrame(
    [
        {
            "artifact":
                name,

            **record,
        }

        for name, record
        in artifact_checks_48.items()
    ]
)


architecture_df_48 = pd.DataFrame(
    [
        {
            "check":
                name,

            "passed":
                status,
        }

        for name, status
        in architecture_checks_48.items()
    ]
)


learned_df_48 = pd.DataFrame(
    [
        {
            "check":
                name,

            "passed":
                status,
        }

        for name, status
        in learned_checks_48.items()
    ]
)


fault_df_48 = pd.DataFrame(
    [
        {
            "check":
                name,

            "passed":
                status,
        }

        for name, status
        in fault_checks_48.items()
    ]
)


# ============================================================
# Console report
# ============================================================

print(
    "=" * 116
)

print(
    "MultiUUV-Link — Final Prototype Validation"
)

print(
    "=" * 116
)


print(
    "\nArchitecture:"
)

print(
    architecture_df_48.to_string(
        index=False
    )
)


print(
    "\nCore notebook objects:"
)

print(
    object_df_48.to_string(
        index=False
    )
)


print(
    "\nExported artifacts:"
)

print(
    artifact_df_48[
        [
            "artifact",
            "exists",
            "nonempty",
            "size_bytes",
            "path",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nLearned model validation:"
)

print(
    learned_df_48.to_string(
        index=False
    )
)


print(
    "\nMeasured learned-model performance:"
)

print(
    f"  Test R²             : "
    f"{test_metrics_38['R2']:.6f}"
)

print(
    f"  Test Spearman       : "
    f"{test_metrics_38['Spearman']:.6f}"
)

print(
    f"  Top-1 agreement     : "
    f"{100 * top1_accuracy_38:.2f}%"
)


print(
    "\nController benchmark winners:"
)

print(
    f"  Mean final coverage       : "
    f"{coverage_winner_48}"
)

print(
    f"  Coverage AUC              : "
    f"{coverage_auc_winner_48}"
)

print(
    f"  Lowest redundancy         : "
    f"{redundancy_winner_48}"
)

print(
    f"  Lowest coverage variance  : "
    f"{coverage_variance_winner_48}"
)


print(
    "\nFault-recovery validation:"
)

print(
    fault_df_48.to_string(
        index=False
    )
)


print(
    "\nFault-recovery metrics:"
)

print(
    f"  Physical failure time     : "
    f"{PHYSICAL_FAILURE_TIME_46}"
)

print(
    f"  Confirmation time         : "
    f"{majority_confirmation_time_46}"
)

print(
    f"  Detection delay           : "
    f"{detection_delay_46} steps"
)

print(
    f"  Recovery completion       : "
    f"{orphan_completion_time_46}"
)

print(
    f"  Total recovery delay      : "
    f"{total_recovery_delay_46} steps"
)

print(
    f"  Final coverage            : "
    f"{100 * final_coverage_46:.4f}%"
)

print(
    f"  Resources discovered      : "
    f"{len(team_discovered_resources_46)}"
    f"/{len(resources)}"
)

print(
    f"  Minimum separation        : "
    f"{mission_min_separation_46:.2f} px"
)


print(
    "\n" + "=" * 116
)

print(
    f"PROTOTYPE STATUS: "
    f"{prototype_status_48}"
)

print(
    "=" * 116
)


if not prototype_complete_48:

    raise RuntimeError(
        "Final MultiUUV-Link prototype validation failed."
    )


print(
    "\nMultiUUV-Link research prototype "
    "completed and validated."
)